
# TA-GSNN: Trust-Aware Adaptive Graph Sparsification for Social Internet of Things

**Research Project:** Trust-Aware Adaptive Graph Sparsification Using Graph Neural Networks for Social Internet of Things  
**Target Journal:** Scientific Reports (Nature Portfolio / Springer Nature)  
**Notebook:** TA_GSNN_Experiments.ipynb  
**Status:** Initial experimental development — no results validated yet.

---

## 1. Research Objective

This project investigates whether a trust-aware Graph Neural Network (GNN) can learn to sparsify signed social trust networks while preserving trust prediction performance and essential graph connectivity.

The primary hypothesis is that adaptive edge selection can achieve a better trade-off between sparsity, prediction performance, and structural preservation than conventional pruning strategies.

## 2. Proposed Method: TA-GSNN

1. **Signed Graph Encoder:** Learns node representations using positive and negative trust relationships.
2. **Adaptive Edge Gate:** Learns the importance of observed edges and selects a sparse graph.
3. **Trust Predictor:** Predicts hidden trust relationships.

The preliminary training objective combines prediction loss, sparsity regularization, and degree preservation.

## 3. Datasets

- Bitcoin Alpha: https://snap.stanford.edu/data/soc-sign-bitcoinalpha.html
- Bitcoin OTC: https://snap.stanford.edu/data/soc-sign-bitcoinotc.html

**Limitation:** These are proxy social trust datasets, not native SIoT device networks.

## 4. Evaluation Protocol

Chronological split:
- 60% observed context graph
- 20% training targets
- 10% validation targets
- 10% held-out test targets

Prevent target-edge and label leakage during graph construction, edge selection, and feature extraction. Check repeated directed pairs.

## 5. Baselines

- Full Signed GNN
- Random Edge Pruning
- Trust Magnitude-Based Pruning
- Degree-Aware Top-K
- Proposed TA-GSNN
- TA-GSNN without degree regularization

## 6. Evaluation Metrics

**Prediction:** ROC-AUC, PR-AUC, Macro-F1, Balanced Accuracy.

**Structure:** Edge removal ratio, largest weakly connected component ratio, isolated-node ratio.

**Efficiency:** Inference time and memory consumption.

**Budgets:** 20%, 50%, and 80% edge removal.

**Seeds:** 42, 43, 44, 45, 46.

## 7. Experiment Stages

1. Environment and data verification
2. Full Signed GNN baseline
3. Conventional sparsification baselines
4. Trust-aware adaptive edge gating
5. Ablation and repeated-seed experiments
6. Reproducibility and result export

## 8. Reproducibility

Record software versions, dataset provenance, random seeds, preprocessing decisions, evaluation settings, and numerical results. Save results as CSV or JSON.

Do not report unverified or fabricated experimental findings.

---

**Current milestone: Stage 1 — Environment and Dataset Preparation**
  

In [ ]:

# ============================================================
# TA-GSNN | Stage 1.1: Environment & GPU Verification
# ============================================================

import sys
import os
import platform
import random
import json
import importlib.util
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 55)
print("TA-GSNN | ENVIRONMENT VERIFICATION")
print("=" * 55)

print(f"Python version : {sys.version.split()[0]}")
print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version  : {np.__version__}")
print(f"Pandas version : {pd.__version__}")
print(f"Platform       : {platform.platform()}")
print(f"Device         : {DEVICE}")
print(f"CUDA available : {torch.cuda.is_available()}")
print(f"CUDA version   : {torch.version.cuda}")

if torch.cuda.is_available():
    print(f"GPU name       : {torch.cuda.get_device_name(0)}")
    gpu_properties = torch.cuda.get_device_properties(0)
    print(
        f"GPU memory     : "
        f"{gpu_properties.total_memory / (1024**3):.2f} GB"
    )

print("\nRequired libraries:")
for package in ["sklearn", "networkx", "matplotlib", "torch_geometric"]:
    installed = importlib.util.find_spec(package) is not None
    print(f"  {package:18s}: {'OK' if installed else 'NOT INSTALLED'}")

# Verify that a simple operation runs on the selected device
x = torch.randn(100, 32, device=DEVICE)
y = torch.matmul(x, x.T)

print(f"\nTensor test    : {'PASS' if y.shape == (100, 100) else 'FAIL'}")

# Save environment information for reproducibility
environment_info = {
    "project": "TA-GSNN",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version.split()[0],
    "pytorch": torch.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "device": str(DEVICE),
    "cuda_version": torch.version.cuda,
    "gpu_name": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available() else None
    ),
    "seed": SEED,
}

os.makedirs("/kaggle/working/ta_gsnn_results", exist_ok=True)

with open(
    "/kaggle/working/ta_gsnn_results/environment.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(environment_info, f, indent=2)

print("\nEnvironment information saved successfully.")
print("=" * 55)


In [ ]:

# ============================================================
# TA-GSNN | Stage 1.2: Bitcoin Alpha Dataset Validation
# ============================================================

from pathlib import Path
import urllib.request
import gzip
import hashlib
import json

import numpy as np
import pandas as pd

DATA_DIR = Path("/kaggle/working/ta_gsnn_data")
RESULTS_DIR = Path("/kaggle/working/ta_gsnn_results")

DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DATASET_URL = (
    "https://snap.stanford.edu/data/"
    "soc-sign-bitcoinalpha.csv.gz"
)

ARCHIVE_PATH = DATA_DIR / "soc-sign-bitcoinalpha.csv.gz"

if not ARCHIVE_PATH.exists():
    print("Downloading Bitcoin Alpha dataset...")
    urllib.request.urlretrieve(DATASET_URL, ARCHIVE_PATH)
else:
    print("Using previously downloaded dataset.")

# Record file checksum for reproducibility
sha256 = hashlib.sha256()

with open(ARCHIVE_PATH, "rb") as file:
    for chunk in iter(lambda: file.read(1024 * 1024), b""):
        sha256.update(chunk)

checksum = sha256.hexdigest()

# Official SNAP columns:
# SOURCE, TARGET, RATING, TIME

with gzip.open(ARCHIVE_PATH, "rt") as file:
    df = pd.read_csv(
        file,
        header=None,
        names=["source", "target", "rating", "timestamp"]
    )

# Basic validation
assert df.shape[1] == 4, "Unexpected number of columns"
assert not df.isna().any().any(), "Missing values detected"
assert df["rating"].between(-10, 10).all(), "Invalid rating"
assert (df["rating"] != 0).all(), "Zero ratings detected"

df["source"] = df["source"].astype(int)
df["target"] = df["target"].astype(int)
df["rating"] = df["rating"].astype(int)
df["timestamp"] = df["timestamp"].astype(int)

# Sort chronologically, preserving original order for ties
df = df.sort_values(
    "timestamp",
    kind="stable"
).reset_index(drop=True)

df["trust_label"] = (df["rating"] > 0).astype(int)

num_nodes = len(
    set(df["source"]).union(set(df["target"]))
)

num_edges = len(df)
positive_edges = int(df["trust_label"].sum())
negative_edges = num_edges - positive_edges

duplicate_pairs = int(
    df.duplicated(subset=["source", "target"]).sum()
)

self_loops = int(
    (df["source"] == df["target"]).sum()
)

time_min = pd.to_datetime(
    df["timestamp"].min(), unit="s", utc=True
)
time_max = pd.to_datetime(
    df["timestamp"].max(), unit="s", utc=True
)

report = {
    "dataset": "Bitcoin Alpha",
    "source_url": DATASET_URL,
    "sha256": checksum,
    "num_nodes": num_nodes,
    "num_edges": num_edges,
    "positive_edges": positive_edges,
    "negative_edges": negative_edges,
    "positive_ratio": positive_edges / num_edges,
    "negative_ratio": negative_edges / num_edges,
    "duplicate_directed_pairs": duplicate_pairs,
    "self_loops": self_loops,
    "timestamp_start": str(time_min),
    "timestamp_end": str(time_max),
}

print("=" * 60)
print("TA-GSNN | BITCOIN ALPHA VALIDATION")
print("=" * 60)

for key, value in report.items():
    print(f"{key:26s}: {value}")

print("\nFirst five chronological interactions:")
print(df.head().to_string(index=False))

print("\nRating distribution:")
print(df["rating"].value_counts().sort_index())

df.to_csv(
    DATA_DIR / "bitcoin_alpha_chronological.csv",
    index=False
)

with open(
    RESULTS_DIR / "bitcoin_alpha_validation.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(report, file, indent=2)

print("\nDataset and validation report saved.")
print("=" * 60)


In [ ]:

# ============================================================
# TA-GSNN | Stage 1.3: Leakage-Free Temporal Splitting
# ============================================================

import numpy as np
import pandas as pd
import json

from pathlib import Path

DATA_DIR = Path("/kaggle/working/ta_gsnn_data")
RESULTS_DIR = Path("/kaggle/working/ta_gsnn_results")

df = pd.read_csv(
    DATA_DIR / "bitcoin_alpha_chronological.csv"
)

df = df.sort_values(
    "timestamp", kind="stable"
).reset_index(drop=True)

N = len(df)

# Do not split interactions sharing the same timestamp.
# Select the nearest valid boundary to each target fraction.

timestamps = df["timestamp"].to_numpy()
valid_boundaries = np.flatnonzero(
    timestamps[1:] != timestamps[:-1]
) + 1

def nearest_boundary(target, lower_bound=0):
    candidates = valid_boundaries[
        valid_boundaries > lower_bound
    ]
    if len(candidates) == 0:
        raise ValueError("No available temporal boundary.")
    return int(candidates[
        np.argmin(np.abs(candidates - target))
    ])

b1 = nearest_boundary(int(N * 0.60))
b2 = nearest_boundary(int(N * 0.80), b1)
b3 = nearest_boundary(int(N * 0.90), b2)

splits = {
    "context": df.iloc[:b1].copy(),
    "train": df.iloc[b1:b2].copy(),
    "validation": df.iloc[b2:b3].copy(),
    "test": df.iloc[b3:].copy()
}

# Verify strict chronological separation.
split_names = list(splits.keys())

for i in range(len(split_names) - 1):
    left = splits[split_names[i]]
    right = splits[split_names[i + 1]]

    assert left["timestamp"].max() < right["timestamp"].min(), (
        "Temporal overlap detected!"
    )

# Check that directed pairs are disjoint.
pair_sets = {
    name: set(zip(part["source"], part["target"]))
    for name, part in splits.items()
}

for i, name_a in enumerate(split_names):
    for name_b in split_names[i + 1:]:
        assert pair_sets[name_a].isdisjoint(
            pair_sets[name_b]
        ), f"Directed pair overlap: {name_a}, {name_b}"

# Check node coverage for future-link prediction.
context_nodes = set(splits["context"]["source"]) | set(
    splits["context"]["target"]
)

summary = []

for name, part in splits.items():

    nodes = set(part["source"]) | set(part["target"])

    unseen_nodes = nodes - context_nodes

    positive = int((part["trust_label"] == 1).sum())
    negative = int((part["trust_label"] == 0).sum())

    summary.append({
        "split": name,
        "edges": len(part),
        "percentage": round(100 * len(part) / N, 2),
        "positive": positive,
        "negative": negative,
        "positive_pct": round(
            100 * positive / len(part), 2
        ),
        "unique_nodes": len(nodes),
        "nodes_not_in_context": len(unseen_nodes),
        "time_start": pd.to_datetime(
            part["timestamp"].min(),
            unit="s",
            utc=True
        ).strftime("%Y-%m-%d"),
        "time_end": pd.to_datetime(
            part["timestamp"].max(),
            unit="s",
            utc=True
        ).strftime("%Y-%m-%d")
    })

    part.to_csv(
        DATA_DIR / f"bitcoin_alpha_{name}.csv",
        index=False
    )

summary_df = pd.DataFrame(summary)

print("=" * 80)
print("TA-GSNN | TEMPORAL SPLIT SUMMARY")
print("=" * 80)

print(summary_df.to_string(index=False))

print("\nTotal edges:", sum(len(x) for x in splits.values()))
print("Original edges:", N)

assert sum(len(x) for x in splits.values()) == N
assert all(len(x) > 0 for x in splits.values())

print("\nTemporal leakage check: PASS")
print("Directed-pair overlap check: PASS")

print("\nContext graph nodes:", len(context_nodes))
print(
    "Total dataset nodes:",
    len(set(df["source"]) | set(df["target"]))
)

summary_df.to_csv(
    RESULTS_DIR / "bitcoin_alpha_split_summary.csv",
    index=False
)

with open(
    RESULTS_DIR / "bitcoin_alpha_split_config.json",
    "w"
) as f:
    json.dump({
        "protocol": "strict_chronological_60_20_10_10",
        "timestamp_ties_kept_together": True,
        "boundaries": [b1, b2, b3],
        "total_edges": N,
        "context_nodes": len(context_nodes),
        "splits": summary
    }, f, indent=2)

print("\nSplit files and summary saved successfully.")


In [ ]:

# ============================================================
# TA-GSNN | Stage 1.4: Node Coverage & Cold-Start Analysis
# ============================================================

from pathlib import Path
import json
import pandas as pd

DATA_DIR = Path("/kaggle/working/ta_gsnn_data")
RESULTS_DIR = Path("/kaggle/working/ta_gsnn_results")

context = pd.read_csv(DATA_DIR / "bitcoin_alpha_context.csv")

context_nodes = (
    set(context["source"]) | set(context["target"])
)

print("=" * 82)
print("TA-GSNN | NODE COVERAGE & COLD-START ANALYSIS")
print("=" * 82)

results = []

for split_name in ["train", "validation", "test"]:

    part = pd.read_csv(
        DATA_DIR / f"bitcoin_alpha_{split_name}.csv"
    )

    source_seen = part["source"].isin(context_nodes)
    target_seen = part["target"].isin(context_nodes)

    both_seen = source_seen & target_seen
    one_unseen = source_seen ^ target_seen
    both_unseen = ~source_seen & ~target_seen

    categories = {
        "both_seen": both_seen,
        "one_unseen": one_unseen,
        "both_unseen": both_unseen
    }

    assert sum(
        int(mask.sum()) for mask in categories.values()
    ) == len(part)

    for category, mask in categories.items():

        subset = part.loc[mask]
        count = len(subset)

        results.append({
            "split": split_name,
            "category": category,
            "edges": count,
            "edge_pct": round(
                100 * count / len(part), 2
            ),
            "positive": int(
                (subset["trust_label"] == 1).sum()
            ),
            "negative": int(
                (subset["trust_label"] == 0).sum()
            ),
            "positive_pct": (
                round(
                    100 * subset["trust_label"].mean(), 2
                )
                if count > 0 else None
            )
        })

summary = pd.DataFrame(results)

print("\nEdge categories by split:")
print(summary.to_string(index=False))

print("\n" + "-" * 82)

for split_name in ["train", "validation", "test"]:

    sub = summary[summary["split"] == split_name]

    total = int(sub["edges"].sum())
    seen = int(
        sub.loc[
            sub["category"] == "both_seen", "edges"
        ].sum()
    )

    cold = total - seen

    print(
        f"{split_name.upper():12s} | "
        f"Seen-node edges: {seen:5d} "
        f"({100 * seen / total:5.1f}%) | "
        f"Cold-start edges: {cold:5d} "
        f"({100 * cold / total:5.1f}%)"
    )

summary.to_csv(
    RESULTS_DIR / "bitcoin_alpha_node_coverage.csv",
    index=False
)

with open(
    RESULTS_DIR / "bitcoin_alpha_node_coverage.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        results, f, indent=2
    )

print("\nAnalysis saved successfully.")
print("=" * 82)


In [ ]:

# ============================================================
# TA-GSNN | Stage 1.5: Expanding Historical Context Diagnostic
# ============================================================

from pathlib import Path
import pandas as pd
import json

DATA_DIR = Path("/kaggle/working/ta_gsnn_data")
RESULTS_DIR = Path("/kaggle/working/ta_gsnn_results")

context = pd.read_csv(DATA_DIR / "bitcoin_alpha_context.csv")
train = pd.read_csv(DATA_DIR / "bitcoin_alpha_train.csv")
validation = pd.read_csv(DATA_DIR / "bitcoin_alpha_validation.csv")
test = pd.read_csv(DATA_DIR / "bitcoin_alpha_test.csv")

def node_set(frame):
    return set(frame["source"]) | set(frame["target"])

def coverage_report(target, history, split_name):
    known = node_set(history)

    src_seen = target["source"].isin(known)
    dst_seen = target["target"].isin(known)

    both_seen = src_seen & dst_seen
    one_new = src_seen ^ dst_seen
    both_new = (~src_seen) & (~dst_seen)

    n = len(target)

    return {
        "split": split_name,
        "history_edges": len(history),
        "history_nodes": len(known),
        "target_edges": n,
        "both_seen_edges": int(both_seen.sum()),
        "both_seen_pct": round(100 * both_seen.mean(), 2),
        "one_new_pct": round(100 * one_new.mean(), 2),
        "both_new_pct": round(100 * both_new.mean(), 2),
        "cold_start_pct": round(
            100 * (~both_seen).mean(), 2
        ),
        "seen_negative": int(
            ((target["trust_label"] == 0) & both_seen).sum()
        ),
        "cold_negative": int(
            ((target["trust_label"] == 0) & (~both_seen)).sum()
        ),
    }

# Original fixed 60% context
fixed_val = coverage_report(
    validation, context, "validation_fixed"
)
fixed_test = coverage_report(
    test, context, "test_fixed"
)

# Expanding history: only previously observed interactions
history_80 = pd.concat(
    [context, train], ignore_index=True
)

history_90 = pd.concat(
    [context, train, validation], ignore_index=True
)

assert history_80["timestamp"].max() < validation["timestamp"].min()
assert history_90["timestamp"].max() < test["timestamp"].min()

expanded_val = coverage_report(
    validation, history_80, "validation_expanding"
)

expanded_test = coverage_report(
    test, history_90, "test_expanding"
)

reports = [
    fixed_val,
    expanded_val,
    fixed_test,
    expanded_test
]

report_df = pd.DataFrame(reports)

print("=" * 90)
print("TA-GSNN | FIXED vs EXPANDING CONTEXT")
print("=" * 90)

print(report_df.to_string(index=False))

print("\nComparison:")
for name, fixed, expanded in [
    ("Validation", fixed_val, expanded_val),
    ("Test", fixed_test, expanded_test),
]:
    change = (
        expanded["both_seen_pct"] - fixed["both_seen_pct"]
    )
    print(
        f"{name}: both-seen coverage "
        f"{fixed['both_seen_pct']:.2f}% -> "
        f"{expanded['both_seen_pct']:.2f}% "
        f"(change: {change:+.2f} percentage points)"
    )

report_df.to_csv(
    RESULTS_DIR / "bitcoin_alpha_expanding_context.csv",
    index=False
)

with open(
    RESULTS_DIR / "bitcoin_alpha_expanding_context.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(reports, f, indent=2)

print("\nDiagnostic saved successfully.")


In [ ]:

# ============================================================
# TA-GSNN | Stage 1.6: Historical Trust Feature Baseline
# ============================================================

from pathlib import Path
from collections import Counter
import json

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix
)

DATA_DIR = Path("/kaggle/working/ta_gsnn_data")
RESULTS_DIR = Path("/kaggle/working/ta_gsnn_results")

parts = {
    name: pd.read_csv(
        DATA_DIR / f"bitcoin_alpha_{name}.csv"
    )
    for name in ["context", "train", "validation", "test"]
}

history = {
    "train": parts["context"],
    "validation": pd.concat(
        [parts["context"], parts["train"]],
        ignore_index=True
    ),
    "test": pd.concat(
        [parts["context"], parts["train"],
         parts["validation"]],
        ignore_index=True
    )
}

FEATURE_NAMES = [
    "src_out_pos", "src_out_neg",
    "src_in_pos", "src_in_neg",
    "dst_out_pos", "dst_out_neg",
    "dst_in_pos", "dst_in_neg",
    "src_seen", "dst_seen",
    "reverse_positive", "reverse_negative"
]

def extract_features(hist, targets):
    out_pos, out_neg = Counter(), Counter()
    in_pos, in_neg = Counter(), Counter()

    reverse_lookup = {}

    for u, v, rating in zip(
        hist["source"],
        hist["target"],
        hist["rating"]
    ):
        u, v = int(u), int(v)
        if rating > 0:
            out_pos[u] += 1
            in_pos[v] += 1
        else:
            out_neg[u] += 1
            in_neg[v] += 1

        reverse_lookup[(u, v)] = 1 if rating > 0 else -1

    seen = (
        set(hist["source"]) |
        set(hist["target"])
    )

    features = []

    for u, v in zip(
        targets["source"], targets["target"]
    ):
        u, v = int(u), int(v)

        reverse = reverse_lookup.get((v, u), 0)

        row = [
            np.log1p(out_pos[u]),
            np.log1p(out_neg[u]),
            np.log1p(in_pos[u]),
            np.log1p(in_neg[u]),
            np.log1p(out_pos[v]),
            np.log1p(out_neg[v]),
            np.log1p(in_pos[v]),
            np.log1p(in_neg[v]),
            float(u in seen),
            float(v in seen),
            float(reverse == 1),
            float(reverse == -1)
        ]

        features.append(row)

    return np.asarray(features, dtype=np.float32)

X = {}
y = {}

for name in ["train", "validation", "test"]:
    X[name] = extract_features(
        history[name], parts[name]
    )

    # Negative trust is the positive class for evaluation.
    y[name] = (
        parts[name]["trust_label"].to_numpy() == 0
    ).astype(int)

    assert np.isfinite(X[name]).all()

    print(
        f"{name:12s}: X={X[name].shape}, "
        f"negative-trust prevalence={y[name].mean():.4f}"
    )

model = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    )
)

model.fit(X["train"], y["train"])

validation_scores = model.predict_proba(
    X["validation"]
)[:, 1]

# Select a classification threshold using validation only.
thresholds = np.linspace(0.05, 0.95, 181)

val_f1 = [
    f1_score(
        y["validation"],
        (validation_scores >= t).astype(int),
        zero_division=0
    )
    for t in thresholds
]

best_threshold = float(
    thresholds[int(np.argmax(val_f1))]
)

def evaluate(name):
    scores = model.predict_proba(X[name])[:, 1]
    preds = (scores >= best_threshold).astype(int)

    return {
        "split": name,
        "roc_auc": float(
            roc_auc_score(y[name], scores)
        ),
        "negative_pr_auc": float(
            average_precision_score(y[name], scores)
        ),
        "macro_f1": float(
            f1_score(
                y[name], preds,
                average="macro",
                zero_division=0
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(y[name], preds)
        ),
        "confusion_matrix": confusion_matrix(
            y[name], preds, labels=[0, 1]
        ).tolist()
    }

reports = [
    evaluate("validation"),
    evaluate("test")
]

print("\n" + "=" * 65)
print("TA-GSNN | HISTORICAL TRUST BASELINE")
print("=" * 65)

print(f"Validation-selected threshold: {best_threshold:.3f}")

for result in reports:
    print(f"\n{result['split'].upper()}")
    print(f"ROC-AUC           : {result['roc_auc']:.4f}")
    print(f"Negative PR-AUC   : {result['negative_pr_auc']:.4f}")
    print(f"Macro-F1          : {result['macro_f1']:.4f}")
    print(f"Balanced Accuracy : {result['balanced_accuracy']:.4f}")
    print(f"Confusion Matrix  : {result['confusion_matrix']}")

with open(
    RESULTS_DIR / "bitcoin_alpha_logistic_baseline.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump({
        "features": FEATURE_NAMES,
        "selected_threshold": best_threshold,
        "results": reports,
        "protocol": "expanding_historical_context"
    }, f, indent=2)

print("\nBaseline results saved.")


In [ ]:

# ============================================================
# TA-GSNN | Stage 1.7: Threshold Calibration Diagnostic
# ============================================================

import numpy as np
import pandas as pd
import json

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score
)

from pathlib import Path

RESULTS_DIR = Path("/kaggle/working/ta_gsnn_results")

# Class 1 = negative trust / distrust
val_scores = model.predict_proba(X["validation"])[:, 1]
test_scores = model.predict_proba(X["test"])[:, 1]

# Candidate thresholds derived from validation probabilities.
# Include extreme thresholds to allow all-one-class predictions.
candidate_thresholds = np.unique(
    np.concatenate([
        [0.0, 0.125, 0.5, 1.0],
        val_scores,
        (np.sort(np.unique(val_scores))[:-1] +
         np.sort(np.unique(val_scores))[1:]) / 2
    ])
)

def macro_f1_at_threshold(t):
    preds = (val_scores >= t).astype(int)
    return f1_score(
        y["validation"],
        preds,
        average="macro",
        zero_division=0
    )

val_macro_scores = np.array([
    macro_f1_at_threshold(t)
    for t in candidate_thresholds
])

# Tie-break toward 0.5 for deterministic selection.
best_indices = np.flatnonzero(
    np.isclose(
        val_macro_scores,
        val_macro_scores.max(),
        rtol=0,
        atol=1e-12
    )
)

best_index = best_indices[
    np.argmin(
        np.abs(candidate_thresholds[best_indices] - 0.5)
    )
]

macro_threshold = float(
    candidate_thresholds[best_index]
)

def evaluate_threshold(split, scores, threshold):
    labels = y[split]
    preds = (scores >= threshold).astype(int)

    return {
        "split": split,
        "threshold": float(threshold),
        "macro_f1": float(
            f1_score(
                labels, preds,
                average="macro",
                zero_division=0
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(labels, preds)
        ),
        "negative_precision": float(
            precision_score(
                labels, preds,
                zero_division=0
            )
        ),
        "negative_recall": float(
            recall_score(
                labels, preds,
                zero_division=0
            )
        ),
        "predicted_negative_pct": float(
            100 * preds.mean()
        ),
        "confusion_matrix": confusion_matrix(
            labels, preds, labels=[0, 1]
        ).tolist()
    }

strategies = {
    "previous_F1_negative": 0.125,
    "default_0.5": 0.5,
    "validation_macro_F1": macro_threshold
}

records = []

for strategy, threshold in strategies.items():
    for split, scores in [
        ("validation", val_scores),
        ("test", test_scores)
    ]:
        result = evaluate_threshold(
            split, scores, threshold
        )
        result["strategy"] = strategy
        records.append(result)

results_df = pd.DataFrame(records)

print("=" * 85)
print("TA-GSNN | THRESHOLD CALIBRATION")
print("=" * 85)

print(f"Macro-F1 optimal validation threshold: {macro_threshold:.6f}")

print("\nValidation probability quantiles:")
print(pd.Series(val_scores).quantile(
    [0, 0.1, 0.25, 0.5, 0.75, 0.9, 1]
).to_string())

print("\nTest probability quantiles:")
print(pd.Series(test_scores).quantile(
    [0, 0.1, 0.25, 0.5, 0.75, 0.9, 1]
).to_string())

print("\nThreshold comparison:")
print(
    results_df[[
        "strategy",
        "split",
        "threshold",
        "macro_f1",
        "balanced_accuracy",
        "negative_precision",
        "negative_recall",
        "predicted_negative_pct"
    ]].round(4).to_string(index=False)
)

print("\nConfusion matrices:")
for row in records:
    print(
        row["strategy"],
        row["split"],
        row["confusion_matrix"]
    )

results_df.to_csv(
    RESULTS_DIR / "bitcoin_alpha_threshold_comparison.csv",
    index=False
)

with open(
    RESULTS_DIR / "bitcoin_alpha_threshold_comparison.json",
    "w"
) as f:
    json.dump(records, f, indent=2)

print("\nThreshold diagnostic saved successfully.")


In [ ]:

# ============================================================
# TA-GSNN | Stage 2.1: Prepare Signed Graph Tensors
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import torch

DATA_DIR = Path("/kaggle/working/ta_gsnn_data")

parts = {
    name: pd.read_csv(
        DATA_DIR / f"bitcoin_alpha_{name}.csv"
    )
    for name in ["context", "train", "validation", "test"]
}

full_df = pd.concat(
    list(parts.values()),
    ignore_index=True
)

# Node IDs are used only for indexing, not as predictive features.
node_ids = sorted(
    set(full_df["source"]) | set(full_df["target"])
)

node_to_idx = {
    node_id: index
    for index, node_id in enumerate(node_ids)
}

NUM_NODES = len(node_ids)

print(f"Total indexed nodes: {NUM_NODES}")

def build_graph_tensors(history):

    src = history["source"].map(node_to_idx).to_numpy()
    dst = history["target"].map(node_to_idx).to_numpy()

    rating = history["rating"].to_numpy()
    positive_mask = rating > 0
    negative_mask = rating < 0

    # Direction-preserving positive and negative edge sets.
    pos_edges = torch.tensor(
        np.stack([
            src[positive_mask],
            dst[positive_mask]
        ]),
        dtype=torch.long,
        device=DEVICE
    )

    neg_edges = torch.tensor(
        np.stack([
            src[negative_mask],
            dst[negative_mask]
        ]),
        dtype=torch.long,
        device=DEVICE
    )

    # Historical node features:
    # positive/negative outgoing and incoming edge counts.
    features = np.zeros(
        (NUM_NODES, 4),
        dtype=np.float32
    )

    np.add.at(features[:, 0], src[positive_mask], 1)
    np.add.at(features[:, 1], src[negative_mask], 1)
    np.add.at(features[:, 2], dst[positive_mask], 1)
    np.add.at(features[:, 3], dst[negative_mask], 1)

    features = np.log1p(features)

    return {
        "x": torch.tensor(
            features, dtype=torch.float32, device=DEVICE
        ),
        "pos_edges": pos_edges,
        "neg_edges": neg_edges
    }

def build_target_tensors(targets):

    edge_pairs = np.stack([
        targets["source"].map(node_to_idx).to_numpy(),
        targets["target"].map(node_to_idx).to_numpy()
    ], axis=1)

    # 1 = negative trust (distrust)
    labels = (
        targets["rating"].to_numpy() < 0
    ).astype(np.float32)

    return (
        torch.tensor(
            edge_pairs, dtype=torch.long, device=DEVICE
        ),
        torch.tensor(
            labels, dtype=torch.float32, device=DEVICE
        )
    )

history_train = parts["context"]

history_validation = pd.concat(
    [parts["context"], parts["train"]],
    ignore_index=True
)

history_test = pd.concat(
    [parts["context"], parts["train"], parts["validation"]],
    ignore_index=True
)

graph_train = build_graph_tensors(history_train)
graph_validation = build_graph_tensors(history_validation)
graph_test = build_graph_tensors(history_test)

targets = {
    name: build_target_tensors(parts[name])
    for name in ["train", "validation", "test"]
}

for name, graph in [
    ("train", graph_train),
    ("validation", graph_validation),
    ("test", graph_test)
]:
    print(
        f"{name:12s} | "
        f"features={tuple(graph['x'].shape)} | "
        f"positive_edges={graph['pos_edges'].shape[1]} | "
        f"negative_edges={graph['neg_edges'].shape[1]}"
    )

print("\nTarget tensor shapes:")
for name, (edges, labels) in targets.items():
    print(
        f"{name:12s} | "
        f"edges={tuple(edges.shape)} | "
        f"labels={tuple(labels.shape)} | "
        f"distrust_rate={labels.mean().item():.4f}"
    )

print("\nStage 2.1 completed successfully.")


In [ ]:

# ============================================================
# TA-GSNN | Stage 2.2: Full Signed GraphSAGE Baseline
# ============================================================

import os
import json
import copy
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    balanced_accuracy_score
)

# ------------------------------------------------------------
# 1. Reproducibility and configuration
# ------------------------------------------------------------

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

RESULTS_DIR = Path("/kaggle/working/ta_gsnn_results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "seed": SEED,
    "hidden_dim": 32,
    "dropout": 0.20,
    "learning_rate": 0.001,
    "weight_decay": 0.0001,
    "max_epochs": 120,
    "patience": 20,
    "selection_metric": "validation_negative_pr_auc",
    "target_class": "negative_trust",
    "graph_protocol": "expanding_historical_context",
}

print("Device:", DEVICE)

# ------------------------------------------------------------
# 2. Directed signed neighborhood aggregation
# ------------------------------------------------------------

def aggregate_mean(h, edges, direction):
    """
    Direction-preserving mean aggregation.

    edges[0] = source
    edges[1] = target

    direction='incoming':
        aggregate source embeddings at target nodes

    direction='outgoing':
        aggregate target embeddings at source nodes
    """

    if direction == "incoming":
        receivers = edges[1]
        senders = edges[0]
    elif direction == "outgoing":
        receivers = edges[0]
        senders = edges[1]
    else:
        raise ValueError("Invalid aggregation direction")

    result = torch.zeros_like(h)

    result = result.index_add(
        0, receivers, h[senders]
    )

    degrees = torch.bincount(
        receivers, minlength=h.shape[0]
    ).to(dtype=h.dtype)

    result = result / degrees.clamp(min=1).unsqueeze(1)

    return result


class SignedSAGELayer(nn.Module):

    def __init__(self, in_dim, out_dim):
        super().__init__()

        # Self + positive incoming/outgoing
        #      + negative incoming/outgoing
        self.linear = nn.Linear(5 * in_dim, out_dim)
        self.norm = nn.LayerNorm(out_dim)

    def forward(self, h, pos_edges, neg_edges):

        pos_in = aggregate_mean(
            h, pos_edges, "incoming"
        )

        pos_out = aggregate_mean(
            h, pos_edges, "outgoing"
        )

        neg_in = aggregate_mean(
            h, neg_edges, "incoming"
        )

        neg_out = aggregate_mean(
            h, neg_edges, "outgoing"
        )

        combined = torch.cat(
            [h, pos_in, pos_out, neg_in, neg_out],
            dim=1
        )

        return self.norm(self.linear(combined))


# ------------------------------------------------------------
# 3. Encoder and trust prediction decoder
# ------------------------------------------------------------

class SignedGraphSAGE(nn.Module):

    def __init__(self, input_dim=4, hidden_dim=32,
                 dropout=0.20):
        super().__init__()

        self.layer1 = SignedSAGELayer(
            input_dim, hidden_dim
        )

        self.layer2 = SignedSAGELayer(
            hidden_dim, hidden_dim
        )

        # Ordered pairs preserve direction.
        # Concatenate source, target, difference, product.
        self.decoder = nn.Sequential(
            nn.Linear(4 * hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

        self.dropout = dropout

    def encode(self, graph):

        h = graph["x"]

        h = self.layer1(
            h,
            graph["pos_edges"],
            graph["neg_edges"]
        )

        h = F.relu(h)
        h = F.dropout(
            h, p=self.dropout, training=self.training
        )

        h = self.layer2(
            h,
            graph["pos_edges"],
            graph["neg_edges"]
        )

        return F.relu(h)

    def forward(self, graph, edge_pairs):

        h = self.encode(graph)

        source_h = h[edge_pairs[:, 0]]
        target_h = h[edge_pairs[:, 1]]

        pair_features = torch.cat(
            [
                source_h,
                target_h,
                torch.abs(source_h - target_h),
                source_h * target_h
            ],
            dim=1
        )

        return self.decoder(pair_features).squeeze(-1)


# ------------------------------------------------------------
# 4. Training data and loss
# ------------------------------------------------------------

gnn = SignedGraphSAGE(
    input_dim=4,
    hidden_dim=CONFIG["hidden_dim"],
    dropout=CONFIG["dropout"]
).to(DEVICE)

train_edges, train_labels = targets["train"]
val_edges, val_labels = targets["validation"]

train_labels = train_labels.float()
val_labels = val_labels.float()

num_negative = train_labels.sum().item()
num_positive = len(train_labels) - num_negative

# Since class 1 means distrust, compensate for rarity.
positive_weight = num_positive / max(num_negative, 1)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        positive_weight,
        device=DEVICE,
        dtype=torch.float32
    )
)

optimizer = torch.optim.Adam(
    gnn.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"]
)

print(f"Trainable parameters: "
      f"{sum(p.numel() for p in gnn.parameters()):,}")
print(f"Distrust class weight: {positive_weight:.4f}")

# ------------------------------------------------------------
# 5. Validation metrics
# ------------------------------------------------------------

def evaluate_gnn(model, graph, edges, labels):

    model.eval()

    with torch.no_grad():
        logits = model(graph, edges)
        scores = torch.sigmoid(logits).cpu().numpy()

    y_true = labels.detach().cpu().numpy().astype(int)

    return {
        "roc_auc": float(
            roc_auc_score(y_true, scores)
        ),
        "negative_pr_auc": float(
            average_precision_score(y_true, scores)
        ),
        "scores": scores,
        "labels": y_true
    }


# ------------------------------------------------------------
# 6. Train with validation early stopping
# ------------------------------------------------------------

best_pr_auc = -1.0
best_epoch = 0
best_state = None
wait = 0
training_log = []

print("\n" + "=" * 70)
print("TA-GSNN | FULL SIGNED GNN TRAINING")
print("=" * 70)

for epoch in range(1, CONFIG["max_epochs"] + 1):

    gnn.train()
    optimizer.zero_grad()

    logits = gnn(graph_train, train_edges)

    loss = criterion(logits, train_labels)

    loss.backward()
    torch.nn.utils.clip_grad_norm_(
        gnn.parameters(), max_norm=5.0
    )

    optimizer.step()

    val_result = evaluate_gnn(
        gnn,
        graph_validation,
        val_edges,
        val_labels
    )

    current_pr = val_result["negative_pr_auc"]

    training_log.append({
        "epoch": epoch,
        "train_loss": float(loss.item()),
        "val_roc_auc": val_result["roc_auc"],
        "val_negative_pr_auc": current_pr
    })

    if current_pr > best_pr_auc + 1e-6:
        best_pr_auc = current_pr
        best_epoch = epoch
        best_state = copy.deepcopy(gnn.state_dict())
        wait = 0
    else:
        wait += 1

    if epoch == 1 or epoch % 10 == 0:
        print(
            f"Epoch {epoch:3d} | "
            f"Loss: {loss.item():.4f} | "
            f"Val ROC-AUC: {val_result['roc_auc']:.4f} | "
            f"Val PR-AUC: {current_pr:.4f}"
        )

    if wait >= CONFIG["patience"]:
        print(f"\nEarly stopping at epoch {epoch}")
        break

assert best_state is not None

gnn.load_state_dict(best_state)
gnn.eval()

# ------------------------------------------------------------
# 7. Select classification threshold on validation only
# ------------------------------------------------------------

best_val = evaluate_gnn(
    gnn,
    graph_validation,
    val_edges,
    val_labels
)

scores = best_val["scores"]
labels = best_val["labels"]

# Threshold based only on validation predictions.
unique_scores = np.unique(scores)

candidate_thresholds = np.unique(
    np.concatenate([
        [0.0, 0.5, 1.0],
        unique_scores,
        (
            unique_scores[:-1] + unique_scores[1:]
        ) / 2
    ])
)

macro_values = np.array([
    f1_score(
        labels,
        (scores >= t).astype(int),
        average="macro",
        zero_division=0
    )
    for t in candidate_thresholds
])

best_indices = np.flatnonzero(
    np.isclose(
        macro_values, macro_values.max(), atol=1e-12
    )
)

chosen_index = best_indices[
    np.argmin(
        np.abs(candidate_thresholds[best_indices] - 0.5)
    )
]

chosen_threshold = float(
    candidate_thresholds[chosen_index]
)

val_predictions = (
    scores >= chosen_threshold
).astype(int)

final_metrics = {
    "best_epoch": best_epoch,
    "validation_roc_auc": best_val["roc_auc"],
    "validation_negative_pr_auc":
        best_val["negative_pr_auc"],
    "validation_macro_f1": float(
        f1_score(
            labels,
            val_predictions,
            average="macro",
            zero_division=0
        )
    ),
    "validation_balanced_accuracy": float(
        balanced_accuracy_score(
            labels, val_predictions
        )
    ),
    "validation_selected_threshold": chosen_threshold
}

print("\n" + "=" * 70)
print("BEST VALIDATION RESULTS")
print("=" * 70)

for key, value in final_metrics.items():
    print(f"{key:38s}: {value}")

# ------------------------------------------------------------
# 8. Save reproducibility artifacts
# ------------------------------------------------------------

checkpoint_path = (
    RESULTS_DIR / "bitcoin_alpha_signed_gnn_seed42.pt"
)

torch.save(
    {
        "model_state_dict": best_state,
        "model_config": CONFIG,
        "input_dim": 4,
        "hidden_dim": CONFIG["hidden_dim"],
        "threshold": chosen_threshold,
        "node_ids": node_ids,
        "best_epoch": best_epoch
    },
    checkpoint_path
)

import pandas as pd

pd.DataFrame(training_log).to_csv(
    RESULTS_DIR / "bitcoin_alpha_gnn_training_log.csv",
    index=False
)

with open(
    RESULTS_DIR / "bitcoin_alpha_gnn_validation.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        {
            "config": CONFIG,
            "metrics": final_metrics,
            "class_weight": positive_weight
        },
        file,
        indent=2
    )

print("\nModel checkpoint and validation results saved.")
print("Test set has NOT been evaluated.")
print("=" * 70)


In [ ]:

# ============================================================
# TA-GSNN | Stage 2.3: Sparsification Sensitivity Test
# ============================================================

import numpy as np
import pandas as pd
import torch
import json

from pathlib import Path
from sklearn.metrics import roc_auc_score, average_precision_score

RESULTS_DIR = Path("/kaggle/working/ta_gsnn_results")
DATA_DIR = Path("/kaggle/working/ta_gsnn_data")

# Ensure the previously selected best model is used.
gnn.eval()

val_edges, val_labels = targets["validation"]
y_val = val_labels.detach().cpu().numpy().astype(int)

# Build pruned graph from raw historical interactions.
validation_history = pd.concat(
    [parts["context"], parts["train"]],
    ignore_index=True
)

def make_pruned_graph(
    history,
    pruning_ratio,
    method,
    seed=42
):
    n = len(history)
    retain = int(round(n * (1 - pruning_ratio)))

    if method == "random":
        rng = np.random.default_rng(seed)
        selected = rng.choice(
            n, size=retain, replace=False
        )

    elif method == "magnitude":
        # Ranking by magnitude of OBSERVED trust rating.
        # Retains both strong positive and strong negative edges.
        scores = np.abs(
            history["rating"].to_numpy()
        )

        # Stable tie-breaking by chronological order.
        selected = np.argsort(
            -scores, kind="stable"
        )[:retain]

    else:
        raise ValueError("Unknown pruning method")

    subset = history.iloc[
        np.sort(selected)
    ].copy()

    return build_graph_tensors(subset)


def evaluate_pruned_graph(graph):
    gnn.eval()

    with torch.no_grad():
        logits = gnn(graph, val_edges)
        scores = torch.sigmoid(logits).cpu().numpy()

    return {
        "roc_auc": float(
            roc_auc_score(y_val, scores)
        ),
        "negative_pr_auc": float(
            average_precision_score(y_val, scores)
        )
    }


experiments = []

# Full graph reference
full_result = evaluate_pruned_graph(graph_validation)

experiments.append({
    "method": "full_graph",
    "pruning_ratio": 0.0,
    "retained_edges": len(validation_history),
    **full_result
})

for pruning_ratio in [0.20, 0.50, 0.80]:

    for method in ["random", "magnitude"]:

        pruned_graph = make_pruned_graph(
            validation_history,
            pruning_ratio,
            method,
            seed=42
        )

        result = evaluate_pruned_graph(
            pruned_graph
        )

        experiments.append({
            "method": method,
            "pruning_ratio": pruning_ratio,
            "retained_edges": int(
                len(validation_history) *
                (1 - pruning_ratio)
            ),
            **result
        })

results_df = pd.DataFrame(experiments)

print("=" * 80)
print("TA-GSNN | SPARSIFICATION SENSITIVITY")
print("=" * 80)
print(results_df.round(4).to_string(index=False))

results_df.to_csv(
    RESULTS_DIR / "bitcoin_alpha_sparsification_sensitivity.csv",
    index=False
)

with open(
    RESULTS_DIR / "bitcoin_alpha_sparsification_sensitivity.json",
    "w"
) as f:
    json.dump(
        experiments, f, indent=2
    )

print("\nSensitivity results saved.")
print("=" * 80)


In [ ]:

# ============================================================
# TA-GSNN | Stage 2.4: Sparsification Robustness Diagnostic
# ============================================================

import numpy as np
import pandas as pd
import networkx as nx
import torch
from pathlib import Path
from sklearn.metrics import roc_auc_score, average_precision_score

RESULTS_DIR = Path("/kaggle/working/ta_gsnn_results")

history_df = pd.concat(
    [parts["context"], parts["train"]],
    ignore_index=True
)

val_edges, val_labels = targets["validation"]
y_true = val_labels.detach().cpu().numpy().astype(int)

RANDOM_SEEDS = list(range(42, 52))
PRUNING_RATIOS = [0.20, 0.50, 0.80]

# Keep the original historical node set fixed for
# meaningful structural comparisons.
historical_nodes = set(history_df["source"]) | set(
    history_df["target"]
)

def graph_structure_metrics(subset):
    graph = nx.Graph()
    graph.add_nodes_from(historical_nodes)
    graph.add_edges_from(
        zip(subset["source"], subset["target"])
    )

    components = list(nx.connected_components(graph))
    largest = max(map(len, components)) if components else 0
    isolated = sum(
        degree == 0 for _, degree in graph.degree()
    )

    return {
        "largest_component_pct": (
            100 * largest / len(historical_nodes)
        ),
        "isolated_nodes_pct": (
            100 * isolated / len(historical_nodes)
        )
    }

def evaluate_subset(subset):
    graph = build_graph_tensors(subset)

    gnn.eval()
    with torch.no_grad():
        logits = gnn(graph, val_edges)
        scores = torch.sigmoid(logits).cpu().numpy()

    metrics = {
        "roc_auc": roc_auc_score(y_true, scores),
        "negative_pr_auc": average_precision_score(
            y_true, scores
        )
    }

    metrics.update(graph_structure_metrics(subset))
    return metrics

records = []

def add_result(method, ratio, seed, subset):
    metrics = evaluate_subset(subset)

    records.append({
        "method": method,
        "pruning_ratio": ratio,
        "seed": seed,
        "retained_edges": len(subset),
        "retained_negative_pct": (
            100 * (subset["rating"] < 0).mean()
        ),
        **metrics
    })

# Full graph
add_result("full", 0.0, -1, history_df)

for ratio in PRUNING_RATIOS:
    keep = int(round(len(history_df) * (1 - ratio)))

    # Deterministic trust-magnitude baseline
    ratings = np.abs(history_df["rating"].to_numpy())
    magnitude_idx = np.argsort(
        -ratings, kind="stable"
    )[:keep]

    add_result(
        "magnitude",
        ratio,
        -1,
        history_df.iloc[np.sort(magnitude_idx)]
    )

    # Repeated random pruning
    for seed in RANDOM_SEEDS:
        rng = np.random.default_rng(seed)
        random_idx = rng.choice(
            len(history_df),
            size=keep,
            replace=False
        )

        add_result(
            "random",
            ratio,
            seed,
            history_df.iloc[np.sort(random_idx)]
        )

results = pd.DataFrame(records)

summary = (
    results.groupby(
        ["method", "pruning_ratio"], as_index=False
    )
    .agg(
        pr_auc_mean=("negative_pr_auc", "mean"),
        pr_auc_std=("negative_pr_auc", "std"),
        roc_auc_mean=("roc_auc", "mean"),
        roc_auc_std=("roc_auc", "std"),
        retained_negative_pct=(
            "retained_negative_pct", "mean"
        ),
        largest_component_pct=(
            "largest_component_pct", "mean"
        ),
        isolated_nodes_pct=(
            "isolated_nodes_pct", "mean"
        )
    )
    .sort_values(["pruning_ratio", "method"])
)

# Standard deviation is undefined for one deterministic run.
summary = summary.fillna({
    "pr_auc_std": 0.0,
    "roc_auc_std": 0.0
})

print("=" * 105)
print("TA-GSNN | SPARSIFICATION ROBUSTNESS")
print("=" * 105)
print(summary.round(4).to_string(index=False))

results.to_csv(
    RESULTS_DIR / "bitcoin_alpha_pruning_robustness_raw.csv",
    index=False
)

summary.to_csv(
    RESULTS_DIR / "bitcoin_alpha_pruning_robustness_summary.csv",
    index=False
)

print("\nSaved raw results and summary.")
print("NOTE: This is a fixed-model validation diagnostic.")
print("No test data were evaluated.")


In [ ]:

# ============================================================
# TA-GSNN | Stage 3.1: Learned Trust-Aware Edge Gate
# Two-stage proof of concept | 50% edge removal
# ============================================================

import copy
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from pathlib import Path
from sklearn.metrics import roc_auc_score, average_precision_score

SEED = 42
KEEP_RATIO = 0.50
torch.manual_seed(SEED)
np.random.seed(SEED)

RESULTS_DIR = Path("/kaggle/working/ta_gsnn_results")

# Freeze the previously trained GNN.
gnn.eval()
for parameter in gnn.parameters():
    parameter.requires_grad_(False)

train_history = parts["context"].copy()
val_history = pd.concat(
    [parts["context"], parts["train"]],
    ignore_index=True
)

def prepare_gate_data(history):
    src = torch.tensor(
        history["source"].map(node_to_idx).to_numpy(),
        dtype=torch.long, device=DEVICE
    )
    dst = torch.tensor(
        history["target"].map(node_to_idx).to_numpy(),
        dtype=torch.long, device=DEVICE
    )
    rating = torch.tensor(
        history["rating"].to_numpy(),
        dtype=torch.float32, device=DEVICE
    )

    positive = rating > 0
    negative = rating < 0

    counts = torch.zeros(
        (NUM_NODES, 4), device=DEVICE
    )
    ones = torch.ones(len(src), device=DEVICE)

    counts[:, 0].index_add_(
        0, src[positive], ones[positive]
    )
    counts[:, 1].index_add_(
        0, src[negative], ones[negative]
    )
    counts[:, 2].index_add_(
        0, dst[positive], ones[positive]
    )
    counts[:, 3].index_add_(
        0, dst[negative], ones[negative]
    )

    node_features = torch.log1p(counts)
    node_features = node_features / (
        node_features.amax(dim=0).clamp(min=1)
    )

    # Only historically observed edge information is used.
    edge_features = torch.cat([
        node_features[src],
        node_features[dst],
        torch.sign(rating).unsqueeze(1),
        (rating.abs() / 10).unsqueeze(1)
    ], dim=1)

    return {
        "src": src,
        "dst": dst,
        "positive": positive,
        "negative": negative,
        "edge_features": edge_features,
        "num_edges": len(src)
    }

train_data = prepare_gate_data(train_history)
val_data = prepare_gate_data(val_history)

class EdgeGate(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(10, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, edge_features):
        return torch.sigmoid(
            self.network(edge_features).squeeze(-1)
        )

gate = EdgeGate().to(DEVICE)

def weighted_aggregate(h, src, dst, weights, incoming):
    receivers = dst if incoming else src
    senders = src if incoming else dst

    numerator = torch.zeros_like(h).index_add(
        0,
        receivers,
        h[senders] * weights.unsqueeze(1)
    )

    denominator = torch.zeros(
        h.shape[0], device=h.device, dtype=h.dtype
    ).index_add(0, receivers, weights)

    return numerator / denominator.clamp(min=1e-6).unsqueeze(1)

def weighted_layer(layer, h, data, weights):
    src, dst = data["src"], data["dst"]
    pos, neg = data["positive"], data["negative"]

    pieces = [h]

    for mask in (pos, neg):
        s, d, w = src[mask], dst[mask], weights[mask]
        pieces.append(weighted_aggregate(h, s, d, w, True))
        pieces.append(weighted_aggregate(h, s, d, w, False))

    return layer.norm(layer.linear(torch.cat(pieces, dim=1)))

def gated_gnn_logits(data, weights, target_edges):
    src, dst = data["src"], data["dst"]
    pos, neg = data["positive"], data["negative"]

    # Reconstruct node features using soft retained edges.
    x = torch.zeros(
        (NUM_NODES, 4), device=DEVICE
    )

    x[:, 0].index_add_(0, src[pos], weights[pos])
    x[:, 1].index_add_(0, src[neg], weights[neg])
    x[:, 2].index_add_(0, dst[pos], weights[pos])
    x[:, 3].index_add_(0, dst[neg], weights[neg])

    h = torch.log1p(x)
    h = F.relu(
        weighted_layer(gnn.layer1, h, data, weights)
    )
    h = F.relu(
        weighted_layer(gnn.layer2, h, data, weights)
    )

    u = h[target_edges[:, 0]]
    v = h[target_edges[:, 1]]

    pair = torch.cat([
        u, v, torch.abs(u - v), u * v
    ], dim=1)

    return gnn.decoder(pair).squeeze(-1)

def degree_preservation_penalty(data, weights):
    degree = torch.zeros(
        NUM_NODES, device=DEVICE
    )
    degree = degree.index_add(
        0, data["src"], weights
    )
    degree = degree.index_add(
        0, data["dst"], weights
    )

    original = torch.zeros(
        NUM_NODES, device=DEVICE
    )
    ones = torch.ones_like(weights)
    original = original.index_add(
        0, data["src"], ones
    )
    original = original.index_add(
        0, data["dst"], ones
    )

    active = original > 0
    return F.relu(0.5 - degree[active]).mean()

optimizer_gate = torch.optim.Adam(
    gate.parameters(), lr=0.002
)

train_edges, train_labels = targets["train"]
val_edges, val_labels = targets["validation"]
y_val = val_labels.cpu().numpy().astype(int)

# Reuse the original class-weighted prediction loss.
gate_criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        positive_weight, device=DEVICE
    )
)

def validate_gate():
    gate.eval()
    with torch.no_grad():
        scores = gate(val_data["edge_features"])
        retain = round(
            KEEP_RATIO * val_data["num_edges"]
        )
        selected = torch.topk(
            scores, retain
        ).indices.cpu().numpy()

        pruned = val_history.iloc[
            np.sort(selected)
        ].copy()

        graph = build_graph_tensors(pruned)
        probabilities = torch.sigmoid(
            gnn(graph, val_edges)
        ).cpu().numpy()

    return (
        float(average_precision_score(
            y_val, probabilities
        )),
        float(roc_auc_score(
            y_val, probabilities
        )),
        pruned
    )

best_score = -1
best_state = None
best_epoch = None
wait = 0
logs = []

print("=" * 76)
print("TA-GSNN | LEARNED EDGE GATE — 50% RETENTION")
print("=" * 76)

for epoch in range(1, 61):
    gate.train()
    optimizer_gate.zero_grad()

    weights = gate(train_data["edge_features"])

    logits = gated_gnn_logits(
        train_data, weights, train_edges
    )

    pred_loss = gate_criterion(
        logits, train_labels
    )

    budget_loss = (
        weights.mean() - KEEP_RATIO
    ).square()

    degree_loss = degree_preservation_penalty(
        train_data, weights
    )

    loss = (
        pred_loss
        + 10.0 * budget_loss
        + 0.10 * degree_loss
    )

    loss.backward()
    optimizer_gate.step()

    val_pr, val_roc, _ = validate_gate()

    logs.append({
        "epoch": epoch,
        "loss": float(loss.item()),
        "prediction_loss": float(pred_loss.item()),
        "gate_mean": float(weights.mean().item()),
        "validation_pr_auc": val_pr,
        "validation_roc_auc": val_roc
    })

    if val_pr > best_score + 1e-6:
        best_score = val_pr
        best_epoch = epoch
        best_state = copy.deepcopy(gate.state_dict())
        wait = 0
    else:
        wait += 1

    if epoch == 1 or epoch % 10 == 0:
        print(
            f"Epoch {epoch:3d} | "
            f"Loss={loss.item():.4f} | "
            f"GateMean={weights.mean().item():.3f} | "
            f"Val PR-AUC={val_pr:.4f} | "
            f"Val ROC-AUC={val_roc:.4f}"
        )

    if wait >= 15:
        print(f"Early stopping at epoch {epoch}")
        break

gate.load_state_dict(best_state)
best_pr, best_roc, best_pruned = validate_gate()

# Structural diagnostics on the final selected subgraph.
structure = graph_structure_metrics(best_pruned)

print("\n" + "=" * 76)
print("BEST LEARNED GATE — VALIDATION")
print("=" * 76)
print(f"Best epoch                 : {best_epoch}")
print(f"Retained edges             : {len(best_pruned)}")
print(f"Actual edge retention      : {len(best_pruned)/len(val_history):.4f}")
print(f"Negative PR-AUC            : {best_pr:.4f}")
print(f"ROC-AUC                    : {best_roc:.4f}")
print(f"Largest component (%)      : {structure['largest_component_pct']:.2f}")
print(f"Isolated nodes (%)         : {structure['isolated_nodes_pct']:.2f}")

torch.save(
    {
        "gate_state_dict": best_state,
        "seed": SEED,
        "keep_ratio": KEEP_RATIO,
        "best_epoch": best_epoch,
        "training_type": "frozen_GNN_two_stage_gate"
    },
    RESULTS_DIR / "bitcoin_alpha_edge_gate_50pct.pt"
)

pd.DataFrame(logs).to_csv(
    RESULTS_DIR / "bitcoin_alpha_edge_gate_training.csv",
    index=False
)

print("\nGate checkpoint and training log saved.")
print("Test set NOT evaluated.")


In [ ]:

# ============================================================
# TA-GSNN | Stage 3.2: Coverage-Constrained Sparsification
# Diagnostic comparison at 50% edge retention
# ============================================================

import numpy as np
import pandas as pd
import torch
from pathlib import Path
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)

RESULTS_DIR = Path("/kaggle/working/ta_gsnn_results")

KEEP_RATIO = 0.50
N_EDGES = len(val_history)
KEEP_COUNT = round(N_EDGES * KEEP_RATIO)

# The historical graph contains all eligible observed edges.
src = val_history["source"].to_numpy()
dst = val_history["target"].to_numpy()
ratings = val_history["rating"].to_numpy()

# Frozen learned gate scores
gate.eval()

with torch.no_grad():
    learned_scores = gate(
        val_data["edge_features"]
    ).cpu().numpy()

magnitude_scores = np.abs(ratings).astype(float)

rng = np.random.default_rng(42)
random_scores = rng.random(N_EDGES)


def select_edges(scores, keep_count, preserve_coverage):
    """
    Select exactly keep_count edges.

    If preserve_coverage=True:
      1. Choose each active node's highest-scored incident edge.
      2. Fill the remaining budget with highest-scored edges.

    This ensures every node with at least one historical
    incident edge retains at least one incident edge,
    provided the budget can accommodate the initial cover.
    """

    # Stable ranking; tie-breaking follows input order.
    ranking = np.argsort(-scores, kind="stable")

    if not preserve_coverage:
        return np.sort(ranking[:keep_count])

    # Map original node IDs to compact indices.
    nodes = np.unique(np.concatenate([src, dst]))
    node_index = {
        int(node): i for i, node in enumerate(nodes)
    }

    best_incident = np.full(
        len(nodes), -1, dtype=np.int64
    )

    # Highest-ranked incident edge for every node.
    for edge_idx in ranking:
        u = node_index[int(src[edge_idx])]
        v = node_index[int(dst[edge_idx])]

        if best_incident[u] == -1:
            best_incident[u] = edge_idx

        if best_incident[v] == -1:
            best_incident[v] = edge_idx

    required = np.unique(best_incident)

    assert np.all(required >= 0)

    if len(required) > keep_count:
        raise ValueError(
            "Coverage requirement exceeds edge budget."
        )

    selected_mask = np.zeros(
        N_EDGES, dtype=bool
    )

    selected_mask[required] = True

    remaining = keep_count - len(required)

    additional = ranking[
        ~selected_mask[ranking]
    ][:remaining]

    selected_mask[additional] = True

    selected = np.flatnonzero(selected_mask)

    assert len(selected) == keep_count

    return selected


def evaluate_selection(scores, method, preserve_coverage):

    selected = select_edges(
        scores,
        KEEP_COUNT,
        preserve_coverage
    )

    subset = val_history.iloc[selected].copy()

    graph = build_graph_tensors(subset)

    gnn.eval()

    with torch.no_grad():
        logits = gnn(graph, val_edges)

        probabilities = torch.sigmoid(
            logits
        ).cpu().numpy()

    graph_metrics = graph_structure_metrics(subset)

    return {
        "method": method,
        "coverage_constraint": preserve_coverage,
        "retained_edges": len(subset),
        "negative_pr_auc": float(
            average_precision_score(
                y_val, probabilities
            )
        ),
        "roc_auc": float(
            roc_auc_score(
                y_val, probabilities
            )
        ),
        "largest_component_pct":
            graph_metrics["largest_component_pct"],
        "isolated_nodes_pct":
            graph_metrics["isolated_nodes_pct"],
        "negative_edges_pct": float(
            100 * (subset["rating"] < 0).mean()
        )
    }


records = []

methods = {
    "random": random_scores,
    "magnitude": magnitude_scores,
    "learned_gate": learned_scores
}

for method, scores in methods.items():

    for constrained in [False, True]:

        result = evaluate_selection(
            scores,
            method,
            constrained
        )

        records.append(result)


results_df = pd.DataFrame(records)

print("=" * 105)
print("TA-GSNN | COVERAGE-CONSTRAINED SPARSIFICATION")
print("=" * 105)

print(
    results_df.round(4).to_string(index=False)
)

results_df.to_csv(
    RESULTS_DIR /
    "bitcoin_alpha_coverage_constrained_comparison.csv",
    index=False
)

print("\nResults saved successfully.")
print("Validation only; test set not evaluated.")


In [ ]:

# ============================================================
# TA-GSNN | Stage 3.3
# Sign-Balanced Coverage-Constrained Edge Selection
# Validation-only diagnostic, no retraining
# ============================================================

import numpy as np
import pandas as pd
import torch

from pathlib import Path
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)

RESULTS_DIR = Path("/kaggle/working/ta_gsnn_results")

history = val_history.reset_index(drop=True)

src = history["source"].to_numpy()
dst = history["target"].to_numpy()
ratings = history["rating"].to_numpy()

N = len(history)
KEEP = round(0.50 * N)

negative_mask = ratings < 0

# Use the already-trained gate.
gate.eval()
gnn.eval()

with torch.no_grad():
    gate_scores = gate(
        val_data["edge_features"]
    ).detach().cpu().numpy()

magnitude_scores = np.abs(ratings).astype(float)

def balanced_coverage_selection(
    scores,
    target_negative_fraction
):
    """
    Select exactly KEEP historical edges.

    1. Cover every historically active node.
    2. Retain at least a target number of negative edges.
    3. Fill the remaining budget with positive edges first,
       then remaining edges if necessary.

    This is a diagnostic heuristic, not a learned optimizer.
    """

    ranking = np.argsort(-scores, kind="stable")

    nodes = np.unique(np.concatenate([src, dst]))
    node_map = {
        int(node): idx for idx, node in enumerate(nodes)
    }

    best_incident = np.full(
        len(nodes), -1, dtype=np.int64
    )

    for idx in ranking:
        u = node_map[int(src[idx])]
        v = node_map[int(dst[idx])]

        if best_incident[u] == -1:
            best_incident[u] = idx
        if best_incident[v] == -1:
            best_incident[v] = idx

    required = np.unique(best_incident)

    if len(required) > KEEP:
        raise ValueError("Coverage exceeds edge budget.")

    selected = np.zeros(N, dtype=bool)
    selected[required] = True

    target_negatives = min(
        round(KEEP * target_negative_fraction),
        int(negative_mask.sum())
    )

    current_negatives = int(
        (selected & negative_mask).sum()
    )

    # Ensure minimum negative-edge retention.
    need_negatives = max(
        0, target_negatives - current_negatives
    )

    negative_candidates = ranking[
        negative_mask[ranking] & ~selected[ranking]
    ]

    selected[
        negative_candidates[:need_negatives]
    ] = True

    remaining = KEEP - int(selected.sum())

    # Prefer positive edges for the remaining slots.
    positive_candidates = ranking[
        (~negative_mask[ranking]) & ~selected[ranking]
    ]

    chosen_positive = positive_candidates[:remaining]
    selected[chosen_positive] = True

    remaining = KEEP - int(selected.sum())

    if remaining > 0:
        other_candidates = ranking[~selected[ranking]]
        selected[other_candidates[:remaining]] = True

    indices = np.flatnonzero(selected)

    assert len(indices) == KEEP

    return indices


records = []

for method, scores in {
    "learned_gate": gate_scores,
    "magnitude": magnitude_scores
}.items():

    for negative_fraction in [0.03, 0.05, 0.08]:

        indices = balanced_coverage_selection(
            scores,
            negative_fraction
        )

        subset = history.iloc[indices].copy()
        graph = build_graph_tensors(subset)

        with torch.no_grad():
            logits = gnn(graph, val_edges)
            probabilities = torch.sigmoid(
                logits
            ).cpu().numpy()

        structure = graph_structure_metrics(subset)

        records.append({
            "method": method,
            "minimum_negative_fraction":
                negative_fraction,
            "actual_negative_pct":
                100 * (subset["rating"] < 0).mean(),
            "retained_edges": len(subset),
            "negative_pr_auc": float(
                average_precision_score(
                    y_val, probabilities
                )
            ),
            "roc_auc": float(
                roc_auc_score(
                    y_val, probabilities
                )
            ),
            "largest_component_pct":
                structure["largest_component_pct"],
            "isolated_nodes_pct":
                structure["isolated_nodes_pct"]
        })

result_df = pd.DataFrame(records)

print("=" * 105)
print("TA-GSNN | SIGN-BALANCED SPARSIFICATION")
print("=" * 105)

print(
    result_df.round(4).to_string(index=False)
)

result_df.to_csv(
    RESULTS_DIR /
    "bitcoin_alpha_sign_balance_diagnostic.csv",
    index=False
)

print("\nDiagnostic saved successfully.")
print("Validation only. Test set NOT evaluated.")


In [ ]:

# ============================================================
# TA-GSNN | Stage 3.4
# Joint Training with Coverage-Constrained Hard Edge Selection
# Validation-only experiment | 50% edge retention
# ============================================================

import copy
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from sklearn.metrics import roc_auc_score, average_precision_score

JOINT_SEED = 42
KEEP_RATIO = 0.50
MAX_EPOCHS = 80
PATIENCE = 15

random.seed(JOINT_SEED)
np.random.seed(JOINT_SEED)
torch.manual_seed(JOINT_SEED)
torch.cuda.manual_seed_all(JOINT_SEED)

RESULTS_DIR = Path("/kaggle/working/ta_gsnn_results")

# ------------------------------------------------------------
# 1. Graph-independent coverage-constrained selector
# ------------------------------------------------------------

def coverage_topk_mask(scores, data, keep_ratio):
    """
    Returns a binary edge mask at the specified budget.
    Covers every historically active node with >= 1 edge.
    """

    n = data["num_edges"]
    budget = round(n * keep_ratio)

    order = torch.argsort(
        scores.detach(), descending=True, stable=True
    ).cpu().numpy()

    sources = data["src"].cpu().numpy()
    destinations = data["dst"].cpu().numpy()

    required = {}
    for idx in order:
        u = int(sources[idx])
        v = int(destinations[idx])
        if u not in required:
            required[u] = int(idx)
        if v not in required:
            required[v] = int(idx)

    mandatory = set(required.values())

    if len(mandatory) > budget:
        raise ValueError(
            "Coverage is infeasible at this edge budget."
        )

    selected = set(mandatory)

    for idx in order:
        if len(selected) >= budget:
            break
        selected.add(int(idx))

    mask = torch.zeros(
        n, device=DEVICE, dtype=torch.float32
    )
    mask[list(selected)] = 1.0

    assert int(mask.sum().item()) == budget

    return mask


# ------------------------------------------------------------
# 2. Independent jointly trained models
# ------------------------------------------------------------

joint_gnn = SignedGraphSAGE(
    input_dim=4, hidden_dim=32, dropout=0.20
).to(DEVICE)

joint_gate = EdgeGate().to(DEVICE)

# Reuse the historical data and target tensors created earlier.
joint_train_data = prepare_gate_data(parts["context"])

joint_val_history = pd.concat(
    [parts["context"], parts["train"]],
    ignore_index=True
)

joint_val_data = prepare_gate_data(joint_val_history)

joint_train_edges, joint_train_labels = targets["train"]
joint_val_edges, joint_val_labels = targets["validation"]

# ------------------------------------------------------------
# 3. Differentiable forward pass
# ------------------------------------------------------------

def joint_forward(model, data, weights, edge_pairs):

    src, dst = data["src"], data["dst"]
    pos, neg = data["positive"], data["negative"]

    x = torch.zeros(
        (NUM_NODES, 4), device=DEVICE
    )

    x[:, 0].index_add_(0, src[pos], weights[pos])
    x[:, 1].index_add_(0, src[neg], weights[neg])
    x[:, 2].index_add_(0, dst[pos], weights[pos])
    x[:, 3].index_add_(0, dst[neg], weights[neg])

    h = torch.log1p(x)

    h = torch.relu(
        weighted_layer(model.layer1, h, data, weights)
    )
    h = torch.nn.functional.dropout(
        h, p=model.dropout, training=model.training
    )

    h = torch.relu(
        weighted_layer(model.layer2, h, data, weights)
    )

    u = h[edge_pairs[:, 0]]
    v = h[edge_pairs[:, 1]]

    pair = torch.cat(
        [u, v, torch.abs(u-v), u*v], dim=1
    )

    return model.decoder(pair).squeeze(-1)


# ------------------------------------------------------------
# 4. Optimization
# ------------------------------------------------------------

joint_optimizer = torch.optim.Adam(
    list(joint_gnn.parameters()) +
    list(joint_gate.parameters()),
    lr=0.001,
    weight_decay=1e-4
)

joint_criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        positive_weight,
        device=DEVICE,
        dtype=torch.float32
    )
)

def joint_validation():

    joint_gnn.eval()
    joint_gate.eval()

    with torch.no_grad():
        probabilities = joint_gate(
            joint_val_data["edge_features"]
        )

        hard_mask = coverage_topk_mask(
            probabilities, joint_val_data, KEEP_RATIO
        )

        logits = joint_forward(
            joint_gnn,
            joint_val_data,
            hard_mask,
            joint_val_edges
        )

        predictions = torch.sigmoid(
            logits
        ).cpu().numpy()

    labels = joint_val_labels.cpu().numpy().astype(int)

    return {
        "pr_auc": float(average_precision_score(
            labels, predictions
        )),
        "roc_auc": float(roc_auc_score(
            labels, predictions
        )),
        "mask": hard_mask
    }


best_joint_pr = -1
best_joint_epoch = None
best_joint_gnn_state = None
best_joint_gate_state = None
waiting = 0
joint_logs = []

print("=" * 76)
print("TA-GSNN | JOINT TRAINING — 50% EDGE RETENTION")
print("=" * 76)

for epoch in range(1, MAX_EPOCHS + 1):

    joint_gnn.train()
    joint_gate.train()
    joint_optimizer.zero_grad()

    soft_scores = joint_gate(
        joint_train_data["edge_features"]
    )

    hard_mask = coverage_topk_mask(
        soft_scores, joint_train_data, KEEP_RATIO
    )

    # Hard mask in forward pass; soft gradient approximation.
    ste_weights = (
        hard_mask + soft_scores - soft_scores.detach()
    )

    logits = joint_forward(
        joint_gnn,
        joint_train_data,
        ste_weights,
        joint_train_edges
    )

    loss = joint_criterion(
        logits, joint_train_labels
    )

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        list(joint_gnn.parameters()) +
        list(joint_gate.parameters()),
        max_norm=5.0
    )

    joint_optimizer.step()

    result = joint_validation()

    joint_logs.append({
        "epoch": epoch,
        "train_loss": float(loss.item()),
        "val_pr_auc": result["pr_auc"],
        "val_roc_auc": result["roc_auc"]
    })

    if result["pr_auc"] > best_joint_pr + 1e-6:
        best_joint_pr = result["pr_auc"]
        best_joint_epoch = epoch
        best_joint_gnn_state = copy.deepcopy(
            joint_gnn.state_dict()
        )
        best_joint_gate_state = copy.deepcopy(
            joint_gate.state_dict()
        )
        waiting = 0
    else:
        waiting += 1

    if epoch == 1 or epoch % 10 == 0:
        print(
            f"Epoch {epoch:3d} | "
            f"Loss={loss.item():.4f} | "
            f"Val PR-AUC={result['pr_auc']:.4f} | "
            f"Val ROC-AUC={result['roc_auc']:.4f}"
        )

    if waiting >= PATIENCE:
        print(f"Early stopping at epoch {epoch}")
        break


# ------------------------------------------------------------
# 5. Restore best model and assess graph structure
# ------------------------------------------------------------

joint_gnn.load_state_dict(best_joint_gnn_state)
joint_gate.load_state_dict(best_joint_gate_state)

best_joint_validation = joint_validation()

selected_indices = torch.nonzero(
    best_joint_validation["mask"]
).flatten().cpu().numpy()

selected_history = joint_val_history.iloc[
    selected_indices
].copy()

structure = graph_structure_metrics(
    selected_history
)

print("\n" + "=" * 76)
print("BEST JOINT MODEL — VALIDATION")
print("=" * 76)
print(f"Best epoch              : {best_joint_epoch}")
print(f"Negative PR-AUC         : {best_joint_validation['pr_auc']:.4f}")
print(f"ROC-AUC                 : {best_joint_validation['roc_auc']:.4f}")
print(f"Retained edges          : {len(selected_history)}")
print(f"Largest component (%)   : {structure['largest_component_pct']:.2f}")
print(f"Isolated nodes (%)      : {structure['isolated_nodes_pct']:.2f}")

torch.save({
    "gnn_state": best_joint_gnn_state,
    "gate_state": best_joint_gate_state,
    "best_epoch": best_joint_epoch,
    "keep_ratio": KEEP_RATIO,
    "seed": JOINT_SEED,
    "training": "joint_straight_through_coverage"
}, RESULTS_DIR / "bitcoin_alpha_joint_gate_seed42.pt")

pd.DataFrame(joint_logs).to_csv(
    RESULTS_DIR / "bitcoin_alpha_joint_training_log.csv",
    index=False
)

print("\nCheckpoint and training log saved.")
print("Test set NOT evaluated.")


In [ ]:

# ============================================================
# TA-GSNN | Stage 3.5
# Fair Retrained Sparsification Baselines
#
# Models:
#   1. Full Signed GNN
#   2. Random + Node Coverage
#   3. Magnitude + Node Coverage
#
# Same architecture, seed, optimizer and training protocol.
# Validation only. Test data are NOT evaluated.
# ============================================================

import copy
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

RESULTS_DIR = Path("/kaggle/working/ta_gsnn_results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FAIR_SEED = 42
KEEP_RATIO = 0.50
MAX_EPOCHS = 120
PATIENCE = 20

# ------------------------------------------------------------
# 1. Set deterministic seeds
# ------------------------------------------------------------

def set_fair_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ------------------------------------------------------------
# 2. Coverage-constrained edge selection
# ------------------------------------------------------------

def select_coverage_subset(history, method, keep_ratio, seed):

    history = history.reset_index(drop=True)
    n = len(history)

    if method == "full":
        return history.copy()

    keep_count = round(n * keep_ratio)

    sources = history["source"].to_numpy()
    destinations = history["target"].to_numpy()
    ratings = history["rating"].to_numpy()

    if method == "random":
        rng = np.random.default_rng(seed)
        scores = rng.random(n)

    elif method == "magnitude":
        scores = np.abs(ratings).astype(float)

    else:
        raise ValueError(f"Unknown method: {method}")

    ranking = np.argsort(-scores, kind="stable")

    # Highest-scored incident edge for each active node.
    required = {}

    for edge_idx in ranking:
        u = int(sources[edge_idx])
        v = int(destinations[edge_idx])

        if u not in required:
            required[u] = int(edge_idx)

        if v not in required:
            required[v] = int(edge_idx)

    mandatory = set(required.values())

    if len(mandatory) > keep_count:
        raise ValueError(
            f"Coverage infeasible for {method}: "
            f"{len(mandatory)} mandatory edges, "
            f"budget {keep_count}"
        )

    selected = set(mandatory)

    for edge_idx in ranking:
        if len(selected) >= keep_count:
            break

        selected.add(int(edge_idx))

    indices = np.array(sorted(selected), dtype=int)

    assert len(indices) == keep_count

    subset = history.iloc[indices].copy()

    # Verify every originally active node stays covered.
    original_nodes = (
        set(history["source"]) |
        set(history["target"])
    )

    retained_nodes = (
        set(subset["source"]) |
        set(subset["target"])
    )

    assert original_nodes == retained_nodes

    return subset


# ------------------------------------------------------------
# 3. Prepare identical temporal evaluation protocol
# ------------------------------------------------------------

history_train_fair = parts["context"].copy()

history_val_fair = pd.concat(
    [parts["context"], parts["train"]],
    ignore_index=True
)

train_target_edges, train_target_labels = targets["train"]
val_target_edges, val_target_labels = targets["validation"]

train_target_labels = train_target_labels.float()
val_target_labels = val_target_labels.float()

y_validation = (
    val_target_labels.detach()
    .cpu().numpy().astype(int)
)

# Class weight computed only from training targets.
n_distrust = float(train_target_labels.sum().item())

n_trust = (
    len(train_target_labels) - n_distrust
)

fair_class_weight = n_trust / max(n_distrust, 1.0)


# ------------------------------------------------------------
# 4. Model training function
# ------------------------------------------------------------

def train_fair_baseline(method, seed=42):

    set_fair_seed(seed)

    selected_train = select_coverage_subset(
        history_train_fair,
        method,
        KEEP_RATIO,
        seed
    )

    selected_val = select_coverage_subset(
        history_val_fair,
        method,
        KEEP_RATIO,
        seed
    )

    train_graph = build_graph_tensors(selected_train)
    val_graph = build_graph_tensors(selected_val)

    # Identical model architecture for all methods.
    model = SignedGraphSAGE(
        input_dim=4,
        hidden_dim=32,
        dropout=0.20
    ).to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.001,
        weight_decay=1e-4
    )

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(
            fair_class_weight,
            dtype=torch.float32,
            device=DEVICE
        )
    )

    best_pr_auc = -1.0
    best_epoch = 0
    best_state = None
    waiting = 0
    training_log = []

    for epoch in range(1, MAX_EPOCHS + 1):

        model.train()
        optimizer.zero_grad()

        logits = model(
            train_graph,
            train_target_edges
        )

        loss = criterion(
            logits,
            train_target_labels
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0
        )

        optimizer.step()

        model.eval()

        with torch.no_grad():

            val_logits = model(
                val_graph,
                val_target_edges
            )

            val_scores = torch.sigmoid(
                val_logits
            ).cpu().numpy()

        pr_auc = float(
            average_precision_score(
                y_validation,
                val_scores
            )
        )

        roc_auc = float(
            roc_auc_score(
                y_validation,
                val_scores
            )
        )

        training_log.append({
            "method": method,
            "seed": seed,
            "epoch": epoch,
            "train_loss": float(loss.item()),
            "validation_pr_auc": pr_auc,
            "validation_roc_auc": roc_auc
        })

        if pr_auc > best_pr_auc + 1e-6:
            best_pr_auc = pr_auc
            best_epoch = epoch
            best_state = copy.deepcopy(
                model.state_dict()
            )
            waiting = 0
        else:
            waiting += 1

        if waiting >= PATIENCE:
            break

    # Restore best validation checkpoint.
    model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():

        scores = torch.sigmoid(
            model(
                val_graph,
                val_target_edges
            )
        ).cpu().numpy()

    final_pr = float(
        average_precision_score(
            y_validation, scores
        )
    )

    final_roc = float(
        roc_auc_score(
            y_validation, scores
        )
    )

    structure = graph_structure_metrics(
        selected_val
    )

    result = {
        "method": method,
        "seed": seed,
        "best_epoch": best_epoch,
        "train_edges": len(selected_train),
        "validation_graph_edges": len(selected_val),
        "retention_ratio": (
            len(selected_val) / len(history_val_fair)
        ),
        "negative_pr_auc": final_pr,
        "roc_auc": final_roc,
        "largest_component_pct": (
            structure["largest_component_pct"]
        ),
        "isolated_nodes_pct": (
            structure["isolated_nodes_pct"]
        ),
        "retained_negative_pct": float(
            100 * (selected_val["rating"] < 0).mean()
        )
    }

    checkpoint = {
        "model_state_dict": best_state,
        "method": method,
        "seed": seed,
        "best_epoch": best_epoch,
        "keep_ratio": (
            1.0 if method == "full" else KEEP_RATIO
        ),
        "architecture": "SignedGraphSAGE",
        "hidden_dim": 32,
        "selection": (
            "full" if method == "full"
            else "coverage_constrained"
        )
    }

    torch.save(
        checkpoint,
        RESULTS_DIR /
        f"fair_retrained_{method}_seed{seed}.pt"
    )

    return result, training_log


# ------------------------------------------------------------
# 5. Train three independently initialized models
# ------------------------------------------------------------

methods = [
    "full",
    "random",
    "magnitude"
]

all_results = []
all_training_logs = []

print("=" * 88)
print("TA-GSNN | FAIR RETRAINED BASELINES")
print("=" * 88)

for method in methods:

    print(f"\nTraining: {method.upper()}")

    result, log = train_fair_baseline(
        method=method,
        seed=FAIR_SEED
    )

    all_results.append(result)
    all_training_logs.extend(log)

    print(
        f"Best epoch: {result['best_epoch']} | "
        f"PR-AUC: {result['negative_pr_auc']:.4f} | "
        f"ROC-AUC: {result['roc_auc']:.4f}"
    )


# ------------------------------------------------------------
# 6. Display and save results
# ------------------------------------------------------------

result_df = pd.DataFrame(all_results)

print("\n" + "=" * 105)
print("FINAL FAIR RETRAINED COMPARISON — VALIDATION")
print("=" * 105)

columns = [
    "method",
    "best_epoch",
    "train_edges",
    "validation_graph_edges",
    "negative_pr_auc",
    "roc_auc",
    "largest_component_pct",
    "isolated_nodes_pct",
    "retained_negative_pct"
]

print(
    result_df[columns].round(4).to_string(index=False)
)

result_df.to_csv(
    RESULTS_DIR /
    "bitcoin_alpha_fair_retrained_baselines.csv",
    index=False
)

pd.DataFrame(all_training_logs).to_csv(
    RESULTS_DIR /
    "bitcoin_alpha_fair_retrained_training_logs.csv",
    index=False
)

with open(
    RESULTS_DIR /
    "bitcoin_alpha_fair_retrained_config.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump({
        "seed": FAIR_SEED,
        "keep_ratio": KEEP_RATIO,
        "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE,
        "hidden_dim": 32,
        "dropout": 0.20,
        "learning_rate": 0.001,
        "weight_decay": 0.0001,
        "model_selection": "validation_negative_pr_auc",
        "methods": methods,
        "evaluation": "validation_only"
    }, f, indent=2)

print("\nCheckpoints, training logs and results saved.")
print("TEST SET NOT EVALUATED.")
print("=" * 105)


In [ ]:

# ============================================================
# TA-GSNN | Stage 3.6
# Five-Seed Fair Retrained Baseline Validation
# Validation only — No test evaluation
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

RESULTS_DIR = Path("/kaggle/working/ta_gsnn_results")

SEEDS = [42, 43, 44, 45, 46]
METHODS = ["full", "random", "magnitude"]

all_results = []
all_logs = []

print("=" * 85)
print("TA-GSNN | FIVE-SEED FAIR BASELINE VALIDATION")
print("=" * 85)

for seed in SEEDS:
    print(f"\n{'=' * 30} SEED {seed} {'=' * 30}")

    for method in METHODS:

        result, logs = train_fair_baseline(
            method=method,
            seed=seed
        )

        all_results.append(result)
        all_logs.extend(logs)

        print(
            f"{method:12s} | "
            f"PR-AUC={result['negative_pr_auc']:.4f} | "
            f"ROC-AUC={result['roc_auc']:.4f} | "
            f"Epoch={result['best_epoch']}"
        )

raw_df = pd.DataFrame(all_results)

summary_df = (
    raw_df.groupby("method", as_index=False)
    .agg(
        pr_auc_mean=("negative_pr_auc", "mean"),
        pr_auc_std=("negative_pr_auc", "std"),
        roc_auc_mean=("roc_auc", "mean"),
        roc_auc_std=("roc_auc", "std"),
        largest_component_mean=(
            "largest_component_pct", "mean"
        ),
        isolated_nodes_mean=(
            "isolated_nodes_pct", "mean"
        ),
        best_epoch_mean=("best_epoch", "mean")
    )
)

print("\n" + "=" * 100)
print("FINAL FIVE-SEED VALIDATION SUMMARY")
print("=" * 100)

print(summary_df.round(4).to_string(index=False))

# Paired comparison across identical seeds
pivot = raw_df.pivot(
    index="seed",
    columns="method",
    values="negative_pr_auc"
)

print("\nPaired PR-AUC differences by seed:")
paired = pd.DataFrame({
    "seed": pivot.index,
    "random_minus_full":
        pivot["random"].values - pivot["full"].values,
    "magnitude_minus_full":
        pivot["magnitude"].values - pivot["full"].values,
    "random_minus_magnitude":
        pivot["random"].values - pivot["magnitude"].values
})

print(paired.round(4).to_string(index=False))

raw_df.to_csv(
    RESULTS_DIR / "five_seed_baselines_raw.csv",
    index=False
)

summary_df.to_csv(
    RESULTS_DIR / "five_seed_baselines_summary.csv",
    index=False
)

paired.to_csv(
    RESULTS_DIR / "five_seed_paired_differences.csv",
    index=False
)

pd.DataFrame(all_logs).to_csv(
    RESULTS_DIR / "five_seed_training_logs.csv",
    index=False
)

print("\nAll five-seed results saved.")
print("Test set NOT evaluated.")
print("=" * 100)


In [ ]:

import os
import shutil
from pathlib import Path

output_dir = Path("/kaggle/working")
results_dir = output_dir / "ta_gsnn_results"
data_dir = output_dir / "ta_gsnn_data"

archive_base = output_dir / "TA_GSNN_Backup"

# Save experimental results, checkpoints and dataset files.
shutil.make_archive(
    str(archive_base),
    "zip",
    root_dir=str(output_dir),
    base_dir="."
)

archive = Path(str(archive_base) + ".zip")

print("Backup created successfully!")
print("File:", archive)
print("Size (MB):", round(archive.stat().st_size / 1024**2, 2))
print("Saved experiment files:", len(list(results_dir.rglob("*"))))
print("Saved data files:", len(list(data_dir.rglob("*"))))


In [ ]:

# ============================================================
# TA-GSNN | Cell 19 - Restart-Safe Version
# Seen vs Cold-Start Distribution
# No GPU or model training required
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("/kaggle/working/ta_gsnn_data")
RESULTS_DIR = Path("/kaggle/working/ta_gsnn_results")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

required_files = [
    DATA_DIR / "bitcoin_alpha_context.csv",
    DATA_DIR / "bitcoin_alpha_train.csv",
    DATA_DIR / "bitcoin_alpha_validation.csv"
]

missing = [str(p) for p in required_files if not p.exists()]

if missing:
    print("ERROR: Dataset files were not found:")
    for path in missing:
        print(" -", path)
    print("\nRestore the files from TA_GSNN_Backup.zip")
else:
    context = pd.read_csv(required_files[0])
    train = pd.read_csv(required_files[1])
    validation = pd.read_csv(required_files[2])

    history = pd.concat(
        [context, train],
        ignore_index=True
    )

    known_nodes = (
        set(history["source"]) |
        set(history["target"])
    )

    both_seen = (
        validation["source"].isin(known_nodes) &
        validation["target"].isin(known_nodes)
    )

    validation["group"] = np.where(
        both_seen,
        "both_seen",
        "cold_start"
    )

    summary = (
        validation.groupby("group")
        .agg(
            edges=("rating", "size"),
            negative_edges=(
                "rating", lambda x: (x < 0).sum()
            ),
            positive_edges=(
                "rating", lambda x: (x > 0).sum()
            )
        )
        .reset_index()
    )

    summary["negative_pct"] = (
        100 * summary["negative_edges"] /
        summary["edges"]
    )

    summary["share_pct"] = (
        100 * summary["edges"] /
        len(validation)
    )

    print("=" * 75)
    print("TA-GSNN | SEEN VS COLD-START DISTRIBUTION")
    print("=" * 75)
    print(summary.round(3).to_string(index=False))

    summary.to_csv(
        RESULTS_DIR /
        "bitcoin_alpha_seen_coldstart_distribution.csv",
        index=False
    )

    print("\nAnalysis completed successfully.")
    print("No training or GPU required.")


In [ ]:
from pathlib import Path
import torch

RESULTS_DIR = Path("/kaggle/working/ta_gsnn_results")

files = sorted(RESULTS_DIR.glob("fair_retrained_*.pt"))

print("=" * 65)
print("TA-GSNN | SAVED MODEL CHECKPOINTS")
print("=" * 65)

print("Checkpoints found:", len(files))

for path in files:
    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=True
    )

    print(
        path.name,
        "| method:", checkpoint.get("method"),
        "| seed:", checkpoint.get("seed"),
        "| best epoch:", checkpoint.get("best_epoch")
    )

if files:
    sample = torch.load(
        files[0],
        map_location="cpu",
        weights_only=True
    )

    print("\nMODEL PARAMETER SHAPES")
    for name, tensor in sample["model_state_dict"].items():
        print(name, tuple(tensor.shape))
else:
    print("\nNo checkpoint files found.")
    print("Check whether the backup was restored.")

In [ ]:

# ============================================================
# TA-GSNN | Cell 21
# Seen vs Cold-Start Evaluation | 15 Saved Checkpoints
# CPU only, no training, validation only
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, average_precision_score

DATA_DIR = Path("/kaggle/working/ta_gsnn_data")
RESULTS_DIR = Path("/kaggle/working/ta_gsnn_results")

DEVICE = torch.device("cpu")
SEEDS = [42, 43, 44, 45, 46]
METHODS = ["full", "random", "magnitude"]

parts_eval = {
    name: pd.read_csv(DATA_DIR / f"bitcoin_alpha_{name}.csv")
    for name in ["context", "train", "validation", "test"]
}

# Reconstruct the same global node indexing as Cell 8.
all_nodes = sorted(set(
    pd.concat(parts_eval.values())["source"].tolist()
    + pd.concat(parts_eval.values())["target"].tolist()
))
node_to_idx = {node: i for i, node in enumerate(all_nodes)}
n_nodes = len(all_nodes)

history = pd.concat(
    [parts_eval["context"], parts_eval["train"]],
    ignore_index=True
)
validation = parts_eval["validation"]

known = set(history["source"]) | set(history["target"])

both_seen = (
    validation["source"].isin(known)
    & validation["target"].isin(known)
).to_numpy()

labels = (validation["rating"].to_numpy() < 0).astype(int)
target_pairs = torch.tensor(
    [
        [node_to_idx[s], node_to_idx[t]]
        for s, t in zip(validation["source"], validation["target"])
    ],
    dtype=torch.long
)


def select_subset(df, method, seed):
    df = df.reset_index(drop=True)

    if method == "full":
        return df

    n = len(df)
    budget = round(n * 0.50)

    if method == "random":
        scores = np.random.default_rng(seed).random(n)
    else:
        scores = np.abs(df["rating"].to_numpy()).astype(float)

    ranking = np.argsort(-scores, kind="stable")

    sources = df["source"].to_numpy()
    destinations = df["target"].to_numpy()

    required = {}
    for i in ranking:
        required.setdefault(int(sources[i]), int(i))
        required.setdefault(int(destinations[i]), int(i))

    selected = set(required.values())
    if len(selected) > budget:
        raise ValueError("Coverage budget is infeasible")

    for i in ranking:
        if len(selected) >= budget:
            break
        selected.add(int(i))

    return df.iloc[sorted(selected)].copy()


def build_graph(df):
    src = np.array(
        [node_to_idx[v] for v in df["source"]],
        dtype=np.int64
    )
    dst = np.array(
        [node_to_idx[v] for v in df["target"]],
        dtype=np.int64
    )

    positive = df["rating"].to_numpy() > 0

    features = np.zeros((n_nodes, 4), dtype=np.float32)

    np.add.at(features[:, 0], src[positive], 1)
    np.add.at(features[:, 1], src[~positive], 1)
    np.add.at(features[:, 2], dst[positive], 1)
    np.add.at(features[:, 3], dst[~positive], 1)

    return {
        "x": torch.from_numpy(np.log1p(features)),
        "pos": torch.tensor(
            np.stack([src[positive], dst[positive]]),
            dtype=torch.long
        ),
        "neg": torch.tensor(
            np.stack([src[~positive], dst[~positive]]),
            dtype=torch.long
        )
    }


def aggregate(h, edges, incoming):
    src, dst = edges[0], edges[1]

    receiver = dst if incoming else src
    sender = src if incoming else dst

    result = torch.zeros_like(h)
    result.index_add_(0, receiver, h[sender])

    counts = torch.bincount(
        receiver, minlength=n_nodes
    ).to(h.dtype).clamp(min=1)

    return result / counts.unsqueeze(1)


def signed_layer(h, graph, state, prefix):
    pieces = [h]

    for sign in ["pos", "neg"]:
        pieces.append(aggregate(h, graph[sign], True))
        pieces.append(aggregate(h, graph[sign], False))

    combined = torch.cat(pieces, dim=1)

    z = F.linear(
        combined,
        state[f"{prefix}.linear.weight"],
        state[f"{prefix}.linear.bias"]
    )

    z = F.layer_norm(
        z,
        (z.shape[1],),
        state[f"{prefix}.norm.weight"],
        state[f"{prefix}.norm.bias"],
        eps=1e-5
    )

    return F.relu(z)


@torch.no_grad()
def predict(graph, state):
    h = signed_layer(graph["x"], graph, state, "layer1")
    h = signed_layer(h, graph, state, "layer2")

    u = h[target_pairs[:, 0]]
    v = h[target_pairs[:, 1]]

    pair = torch.cat(
        [u, v, torch.abs(u - v), u * v],
        dim=1
    )

    z = F.relu(F.linear(
        pair,
        state["decoder.0.weight"],
        state["decoder.0.bias"]
    ))

    logits = F.linear(
        z,
        state["decoder.3.weight"],
        state["decoder.3.bias"]
    ).squeeze(1)

    return torch.sigmoid(logits).numpy()


records = []

for seed in SEEDS:
    for method in METHODS:
        path = RESULTS_DIR / f"fair_retrained_{method}_seed{seed}.pt"

        checkpoint = torch.load(
            path, map_location="cpu", weights_only=True
        )

        subset = select_subset(history, method, seed)
        graph = build_graph(subset)
        scores = predict(graph, checkpoint["model_state_dict"])

        for group, mask in [
            ("overall", np.ones(len(labels), dtype=bool)),
            ("both_seen", both_seen),
            ("cold_start", ~both_seen)
        ]:
            y = labels[mask]
            p = scores[mask]

            records.append({
                "method": method,
                "seed": seed,
                "group": group,
                "edges": len(y),
                "negative_prevalence": y.mean(),
                "pr_auc": average_precision_score(y, p),
                "roc_auc": roc_auc_score(y, p)
            })

raw = pd.DataFrame(records)

summary = (
    raw.groupby(["method", "group"], as_index=False)
    .agg(
        edges=("edges", "mean"),
        negative_prevalence=("negative_prevalence", "mean"),
        pr_auc_mean=("pr_auc", "mean"),
        pr_auc_std=("pr_auc", "std"),
        roc_auc_mean=("roc_auc", "mean"),
        roc_auc_std=("roc_auc", "std")
    )
)

print("=" * 100)
print("TA-GSNN | SEEN VS COLD-START PERFORMANCE")
print("=" * 100)
print(summary.round(4).to_string(index=False))

# Verify reconstruction against previously saved results.
reference_path = RESULTS_DIR / "five_seed_baselines_raw.csv"

if reference_path.exists():
    reference = pd.read_csv(reference_path)
    reconstructed = raw[raw["group"] == "overall"]

    comparison = reconstructed.merge(
        reference[["method", "seed", "negative_pr_auc"]],
        on=["method", "seed"]
    )

    comparison["absolute_difference"] = np.abs(
        comparison["pr_auc"] - comparison["negative_pr_auc"]
    )

    max_diff = comparison["absolute_difference"].max()

    print("\nRECONSTRUCTION CHECK")
    print("Maximum PR-AUC difference:", round(float(max_diff), 6))

    if max_diff > 0.0005:
        print("WARNING: Predictions do not match saved baselines.")
        print("Do not interpret subgroup results until verified.")
    else:
        print("PASS: Reconstruction matches saved baselines.")
else:
    print("\nReference CSV unavailable; reconstruction unverified.")

raw.to_csv(
    RESULTS_DIR / "seen_coldstart_five_seed_raw.csv",
    index=False
)
summary.to_csv(
    RESULTS_DIR / "seen_coldstart_five_seed_summary.csv",
    index=False
)

print("\nSaved subgroup results. No test predictions evaluated.")


In [ ]:

from pathlib import Path
import urllib.request
import hashlib
import pandas as pd

DATA_DIR = Path("/kaggle/working/ta_gsnn_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

url = "https://snap.stanford.edu/data/soc-sign-bitcoinotc.csv.gz"
path = DATA_DIR / "soc-sign-bitcoinotc.csv.gz"

if not path.exists():
    urllib.request.urlretrieve(url, path)

checksum = hashlib.sha256(path.read_bytes()).hexdigest()

df = pd.read_csv(
    path,
    compression="gzip",
    header=None,
    names=["source", "target", "rating", "timestamp"]
)

print("=" * 65)
print("BITCOIN OTC | DATASET VERIFICATION")
print("=" * 65)
print("Rows:", len(df))
print("Nodes:", len(set(df.source) | set(df.target)))
print("Positive edges:", int((df.rating > 0).sum()))
print("Negative edges:", int((df.rating < 0).sum()))
print("Duplicate directed pairs:", int(
    df.duplicated(["source", "target"]).sum()
))
print("Missing values:", int(df.isna().sum().sum()))
print("SHA256:", checksum)
print("First timestamp:", df.timestamp.min())
print("Last timestamp:", df.timestamp.max())
print("Data file:", path)


In [ ]:

# ============================================================
# TA-GSNN | Cell 23
# Bitcoin OTC - Chronological Data Preparation
# CPU only | No model training | No test predictions
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import hashlib

DATA_DIR = Path("/kaggle/working/ta_gsnn_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

raw_path = DATA_DIR / "soc-sign-bitcoinotc.csv.gz"

df = pd.read_csv(
    raw_path,
    compression="gzip",
    header=None,
    names=["source", "target", "rating", "timestamp"]
)

assert df.notna().all().all()
assert (df["rating"] != 0).all()
assert not df.duplicated(["source", "target"]).any()

df = df.sort_values(
    "timestamp", kind="stable"
).reset_index(drop=True)

N = len(df)

# Find the nearest split boundary that does not
# divide records sharing an identical timestamp.
def time_safe_boundary(data, target_fraction):
    target = round(len(data) * target_fraction)

    times = data["timestamp"].to_numpy()

    candidates = np.flatnonzero(
        times[1:] != times[:-1]
    ) + 1

    boundary = int(
        candidates[np.argmin(np.abs(
            candidates - target
        ))]
    )

    return boundary


b60 = time_safe_boundary(df, 0.60)
b80 = time_safe_boundary(df, 0.80)
b90 = time_safe_boundary(df, 0.90)

assert 0 < b60 < b80 < b90 < N

splits = {
    "context": df.iloc[:b60].copy(),
    "train": df.iloc[b60:b80].copy(),
    "validation": df.iloc[b80:b90].copy(),
    "test": df.iloc[b90:].copy()
}

# Sanity check: no target edge leaks into older history.
assert sum(len(x) for x in splits.values()) == N

for earlier, later in [
    ("context", "train"),
    ("train", "validation"),
    ("validation", "test")
]:
    assert (
        splits[earlier]["timestamp"].max()
        < splits[later]["timestamp"].min()
    )

records = []
all_previous_nodes = set()

for name, subset in splits.items():

    current_nodes = (
        set(subset["source"]) |
        set(subset["target"])
    )

    cold_edges = (
        ~subset["source"].isin(all_previous_nodes)
        | ~subset["target"].isin(all_previous_nodes)
    ).sum()

    records.append({
        "split": name,
        "edges": len(subset),
        "positive": int((subset["rating"] > 0).sum()),
        "negative": int((subset["rating"] < 0).sum()),
        "negative_pct": round(
            100 * (subset["rating"] < 0).mean(), 3
        ),
        "cold_start_pct": (
            None if name == "context"
            else round(100 * cold_edges / len(subset), 3)
        ),
        "start_timestamp": subset["timestamp"].min(),
        "end_timestamp": subset["timestamp"].max()
    })

    all_previous_nodes.update(current_nodes)

    # Preserve the original ordered edge attributes.
    output = DATA_DIR / f"bitcoin_otc_{name}.csv"
    subset.to_csv(output, index=False)

summary = pd.DataFrame(records)

print("=" * 105)
print("TA-GSNN | BITCOIN OTC CHRONOLOGICAL SPLITS")
print("=" * 105)

print(summary.to_string(index=False))

print("\nTotal edges:", N)
print("Unique nodes:", len(set(df.source) | set(df.target)))
print("Split boundaries:", b60, b80, b90)

summary.to_csv(
    DATA_DIR / "bitcoin_otc_split_summary.csv",
    index=False
)

print("\nFour chronological split files saved.")
print("No GNN training or test-set prediction performed.")


In [ ]:

# ============================================================
# TA-GSNN | Cell 24
# Connectivity-Preserving Trust-Aware Baseline
# CPU only | Validation history only | No test evaluation
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path("/kaggle/working/ta_gsnn_data")
RESULTS_DIR = Path("/kaggle/working/ta_gsnn_results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

KEEP_RATIO = 0.50


def component_count(src, dst):
    nodes = np.unique(np.concatenate([src, dst]))
    lookup = {v: i for i, v in enumerate(nodes)}
    parent = list(range(len(nodes)))
    size = [1] * len(nodes)

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        a, b = find(a), find(b)
        if a == b:
            return False
        if size[a] < size[b]:
            a, b = b, a
        parent[b] = a
        size[a] += size[b]
        return True

    for u, v in zip(src, dst):
        union(lookup[u], lookup[v])

    return len({find(i) for i in range(len(nodes))})


def preserve_connectivity(history):
    history = history.reset_index(drop=True).copy()

    src = history["source"].to_numpy()
    dst = history["target"].to_numpy()
    ratings = history["rating"].to_numpy()

    n = len(history)
    budget = round(n * KEEP_RATIO)

    nodes = np.unique(np.concatenate([src, dst]))
    lookup = {v: i for i, v in enumerate(nodes)}

    degree = np.zeros(len(nodes), dtype=np.int64)

    for u, v in zip(src, dst):
        degree[lookup[u]] += 1
        degree[lookup[v]] += 1

    # Historical attributes only.
    # Magnitude + preference for edges touching low-degree nodes.
    structural = np.array([
        1 / np.sqrt(degree[lookup[u]]) +
        1 / np.sqrt(degree[lookup[v]])
        for u, v in zip(src, dst)
    ])

    score = (
        np.abs(ratings) / 10.0
        + 0.5 * structural
    )

    ranking = np.argsort(-score, kind="stable")

    parent = list(range(len(nodes)))
    size = [1] * len(nodes)

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        a, b = find(a), find(b)

        if a == b:
            return False

        if size[a] < size[b]:
            a, b = b, a

        parent[b] = a
        size[a] += size[b]
        return True

    selected = np.zeros(n, dtype=bool)

    # Step 1: maximum-score spanning forest.
    for i in ranking:
        if union(lookup[src[i]], lookup[dst[i]]):
            selected[i] = True

    forest_edges = int(selected.sum())

    if forest_edges > budget:
        raise ValueError(
            "Connectivity cannot be preserved at this budget."
        )

    # Step 2: approximately preserve historical sign ratio.
    negative = ratings < 0
    target_negative = round(budget * negative.mean())

    selected_negative = int((selected & negative).sum())
    needed_negative = max(0, target_negative - selected_negative)

    negative_candidates = ranking[
        negative[ranking] & ~selected[ranking]
    ]

    capacity = budget - int(selected.sum())

    chosen_negative = negative_candidates[
        :min(needed_negative, capacity)
    ]
    selected[chosen_negative] = True

    # Step 3: fill remaining budget by importance.
    capacity = budget - int(selected.sum())
    remaining = ranking[~selected[ranking]]
    selected[remaining[:capacity]] = True

    assert int(selected.sum()) == budget

    subset = history.iloc[np.flatnonzero(selected)].copy()

    original_components = component_count(src, dst)

    retained_components = component_count(
        subset["source"].to_numpy(),
        subset["target"].to_numpy()
    )

    original_nodes = set(src) | set(dst)
    retained_nodes = (
        set(subset["source"]) | set(subset["target"])
    )

    assert original_nodes == retained_nodes
    assert original_components == retained_components

    return subset, forest_edges, original_components


records = []

for dataset in ["alpha", "otc"]:

    prefix = f"bitcoin_{dataset}"

    context = pd.read_csv(
        DATA_DIR / f"{prefix}_context.csv"
    )
    train = pd.read_csv(
        DATA_DIR / f"{prefix}_train.csv"
    )

    history = pd.concat(
        [context, train], ignore_index=True
    )

    selected, forest_edges, components = (
        preserve_connectivity(history)
    )

    filename = (
        RESULTS_DIR /
        f"{prefix}_connectivity_preserving_50pct.csv"
    )
    selected.to_csv(filename, index=False)

    records.append({
        "dataset": dataset,
        "original_edges": len(history),
        "retained_edges": len(selected),
        "forest_edges": forest_edges,
        "weak_components": components,
        "original_negative_pct":
            100 * (history["rating"] < 0).mean(),
        "retained_negative_pct":
            100 * (selected["rating"] < 0).mean(),
        "isolated_historical_nodes": 0
    })

results = pd.DataFrame(records)

print("=" * 100)
print("TA-GSNN | CONNECTIVITY-PRESERVING SPARSIFICATION")
print("=" * 100)
print(results.round(3).to_string(index=False))

results.to_csv(
    RESULTS_DIR / "connectivity_preserving_summary.csv",
    index=False
)

print("\nConnectivity verified for both datasets.")
print("No model training. No test predictions.")


In [ ]:

# ============================================================
# TA-GSNN | Cell 25
# Connectivity-Preserving GNN - Alpha, Seed 42
# One-seed pilot | Validation only | No test evaluation
# ============================================================

import copy
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, roc_auc_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Use CPU for this small pilot.
device = torch.device("cpu")

# ------------------------------------------------------------
# 1. Reconstruct trainable Signed GraphSAGE
# ------------------------------------------------------------

class SignedLayer25(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(5 * in_dim, out_dim)
        self.norm = nn.LayerNorm(out_dim)

    def forward(self, h, graph):
        pieces = [h]

        for sign in ("pos", "neg"):
            pieces.append(aggregate(h, graph[sign], True))
            pieces.append(aggregate(h, graph[sign], False))

        return self.norm(
            self.linear(torch.cat(pieces, dim=1))
        )


class SignedGraphSAGE25(nn.Module):
    def __init__(self):
        super().__init__()

        self.layer1 = SignedLayer25(4, 32)
        self.layer2 = SignedLayer25(32, 32)
        self.dropout = 0.20

        self.decoder = nn.Sequential(
            nn.Linear(128, 32),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(32, 1)
        )

    def forward(self, graph, edge_pairs):
        h = F.relu(self.layer1(graph["x"], graph))
        h = F.dropout(
            h, p=self.dropout, training=self.training
        )

        h = F.relu(self.layer2(h, graph))
        h = F.dropout(
            h, p=self.dropout, training=self.training
        )

        u = h[edge_pairs[:, 0]]
        v = h[edge_pairs[:, 1]]

        pair = torch.cat(
            [u, v, torch.abs(u - v), u * v],
            dim=1
        )

        return self.decoder(pair).squeeze(-1)


# ------------------------------------------------------------
# 2. Verify architecture against original saved checkpoint
# ------------------------------------------------------------

reference_checkpoint = torch.load(
    RESULTS_DIR / "fair_retrained_full_seed42.pt",
    map_location="cpu",
    weights_only=True
)

reference_model = SignedGraphSAGE25()
reference_model.load_state_dict(
    reference_checkpoint["model_state_dict"],
    strict=True
)
reference_model.eval()

alpha_context = pd.read_csv(
    DATA_DIR / "bitcoin_alpha_context.csv"
)
alpha_train = pd.read_csv(
    DATA_DIR / "bitcoin_alpha_train.csv"
)
alpha_validation = pd.read_csv(
    DATA_DIR / "bitcoin_alpha_validation.csv"
)

alpha_history = pd.concat(
    [alpha_context, alpha_train],
    ignore_index=True
)

val_graph_full = build_graph(alpha_history)

with torch.no_grad():
    ref_scores = torch.sigmoid(
        reference_model(val_graph_full, target_pairs)
    ).numpy()

ref_pr = average_precision_score(labels, ref_scores)

print("Original model reconstruction PR-AUC:", round(ref_pr, 6))

if abs(ref_pr - 0.1599875976481427) > 0.0005:
    raise RuntimeError(
        "Architecture reconstruction mismatch. "
        "Stop before training."
    )

print("Architecture reconstruction: PASS")


# ------------------------------------------------------------
# 3. Prepare connectivity-preserving train/val graphs
# ------------------------------------------------------------

cp_train_df, _, _ = preserve_connectivity(
    alpha_context
)

cp_val_df = pd.read_csv(
    RESULTS_DIR /
    "bitcoin_alpha_connectivity_preserving_50pct.csv"
)

cp_train_graph = build_graph(cp_train_df)
cp_val_graph = build_graph(cp_val_df)

train_pairs = torch.tensor(
    [
        [node_to_idx[s], node_to_idx[t]]
        for s, t in zip(
            alpha_train["source"], alpha_train["target"]
        )
    ],
    dtype=torch.long
)

train_y = torch.tensor(
    (alpha_train["rating"].to_numpy() < 0).astype(float),
    dtype=torch.float32
)

validation_y = labels.astype(int)

positive_count = float((train_y == 0).sum())
negative_count = float((train_y == 1).sum())

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        positive_count / negative_count
    )
)


# ------------------------------------------------------------
# 4. Train with the same main optimizer configuration
# ------------------------------------------------------------

torch.manual_seed(SEED)

model = SignedGraphSAGE25().to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

best_pr = -1.0
best_roc = None
best_epoch = None
best_state = None
waiting = 0

training_records = []

print("\n" + "=" * 70)
print("CONNECTIVITY-PRESERVING GNN TRAINING")
print("=" * 70)

for epoch in range(1, 121):
    model.train()
    optimizer.zero_grad()

    train_logits = model(
        cp_train_graph, train_pairs
    )

    loss = criterion(train_logits, train_y)
    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        model.parameters(), 5.0
    )

    optimizer.step()

    model.eval()

    with torch.no_grad():
        val_scores = torch.sigmoid(
            model(cp_val_graph, target_pairs)
        ).numpy()

    pr = average_precision_score(
        validation_y, val_scores
    )
    roc = roc_auc_score(
        validation_y, val_scores
    )

    training_records.append({
        "epoch": epoch,
        "train_loss": float(loss.item()),
        "validation_pr_auc": float(pr),
        "validation_roc_auc": float(roc)
    })

    if pr > best_pr + 1e-6:
        best_pr = float(pr)
        best_roc = float(roc)
        best_epoch = epoch
        best_state = copy.deepcopy(
            model.state_dict()
        )
        waiting = 0
    else:
        waiting += 1

    if epoch == 1 or epoch % 10 == 0:
        print(
            f"Epoch {epoch:3d} | "
            f"Loss={loss.item():.4f} | "
            f"PR-AUC={pr:.4f} | "
            f"ROC-AUC={roc:.4f}"
        )

    if waiting >= 20:
        print("Early stopping:", epoch)
        break

# ------------------------------------------------------------
# 5. Save pilot results
# ------------------------------------------------------------

torch.save(
    {
        "model_state_dict": best_state,
        "method": "connectivity_preserving",
        "seed": SEED,
        "best_epoch": best_epoch,
        "keep_ratio": 0.50
    },
    RESULTS_DIR /
    "connectivity_preserving_alpha_seed42.pt"
)

pd.DataFrame(training_records).to_csv(
    RESULTS_DIR /
    "connectivity_preserving_alpha_seed42_log.csv",
    index=False
)

print("\n" + "=" * 70)
print("CONNECTIVITY-PRESERVING PILOT RESULTS")
print("=" * 70)
print("Best epoch:", best_epoch)
print("Validation PR-AUC:", round(best_pr, 4))
print("Validation ROC-AUC:", round(best_roc, 4))
print("Training graph edges:", len(cp_train_df))
print("Validation history edges:", len(cp_val_df))
print("Checkpoint saved. Test set NOT evaluated.")


In [ ]:

# ============================================================
# TA-GSNN | Cell 27 - CUDA VERSION
# Gradient-Based Task-Aware Edge Importance
#
# Dataset: Bitcoin Alpha
# Model: Saved Full Signed GNN, Seed 42
# Labels: TRAIN ONLY
# No retraining, no test evaluation
# ============================================================

from pathlib import Path
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

DATA_DIR = Path("/kaggle/working/ta_gsnn_data")
RESULTS_DIR = Path("/kaggle/working/ta_gsnn_results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 75, flush=True)
print("TA-GSNN | GPU TASK-AWARE EDGE IMPORTANCE", flush=True)
print("Device:", DEVICE, flush=True)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0), flush=True)

start_time = time.perf_counter()

# ------------------------------------------------------------
# 1. Load data
# ------------------------------------------------------------

context = pd.read_csv(
    DATA_DIR / "bitcoin_alpha_context.csv"
)

train = pd.read_csv(
    DATA_DIR / "bitcoin_alpha_train.csv"
)

# Global IDs are used for consistent indexing ONLY.
# No future ratings or edge attributes are used.
node_ids = set()

for split in ["context", "train", "validation", "test"]:
    frame = pd.read_csv(
        DATA_DIR / f"bitcoin_alpha_{split}.csv",
        usecols=["source", "target"]
    )
    node_ids.update(frame["source"].tolist())
    node_ids.update(frame["target"].tolist())

node_ids = sorted(node_ids)
node_map = {node: i for i, node in enumerate(node_ids)}

N_NODES = len(node_ids)
N_EDGES = len(context)

def index_tensor(series):
    values = series.map(node_map).to_numpy()

    if pd.isna(values).any():
        raise ValueError("Unknown node ID")

    return torch.as_tensor(
        values, dtype=torch.long, device=DEVICE
    )

src = index_tensor(context["source"])
dst = index_tensor(context["target"])

ratings = context["rating"].to_numpy()

pos_mask = torch.as_tensor(
    ratings > 0,
    dtype=torch.bool,
    device=DEVICE
)

neg_mask = ~pos_mask

target_edges = torch.stack([
    index_tensor(train["source"]),
    index_tensor(train["target"])
], dim=1)

target_y = torch.as_tensor(
    (train["rating"].to_numpy() < 0).astype(np.float32),
    device=DEVICE
)

print(
    f"Nodes: {N_NODES} | "
    f"Historical edges: {N_EDGES} | "
    f"Train targets: {len(train)}",
    flush=True
)

# ------------------------------------------------------------
# 2. Load frozen checkpoint directly onto GPU
# ------------------------------------------------------------

checkpoint_path = (
    RESULTS_DIR / "fair_retrained_full_seed42.pt"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location=DEVICE,
    weights_only=True
)

state = {
    name: tensor.detach().to(DEVICE)
    for name, tensor in
    checkpoint["model_state_dict"].items()
}

print("Checkpoint loaded successfully.", flush=True)

# ------------------------------------------------------------
# 3. Differentiable node features
# ------------------------------------------------------------

def weighted_features(weights):

    features = torch.zeros(
        (N_NODES, 4),
        dtype=torch.float32,
        device=DEVICE
    )

    # Positive outgoing
    contributions = torch.zeros(
        (int(pos_mask.sum()), 4),
        device=DEVICE
    )
    contributions[:, 0] = weights[pos_mask]

    features = features.index_add(
        0, src[pos_mask], contributions
    )

    # Negative outgoing
    contributions = torch.zeros(
        (int(neg_mask.sum()), 4),
        device=DEVICE
    )
    contributions[:, 1] = weights[neg_mask]

    features = features.index_add(
        0, src[neg_mask], contributions
    )

    # Positive incoming
    contributions = torch.zeros(
        (int(pos_mask.sum()), 4),
        device=DEVICE
    )
    contributions[:, 2] = weights[pos_mask]

    features = features.index_add(
        0, dst[pos_mask], contributions
    )

    # Negative incoming
    contributions = torch.zeros(
        (int(neg_mask.sum()), 4),
        device=DEVICE
    )
    contributions[:, 3] = weights[neg_mask]

    features = features.index_add(
        0, dst[neg_mask], contributions
    )

    return torch.log1p(features)


# ------------------------------------------------------------
# 4. Weighted signed message aggregation
# ------------------------------------------------------------

def weighted_mean(h, s, d, weights, incoming):

    receiver = d if incoming else s
    sender = s if incoming else d

    numerator = torch.zeros_like(h).index_add(
        0,
        receiver,
        h[sender] * weights.unsqueeze(1)
    )

    denominator = torch.zeros(
        N_NODES,
        dtype=h.dtype,
        device=DEVICE
    ).index_add(
        0, receiver, weights
    )

    # At unit weights, this matches the original
    # count-based mean aggregation exactly.
    return (
        numerator /
        denominator.clamp(min=1.0).unsqueeze(1)
    )


def signed_layer(h, edge_weights, prefix):

    pieces = [h]

    for mask in (pos_mask, neg_mask):

        s = src[mask]
        d = dst[mask]
        w = edge_weights[mask]

        pieces.append(
            weighted_mean(h, s, d, w, True)
        )

        pieces.append(
            weighted_mean(h, s, d, w, False)
        )

    combined = torch.cat(pieces, dim=1)

    z = F.linear(
        combined,
        state[f"{prefix}.linear.weight"],
        state[f"{prefix}.linear.bias"]
    )

    z = F.layer_norm(
        z,
        (z.shape[1],),
        state[f"{prefix}.norm.weight"],
        state[f"{prefix}.norm.bias"],
        eps=1e-5
    )

    return F.relu(z)


# ------------------------------------------------------------
# 5. Full weighted GNN forward
# ------------------------------------------------------------

def weighted_predict(edge_weights):

    h = weighted_features(edge_weights)

    h = signed_layer(
        h, edge_weights, "layer1"
    )

    h = signed_layer(
        h, edge_weights, "layer2"
    )

    u = h[target_edges[:, 0]]
    v = h[target_edges[:, 1]]

    pair = torch.cat([
        u,
        v,
        torch.abs(u - v),
        u * v
    ], dim=1)

    z = F.relu(F.linear(
        pair,
        state["decoder.0.weight"],
        state["decoder.0.bias"]
    ))

    logits = F.linear(
        z,
        state["decoder.3.weight"],
        state["decoder.3.bias"]
    ).squeeze(-1)

    return logits


# ------------------------------------------------------------
# 6. Original unweighted model forward for verification
# ------------------------------------------------------------

def ordinary_mean(h, s, d, incoming):

    receiver = d if incoming else s
    sender = s if incoming else d

    result = torch.zeros_like(h).index_add(
        0, receiver, h[sender]
    )

    counts = torch.zeros(
        N_NODES, device=DEVICE, dtype=h.dtype
    ).index_add(
        0,
        receiver,
        torch.ones(
            receiver.numel(),
            device=DEVICE,
            dtype=h.dtype
        )
    )

    return result / counts.clamp(min=1).unsqueeze(1)


def ordinary_predict():

    x = torch.zeros(
        (N_NODES, 4), device=DEVICE
    )

    for column, indices in [
        (0, src[pos_mask]),
        (1, src[neg_mask]),
        (2, dst[pos_mask]),
        (3, dst[neg_mask])
    ]:
        values = torch.ones(
            len(indices), device=DEVICE
        )
        x[:, column].index_add_(
            0, indices, values
        )

    h = torch.log1p(x)

    for prefix in ("layer1", "layer2"):

        pieces = [h]

        for mask in (pos_mask, neg_mask):
            s, d = src[mask], dst[mask]

            pieces.append(
                ordinary_mean(h, s, d, True)
            )
            pieces.append(
                ordinary_mean(h, s, d, False)
            )

        z = F.linear(
            torch.cat(pieces, dim=1),
            state[f"{prefix}.linear.weight"],
            state[f"{prefix}.linear.bias"]
        )

        h = F.relu(F.layer_norm(
            z,
            (z.shape[1],),
            state[f"{prefix}.norm.weight"],
            state[f"{prefix}.norm.bias"],
            eps=1e-5
        ))

    u = h[target_edges[:, 0]]
    v = h[target_edges[:, 1]]

    pair = torch.cat([
        u, v, torch.abs(u - v), u * v
    ], dim=1)

    z = F.relu(F.linear(
        pair,
        state["decoder.0.weight"],
        state["decoder.0.bias"]
    ))

    return F.linear(
        z,
        state["decoder.3.weight"],
        state["decoder.3.bias"]
    ).squeeze(-1)


# ------------------------------------------------------------
# 7. Verify weighted forward == original forward
# ------------------------------------------------------------

unit_weights = torch.ones(
    N_EDGES, device=DEVICE
)

with torch.no_grad():

    original_logits = ordinary_predict()

    weighted_logits = weighted_predict(
        unit_weights
    )

    max_difference = (
        original_logits - weighted_logits
    ).abs().max().item()

print("\n" + "=" * 75, flush=True)
print("RECONSTRUCTION CHECK", flush=True)
print("Maximum logit difference:", max_difference, flush=True)

if not np.isfinite(max_difference) or max_difference > 1e-4:
    raise RuntimeError(
        "Weighted GNN differs from original model. "
        "Stop before computing importance."
    )

print("PASS: Weighted GNN matches original.", flush=True)

# ------------------------------------------------------------
# 8. Compute gradient-based importance
# ------------------------------------------------------------

edge_weights = torch.ones(
    N_EDGES,
    dtype=torch.float32,
    device=DEVICE,
    requires_grad=True
)

n_negative = float(target_y.sum().item())
n_positive = len(target_y) - n_negative

class_weight = torch.tensor(
    n_positive / n_negative,
    device=DEVICE,
    dtype=torch.float32
)

print(
    "\nComputing training-loss gradients...",
    flush=True
)

logits = weighted_predict(edge_weights)

loss = F.binary_cross_entropy_with_logits(
    logits,
    target_y,
    pos_weight=class_weight
)

gradient_tensor = torch.autograd.grad(
    outputs=loss,
    inputs=edge_weights
)[0]

gradient = (
    gradient_tensor.detach()
    .cpu().numpy()
)

importance = -gradient

if not np.isfinite(importance).all():
    raise RuntimeError(
        "Non-finite edge importance detected."
    )

# ------------------------------------------------------------
# 9. Export results
# ------------------------------------------------------------

output = context.copy()

output["gradient"] = gradient
output["task_importance"] = importance
output["absolute_gradient"] = np.abs(gradient)

output_path = (
    RESULTS_DIR /
    "bitcoin_alpha_task_importance_seed42.csv"
)

output.to_csv(output_path, index=False)

print("\n" + "=" * 75, flush=True)
print("TASK-AWARE EDGE IMPORTANCE | GPU RESULTS", flush=True)
print("=" * 75, flush=True)

print("Train loss:", round(loss.item(), 6))
print("Positive importance:", int((importance > 0).sum()))
print("Negative importance:", int((importance < 0).sum()))
print(
    "Near-zero gradients (<1e-9):",
    int((np.abs(importance) < 1e-9).sum())
)

print("\nImportance statistics:")
print(
    pd.Series(importance).describe(
        percentiles=[0.01, 0.10, 0.50, 0.90, 0.99]
    ).to_string()
)

print("\nImportance by trust sign:")

output["sign"] = np.where(
    output["rating"] > 0,
    "positive",
    "negative"
)

print(
    output.groupby("sign")["task_importance"]
    .agg(["count", "mean", "median", "std"])
    .to_string()
)

if DEVICE.type == "cuda":
    torch.cuda.synchronize()

elapsed = time.perf_counter() - start_time

print("\nExecution time (seconds):", round(elapsed, 2))
print("Saved:", output_path)
print("No retraining. Training labels only.")
print("Validation/test labels NOT evaluated.")
print("=" * 75)


In [ ]:

# ============================================================
# TA-GSNN | Cell 28
# Validate Task-Aware Gradient Importance
# Finite Difference vs Actual Edge Removal
# GPU | Bitcoin Alpha | Training Loss Only
# Requires Cell 27 to have completed
# ============================================================

import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from scipy.stats import spearmanr, pearsonr

assert "weighted_predict" in globals()
assert "importance" in globals()
assert "edge_weights" in globals()
assert "target_y" in globals()

rng = np.random.default_rng(2026)
n_edges = len(importance)

# Sample 60 distinct edges:
# 15 highest, 15 lowest and 30 random middle-ranked.
ranking = np.argsort(importance)

top_ids = ranking[-15:]
bottom_ids = ranking[:15]

excluded = np.concatenate([top_ids, bottom_ids])
remaining = np.setdiff1d(
    np.arange(n_edges), excluded
)

random_ids = rng.choice(
    remaining, size=30, replace=False
)

selected_ids = np.concatenate([
    top_ids, bottom_ids, random_ids
])

selected_groups = (
    ["top_positive"] * 15
    + ["bottom_negative"] * 15
    + ["random"] * 30
)

# Consistent, frozen model; same training objective.
def training_loss_for_weights(weights):
    logits = weighted_predict(weights)
    return F.binary_cross_entropy_with_logits(
        logits, target_y, pos_weight=class_weight
    )

base_weights = torch.ones(
    n_edges, dtype=torch.float32, device=DEVICE
)

with torch.no_grad():
    baseline_loss = float(
        training_loss_for_weights(base_weights).item()
    )

records = []
started = time.perf_counter()

print("=" * 76, flush=True)
print("TA-GSNN | GRADIENT VALIDATION", flush=True)
print("Edges sampled:", len(selected_ids), flush=True)
print("Baseline train loss:", round(baseline_loss, 8), flush=True)

with torch.no_grad():
    for position, (edge_id, group) in enumerate(
        zip(selected_ids, selected_groups), start=1
    ):
        edge_id = int(edge_id)

        # Small perturbation: m_e = 0.99
        weights_small = base_weights.clone()
        weights_small[edge_id] = 0.99

        small_loss = float(
            training_loss_for_weights(weights_small).item()
        )

        # Complete removal: m_e = 0
        weights_deleted = base_weights.clone()
        weights_deleted[edge_id] = 0.0

        deleted_loss = float(
            training_loss_for_weights(weights_deleted).item()
        )

        records.append({
            "edge_id": edge_id,
            "group": group,
            "historical_sign": (
                "negative" if ratings[edge_id] < 0
                else "positive"
            ),
            "importance": float(importance[edge_id]),
            "predicted_small_delta": float(
                0.01 * importance[edge_id]
            ),
            "actual_small_delta": (
                small_loss - baseline_loss
            ),
            "predicted_deletion_delta": float(
                importance[edge_id]
            ),
            "actual_deletion_delta": (
                deleted_loss - baseline_loss
            )
        })

        if position % 15 == 0:
            print(
                f"Evaluated {position}/{len(selected_ids)} edges",
                flush=True
            )

df28 = pd.DataFrame(records)

def valid_corr(a, b, method):
    if np.std(a) == 0 or np.std(b) == 0:
        return np.nan
    if method == "spearman":
        return spearmanr(a, b).statistic
    return pearsonr(a, b).statistic

small_corr = valid_corr(
    df28["predicted_small_delta"],
    df28["actual_small_delta"],
    "spearman"
)

delete_corr = valid_corr(
    df28["predicted_deletion_delta"],
    df28["actual_deletion_delta"],
    "spearman"
)

# Ignore effectively zero changes when assessing sign.
small_threshold = 2e-7
deletion_threshold = 2e-7

small_reliable = (
    df28["actual_small_delta"].abs()
    > small_threshold
)
delete_reliable = (
    df28["actual_deletion_delta"].abs()
    > deletion_threshold
)

small_sign_agreement = (
    (
        np.sign(df28.loc[small_reliable, "importance"])
        == np.sign(
            df28.loc[small_reliable, "actual_small_delta"]
        )
    ).mean()
    if small_reliable.any() else np.nan
)

delete_sign_agreement = (
    (
        np.sign(df28.loc[delete_reliable, "importance"])
        == np.sign(
            df28.loc[delete_reliable, "actual_deletion_delta"]
        )
    ).mean()
    if delete_reliable.any() else np.nan
)

print("\n" + "=" * 76)
print("GRADIENT VALIDATION RESULTS")
print("=" * 76)

print(
    "Small perturbation Spearman:",
    round(float(small_corr), 4)
)
print(
    "Full deletion Spearman:",
    round(float(delete_corr), 4)
)
print(
    "Small perturbation sign agreement:",
    round(float(small_sign_agreement), 4)
)
print(
    "Full deletion sign agreement:",
    round(float(delete_sign_agreement), 4)
)
print(
    "Reliable small changes:",
    int(small_reliable.sum()), "/ 60"
)
print(
    "Reliable deletion changes:",
    int(delete_reliable.sum()), "/ 60"
)

print("\nACTUAL TRAIN LOSS CHANGE BY GROUP")
print(
    df28.groupby("group")["actual_deletion_delta"]
    .agg(["count", "mean", "median", "min", "max"])
    .to_string()
)

print("\nHistorical sign composition:")
print(
    pd.crosstab(
        df28["group"], df28["historical_sign"]
    ).to_string()
)

output_path28 = (
    RESULTS_DIR /
    "bitcoin_alpha_gradient_validation_seed42.csv"
)

df28.to_csv(output_path28, index=False)

if DEVICE.type == "cuda":
    torch.cuda.synchronize()

print(
    "\nElapsed seconds:",
    round(time.perf_counter() - started, 2)
)
print("Saved:", output_path28)
print("No retraining; no validation/test labels evaluated.")


In [ ]:

# ============================================================
# TA-GSNN | Cell 29
# Gradient-Based Connectivity-Constrained Selection
# Bitcoin Alpha | Context graph | Seed 42
# GPU inference | No retraining | Train loss only
# Requires Cell 27
# ============================================================

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from pathlib import Path

assert "weighted_predict" in globals()
assert "importance" in globals()
assert "context" in globals()
assert "target_y" in globals()
assert "class_weight" in globals()

RESULTS_DIR = Path("/kaggle/working/ta_gsnn_results")
KEEP_RATIO = 0.50

historical = context.reset_index(drop=True).copy()
n = len(historical)
budget = round(n * KEEP_RATIO)

sources = historical["source"].to_numpy()
targets = historical["target"].to_numpy()
ratings29 = historical["rating"].to_numpy()

nodes = sorted(set(sources) | set(targets))
node_index = {v: i for i, v in enumerate(nodes)}


def select_with_connectivity(scores, method_name):
    scores = np.asarray(scores, dtype=np.float64)

    assert len(scores) == n
    assert np.isfinite(scores).all()

    parent = list(range(len(nodes)))
    sizes = [1] * len(nodes)

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        a, b = find(a), find(b)

        if a == b:
            return False

        if sizes[a] < sizes[b]:
            a, b = b, a

        parent[b] = a
        sizes[a] += sizes[b]
        return True

    ranking = np.argsort(-scores, kind="stable")
    selected = np.zeros(n, dtype=bool)

    # Step 1: maximum-score spanning forest
    for i in ranking:
        u = node_index[sources[i]]
        v = node_index[targets[i]]

        if union(u, v):
            selected[i] = True

    forest_count = int(selected.sum())

    if forest_count > budget:
        raise RuntimeError(
            f"{method_name}: forest exceeds edge budget"
        )

    # Step 2: task-score-based budget completion
    remaining = ranking[~selected[ranking]]
    capacity = budget - forest_count

    selected[remaining[:capacity]] = True

    assert int(selected.sum()) == budget

    # Check preservation of historical connected components
    def components(mask):
        p = list(range(len(nodes)))

        def root(x):
            while p[x] != x:
                p[x] = p[p[x]]
                x = p[x]
            return x

        for i in np.flatnonzero(mask):
            a = root(node_index[sources[i]])
            b = root(node_index[targets[i]])
            if a != b:
                p[b] = a

        return len({root(i) for i in range(len(nodes))})

    full_components = components(np.ones(n, dtype=bool))
    sparse_components = components(selected)

    assert full_components == sparse_components

    return selected, forest_count, sparse_components


rng = np.random.default_rng(42)

score_methods = {
    "random_connectivity": rng.random(n),
    "magnitude_connectivity": np.abs(ratings29).astype(float),
    "task_aware_connectivity": np.asarray(importance)
}

selections = {
    "full": np.ones(n, dtype=bool)
}

metadata = {
    "full": (n, None)
}

for method, scores in score_methods.items():
    mask, forest_size, component_count = (
        select_with_connectivity(scores, method)
    )

    selections[method] = mask
    metadata[method] = (forest_size, component_count)

# Evaluate the frozen GNN on training targets.
# This is a diagnostic, NOT validation performance.

def fixed_train_loss(mask):
    edge_mask = torch.tensor(
        mask.astype(np.float32),
        device=DEVICE
    )

    with torch.no_grad():
        logits = weighted_predict(edge_mask)

        loss = F.binary_cross_entropy_with_logits(
            logits,
            target_y,
            pos_weight=class_weight
        )

    return float(loss.item())


records29 = []

full_loss = fixed_train_loss(selections["full"])

for method, mask in selections.items():
    loss = fixed_train_loss(mask)
    forest_size, components = metadata[method]

    records29.append({
        "method": method,
        "retained_edges": int(mask.sum()),
        "retained_pct": 100 * mask.mean(),
        "forest_edges": forest_size,
        "weak_components": components,
        "negative_pct": 100 * (ratings29[mask] < 0).mean(),
        "frozen_train_loss": loss,
        "loss_change_vs_full": loss - full_loss
    })

    if method != "full":
        selected_data = historical.iloc[
            np.flatnonzero(mask)
        ].copy()

        selected_data.to_csv(
            RESULTS_DIR /
            f"bitcoin_alpha_{method}_context_50pct.csv",
            index=False
        )

summary29 = pd.DataFrame(records29)

summary29.to_csv(
    RESULTS_DIR /
    "task_aware_selection_diagnostic_seed42.csv",
    index=False
)

print("=" * 110)
print("TA-GSNN | CELL 29 | STRUCTURE AND TRAIN LOSS")
print("=" * 110)

print(
    summary29.round(7).to_string(index=False)
)

task_mask = selections["task_aware_connectivity"]

print("\nTASK-AWARE SELECTION DETAILS")
print("Selected edges:", int(task_mask.sum()))
print(
    "Selected positive-importance edges:",
    int((task_mask & (importance > 0)).sum())
)
print(
    "Selected negative-importance edges:",
    int((task_mask & (importance < 0)).sum())
)

print("\nFiles saved.")
print("No training performed.")
print("Validation/test labels NOT used.")


In [ ]:

# ============================================================
# TA-GSNN | Cell 30
# Task-Aware Validation History Construction
# Bitcoin Alpha | Seed 42 | CUDA
# Labels used: TRAIN ONLY
# No retraining / No validation or test evaluation
# Requires Cell 27 variables
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

assert "state" in globals()
assert "node_map" in globals()
assert "target_edges" in globals()
assert "target_y" in globals()
assert "class_weight" in globals()
assert "DEVICE" in globals()

DATA_DIR = Path("/kaggle/working/ta_gsnn_data")
RESULTS_DIR = Path("/kaggle/working/ta_gsnn_results")

context30 = pd.read_csv(
    DATA_DIR / "bitcoin_alpha_context.csv"
)
train30 = pd.read_csv(
    DATA_DIR / "bitcoin_alpha_train.csv"
)

history30 = pd.concat(
    [context30, train30], ignore_index=True
)

src30 = torch.as_tensor(
    history30["source"].map(node_map).to_numpy(),
    dtype=torch.long, device=DEVICE
)
dst30 = torch.as_tensor(
    history30["target"].map(node_map).to_numpy(),
    dtype=torch.long, device=DEVICE
)

pos30 = torch.as_tensor(
    history30["rating"].to_numpy() > 0,
    dtype=torch.bool, device=DEVICE
)
neg30 = ~pos30

n30 = len(history30)
n_nodes30 = len(node_map)

print("=" * 78, flush=True)
print("TA-GSNN | TASK-AWARE VALIDATION HISTORY", flush=True)
print("Device:", DEVICE, flush=True)
print("Historical edges:", n30, flush=True)
print("Training labels:", len(target_y), flush=True)


def features30(w):
    columns = []

    for indices, weights in [
        (src30[pos30], w[pos30]),
        (src30[neg30], w[neg30]),
        (dst30[pos30], w[pos30]),
        (dst30[neg30], w[neg30])
    ]:
        col = torch.zeros(
            n_nodes30, dtype=w.dtype, device=DEVICE
        ).index_add(0, indices, weights)

        columns.append(col)

    return torch.log1p(torch.stack(columns, dim=1))


def mean30(h, s, d, w, incoming):
    receiver = d if incoming else s
    sender = s if incoming else d

    numerator = torch.zeros_like(h).index_add(
        0, receiver, h[sender] * w.unsqueeze(1)
    )

    denominator = torch.zeros(
        n_nodes30, device=DEVICE, dtype=h.dtype
    ).index_add(0, receiver, w)

    return numerator / denominator.clamp(min=1).unsqueeze(1)


def layer30(h, w, prefix):
    pieces = [h]

    for mask in (pos30, neg30):
        s, d, weight = (
            src30[mask],
            dst30[mask],
            w[mask]
        )

        pieces.append(mean30(h, s, d, weight, True))
        pieces.append(mean30(h, s, d, weight, False))

    z = F.linear(
        torch.cat(pieces, dim=1),
        state[f"{prefix}.linear.weight"],
        state[f"{prefix}.linear.bias"]
    )

    return F.relu(F.layer_norm(
        z,
        (z.shape[1],),
        state[f"{prefix}.norm.weight"],
        state[f"{prefix}.norm.bias"],
        eps=1e-5
    ))


def predict30(w):
    h = features30(w)
    h = layer30(h, w, "layer1")
    h = layer30(h, w, "layer2")

    u = h[target_edges[:, 0]]
    v = h[target_edges[:, 1]]

    pair = torch.cat([
        u, v, torch.abs(u - v), u * v
    ], dim=1)

    hidden = F.relu(F.linear(
        pair,
        state["decoder.0.weight"],
        state["decoder.0.bias"]
    ))

    return F.linear(
        hidden,
        state["decoder.3.weight"],
        state["decoder.3.bias"]
    ).squeeze(-1)


# ------------------------------------------------------------
# Task-aware importance using TRAIN labels.
# NOTE: Train edges are historical inputs at validation time.
# Therefore this is an in-sample sensitivity score.
# ------------------------------------------------------------

w30 = torch.ones(
    n30, dtype=torch.float32,
    device=DEVICE, requires_grad=True
)

logits30 = predict30(w30)

loss30 = F.binary_cross_entropy_with_logits(
    logits30, target_y, pos_weight=class_weight
)

grad30 = torch.autograd.grad(loss30, w30)[0]

importance30 = (
    -grad30.detach().cpu().numpy()
)

assert np.isfinite(importance30).all()

print("Full-history train loss:", round(loss30.item(), 6))
print("Finite gradients: PASS", flush=True)


# ------------------------------------------------------------
# Connectivity-preserving selection
# ------------------------------------------------------------

u30 = history30["source"].to_numpy()
v30 = history30["target"].to_numpy()

nodes30 = sorted(set(u30) | set(v30))
lookup30 = {node: i for i, node in enumerate(nodes30)}

parent30 = list(range(len(nodes30)))
sizes30 = [1] * len(nodes30)

def find30(x):
    while parent30[x] != x:
        parent30[x] = parent30[parent30[x]]
        x = parent30[x]
    return x

def union30(a, b):
    a, b = find30(a), find30(b)

    if a == b:
        return False

    if sizes30[a] < sizes30[b]:
        a, b = b, a

    parent30[b] = a
    sizes30[a] += sizes30[b]
    return True


ranking30 = np.argsort(
    -importance30, kind="stable"
)

selected30 = np.zeros(n30, dtype=bool)

for i in ranking30:
    if union30(
        lookup30[u30[i]],
        lookup30[v30[i]]
    ):
        selected30[i] = True

forest_edges30 = int(selected30.sum())

budget30 = round(n30 * 0.50)

if forest_edges30 > budget30:
    raise RuntimeError(
        "Spanning forest exceeds the 50% edge budget."
    )

remaining30 = ranking30[
    ~selected30[ranking30]
]

selected30[
    remaining30[:budget30 - forest_edges30]
] = True

assert int(selected30.sum()) == budget30

# The forest construction preserves the original
# weakly connected components and all historical nodes.
components30 = len({
    find30(i) for i in range(len(nodes30))
})

selected_nodes30 = (
    set(u30[selected30]) | set(v30[selected30])
)

assert selected_nodes30 == set(nodes30)

selected_history30 = history30.iloc[
    np.flatnonzero(selected30)
].copy()

output30 = (
    RESULTS_DIR /
    "bitcoin_alpha_task_aware_connectivity_"
    "validation_history_seed42_50pct.csv"
)

selected_history30.to_csv(output30, index=False)

scores30 = history30.copy()
scores30["task_importance"] = importance30
scores30["selected"] = selected30

scores30.to_csv(
    RESULTS_DIR /
    "bitcoin_alpha_task_scores_validation_history_seed42.csv",
    index=False
)

# Frozen-model train loss is ONLY a diagnostic.
with torch.no_grad():
    sparse_weights30 = torch.as_tensor(
        selected30.astype(np.float32),
        device=DEVICE
    )

    sparse_loss30 = F.binary_cross_entropy_with_logits(
        predict30(sparse_weights30),
        target_y,
        pos_weight=class_weight
    ).item()

print("\n" + "=" * 78)
print("CELL 30 | SPARSE VALIDATION HISTORY RESULTS")
print("=" * 78)

print("Historical edges:", n30)
print("Retained edges:", int(selected30.sum()))
print("Retained percentage:", round(100 * selected30.mean(), 3))
print("Forest edges:", forest_edges30)
print("Weak components:", components30)
print("Historical nodes covered:", len(selected_nodes30))
print(
    "Original negative percentage:",
    round(100 * (history30["rating"] < 0).mean(), 3)
)
print(
    "Retained negative percentage:",
    round(100 * (selected_history30["rating"] < 0).mean(), 3)
)
print("Full-history train loss:", round(loss30.item(), 6))
print("Sparse-history train loss:", round(sparse_loss30, 6))
print(
    "Train loss difference:",
    round(sparse_loss30 - loss30.item(), 6)
)

print("\nSaved:", output30)
print("No retraining. No validation/test labels evaluated.")


In [ ]:

# ============================================================
# TA-GSNN | Cell 31
# Frozen GNN Validation Evaluation
# Full vs Task-Aware + Connectivity
# Bitcoin Alpha | Seed 42 | CUDA
# No retraining | Test untouched
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)

assert "features30" in globals()
assert "layer30" in globals()
assert "history30" in globals()
assert "state" in globals()
assert "node_map" in globals()
assert "DEVICE" in globals()

DATA_DIR = Path("/kaggle/working/ta_gsnn_data")
RESULTS_DIR = Path("/kaggle/working/ta_gsnn_results")

validation31 = pd.read_csv(
    DATA_DIR / "bitcoin_alpha_validation.csv"
)

val_pairs31 = torch.as_tensor(
    np.column_stack([
        validation31["source"].map(node_map).to_numpy(),
        validation31["target"].map(node_map).to_numpy()
    ]),
    dtype=torch.long,
    device=DEVICE
)

val_y31 = (
    validation31["rating"].to_numpy() < 0
).astype(int)

# Load selection from the saved artifact.
selected_path31 = (
    RESULTS_DIR /
    "bitcoin_alpha_task_scores_validation_history_seed42.csv"
)

scores_file31 = pd.read_csv(selected_path31)

assert len(scores_file31) == len(history30)
assert np.array_equal(
    scores_file31[["source", "target"]].to_numpy(),
    history30[["source", "target"]].to_numpy()
)

mask31 = (
    scores_file31["selected"]
    .astype(bool)
    .to_numpy()
)

assert int(mask31.sum()) == 9670

# Build val prediction without referring to train targets.
def predict_validation31(weights):
    h = features30(weights)
    h = layer30(h, weights, "layer1")
    h = layer30(h, weights, "layer2")

    u = h[val_pairs31[:, 0]]
    v = h[val_pairs31[:, 1]]

    pair = torch.cat(
        [u, v, torch.abs(u - v), u * v], dim=1
    )

    hidden = F.relu(F.linear(
        pair,
        state["decoder.0.weight"],
        state["decoder.0.bias"]
    ))

    return F.linear(
        hidden,
        state["decoder.3.weight"],
        state["decoder.3.bias"]
    ).squeeze(-1)


all_weights31 = torch.ones(
    len(history30),
    dtype=torch.float32,
    device=DEVICE
)

sparse_weights31 = torch.as_tensor(
    mask31.astype(np.float32),
    device=DEVICE
)

with torch.no_grad():
    logits_full31 = predict_validation31(
        all_weights31
    )

    logits_sparse31 = predict_validation31(
        sparse_weights31
    )

    probabilities31 = {
        "full": torch.sigmoid(
            logits_full31
        ).cpu().numpy(),

        "task_aware_connectivity": torch.sigmoid(
            logits_sparse31
        ).cpu().numpy()
    }



# Split validation edges by historical node exposure.

historical_nodes31 = (
    set(history30["source"]) |
    set(history30["target"])
)

both_seen31 = (
    validation31["source"].isin(historical_nodes31)
    & validation31["target"].isin(historical_nodes31)
).to_numpy()

groups31 = {
    "overall": np.ones(len(validation31), dtype=bool),
    "both_seen": both_seen31,
    "cold_start": ~both_seen31
}


records31 = []

for method, predictions in probabilities31.items():
    for group, indices in groups31.items():
        y = val_y31[indices]
        p = predictions[indices]

        records31.append({
            "method": method,
            "group": group,
            "edges": int(indices.sum()),
            "negative_pct": 100 * float(y.mean()),
            "pr_auc": float(
                average_precision_score(y, p)
            ),
            "roc_auc": float(
                roc_auc_score(y, p)
            )
        })

results31 = pd.DataFrame(records31)

print("=" * 90)
print("TA-GSNN | CELL 31 | FROZEN VALIDATION RESULTS")
print("=" * 90)
print(results31.round(5).to_string(index=False))

overall31 = results31[
    results31["group"] == "overall"
].set_index("method")

pr_change31 = (
    overall31.loc[
        "task_aware_connectivity", "pr_auc"
    ] - overall31.loc["full", "pr_auc"]
)

roc_change31 = (
    overall31.loc[
        "task_aware_connectivity", "roc_auc"
    ] - overall31.loc["full", "roc_auc"]
)

print("\nTASK-AWARE MINUS FULL")
print("PR-AUC difference:", round(pr_change31, 6))
print("ROC-AUC difference:", round(roc_change31, 6))

# Consistency check with earlier full-model validation.
reference31 = 0.1599875976481427

full_pr31 = overall31.loc["full", "pr_auc"]

print("\nREFERENCE CHECK")
print("Full GNN PR-AUC:", round(full_pr31, 6))
print(
    "Difference from Cell 25:",
    round(abs(full_pr31 - reference31), 8)
)

if abs(full_pr31 - reference31) > 0.0005:
    raise RuntimeError(
        "Full-model validation reconstruction mismatch."
    )

print("PASS: Full-model validation matches.")
print("No retraining; test set untouched.")

results31.to_csv(
    RESULTS_DIR /
    "bitcoin_alpha_frozen_task_aware_validation_seed42.csv",
    index=False
)

print("\nResults saved.")


In [ ]:

# ============================================================
# TA-GSNN | Cell 32
# Controlled Frozen-Model Comparison | Bitcoin Alpha
# Same model, same history, same 50% budget
# Validation only | GPU | No retraining
# Requires Cell 30 and Cell 31
# ============================================================

import numpy as np
import pandas as pd
import torch
from pathlib import Path
from sklearn.metrics import average_precision_score, roc_auc_score

RESULTS_DIR = Path("/kaggle/working/ta_gsnn_results")

history32 = history30.reset_index(drop=True)
n32 = len(history32)
budget32 = round(n32 * 0.50)

src32 = history32["source"].to_numpy()
dst32 = history32["target"].to_numpy()
rating32 = history32["rating"].to_numpy()

nodes32 = sorted(set(src32) | set(dst32))
index32 = {node: i for i, node in enumerate(nodes32)}

assert not history32.duplicated(["source", "target"]).any()


def select_connected32(scores):
    ranking = np.argsort(
        -np.asarray(scores), kind="stable"
    )

    parent = np.arange(len(nodes32))
    size = np.ones(len(nodes32), dtype=int)

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        a, b = find(a), find(b)
        if a == b:
            return False
        if size[a] < size[b]:
            a, b = b, a
        parent[b] = a
        size[a] += size[b]
        return True

    selected = np.zeros(n32, dtype=bool)

    for i in ranking:
        if union(index32[src32[i]], index32[dst32[i]]):
            selected[i] = True

    forest_edges = int(selected.sum())

    if forest_edges > budget32:
        raise RuntimeError("Connectivity budget infeasible")

    remaining = ranking[~selected[ranking]]
    selected[remaining[:budget32 - forest_edges]] = True

    assert selected.sum() == budget32

    return selected


def mask_from_file32(path):
    saved = pd.read_csv(path)

    original_index = pd.MultiIndex.from_frame(
        history32[["source", "target"]]
    )
    saved_index = pd.MultiIndex.from_frame(
        saved[["source", "target"]]
    )

    assert saved_index.is_unique
    assert saved_index.isin(original_index).all()

    mask = original_index.isin(saved_index)
    assert mask.sum() == budget32

    return np.asarray(mask, dtype=bool)


rng32 = np.random.default_rng(42)

masks32 = {
    "full": np.ones(n32, dtype=bool),

    "random_connectivity":
        select_connected32(rng32.random(n32)),

    "magnitude_connectivity":
        select_connected32(np.abs(rating32)),

    "structural_connectivity":
        mask_from_file32(
            RESULTS_DIR /
            "bitcoin_alpha_connectivity_preserving_50pct.csv"
        ),

    "task_aware_connectivity":
        mask_from_file32(
            RESULTS_DIR /
            "bitcoin_alpha_task_aware_connectivity_"
            "validation_history_seed42_50pct.csv"
        )
}

# Verify all sparse methods preserve the full graph's
# weakly connected components.
def component_count32(mask):
    parent = np.arange(len(nodes32))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for i in np.flatnonzero(mask):
        a = find(index32[src32[i]])
        b = find(index32[dst32[i]])
        if a != b:
            parent[b] = a

    return len({find(i) for i in range(len(nodes32))})


reference_components32 = component_count32(masks32["full"])

records32 = []

for method, mask in masks32.items():

    components = component_count32(mask)
    assert components == reference_components32

    weights = torch.as_tensor(
        mask.astype(np.float32),
        device=DEVICE
    )

    with torch.no_grad():
        probabilities = torch.sigmoid(
            predict_validation31(weights)
        ).cpu().numpy()

    for group, group_mask in groups31.items():
        y = val_y31[group_mask]
        p = probabilities[group_mask]

        records32.append({
            "method": method,
            "group": group,
            "edges_retained": int(mask.sum()),
            "negative_edges_retained": int(
                (rating32[mask] < 0).sum()
            ),
            "negative_pct_retained": float(
                100 * (rating32[mask] < 0).mean()
            ),
            "weak_components": components,
            "pr_auc": float(
                average_precision_score(y, p)
            ),
            "roc_auc": float(
                roc_auc_score(y, p)
            )
        })

results32 = pd.DataFrame(records32)

overall32 = results32[
    results32["group"] == "overall"
].copy()

overall32 = overall32.sort_values(
    "pr_auc", ascending=False
)

# Confirm identical full and task results from Cell 31.
for method in ["full", "task_aware_connectivity"]:
    old = results31[
        (results31["method"] == method)
        & (results31["group"] == "overall")
    ]["pr_auc"].iloc[0]

    new = overall32[
        overall32["method"] == method
    ]["pr_auc"].iloc[0]

    assert abs(old - new) < 1e-6

print("=" * 105)
print("CELL 32 | CONTROLLED FROZEN VALIDATION COMPARISON")
print("=" * 105)

print(overall32.round(5).to_string(index=False))

print("\n" + "=" * 105)
print("CELL 32 | SEEN VS COLD-START PR-AUC")
print("=" * 105)

print(
    results32.pivot(
        index="method",
        columns="group",
        values="pr_auc"
    ).round(5).to_string()
)

results32.to_csv(
    RESULTS_DIR /
    "bitcoin_alpha_controlled_frozen_comparison_seed42.csv",
    index=False
)

print("\nConsistency check: PASS")
print("All methods use the same frozen GNN.")
print("No retraining. Test set untouched.")


In [ ]:

# ============================================================
# TA-GSNN | Cell 33
# Five-Seed Frozen Task-Aware Evaluation
# Bitcoin Alpha | CUDA | No retraining
# Validation only | Test untouched
#
# Requires Cells 30, 31 and 32
# ============================================================

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, roc_auc_score

required = [
    "predict30", "predict_validation31",
    "select_connected32", "mask_from_file32",
    "history30", "target_y", "class_weight",
    "val_y31", "groups31", "state", "DEVICE"
]

missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f"Missing previous definitions: {missing}")

SEEDS33 = [42, 43, 44, 45, 46]
original_state33 = state

n33 = len(history30)

structural_mask33 = mask_from_file32(
    RESULTS_DIR /
    "bitcoin_alpha_connectivity_preserving_50pct.csv"
)

full_mask33 = np.ones(n33, dtype=bool)

magnitude_mask33 = select_connected32(
    np.abs(history30["rating"].to_numpy())
)

records33 = []

print("=" * 85, flush=True)
print("TA-GSNN | FIVE-SEED FROZEN VALIDATION", flush=True)
print("Device:", DEVICE, flush=True)
print("=" * 85, flush=True)

try:
    for seed in SEEDS33:

        checkpoint = torch.load(
            RESULTS_DIR /
            f"fair_retrained_full_seed{seed}.pt",
            map_location=DEVICE,
            weights_only=True
        )

        # The global forward functions reference 'state'.
        state = {
            name: value.detach().to(DEVICE)
            for name, value in
            checkpoint["model_state_dict"].items()
        }

        # Gradient importance: training labels only.
        weights33 = torch.ones(
            n33,
            dtype=torch.float32,
            device=DEVICE,
            requires_grad=True
        )

        train_logits33 = predict30(weights33)

        loss33 = F.binary_cross_entropy_with_logits(
            train_logits33,
            target_y,
            pos_weight=class_weight
        )

        gradient33 = torch.autograd.grad(
            loss33, weights33
        )[0]

        importance33 = (
            -gradient33.detach().cpu().numpy()
        )

        if not np.isfinite(importance33).all():
            raise RuntimeError(
                f"Non-finite importance for seed {seed}"
            )

        task_mask33 = select_connected32(
            importance33
        )

        random_mask33 = select_connected32(
            np.random.default_rng(seed).random(n33)
        )

        methods33 = {
            "full": full_mask33,
            "random_connectivity": random_mask33,
            "magnitude_connectivity": magnitude_mask33,
            "structural_connectivity": structural_mask33,
            "task_aware_connectivity": task_mask33
        }

        # Save exact seed-specific task-aware choices.
        scored33 = history30.copy()
        scored33["importance"] = importance33
        scored33["selected"] = task_mask33

        scored33.to_csv(
            RESULTS_DIR /
            f"task_aware_validation_history_seed{seed}.csv",
            index=False
        )

        for method, mask in methods33.items():

            edge_weights33 = torch.as_tensor(
                mask.astype(np.float32),
                device=DEVICE
            )

            with torch.no_grad():
                probs33 = torch.sigmoid(
                    predict_validation31(edge_weights33)
                ).cpu().numpy()

            for group, group_mask in groups31.items():

                y = val_y31[group_mask]
                p = probs33[group_mask]

                records33.append({
                    "seed": seed,
                    "method": method,
                    "group": group,
                    "retained_edges": int(mask.sum()),
                    "negative_pct": float(
                        100 * (
                            history30["rating"]
                            .to_numpy()[mask] < 0
                        ).mean()
                    ),
                    "pr_auc": float(
                        average_precision_score(y, p)
                    ),
                    "roc_auc": float(
                        roc_auc_score(y, p)
                    )
                })

        seed_rows = [
            r for r in records33
            if r["seed"] == seed and r["group"] == "overall"
        ]

        print(f"\nSeed {seed}", flush=True)
        for row in seed_rows:
            print(
                f"  {row['method']:27s} "
                f"PR={row['pr_auc']:.4f} "
                f"ROC={row['roc_auc']:.4f}",
                flush=True
            )

finally:
    state = original_state33

raw33 = pd.DataFrame(records33)

summary33 = (
    raw33.groupby(["method", "group"], as_index=False)
    .agg(
        pr_auc_mean=("pr_auc", "mean"),
        pr_auc_std=("pr_auc", "std"),
        roc_auc_mean=("roc_auc", "mean"),
        roc_auc_std=("roc_auc", "std")
    )
)

overall33 = (
    summary33[summary33["group"] == "overall"]
    .sort_values("pr_auc_mean", ascending=False)
)

print("\n" + "=" * 95)
print("CELL 33 | FIVE-SEED OVERALL COMPARISON")
print("=" * 95)
print(overall33.round(5).to_string(index=False))

print("\n" + "=" * 95)
print("CELL 33 | SEEN VS COLD-START PR-AUC")
print("=" * 95)

print(
    summary33.pivot(
        index="method",
        columns="group",
        values="pr_auc_mean"
    ).round(5).to_string()
)

# Paired comparisons at matching seeds.
paired33 = raw33[
    raw33["group"] == "overall"
].pivot(index="seed", columns="method", values="pr_auc")

task33 = paired33["task_aware_connectivity"]

print("\n" + "=" * 95)
print("CELL 33 | PAIRED PR-AUC DIFFERENCES")
print("=" * 95)

for method in [
    "full",
    "random_connectivity",
    "magnitude_connectivity",
    "structural_connectivity"
]:
    differences = task33 - paired33[method]

    print(
        f"Task-Aware minus {method}: "
        f"mean={differences.mean():+.5f}, "
        f"std={differences.std():.5f}, "
        f"wins={int((differences > 0).sum())}/5"
    )

raw33.to_csv(
    RESULTS_DIR / "five_seed_frozen_task_aware_raw.csv",
    index=False
)

summary33.to_csv(
    RESULTS_DIR / "five_seed_frozen_task_aware_summary.csv",
    index=False
)

print("\nSaved five-seed results and selection files.")
print("No retraining. Test set untouched.")


In [ ]:

# ============================================================
# TA-GSNN | Cell 34
# Task-Aware GNN Retraining | Seed 42 Pilot
# CUDA | Bitcoin Alpha | Validation only
# Fixed previously selected graph structures
# ============================================================

import copy
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import average_precision_score, roc_auc_score

DATA_DIR = Path("/kaggle/working/ta_gsnn_data")
RESULTS_DIR = Path("/kaggle/working/ta_gsnn_results")

DEVICE34 = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

SEED34 = 42
MAX_EPOCHS34 = 120
PATIENCE34 = 20

assert "SignedGraphSAGE25" in globals()
assert "build_graph" in globals()
assert "target_pairs" in globals()
assert "node_to_idx" in globals()

random.seed(SEED34)
np.random.seed(SEED34)
torch.manual_seed(SEED34)

if DEVICE34.type == "cuda":
    torch.cuda.manual_seed_all(SEED34)

print("=" * 75, flush=True)
print("TA-GSNN | TASK-AWARE RETRAINING PILOT", flush=True)
print("Device:", DEVICE34, flush=True)

# ------------------------------------------------------------
# 1. Load data and FIXED selections
# ------------------------------------------------------------

train_df34 = pd.read_csv(
    DATA_DIR / "bitcoin_alpha_train.csv"
)

val_df34 = pd.read_csv(
    DATA_DIR / "bitcoin_alpha_validation.csv"
)

context_selected34 = pd.read_csv(
    RESULTS_DIR /
    "bitcoin_alpha_task_aware_connectivity_context_50pct.csv"
)

history_selected34 = pd.read_csv(
    RESULTS_DIR /
    "bitcoin_alpha_task_aware_connectivity_"
    "validation_history_seed42_50pct.csv"
)

assert len(context_selected34) == 7248
assert len(history_selected34) == 9670

def graph_to_device34(frame):
    graph = build_graph(frame)

    return {
        name: tensor.to(DEVICE34)
        for name, tensor in graph.items()
    }

train_graph34 = graph_to_device34(context_selected34)
val_graph34 = graph_to_device34(history_selected34)

train_pairs34 = torch.tensor(
    [
        [node_to_idx[s], node_to_idx[t]]
        for s, t in zip(
            train_df34["source"],
            train_df34["target"]
        )
    ],
    dtype=torch.long,
    device=DEVICE34
)

val_pairs34 = target_pairs.to(DEVICE34)

train_y34 = torch.tensor(
    (train_df34["rating"].to_numpy() < 0).astype(float),
    dtype=torch.float32,
    device=DEVICE34
)

val_y34 = (
    val_df34["rating"].to_numpy() < 0
).astype(int)

n_positive34 = float((train_y34 == 0).sum().item())
n_negative34 = float((train_y34 == 1).sum().item())

criterion34 = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        n_positive34 / n_negative34,
        device=DEVICE34
    )
)

# ------------------------------------------------------------
# 2. Initialize and train the SAME GNN architecture
# ------------------------------------------------------------

model34 = SignedGraphSAGE25().to(DEVICE34)

optimizer34 = torch.optim.Adam(
    model34.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

best_pr34 = -1.0
best_roc34 = None
best_epoch34 = 0
best_state34 = None
waiting34 = 0

records34 = []

for epoch in range(1, MAX_EPOCHS34 + 1):

    model34.train()
    optimizer34.zero_grad()

    train_logits = model34(
        train_graph34, train_pairs34
    )

    loss34 = criterion34(
        train_logits, train_y34
    )

    loss34.backward()

    torch.nn.utils.clip_grad_norm_(
        model34.parameters(), 5.0
    )

    optimizer34.step()

    model34.eval()

    with torch.no_grad():
        scores34 = torch.sigmoid(
            model34(val_graph34, val_pairs34)
        ).detach().cpu().numpy()

    pr34 = float(
        average_precision_score(val_y34, scores34)
    )

    roc34 = float(
        roc_auc_score(val_y34, scores34)
    )

    records34.append({
        "epoch": epoch,
        "train_loss": float(loss34.item()),
        "validation_pr_auc": pr34,
        "validation_roc_auc": roc34
    })

    if pr34 > best_pr34 + 1e-6:
        best_pr34 = pr34
        best_roc34 = roc34
        best_epoch34 = epoch
        best_state34 = copy.deepcopy(
            model34.state_dict()
        )
        waiting34 = 0
    else:
        waiting34 += 1

    if epoch == 1 or epoch % 10 == 0:
        print(
            f"Epoch {epoch:3d} | "
            f"Loss={loss34.item():.4f} | "
            f"PR={pr34:.4f} | "
            f"ROC={roc34:.4f}",
            flush=True
        )

    if waiting34 >= PATIENCE34:
        print(
            f"Early stopping: epoch {epoch}",
            flush=True
        )
        break

# ------------------------------------------------------------
# 3. Save reproducible artifacts
# ------------------------------------------------------------

checkpoint_path34 = (
    RESULTS_DIR /
    "task_aware_retrained_alpha_seed42.pt"
)

torch.save({
    "model_state_dict": {
        k: v.detach().cpu()
        for k, v in best_state34.items()
    },
    "method": "task_aware_connectivity",
    "seed": SEED34,
    "best_epoch": best_epoch34,
    "keep_ratio": 0.50
}, checkpoint_path34)

pd.DataFrame(records34).to_csv(
    RESULTS_DIR /
    "task_aware_retrained_alpha_seed42_log.csv",
    index=False
)

print("\n" + "=" * 75)
print("CELL 34 | TASK-AWARE RETRAINED PILOT RESULTS")
print("=" * 75)
print("Best epoch:", best_epoch34)
print("Validation PR-AUC:", round(best_pr34, 5))
print("Validation ROC-AUC:", round(best_roc34, 5))
print("Training graph edges:", len(context_selected34))
print("Validation history edges:", len(history_selected34))
print("Saved checkpoint:", checkpoint_path34)
print("Test set untouched.")


In [ ]:

# ============================================================
# TA-GSNN | Cell 35
# Five-Seed Task-Aware Retraining
# Alpha | CUDA | Validation only | Test untouched
# ============================================================

import copy, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, roc_auc_score

R35 = Path("/kaggle/working/ta_gsnn_results")
D35 = Path("/kaggle/working/ta_gsnn_data")
DEV35 = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEEDS35 = [42, 43, 44, 45, 46]

required35 = [
    "weighted_predict", "predict30", "state",
    "SignedGraphSAGE25", "build_graph", "node_to_idx",
    "context", "history30", "target_y", "class_weight"
]
missing35 = [x for x in required35 if x not in globals()]
if missing35:
    raise RuntimeError("Missing definitions: " + str(missing35))

context35 = pd.read_csv(D35 / "bitcoin_alpha_context.csv")
train35 = pd.read_csv(D35 / "bitcoin_alpha_train.csv")
val35 = pd.read_csv(D35 / "bitcoin_alpha_validation.csv")
history35 = pd.concat([context35, train35], ignore_index=True)

assert len(context35) == 14496
assert len(history35) == 19339

def connected_select35(frame, scores, ratio=0.50):
    scores = np.asarray(scores, dtype=float)
    n = len(frame)
    budget = round(n * ratio)
    assert len(scores) == n and np.isfinite(scores).all()

    u = frame["source"].to_numpy()
    v = frame["target"].to_numpy()
    nodes = sorted(set(u) | set(v))
    idx = {x: i for i, x in enumerate(nodes)}

    parent = list(range(len(nodes)))
    sizes = [1] * len(nodes)

    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    def union(a, b):
        a, b = find(a), find(b)
        if a == b:
            return False
        if sizes[a] < sizes[b]:
            a, b = b, a
        parent[b] = a
        sizes[a] += sizes[b]
        return True

    order = np.argsort(-scores, kind="stable")
    mask = np.zeros(n, dtype=bool)

    for i in order:
        if union(idx[u[i]], idx[v[i]]):
            mask[i] = True

    forest_size = int(mask.sum())
    if forest_size > budget:
        raise ValueError("Spanning forest exceeds budget")

    remaining = order[~mask[order]]
    mask[remaining[:budget - forest_size]] = True
    assert mask.sum() == budget

    return mask

def pairs35(df):
    a = df["source"].map(node_to_idx).to_numpy()
    b = df["target"].map(node_to_idx).to_numpy()
    return torch.tensor(
        np.column_stack([a, b]), dtype=torch.long, device=DEV35
    )

train_pairs35 = pairs35(train35)
val_pairs35 = pairs35(val35)

train_labels35 = torch.tensor(
    (train35["rating"].to_numpy() < 0).astype(np.float32),
    device=DEV35
)
val_labels35 = (val35["rating"].to_numpy() < 0).astype(int)

pos35 = (train_labels35 == 0).sum().item()
neg35 = (train_labels35 == 1).sum().item()

criterion35 = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(pos35 / neg35, device=DEV35)
)

def graph35(frame):
    return {
        k: v.to(DEV35)
        for k, v in build_graph(frame).items()
    }

def score_graph35(model, graph):
    model.eval()
    with torch.no_grad():
        p = torch.sigmoid(
            model(graph, val_pairs35)
        ).cpu().numpy()
    return (
        float(average_precision_score(val_labels35, p)),
        float(roc_auc_score(val_labels35, p))
    )

original_state35 = state
rows35 = []

print("=" * 78, flush=True)
print("CELL 35 | TASK-AWARE FIVE-SEED RETRAINING", flush=True)
print("Device:", DEV35, flush=True)

try:
    for seed in SEEDS35:
        output_ckpt = R35 / f"task_aware_retrained_alpha_seed{seed}.pt"

        # Each seed uses its own original full-GNN teacher.
        teacher = torch.load(
            R35 / f"fair_retrained_full_seed{seed}.pt",
            map_location=DEV35,
            weights_only=True
        )

        state = {
            k: v.detach().to(DEV35)
            for k, v in teacher["model_state_dict"].items()
        }

        # Gradients computed ONLY from training targets.
        # Context graph for train, expanded history for validation.
        w_context = torch.ones(
            len(context35), device=DEV35, requires_grad=True
        )
        loss_context = F.binary_cross_entropy_with_logits(
            weighted_predict(w_context),
            target_y,
            pos_weight=class_weight
        )
        imp_context = -torch.autograd.grad(
            loss_context, w_context
        )[0].detach().cpu().numpy()

        w_history = torch.ones(
            len(history35), device=DEV35, requires_grad=True
        )
        loss_history = F.binary_cross_entropy_with_logits(
            predict30(w_history),
            target_y,
            pos_weight=class_weight
        )
        imp_history = -torch.autograd.grad(
            loss_history, w_history
        )[0].detach().cpu().numpy()

        mask_train = connected_select35(context35, imp_context)
        mask_val = connected_select35(history35, imp_history)

        train_selected = context35.iloc[np.flatnonzero(mask_train)]
        val_selected = history35.iloc[np.flatnonzero(mask_val)]

        # Persist exactly which edges each seed receives.
        for split_name, frame, scores, mask in [
            ("context", context35, imp_context, mask_train),
            ("validation_history", history35, imp_history, mask_val)
        ]:
            scored = frame.copy()
            scored["importance"] = scores
            scored["selected"] = mask
            scored.to_csv(
                R35 / f"task_aware_seed{seed}_{split_name}_selection.csv",
                index=False
            )

        g_train = graph35(train_selected)
        g_val = graph35(val_selected)

        if seed == 42 and output_ckpt.exists():
            saved = torch.load(
                output_ckpt, map_location="cpu", weights_only=True
            )
            model = SignedGraphSAGE25().to(DEV35)
            model.load_state_dict(saved["model_state_dict"], strict=True)
            pr, roc = score_graph35(model, g_val)
            best_epoch = int(saved["best_epoch"])
            origin = "saved"

            if abs(pr - 0.24401) > 0.0005:
                raise RuntimeError(
                    "Seed 42 mismatch: check graph selections."
                )
        else:
            random.seed(seed)
            np.random.seed(seed)
            torch.manual_seed(seed)
            if DEV35.type == "cuda":
                torch.cuda.manual_seed_all(seed)

            model = SignedGraphSAGE25().to(DEV35)
            optimizer = torch.optim.Adam(
                model.parameters(), lr=0.001, weight_decay=1e-4
            )

            best_pr = -1.0
            best_epoch = 0
            best_state = None
            waiting = 0

            for epoch in range(1, 121):
                model.train()
                optimizer.zero_grad()

                logits = model(g_train, train_pairs35)
                loss = criterion35(logits, train_labels35)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(), 5.0
                )
                optimizer.step()

                current_pr, current_roc = score_graph35(
                    model, g_val
                )

                if current_pr > best_pr + 1e-6:
                    best_pr = current_pr
                    best_epoch = epoch
                    best_state = copy.deepcopy(model.state_dict())
                    waiting = 0
                else:
                    waiting += 1

                if waiting >= 20:
                    break

            model.load_state_dict(best_state, strict=True)
            pr, roc = score_graph35(model, g_val)
            origin = "trained"

            torch.save({
                "model_state_dict": {
                    k: v.detach().cpu()
                    for k, v in best_state.items()
                },
                "method": "task_aware_connectivity",
                "seed": seed,
                "best_epoch": best_epoch,
                "keep_ratio": 0.50
            }, output_ckpt)

        rows35.append({
            "method": "task_aware_connectivity",
            "seed": seed,
            "best_epoch": best_epoch,
            "pr_auc": pr,
            "roc_auc": roc,
            "source": origin
        })

        print(
            f"Seed {seed}: PR={pr:.5f} | ROC={roc:.5f} "
            f"| epoch={best_epoch} | {origin}",
            flush=True
        )

finally:
    state = original_state35

results35 = pd.DataFrame(rows35)
results35.to_csv(
    R35 / "task_aware_retrained_five_seed.csv",
    index=False
)

print("\n" + "=" * 78)
print("CELL 35 | FIVE-SEED RETRAINED TASK-AWARE RESULTS")
print("=" * 78)
print(results35.round(5).to_string(index=False))
print("\nMean PR-AUC:", round(results35.pr_auc.mean(), 5))
print("Std PR-AUC:", round(results35.pr_auc.std(), 5))
print("Mean ROC-AUC:", round(results35.roc_auc.mean(), 5))
print("Std ROC-AUC:", round(results35.roc_auc.std(), 5))
print("\nAll checkpoint and selection files saved.")
print("Test set untouched.")


In [ ]:

# ============================================================
# TA-GSNN | Cell 36
# Paired Comparison of Retrained Models
# Bitcoin Alpha | Validation only | No training
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

R = Path("/kaggle/working/ta_gsnn_results")

task = pd.read_csv(
    R / "task_aware_retrained_five_seed.csv"
)

baseline = pd.read_csv(
    R / "five_seed_baselines_raw.csv"
)

structural = pd.read_csv(
    R / "connectivity_preserving_five_seed.csv"
)

task = task.rename(columns={
    "pr_auc": "negative_pr_auc"
})

structural["method"] = "structural_connectivity"

comparison = pd.concat([
    baseline[["seed", "method", "negative_pr_auc", "roc_auc"]],
    structural[["seed", "method", "negative_pr_auc", "roc_auc"]],
    task[["seed", "method", "negative_pr_auc", "roc_auc"]]
], ignore_index=True)

assert not comparison.duplicated(
    ["seed", "method"]
).any(), "Duplicate seed-method combinations"

pivot = comparison.pivot(
    index="seed",
    columns="method",
    values="negative_pr_auc"
)

expected_seeds = [42, 43, 44, 45, 46]
assert pivot.index.tolist() == expected_seeds
assert not pivot.isna().any().any()

print("=" * 95)
print("CELL 36 | PER-SEED RETRAINED PR-AUC")
print("=" * 95)
print(pivot.round(5).to_string())

task_scores = pivot["task_aware_connectivity"]

records = []

for method in pivot.columns:
    if method == "task_aware_connectivity":
        continue

    diff = task_scores - pivot[method]

    records.append({
        "baseline": method,
        "mean_difference": diff.mean(),
        "std_difference": diff.std(ddof=1),
        "wins": int((diff > 0).sum()),
        "ties": int((diff == 0).sum()),
        "losses": int((diff < 0).sum()),
        "minimum_difference": diff.min(),
        "maximum_difference": diff.max()
    })

paired = pd.DataFrame(records).sort_values(
    "mean_difference", ascending=False
)

print("\n" + "=" * 95)
print("CELL 36 | PAIRED RETRAINED PR-AUC DIFFERENCES")
print("=" * 95)
print(paired.round(5).to_string(index=False))

summary = comparison.groupby("method", as_index=False).agg(
    pr_auc_mean=("negative_pr_auc", "mean"),
    pr_auc_std=("negative_pr_auc", "std"),
    roc_auc_mean=("roc_auc", "mean"),
    roc_auc_std=("roc_auc", "std")
)

print("\n" + "=" * 95)
print("CELL 36 | RETRAINED METHOD SUMMARY")
print("=" * 95)
print(
    summary.sort_values(
        "pr_auc_mean", ascending=False
    ).round(5).to_string(index=False)
)

comparison.to_csv(
    R / "retrained_five_seed_all_methods.csv",
    index=False
)

paired.to_csv(
    R / "retrained_five_seed_paired_differences.csv",
    index=False
)

summary.to_csv(
    R / "retrained_five_seed_final_summary.csv",
    index=False
)

print("\nSaved comparison tables.")
print("Validation analysis only. Test not accessed.")


In [ ]:

# ============================================================
# TA-GSNN | Cell 37
# Audit Historical Edge Selection
# Bitcoin Alpha | Five Seeds | No model execution
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

D37 = Path("/kaggle/working/ta_gsnn_data")
R37 = Path("/kaggle/working/ta_gsnn_results")

context37 = pd.read_csv(D37 / "bitcoin_alpha_context.csv")
train37 = pd.read_csv(D37 / "bitcoin_alpha_train.csv")

history37 = pd.concat(
    [context37, train37], ignore_index=True
)

n_context37 = len(context37)
n_train37 = len(train37)

records37 = []

for seed in [42, 43, 44, 45, 46]:

    path = R37 / (
        f"task_aware_seed{seed}_validation_history_selection.csv"
    )

    df = pd.read_csv(path)

    assert len(df) == len(history37)
    assert np.array_equal(
        df[["source", "target", "rating"]].to_numpy(),
        history37[["source", "target", "rating"]].to_numpy()
    )

    selected = df["selected"].astype(bool).to_numpy()
    importance = df["importance"].to_numpy()

    for name, sl in [
        ("context", slice(0, n_context37)),
        ("train_history", slice(n_context37, None))
    ]:
        m = selected[sl]
        scores = importance[sl]

        records37.append({
            "seed": seed,
            "edge_origin": name,
            "available_edges": len(m),
            "selected_edges": int(m.sum()),
            "selection_pct": 100 * m.mean(),
            "mean_importance": float(scores.mean()),
            "median_importance": float(np.median(scores)),
            "positive_score_pct": 100 * (scores > 0).mean()
        })

audit37 = pd.DataFrame(records37)

print("=" * 96)
print("CELL 37 | TRAIN-EDGE OVERLAP AUDIT")
print("=" * 96)
print(audit37.round(6).to_string(index=False))

print("\n" + "=" * 96)
print("CELL 37 | MEAN SELECTION RATE BY EDGE ORIGIN")
print("=" * 96)

print(
    audit37.groupby("edge_origin")["selection_pct"]
    .agg(["mean", "std", "min", "max"])
    .round(3).to_string()
)

audit37.to_csv(
    R37 / "task_aware_train_edge_overlap_audit.csv",
    index=False
)

print("\nAudit saved. No model training or evaluation.")


In [ ]:

# ============================================================
# TA-GSNN | Cell 38
# Experiment Protocol Freeze & Artifact Verification
# No training | No GPU required | No test access
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import pandas as pd
import torch

BASE = Path("/kaggle/working")
DATA = BASE / "ta_gsnn_data"
RESULTS = BASE / "ta_gsnn_results"
RESULTS.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 43, 44, 45, 46]

protocol = {
    "protocol_id": "TASP-GNN-v1.0",
    "status": "frozen_before_OTC_evaluation",
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "method": {
        "name": "Gradient-Based Task-Aware "
                "Connectivity-Preserving Sparsification",
        "working_acronym": "TASP-GNN",
        "edge_importance":
            "negative gradient of weighted training "
            "BCE loss with respect to edge weights at 1",
        "selection":
            "descending-importance maximum spanning "
            "forest followed by descending-importance fill",
        "graph_connectivity":
            "preserve weakly connected components "
            "and all historical nodes",
        "retention_ratio": 0.50,
        "edge_selection_is_learned_gate": False,
        "teacher_model_required": True
    },

    "model": {
        "architecture": "Two-layer Signed GraphSAGE",
        "input_dimension": 4,
        "hidden_dimension": 32,
        "dropout": 0.20,
        "optimizer": "Adam",
        "learning_rate": 0.001,
        "weight_decay": 0.0001,
        "gradient_clip_norm": 5.0,
        "maximum_epochs": 120,
        "early_stopping_patience": 20,
        "early_stopping_metric":
            "validation negative PR-AUC",
        "loss": "BCEWithLogitsLoss",
        "class_weight":
            "positive_train_count / negative_train_count"
    },

    "evaluation": {
        "seeds": SEEDS,
        "chronological_split":
            "60% context, 20% train, "
            "10% validation, 10% test; "
            "timestamp-safe boundaries",
        "primary_metric": "negative PR-AUC",
        "secondary_metrics": [
            "ROC-AUC",
            "weak component preservation",
            "historical node coverage",
            "retained negative edge percentage",
            "runtime",
            "peak memory"
        ],
        "baselines": [
            "full",
            "random_with_coverage",
            "magnitude_with_coverage",
            "structural_connectivity",
            "random_connectivity",
            "magnitude_connectivity"
        ],
        "dataset_development": "Bitcoin Alpha",
        "dataset_replication": "Bitcoin OTC",
        "test_access_permitted": False
    },

    "methodological_disclosures": {
        "alpha_validation_used_during_development": True,
        "teacher_selected_via_validation_early_stopping": True,
        "expanded_history_uses_training_labels_for_scoring": True,
        "expanded_history_importance_is_in_sample": True,
        "gradient_is_local_approximation": True,
        "bitcoin_networks_are_SIoT_proxies": True,
        "mininet_synthetic_experiment_is_not_SIoT_validation": True,
        "historical_global_node_mapping_used_future_node_ids": True,
        "future_node_ids_used_for_indexing_only": True,
        "otc_policy":
            "No architecture or algorithm changes "
            "based on OTC results"
    }
}

# Required reproducibility artifacts.
required_files = []

for split in ["context", "train", "validation"]:
    required_files.append(
        DATA / f"bitcoin_alpha_{split}.csv"
    )
    required_files.append(
        DATA / f"bitcoin_otc_{split}.csv"
    )

required_files.append(
    DATA / "soc-sign-bitcoinotc.csv.gz"
)

required_files.extend([
    RESULTS / "task_aware_retrained_five_seed.csv",
    RESULTS / "five_seed_baselines_raw.csv",
    RESULTS / "connectivity_preserving_five_seed.csv",
    RESULTS / "five_seed_frozen_task_aware_raw.csv",
    RESULTS / "retrained_five_seed_final_summary.csv",
    RESULTS / "task_aware_train_edge_overlap_audit.csv"
])

for seed in SEEDS:
    required_files.extend([
        RESULTS / f"fair_retrained_full_seed{seed}.pt",
        RESULTS / f"task_aware_retrained_alpha_seed{seed}.pt",
        RESULTS / f"task_aware_seed{seed}_context_selection.csv",
        RESULTS / f"task_aware_seed{seed}_validation_history_selection.csv"
    ])

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024), b""
        ):
            digest.update(chunk)
    return digest.hexdigest()

records = []

for path in required_files:
    exists = path.is_file()
    records.append({
        "file": str(path.relative_to(BASE)),
        "exists": exists,
        "size_bytes": path.stat().st_size if exists else None,
        "sha256": sha256_file(path) if exists else None
    })

manifest = pd.DataFrame(records)
missing = manifest[~manifest["exists"]]

print("=" * 85)
print("CELL 38 | EXPERIMENT PROTOCOL FREEZE")
print("=" * 85)

print("Protocol:", protocol["protocol_id"])
print("Alpha: development dataset")
print("OTC: independent replication dataset")
print("Required artifact count:", len(manifest))
print("Available artifacts:", int(manifest["exists"].sum()))
print("Missing artifacts:", len(missing))

if len(missing):
    print("\nMISSING FILES")
    print(missing["file"].to_string(index=False))
    print("\nProtocol NOT frozen: missing artifacts.")
else:
    # Save protocol and manifest only after verification.
    protocol_path = (
        RESULTS / "experiment_protocol_v1.json"
    )
    manifest_path = (
        RESULTS / "artifact_manifest_v1.csv"
    )

    with open(protocol_path, "w", encoding="utf-8") as f:
        json.dump(
            protocol, f, indent=2, ensure_ascii=False
        )

    manifest.to_csv(manifest_path, index=False)

    # Verify five-seed completeness without rerunning models.
    task = pd.read_csv(
        RESULTS / "task_aware_retrained_five_seed.csv"
    )

    assert sorted(task["seed"].tolist()) == SEEDS
    assert task["pr_auc"].notna().all()
    assert task["roc_auc"].notna().all()

    print("\nFive-seed checkpoint verification: PASS")
    print("Artifact integrity manifest: SAVED")
    print("Protocol manifest: SAVED")
    print("Protocol status: FROZEN")

    print("\nPROTOCOL OUTPUTS")
    print(protocol_path)
    print(manifest_path)

print("\nNo model training performed.")
print("No test files accessed.")


In [ ]:

# ============================================================
# TASP-GNN | Cell 39
# Bitcoin OTC - Graph Preparation and Integrity Audit
#
# Protocol: TASP-GNN-v1.0
# No training | No test access
# ============================================================

from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd
import torch

DATA39 = Path("/kaggle/working/ta_gsnn_data")
RESULTS39 = Path("/kaggle/working/ta_gsnn_results")

with open(
    RESULTS39 / "experiment_protocol_v1.json",
    encoding="utf-8"
) as f:
    protocol39 = json.load(f)

assert protocol39["protocol_id"] == "TASP-GNN-v1.0"
assert protocol39["method"]["retention_ratio"] == 0.50

DEVICE39 = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

context39 = pd.read_csv(
    DATA39 / "bitcoin_otc_context.csv"
)
train39 = pd.read_csv(
    DATA39 / "bitcoin_otc_train.csv"
)
validation39 = pd.read_csv(
    DATA39 / "bitcoin_otc_validation.csv"
)

history39 = pd.concat(
    [context39, train39],
    ignore_index=True
)

# Node indexing uses historical and validation endpoint IDs
# only; no validation labels are used to build features.
# Unknown future nodes will have zero initial features.
known_ids39 = sorted(
    set(history39["source"])
    | set(history39["target"])
)

node_to_idx39 = {
    node: i for i, node in enumerate(known_ids39)
}

N39 = len(known_ids39)

# Extend indexing for validation endpoints, but never add
# validation edges to the message-passing graph.
validation_ids39 = sorted(
    (set(validation39["source"])
     | set(validation39["target"]))
    - set(known_ids39)
)

for node in validation_ids39:
    node_to_idx39[node] = len(node_to_idx39)

N_TOTAL39 = len(node_to_idx39)


def graph_tensors39(frame):
    source = frame["source"].map(node_to_idx39).to_numpy()
    target = frame["target"].map(node_to_idx39).to_numpy()

    assert not pd.isna(source).any()
    assert not pd.isna(target).any()

    source = source.astype(np.int64)
    target = target.astype(np.int64)

    is_positive = frame["rating"].to_numpy() > 0

    features = np.zeros(
        (N_TOTAL39, 4),
        dtype=np.float32
    )

    np.add.at(
        features[:, 0], source[is_positive], 1
    )
    np.add.at(
        features[:, 1], source[~is_positive], 1
    )
    np.add.at(
        features[:, 2], target[is_positive], 1
    )
    np.add.at(
        features[:, 3], target[~is_positive], 1
    )

    return {
        "x": torch.tensor(
            np.log1p(features),
            dtype=torch.float32,
            device=DEVICE39
        ),
        "pos": torch.tensor(
            np.stack([
                source[is_positive],
                target[is_positive]
            ]),
            dtype=torch.long,
            device=DEVICE39
        ),
        "neg": torch.tensor(
            np.stack([
                source[~is_positive],
                target[~is_positive]
            ]),
            dtype=torch.long,
            device=DEVICE39
        )
    }


# The same graph representation as Alpha, but with dynamic
# node dimensions rather than a hard-coded node count.
graph_context39 = graph_tensors39(context39)
graph_validation39 = graph_tensors39(history39)


def forest_statistics39(frame):
    source = frame["source"].to_numpy()
    target = frame["target"].to_numpy()

    nodes = sorted(set(source) | set(target))
    lookup = {node: i for i, node in enumerate(nodes)}

    parent = list(range(len(nodes)))
    sizes = [1] * len(nodes)

    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    def union(a, b):
        a, b = find(a), find(b)

        if a == b:
            return False

        if sizes[a] < sizes[b]:
            a, b = b, a

        parent[b] = a
        sizes[a] += sizes[b]
        return True

    forest_edges = 0

    for u, v in zip(source, target):
        if union(lookup[u], lookup[v]):
            forest_edges += 1

    components = len({
        find(i) for i in range(len(nodes))
    })

    budget = round(len(frame) * 0.50)

    return {
        "edges": len(frame),
        "historical_nodes": len(nodes),
        "weak_components": components,
        "minimum_forest_edges": forest_edges,
        "budget_50pct": budget,
        "forest_feasible": forest_edges <= budget,
        "negative_pct": 100 * (
            frame["rating"] < 0
        ).mean()
    }


summary39 = pd.DataFrame([
    {
        "graph": "training_context",
        **forest_statistics39(context39)
    },
    {
        "graph": "validation_history",
        **forest_statistics39(history39)
    }
])

# Check chronological separation.
assert context39["timestamp"].max() < train39["timestamp"].min()
assert train39["timestamp"].max() < validation39["timestamp"].min()

assert summary39["forest_feasible"].all()
assert graph_context39["x"].shape == (
    N_TOTAL39, 4
)
assert graph_validation39["x"].shape == (
    N_TOTAL39, 4
)

# Validate that the test split was not consulted.
print("=" * 100)
print("CELL 39 | BITCOIN OTC GRAPH READINESS")
print("=" * 100)

print("Protocol:", protocol39["protocol_id"])
print("Device:", DEVICE39)
print("Historical nodes:", N39)
print("Validation-only new nodes:", len(validation_ids39))
print("Indexed nodes:", N_TOTAL39)

print("\nGRAPH STRUCTURE:")
print(summary39.round(3).to_string(index=False))

print("\nGraph feature shapes:")
print(
    "Training graph:",
    tuple(graph_context39["x"].shape)
)
print(
    "Validation-history graph:",
    tuple(graph_validation39["x"].shape)
)

print("\nChronological integrity: PASS")
print("50% connectivity feasibility: PASS")
print("No model training performed.")
print("No test data accessed.")

summary39.to_csv(
    RESULTS39 / "otc_graph_readiness_v1.csv",
    index=False
)


In [ ]:

# ============================================================
# TASP-GNN | Cell 40
# Bitcoin OTC | Full Signed GNN Teacher Training
# 5 seeds | CUDA | Frozen protocol v1.0
# Validation early stopping | TEST UNTOUCHED
# Requires Cell 39
# ============================================================

import copy
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)

required40 = [
    "context39", "train39", "validation39",
    "graph_context39", "graph_validation39",
    "node_to_idx39", "DEVICE39", "RESULTS39"
]

missing40 = [x for x in required40 if x not in globals()]
if missing40:
    raise RuntimeError(
        f"Run Cell 39 first. Missing: {missing40}"
    )

DEVICE40 = DEVICE39
SEEDS40 = [42, 43, 44, 45, 46]

print("=" * 78, flush=True)
print("CELL 40 | OTC FULL GNN TEACHER TRAINING", flush=True)
print("Device:", DEVICE40, flush=True)
print("=" * 78, flush=True)

# ------------------------------------------------------------
# 1. Prepare target edges and labels
# ------------------------------------------------------------

def target_tensors40(frame):
    source = frame["source"].map(node_to_idx39).to_numpy()
    target = frame["target"].map(node_to_idx39).to_numpy()

    assert pd.notna(source).all()
    assert pd.notna(target).all()

    edges = torch.as_tensor(
        np.column_stack([source, target]).astype(np.int64),
        dtype=torch.long,
        device=DEVICE40
    )

    labels = torch.as_tensor(
        (frame["rating"].to_numpy() < 0).astype(np.float32),
        dtype=torch.float32,
        device=DEVICE40
    )

    return edges, labels


train_pairs40, train_y40 = target_tensors40(train39)
val_pairs40, val_y40 = target_tensors40(validation39)

val_labels_np40 = val_y40.cpu().numpy().astype(int)

n_pos40 = float((train_y40 == 0).sum().item())
n_neg40 = float((train_y40 == 1).sum().item())

criterion40 = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        n_pos40 / n_neg40,
        dtype=torch.float32,
        device=DEVICE40
    )
)

assert n_neg40 > 0
assert len(np.unique(val_labels_np40)) == 2

# ------------------------------------------------------------
# 2. Dynamic Signed GraphSAGE architecture
# ------------------------------------------------------------

class SignedLayer40(nn.Module):

    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim * 5, out_dim)
        self.norm = nn.LayerNorm(out_dim)

    @staticmethod
    def aggregate(h, edges, incoming):
        src, dst = edges[0], edges[1]

        receiver = dst if incoming else src
        sender = src if incoming else dst

        n = h.shape[0]

        aggregated = torch.zeros_like(h)
        aggregated.index_add_(0, receiver, h[sender])

        counts = torch.bincount(
            receiver, minlength=n
        ).to(h.dtype).clamp(min=1)

        return aggregated / counts.unsqueeze(1)

    def forward(self, h, graph):

        pieces = [h]

        for sign in ("pos", "neg"):
            edges = graph[sign]

            pieces.append(
                self.aggregate(h, edges, True)
            )
            pieces.append(
                self.aggregate(h, edges, False)
            )

        return self.norm(
            self.linear(torch.cat(pieces, dim=1))
        )


class SignedGraphSAGE40(nn.Module):

    def __init__(self):
        super().__init__()

        self.layer1 = SignedLayer40(4, 32)
        self.layer2 = SignedLayer40(32, 32)
        self.dropout = 0.20

        self.decoder = nn.Sequential(
            nn.Linear(128, 32),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(32, 1)
        )

    def forward(self, graph, edge_pairs):

        h = F.relu(
            self.layer1(graph["x"], graph)
        )
        h = F.dropout(
            h, p=self.dropout, training=self.training
        )

        h = F.relu(
            self.layer2(h, graph)
        )
        h = F.dropout(
            h, p=self.dropout, training=self.training
        )

        u = h[edge_pairs[:, 0]]
        v = h[edge_pairs[:, 1]]

        pair = torch.cat(
            [u, v, torch.abs(u-v), u*v],
            dim=1
        )

        return self.decoder(pair).squeeze(-1)


# ------------------------------------------------------------
# 3. Train and save each teacher
# ------------------------------------------------------------

def evaluate40(model):
    model.eval()

    with torch.no_grad():
        probabilities = torch.sigmoid(
            model(
                graph_validation39,
                val_pairs40
            )
        ).cpu().numpy()

    return (
        float(average_precision_score(
            val_labels_np40, probabilities
        )),
        float(roc_auc_score(
            val_labels_np40, probabilities
        ))
    )


rows40 = []
start40 = time.perf_counter()

for seed in SEEDS40:

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if DEVICE40.type == "cuda":
        torch.cuda.manual_seed_all(seed)

    model = SignedGraphSAGE40().to(DEVICE40)

    # Verify the same parameter count as Alpha.
    parameters = sum(
        p.numel() for p in model.parameters()
    )
    assert parameters == 10113, (
        f"Unexpected parameter count: {parameters}"
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.001,
        weight_decay=1e-4
    )

    best_pr = -1.0
    best_roc = None
    best_epoch = None
    best_state = None
    waiting = 0

    print(f"\nTraining seed {seed}...", flush=True)

    for epoch in range(1, 121):

        model.train()
        optimizer.zero_grad()

        train_logits = model(
            graph_context39, train_pairs40
        )

        loss = criterion40(
            train_logits, train_y40
        )

        loss.backward()

        nn.utils.clip_grad_norm_(
            model.parameters(), max_norm=5.0
        )

        optimizer.step()

        pr, roc = evaluate40(model)

        if pr > best_pr + 1e-6:
            best_pr = pr
            best_roc = roc
            best_epoch = epoch
            best_state = copy.deepcopy(
                model.state_dict()
            )
            waiting = 0
        else:
            waiting += 1

        if waiting >= 20:
            break

    # Explicitly verify the saved best weights.
    model.load_state_dict(best_state, strict=True)
    final_pr, final_roc = evaluate40(model)

    assert abs(final_pr - best_pr) < 1e-6

    checkpoint_path = (
        RESULTS39 /
        f"otc_full_teacher_seed{seed}.pt"
    )

    torch.save(
        {
            "model_state_dict": {
                k: v.detach().cpu()
                for k, v in best_state.items()
            },
            "method": "full",
            "dataset": "bitcoin_otc",
            "seed": seed,
            "best_epoch": best_epoch,
            "protocol_id": "TASP-GNN-v1.0"
        },
        checkpoint_path
    )

    rows40.append({
        "seed": seed,
        "best_epoch": best_epoch,
        "validation_pr_auc": final_pr,
        "validation_roc_auc": final_roc,
        "train_edges": len(context39),
        "validation_history_edges": len(history39),
        "parameter_count": parameters
    })

    print(
        f"Seed {seed}: "
        f"PR-AUC={final_pr:.5f} | "
        f"ROC-AUC={final_roc:.5f} | "
        f"Best epoch={best_epoch}",
        flush=True
    )


# ------------------------------------------------------------
# 4. Final summary and export
# ------------------------------------------------------------

results40 = pd.DataFrame(rows40)

results40.to_csv(
    RESULTS39 / "otc_full_teacher_five_seed.csv",
    index=False
)

if DEVICE40.type == "cuda":
    torch.cuda.synchronize()

print("\n" + "=" * 88)
print("CELL 40 | OTC FULL TEACHER FIVE-SEED RESULTS")
print("=" * 88)

print(
    results40.round(5).to_string(index=False)
)

print("\nMean PR-AUC:",
      round(results40["validation_pr_auc"].mean(), 5))

print("Std PR-AUC:",
      round(results40["validation_pr_auc"].std(), 5))

print("Mean ROC-AUC:",
      round(results40["validation_roc_auc"].mean(), 5))

print("Std ROC-AUC:",
      round(results40["validation_roc_auc"].std(), 5))

print(
    "Elapsed seconds:",
    round(time.perf_counter() - start40, 2)
)

print("\nTeacher checkpoints saved.")
print("Protocol unchanged.")
print("Test split NOT accessed.")


In [ ]:

# ============================================================
# TASP-GNN | Cell 41
# OTC Task-Aware Gradient Scoring + Connectivity Selection
# Five seeds | CUDA | Protocol v1.0
# No retraining | No validation/test scoring
# Requires Cells 39 and 40
# ============================================================

import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

required41 = [
    "context39", "history39", "node_to_idx39",
    "DEVICE40", "RESULTS39", "train_pairs40",
    "train_y40", "criterion40", "SignedGraphSAGE40",
    "graph_context39", "graph_validation39"
]
missing41 = [x for x in required41 if x not in globals()]
if missing41:
    raise RuntimeError(f"Missing previous cells: {missing41}")

SEEDS41 = [42, 43, 44, 45, 46]
DEVICE41 = DEVICE40
N41 = len(node_to_idx39)

def prepare_edges41(frame):
    s = torch.tensor(
        frame["source"].map(node_to_idx39).to_numpy(),
        dtype=torch.long, device=DEVICE41
    )
    d = torch.tensor(
        frame["target"].map(node_to_idx39).to_numpy(),
        dtype=torch.long, device=DEVICE41
    )
    positive = torch.tensor(
        (frame["rating"].to_numpy() > 0),
        dtype=torch.bool, device=DEVICE41
    )
    return s, d, positive


def weighted_forward41(frame_data, weights, state):
    s, d, positive = frame_data
    negative = ~positive

    # Differentiable degree features.
    columns = []
    for indices, w in [
        (s[positive], weights[positive]),
        (s[negative], weights[negative]),
        (d[positive], weights[positive]),
        (d[negative], weights[negative])
    ]:
        col = torch.zeros(
            N41, dtype=weights.dtype, device=DEVICE41
        ).index_add(0, indices, w)
        columns.append(col)

    h = torch.log1p(torch.stack(columns, dim=1))

    # Differentiable weighted signed aggregation.
    for layer in ("layer1", "layer2"):
        pieces = [h]

        for sign_mask in (positive, negative):
            ss, dd = s[sign_mask], d[sign_mask]
            ww = weights[sign_mask]

            for incoming in (True, False):
                receiver = dd if incoming else ss
                sender = ss if incoming else dd

                numerator = torch.zeros_like(h).index_add(
                    0, receiver,
                    h[sender] * ww.unsqueeze(1)
                )
                denominator = torch.zeros(
                    N41, dtype=h.dtype, device=DEVICE41
                ).index_add(0, receiver, ww)

                pieces.append(
                    numerator /
                    denominator.clamp(min=1).unsqueeze(1)
                )

        z = F.linear(
            torch.cat(pieces, dim=1),
            state[f"{layer}.linear.weight"],
            state[f"{layer}.linear.bias"]
        )
        h = F.relu(F.layer_norm(
            z, (z.shape[1],),
            state[f"{layer}.norm.weight"],
            state[f"{layer}.norm.bias"],
            eps=1e-5
        ))

    u = h[train_pairs40[:, 0]]
    v = h[train_pairs40[:, 1]]
    pair = torch.cat(
        [u, v, torch.abs(u - v), u * v],
        dim=1
    )

    hidden = F.relu(F.linear(
        pair,
        state["decoder.0.weight"],
        state["decoder.0.bias"]
    ))

    return F.linear(
        hidden,
        state["decoder.3.weight"],
        state["decoder.3.bias"]
    ).squeeze(-1)


def connected_selection41(frame, importance):
    n = len(frame)
    budget = round(n * 0.50)

    source = frame["source"].to_numpy()
    target = frame["target"].to_numpy()

    nodes = sorted(set(source) | set(target))
    lookup = {node: i for i, node in enumerate(nodes)}

    parent = np.arange(len(nodes))
    sizes = np.ones(len(nodes), dtype=np.int64)

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        a, b = find(a), find(b)
        if a == b:
            return False
        if sizes[a] < sizes[b]:
            a, b = b, a
        parent[b] = a
        sizes[a] += sizes[b]
        return True

    ranking = np.argsort(-importance, kind="stable")
    selected = np.zeros(n, dtype=bool)

    for i in ranking:
        if union(lookup[source[i]], lookup[target[i]]):
            selected[i] = True

    forest_edges = int(selected.sum())
    components = len({
        find(i) for i in range(len(nodes))
    })

    if forest_edges > budget:
        raise RuntimeError("Spanning forest exceeds budget")

    remaining = ranking[~selected[ranking]]
    selected[remaining[:budget - forest_edges]] = True

    assert selected.sum() == budget

    return selected, forest_edges, components, len(nodes)


context_data41 = prepare_edges41(context39)
history_data41 = prepare_edges41(history39)

rows41 = []
start41 = time.perf_counter()

print("=" * 95, flush=True)
print("CELL 41 | OTC TASK-AWARE STRUCTURE SELECTION", flush=True)
print("Device:", DEVICE41, flush=True)
print("=" * 95, flush=True)

for seed in SEEDS41:
    checkpoint = torch.load(
        RESULTS39 / f"otc_full_teacher_seed{seed}.pt",
        map_location=DEVICE41,
        weights_only=True
    )
    state41 = {
        k: v.detach()
        for k, v in checkpoint["model_state_dict"].items()
    }

    teacher41 = SignedGraphSAGE40().to(DEVICE41)
    teacher41.load_state_dict(state41, strict=True)
    teacher41.eval()

    for phase, frame, prepared, original_graph in [
        ("context", context39, context_data41, graph_context39),
        ("validation_history", history39,
         history_data41, graph_validation39)
    ]:
        weights = torch.ones(
            len(frame), device=DEVICE41,
            dtype=torch.float32, requires_grad=True
        )

        # Verify exactly the same predictions at weights=1.
        with torch.no_grad():
            original = teacher41(original_graph, train_pairs40)
            reconstructed = weighted_forward41(
                prepared, weights.detach(), state41
            )
            max_diff = (
                original - reconstructed
            ).abs().max().item()

        if max_diff > 1e-4:
            raise RuntimeError(
                f"Reconstruction failed: seed={seed}, "
                f"phase={phase}, diff={max_diff}"
            )

        logits = weighted_forward41(prepared, weights, state41)
        loss = criterion40(logits, train_y40)

        gradient = torch.autograd.grad(
            loss, weights
        )[0].detach().cpu().numpy()

        importance = -gradient
        assert np.isfinite(importance).all()

        mask, forest_size, components, nodes = (
            connected_selection41(frame, importance)
        )

        # Preserve all historical nodes and weak components.
        selected_frame = frame.iloc[
            np.flatnonzero(mask)
        ].copy()

        assert (
            set(selected_frame["source"])
            | set(selected_frame["target"])
        ) == (set(frame["source"]) | set(frame["target"]))

        scored = frame.copy()
        scored["importance"] = importance
        scored["selected"] = mask

        scored.to_csv(
            RESULTS39 /
            f"otc_task_aware_seed{seed}_{phase}_selection.csv",
            index=False
        )

        selected_frame.to_csv(
            RESULTS39 /
            f"otc_task_aware_seed{seed}_{phase}_50pct.csv",
            index=False
        )

        rows41.append({
            "seed": seed,
            "phase": phase,
            "original_edges": len(frame),
            "retained_edges": int(mask.sum()),
            "forest_edges": forest_size,
            "weak_components": components,
            "historical_nodes": nodes,
            "negative_pct_before": 100 * (
                frame["rating"] < 0
            ).mean(),
            "negative_pct_after": 100 * (
                selected_frame["rating"] < 0
            ).mean(),
            "reconstruction_max_diff": max_diff
        })

        print(
            f"Seed {seed} | {phase:19s} | "
            f"retained={mask.sum():5d} | "
            f"components={components} | "
            f"check={max_diff:.2e}",
            flush=True
        )

summary41 = pd.DataFrame(rows41)
summary41.to_csv(
    RESULTS39 / "otc_task_aware_selection_summary.csv",
    index=False
)

if DEVICE41.type == "cuda":
    torch.cuda.synchronize()

print("\n" + "=" * 105)
print("CELL 41 | OTC TASK-AWARE SELECTION SUMMARY")
print("=" * 105)
print(summary41.round(5).to_string(index=False))

print("\nElapsed seconds:",
      round(time.perf_counter() - start41, 2))
print("All selections saved.")
print("No retraining or validation/test scoring.")


In [ ]:

# ============================================================
# TASP-GNN | Cell 42
# OTC Task-Aware GNN Retraining | Five Seeds
# Protocol v1.0 | CUDA | Validation only
# Uses frozen selections from Cell 41
# Requires Cells 39 and 40
# ============================================================

import copy
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import average_precision_score, roc_auc_score

required42 = [
    "RESULTS39", "DEVICE40", "graph_tensors39",
    "SignedGraphSAGE40", "train_pairs40", "train_y40",
    "val_pairs40", "val_labels_np40", "criterion40"
]

missing42 = [x for x in required42 if x not in globals()]
if missing42:
    raise RuntimeError(
        f"Missing definitions: {missing42}. "
        "Run Cells 39 and 40 first."
    )

SEEDS42 = [42, 43, 44, 45, 46]
DEVICE42 = DEVICE40
MAX_EPOCHS42 = 120
PATIENCE42 = 20

rows42 = []
started42 = time.perf_counter()

print("=" * 85, flush=True)
print("CELL 42 | OTC TASK-AWARE FIVE-SEED RETRAINING", flush=True)
print("Device:", DEVICE42, flush=True)
print("=" * 85, flush=True)


def evaluate42(model, graph):
    model.eval()

    with torch.no_grad():
        predictions = torch.sigmoid(
            model(graph, val_pairs40)
        ).cpu().numpy()

    return (
        float(average_precision_score(
            val_labels_np40, predictions
        )),
        float(roc_auc_score(
            val_labels_np40, predictions
        ))
    )


for seed in SEEDS42:

    # --------------------------------------------------------
    # 1. Load frozen graph selections
    # --------------------------------------------------------

    train_path = RESULTS39 / (
        f"otc_task_aware_seed{seed}_context_50pct.csv"
    )

    validation_path = RESULTS39 / (
        f"otc_task_aware_seed{seed}_validation_history_50pct.csv"
    )

    train_selected = pd.read_csv(train_path)
    validation_selected = pd.read_csv(validation_path)

    assert len(train_selected) == 10678
    assert len(validation_selected) == 14237

    train_graph = graph_tensors39(train_selected)
    validation_graph = graph_tensors39(validation_selected)

    # --------------------------------------------------------
    # 2. Same training configuration as Full GNN
    # --------------------------------------------------------

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if DEVICE42.type == "cuda":
        torch.cuda.manual_seed_all(seed)

    model = SignedGraphSAGE40().to(DEVICE42)

    assert sum(
        p.numel() for p in model.parameters()
    ) == 10113

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.001,
        weight_decay=1e-4
    )

    best_pr = -1.0
    best_roc = None
    best_epoch = None
    best_state = None
    waiting = 0

    for epoch in range(1, MAX_EPOCHS42 + 1):

        model.train()
        optimizer.zero_grad()

        logits = model(train_graph, train_pairs40)
        loss = criterion40(logits, train_y40)

        loss.backward()

        nn.utils.clip_grad_norm_(
            model.parameters(), max_norm=5.0
        )

        optimizer.step()

        pr, roc = evaluate42(model, validation_graph)

        if pr > best_pr + 1e-6:
            best_pr = pr
            best_roc = roc
            best_epoch = epoch
            best_state = copy.deepcopy(
                model.state_dict()
            )
            waiting = 0
        else:
            waiting += 1

        if waiting >= PATIENCE42:
            break

    # --------------------------------------------------------
    # 3. Verify and save best checkpoint
    # --------------------------------------------------------

    model.load_state_dict(best_state, strict=True)

    final_pr, final_roc = evaluate42(
        model, validation_graph
    )

    assert abs(final_pr - best_pr) < 1e-6

    output_checkpoint = RESULTS39 / (
        f"otc_task_aware_retrained_seed{seed}.pt"
    )

    torch.save(
        {
            "model_state_dict": {
                k: v.detach().cpu()
                for k, v in best_state.items()
            },
            "method": "task_aware_connectivity",
            "dataset": "bitcoin_otc",
            "seed": seed,
            "best_epoch": best_epoch,
            "keep_ratio": 0.50,
            "protocol_id": "TASP-GNN-v1.0",
            "teacher_checkpoint": (
                f"otc_full_teacher_seed{seed}.pt"
            )
        },
        output_checkpoint
    )

    rows42.append({
        "seed": seed,
        "best_epoch": best_epoch,
        "validation_pr_auc": final_pr,
        "validation_roc_auc": final_roc,
        "train_edges": len(train_selected),
        "validation_history_edges": len(validation_selected)
    })

    print(
        f"Seed {seed}: "
        f"PR-AUC={final_pr:.5f} | "
        f"ROC-AUC={final_roc:.5f} | "
        f"Best epoch={best_epoch}",
        flush=True
    )


# ------------------------------------------------------------
# 4. Summarize and compare with Full teachers
# ------------------------------------------------------------

task_results42 = pd.DataFrame(rows42)

task_results42.to_csv(
    RESULTS39 / "otc_task_aware_retrained_five_seed.csv",
    index=False
)

full_results42 = pd.read_csv(
    RESULTS39 / "otc_full_teacher_five_seed.csv"
)

paired42 = task_results42.merge(
    full_results42[
        ["seed", "validation_pr_auc", "validation_roc_auc"]
    ],
    on="seed",
    suffixes=("_task", "_full"),
    validate="one_to_one"
)

paired42["pr_auc_difference"] = (
    paired42["validation_pr_auc_task"]
    - paired42["validation_pr_auc_full"]
)

paired42["roc_auc_difference"] = (
    paired42["validation_roc_auc_task"]
    - paired42["validation_roc_auc_full"]
)

if DEVICE42.type == "cuda":
    torch.cuda.synchronize()

print("\n" + "=" * 90)
print("CELL 42 | OTC TASK-AWARE RETRAINED RESULTS")
print("=" * 90)

print(
    task_results42.round(5).to_string(index=False)
)

print("\n" + "=" * 90)
print("CELL 42 | PAIRED TASK-AWARE VS FULL")
print("=" * 90)

print(
    paired42[
        [
            "seed",
            "validation_pr_auc_full",
            "validation_pr_auc_task",
            "pr_auc_difference",
            "roc_auc_difference"
        ]
    ].round(5).to_string(index=False)
)

print("\nMean Task-Aware PR-AUC:",
      round(task_results42.validation_pr_auc.mean(), 5))

print("Std Task-Aware PR-AUC:",
      round(task_results42.validation_pr_auc.std(), 5))

print("Mean Full PR-AUC:",
      round(full_results42.validation_pr_auc.mean(), 5))

print("Mean paired PR-AUC difference:",
      round(paired42.pr_auc_difference.mean(), 5))

print("Wins over Full:",
      int((paired42.pr_auc_difference > 0).sum()), "/ 5")

print("Elapsed seconds:",
      round(time.perf_counter() - started42, 2))

paired42.to_csv(
    RESULTS39 / "otc_task_aware_vs_full_paired.csv",
    index=False
)

print("\nAll results and checkpoints saved.")
print("Protocol unchanged. Test split untouched.")


In [ ]:

# ============================================================
# TASP-GNN | Cell 43
# OTC Connectivity-Constrained Baselines
# Random / Magnitude / Structural
# 5 seeds | CUDA | Frozen protocol v1.0
# Validation only | TEST UNTOUCHED
# Requires Cells 39 and 40
# ============================================================

import copy
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import average_precision_score, roc_auc_score

R43 = Path("/kaggle/working/ta_gsnn_results")
SEEDS43 = [42, 43, 44, 45, 46]
METHODS43 = [
    "random_connectivity",
    "magnitude_connectivity",
    "structural_connectivity"
]
DEV43 = DEVICE40

# ------------------------------------------------------------
# 1. Selection function: same 50% connectivity constraint
# ------------------------------------------------------------

def select43(frame, method, seed):
    df = frame.reset_index(drop=True)
    n = len(df)
    budget = round(n * 0.50)

    u = df["source"].to_numpy()
    v = df["target"].to_numpy()
    ratings = df["rating"].to_numpy()

    nodes = sorted(set(u) | set(v))
    idx = {node: i for i, node in enumerate(nodes)}

    if method == "random_connectivity":
        scores = np.random.default_rng(seed).random(n)

    elif method == "magnitude_connectivity":
        scores = np.abs(ratings).astype(float)

    elif method == "structural_connectivity":
        degree = np.zeros(len(nodes), dtype=np.int64)

        for a, b in zip(u, v):
            degree[idx[a]] += 1
            degree[idx[b]] += 1

        structural = np.array([
            1 / np.sqrt(degree[idx[a]]) +
            1 / np.sqrt(degree[idx[b]])
            for a, b in zip(u, v)
        ])

        scores = np.abs(ratings) / 10.0 + 0.5 * structural

    else:
        raise ValueError(method)

    ranking = np.argsort(-scores, kind="stable")

    parent = list(range(len(nodes)))
    size = [1] * len(nodes)

    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    def union(a, b):
        a, b = find(a), find(b)
        if a == b:
            return False
        if size[a] < size[b]:
            a, b = b, a
        parent[b] = a
        size[a] += size[b]
        return True

    selected = np.zeros(n, dtype=bool)

    # Maximum-score spanning forest
    for i in ranking:
        if union(idx[u[i]], idx[v[i]]):
            selected[i] = True

    forest_count = int(selected.sum())

    if forest_count > budget:
        raise RuntimeError("Forest exceeds 50% edge budget")

    # Structural baseline reproduces Cell 24:
    # approximate historical negative-edge preservation,
    # followed by score-ranked fill.
    if method == "structural_connectivity":
        negative = ratings < 0
        negative_target = round(budget * negative.mean())

        needed = max(
            0,
            negative_target - int((selected & negative).sum())
        )

        capacity = budget - int(selected.sum())
        neg_candidates = ranking[
            negative[ranking] & ~selected[ranking]
        ]

        chosen = neg_candidates[:min(needed, capacity)]
        selected[chosen] = True

    capacity = budget - int(selected.sum())
    remaining = ranking[~selected[ranking]]
    selected[remaining[:capacity]] = True

    assert int(selected.sum()) == budget

    selected_df = df.iloc[np.flatnonzero(selected)].copy()

    assert (
        set(selected_df["source"]) |
        set(selected_df["target"])
    ) == set(nodes)

    components = len({find(i) for i in range(len(nodes))})

    return selected_df, forest_count, components


# ------------------------------------------------------------
# 2. Same training and validation settings as Cell 40
# ------------------------------------------------------------

def evaluate43(model, graph):
    model.eval()

    with torch.no_grad():
        p = torch.sigmoid(
            model(graph, val_pairs40)
        ).cpu().numpy()

    return (
        float(average_precision_score(val_labels_np40, p)),
        float(roc_auc_score(val_labels_np40, p))
    )


rows43 = []
start43 = time.perf_counter()

print("=" * 85, flush=True)
print("CELL 43 | OTC FIVE-SEED BASELINE TRAINING", flush=True)
print("Device:", DEV43, flush=True)
print("=" * 85, flush=True)

for method in METHODS43:

    for seed in SEEDS43:

        # Match seed-specific graph selection.
        train_df = select43(
            context39, method, seed
        )[0]

        val_df = select43(
            history39, method, seed
        )[0]

        assert len(train_df) == 10678
        assert len(val_df) == 14237

        graph_train = graph_tensors39(train_df)
        graph_val = graph_tensors39(val_df)

        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)

        if DEV43.type == "cuda":
            torch.cuda.manual_seed_all(seed)

        model = SignedGraphSAGE40().to(DEV43)
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=0.001,
            weight_decay=1e-4
        )

        best_pr = -1.0
        best_epoch = 0
        best_state = None
        waiting = 0

        for epoch in range(1, 121):

            model.train()
            optimizer.zero_grad()

            logits = model(graph_train, train_pairs40)
            loss = criterion40(logits, train_y40)

            loss.backward()
            nn.utils.clip_grad_norm_(
                model.parameters(), max_norm=5.0
            )
            optimizer.step()

            pr, roc = evaluate43(model, graph_val)

            if pr > best_pr + 1e-6:
                best_pr = pr
                best_epoch = epoch
                best_state = copy.deepcopy(model.state_dict())
                waiting = 0
            else:
                waiting += 1

            if waiting >= 20:
                break

        model.load_state_dict(best_state, strict=True)
        final_pr, final_roc = evaluate43(model, graph_val)

        torch.save({
            "model_state_dict": {
                k: v.detach().cpu()
                for k, v in best_state.items()
            },
            "method": method,
            "seed": seed,
            "best_epoch": best_epoch,
            "keep_ratio": 0.50,
            "dataset": "bitcoin_otc",
            "protocol_id": "TASP-GNN-v1.0"
        }, R43 / f"otc_{method}_seed{seed}.pt")

        rows43.append({
            "seed": seed,
            "method": method,
            "best_epoch": best_epoch,
            "validation_pr_auc": final_pr,
            "validation_roc_auc": final_roc
        })

        print(
            f"{method:25s} | Seed {seed} | "
            f"PR={final_pr:.5f} | ROC={final_roc:.5f} "
            f"| Epoch={best_epoch}",
            flush=True
        )

# ------------------------------------------------------------
# 3. Combined comparison of all five methods
# ------------------------------------------------------------

baseline43 = pd.DataFrame(rows43)

baseline43.to_csv(
    R43 / "otc_connected_baselines_five_seed.csv",
    index=False
)

full43 = pd.read_csv(
    R43 / "otc_full_teacher_five_seed.csv"
)

task43 = pd.read_csv(
    R43 / "otc_task_aware_retrained_five_seed.csv"
)

combined43 = pd.concat([
    baseline43[
        ["seed", "method",
         "validation_pr_auc", "validation_roc_auc"]
    ],
    full43.assign(method="full")[
        ["seed", "method",
         "validation_pr_auc", "validation_roc_auc"]
    ],
    task43.assign(method="task_aware_connectivity")[
        ["seed", "method",
         "validation_pr_auc", "validation_roc_auc"]
    ]
], ignore_index=True)

assert not combined43.duplicated(
    ["seed", "method"]
).any()

summary43 = (
    combined43.groupby("method", as_index=False)
    .agg(
        pr_mean=("validation_pr_auc", "mean"),
        pr_std=("validation_pr_auc", "std"),
        roc_mean=("validation_roc_auc", "mean"),
        roc_std=("validation_roc_auc", "std")
    )
    .sort_values("pr_mean", ascending=False)
)

paired43 = combined43.pivot(
    index="seed",
    columns="method",
    values="validation_pr_auc"
)

print("\n" + "=" * 95)
print("CELL 43 | OTC FIVE-METHOD RETRAINED COMPARISON")
print("=" * 95)
print(summary43.round(5).to_string(index=False))

print("\n" + "=" * 95)
print("CELL 43 | PAIRED PR-AUC COMPARISON")
print("=" * 95)

for method in [
    "full",
    "random_connectivity",
    "magnitude_connectivity",
    "structural_connectivity"
]:
    delta = (
        paired43["task_aware_connectivity"]
        - paired43[method]
    )
    print(
        f"Task-Aware minus {method:25s} | "
        f"Mean={delta.mean():+.5f} | "
        f"Wins={int((delta > 0).sum())}/5"
    )

combined43.to_csv(
    R43 / "otc_all_methods_five_seed.csv",
    index=False
)

summary43.to_csv(
    R43 / "otc_all_methods_summary.csv",
    index=False
)

if DEV43.type == "cuda":
    torch.cuda.synchronize()

print(
    "\nElapsed seconds:",
    round(time.perf_counter() - start43, 2)
)

print("Protocol unchanged.")
print("Test set untouched.")


In [ ]:

# ============================================================
# TASP-GNN | Cell 44
# Pre-Test Audit and Evaluation Lock
# OTC | No training | No test access
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import hashlib
import pandas as pd

R44 = Path("/kaggle/working/ta_gsnn_results")
SEEDS44 = [42, 43, 44, 45, 46]

METHODS44 = [
    "full",
    "random_connectivity",
    "magnitude_connectivity",
    "structural_connectivity",
    "task_aware_connectivity"
]

# Existing validation results.
results44 = pd.read_csv(
    R44 / "otc_all_methods_five_seed.csv"
)

assert len(results44) == 25
assert not results44.duplicated(
    ["seed", "method"]
).any()

for method in METHODS44:
    observed = sorted(
        results44.loc[
            results44["method"] == method, "seed"
        ].tolist()
    )
    assert observed == SEEDS44, method

# Confirm all 25 checkpoints exist.
checkpoint_paths44 = []

for seed in SEEDS44:
    checkpoint_paths44.extend([
        R44 / f"otc_full_teacher_seed{seed}.pt",
        R44 / f"otc_task_aware_retrained_seed{seed}.pt"
    ])

    for method in METHODS44[1:4]:
        checkpoint_paths44.append(
            R44 / f"otc_{method}_seed{seed}.pt"
        )

selection_paths44 = []

for seed in SEEDS44:
    for phase in ["context", "validation_history"]:
        selection_paths44.append(
            R44 /
            f"otc_task_aware_seed{seed}_{phase}_selection.csv"
        )

required44 = (
    checkpoint_paths44
    + selection_paths44
    + [
        R44 / "experiment_protocol_v1.json",
        R44 / "otc_all_methods_five_seed.csv",
        R44 / "otc_task_aware_selection_summary.csv"
    ]
)

missing44 = [str(p) for p in required44 if not p.is_file()]

if missing44:
    raise FileNotFoundError(
        "Missing artifacts:\n" + "\n".join(missing44)
    )

def sha44(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(block)
    return h.hexdigest()

manifest44 = pd.DataFrame([
    {
        "file": p.name,
        "bytes": p.stat().st_size,
        "sha256": sha44(p)
    }
    for p in required44
])

pivot44 = results44.pivot(
    index="seed",
    columns="method",
    values="validation_pr_auc"
)

task44 = pivot44["task_aware_connectivity"]

comparisons44 = []

for method in METHODS44[:-1]:
    diff = task44 - pivot44[method]
    comparisons44.append({
        "baseline": method,
        "mean_pr_difference": float(diff.mean()),
        "wins": int((diff > 0).sum()),
        "losses": int((diff < 0).sum())
    })

paired44 = pd.DataFrame(comparisons44)

test_plan44 = {
    "protocol_id": "TASP-GNN-v1.0",
    "audit_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "development_dataset": "Bitcoin Alpha",
    "replication_dataset": "Bitcoin OTC",
    "seeds": SEEDS44,
    "methods": METHODS44,
    "retention_ratio": 0.50,
    "primary_metric": "negative_pr_auc",
    "secondary_metric": "roc_auc",
    "checkpoint_selection": "already finalized on validation",
    "test_history": "context + train + validation",
    "test_targets": "chronologically held-out test edges",
    "task_aware_test_scoring_labels": "train labels only",
    "test_scoring_rule": "unchanged frozen gradient and "
                         "connectivity selection",
    "retraining_after_test": False,
    "parameter_tuning_after_test": False,
    "test_results_available": False,
    "baseline_selection_reproduction":
        "Use exact deterministic Cell 43 algorithms "
        "with their existing checkpoints",
    "disclosure":
        "Validation-guided development and in-sample "
        "training-edge attribution must be reported"
}

manifest44.to_csv(
    R44 / "otc_pretest_artifact_manifest.csv",
    index=False
)

with open(
    R44 / "final_test_evaluation_plan.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        test_plan44, f, indent=2, ensure_ascii=False
    )

print("=" * 80)
print("CELL 44 | PRE-TEST AUDIT")
print("=" * 80)

print("Models:", len(METHODS44))
print("Seeds:", len(SEEDS44))
print("Verified checkpoints:", len(checkpoint_paths44))
print("Verified selection files:", len(selection_paths44))
print("Missing files:", len(missing44))

print("\nVALIDATION PAIRED COMPARISONS")
print(paired44.round(5).to_string(index=False))

print("\nArtifact SHA-256 manifest: SAVED")
print("Final test evaluation plan: SAVED")
print("Algorithm changes after this point: NOT PERMITTED")
print("Test files accessed: NO")
print("Status: READY FOR FINAL TEST IMPLEMENTATION")


In [ ]:

# ============================================================
# TASP-GNN | Cell 45
# FINAL HELD-OUT TEST EVALUATION - BITCOIN OTC
# 5 methods x 5 seeds | GPU | NO RETRAINING
# Frozen Protocol: TASP-GNN-v1.0
#
# Requires definitions from Cells 39, 40, 41, 43
# ============================================================

from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, roc_auc_score

R45 = Path("/kaggle/working/ta_gsnn_results")
D45 = Path("/kaggle/working/ta_gsnn_data")
DEV45 = DEVICE40
SEEDS45 = [42, 43, 44, 45, 46]

METHODS45 = [
    "full",
    "random_connectivity",
    "magnitude_connectivity",
    "structural_connectivity",
    "task_aware_connectivity"
]

required45 = [
    "context39", "train39", "validation39",
    "node_to_idx39", "train_pairs40", "train_y40",
    "criterion40", "SignedGraphSAGE40",
    "prepare_edges41", "weighted_forward41",
    "connected_selection41", "select43"
]

missing45 = [x for x in required45 if x not in globals()]
if missing45:
    raise RuntimeError(
        "Missing definitions. Run the relevant "
        "previous cells: " + str(missing45)
    )

# Guard against accidentally repeating the final evaluation.
output45 = R45 / "otc_final_test_results_v1.csv"

if output45.exists():
    raise RuntimeError(
        "Final OTC test results already exist. "
        "Do not rerun or revise the test procedure."
    )

with open(
    R45 / "final_test_evaluation_plan.json",
    encoding="utf-8"
) as f:
    plan45 = json.load(f)

assert plan45["protocol_id"] == "TASP-GNN-v1.0"
assert plan45["retention_ratio"] == 0.50
assert plan45["seeds"] == SEEDS45
assert plan45["methods"] == METHODS45

print("=" * 85, flush=True)
print("CELL 45 | FROZEN OTC TEST EVALUATION", flush=True)
print("Device:", DEV45, flush=True)
print("Protocol:", plan45["protocol_id"], flush=True)

# ------------------------------------------------------------
# 1. Construct history WITHOUT loading Test.
# ------------------------------------------------------------

history45 = pd.concat(
    [context39, train39, validation39],
    ignore_index=True
)

assert len(history45) == 32033
assert (
    history45["timestamp"].is_monotonic_increasing
)

print("Historical edges:", len(history45), flush=True)
print("Scoring and selecting historical edges...", flush=True)

prepared45 = prepare_edges41(history45)

# ------------------------------------------------------------
# 2. Generate all frozen method selections.
#    Task-aware scores use TRAIN labels only.
#    No Test data has been loaded.
# ------------------------------------------------------------

selections45 = {}

# These two selections do not depend on seed.
magnitude_df45 = select43(
    history45, "magnitude_connectivity", 42
)[0]

structural_df45 = select43(
    history45, "structural_connectivity", 42
)[0]


def selection_mask45(original, selected):
    original_keys = pd.MultiIndex.from_frame(
        original[["source", "target"]]
    )
    selected_keys = pd.MultiIndex.from_frame(
        selected[["source", "target"]]
    )

    assert original_keys.is_unique
    assert selected_keys.is_unique

    mask = np.asarray(
        original_keys.isin(selected_keys),
        dtype=bool
    )

    assert mask.sum() == len(selected)
    return mask


magnitude_mask45 = selection_mask45(
    history45, magnitude_df45
)
structural_mask45 = selection_mask45(
    history45, structural_df45
)

budget45 = round(len(history45) * 0.50)

for seed in SEEDS45:

    teacher_checkpoint = torch.load(
        R45 / f"otc_full_teacher_seed{seed}.pt",
        map_location=DEV45,
        weights_only=True
    )

    teacher_state = {
        k: v.detach()
        for k, v in
        teacher_checkpoint["model_state_dict"].items()
    }

    # No model training: gradient only w.r.t. edge weights.
    w45 = torch.ones(
        len(history45),
        dtype=torch.float32,
        device=DEV45,
        requires_grad=True
    )

    logits45 = weighted_forward41(
        prepared45, w45, teacher_state
    )

    loss45 = criterion40(logits45, train_y40)

    gradient45 = torch.autograd.grad(
        loss45, w45
    )[0]

    importance45 = (
        -gradient45.detach().cpu().numpy()
    )

    assert np.isfinite(importance45).all()

    task_mask45, forest45, comps45, nodes45 = (
        connected_selection41(
            history45, importance45
        )
    )

    random_df45 = select43(
        history45, "random_connectivity", seed
    )[0]

    random_mask45 = selection_mask45(
        history45, random_df45
    )

    selections45[seed] = {
        "full": np.ones(
            len(history45), dtype=bool
        ),
        "random_connectivity": random_mask45,
        "magnitude_connectivity": magnitude_mask45,
        "structural_connectivity": structural_mask45,
        "task_aware_connectivity": task_mask45
    }

    # Save exact selection and gradient for reproducibility.
    scored45 = history45.copy()
    scored45["importance"] = importance45
    scored45["selected"] = task_mask45

    scored45.to_csv(
        R45 / f"otc_test_history_task_selection_seed{seed}.csv",
        index=False
    )

    for method, mask in selections45[seed].items():
        if method != "full":
            assert int(mask.sum()) == budget45

    print(
        f"Seed {seed}: selections ready | "
        f"task forest={forest45} | "
        f"components={comps45}",
        flush=True
    )

print(
    "\nAll graph selections finished BEFORE "
    "loading Test data.",
    flush=True
)

# ------------------------------------------------------------
# 3. Load held-out Test ONLY NOW.
#    Test edges NEVER enter graph or edge scoring.
# ------------------------------------------------------------

test45 = pd.read_csv(
    D45 / "bitcoin_otc_test.csv"
)

assert (
    history45["timestamp"].max()
    < test45["timestamp"].min()
)

assert not test45[["source", "target"]].isna().any().any()

# Preserve the old indexing exactly. Add unseen test nodes
# at the end; their graph features remain zero.
node_map45 = dict(node_to_idx39)

test_nodes45 = (
    set(test45["source"])
    | set(test45["target"])
)

for node in sorted(test_nodes45 - set(node_map45)):
    node_map45[node] = len(node_map45)

N45 = len(node_map45)


def indices45(series):
    values = series.map(node_map45).to_numpy()
    assert pd.notna(values).all()

    return values.astype(np.int64)


test_pairs45 = torch.tensor(
    np.column_stack([
        indices45(test45["source"]),
        indices45(test45["target"])
    ]),
    dtype=torch.long,
    device=DEV45
)

test_labels45 = (
    test45["rating"].to_numpy() < 0
).astype(int)

assert len(np.unique(test_labels45)) == 2

# Full and sparse graph tensors, using ONLY historical edges.
def graph_for_mask45(mask):
    frame = history45.iloc[np.flatnonzero(mask)]

    s = indices45(frame["source"])
    d = indices45(frame["target"])
    positive = frame["rating"].to_numpy() > 0

    features = np.zeros((N45, 4), dtype=np.float32)

    np.add.at(features[:, 0], s[positive], 1)
    np.add.at(features[:, 1], s[~positive], 1)
    np.add.at(features[:, 2], d[positive], 1)
    np.add.at(features[:, 3], d[~positive], 1)

    return {
        "x": torch.tensor(
            np.log1p(features),
            dtype=torch.float32,
            device=DEV45
        ),
        "pos": torch.tensor(
            np.stack([s[positive], d[positive]]),
            dtype=torch.long,
            device=DEV45
        ),
        "neg": torch.tensor(
            np.stack([s[~positive], d[~positive]]),
            dtype=torch.long,
            device=DEV45
        )
    }


def checkpoint_path45(method, seed):
    if method == "full":
        return R45 / f"otc_full_teacher_seed{seed}.pt"

    if method == "task_aware_connectivity":
        return R45 / f"otc_task_aware_retrained_seed{seed}.pt"

    return R45 / f"otc_{method}_seed{seed}.pt"


# ------------------------------------------------------------
# 4. Evaluate all frozen checkpoints ONCE.
# ------------------------------------------------------------

records45 = []

for seed in SEEDS45:
    for method in METHODS45:

        mask = selections45[seed][method]
        graph = graph_for_mask45(mask)

        checkpoint = torch.load(
            checkpoint_path45(method, seed),
            map_location=DEV45,
            weights_only=True
        )

        model = SignedGraphSAGE40().to(DEV45)

        model.load_state_dict(
            checkpoint["model_state_dict"],
            strict=True
        )
        model.eval()

        with torch.no_grad():
            prob = torch.sigmoid(
                model(graph, test_pairs45)
            ).cpu().numpy()

        pr = float(
            average_precision_score(test_labels45, prob)
        )
        roc = float(
            roc_auc_score(test_labels45, prob)
        )

        records45.append({
            "seed": seed,
            "method": method,
            "test_pr_auc": pr,
            "test_roc_auc": roc,
            "retained_edges": int(mask.sum()),
            "retained_pct": 100 * mask.mean(),
            "checkpoint_epoch": int(checkpoint["best_epoch"])
        })

# ------------------------------------------------------------
# 5. Save results; never tune based on this output.
# ------------------------------------------------------------

raw45 = pd.DataFrame(records45)

summary45 = (
    raw45.groupby("method", as_index=False)
    .agg(
        pr_mean=("test_pr_auc", "mean"),
        pr_std=("test_pr_auc", "std"),
        roc_mean=("test_roc_auc", "mean"),
        roc_std=("test_roc_auc", "std")
    )
    .sort_values("pr_mean", ascending=False)
)

paired45 = raw45.pivot(
    index="seed",
    columns="method",
    values="test_pr_auc"
)

raw45.to_csv(output45, index=False)

summary45.to_csv(
    R45 / "otc_final_test_summary_v1.csv",
    index=False
)

if DEV45.type == "cuda":
    torch.cuda.synchronize()

print("\n" + "=" * 95)
print("CELL 45 | FINAL OTC TEST RESULTS")
print("=" * 95)
print("Test edges:", len(test45))
print("Negative prevalence:",
      round(100 * test_labels45.mean(), 3), "%")
print("Indexed nodes:", N45)

print("\n" + "=" * 95)
print("FINAL TEST | FIVE-METHOD SUMMARY")
print("=" * 95)
print(summary45.round(5).to_string(index=False))

print("\n" + "=" * 95)
print("FINAL TEST | PER-SEED PR-AUC")
print("=" * 95)
print(paired45.round(5).to_string())

print("\n" + "=" * 95)
print("FINAL TEST | PAIRED COMPARISONS")
print("=" * 95)

for method in METHODS45:
    if method == "task_aware_connectivity":
        continue

    delta = (
        paired45["task_aware_connectivity"]
        - paired45[method]
    )

    print(
        f"Task-Aware minus {method:25s} | "
        f"Mean={delta.mean():+.5f} | "
        f"Wins={int((delta > 0).sum())}/5"
    )

print("\nFinal results saved.")
print("No training or parameter tuning performed.")
print("Frozen protocol maintained.")


In [ ]:

# ============================================================
# TASP-GNN | CELL 46
# FINAL HELD-OUT TEST — BITCOIN ALPHA
# Protocol v1.0 | Five seeds | No retraining
# Selection verification BEFORE test access
# ============================================================

from pathlib import Path
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, roc_auc_score

D46 = Path("/kaggle/working/ta_gsnn_data")
R46 = Path("/kaggle/working/ta_gsnn_results")
DEV46 = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEEDS46 = [42, 43, 44, 45, 46]
METHODS46 = [
    "full", "random", "magnitude",
    "structural_connectivity", "task_aware_connectivity"
]
FINAL46 = R46 / "alpha_final_test_results_v1.csv"

if FINAL46.exists():
    raise RuntimeError(
        "Final Alpha test already exists. Do not rerun."
    )

context46 = pd.read_csv(D46 / "bitcoin_alpha_context.csv")
train46 = pd.read_csv(D46 / "bitcoin_alpha_train.csv")
val46 = pd.read_csv(D46 / "bitcoin_alpha_validation.csv")

validation_history46 = pd.concat(
    [context46, train46], ignore_index=True
)
test_history46 = pd.concat(
    [context46, train46, val46], ignore_index=True
)

# Test is NOT loaded yet.
nodes46 = sorted(
    set(test_history46.source) | set(test_history46.target)
)
index46 = {node: i for i, node in enumerate(nodes46)}

def pairs46(frame):
    return torch.tensor(
        np.column_stack([
            frame.source.map(index46).to_numpy(),
            frame.target.map(index46).to_numpy()
        ]),
        dtype=torch.long, device=DEV46
    )

train_pairs46 = pairs46(train46)
train_y46 = torch.tensor(
    (train46.rating.to_numpy() < 0).astype(np.float32),
    device=DEV46
)
class_weight46 = torch.tensor(
    (train_y46 == 0).sum().item()
    / (train_y46 == 1).sum().item(),
    device=DEV46
)

# ------------------------------------------------------------
# 1. Recreate signed GNN architecture
# ------------------------------------------------------------

class SignedLayer46(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(5 * in_dim, out_dim)
        self.norm = nn.LayerNorm(out_dim)

    def forward(self, h, graph):
        n = h.shape[0]
        pieces = [h]

        for sign in ("pos", "neg"):
            src, dst = graph[sign]

            for incoming in (True, False):
                receiver = dst if incoming else src
                sender = src if incoming else dst

                total = torch.zeros_like(h).index_add(
                    0, receiver, h[sender]
                )
                count = torch.bincount(
                    receiver, minlength=n
                ).to(h.dtype).clamp(min=1)

                pieces.append(total / count[:, None])

        return self.norm(
            self.linear(torch.cat(pieces, dim=1))
        )


class SignedGraph46(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = SignedLayer46(4, 32)
        self.layer2 = SignedLayer46(32, 32)
        self.dropout = 0.2
        self.decoder = nn.Sequential(
            nn.Linear(128, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1)
        )

    def forward(self, graph, pairs):
        h = F.relu(self.layer1(graph["x"], graph))
        h = F.dropout(h, p=self.dropout, training=self.training)
        h = F.relu(self.layer2(h, graph))
        h = F.dropout(h, p=self.dropout, training=self.training)

        u, v = h[pairs[:, 0]], h[pairs[:, 1]]
        z = torch.cat([u, v, torch.abs(u-v), u*v], dim=1)
        return self.decoder(z).squeeze(-1)


def graph46(frame):
    s = frame.source.map(index46).to_numpy(dtype=np.int64)
    d = frame.target.map(index46).to_numpy(dtype=np.int64)
    positive = frame.rating.to_numpy() > 0

    x = np.zeros((len(index46), 4), dtype=np.float32)

    np.add.at(x[:, 0], s[positive], 1)
    np.add.at(x[:, 1], s[~positive], 1)
    np.add.at(x[:, 2], d[positive], 1)
    np.add.at(x[:, 3], d[~positive], 1)

    return {
        "x": torch.tensor(
            np.log1p(x), device=DEV46
        ),
        "pos": torch.tensor(
            np.stack([s[positive], d[positive]]),
            dtype=torch.long, device=DEV46
        ),
        "neg": torch.tensor(
            np.stack([s[~positive], d[~positive]]),
            dtype=torch.long, device=DEV46
        )
    }


# ------------------------------------------------------------
# 2. Frozen selection algorithms
# ------------------------------------------------------------

def select46(frame, method, seed=42, importance=None):
    frame = frame.reset_index(drop=True)
    n = len(frame)

    if method == "full":
        return np.ones(n, dtype=bool)

    budget = round(0.5 * n)

    u = frame.source.to_numpy()
    v = frame.target.to_numpy()
    rating = frame.rating.to_numpy()

    nodes = sorted(set(u) | set(v))
    lookup = {node: i for i, node in enumerate(nodes)}

    if method == "random":
        scores = np.random.default_rng(seed).random(n)
    elif method == "random_connectivity":
        scores = np.random.default_rng(seed).random(n)
    elif method in ("magnitude", "magnitude_connectivity"):
        scores = np.abs(rating).astype(float)
    elif method == "structural_connectivity":
        degree = np.zeros(len(nodes), dtype=np.int64)
        for a, b in zip(u, v):
            degree[lookup[a]] += 1
            degree[lookup[b]] += 1
        scores = np.array([
            abs(rating[i]) / 10.0
            + 0.5 * (
                1 / np.sqrt(degree[lookup[u[i]]])
                + 1 / np.sqrt(degree[lookup[v[i]]])
            )
            for i in range(n)
        ])
    elif method == "task_aware_connectivity":
        scores = np.asarray(importance, dtype=float)
    else:
        raise ValueError(method)

    assert len(scores) == n
    assert np.isfinite(scores).all()

    order = np.argsort(-scores, kind="stable")
    selected = np.zeros(n, dtype=bool)

    # Original Alpha Random/Magnitude: node coverage,
    # not necessarily weak-component preservation.
    if method in ("random", "magnitude"):
        required = {}
        for i in order:
            required.setdefault(int(u[i]), int(i))
            required.setdefault(int(v[i]), int(i))

        selected[list(set(required.values()))] = True
        if selected.sum() > budget:
            raise ValueError("Coverage budget infeasible")

    else:
        # Connectivity-preserving maximum spanning forest.
        parent = np.arange(len(nodes))
        size = np.ones(len(nodes), dtype=np.int64)

        def find(a):
            while parent[a] != a:
                parent[a] = parent[parent[a]]
                a = parent[a]
            return a

        def union(a, b):
            a, b = find(a), find(b)
            if a == b:
                return False
            if size[a] < size[b]:
                a, b = b, a
            parent[b] = a
            size[a] += size[b]
            return True

        for i in order:
            if union(lookup[u[i]], lookup[v[i]]):
                selected[i] = True

        if selected.sum() > budget:
            raise ValueError("Connectivity budget infeasible")

        # Exact Structural policy of Cell 24.
        if method == "structural_connectivity":
            negative = rating < 0
            desired = round(budget * negative.mean())
            needed = max(
                0, desired - int((selected & negative).sum())
            )
            capacity = budget - int(selected.sum())
            candidates = order[
                negative[order] & ~selected[order]
            ]
            selected[candidates[:min(needed, capacity)]] = True

    capacity = budget - int(selected.sum())
    remaining = order[~selected[order]]
    selected[remaining[:capacity]] = True

    assert int(selected.sum()) == budget
    return selected


# ------------------------------------------------------------
# 3. Exact differentiable weighted forward for teacher
# ------------------------------------------------------------

def weighted_logits46(frame, weights, state):
    s = torch.tensor(
        frame.source.map(index46).to_numpy(),
        dtype=torch.long, device=DEV46
    )
    d = torch.tensor(
        frame.target.map(index46).to_numpy(),
        dtype=torch.long, device=DEV46
    )
    pos = torch.tensor(
        frame.rating.to_numpy() > 0,
        dtype=torch.bool, device=DEV46
    )

    n_nodes = len(index46)
    columns = []

    for receiver, mask in (
        (s, pos), (s, ~pos), (d, pos), (d, ~pos)
    ):
        col = torch.zeros(
            n_nodes, device=DEV46
        ).index_add(0, receiver[mask], weights[mask])
        columns.append(col)

    h = torch.log1p(torch.stack(columns, dim=1))

    for layer in ("layer1", "layer2"):
        pieces = [h]

        for mask in (pos, ~pos):
            ss, dd, ww = s[mask], d[mask], weights[mask]

            for incoming in (True, False):
                receiver = dd if incoming else ss
                sender = ss if incoming else dd

                numerator = torch.zeros_like(h).index_add(
                    0, receiver, h[sender] * ww[:, None]
                )

                denominator = torch.zeros(
                    n_nodes, device=DEV46
                ).index_add(0, receiver, ww)

                pieces.append(
                    numerator / denominator.clamp(min=1)[:, None]
                )

        z = F.linear(
            torch.cat(pieces, dim=1),
            state[f"{layer}.linear.weight"],
            state[f"{layer}.linear.bias"]
        )
        h = F.relu(F.layer_norm(
            z, (z.shape[1],),
            state[f"{layer}.norm.weight"],
            state[f"{layer}.norm.bias"],
            eps=1e-5
        ))

    u = h[train_pairs46[:, 0]]
    v = h[train_pairs46[:, 1]]
    pair = torch.cat(
        [u, v, torch.abs(u-v), u*v], dim=1
    )

    z = F.relu(F.linear(
        pair,
        state["decoder.0.weight"],
        state["decoder.0.bias"]
    ))

    return F.linear(
        z,
        state["decoder.3.weight"],
        state["decoder.3.bias"]
    ).squeeze(-1)


def importance46(frame, teacher_state):
    w = torch.ones(
        len(frame), device=DEV46, requires_grad=True
    )

    logits = weighted_logits46(frame, w, teacher_state)

    loss = F.binary_cross_entropy_with_logits(
        logits, train_y46, pos_weight=class_weight46
    )

    gradients = torch.autograd.grad(loss, w)[0]
    return -gradients.detach().cpu().numpy()


# ------------------------------------------------------------
# 4. Mandatory PRE-TEST verification
# ------------------------------------------------------------

print("=" * 82, flush=True)
print("CELL 46 | PRE-TEST ALPHA SELECTION VERIFICATION", flush=True)
print("Device:", DEV46, flush=True)

structural_check46 = select46(
    validation_history46, "structural_connectivity"
)

stored_structural46 = pd.read_csv(
    R46 / "bitcoin_alpha_connectivity_preserving_50pct.csv"
)

saved_pairs46 = set(zip(
    stored_structural46.source, stored_structural46.target
))

computed_pairs46 = set(zip(
    validation_history46.loc[
        structural_check46, "source"
    ],
    validation_history46.loc[
        structural_check46, "target"
    ]
))

assert saved_pairs46 == computed_pairs46, (
    "Structural selection mismatch — stop before Test."
)

teacher_states46 = {}
for seed in SEEDS46:
    teacher = torch.load(
        R46 / f"fair_retrained_full_seed{seed}.pt",
        map_location=DEV46,
        weights_only=True
    )
    teacher_states46[seed] = {
        k: v.detach()
        for k, v in teacher["model_state_dict"].items()
    }

    scores_check = importance46(
        validation_history46, teacher_states46[seed]
    )
    mask_check = select46(
        validation_history46,
        "task_aware_connectivity",
        importance=scores_check
    )

    saved = pd.read_csv(
        R46 / (
            f"task_aware_seed{seed}_"
            "validation_history_selection.csv"
        )
    )

    assert np.array_equal(
        saved[["source", "target"]].to_numpy(),
        validation_history46[["source", "target"]].to_numpy()
    )

    saved_mask = saved["selected"].astype(bool).to_numpy()
    agreement = float((mask_check == saved_mask).mean())

    print(
        f"Seed {seed} | selection agreement={agreement:.6f}",
        flush=True
    )

    if not np.array_equal(mask_check, saved_mask):
        raise RuntimeError(
            f"Seed {seed} selection mismatch. "
            "Test must NOT be opened."
        )

print("PRE-TEST VERIFICATION: PASS", flush=True)

# ------------------------------------------------------------
# 5. Select ALL test-history edges BEFORE opening Test
# ------------------------------------------------------------

selections46 = {}
for seed in SEEDS46:
    scores = importance46(
        test_history46, teacher_states46[seed]
    )

    selections46[seed] = {
        "full": select46(test_history46, "full"),
        "random": select46(
            test_history46, "random", seed
        ),
        "magnitude": select46(
            test_history46, "magnitude", seed
        ),
        "structural_connectivity": select46(
            test_history46, "structural_connectivity", seed
        ),
        "task_aware_connectivity": select46(
            test_history46,
            "task_aware_connectivity",
            importance=scores
        )
    }

    print(
        f"Seed {seed}: test-history selections prepared",
        flush=True
    )

print("All selections completed before Test access.", flush=True)

# ------------------------------------------------------------
# 6. FIRST TEST ACCESS
# ------------------------------------------------------------

test46 = pd.read_csv(D46 / "bitcoin_alpha_test.csv")

assert test_history46.timestamp.max() < test46.timestamp.min()

for node in sorted(
    (set(test46.source) | set(test46.target))
    - set(index46)
):
    index46[node] = len(index46)

test_pairs46 = pairs46(test46)
test_y46 = (test46.rating.to_numpy() < 0).astype(int)

assert len(np.unique(test_y46)) == 2

def checkpoint_path46(method, seed):
    if method in ("full", "random", "magnitude"):
        return R46 / f"fair_retrained_{method}_seed{seed}.pt"
    if method == "structural_connectivity":
        return R46 / f"connectivity_preserving_alpha_seed{seed}.pt"
    return R46 / f"task_aware_retrained_alpha_seed{seed}.pt"


# ------------------------------------------------------------
# 7. ONE-TIME held-out evaluation
# ------------------------------------------------------------

rows46 = []

for seed in SEEDS46:
    for method in METHODS46:
        mask = selections46[seed][method]
        selected_frame = test_history46.iloc[
            np.flatnonzero(mask)
        ]
        graph = graph46(selected_frame)

        checkpoint = torch.load(
            checkpoint_path46(method, seed),
            map_location=DEV46,
            weights_only=True
        )

        model = SignedGraph46().to(DEV46)
        model.load_state_dict(
            checkpoint["model_state_dict"],
            strict=True
        )
        model.eval()

        with torch.no_grad():
            predictions = torch.sigmoid(
                model(graph, test_pairs46)
            ).cpu().numpy()

        rows46.append({
            "seed": seed,
            "method": method,
            "test_pr_auc": float(
                average_precision_score(test_y46, predictions)
            ),
            "test_roc_auc": float(
                roc_auc_score(test_y46, predictions)
            ),
            "retained_edges": len(selected_frame),
            "checkpoint_epoch": checkpoint["best_epoch"]
        })

raw46 = pd.DataFrame(rows46)

summary46 = (
    raw46.groupby("method", as_index=False)
    .agg(
        pr_mean=("test_pr_auc", "mean"),
        pr_std=("test_pr_auc", "std"),
        roc_mean=("test_roc_auc", "mean"),
        roc_std=("test_roc_auc", "std")
    )
    .sort_values("pr_mean", ascending=False)
)

paired46 = raw46.pivot(
    index="seed", columns="method", values="test_pr_auc"
)

raw46.to_csv(FINAL46, index=False)
summary46.to_csv(
    R46 / "alpha_final_test_summary_v1.csv",
    index=False
)

print("\n" + "=" * 95)
print("CELL 46 | FINAL ALPHA TEST")
print("=" * 95)
print("Test edges:", len(test46))
print("Negative prevalence (%):", round(100 * test_y46.mean(), 3))

print("\nFINAL ALPHA TEST | FIVE-METHOD SUMMARY")
print(summary46.round(5).to_string(index=False))

print("\nFINAL ALPHA TEST | PER-SEED PR-AUC")
print(paired46.round(5).to_string())

print("\nFINAL ALPHA TEST | PAIRED COMPARISONS")
for method in METHODS46:
    if method == "task_aware_connectivity":
        continue

    diff = paired46["task_aware_connectivity"] - paired46[method]
    print(
        f"Task-Aware minus {method:25s} | "
        f"Mean={diff.mean():+.5f} | "
        f"Wins={(diff > 0).sum()}/5"
    )

print("\nFinal results saved. No retraining performed.")
print("Do NOT modify the protocol after reviewing Test.")


In [ ]:

# ============================================================
# TASP-GNN | Cell 47
# Final Cross-Dataset Test Results
# Read-only analysis | No model execution
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

R47 = Path("/kaggle/working/ta_gsnn_results")

methods47 = [
    "full",
    "random",
    "magnitude",
    "structural_connectivity",
    "task_aware_connectivity"
]

alpha47 = pd.read_csv(
    R47 / "alpha_final_test_results_v1.csv"
)

otc47 = pd.read_csv(
    R47 / "otc_final_test_results_v1.csv"
)

# Normalize method names for the combined table.
otc47["method"] = otc47["method"].replace({
    "random_connectivity": "random",
    "magnitude_connectivity": "magnitude"
})

alpha47["dataset"] = "Bitcoin Alpha"
otc47["dataset"] = "Bitcoin OTC"

combined47 = pd.concat(
    [alpha47, otc47], ignore_index=True
)

assert len(combined47) == 50
assert not combined47.duplicated(
    ["dataset", "method", "seed"]
).any()

for dataset in ["Bitcoin Alpha", "Bitcoin OTC"]:
    for method in methods47:
        subset = combined47[
            (combined47["dataset"] == dataset)
            & (combined47["method"] == method)
        ]
        assert sorted(subset.seed.tolist()) == [
            42, 43, 44, 45, 46
        ]

summary47 = (
    combined47.groupby(
        ["dataset", "method"], as_index=False
    )
    .agg(
        pr_mean=("test_pr_auc", "mean"),
        pr_std=("test_pr_auc", "std"),
        roc_mean=("test_roc_auc", "mean"),
        roc_std=("test_roc_auc", "std"),
        retained_edges=("retained_edges", "mean")
    )
)

summary47["method"] = pd.Categorical(
    summary47["method"],
    categories=methods47,
    ordered=True
)

summary47 = summary47.sort_values(
    ["dataset", "method"]
)

paired_records47 = []

for dataset in ["Bitcoin Alpha", "Bitcoin OTC"]:
    frame = combined47[
        combined47["dataset"] == dataset
    ]

    pivot = frame.pivot(
        index="seed",
        columns="method",
        values="test_pr_auc"
    )

    task = pivot["task_aware_connectivity"]

    for method in methods47:
        if method == "task_aware_connectivity":
            continue

        delta = task - pivot[method]

        paired_records47.append({
            "dataset": dataset,
            "baseline": method,
            "mean_delta": float(delta.mean()),
            "sd_delta": float(delta.std(ddof=1)),
            "wins": int((delta > 0).sum()),
            "losses": int((delta < 0).sum())
        })

paired47 = pd.DataFrame(paired_records47)

print("=" * 95)
print("CELL 47 | FINAL CROSS-DATASET TEST SUMMARY")
print("=" * 95)
print(summary47.round(5).to_string(index=False))

print("\n" + "=" * 95)
print("CELL 47 | FINAL PAIRED TEST DIFFERENCES")
print("=" * 95)
print(paired47.round(5).to_string(index=False))

summary47.to_csv(
    R47 / "paper_final_test_summary.csv",
    index=False
)

paired47.to_csv(
    R47 / "paper_final_paired_differences.csv",
    index=False
)

combined47.to_csv(
    R47 / "paper_final_test_raw.csv",
    index=False
)

print("\nPaper result tables saved.")
print("No models trained or evaluated.")
print("No parameter changes performed.")


In [ ]:

# ============================================================
# TASP-GNN | CELL 48
# Correct final labels + benchmark inference latency
# Bitcoin OTC | CUDA | Seed 42 | No test access
# Requires Cells 39, 40 and 43 definitions
# ============================================================

from pathlib import Path
import time
import gc
import numpy as np
import pandas as pd
import torch

R48 = Path("/kaggle/working/ta_gsnn_results")
DEV48 = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ------------------------------------------------------------
# A. Rebuild correctly labeled final paper table
# ------------------------------------------------------------

alpha48 = pd.read_csv(
    R48 / "alpha_final_test_results_v1.csv"
)
otc48 = pd.read_csv(
    R48 / "otc_final_test_results_v1.csv"
)

alpha48["dataset"] = "Bitcoin Alpha"
otc48["dataset"] = "Bitcoin OTC"

# Preserve original method identifiers: NO renaming.
combined48 = pd.concat(
    [alpha48, otc48], ignore_index=True
)

assert len(combined48) == 50
assert not combined48.duplicated(
    ["dataset", "method", "seed"]
).any()

summary48 = (
    combined48.groupby(["dataset", "method"], as_index=False)
    .agg(
        pr_mean=("test_pr_auc", "mean"),
        pr_std=("test_pr_auc", "std"),
        roc_mean=("test_roc_auc", "mean"),
        roc_std=("test_roc_auc", "std"),
        retained_edges=("retained_edges", "mean")
    )
)

summary48.to_csv(
    R48 / "paper_final_test_summary_corrected.csv",
    index=False
)

print("=" * 85)
print("CELL 48 | CORRECTED FINAL TEST TABLE")
print("=" * 85)
print(summary48.round(5).to_string(index=False))

# ------------------------------------------------------------
# B. Inference-only benchmark on OTC validation history
# ------------------------------------------------------------

required48 = [
    "history39", "graph_tensors39",
    "SignedGraphSAGE40", "val_pairs40",
    "select43"
]

missing48 = [x for x in required48 if x not in globals()]
if missing48:
    raise RuntimeError(
        "Missing definitions from earlier cells: "
        + str(missing48)
    )

seed48 = 42

full_graph48 = graph_tensors39(history39)

structural_frame48 = select43(
    history39, "structural_connectivity", seed48
)[0]

task_frame48 = pd.read_csv(
    R48 /
    f"otc_task_aware_seed{seed48}_validation_history_50pct.csv"
)

assert len(structural_frame48) == 14237
assert len(task_frame48) == 14237

benchmark_items48 = [
    (
        "full",
        history39,
        full_graph48,
        R48 / f"otc_full_teacher_seed{seed48}.pt"
    ),
    (
        "structural_connectivity",
        structural_frame48,
        graph_tensors39(structural_frame48),
        R48 / f"otc_structural_connectivity_seed{seed48}.pt"
    ),
    (
        "task_aware_connectivity",
        task_frame48,
        graph_tensors39(task_frame48),
        R48 / f"otc_task_aware_retrained_seed{seed48}.pt"
    )
]

def sync48():
    if DEV48.type == "cuda":
        torch.cuda.synchronize()


def benchmark48(name, frame, graph, checkpoint_path):
    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEV48,
        weights_only=True
    )

    model = SignedGraphSAGE40().to(DEV48)
    model.load_state_dict(
        checkpoint["model_state_dict"],
        strict=True
    )
    model.eval()

    # Warmup before measuring latency.
    with torch.inference_mode():
        for _ in range(10):
            _ = model(graph, val_pairs40)

    sync48()

    # Measure synchronized wall-clock forward time.
    timings_ms = []

    with torch.inference_mode():
        for _ in range(50):
            sync48()
            start = time.perf_counter()

            _ = model(graph, val_pairs40)

            sync48()
            timings_ms.append(
                1000 * (time.perf_counter() - start)
            )

    # Peak active GPU allocation during the forward pass.
    peak_mib = np.nan

    if DEV48.type == "cuda":
        torch.cuda.reset_peak_memory_stats(DEV48)

        with torch.inference_mode():
            _ = model(graph, val_pairs40)

        sync48()
        peak_mib = (
            torch.cuda.max_memory_allocated(DEV48)
            / (1024 ** 2)
        )

    result = {
        "method": name,
        "seed": seed48,
        "edges": len(frame),
        "median_forward_ms": float(np.median(timings_ms)),
        "mean_forward_ms": float(np.mean(timings_ms)),
        "std_forward_ms": float(np.std(timings_ms, ddof=1)),
        "peak_gpu_allocated_mib": peak_mib
    }

    del model, checkpoint
    gc.collect()

    return result


print("\n" + "=" * 85)
print("CELL 48 | OTC INFERENCE BENCHMARK")
print("=" * 85)
print("Device:", DEV48)
print("Seed:", seed48)
print("Warmup iterations: 10")
print("Timed iterations: 50")

benchmark_results48 = []

for name, frame, graph, path in benchmark_items48:
    result = benchmark48(name, frame, graph, path)
    benchmark_results48.append(result)

    print(
        f"{name:27s} | "
        f"edges={result['edges']:5d} | "
        f"median={result['median_forward_ms']:.4f} ms",
        flush=True
    )

benchmark_df48 = pd.DataFrame(benchmark_results48)

full_latency48 = benchmark_df48.loc[
    benchmark_df48["method"] == "full",
    "median_forward_ms"
].iloc[0]

benchmark_df48["speedup_vs_full"] = (
    full_latency48 / benchmark_df48["median_forward_ms"]
)

print("\n" + "=" * 95)
print("CELL 48 | INFERENCE PERFORMANCE SUMMARY")
print("=" * 95)
print(benchmark_df48.round(4).to_string(index=False))

benchmark_df48.to_csv(
    R48 / "otc_inference_benchmark_seed42.csv",
    index=False
)

print("\nCorrected paper table and benchmark saved.")
print("Inference only; no training or test access.")
print("Teacher training and edge-selection costs NOT included.")


In [ ]:

# ============================================================
# TASP-GNN | Cell 49
# Edge Scoring and Selection Runtime
# Bitcoin OTC | Seed 42 | CUDA
# No training | No test access | No parameter tuning
# Requires Cells 39, 40, 41, 43
# ============================================================

import time
import gc
import numpy as np
import pandas as pd
import torch

SEED49 = 42
REPEATS49 = 10
DEVICE49 = DEVICE40

history49 = history39.reset_index(drop=True)
prepared49 = prepare_edges41(history49)

checkpoint49 = torch.load(
    RESULTS39 / f"otc_full_teacher_seed{SEED49}.pt",
    map_location=DEVICE49,
    weights_only=True
)

state49 = {
    k: v.detach()
    for k, v in checkpoint49["model_state_dict"].items()
}

def sync49():
    if DEVICE49.type == "cuda":
        torch.cuda.synchronize()

def gradient_scores49():
    weights = torch.ones(
        len(history49),
        device=DEVICE49,
        dtype=torch.float32,
        requires_grad=True
    )

    logits = weighted_forward41(
        prepared49, weights, state49
    )

    loss = criterion40(logits, train_y40)

    gradient = torch.autograd.grad(
        loss, weights
    )[0]

    return -gradient.detach().cpu().numpy()

def task_selection49():
    scores = gradient_scores49()
    mask, forest, components, nodes = connected_selection41(
        history49, scores
    )
    return mask

def structural_selection49():
    return select43(
        history49,
        "structural_connectivity",
        SEED49
    )[0]

# Warmup: not included in timed runs.
_ = gradient_scores49()
_ = task_selection49()
_ = structural_selection49()
sync49()

def measure49(fn):
    times = []

    for _ in range(REPEATS49):
        sync49()
        start = time.perf_counter()

        result = fn()

        sync49()
        times.append(
            1000 * (time.perf_counter() - start)
        )

    return np.array(times), result

gradient_times49, gradients49 = measure49(
    gradient_scores49
)

task_total_times49, task_mask49 = measure49(
    task_selection49
)

structural_times49, structural_df49 = measure49(
    structural_selection49
)

# Validate outputs against existing frozen selection.
saved49 = pd.read_csv(
    RESULTS39 /
    f"otc_task_aware_seed{SEED49}_validation_history_selection.csv"
)

assert np.array_equal(
    task_mask49,
    saved49["selected"].astype(bool).to_numpy()
)

assert len(structural_df49) == 14237

rows49 = []

for method, values in [
    ("gradient_scoring_only", gradient_times49),
    ("task_scoring_plus_selection", task_total_times49),
    ("structural_selection_only", structural_times49)
]:
    rows49.append({
        "operation": method,
        "repetitions": REPEATS49,
        "median_ms": float(np.median(values)),
        "mean_ms": float(np.mean(values)),
        "std_ms": float(np.std(values, ddof=1)),
        "min_ms": float(np.min(values)),
        "max_ms": float(np.max(values))
    })

result49 = pd.DataFrame(rows49)

print("=" * 95)
print("CELL 49 | EDGE SCORING AND SELECTION COST")
print("=" * 95)
print("Device:", DEVICE49)
print("Dataset: Bitcoin OTC")
print("Historical edges:", len(history49))
print("Seed:", SEED49)
print("Selection consistency: PASS")
print()
print(result49.round(4).to_string(index=False))

print("\nIMPORTANT:")
print("Task scoring + selection includes gradient calculation.")
print("Do NOT add gradient time to task total again.")
print("Teacher training is NOT included.")
print("Inference and final GNN training are NOT included.")

result49.to_csv(
    RESULTS39 / "otc_selection_runtime_seed42.csv",
    index=False
)

print("\nRuntime results saved.")
print("No training or test access.")


In [ ]:

# ============================================================
# TASP-GNN | Cell 49B
# Selection mismatch diagnostics + runtime summary
# No retraining | No test access
# ============================================================

import numpy as np
import pandas as pd

saved_mask49 = saved49["selected"].astype(bool).to_numpy()
current_mask49 = np.asarray(task_mask49, dtype=bool)

assert len(saved_mask49) == len(current_mask49)

different49 = saved_mask49 != current_mask49
difference_count49 = int(different49.sum())

intersection49 = int(
    (saved_mask49 & current_mask49).sum()
)
union49 = int(
    (saved_mask49 | current_mask49).sum()
)

agreement49 = float(
    (saved_mask49 == current_mask49).mean()
)
jaccard49 = intersection49 / union49

saved_scores49 = saved49["importance"].to_numpy(
    dtype=np.float64
)
current_scores49 = np.asarray(
    gradients49, dtype=np.float64
)

score_diff49 = np.abs(
    saved_scores49 - current_scores49
)

print("=" * 78)
print("CELL 49B | SELECTION CONSISTENCY DIAGNOSTICS")
print("=" * 78)

print("Historical edges:", len(saved_mask49))
print("Saved selected:", int(saved_mask49.sum()))
print("Current selected:", int(current_mask49.sum()))
print("Different mask positions:", difference_count49)
print("Selection agreement:", round(agreement49, 8))
print("Selection Jaccard:", round(jaccard49, 8))
print("Maximum absolute score difference:", score_diff49.max())
print("Mean absolute score difference:", score_diff49.mean())

if difference_count49 == 0:
    print("Selection verification: EXACT MATCH")
elif score_diff49.max() <= 1e-6:
    print(
        "Selection verification: small score differences; "
        "inspect near-tied rankings."
    )
else:
    print(
        "WARNING: Score differences require investigation."
    )

# ------------------------------------------------------------
# Use existing timing results; do not rerun benchmarks.
# ------------------------------------------------------------

runtime49b = pd.DataFrame([
    {
        "operation": name,
        "median_ms": float(np.median(values)),
        "mean_ms": float(np.mean(values)),
        "std_ms": float(np.std(values, ddof=1))
    }
    for name, values in [
        ("gradient_scoring_only", gradient_times49),
        ("task_scoring_plus_selection", task_total_times49),
        ("structural_selection_only", structural_times49)
    ]
])

print("\n" + "=" * 78)
print("CELL 49B | EDGE SCORING AND SELECTION COST")
print("=" * 78)
print(runtime49b.round(4).to_string(index=False))

runtime49b.to_csv(
    RESULTS39 / "otc_selection_runtime_seed42.csv",
    index=False
)

print("\nRuntime results saved.")
print("Frozen Test results unchanged.")


In [ ]:

# ============================================================
# TASP-GNN | Cell 50
# Numerical Reproducibility Audit
# OTC | Seed 42 | No training | No test access
# ============================================================

import numpy as np
import pandas as pd

assert "saved49" in globals()
assert "task_mask49" in globals()
assert "gradients49" in globals()
assert "gradient_scores49" in globals()
assert "connected_selection41" in globals()
assert "history49" in globals()

saved_mask50 = saved49["selected"].astype(bool).to_numpy()
current_mask50 = np.asarray(task_mask49, dtype=bool)

saved_scores50 = saved49["importance"].to_numpy(dtype=float)
current_scores50 = np.asarray(gradients49, dtype=float)

difference_ids50 = np.flatnonzero(
    saved_mask50 != current_mask50
)

# Show exact differences without modifying selections.
details50 = pd.DataFrame({
    "edge_id": difference_ids50,
    "source": history49.iloc[difference_ids50]["source"].to_numpy(),
    "target": history49.iloc[difference_ids50]["target"].to_numpy(),
    "rating": history49.iloc[difference_ids50]["rating"].to_numpy(),
    "saved_selected": saved_mask50[difference_ids50],
    "current_selected": current_mask50[difference_ids50],
    "saved_importance": saved_scores50[difference_ids50],
    "current_importance": current_scores50[difference_ids50],
    "score_difference": (
        current_scores50[difference_ids50]
        - saved_scores50[difference_ids50]
    )
})

pd.set_option("display.float_format", "{:.12e}".format)

print("=" * 88)
print("CELL 50 | DIFFERENTLY SELECTED EDGES")
print("=" * 88)
print(details50.to_string(index=False))

# Compare saved and current gradient arrays.
all_diffs50 = np.abs(saved_scores50 - current_scores50)

largest50 = np.argsort(-all_diffs50)[:10]

print("\n" + "=" * 88)
print("CELL 50 | LARGEST GRADIENT DIFFERENCES")
print("=" * 88)

print(pd.DataFrame({
    "edge_id": largest50,
    "saved_score": saved_scores50[largest50],
    "current_score": current_scores50[largest50],
    "absolute_difference": all_diffs50[largest50]
}).to_string(index=False))

# Check stability across repeated GPU gradient calculations.
repeated_masks50 = []
repeated_scores50 = []

for repeat in range(3):
    scores = gradient_scores49()
    mask, _, _, _ = connected_selection41(
        history49, scores
    )

    repeated_scores50.append(scores)
    repeated_masks50.append(mask)

print("\n" + "=" * 88)
print("CELL 50 | REPEATED COMPUTATION STABILITY")
print("=" * 88)

for i in range(3):
    changed = int(np.sum(
        repeated_masks50[i] != saved_mask50
    ))

    max_score_diff = float(np.max(np.abs(
        repeated_scores50[i] - saved_scores50
    )))

    print(
        f"Repeat {i+1}: "
        f"different_edges_vs_saved={changed}, "
        f"max_score_difference={max_score_diff:.12e}"
    )

for i in range(1, 3):
    changed = int(np.sum(
        repeated_masks50[i] != repeated_masks50[0]
    ))

    print(
        f"Repeat {i+1} vs repeat 1: "
        f"different_mask_positions={changed}"
    )

print("\nNo selection files overwritten.")
print("No training or test evaluation.")


In [ ]:

# ============================================================
# TASP-GNN | Cell 51
# CPU vs GPU Gradient Reproducibility Audit
# Bitcoin OTC | Seed 42 | Historical data only
# No training | No test access | No algorithm changes
# ============================================================

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from pathlib import Path

R51 = Path("/kaggle/working/ta_gsnn_results")
SEED51 = 42
REPEATS51 = 3

checkpoint51 = torch.load(
    R51 / f"otc_full_teacher_seed{SEED51}.pt",
    map_location="cpu",
    weights_only=True
)

devices51 = [torch.device("cpu")]

if torch.cuda.is_available():
    devices51.append(torch.device("cuda"))

# Read only the existing historical graph and training labels.
frame51 = history39.reset_index(drop=True)

source_np51 = frame51["source"].map(
    node_to_idx39
).to_numpy(dtype=np.int64)

target_np51 = frame51["target"].map(
    node_to_idx39
).to_numpy(dtype=np.int64)

positive_np51 = (
    frame51["rating"].to_numpy() > 0
)

train_source_np51 = train39["source"].map(
    node_to_idx39
).to_numpy(dtype=np.int64)

train_target_np51 = train39["target"].map(
    node_to_idx39
).to_numpy(dtype=np.int64)

labels_np51 = (
    train39["rating"].to_numpy() < 0
).astype(np.float32)

N51 = len(node_to_idx39)


def gradients_on_device51(device):

    source = torch.tensor(
        source_np51, dtype=torch.long, device=device
    )
    target = torch.tensor(
        target_np51, dtype=torch.long, device=device
    )
    positive = torch.tensor(
        positive_np51, dtype=torch.bool, device=device
    )

    pairs = torch.tensor(
        np.column_stack([
            train_source_np51,
            train_target_np51
        ]),
        dtype=torch.long, device=device
    )

    labels = torch.tensor(
        labels_np51, dtype=torch.float32, device=device
    )

    class_weight = torch.tensor(
        (labels == 0).sum().item()
        / (labels == 1).sum().item(),
        dtype=torch.float32, device=device
    )

    state = {
        k: v.to(device).detach()
        for k, v in
        checkpoint51["model_state_dict"].items()
    }

    results = []

    for repeat in range(REPEATS51):
        weights = torch.ones(
            len(frame51),
            dtype=torch.float32,
            device=device,
            requires_grad=True
        )

        columns = []

        for idx, mask in [
            (source, positive),
            (source, ~positive),
            (target, positive),
            (target, ~positive)
        ]:
            col = torch.zeros(
                N51, device=device
            ).index_add(
                0, idx[mask], weights[mask]
            )
            columns.append(col)

        h = torch.log1p(
            torch.stack(columns, dim=1)
        )

        for layer in ["layer1", "layer2"]:
            pieces = [h]

            for mask in (positive, ~positive):

                s = source[mask]
                d = target[mask]
                w = weights[mask]

                for incoming in (True, False):
                    receiver = d if incoming else s
                    sender = s if incoming else d

                    numerator = torch.zeros_like(h).index_add(
                        0, receiver,
                        h[sender] * w[:, None]
                    )

                    denominator = torch.zeros(
                        N51, device=device
                    ).index_add(0, receiver, w)

                    pieces.append(
                        numerator /
                        denominator.clamp(min=1)[:, None]
                    )

            z = F.linear(
                torch.cat(pieces, dim=1),
                state[f"{layer}.linear.weight"],
                state[f"{layer}.linear.bias"]
            )

            h = F.relu(F.layer_norm(
                z,
                (z.shape[1],),
                state[f"{layer}.norm.weight"],
                state[f"{layer}.norm.bias"],
                eps=1e-5
            ))

        u = h[pairs[:, 0]]
        v = h[pairs[:, 1]]

        pair = torch.cat([
            u, v, torch.abs(u-v), u*v
        ], dim=1)

        hidden = F.relu(F.linear(
            pair,
            state["decoder.0.weight"],
            state["decoder.0.bias"]
        ))

        logits = F.linear(
            hidden,
            state["decoder.3.weight"],
            state["decoder.3.bias"]
        ).squeeze(-1)

        loss = F.binary_cross_entropy_with_logits(
            logits,
            labels,
            pos_weight=class_weight
        )

        gradient = torch.autograd.grad(
            loss, weights
        )[0]

        results.append(
            -gradient.detach().cpu().numpy()
        )

    return results


results51 = {}

for device in devices51:
    print(
        f"Computing gradients on {device}...",
        flush=True
    )
    results51[str(device)] = gradients_on_device51(
        device
    )

print("\n" + "=" * 90)
print("CELL 51 | GRADIENT REPRODUCIBILITY")
print("=" * 90)

records51 = []

for device_name, arrays in results51.items():

    for repeat in [1, 2]:

        delta = np.abs(
            arrays[0] - arrays[repeat]
        )

        records51.append({
            "device": device_name,
            "comparison": f"repeat1_vs_repeat{repeat+1}",
            "max_abs_difference": float(delta.max()),
            "mean_abs_difference": float(delta.mean()),
            "changed_scores_gt_1e-8": int(
                (delta > 1e-8).sum()
            ),
            "exact_equal": bool(
                np.array_equal(
                    arrays[0], arrays[repeat]
                )
            )
        })

audit51 = pd.DataFrame(records51)

print(audit51.to_string(index=False))

if "cuda" in results51:
    cpu_scores = results51["cpu"][0]
    gpu_scores = results51["cuda"][0]

    print("\nCPU vs GPU:")
    print(
        "Max absolute score difference:",
        float(np.max(np.abs(
            cpu_scores - gpu_scores
        )))
    )

audit51.to_csv(
    R51 / "otc_cpu_gpu_gradient_reproducibility.csv",
    index=False
)

print("\nDiagnostic results saved.")
print("No training, selection modification or test access.")


In [ ]:

# ============================================================
# TASP-GNN | CELL 52
# COMPLETE PROJECT BACKUP BEFORE STOPPING KAGGLE
# Saves data, checkpoints, results, selections, manifest
# No training | No inference | No test evaluation
# ============================================================

from pathlib import Path
import os
import sys
import json
import hashlib
import zipfile
import platform
import datetime
import numpy as np
import pandas as pd
import torch
import sklearn

ROOT52 = Path("/kaggle/working")
DATA52 = ROOT52 / "ta_gsnn_data"
RESULTS52 = ROOT52 / "ta_gsnn_results"
EXTRA52 = ROOT52 / "ta_gsnn_backup_metadata"
EXTRA52.mkdir(parents=True, exist_ok=True)

ARCHIVE52 = ROOT52 / "TASP_GNN_COMPLETE_BACKUP.zip"

print("=" * 85, flush=True)
print("CELL 52 | COMPLETE BACKUP", flush=True)
print("=" * 85, flush=True)

# ------------------------------------------------------------
# 1. Preserve test-history graph selections already in RAM.
#    No gradients, predictions or tests are recomputed.
# ------------------------------------------------------------

selection_status52 = {}

def export_selections52(dataset, frame, selections):
    output_dir = RESULTS52 / f"{dataset}_frozen_test_selections"
    output_dir.mkdir(parents=True, exist_ok=True)

    frame = frame.reset_index(drop=True)

    count = 0

    for seed, methods in selections.items():
        for method, mask in methods.items():
            mask = np.asarray(mask, dtype=bool)

            if len(mask) != len(frame):
                raise ValueError(
                    f"{dataset}/{seed}/{method}: mask length mismatch"
                )

            filename = (
                f"{dataset}_seed{seed}_{method}_test_history.csv"
            )

            selected_frame = frame.loc[mask].copy()

            selected_frame.to_csv(
                output_dir / filename, index=False
            )

            count += 1

    return count


if (
    "selections46" in globals()
    and "test_history46" in globals()
):
    count = export_selections52(
        "bitcoin_alpha",
        test_history46,
        selections46
    )
    selection_status52["alpha"] = f"EXPORTED {count} files"
    print(f"Alpha frozen selections: {count} files saved")
else:
    selection_status52["alpha"] = "NOT IN MEMORY"
    print("WARNING: Alpha selection variables not in memory")


if (
    "selections45" in globals()
    and "history45" in globals()
):
    count = export_selections52(
        "bitcoin_otc",
        history45,
        selections45
    )
    selection_status52["otc"] = f"EXPORTED {count} files"
    print(f"OTC frozen selections: {count} files saved")
else:
    selection_status52["otc"] = "NOT IN MEMORY"
    print("WARNING: OTC selection variables not in memory")

# ------------------------------------------------------------
# 2. Environment and experiment metadata.
# ------------------------------------------------------------

metadata52 = {
    "project": "TASP-GNN",
    "protocol": "TASP-GNN-v1.0",
    "created_utc": datetime.datetime.now(
        datetime.timezone.utc
    ).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "sklearn": sklearn.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "gpu_name": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available() else None
    ),
    "seeds": [42, 43, 44, 45, 46],
    "datasets": ["Bitcoin Alpha", "Bitcoin OTC"],
    "selection_status": selection_status52,
    "notes": [
        "Frozen test results must not be tuned or overwritten.",
        "Alpha Random/Magnitude use node coverage.",
        "OTC Random/Magnitude use connectivity constraints.",
        "Gradient edge scores exhibit CUDA numerical variability.",
        "Teacher training cost is not included in inference timing.",
        "Bitcoin networks are signed trust proxies, not real SIoT."
    ]
}

with open(
    EXTRA52 / "experiment_environment.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        metadata52, f, indent=2, ensure_ascii=False
    )

# ------------------------------------------------------------
# 3. Locate all project files and verify essentials.
# ------------------------------------------------------------

if not DATA52.is_dir():
    raise FileNotFoundError(DATA52)

if not RESULTS52.is_dir():
    raise FileNotFoundError(RESULTS52)

essential52 = [
    RESULTS52 / "experiment_protocol_v1.json",
    RESULTS52 / "alpha_final_test_results_v1.csv",
    RESULTS52 / "otc_final_test_results_v1.csv",
    RESULTS52 / "paper_final_test_summary_corrected.csv",
    RESULTS52 / "otc_selection_runtime_seed42.csv",
    RESULTS52 / "otc_cpu_gpu_gradient_reproducibility.csv"
]

missing52 = [
    str(path) for path in essential52
    if not path.is_file()
]

if missing52:
    raise FileNotFoundError(
        "Missing critical result files:\n"
        + "\n".join(missing52)
    )

paths52 = []

for directory in [DATA52, RESULTS52, EXTRA52]:
    paths52.extend(
        path for path in directory.rglob("*")
        if path.is_file()
    )

paths52 = sorted(set(paths52))

def sha256_file52(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024), b""
        ):
            digest.update(chunk)
    return digest.hexdigest()

manifest52 = []

for path in paths52:
    manifest52.append({
        "relative_path": str(path.relative_to(ROOT52)),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file52(path)
    })

manifest_path52 = (
    EXTRA52 / "backup_integrity_manifest.csv"
)

pd.DataFrame(manifest52).to_csv(
    manifest_path52, index=False
)

# Include manifest itself in the archive.
paths52.append(manifest_path52)

# ------------------------------------------------------------
# 4. Create ZIP with original directory structure.
# ------------------------------------------------------------

if ARCHIVE52.exists():
    raise FileExistsError(
        f"Backup ZIP already exists: {ARCHIVE52}. "
        "Verify it before replacing."
    )

with zipfile.ZipFile(
    ARCHIVE52,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
    allowZip64=True
) as archive:

    for i, path in enumerate(paths52, 1):

        archive.write(
            path,
            arcname=str(path.relative_to(ROOT52))
        )

        if i % 25 == 0:
            print(
                f"Archived {i}/{len(paths52)} files...",
                flush=True
            )

# ------------------------------------------------------------
# 5. Verify ZIP archive integrity.
# ------------------------------------------------------------

with zipfile.ZipFile(ARCHIVE52, "r") as archive:
    bad_file52 = archive.testzip()
    archived_files52 = archive.namelist()

assert bad_file52 is None, (
    f"Corrupted ZIP entry: {bad_file52}"
)

assert len(archived_files52) == len(paths52)

for path in essential52:
    assert str(path.relative_to(ROOT52)) in archived_files52

print("\n" + "=" * 85)
print("CELL 52 | BACKUP VERIFIED")
print("=" * 85)

print("Archive:", ARCHIVE52)
print("Files:", len(archived_files52))
print(
    "Archive size (MB):",
    round(ARCHIVE52.stat().st_size / 1024**2, 2)
)

print("ZIP integrity: PASS")
print("Critical result files: PASS")

print("\nSelection preservation:")
for dataset, status in selection_status52.items():
    print(f"  {dataset}: {status}")

print("\nSHA256 archive:")
print(sha256_file52(ARCHIVE52))

print("\nBACKUP READY FOR DOWNLOAD")


In [ ]:

import os
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Working directory:", os.getcwd())
print("Backup exists:", os.path.exists(
    "/kaggle/working/TASP_GNN_COMPLETE_BACKUP.zip"
))


In [ ]:

import zipfile
from pathlib import Path

backup = Path("/kaggle/working/TASP_GNN_COMPLETE_BACKUP.zip")

with zipfile.ZipFile(backup) as z:
    names = z.namelist()

print("Total files:", len(names))

print("\n=== MODEL CHECKPOINTS ===")
for name in names:
    if name.endswith((".pt", ".pth")):
        print(name)

print("\n=== NOTEBOOKS AND PYTHON FILES ===")
for name in names:
    if name.endswith((".ipynb", ".py")):
        print(name)

print("\n=== EDGE SCORES AND SELECTION FILES ===")
for name in names:
    if name.endswith(".csv") and any(
        key in name.lower()
        for key in ["score", "importance", "selection", "selected"]
    ):
        print(name)


In [ ]:

from pathlib import Path
import zipfile

backup = Path("/kaggle/working/TASP_GNN_COMPLETE_BACKUP.zip")
output_dir = Path("/kaggle/working")

assert backup.exists(), "Backup ZIP not found!"

with zipfile.ZipFile(backup, "r") as z:
    z.extractall(output_dir)

results_dir = output_dir / "ta_gsnn_results"

print("Backup extracted:", results_dir.exists())
print("Model checkpoints:", len(list(results_dir.rglob("*.pt"))))
print("CSV files:", len(list(results_dir.rglob("*.csv"))))

teacher_path = results_dir / "otc_full_teacher_seed42.pt"
print("Teacher checkpoint available:", teacher_path.exists())


In [ ]:

from pathlib import Path
import torch

path = Path(
    "/kaggle/working/ta_gsnn_results/"
    "otc_full_teacher_seed42.pt"
)

checkpoint = torch.load(
    path,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint type:", type(checkpoint).__name__)

if isinstance(checkpoint, dict):
    print("\n=== TOP-LEVEL KEYS ===")
    for key in checkpoint.keys():
        print(key)

    for key in ["model_state", "model_state_dict", "state_dict"]:
        if key in checkpoint:
            print("\nModel weights found under:", key)
            print("Number of tensors:", len(checkpoint[key]))
            break
else:
    print("Unexpected checkpoint structure")


In [ ]:

from pathlib import Path
import torch

path = Path(
    "/kaggle/working/ta_gsnn_results/"
    "otc_full_teacher_seed42.pt"
)

checkpoint = torch.load(
    path,
    map_location="cpu",
    weights_only=True
)

print("=== CHECKPOINT METADATA ===")
for key in ["method", "dataset", "seed",
            "best_epoch", "protocol_id"]:
    print(f"{key}: {checkpoint.get(key)}")

print("\n=== MODEL PARAMETERS ===")
state = checkpoint["model_state_dict"]

total = 0
for name, tensor in state.items():
    count = tensor.numel()
    total += count
    print(f"{name:45s} {str(tuple(tensor.shape)):20s} {count}")

print("\nTotal trainable-size parameters:", total)


In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path

class SignedLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim * 5, out_dim)
        self.norm = nn.LayerNorm(out_dim)

    @staticmethod
    def aggregate(h, edges, incoming):
        src, dst = edges[0], edges[1]
        receiver = dst if incoming else src
        sender = src if incoming else dst

        aggregated = torch.zeros_like(h)
        aggregated.index_add_(0, receiver, h[sender])

        counts = torch.bincount(
            receiver, minlength=h.shape[0]
        ).to(h.dtype).clamp(min=1)

        return aggregated / counts.unsqueeze(1)

    def forward(self, h, graph):
        pieces = [h]

        for sign in ("pos", "neg"):
            edges = graph[sign]
            pieces.append(self.aggregate(h, edges, True))
            pieces.append(self.aggregate(h, edges, False))

        return self.norm(
            self.linear(torch.cat(pieces, dim=1))
        )


class SignedGraphSAGE(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = SignedLayer(4, 32)
        self.layer2 = SignedLayer(32, 32)
        self.dropout = 0.20

        self.decoder = nn.Sequential(
            nn.Linear(128, 32),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(32, 1)
        )

    def forward(self, graph, edge_pairs):
        h = F.relu(self.layer1(graph["x"], graph))
        h = F.dropout(
            h, p=self.dropout, training=self.training
        )

        h = F.relu(self.layer2(h, graph))
        h = F.dropout(
            h, p=self.dropout, training=self.training
        )

        u = h[edge_pairs[:, 0]]
        v = h[edge_pairs[:, 1]]

        pair = torch.cat(
            [u, v, torch.abs(u-v), u*v], dim=1
        )

        return self.decoder(pair).squeeze(-1)


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

path = Path(
    "/kaggle/working/ta_gsnn_results/"
    "otc_full_teacher_seed42.pt"
)

checkpoint = torch.load(
    path, map_location="cpu", weights_only=True
)

teacher = SignedGraphSAGE().to(device)

teacher.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

teacher.eval()

for parameter in teacher.parameters():
    parameter.requires_grad_(False)

print("Teacher loaded successfully!")
print("Device:", device)
print("Dataset:", checkpoint["dataset"])
print("Seed:", checkpoint["seed"])
print("Best epoch:", checkpoint["best_epoch"])
print("Parameters:", sum(
    p.numel() for p in teacher.parameters()
))


In [ ]:

from pathlib import Path
import pandas as pd

matches = list(Path("/kaggle").rglob("siot_trust_edges.csv"))

if not matches:
    print("CSV not found. Please upload it to Kaggle.")
else:
    path = matches[0]
    df = pd.read_csv(path)

    print("File:", path)
    print("Edges:", len(df))
    print("Nodes:", len(set(df.source) | set(df.target)))
    print("Positive:", (df.rating > 0).sum())
    print("Negative:", (df.rating < 0).sum())
    print("Duplicates:", df.duplicated(["source", "target"]).sum())
    print("\nFirst five rows:")
    print(df.head())


In [ ]:

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path

# 1. Load Mininet trust graph
path = Path(
    "/kaggle/input/datasets/mahomaattorney/"
    "siot-trust-edges-csv/siot_trust_edges.csv"
)
df_siot = pd.read_csv(path)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

nodes = sorted(set(df_siot.source) | set(df_siot.target))
node_map = {node: i for i, node in enumerate(nodes)}
n = len(nodes)

src = df_siot.source.map(node_map).to_numpy()
dst = df_siot.target.map(node_map).to_numpy()
positive = df_siot.rating.to_numpy() > 0

# 2. Signed degree features
features = np.zeros((n, 4), dtype=np.float32)

np.add.at(features[:, 0], src[positive], 1)
np.add.at(features[:, 1], src[~positive], 1)
np.add.at(features[:, 2], dst[positive], 1)
np.add.at(features[:, 3], dst[~positive], 1)

graph_siot = {
    "x": torch.tensor(
        np.log1p(features), dtype=torch.float32,
        device=device
    ),
    "pos": torch.tensor(
        np.stack([src[positive], dst[positive]]),
        dtype=torch.long, device=device
    ),
    "neg": torch.tensor(
        np.stack([src[~positive], dst[~positive]]),
        dtype=torch.long, device=device
    )
}

pairs_siot = torch.tensor(
    np.column_stack([src, dst]),
    dtype=torch.long, device=device
)

labels_siot = torch.tensor(
    (~positive).astype(np.float32),
    device=device
)

# 3. Train a separate pilot teacher
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

pilot_teacher = SignedGraphSAGE().to(device)

negative_count = float(labels_siot.sum())
positive_count = float(len(labels_siot) - negative_count)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        positive_count / negative_count,
        device=device
    )
)

optimizer = torch.optim.Adam(
    pilot_teacher.parameters(),
    lr=0.005,
    weight_decay=1e-4
)

for epoch in range(1, 201):
    pilot_teacher.train()
    optimizer.zero_grad()

    logits = pilot_teacher(graph_siot, pairs_siot)
    loss = criterion(logits, labels_siot)

    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(
            f"Epoch {epoch:3d}/200 | "
            f"Training loss: {loss.item():.6f}"
        )

# 4. Save pilot separately
output_path = Path(
    "/kaggle/working/siot_pilot_teacher.pt"
)

torch.save(
    {
        "model_state_dict": pilot_teacher.state_dict(),
        "dataset": "siot_synthetic_10_nodes",
        "seed": 42,
        "purpose": "in_sample_feasibility_pilot",
        "epochs": 200
    },
    output_path
)

print("\nPilot teacher saved:", output_path)
print("Nodes:", n)
print("Edges:", len(df_siot))
print("Negative edges:", int(negative_count))
print("Original OTC teacher: unchanged")


In [ ]:

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from pathlib import Path

# Recover the separately trained SIoT pilot
pilot_path = Path("/kaggle/working/siot_pilot_teacher.pt")
ckpt = torch.load(pilot_path, map_location=device, weights_only=True)

pilot_teacher = SignedGraphSAGE().to(device)
pilot_teacher.load_state_dict(ckpt["model_state_dict"], strict=True)
pilot_teacher.eval()

state = {k: v.detach() for k, v in pilot_teacher.state_dict().items()}

s = torch.tensor(src, dtype=torch.long, device=device)
d = torch.tensor(dst, dtype=torch.long, device=device)
pos = torch.tensor(positive, dtype=torch.bool, device=device)
N = len(nodes)

def weighted_forward_siot(weights):
    # Differentiable signed-degree features
    cols = []
    for indices, w in [
        (s[pos], weights[pos]),
        (s[~pos], weights[~pos]),
        (d[pos], weights[pos]),
        (d[~pos], weights[~pos])
    ]:
        col = torch.zeros(N, device=device).index_add(0, indices, w)
        cols.append(col)

    h = torch.log1p(torch.stack(cols, dim=1))

    # Exactly the four directed signed aggregation channels
    for layer in ("layer1", "layer2"):
        pieces = [h]

        for mask in (pos, ~pos):
            ss, dd, ww = s[mask], d[mask], weights[mask]

            for incoming in (True, False):
                receiver = dd if incoming else ss
                sender = ss if incoming else dd

                numerator = torch.zeros_like(h).index_add(
                    0, receiver, h[sender] * ww.unsqueeze(1)
                )
                denominator = torch.zeros(
                    N, device=device
                ).index_add(0, receiver, ww)

                pieces.append(
                    numerator /
                    denominator.clamp(min=1).unsqueeze(1)
                )

        z = F.linear(
            torch.cat(pieces, dim=1),
            state[f"{layer}.linear.weight"],
            state[f"{layer}.linear.bias"]
        )
        h = F.relu(F.layer_norm(
            z, (32,),
            state[f"{layer}.norm.weight"],
            state[f"{layer}.norm.bias"],
            eps=1e-5
        ))

    u, v = h[pairs_siot[:, 0]], h[pairs_siot[:, 1]]
    pair = torch.cat([u, v, torch.abs(u-v), u*v], dim=1)

    hidden = F.relu(F.linear(
        pair, state["decoder.0.weight"],
        state["decoder.0.bias"]
    ))

    return F.linear(
        hidden,
        state["decoder.3.weight"],
        state["decoder.3.bias"]
    ).squeeze(-1)

# Verify that weighted implementation reproduces Teacher
weights = torch.ones(
    len(df_siot), device=device, requires_grad=True
)

with torch.no_grad():
    original = pilot_teacher(graph_siot, pairs_siot)
    reconstructed = weighted_forward_siot(weights.detach())
    max_diff = (original - reconstructed).abs().max().item()

print("Maximum reconstruction difference:", max_diff)
assert max_diff < 1e-4, "Model reconstruction mismatch!"

# Compute task-aware importance: -dL/dw
criterion_pilot = torch.nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        float((~positive).sum() and positive.sum() / (~positive).sum()),
        device=device
    )
)

loss = criterion_pilot(
    weighted_forward_siot(weights),
    labels_siot
)

gradient = torch.autograd.grad(loss, weights)[0]
importance = (-gradient).detach().cpu().numpy()

assert np.isfinite(importance).all()

# Connectivity-preserving maximum-score spanning forest
parent = list(range(N))
size = [1] * N

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(a, b):
    a, b = find(a), find(b)
    if a == b:
        return False
    if size[a] < size[b]:
        a, b = b, a
    parent[b] = a
    size[a] += size[b]
    return True

budget = round(0.5 * len(df_siot))
ranking = np.argsort(-importance, kind="stable")
selected = np.zeros(len(df_siot), dtype=bool)

for i in ranking:
    if union(int(src[i]), int(dst[i])):
        selected[i] = True

forest_edges = int(selected.sum())
assert forest_edges <= budget

remaining = ranking[~selected[ranking]]
selected[remaining[:budget - forest_edges]] = True

components = len({find(i) for i in range(N)})
assert selected.sum() == budget
assert components == 1

# Export Mininet-compatible policy
policy = df_siot.loc[
    selected, ["source", "target", "rating"]
].copy()

output = Path("/kaggle/working/siot_policy_tasp.csv")
policy.to_csv(output, index=False)

# Save audit details separately
audit = df_siot.copy()
audit["importance"] = importance
audit["selected"] = selected
audit.to_csv(
    "/kaggle/working/siot_tasp_importance_audit.csv",
    index=False
)

print("\n=== TASP-GNN MININET PILOT ===")
print("Initial edges:", len(df_siot))
print("Selected edges:", int(selected.sum()))
print("Positive selected:", int((policy.rating > 0).sum()))
print("Negative selected:", int((policy.rating < 0).sum()))
print("Forest edges:", forest_edges)
print("Weak components:", components)
print("Finite importance scores:", bool(np.isfinite(importance).all()))
print("Policy saved:", output)
print("Audit saved successfully")


In [ ]:

from pathlib import Path

p = Path("/kaggle/working/siot_policy_tasp.csv")

print("File exists:", p.exists())

if p.exists():
    print("File size:", p.stat().st_size, "bytes")
    print("File location:", p)


In [ ]:

from pathlib import Path

print("=== BITCOIN OTC DATA FILES ===")

for base in [
    Path("/kaggle/input"),
    Path("/kaggle/working")
]:
    for path in base.rglob("*"):
        if path.is_file():
            name = path.name.lower()
            if "otc" in name and path.suffix.lower() in [
                ".csv", ".txt", ".gz"
            ]:
                print(path)


In [ ]:

import pandas as pd
from pathlib import Path

data_dir = Path("/kaggle/working/ta_gsnn_data")
results_dir = Path("/kaggle/working/ta_gsnn_results")

files = {
    "CONTEXT": data_dir / "bitcoin_otc_context.csv",
    "TRAIN": data_dir / "bitcoin_otc_train.csv",
    "VALIDATION": data_dir / "bitcoin_otc_validation.csv",
    "TEST": data_dir / "bitcoin_otc_test.csv",
    "TASP_SELECTION": results_dir /
        "bitcoin_otc_frozen_test_selections" /
        "bitcoin_otc_seed42_task_aware_connectivity_test_history.csv",
    "STRUCTURAL_SELECTION": results_dir /
        "bitcoin_otc_frozen_test_selections" /
        "bitcoin_otc_seed42_structural_connectivity_test_history.csv",
}

for name, path in files.items():
    print("\n===", name, "===")

    if not path.exists():
        print("NOT FOUND")
        continue

    df = pd.read_csv(path)

    print("Rows:", len(df))
    print("Columns:", list(df.columns))
    print("First two rows:")
    print(df.head(2).to_string(index=False))


In [ ]:

import pandas as pd
import numpy as np
import torch
from pathlib import Path

D = Path("/kaggle/working/ta_gsnn_data")
R = Path("/kaggle/working/ta_gsnn_results")
S = R / "bitcoin_otc_frozen_test_selections"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Historical edges only: no Test edges in the graph
history = pd.concat([
    pd.read_csv(D / "bitcoin_otc_context.csv"),
    pd.read_csv(D / "bitcoin_otc_train.csv"),
    pd.read_csv(D / "bitcoin_otc_validation.csv")
], ignore_index=True)

test = pd.read_csv(D / "bitcoin_otc_test.csv")

assert history.timestamp.max() < test.timestamp.min()

# Match original OTC node indexing
train_history = pd.concat([
    pd.read_csv(D / "bitcoin_otc_context.csv"),
    pd.read_csv(D / "bitcoin_otc_train.csv")
], ignore_index=True)

known = sorted(
    set(train_history.source) | set(train_history.target)
)

mapping = {v: i for i, v in enumerate(known)}

validation = pd.read_csv(D / "bitcoin_otc_validation.csv")

for v in sorted(
    (set(validation.source) | set(validation.target))
    - set(mapping)
):
    mapping[v] = len(mapping)

for v in sorted(
    (set(test.source) | set(test.target))
    - set(mapping)
):
    mapping[v] = len(mapping)

N = len(mapping)

def make_graph(frame):
    src = frame.source.map(mapping).to_numpy(dtype=np.int64)
    dst = frame.target.map(mapping).to_numpy(dtype=np.int64)
    pos = frame.rating.to_numpy() > 0

    features = np.zeros((N, 4), dtype=np.float32)
    np.add.at(features[:, 0], src[pos], 1)
    np.add.at(features[:, 1], src[~pos], 1)
    np.add.at(features[:, 2], dst[pos], 1)
    np.add.at(features[:, 3], dst[~pos], 1)

    return {
        "x": torch.tensor(
            np.log1p(features), device=device
        ),
        "pos": torch.tensor(
            np.stack([src[pos], dst[pos]]),
            dtype=torch.long, device=device
        ),
        "neg": torch.tensor(
            np.stack([src[~pos], dst[~pos]]),
            dtype=torch.long, device=device
        )
    }

pairs = torch.tensor(
    np.column_stack([
        test.source.map(mapping).to_numpy(dtype=np.int64),
        test.target.map(mapping).to_numpy(dtype=np.int64)
    ]),
    dtype=torch.long, device=device
)

methods = {
    "full": (
        "full",
        "otc_full_teacher_seed42.pt"
    ),
    "random": (
        "random_connectivity",
        "otc_random_connectivity_seed42.pt"
    ),
    "magnitude": (
        "magnitude_connectivity",
        "otc_magnitude_connectivity_seed42.pt"
    ),
    "structural": (
        "structural_connectivity",
        "otc_structural_connectivity_seed42.pt"
    ),
    "tasp": (
        "task_aware_connectivity",
        "otc_task_aware_retrained_seed42.pt"
    )
}

output = test[["source", "target", "timestamp"]].copy()

for label, (method, ckpt_name) in methods.items():
    selection_path = S / (
        f"bitcoin_otc_seed42_{method}_test_history.csv"
    )

    selected = pd.read_csv(selection_path)

    checkpoint = torch.load(
        R / ckpt_name,
        map_location=device,
        weights_only=True
    )

    model = SignedGraphSAGE().to(device)
    model.load_state_dict(
        checkpoint["model_state_dict"],
        strict=True
    )
    model.eval()

    graph = make_graph(selected)

    with torch.no_grad():
        negative_probability = torch.sigmoid(
            model(graph, pairs)
        ).cpu().numpy()

    output[f"{label}_p_negative"] = negative_probability

    # Fixed pilot threshold; not fitted to Test labels
    output[f"{label}_allow"] = (
        negative_probability < 0.5
    ).astype(int)

    print(
        label,
        "| graph edges:", len(selected),
        "| predicted allowed:",
        int(output[f"{label}_allow"].sum())
    )

# Labels used only for later audit, never as model inputs
output["actual_negative"] = (
    test.rating.to_numpy() < 0
).astype(int)

destination = Path(
    "/kaggle/working/otc_mininet_test_predictions_seed42.csv"
)

output.to_csv(destination, index=False)

print("\nPredictions saved:", destination)
print("Test requests:", len(output))
print("Methods:", len(methods))
print("Missing predictions:",
      int(output.isna().sum().sum()))


In [ ]:

import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    confusion_matrix
)

pred = pd.read_csv(
    "/kaggle/working/otc_mininet_test_predictions_seed42.csv"
)

y = pred["actual_negative"].astype(int)

print("=== UNSEEN TEST EVALUATION ===")
print("Total requests:", len(pred))
print("Actual positive:", int((y == 0).sum()))
print("Actual negative:", int((y == 1).sum()))

for method in ["full", "random", "magnitude",
               "structural", "tasp"]:

    prob = pred[f"{method}_p_negative"]
    predicted_negative = 1 - pred[f"{method}_allow"]

    tn, fp, fn, tp = confusion_matrix(
        y, predicted_negative, labels=[0, 1]
    ).ravel()

    print(
        f"{method:11s} "
        f"PR-AUC={average_precision_score(y, prob):.4f} "
        f"ROC-AUC={roc_auc_score(y, prob):.4f} "
        f"false_allow={fn} "
        f"correct_block={tp}"
    )

print("\n=== MININET HOST-SIZE FEASIBILITY ===")

history = pd.concat([
    pd.read_csv(
        "/kaggle/working/ta_gsnn_data/bitcoin_otc_context.csv"
    ),
    pd.read_csv(
        "/kaggle/working/ta_gsnn_data/bitcoin_otc_train.csv"
    ),
    pd.read_csv(
        "/kaggle/working/ta_gsnn_data/bitcoin_otc_validation.csv"
    )
], ignore_index=True)

# Hosts selected only from historical degree;
# future Test labels are not used in selection.
degree = pd.concat([
    history.source, history.target
]).value_counts()

for k in [10, 20, 30, 50, 100]:
    hosts = set(degree.head(k).index)

    subset = pred[
        pred.source.isin(hosts)
        & pred.target.isin(hosts)
    ]

    print(
        f"{k:3d} hosts | "
        f"Test requests: {len(subset):4d}"
    )


In [ ]:

import pandas as pd
from pathlib import Path

R = Path("/kaggle/working/ta_gsnn_results")

for filename in [
    "otc_final_test_results_v1.csv",
    "otc_final_test_summary_v1.csv"
]:
    path = R / filename
    print("\n===", filename, "===")

    df = pd.read_csv(path)
    print("Columns:", list(df.columns))
    print(df.head(10).to_string(index=False))


In [ ]:

import pandas as pd
from pathlib import Path

D = Path("/kaggle/working/ta_gsnn_data")
P = Path(
    "/kaggle/working/otc_mininet_test_predictions_seed42.csv"
)

history = pd.concat([
    pd.read_csv(D / "bitcoin_otc_context.csv"),
    pd.read_csv(D / "bitcoin_otc_train.csv"),
    pd.read_csv(D / "bitcoin_otc_validation.csv")
], ignore_index=True)

pred = pd.read_csv(P)

# Deterministic host selection using historical degree only
degree = pd.concat(
    [history["source"], history["target"]],
    ignore_index=True
).value_counts()

host_nodes = (
    degree.rename_axis("bitcoin_node")
    .reset_index(name="historical_degree")
    .sort_values(
        ["historical_degree", "bitcoin_node"],
        ascending=[False, True]
    )
    .head(30)
    .copy()
)

host_nodes["mininet_host"] = [
    f"h{i}" for i in range(1, len(host_nodes) + 1)
]

selected_nodes = set(host_nodes["bitcoin_node"])

pilot = pred.loc[
    pred["source"].isin(selected_nodes)
    & pred["target"].isin(selected_nodes)
].copy()

print("=== BITCOIN OTC OPENFLOW PILOT ===")
print("Mininet hosts:", len(host_nodes))
print("Test requests:", len(pilot))
print("Actual positive:", int((pilot.actual_negative == 0).sum()))
print("Actual negative:", int((pilot.actual_negative == 1).sum()))
print(
    "Distinct directed pairs:",
    len(pilot.drop_duplicates(["source", "target"]))
)

print("\n=== MODEL DECISIONS ===")
for method in ["full", "random", "magnitude",
               "structural", "tasp"]:
    allowed = pilot[f"{method}_allow"].astype(bool)

    false_allow = int(
        (allowed & (pilot.actual_negative == 1)).sum()
    )

    correct_allow = int(
        (allowed & (pilot.actual_negative == 0)).sum()
    )

    correct_block = int(
        (~allowed & (pilot.actual_negative == 1)).sum()
    )

    print(
        f"{method:11s} "
        f"allowed={int(allowed.sum()):3d} "
        f"correct_allow={correct_allow:3d} "
        f"false_allow={false_allow:3d} "
        f"correct_block={correct_block:3d}"
    )

# Check repeated pairs before constructing OpenFlow rules
duplicates = pilot.duplicated(
    ["source", "target"], keep=False
).sum()

print("\nRows with repeated directed pairs:", int(duplicates))

# Export without overwriting existing experiments
host_nodes.to_csv(
    "/kaggle/working/otc_mininet_30host_mapping.csv",
    index=False
)

pilot.to_csv(
    "/kaggle/working/otc_mininet_30host_requests.csv",
    index=False
)

print("\nPilot and host mapping saved.")



In [ ]:

from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import FileLink, display

working = Path("/kaggle/working")

filenames = [
    "otc_mininet_30host_mapping.csv",
    "otc_mininet_30host_requests.csv"
]

archive = working / "otc_mininet_30host_pilot.zip"

with ZipFile(archive, "w", ZIP_DEFLATED) as z:
    for name in filenames:
        path = working / name
        assert path.exists(), f"Missing: {name}"
        z.write(path, arcname=name)

print("Archive created successfully")
print("Files:", len(filenames))
print("Size:", archive.stat().st_size, "bytes")

display(FileLink(str(archive)))


In [ ]:

from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import FileLink, display

working = Path("/kaggle/working")

filenames = [
    "otc_mininet_30host_mapping.csv",
    "otc_mininet_30host_requests.csv"
]

archive = working / "otc_mininet_30host_pilot.zip"

with ZipFile(archive, "w", ZIP_DEFLATED) as z:
    for name in filenames:
        path = working / name
        assert path.exists(), f"Missing: {name}"
        z.write(path, arcname=name)

print("Archive created successfully")
print("Files:", len(filenames))
print("Size:", archive.stat().st_size, "bytes")

display(FileLink(str(archive)))



# Phase II — Trust-Aware Adaptive Graph Sparsification for SIoT

## Research Direction

**Original supervisor-proposed topic:**

Trust-Aware Adaptive Graph Sparsification Using
Graph Neural Networks for Social Internet of Things

**Phase II objective:**

Develop and evaluate a GNN-guided graph sparsification
method that explicitly incorporates historical trust
information and adapts edge retention to local graph
conditions and temporal changes.

This phase is a new research investigation, not a
renaming of the previously evaluated TASP-GNN method.

---

## 1. Previous Work — Phase I (Completed)

Phase I investigated:

**TASP-GNN: Task-Aware and Connectivity-Preserving
Graph Sparsification for Signed Trust Prediction**

Completed experiments:

- Bitcoin Alpha and Bitcoin OTC datasets
- Chronological context/train/validation/test partitions
- Signed and direction-aware GraphSAGE predictor
- Gradient-based teacher edge importance
- Connectivity-preserving sparsification
- Full, Random, Magnitude, Structural and TASP baselines
- Five random seeds: 42, 43, 44, 45 and 46
- Predictive and structural evaluations
- Runtime measurements for selected operations
- Synthetic Mininet and real-data OpenFlow experiments

Phase I artifacts and results must remain unchanged.

The existing TASP-GNN method will serve as an internal
baseline for Phase II.

---

## 2. Phase II Research Questions

**RQ1 — Trust Awareness**

Does explicit historical trust information improve
GNN-guided edge selection compared with task-only
or structural selection?

**RQ2 — Adaptive Sparsification**

Does adaptive allocation of edge-retention budgets
improve prediction at the same global edge budget?

**RQ3 — Temporal Adaptation**

Can the sparsifier respond to changes in historical
trust relationships without future-information leakage?

**RQ4 — Structural Preservation**

Can adaptive selection preserve the weakly connected
component partition of the observed historical graph?

**RQ5 — Computational Cost**

Do the predictive or structural benefits justify
the additional scoring and adaptation costs?

**RQ6 — SIoT Applicability**

How does the method perform in a documented SIoT
trust-network scenario?

---

## 3. Experimental Protocol

Initial benchmark datasets:

- Bitcoin Alpha
- Bitcoin OTC

Reuse the Phase I chronological splits.

Random seeds:

- Pilot experiments: 42
- Final experiments: 42, 43, 44, 45, 46

Primary predictive metrics:

- Negative-class Average Precision (AP)
- ROC-AUC

Additional metrics:

- Retained edge fraction
- Weakly connected component preservation
- Node coverage
- Local edge-budget allocation
- Temporal edge-selection stability
- Edge-scoring runtime
- Selection runtime
- Student training and inference runtime
- Peak memory usage, when available

All hyperparameter selection must use
training/validation data only.

The test partition must remain untouched
until the final frozen evaluation.

---

## 4. Planned Experiments

E0 — Environment recovery and Phase I baseline audit

E1 — Trust-aware versus trust-agnostic edge scoring

E2 — Fixed versus adaptive local edge allocation

E3 — Temporal adaptation under chronological updates

E4 — Ablation study

E5 — Comparison with learned sparsification baselines

E6 — Retention-budget sensitivity analysis

E7 — SIoT-specific evaluation

E8 — End-to-end computational cost analysis

E9 — Optional trust-manipulation robustness analysis

---

## 5. Experiment Management Rules

1. Do not overwrite Phase I artifacts.

2. Save all new artifacts under:
   /kaggle/working/phase2_adaptive_siot/

3. Keep separate outputs for datasets, seeds,
   experimental variants and checkpoints.

4. Record configurations before training.

5. Use the same chronological splits and test
   protocol across comparable methods.

6. Distinguish validation-selected results
   from final held-out test results.

7. Report unsuccessful experiments and negative results.

8. Do not claim improvement before the final
   multi-seed evaluation.

9. Do not introduce new model components without
   a corresponding hypothesis and ablation.

10. Separate real SIoT datasets from synthetic
    SIoT simulation results.

---

## 6. Initial Execution Order

Step 1: Recover environment and verify data splits.

Step 2: Implement a small trust-aware edge scorer.

Step 3: Compare trust-aware and trust-agnostic scoring.

Step 4: Implement adaptive local edge allocation.

Step 5: Run pilot comparisons on Alpha and OTC
        using seed 42 and a 50% edge budget.

Step 6: Review validation results before proceeding
        to temporal adaptation and five-seed evaluation.

---

## 7. Status

Phase II: NOT STARTED

Current task: E0 — Environment and artifact recovery.

No new performance claims have been established.


In [ ]:

# ============================================================
# PHASE II — E0: ENVIRONMENT & BASELINE AUDIT
# Trust-Aware Adaptive Graph Sparsification for SIoT
# ============================================================
# Read-only audit of Phase I artifacts.
# No model training, no changes to previous results.
# New files are saved ONLY under phase2_adaptive_siot.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import os
import sys
import json
import csv
import platform
import zipfile
import tarfile
import importlib.metadata as metadata

PHASE2_ROOT = Path(
    "/kaggle/working/phase2_adaptive_siot"
)

OUTPUT_DIRS = [
    "configs",
    "data_audit",
    "E0_baseline",
    "E1_trust_scoring",
    "E2_adaptive_allocation",
    "E3_temporal",
    "E4_ablation",
    "E5_baselines",
    "E6_budget_sensitivity",
    "E7_siot",
    "E8_runtime",
    "E9_robustness",
    "final_reports",
]

for folder in OUTPUT_DIRS:
    (PHASE2_ROOT / folder).mkdir(
        parents=True, exist_ok=True
    )

AUDIT_DIR = PHASE2_ROOT / "E0_baseline"

print("=" * 72)
print("PHASE II — E0: ENVIRONMENT & BASELINE AUDIT")
print("=" * 72)

# ------------------------------------------------------------
# 1. Environment
# ------------------------------------------------------------

def get_version(package):
    try:
        return metadata.version(package)
    except metadata.PackageNotFoundError:
        return "not installed"

env = {
    "timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "numpy": get_version("numpy"),
    "pandas": get_version("pandas"),
    "scikit-learn": get_version("scikit-learn"),
    "torch": get_version("torch"),
    "torch-geometric": get_version("torch-geometric"),
    "networkx": get_version("networkx"),
}

try:
    import torch

    env["cuda_available"] = torch.cuda.is_available()
    env["torch_cuda_version"] = torch.version.cuda

    if torch.cuda.is_available():
        env["gpu_name"] = torch.cuda.get_device_name(0)
        props = torch.cuda.get_device_properties(0)
        env["gpu_memory_gb"] = round(
            props.total_memory / (1024 ** 3), 2
        )
    else:
        env["gpu_name"] = None
        env["gpu_memory_gb"] = None

except Exception as exc:
    env["torch_error"] = repr(exc)
    env["cuda_available"] = False

print("\n[1] ENVIRONMENT")
for key, value in env.items():
    print(f"  {key}: {value}")

# ------------------------------------------------------------
# 2. Input roots and artifact discovery
# ------------------------------------------------------------

INPUT_ROOTS = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
]

ARCHIVE_EXTENSIONS = (
    ".zip",
    ".tar",
    ".tar.gz",
    ".tgz",
)

DATA_EXTENSIONS = {
    ".csv", ".tsv", ".json", ".jsonl",
    ".parquet", ".pkl", ".pickle",
    ".pt", ".pth", ".npz", ".npy",
    ".txt",
}

KEYWORDS = (
    "alpha",
    "otc",
    "bitcoin",
    "tasp",
    "ta_gsnn",
    "split",
    "checkpoint",
    "seed",
    "summary",
    "runtime",
    "selection",
    "history",
    "protocol",
    "model",
    "trust",
)

def is_archive(path):
    name = path.name.lower()
    return any(
        name.endswith(ext)
        for ext in ARCHIVE_EXTENSIONS
    )

def candidate_file(path):
    name = path.name.lower()

    return (
        path.suffix.lower() in DATA_EXTENSIONS
        or is_archive(path)
        or any(word in name for word in KEYWORDS)
    )

def scan_files(root):
    found = []

    if not root.exists():
        return found

    for current_dir, dirs, filenames in os.walk(root):
        current = Path(current_dir)

        # Never scan or modify new Phase II outputs.
        dirs[:] = [
            name for name in dirs
            if (current / name) != PHASE2_ROOT
        ]

        for filename in filenames:
            path = current / filename

            if PHASE2_ROOT in path.parents:
                continue

            if candidate_file(path):
                try:
                    size_mb = round(
                        path.stat().st_size / 1048576, 3
                    )
                except OSError:
                    size_mb = None

                found.append({
                    "path": str(path),
                    "name": filename,
                    "size_mb": size_mb,
                    "is_archive": is_archive(path),
                })

    return found

artifacts = []

for root in INPUT_ROOTS:
    artifacts.extend(scan_files(root))

artifacts.sort(key=lambda x: x["path"])

print("\n[2] ARTIFACT DISCOVERY")
print(f"Candidate files found: {len(artifacts)}")

for item in artifacts[:100]:
    print(
        f"  {item['size_mb']} MB | "
        f"{item['path']}"
    )

if len(artifacts) > 100:
    print(
        f"  ... {len(artifacts)-100} more files "
        "recorded in the audit JSON."
    )

# ------------------------------------------------------------
# 3. Inspect archives without extracting them
# ------------------------------------------------------------

archives = [
    item for item in artifacts
    if item["is_archive"]
]

archive_reports = []

for item in archives:
    path = Path(item["path"])

    report = {
        "path": str(path),
        "entries": [],
        "error": None,
    }

    try:
        if zipfile.is_zipfile(path):
            with zipfile.ZipFile(path) as zf:
                report["entries"] = [
                    x.filename
                    for x in zf.infolist()
                    if not x.is_dir()
                ]

        elif tarfile.is_tarfile(path):
            with tarfile.open(path, "r:*") as tf:
                report["entries"] = [
                    x.name
                    for x in tf.getmembers()
                    if x.isfile()
                ]

        else:
            report["error"] = "Unsupported archive format"

    except Exception as exc:
        report["error"] = repr(exc)

    archive_reports.append(report)

print("\n[3] ARCHIVE INSPECTION")
for report in archive_reports:
    print(
        f"\n  {report['path']}\n"
        f"  Entries: {len(report['entries'])}"
    )

    for name in report["entries"][:12]:
        print("   -", name)

    if report["error"]:
        print("  ERROR:", report["error"])

# ------------------------------------------------------------
# 4. Check expected Phase I dataset metadata
# ------------------------------------------------------------

EXPECTED = {
    "bitcoin_alpha": {
        "all_interactions": 24186,
        "all_nodes": 3783,
        "test_history_edges": 21772,
        "sparse_test_history_edges": 10886,
        "test_history_nodes": 3498,
        "test_history_weak_components": 4,
    },
    "bitcoin_otc": {
        "all_interactions": 35592,
        "all_nodes": 5881,
        "test_history_edges": 32033,
        "sparse_test_history_edges": 16016,
        "test_history_nodes": 5437,
        "test_history_weak_components": 3,
    },
}

# These are reference counts from Phase I.
# They are NOT automatically verified by this E0 cell.

print("\n[4] PHASE I REFERENCE METADATA")

for dataset, info in EXPECTED.items():
    print(f"\n  {dataset}")
    for key, value in info.items():
        print(f"    {key}: {value}")

# ------------------------------------------------------------
# 5. Detect likely datasets, checkpoints, results
# ------------------------------------------------------------

all_names = [
    item["path"].lower()
    for item in artifacts
]

for archive in archive_reports:
    all_names.extend(
        name.lower()
        for name in archive["entries"]
    )

search_groups = {
    "alpha_artifacts": ["alpha"],
    "otc_artifacts": ["otc"],
    "checkpoints": [
        "checkpoint", ".pt", ".pth"
    ],
    "seed_results": [
        "seed42", "seed_42", "final_test",
        "test_summary"
    ],
    "runtime_artifacts": [
        "runtime", "timing"
    ],
    "protocol_artifacts": [
        "protocol", "split"
    ],
}

detected = {}

for group, words in search_groups.items():
    matches = sorted({
        name for name in all_names
        if any(word in name for word in words)
    })

    detected[group] = {
        "count": len(matches),
        "examples": matches[:15],
    }

print("\n[5] ARTIFACT CATEGORIES")

for group, info in detected.items():
    print(f"\n  {group}: {info['count']} matches")

    for example in info["examples"][:5]:
        print("   -", example)

# ------------------------------------------------------------
# 6. Freeze Phase II initial protocol
# ------------------------------------------------------------

protocol = {
    "phase": "Phase II",
    "experiment": "E0",
    "research_topic": (
        "Trust-Aware Adaptive Graph Sparsification "
        "Using Graph Neural Networks for SIoT"
    ),
    "datasets": [
        "bitcoin_alpha",
        "bitcoin_otc"
    ],
    "pilot_seed": 42,
    "final_seeds": [42, 43, 44, 45, 46],
    "pilot_global_edge_retention": 0.50,
    "primary_metric": (
        "negative-class average precision"
    ),
    "secondary_metric": "ROC-AUC",
    "temporal_splits": "reuse Phase I",
    "selection_data": "train/validation only",
    "test_policy": (
        "Do not use the final test labels "
        "to select Phase II methods or hyperparameters."
    ),
    "phase1_artifacts": "read-only",
    "status": "environment audit only",
}

# ------------------------------------------------------------
# 7. Save audit files in Phase II only
# ------------------------------------------------------------

audit = {
    "environment": env,
    "expected_phase1_metadata": EXPECTED,
    "artifacts": artifacts,
    "archives": archive_reports,
    "detected_categories": detected,
    "phase2_protocol": protocol,
}

with open(
    AUDIT_DIR / "e0_environment_audit.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        audit, f, indent=2,
        ensure_ascii=False
    )

with open(
    AUDIT_DIR / "e0_artifact_inventory.csv",
    "w",
    newline="",
    encoding="utf-8"
) as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "path", "name", "size_mb", "is_archive"
        ]
    )
    writer.writeheader()
    writer.writerows(artifacts)

with open(
    PHASE2_ROOT / "configs" / "phase2_protocol.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(protocol, f, indent=2)

# ------------------------------------------------------------
# 8. Final audit status
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("E0 AUDIT FINISHED")
print("=" * 72)

print("GPU available:", env["cuda_available"])
print("Candidate files:", len(artifacts))
print("Archives inspected:", len(archive_reports))

for group, info in detected.items():
    print(f"{group}: {info['count']} matches")

print("\nAudit saved to:")
print(AUDIT_DIR / "e0_environment_audit.json")

print("\nSTATUS:")
print("Environment and file discovery completed.")
print("Dataset integrity has NOT yet been verified.")
print("Phase I models have NOT been retrained.")
print("Phase I files have NOT been modified.")
print("Phase II experiments have NOT started.")

print("\nNEXT: E0 dataset/checkpoint integrity verification.")


In [ ]:

# ============================================================
# PHASE II — E0.2: DATA & CHECKPOINT INTEGRITY
# Read-only verification of Phase I
# ============================================================

from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np
import json
import zipfile
import hashlib

ROOT = Path("/kaggle/working")
DATA = ROOT / "ta_gsnn_data"
RESULTS = ROOT / "ta_gsnn_results"
PHASE2 = ROOT / "phase2_adaptive_siot"
OUT = PHASE2 / "E0_baseline"
OUT.mkdir(parents=True, exist_ok=True)

EXPECTED_SPLITS = {
    "alpha": {
        "context": 14496,
        "train": 4843,
        "validation": 2433,
        "test": 2414,
    },
    "otc": {
        "context": 21355,
        "train": 7119,
        "validation": 3559,
        "test": 3559,
    },
}

REQUIRED_COLUMNS = {
    "source", "target", "rating", "timestamp"
}

report = {
    "datasets": {},
    "checkpoints": {},
    "archive": {},
    "issues": [],
}

def check(condition, message):
    if not condition:
        report["issues"].append(message)
        print("FAIL:", message)
    return bool(condition)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(chunk)
    return h.hexdigest()

def row_signature(df):
    """Order-independent, duplicate-aware row comparison."""
    cols = ["source", "target", "rating", "timestamp"]
    return Counter(
        tuple(row)
        for row in df[cols].itertuples(
            index=False, name=None
        )
    )

print("=" * 68)
print("E0.2 — DATASET INTEGRITY")
print("=" * 68)

# ------------------------------------------------------------
# 1. Validate chronological data splits
# ------------------------------------------------------------

for dataset in ("alpha", "otc"):
    print(f"\nDATASET: BITCOIN {dataset.upper()}")
    frames = {}
    metadata = {}

    for split, expected_n in EXPECTED_SPLITS[dataset].items():
        path = DATA / f"bitcoin_{dataset}_{split}.csv"

        if not check(
            path.exists(), f"Missing file: {path}"
        ):
            continue

        df = pd.read_csv(path)
        frames[split] = df

        columns_ok = REQUIRED_COLUMNS.issubset(df.columns)
        check(
            columns_ok,
            f"{dataset}/{split}: missing required columns"
        )
        if not columns_ok:
            continue

        n = len(df)
        check(
            n == expected_n,
            f"{dataset}/{split}: expected {expected_n}, got {n}"
        )

        timestamps = pd.to_numeric(
            df["timestamp"], errors="coerce"
        )
        ratings = pd.to_numeric(
            df["rating"], errors="coerce"
        )

        check(
            timestamps.notna().all(),
            f"{dataset}/{split}: invalid timestamps"
        )
        check(
            ratings.notna().all(),
            f"{dataset}/{split}: invalid ratings"
        )
        check(
            not (ratings == 0).any(),
            f"{dataset}/{split}: zero ratings"
        )

        chronological = bool(
            timestamps.is_monotonic_increasing
        )
        check(
            chronological,
            f"{dataset}/{split}: timestamps not sorted"
        )

        negative = int((ratings < 0).sum())

        metadata[split] = {
            "rows": n,
            "negative": negative,
            "positive": int((ratings > 0).sum()),
            "min_timestamp": float(timestamps.min()),
            "max_timestamp": float(timestamps.max()),
            "chronologically_sorted": chronological,
            "sha256": sha256_file(path),
        }

        print(
            f"  {split:11s} "
            f"rows={n:6d} "
            f"negative={negative:5d} "
            f"sorted={chronological}"
        )

        if "trust_label" in df.columns:
            # Phase I Alpha uses 1 for positive trust,
            # 0 for negative trust in the data files.
            actual = pd.to_numeric(
                df["trust_label"], errors="coerce"
            )

            check(
                actual.isin([0, 1]).all(),
                f"{dataset}/{split}: invalid trust_label"
            )
            expected_label = (ratings > 0).astype(int)
            check(
                np.array_equal(
                    actual.to_numpy(),
                    expected_label.to_numpy()
                ),
                f"{dataset}/{split}: trust_label "
                "does not match rating sign"
            )

    # Verify all four parts are present.
    if len(frames) != 4:
        report["datasets"][dataset] = metadata
        continue

    parts = ["context", "train", "validation", "test"]

    # Strict temporal boundaries.
    for previous, following in zip(parts[:-1], parts[1:]):
        left = metadata[previous]["max_timestamp"]
        right = metadata[following]["min_timestamp"]

        check(
            left <= right,
            f"{dataset}: temporal overlap between "
            f"{previous} and {following}"
        )

    combined = pd.concat(
        [frames[p] for p in parts],
        ignore_index=True
    )

    check(
        combined["timestamp"].is_monotonic_increasing,
        f"{dataset}: combined chronological order invalid"
    )

    if dataset == "alpha":
        original_path = DATA / "bitcoin_alpha_chronological.csv"
    else:
        original_path = DATA / "soc-sign-bitcoinotc.csv.gz"

    if original_path.exists():
        original = pd.read_csv(
            original_path,
            header=None if dataset == "otc" else "infer"
        )

        if dataset == "otc":
            original = original.iloc[:, :4].copy()
            original.columns = [
                "source", "target", "rating", "timestamp"
            ]

        same_rows = (
            len(original) == len(combined)
            and row_signature(original) == row_signature(combined)
        )

        check(
            same_rows,
            f"{dataset}: split union does not match source data"
        )
        metadata["source_matches_split_union"] = same_rows

    metadata["total_rows"] = len(combined)
    metadata["total_negative"] = int(
        (combined["rating"] < 0).sum()
    )

    report["datasets"][dataset] = metadata

# ------------------------------------------------------------
# 2. Check final Phase I results
# ------------------------------------------------------------

print("\n" + "=" * 68)
print("FINAL PHASE I RESULT FILES")
print("=" * 68)

required_results = [
    "alpha_final_test_results_v1.csv",
    "alpha_final_test_summary_v1.csv",
    "otc_final_test_results_v1.csv",
    "otc_final_test_summary_v1.csv",
    "experiment_protocol_v1.json",
]

report["results"] = {}

for name in required_results:
    path = RESULTS / name
    exists = path.is_file()

    check(exists, f"Missing final result: {name}")

    report["results"][name] = {
        "exists": exists,
        "bytes": path.stat().st_size if exists else None,
    }

    print(f"  {'OK' if exists else 'MISSING'}  {name}")

# ------------------------------------------------------------
# 3. Check saved test-history graph selections
# ------------------------------------------------------------

print("\n" + "=" * 68)
print("FROZEN GRAPH SELECTION FILES")
print("=" * 68)

for dataset in ("alpha", "otc"):
    directory = RESULTS / (
        f"bitcoin_{dataset}_frozen_test_selections"
    )

    files = sorted(directory.glob("*.csv")) if directory.exists() else []

    expected = 25  # 5 methods × 5 seeds
    check(
        len(files) == expected,
        f"{dataset}: expected {expected} selections, got {len(files)}"
    )

    print(
        f"  {dataset.upper()}: {len(files)}/{expected} CSV files"
    )

# ------------------------------------------------------------
# 4. Inspect checkpoint presence and file signatures
# Do not unpickle or execute checkpoint contents.
# ------------------------------------------------------------

print("\n" + "=" * 68)
print("CHECKPOINT AUDIT")
print("=" * 68)

checkpoints = sorted(
    list(RESULTS.rglob("*.pt"))
    + list(RESULTS.rglob("*.pth"))
)

checkpoint_rows = []

for path in checkpoints:
    row = {
        "name": path.name,
        "path": str(path),
        "bytes": path.stat().st_size,
        "sha256": sha256_file(path),
        "is_zip_container": zipfile.is_zipfile(path),
    }
    checkpoint_rows.append(row)

check(
    len(checkpoints) > 0,
    "No saved PyTorch checkpoints were found"
)

print("Checkpoint files:", len(checkpoints))
for row in checkpoint_rows[:12]:
    print(
        f"  {row['name']}: "
        f"{row['bytes']} bytes"
    )

if len(checkpoint_rows) > 12:
    print("  ... remaining files saved in report")

report["checkpoints"] = {
    "count": len(checkpoints),
    "files": checkpoint_rows,
    "note": (
        "File presence and SHA256 audited; "
        "model-state compatibility not yet tested."
    ),
}

# ------------------------------------------------------------
# 5. Validate main backup archive integrity
# ------------------------------------------------------------

print("\n" + "=" * 68)
print("BACKUP ARCHIVE INTEGRITY")
print("=" * 68)

archive = ROOT / "TASP_GNN_COMPLETE_BACKUP.zip"

if check(
    archive.exists(),
    "Main Phase I ZIP archive is missing"
):
    with zipfile.ZipFile(archive) as z:
        corrupt_member = z.testzip()
        members = [x for x in z.infolist() if not x.is_dir()]

    check(
        corrupt_member is None,
        f"Corrupt ZIP member: {corrupt_member}"
    )

    report["archive"] = {
        "path": str(archive),
        "file_count": len(members),
        "crc_passed": corrupt_member is None,
        "sha256": sha256_file(archive),
    }

    print("Archived files:", len(members))
    print("ZIP CRC:", "PASS" if corrupt_member is None else "FAIL")

# ------------------------------------------------------------
# 6. Save audit report
# ------------------------------------------------------------

report_path = OUT / "e0_integrity_report.json"

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print("\n" + "=" * 68)
print("E0.2 FINAL STATUS")
print("=" * 68)

print("Issue count:", len(report["issues"]))

if report["issues"]:
    print("\nIssues requiring review:")
    for issue in report["issues"]:
        print("  -", issue)
else:
    print("All implemented integrity checks passed.")

print("\nReport saved:")
print(report_path)

print("\nNo Phase I artifacts were modified.")
print("No training was performed.")
print("Final test results were not used for model selection.")


In [ ]:

# ============================================================
# PHASE II — E1.0
# HISTORICAL TRUST FEATURE AUDIT
# Trust-Aware Adaptive Graph Sparsification for SIoT
# ============================================================
# Uses CONTEXT only.
# No validation/test data access.
# No training or model selection.
# No modification of Phase I files.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import json
import hashlib

DATA_ROOT = Path("/kaggle/working/ta_gsnn_data")
OUT = Path(
    "/kaggle/working/phase2_adaptive_siot/"
    "E1_trust_scoring"
)
OUT.mkdir(parents=True, exist_ok=True)

EPS = 1e-12

def binary_entropy(p):
    p = np.clip(np.asarray(p), EPS, 1 - EPS)
    return -(p * np.log2(p)
             + (1 - p) * np.log2(1 - p))

def make_historical_node_features(df):
    """
    Construct node-level trust evidence from historical
    CONTEXT edges only.

    Features:
      out_degree, in_degree
      out_negative_count, in_negative_count
      out_negative_rate, in_negative_rate
      out_trust_entropy, in_trust_entropy
      out_mean_abs_rating, in_mean_abs_rating

    Every edge represents an observed historical rating.
    No future edge labels are involved.
    """

    required = {
        "source", "target", "rating", "timestamp"
    }
    assert required.issubset(df.columns)

    df = df.copy()
    df["is_negative"] = (df["rating"] < 0).astype(int)
    df["abs_rating"] = df["rating"].abs()

    source_stats = (
        df.groupby("source")
        .agg(
            out_degree=("rating", "size"),
            out_negative_count=("is_negative", "sum"),
            out_mean_abs_rating=("abs_rating", "mean"),
        )
    )

    target_stats = (
        df.groupby("target")
        .agg(
            in_degree=("rating", "size"),
            in_negative_count=("is_negative", "sum"),
            in_mean_abs_rating=("abs_rating", "mean"),
        )
    )

    all_nodes = pd.Index(
        pd.concat([
            df["source"],
            df["target"]
        ]).unique(),
        name="node"
    )

    nodes = pd.DataFrame(index=all_nodes)
    nodes = nodes.join(source_stats, how="left")
    nodes = nodes.join(target_stats, how="left")

    count_cols = [
        "out_degree", "in_degree",
        "out_negative_count", "in_negative_count"
    ]
    nodes[count_cols] = nodes[count_cols].fillna(0)

    for direction in ("out", "in"):
        degree = nodes[f"{direction}_degree"]
        negative = nodes[f"{direction}_negative_count"]

        # Smoothed negative-trust fraction.
        # Beta(1,1) prior prevents undefined rates
        # for nodes with no observed edges.
        rate = (negative + 1.0) / (degree + 2.0)

        nodes[f"{direction}_negative_rate"] = rate
        nodes[f"{direction}_trust_entropy"] = (
            binary_entropy(rate)
        )

    nodes[[
        "out_mean_abs_rating",
        "in_mean_abs_rating"
    ]] = nodes[[
        "out_mean_abs_rating",
        "in_mean_abs_rating"
    ]].fillna(0.0)

    nodes["total_degree"] = (
        nodes["out_degree"] + nodes["in_degree"]
    )

    nodes["local_trust_uncertainty"] = (
        nodes["out_trust_entropy"]
        + nodes["in_trust_entropy"]
    ) / 2.0

    return nodes.reset_index()


reports = {}

print("=" * 70)
print("PHASE II — E1.0 HISTORICAL TRUST FEATURE AUDIT")
print("=" * 70)

for dataset in ("alpha", "otc"):

    path = DATA_ROOT / f"bitcoin_{dataset}_context.csv"
    assert path.is_file(), f"Missing context file: {path}"

    context = pd.read_csv(path)

    assert context["timestamp"].is_monotonic_increasing
    assert (context["rating"] != 0).all()

    features = make_historical_node_features(context)

    assert features["node"].is_unique
    assert not features.isna().any().any()
    assert (
        features["local_trust_uncertainty"].between(0, 1)
    ).all()

    output_file = OUT / (
        f"bitcoin_{dataset}_context_trust_features.csv"
    )

    features.to_csv(output_file, index=False)

    density_summary = {
        "node_count": int(len(features)),
        "historical_edges": int(len(context)),
        "negative_edges": int(
            (context["rating"] < 0).sum()
        ),
        "negative_fraction": float(
            (context["rating"] < 0).mean()
        ),
        "mean_total_degree": float(
            features["total_degree"].mean()
        ),
        "median_total_degree": float(
            features["total_degree"].median()
        ),
        "mean_trust_uncertainty": float(
            features["local_trust_uncertainty"].mean()
        ),
        "feature_rows": int(len(features)),
        "source_scope": "historical context only",
        "output_csv": str(output_file),
    }

    reports[dataset] = density_summary

    print(f"\nBITCOIN {dataset.upper()}")
    for key, value in density_summary.items():
        print(f"  {key}: {value}")

    print("\n  Feature preview:")
    print(features.head(5).to_string(index=False))

    print("\n  Uncertainty distribution:")
    print(
        features["local_trust_uncertainty"]
        .describe()
        .round(4)
        .to_string()
    )

with open(
    OUT / "e1_0_trust_feature_audit.json",
    "w", encoding="utf-8"
) as f:
    json.dump(reports, f, indent=2)

print("\n" + "=" * 70)
print("E1.0 COMPLETED")
print("=" * 70)
print("Historical trust features generated.")
print("Validation/test files were not accessed.")
print("No new model was trained.")
print("No Phase I results were modified.")
print("Next: E1.1 — Define trust-aware edge scoring.")


In [ ]:

# ============================================================
# PHASE II — E1.1: SIoT DATA SUITABILITY AUDIT
# Original supervisor research direction
# ============================================================
# Read-only inspection.
# No training, no sparsification, no data modification.
# ============================================================

from pathlib import Path
import pandas as pd
import json

PATH = Path(
    "/kaggle/input/datasets/mahomaattorney/"
    "siot-trust-edges-csv/siot_trust_edges.csv"
)

OUT = Path(
    "/kaggle/working/phase2_adaptive_siot/"
    "E1_trust_scoring"
)
OUT.mkdir(parents=True, exist_ok=True)

assert PATH.is_file(), f"Dataset not found: {PATH}"

df = pd.read_csv(PATH)

print("=" * 68)
print("E1.1 — SIoT DATA SUITABILITY AUDIT")
print("=" * 68)

print("\nFILE:")
print(PATH)

print("\nSHAPE:")
print(df.shape)

print("\nCOLUMNS:")
print(list(df.columns))

print("\nDATA TYPES:")
print(df.dtypes.to_string())

print("\nFIRST 10 ROWS:")
print(df.head(10).to_string(index=False))

print("\nMISSING VALUES:")
print(df.isna().sum().to_string())

print("\nUNIQUE VALUES PER COLUMN:")
print(df.nunique(dropna=False).to_string())

print("\nNUMERICAL COLUMN STATISTICS:")
numerical = df.select_dtypes(include="number")
if not numerical.empty:
    print(numerical.describe().to_string())
else:
    print("No numeric columns detected.")

print("\nPOTENTIAL SEMANTIC FIELDS:")

groups = {
    "source_node": [
        "source", "src", "from", "trustor"
    ],
    "target_node": [
        "target", "dst", "to", "trustee"
    ],
    "trust": [
        "trust", "rating", "weight", "score", "sign"
    ],
    "time": [
        "time", "timestamp", "date", "epoch"
    ],
    "device": [
        "device", "object", "iot", "type"
    ],
    "context": [
        "context", "service", "location", "social"
    ],
}

detected = {}

for group, terms in groups.items():
    matches = [
        col for col in df.columns
        if any(
            term in col.lower()
            for term in terms
        )
    ]
    detected[group] = matches
    print(f"{group}: {matches}")

# These are name-based candidates only,
# NOT verified semantic interpretations.

print("\nDUPLICATE ROWS:")
print(int(df.duplicated().sum()))

print("\nSOURCE DOCUMENTATION CHECK:")
print(
    "This cell cannot determine whether the data "
    "are real or synthetic from column names alone."
)
print(
    "Dataset provenance, construction procedure, "
    "trust-label meaning, and time semantics "
    "must be verified separately."
)

summary = {
    "file": str(PATH),
    "rows": int(len(df)),
    "columns": list(df.columns),
    "missing_by_column": {
        k: int(v) for k, v in df.isna().sum().items()
    },
    "duplicate_rows": int(df.duplicated().sum()),
    "candidate_fields": detected,
    "provenance_verified": False,
    "temporal_semantics_verified": False,
    "siot_validity_verified": False,
}

with open(
    OUT / "e1_1_siot_suitability_audit.json",
    "w", encoding="utf-8"
) as f:
    json.dump(
        summary,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\n" + "=" * 68)
print("E1.1 COMPLETED — SCHEMA INSPECTION ONLY")
print("=" * 68)

print("No model trained.")
print("No Phase I files modified.")
print("No claim of SIoT validity made.")
print("\nNEXT: Verify provenance and define")
print("the trust-aware adaptive learning objective.")


In [ ]:

# ============================================================
# PHASE II — E1.2
# SANTANDER SIoT DATASET ACQUISITION AUDIT
# ============================================================

from pathlib import Path
import requests
import json

OUT = Path(
    "/kaggle/working/phase2_adaptive_siot/"
    "E1_trust_scoring/santander"
)
OUT.mkdir(parents=True, exist_ok=True)

API = (
    "https://api.github.com/repos/"
    "Net4uCA/SIoT-IoT-Dataset/contents"
)

print("=" * 68)
print("E1.2 — SANTANDER SIoT DATASET AUDIT")
print("=" * 68)

try:
    response = requests.get(
        API,
        headers={"Accept": "application/vnd.github+json"},
        timeout=25
    )
    response.raise_for_status()
    files = response.json()

    inventory = []

    for item in files:
        if item.get("type") != "file":
            continue

        entry = {
            "name": item["name"],
            "size_mb": round(
                item["size"] / (1024 ** 2), 3
            ),
            "download_url": item.get("download_url"),
            "sha": item.get("sha"),
        }
        inventory.append(entry)

    inventory.sort(key=lambda x: x["name"])

    print("\nREPOSITORY FILES:")
    for item in inventory:
        print(
            f"{item['name']:<44} "
            f"{item['size_mb']:>10.3f} MB"
        )

    with open(
        OUT / "santander_repository_inventory.json",
        "w", encoding="utf-8"
    ) as f:
        json.dump(
            inventory, f,
            ensure_ascii=False, indent=2
        )

    required = [
        "objects_description.zip",
        "objects_profile.zip",
        "adjacency_matricies.zip",
        "private_devices.zip",
        "public_devices.zip",
    ]

    names = {x["name"] for x in inventory}

    print("\nESSENTIAL DATA COMPONENTS:")
    for name in required:
        print(
            f"{'FOUND' if name in names else 'MISSING'}"
            f"  {name}"
        )

    print("\nE1.2 STATUS: REPOSITORY ACCESS SUCCESSFUL")
    print("No files downloaded or extracted.")

except requests.RequestException as error:
    print("\nREPOSITORY ACCESS FAILED")
    print(type(error).__name__, str(error))
    print(
        "\nEnable Internet in Kaggle Notebook settings "
        "or add the repository files as a Kaggle dataset."
    )



In [ ]:

# ============================================================
# PHASE II — E1.3
# SANTANDER SIoT DATA ACQUISITION & SCHEMA INSPECTION
# ============================================================
# Downloads official repository archives.
# Does not modify Phase I.
# Does not load large adjacency matrices into RAM.
# Does not train or generate synthetic trust labels.
# ============================================================

from pathlib import Path
from io import TextIOWrapper
import requests
import zipfile
import json
import hashlib
import csv

BASE = Path(
    "/kaggle/working/phase2_adaptive_siot/"
    "E1_trust_scoring/santander"
)
RAW = BASE / "raw"
RAW.mkdir(parents=True, exist_ok=True)

API = (
    "https://api.github.com/repos/"
    "Net4uCA/SIoT-IoT-Dataset/contents"
)

WANTED = [
    "README.md",
    "objects_description.zip",
    "objects_profile.zip",
    "adjacency_matricies.zip",
    "private_devices.zip",
    "public_devices.zip",
    "service_and_application_description.zip",
    "Lysis_Dataset.zip",
]

MAX_DOWNLOAD_BYTES = 40 * 1024 * 1024
MAX_MEMBER_PREVIEW_BYTES = 64 * 1024
MAX_PREVIEW_LINES = 5

session = requests.Session()
session.headers.update({
    "Accept": "application/vnd.github+json",
    "User-Agent": "SIoT-Research-Audit"
})

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            digest.update(chunk)
    return digest.hexdigest()

def download_file(url, destination, expected_size):
    # Existing files are reused only when size matches.
    if destination.exists():
        if destination.stat().st_size == expected_size:
            return "reused"
        raise RuntimeError(
            f"Existing file size mismatch: {destination}"
        )

    if expected_size > MAX_DOWNLOAD_BYTES:
        raise RuntimeError(
            f"File exceeds safety limit: {destination.name}"
        )

    temp = destination.with_suffix(
        destination.suffix + ".partial"
    )

    total = 0
    try:
        with session.get(
            url, stream=True, timeout=(15, 90)
        ) as response:
            response.raise_for_status()

            with open(temp, "wb") as out:
                for chunk in response.iter_content(
                    chunk_size=1024 * 1024
                ):
                    if not chunk:
                        continue
                    total += len(chunk)
                    if total > MAX_DOWNLOAD_BYTES:
                        raise RuntimeError(
                            "Download exceeded safety limit."
                        )
                    out.write(chunk)

        if total != expected_size:
            raise RuntimeError(
                f"Size mismatch: expected {expected_size}, "
                f"downloaded {total}"
            )

        temp.replace(destination)
        return "downloaded"

    finally:
        if temp.exists():
            temp.unlink()

def preview_zip_member(zf, member):
    """
    Read only a small beginning of a text-like member.
    Never extract files to disk.
    """
    if member.is_dir():
        return []

    if member.file_size > 200 * 1024 * 1024:
        return ["[large member: preview skipped]"]

    extension = Path(member.filename).suffix.lower()

    if extension not in {
        ".txt", ".csv", ".tsv", ".dat",
        ".json", ".md", ".edges"
    }:
        return ["[non-text or unknown extension]"]

    with zf.open(member) as f:
        sample = f.read(MAX_MEMBER_PREVIEW_BYTES)

    if b"\x00" in sample:
        return ["[binary content: preview skipped]"]

    text = sample.decode(
        "utf-8-sig", errors="replace"
    )
    lines = [
        line[:180]
        for line in text.splitlines()
        if line.strip()
    ]

    return lines[:MAX_PREVIEW_LINES]

print("=" * 72)
print("E1.3 — SANTANDER DATA ACQUISITION")
print("=" * 72)

response = session.get(API, timeout=30)
response.raise_for_status()

items = {
    item["name"]: item
    for item in response.json()
    if item.get("type") == "file"
}

report = {
    "repository": (
        "https://github.com/Net4uCA/SIoT-IoT-Dataset"
    ),
    "archives": {},
    "notes": [
        "ZIP members inspected without extraction.",
        "Dataset trust labels have not been assumed.",
        "Temporal graph snapshots not yet verified."
    ],
}

for filename in WANTED:
    if filename not in items:
        print("MISSING:", filename)
        continue

    info = items[filename]
    destination = RAW / filename

    print(f"\n{'-' * 60}")
    print("FILE:", filename)

    status = download_file(
        info["download_url"],
        destination,
        info["size"]
    )

    print("Transfer:", status)
    print(
        "Size MB:",
        round(destination.stat().st_size / 1048576, 3)
    )

    record = {
        "download_status": status,
        "bytes": destination.stat().st_size,
        "sha256": sha256_file(destination),
        "github_blob_sha": info.get("sha"),
        "members": [],
    }

    if filename.lower().endswith(".zip"):

        if not zipfile.is_zipfile(destination):
            raise RuntimeError(
                f"Invalid ZIP: {destination}"
            )

        with zipfile.ZipFile(destination) as zf:
            members = [
                m for m in zf.infolist()
                if not m.is_dir()
            ]

            print("Contained files:", len(members))

            # Show at most 30 filenames in console.
            for m in members[:30]:
                print(
                    f"  {m.filename} "
                    f"({m.file_size / 1048576:.3f} MB)"
                )

            if len(members) > 30:
                print(
                    f"  ... {len(members) - 30} more"
                )

            # Preview only the first three small members.
            preview_candidates = sorted(
                members,
                key=lambda m: m.file_size
            )[:3]

            previews = {}

            for m in preview_candidates:
                lines = preview_zip_member(zf, m)
                previews[m.filename] = lines

                print(f"\n  PREVIEW: {m.filename}")
                for line in lines:
                    print("   ", line)

            record["members"] = [
                {
                    "name": m.filename,
                    "bytes": m.file_size,
                    "compressed_bytes": m.compress_size,
                }
                for m in members
            ]
            record["previews"] = previews

    elif filename == "README.md":
        text = destination.read_text(
            encoding="utf-8", errors="replace"
        )
        record["readme_characters"] = len(text)

    report["archives"][filename] = record

report_path = BASE / "e1_3_acquisition_report.json"

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(
        report,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\n" + "=" * 72)
print("E1.3 COMPLETED")
print("=" * 72)

print("Downloaded/reused files:", len(report["archives"]))
print("Audit report:", report_path)
print("No Phase I files modified.")
print("No trust labels generated.")
print("No model trained.")

print("\nNEXT: E1.4 — Relationship and temporal schema audit.")


In [ ]:

# ============================================================
# PHASE II — E1.3 RECOVERY
# Validate downloaded Santander files and inspect ZIP schemas
# ============================================================

from pathlib import Path
import zipfile
import json
import hashlib

BASE = Path(
    "/kaggle/working/phase2_adaptive_siot/"
    "E1_trust_scoring/santander"
)
RAW = BASE / "raw"

EXPECTED_FILES = [
    "adjacency_matricies.zip",
    "objects_description.zip",
    "objects_profile.zip",
    "private_devices.zip",
    "public_devices.zip",
    "service_and_application_description.zip",
    "Lysis_Dataset.zip",
]

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1048576), b""):
            h.update(block)
    return h.hexdigest()

report = {"files": {}, "warnings": []}

print("=" * 72)
print("E1.3 — RECOVERY AND ZIP STRUCTURE AUDIT")
print("=" * 72)

for filename in EXPECTED_FILES:
    path = RAW / filename

    print(f"\n{'-' * 60}")
    print("FILE:", filename)

    if not path.exists():
        print("STATUS: NOT DOWNLOADED")
        report["warnings"].append(f"Missing: {filename}")
        continue

    info = {
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
        "header_hex": path.read_bytes()[:16].hex(),
        "valid_zip": zipfile.is_zipfile(path),
        "members": []
    }

    print("Size:", info["size_bytes"], "bytes")
    print("Header:", info["header_hex"])

    if not info["valid_zip"]:
        print("STATUS: NOT A VALID ZIP — SKIPPED")
        report["warnings"].append(
            f"Invalid ZIP: {filename}"
        )
        report["files"][filename] = info
        continue

    with zipfile.ZipFile(path) as z:
        members = [
            m for m in z.infolist()
            if not m.is_dir()
        ]

        info["members"] = [
            {
                "name": m.filename,
                "bytes": m.file_size
            }
            for m in members
        ]

        print("STATUS: VALID ZIP")
        print("Number of files:", len(members))

        for member in members[:35]:
            print(
                " ",
                member.filename,
                f"({member.file_size:,} bytes)"
            )

        if len(members) > 35:
            print(f" ... {len(members)-35} more files")

        print("\nSMALL TEXT FILE PREVIEWS:")

        candidates = [
            m for m in members
            if m.file_size < 2_000_000
            and Path(m.filename).suffix.lower()
            in {".csv", ".txt", ".dat", ".md", ".edges"}
        ]

        for member in candidates[:3]:
            try:
                with z.open(member) as f:
                    sample = f.read(4096)

                if b"\x00" in sample:
                    print(
                        member.filename,
                        ": binary-like content; skipped"
                    )
                    continue

                lines = sample.decode(
                    "utf-8-sig",
                    errors="replace"
                ).splitlines()

                print("\n", member.filename)
                for line in lines[:4]:
                    print("   ", line[:200])

            except Exception as exc:
                print("Preview error:", repr(exc))

    report["files"][filename] = info

report_path = BASE / "e1_3_recovery_report.json"

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(
        report, f,
        indent=2,
        ensure_ascii=False
    )

print("\n" + "=" * 72)
print("E1.3 RECOVERY COMPLETED")
print("=" * 72)

valid_count = sum(
    item["valid_zip"]
    for item in report["files"].values()
)

print("Valid ZIP archives:", valid_count)
print("Warnings:", len(report["warnings"]))

for warning in report["warnings"]:
    print(" -", warning)

print("Report:", report_path)
print("No files were deleted or overwritten.")


In [ ]:

# ============================================================
# PHASE II — E1.4
# SANTANDER SIoT: MATRIX STRUCTURE & DEVICE AUDIT
# ============================================================
# Read-only, memory-safe sampling.
# No full adjacency matrix loading.
# No training or synthetic trust generation.
# ============================================================

from pathlib import Path
from collections import Counter
import zipfile
import csv
import json
import io
import numpy as np
import pandas as pd

BASE = Path(
    "/kaggle/working/phase2_adaptive_siot/"
    "E1_trust_scoring/santander"
)
RAW = BASE / "raw"

ADJ_ZIP = RAW / "adjacency_matricies.zip"
DESC_ZIP = RAW / "objects_description.zip"

SAMPLE_ROWS = 24
MAX_DISPLAY_COLUMNS = 15

assert ADJ_ZIP.is_file()
assert DESC_ZIP.is_file()

report = {
    "experiment": "E1.4",
    "sampling_only": True,
    "sample_rows_per_matrix": SAMPLE_ROWS,
    "matrices": {},
    "devices": {},
    "warnings": [],
}

print("=" * 72)
print("E1.4 — SIoT GRAPH SCHEMA AUDIT")
print("=" * 72)

# ------------------------------------------------------------
# 1. Read device descriptions
# ------------------------------------------------------------

with zipfile.ZipFile(DESC_ZIP) as z:
    with z.open("objects_description.csv") as f:
        devices = pd.read_csv(f)

print("\n[1] DEVICE DESCRIPTION")
print("Rows:", len(devices))
print("Columns:", list(devices.columns))
print("\nPreview:")
print(devices.head(5).to_string(index=False))

if "id_device" in devices.columns:
    ids = pd.to_numeric(
        devices["id_device"], errors="coerce"
    )

    print("\nDevice ID range:", ids.min(), "to", ids.max())
    print("Unique device IDs:", ids.nunique())
    print("Duplicate IDs:", int(ids.duplicated().sum()))

    report["devices"]["id_min"] = (
        int(ids.min()) if ids.notna().any() else None
    )
    report["devices"]["id_max"] = (
        int(ids.max()) if ids.notna().any() else None
    )
    report["devices"]["unique_ids"] = int(ids.nunique())
    report["devices"]["duplicate_ids"] = int(
        ids.duplicated().sum()
    )

if "device_type" in devices.columns:
    print("\nDevice types:")
    print(
        devices["device_type"]
        .value_counts()
        .head(15)
        .to_string()
    )

if "id_user" in devices.columns:
    print("\nUnique owners:", devices["id_user"].nunique())

report["devices"]["rows"] = len(devices)
report["devices"]["columns"] = list(devices.columns)
report["devices"]["missing_values"] = {
    str(k): int(v)
    for k, v in devices.isna().sum().items()
}

# ------------------------------------------------------------
# 2. Inspect compressed adjacency matrix samples
# ------------------------------------------------------------

def inspect_matrix_sample(zf, member_name, sample_rows=24):
    """
    Read first sample_rows lines only.

    Detect:
      - column count
      - numeric vs nonnumeric cells
      - sampled distinct values
      - sampled nonzero fractions
      - possible header/index columns

    Do NOT assume that first row is a header.
    Do NOT infer whole-matrix density from this sample.
    """
    sampled = []
    lengths = []
    value_counts = Counter()
    numeric_count = 0
    zero_count = 0
    nonzero_count = 0
    nonnumeric_count = 0
    examples_nonnumeric = []

    with zf.open(member_name) as binary:
        text = io.TextIOWrapper(
            binary, encoding="utf-8-sig",
            errors="replace", newline=""
        )
        reader = csv.reader(text)

        for row_index, row in enumerate(reader):
            if row_index >= sample_rows:
                break

            lengths.append(len(row))

            if row_index < 3:
                sampled.append(
                    row[:MAX_DISPLAY_COLUMNS]
                )

            for cell in row:
                value = cell.strip()

                try:
                    number = float(value)
                    if not np.isfinite(number):
                        raise ValueError("Nonfinite value")

                    numeric_count += 1
                    if number == 0:
                        zero_count += 1
                    else:
                        nonzero_count += 1

                    if len(value_counts) < 40 or value in value_counts:
                        value_counts[value] += 1

                except (ValueError, TypeError):
                    nonnumeric_count += 1
                    if len(examples_nonnumeric) < 8:
                        examples_nonnumeric.append(value[:70])

    return {
        "rows_sampled": len(lengths),
        "column_count_first_row": (
            lengths[0] if lengths else 0
        ),
        "column_count_min": min(lengths) if lengths else 0,
        "column_count_max": max(lengths) if lengths else 0,
        "numeric_cells": numeric_count,
        "zero_cells": zero_count,
        "nonzero_cells": nonzero_count,
        "nonnumeric_cells": nonnumeric_count,
        "sample_nonzero_fraction": (
            nonzero_count / numeric_count
            if numeric_count else None
        ),
        "sample_value_counts": dict(
            value_counts.most_common(12)
        ),
        "nonnumeric_examples": examples_nonnumeric,
        "first_rows_first_columns": sampled,
    }

print("\n[2] ADJACENCY MATRIX SAMPLING")

with zipfile.ZipFile(ADJ_ZIP) as z:
    members = [
        m for m in z.namelist()
        if m.lower().endswith(".csv")
    ]

    for name in sorted(members):
        print("\n" + "-" * 60)
        print("MATRIX:", name)

        metadata = z.getinfo(name)

        result = inspect_matrix_sample(
            z, name, SAMPLE_ROWS
        )

        result["uncompressed_bytes"] = metadata.file_size
        result["compressed_bytes"] = metadata.compress_size

        report["matrices"][name] = result

        print(
            "Uncompressed MB:",
            round(metadata.file_size / 1048576, 2)
        )
        print("Rows sampled:", result["rows_sampled"])
        print(
            "Columns in first row:",
            result["column_count_first_row"]
        )
        print(
            "Column range:",
            result["column_count_min"],
            "to",
            result["column_count_max"]
        )
        print("Numeric cells:", result["numeric_cells"])
        print("Nonnumeric cells:", result["nonnumeric_cells"])

        print(
            "Sample nonzero fraction:",
            round(
                result["sample_nonzero_fraction"], 6
            )
            if result["sample_nonzero_fraction"] is not None
            else "N/A"
        )

        print(
            "Sample values:",
            result["sample_value_counts"]
        )

        if result["nonnumeric_examples"]:
            print(
                "Nonnumeric examples:",
                result["nonnumeric_examples"]
            )

        print("First rows (first 15 columns):")
        for row in result["first_rows_first_columns"]:
            print(" ", row)

# ------------------------------------------------------------
# 3. Compare dimensions with device descriptions
# ------------------------------------------------------------

print("\n[3] DIMENSION CONSISTENCY")

device_count = len(devices)

for name, info in report["matrices"].items():
    columns = info["column_count_first_row"]

    if columns == device_count:
        interpretation = "Compatible with N x N matrix width"
    elif columns == device_count + 1:
        interpretation = "Possible row-index column"
    else:
        interpretation = (
            "Dimensions need further examination"
        )
        report["warnings"].append(
            f"{name}: {columns} columns vs "
            f"{device_count} device records"
        )

    print(f"{name}: {interpretation}")
    info["dimension_interpretation"] = interpretation

# ------------------------------------------------------------
# 4. Save report
# ------------------------------------------------------------

report_path = BASE / "e1_4_matrix_schema_audit.json"

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(
        report, f, ensure_ascii=False, indent=2
    )

print("\n" + "=" * 72)
print("E1.4 COMPLETED")
print("=" * 72)
print("Device records:", device_count)
print("Matrices inspected:", len(report["matrices"]))
print("Warnings:", len(report["warnings"]))
print("Report:", report_path)

print("\nNo full matrix was loaded into memory.")
print("No trust values were synthesized.")
print("No model was trained.")
print("\nNEXT: E1.5 — Sparse graph extraction and audit.")


In [ ]:

# ============================================================
# PHASE II — E1.5
# SANTANDER SIoT: STREAMING SPARSE GRAPH EXTRACTION
# ============================================================
# Correct CSV delimiter: semicolon (;)
# Extracts only NONZERO entries of SIoT.csv.
# No dense matrix allocation.
# No synthetic trust labels.
# No model training.
# ============================================================

from pathlib import Path
from collections import Counter
import zipfile
import csv
import io
import json
import os
import time

BASE = Path(
    "/kaggle/working/phase2_adaptive_siot/"
    "E1_trust_scoring/santander"
)
RAW = BASE / "raw"
OUT = BASE / "sparse_graphs"
OUT.mkdir(parents=True, exist_ok=True)

ZIP_PATH = RAW / "adjacency_matricies.zip"
MATRIX_NAME = "SIoT.csv"

N_EXPECTED = 16216

EDGE_PATH = OUT / "siot_edges.csv"
TEMP_PATH = OUT / "siot_edges.csv.partial"
REPORT_PATH = OUT / "e1_5_siot_sparse_audit.json"

assert ZIP_PATH.is_file()

start_time = time.perf_counter()

print("=" * 70)
print("E1.5 — SIoT SPARSE GRAPH EXTRACTION")
print("=" * 70)

print("Matrix:", MATRIX_NAME)
print("Expected devices:", N_EXPECTED)
print("Parsing delimiter: ';'")
print("Dense matrix loading: DISABLED")

# ------------------------------------------------------------
# 1. Stream matrix from ZIP
# ------------------------------------------------------------

nonzero_edges = 0
self_loops = 0
rows_processed = 0

value_counts = Counter()

out_degree = Counter()
in_degree = Counter()

edge_preview = []
first_error = None

# Ensure an incomplete previous extraction is not confused
# with a valid result.
if TEMP_PATH.exists():
    TEMP_PATH.unlink()

try:
    with zipfile.ZipFile(ZIP_PATH) as z:

        assert MATRIX_NAME in z.namelist(), (
            f"Missing matrix: {MATRIX_NAME}"
        )

        with z.open(MATRIX_NAME) as binary:

            text_stream = io.TextIOWrapper(
                binary,
                encoding="utf-8-sig",
                newline=""
            )

            reader = csv.reader(
                text_stream,
                delimiter=";"
            )

            # --------------------------------------------
            # 2. Header validation
            # --------------------------------------------

            header = next(reader)

            if header and header[-1] == "":
                header = header[:-1]

            column_ids = [
                int(value)
                for value in header[1:]
            ]

            assert len(column_ids) == N_EXPECTED, (
                f"Expected {N_EXPECTED} columns, "
                f"got {len(column_ids)}"
            )

            assert column_ids == list(
                range(1, N_EXPECTED + 1)
            ), "Unexpected column identifiers"

            print("\nHeader verified:")
            print("  Columns:", len(column_ids))
            print("  First ID:", column_ids[0])
            print("  Last ID:", column_ids[-1])

            # --------------------------------------------
            # 3. Sparse extraction
            # --------------------------------------------

            with open(
                TEMP_PATH,
                "w",
                newline="",
                encoding="utf-8"
            ) as output:

                writer = csv.writer(output)

                writer.writerow([
                    "source",
                    "target",
                    "relation_value"
                ])

                for row in reader:

                    if row and row[-1] == "":
                        row = row[:-1]

                    if not row:
                        continue

                    source = int(row[0])
                    values = row[1:]

                    expected_source = rows_processed + 1

                    if source != expected_source:
                        raise ValueError(
                            f"Unexpected row ID {source}; "
                            f"expected {expected_source}"
                        )

                    if len(values) != N_EXPECTED:
                        raise ValueError(
                            f"Row {source}: expected "
                            f"{N_EXPECTED} values; "
                            f"got {len(values)}"
                        )

                    for target, value in zip(
                        column_ids, values
                    ):

                        value = value.strip()

                        if value in ("", "0", "0.0"):
                            continue

                        try:
                            numeric = float(value)
                        except ValueError:
                            raise ValueError(
                                f"Invalid value at "
                                f"({source}, {target}): {value}"
                            )

                        if not (
                            numeric == numeric
                            and abs(numeric) != float("inf")
                        ):
                            raise ValueError(
                                "Nonfinite matrix value"
                            )

                        if numeric == 0:
                            continue

                        writer.writerow([
                            source,
                            target,
                            value
                        ])

                        nonzero_edges += 1
                        value_counts[value] += 1
                        out_degree[source] += 1
                        in_degree[target] += 1

                        if source == target:
                            self_loops += 1

                        if len(edge_preview) < 8:
                            edge_preview.append([
                                source, target, value
                            ])

                    rows_processed += 1

                    if rows_processed % 2000 == 0:
                        print(
                            f"  Processed {rows_processed:,} "
                            f"rows | "
                            f"{nonzero_edges:,} nonzero entries",
                            flush=True
                        )

    assert rows_processed == N_EXPECTED, (
        f"Expected {N_EXPECTED} matrix rows, "
        f"got {rows_processed}"
    )

    # Only rename after successful full extraction.
    os.replace(TEMP_PATH, EDGE_PATH)

except Exception:
    # Preserve the partial file for debugging,
    # but never treat it as a complete graph.
    print("\nEXTRACTION FAILED")
    print("Partial file:", TEMP_PATH)
    raise

# ------------------------------------------------------------
# 4. Structural summary
# ------------------------------------------------------------

elapsed = time.perf_counter() - start_time

active_nodes = set(out_degree) | set(in_degree)

density = (
    nonzero_edges / (N_EXPECTED * N_EXPECTED)
)

report = {
    "matrix": MATRIX_NAME,
    "nodes_defined": N_EXPECTED,
    "rows_processed": rows_processed,
    "nonzero_entries": nonzero_edges,
    "active_nodes": len(active_nodes),
    "isolated_nodes": N_EXPECTED - len(active_nodes),
    "self_loops": self_loops,
    "directed_density_including_diagonal": density,
    "observed_nonzero_values": dict(value_counts),
    "mean_out_degree_all_nodes": (
        nonzero_edges / N_EXPECTED
    ),
    "max_out_degree": max(
        out_degree.values(), default=0
    ),
    "max_in_degree": max(
        in_degree.values(), default=0
    ),
    "elapsed_seconds": round(elapsed, 3),
    "output_csv": str(EDGE_PATH),
    "edge_preview": edge_preview,
    "semantics": (
        "Nonzero adjacency entries. "
        "Not verified trust ratings."
    ),
    "temporal_information": False,
}

with open(
    REPORT_PATH, "w", encoding="utf-8"
) as f:
    json.dump(
        report, f,
        indent=2,
        ensure_ascii=False
    )

print("\n" + "=" * 70)
print("E1.5 COMPLETED")
print("=" * 70)

print("Matrix rows:", rows_processed)
print("Defined devices:", N_EXPECTED)
print("Nonzero adjacency entries:", nonzero_edges)
print("Active nodes:", len(active_nodes))
print("Isolated nodes:", N_EXPECTED - len(active_nodes))
print("Self-loops:", self_loops)
print("Directed density:", round(density, 8))
print("Nonzero value distribution:", dict(value_counts))
print("Elapsed seconds:", round(elapsed, 2))

print("\nFirst extracted edges:")
for edge in edge_preview:
    print(" ", edge)

print("\nOutput:")
print(EDGE_PATH)
print("Audit:")
print(REPORT_PATH)

print("\nIMPORTANT:")
print("Nonzero entries represent observed SIoT adjacency.")
print("They are NOT automatically trust ratings.")
print("No temporal adaptation has been tested.")


In [ ]:

# ============================================================
# PHASE II — E1.6
# SANTANDER MULTI-RELATIONAL SIoT GRAPH AUDIT
# ============================================================
# Reads five social relationship matrices from ZIP.
# Produces compressed edge lists with relation types.
# No synthetic trust scores.
# No model training.
# No Phase I modifications.
# ============================================================

from pathlib import Path
from collections import Counter
import zipfile
import io
import csv
import json
import time
import numpy as np
import pandas as pd

BASE = Path(
    "/kaggle/working/phase2_adaptive_siot/"
    "E1_trust_scoring/santander"
)
RAW = BASE / "raw"
OUT = BASE / "sparse_graphs"
OUT.mkdir(parents=True, exist_ok=True)

ZIP_PATH = RAW / "adjacency_matricies.zip"

N = 16216
RELATIONS = [
    "OOR", "POR", "C_LOR", "SOR", "SOR2"
]

# Includes 6 relation indicators total,
# where SIoT is the composite adjacency reference.
ALL_RELATIONS = ["SIoT"] + RELATIONS

REPORT_PATH = OUT / "e1_6_multirelational_audit.json"
MULTI_PATH = OUT / "siot_multirelational_edges.csv.gz"

print("=" * 72)
print("E1.6 — MULTI-RELATIONAL SIoT GRAPH AUDIT")
print("=" * 72)

start_time = time.perf_counter()

# ------------------------------------------------------------
# 1. Read SIoT graph produced by E1.5
# ------------------------------------------------------------

siot_path = OUT / "siot_edges.csv"

assert siot_path.is_file(), (
    "Run E1.5 before E1.6"
)

siot = pd.read_csv(
    siot_path,
    usecols=["source", "target"],
    dtype={"source": "int32", "target": "int32"}
)

assert len(siot) == 292234
assert not siot.duplicated().any()

# Create compact edge-key representation.
# Device IDs are 1 ... N.
siot_keys = (
    (siot["source"].to_numpy(dtype=np.int64) - 1) * N
    + (siot["target"].to_numpy(dtype=np.int64) - 1)
)

siot_keys = np.sort(siot_keys)

print("Composite SIoT edges:", len(siot_keys))

# ------------------------------------------------------------
# 2. Streaming relationship-matrix extraction
# ------------------------------------------------------------

def get_nonzero_keys(zf, member_name):
    keys = []
    values_found = Counter()
    rows = 0

    with zf.open(member_name) as binary:
        reader = csv.reader(
            io.TextIOWrapper(
                binary,
                encoding="utf-8-sig",
                newline=""
            ),
            delimiter=";"
        )

        header = next(reader)
        if header and header[-1] == "":
            header = header[:-1]

        cols = np.asarray(
            [int(c) for c in header[1:]],
            dtype=np.int32
        )

        if len(cols) != N or not np.array_equal(
            cols, np.arange(1, N + 1, dtype=np.int32)
        ):
            raise ValueError(
                f"{member_name}: unexpected column IDs"
            )

        for row in reader:
            if row and row[-1] == "":
                row = row[:-1]

            if not row:
                continue

            source = int(row[0])
            values = row[1:]

            if source != rows + 1:
                raise ValueError(
                    f"{member_name}: row ID mismatch"
                )

            if len(values) != N:
                raise ValueError(
                    f"{member_name}: invalid row width"
                )

            # Parse one row at a time. No dense NxN array.
            arr = np.fromiter(
                (int(v) for v in values),
                dtype=np.int8,
                count=N
            )

            nonzero = np.flatnonzero(arr)

            for value, count in zip(
                *np.unique(
                    arr[nonzero],
                    return_counts=True
                )
            ):
                values_found[int(value)] += int(count)

            if nonzero.size:
                row_keys = (
                    np.int64(source - 1) * N
                    + nonzero.astype(np.int64)
                )
                keys.extend(row_keys.tolist())

            rows += 1

            if rows % 4000 == 0:
                print(
                    f"  {member_name}: "
                    f"{rows:,}/{N:,} rows",
                    flush=True
                )

    if rows != N:
        raise ValueError(
            f"{member_name}: expected {N} rows; got {rows}"
        )

    return np.unique(
        np.asarray(keys, dtype=np.int64)
    ), dict(values_found)

relation_keys = {}
relation_reports = {}

with zipfile.ZipFile(ZIP_PATH) as z:
    for relation in RELATIONS:
        name = f"{relation}.csv"

        print("\nProcessing:", name)
        keys, observed_values = get_nonzero_keys(z, name)

        relation_keys[relation] = keys

        overlap_with_siot = int(
            np.isin(keys, siot_keys).sum()
        )

        relation_reports[relation] = {
            "edge_count": int(len(keys)),
            "overlap_with_siot": overlap_with_siot,
            "outside_siot": int(
                len(keys) - overlap_with_siot
            ),
            "observed_nonzero_values": observed_values,
        }

        print(
            "  Edges:", len(keys),
            "| Overlap SIoT:", overlap_with_siot,
            "| Outside SIoT:",
            len(keys) - overlap_with_siot
        )

# ------------------------------------------------------------
# 3. Build relation indicators on composite SIoT edges
# ------------------------------------------------------------

multi = pd.DataFrame({
    "source": (
        siot_keys // N + 1
    ).astype(np.int32),
    "target": (
        siot_keys % N + 1
    ).astype(np.int32),
})

for relation in RELATIONS:
    multi[relation] = np.isin(
        siot_keys,
        relation_keys[relation]
    ).astype(np.uint8)

multi["relation_count"] = (
    multi[RELATIONS].sum(axis=1)
).astype(np.uint8)

# It is possible that some composite SIoT edges have
# no identified relation among these five matrices.
unexplained = int(
    (multi["relation_count"] == 0).sum()
)

print("\nCOMPOSITE EDGE COVERAGE")
print("SIoT edges:", len(multi))
print("Unexplained by five matrices:", unexplained)

print("\nRELATION COUNT DISTRIBUTION")
print(
    multi["relation_count"]
    .value_counts()
    .sort_index()
    .to_string()
)

# ------------------------------------------------------------
# 4. Save compressed output
# ------------------------------------------------------------

multi.to_csv(
    MULTI_PATH,
    index=False,
    compression="gzip"
)

report = {
    "experiment": "E1.6",
    "nodes": N,
    "composite_siot_edges": len(multi),
    "relations": relation_reports,
    "composite_unexplained_edges": unexplained,
    "relation_count_distribution": {
        str(k): int(v)
        for k, v in multi["relation_count"]
        .value_counts()
        .sort_index()
        .items()
    },
    "elapsed_seconds": round(
        time.perf_counter() - start_time, 2
    ),
    "output": str(MULTI_PATH),
    "trust_labels_present": False,
    "temporal_snapshots_verified": False,
}

with open(
    REPORT_PATH, "w", encoding="utf-8"
) as f:
    json.dump(
        report,
        f, indent=2, ensure_ascii=False
    )

print("\n" + "=" * 72)
print("E1.6 COMPLETED")
print("=" * 72)

print("Composite edges:", len(multi))
print("Unexplained edges:", unexplained)

for relation, info in relation_reports.items():
    print(
        relation,
        "edges:", info["edge_count"],
        "outside composite:", info["outside_siot"]
    )

print("\nSaved:", MULTI_PATH)
print("Report:", REPORT_PATH)

print("\nNo trust labels were generated.")
print("No temporal evolution was inferred.")
print("No model was trained.")


In [ ]:

# ============================================================
# PHASE II — E1.7
# SIoT RELATION OVERLAP AND STRUCTURAL SEMANTICS AUDIT
# ============================================================
# Uses E1.6 output; does NOT reread dense matrices.
# No synthetic trust and no model training.
# ============================================================

from pathlib import Path
from itertools import combinations
import zipfile
import json
import numpy as np
import pandas as pd

BASE = Path(
    "/kaggle/working/phase2_adaptive_siot/"
    "E1_trust_scoring/santander"
)
GRAPH_DIR = BASE / "sparse_graphs"
RELATIONS = ["OOR", "POR", "C_LOR", "SOR", "SOR2"]
N = 16216

edges = pd.read_csv(
    GRAPH_DIR / "siot_multirelational_edges.csv.gz"
)

with zipfile.ZipFile(
    BASE / "raw" / "objects_description.zip"
) as z:
    with z.open("objects_description.csv") as f:
        devices = pd.read_csv(f)

assert len(edges) == 292234
assert devices["id_device"].is_unique
assert set(RELATIONS).issubset(edges.columns)

for col in RELATIONS:
    assert edges[col].isin([0, 1]).all()

assert (edges[RELATIONS].sum(axis=1) >= 1).all()
assert (edges["source"] != edges["target"]).all()

# Encode directed edges as unique integer keys.
keys = (
    (edges["source"].to_numpy(np.int64) - 1) * N
    + edges["target"].to_numpy(np.int64) - 1
)
reverse_keys = (
    (edges["target"].to_numpy(np.int64) - 1) * N
    + edges["source"].to_numpy(np.int64) - 1
)

assert len(np.unique(keys)) == len(keys)

is_reciprocal = np.isin(reverse_keys, keys)

print("=" * 68)
print("E1.7 — SIoT STRUCTURAL SEMANTICS")
print("=" * 68)

print("\n[1] GRAPH RECIPROCITY")
print("Directed edges:", len(edges))
print("Reciprocated directed edges:", int(is_reciprocal.sum()))
print("Reciprocity fraction:", round(is_reciprocal.mean(), 6))

# Exact pairwise overlaps.
print("\n[2] PAIRWISE RELATION OVERLAP")

pairs = []
for a, b in combinations(RELATIONS, 2):
    mask_a = edges[a].to_numpy(bool)
    mask_b = edges[b].to_numpy(bool)

    intersection = int((mask_a & mask_b).sum())
    union = int((mask_a | mask_b).sum())

    record = {
        "relation_a": a,
        "relation_b": b,
        "intersection": intersection,
        "union": union,
        "jaccard": intersection / union if union else 0,
        "identical": bool(np.array_equal(mask_a, mask_b))
    }
    pairs.append(record)

pair_df = pd.DataFrame(pairs)
print(pair_df.to_string(index=False))

# Count exact relationship-type combinations.
print("\n[3] OBSERVED RELATION COMBINATIONS")

combination_counts = (
    edges[RELATIONS]
    .astype(str)
    .agg("".join, axis=1)
    .value_counts()
)

def decode_combo(bits):
    return "+".join(
        name for name, bit in zip(RELATIONS, bits)
        if bit == "1"
    )

combinations_df = pd.DataFrame({
    "relation_combination": [
        decode_combo(bits)
        for bits in combination_counts.index
    ],
    "edge_count": combination_counts.to_numpy()
})

print(combinations_df.to_string(index=False))

# Verify that edge endpoints match device inventory.
print("\n[4] DEVICE ID AND OWNER AUDIT")

valid_ids = set(devices["id_device"].astype(int))
endpoints_valid = (
    edges["source"].isin(valid_ids).all()
    and edges["target"].isin(valid_ids).all()
)

print("All endpoints mapped to devices:", endpoints_valid)
assert endpoints_valid

owner = devices.set_index("id_device")["id_user"]

source_owner = edges["source"].map(owner)
target_owner = edges["target"].map(owner)

known_owners = source_owner.notna() & target_owner.notna()
same_owner = (
    known_owners & source_owner.eq(target_owner)
)

print("Known-owner directed edges:", int(known_owners.sum()))
print("Same-owner directed edges:", int(same_owner.sum()))

# Same-owner statistics by relationship type.
owner_rows = []
for rel in RELATIONS:
    mask = edges[rel].eq(1)
    count = int(mask.sum())
    owner_rows.append({
        "relation": rel,
        "edges": count,
        "same_owner_edges": int((mask & same_owner).sum()),
        "same_owner_fraction": (
            float((mask & same_owner).sum() / count)
            if count else None
        )
    })

owner_df = pd.DataFrame(owner_rows)
print("\n[5] OWNER ASSOCIATION BY RELATION")
print(owner_df.to_string(index=False))

# Save new Phase II audit artifacts.
pair_df.to_csv(
    GRAPH_DIR / "e1_7_relation_overlap.csv",
    index=False
)
combinations_df.to_csv(
    GRAPH_DIR / "e1_7_relation_combinations.csv",
    index=False
)
owner_df.to_csv(
    GRAPH_DIR / "e1_7_owner_associations.csv",
    index=False
)

report = {
    "nodes": N,
    "directed_edges": len(edges),
    "reciprocated_directed_edges": int(is_reciprocal.sum()),
    "reciprocity_fraction": float(is_reciprocal.mean()),
    "all_endpoints_valid": bool(endpoints_valid),
    "pairwise_overlap": pairs,
    "relation_combinations": combinations_df.to_dict("records"),
    "owner_associations": owner_rows,
    "trust_labels_present": False,
    "temporal_observations_present": False,
}

with open(
    GRAPH_DIR / "e1_7_structural_semantics.json",
    "w", encoding="utf-8"
) as f:
    json.dump(report, f, indent=2)

print("\n" + "=" * 68)
print("E1.7 COMPLETED")
print("=" * 68)
print("No synthetic trust was generated.")
print("No temporal behavior was inferred.")
print("No model was trained.")
print("NEXT: Freeze mathematical definition of the new method.")



# Phase II — Model Development
## Trust-Aware Adaptive GNN Sparsification for SIoT

### Structural Dataset Preparation — Completed

- Santander SIoT devices: 16,216
- Directed adjacency entries: 292,234
- Unique undirected social relationships: 146,117
- Relation types: OOR, POR, C-LOR, SOR, SOR2
- Fully reciprocal composite adjacency
- Device ownership and device-type metadata available

### Scientific Objective

Learn a GNN-based adaptive sparsification policy that
reduces the number of retained SIoT relationships while
preserving trust-relevant information and downstream
network-service utility.

### Evaluation Tracks

**Track A — SIoT structural evaluation**

Use the observed Santander topology and documented
synthetic trust observations with controlled temporal
changes. Synthetic trust is not ground truth from
deployed IoT devices.

**Track B — Observed signed trust evaluation**

Use Bitcoin Alpha and Bitcoin OTC with chronological
splits and genuinely observed positive/negative ratings.

### Required Method Properties

1. A GNN participates in edge-retention decisions.
2. Historical trust evidence explicitly influences
   edge selection.
3. The retention policy responds to changing graph
   or trust conditions.
4. Sparsification quality is assessed at matched budgets.
5. Computational savings are measured end to end.
6. Future trust observations are never used in the
   current decision.

### Planned Model Comparison

- GNN-Fixed
- GNN-Adaptive
- Trust-GNN-Adaptive
- Random and structural sparsification baselines

### Primary Outcomes

- Edge retention and reduction
- Predictive trust utility
- Graph connectivity and coverage
- GNN computation time and peak memory
- Temporal adaptation performance
- Performance across random seeds

### Next Experiment

E2.0 — Controlled temporal trust scenario generation
on the verified Santander topology.

All simulated latent variables must remain hidden
from the learning models.

Status: MODEL DEVELOPMENT STARTED


In [ ]:

# ============================================================
# PHASE II — E2.0
# CONTROLLED TEMPORAL TRUST SIMULATION FOR SIoT
# ============================================================
# Observed topology: Santander SIoT
# Trust and interaction outcomes: SYNTHETIC
# No observed real-world SIoT trust labels are claimed.
# No model training in this cell.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import json

SEED = 42
rng = np.random.default_rng(SEED)

N = 16216
T = 12
INTERACTION_RATE = 0.15
SHOCK_START = 6
SHOCK_NODE_FRACTION = 0.08

BASE = Path(
    "/kaggle/working/phase2_adaptive_siot/"
    "E1_trust_scoring/santander"
)

SOURCE = (
    BASE / "sparse_graphs" /
    "siot_multirelational_edges.csv.gz"
)

OUT = Path(
    "/kaggle/working/phase2_adaptive_siot/"
    "E2_temporal_trust"
)

OBS = OUT / "observed"
HIDDEN = OUT / "simulation_ground_truth"

OBS.mkdir(parents=True, exist_ok=True)
HIDDEN.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Load Santander social topology
# ------------------------------------------------------------

edges = pd.read_csv(SOURCE)

RELATIONS = ["OOR", "POR", "C_LOR", "SOR", "SOR2"]

assert len(edges) == 292234
assert set(RELATIONS).issubset(edges.columns)

# Convert fully reciprocal directed topology to unique
# undirected relationships. This does NOT make trust
# inherently symmetric; it defines a shared service link.
edges["u"] = np.minimum(
    edges["source"], edges["target"]
)
edges["v"] = np.maximum(
    edges["source"], edges["target"]
)

undirected = (
    edges[["u", "v"] + RELATIONS]
    .drop_duplicates(subset=["u", "v"])
    .sort_values(["u", "v"])
    .reset_index(drop=True)
)

assert len(undirected) == 146117

# Check symmetry of relation-type indicators.
reverse_check = edges.set_index(
    ["source", "target"]
)[RELATIONS]

forward_values = reverse_check.to_numpy()

reverse_values = reverse_check.reindex(
    pd.MultiIndex.from_arrays([
        edges["target"].to_numpy(),
        edges["source"].to_numpy()
    ])
).to_numpy()

assert np.array_equal(
    forward_values, reverse_values
), "Relation types differ in opposite directions."

M = len(undirected)

u = undirected["u"].to_numpy(np.int32) - 1
v = undirected["v"].to_numpy(np.int32) - 1

print("=" * 70)
print("E2.0 — TEMPORAL TRUST SIMULATION")
print("=" * 70)
print("Devices:", N)
print("Unique social links:", M)
print("Time steps:", T)

# ------------------------------------------------------------
# 2. Hidden device reliability
# ------------------------------------------------------------
# Independent from the categorical relationship labels.
# Beta distribution models heterogeneous reliability.
# These values must NEVER be model input features.

latent_reliability = rng.beta(
    a=5.0, b=2.0, size=N
)

def logit(x):
    x = np.clip(x, 1e-6, 1 - 1e-6)
    return np.log(x / (1 - x))

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -30, 30)))

base_logit = logit(latent_reliability)

# Link-specific hidden effects: NOT determined by type.
edge_effect = rng.normal(
    loc=0.0, scale=0.45, size=M
)

# ------------------------------------------------------------
# 3. Controlled temporal changes
# ------------------------------------------------------------
# One unobserved device shock begins at time step 6.
# Affected nodes are selected independently of labels.

n_shocked = int(round(
    SHOCK_NODE_FRACTION * N
))

shocked_nodes = rng.choice(
    N, size=n_shocked, replace=False
)

shock_indicator = np.zeros(N, dtype=np.float32)
shock_indicator[shocked_nodes] = 1.0

# Latent environment evolves through an AR(1) process.
temporal_state = np.zeros(N, dtype=np.float32)

events = []
latent_summary = []

for t in range(T):

    temporal_state = (
        0.85 * temporal_state
        + rng.normal(0, 0.12, size=N)
    ).astype(np.float32)

    # Shock decays gradually after its introduction.
    if t >= SHOCK_START:
        elapsed = t - SHOCK_START
        shock_strength = -1.6 * (0.90 ** elapsed)
    else:
        shock_strength = 0.0

    current_logit = (
        base_logit
        + temporal_state
        + shock_strength * shock_indicator
    )

    # A shared undirected link is treated as a
    # symmetric service-interaction opportunity.
    # Success depends on BOTH endpoints.
    #
    # This is a simulation assumption, NOT a claim
    # that real SIoT trust is symmetric.

    link_logit = (
        0.5 * (
            current_logit[u]
            + current_logit[v]
        )
        + edge_effect
    )

    success_probability = sigmoid(link_logit)

    # Only some eligible social links interact
    # at each time step.
    observed_mask = (
        rng.random(M) < INTERACTION_RATE
    )

    selected = np.flatnonzero(observed_mask)

    outcomes = (
        rng.random(len(selected))
        < success_probability[selected]
    ).astype(np.int8)

    # Independent observation noise: occasional
    # mistaken reports of interaction outcomes.
    flip_mask = rng.random(len(selected)) < 0.03
    outcomes[flip_mask] = 1 - outcomes[flip_mask]

    step = pd.DataFrame({
        "time": np.full(
            len(selected), t, dtype=np.int16
        ),
        "source": (u[selected] + 1).astype(np.int32),
        "target": (v[selected] + 1).astype(np.int32),
        "success": outcomes,
    })

    events.append(step)

    latent_summary.append({
        "time": t,
        "shock_strength": float(shock_strength),
        "mean_hidden_success_probability": float(
            success_probability.mean()
        )
    })

    print(
        f"t={t:02d} | "
        f"observations={len(step):6,d} | "
        f"observed_success={outcomes.mean():.4f}"
    )

# ------------------------------------------------------------
# 4. Chronological splits
# ------------------------------------------------------------

observations = pd.concat(
    events, ignore_index=True
)

observations = observations.sort_values(
    ["time", "source", "target"]
).reset_index(drop=True)

# Train: 0-7
# Validation: 8-9
# Test: 10-11
#
# Within each period, any prediction must use only
# observations strictly earlier than the prediction
# time step. This is enforced in later model code.

train = observations[
    observations["time"] <= 7
].copy()

validation = observations[
    observations["time"].between(8, 9)
].copy()

test = observations[
    observations["time"] >= 10
].copy()

assert train["time"].max() < validation["time"].min()
assert validation["time"].max() < test["time"].min()

# ------------------------------------------------------------
# 5. Persist observed data separately
# ------------------------------------------------------------

observations.to_csv(
    OBS / "all_interactions.csv.gz",
    index=False, compression="gzip"
)

train.to_csv(
    OBS / "train.csv.gz",
    index=False, compression="gzip"
)

validation.to_csv(
    OBS / "validation.csv.gz",
    index=False, compression="gzip"
)

test.to_csv(
    OBS / "test.csv.gz",
    index=False, compression="gzip"
)

# Save original edge identities and relation types.
# IMPORTANT: No hidden reliability values here.
undirected.to_csv(
    OBS / "siot_social_topology.csv.gz",
    index=False, compression="gzip"
)

# ------------------------------------------------------------
# 6. Save hidden simulator state
# ------------------------------------------------------------
# Audit only. Later learning code must NOT read this
# directory or use any latent fields as features.

np.savez_compressed(
    HIDDEN / "hidden_simulator_state.npz",
    latent_reliability=latent_reliability,
    edge_effect=edge_effect,
    shocked_device_ids=shocked_nodes + 1
)

pd.DataFrame(latent_summary).to_csv(
    HIDDEN / "latent_temporal_summary.csv",
    index=False
)

# ------------------------------------------------------------
# 7. Save transparent simulation protocol
# ------------------------------------------------------------

protocol = {
    "name": "Santander SIoT synthetic temporal trust",
    "seed": SEED,
    "real_topology": True,
    "real_trust_observations": False,
    "simulated_trust_observations": True,
    "time_steps": T,
    "interaction_rate": INTERACTION_RATE,
    "shock_start": SHOCK_START,
    "shock_node_fraction": SHOCK_NODE_FRACTION,
    "device_reliability_distribution": "Beta(5,2)",
    "edge_effect_distribution": "Normal(0,0.45)",
    "temporal_process": "AR(1), rho=0.85, sigma=0.12",
    "observation_flip_probability": 0.03,
    "relation_type_used_in_simulator": False,
    "split": {
        "train": [0, 7],
        "validation": [8, 9],
        "test": [10, 11]
    },
    "scientific_limitations": [
        "Trust observations are simulated, not measured.",
        "The shock mechanism is a controlled assumption.",
        "Social link eligibility is static.",
        "Interactions are generated independently per time step.",
        "Device hidden reliability is unavailable to models.",
        "Shared-link interaction success is symmetric.",
        "Temporal adaptability has not yet been demonstrated."
    ]
}

with open(
    OUT / "e2_0_simulation_protocol.json",
    "w", encoding="utf-8"
) as f:
    json.dump(
        protocol, f,
        indent=2, ensure_ascii=False
    )

# ------------------------------------------------------------
# 8. Final audit
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("E2.0 COMPLETED")
print("=" * 70)

print("Total observations:", len(observations))
print("Train:", len(train))
print("Validation:", len(validation))
print("Test:", len(test))

print(
    "Overall observed success rate:",
    round(float(observations["success"].mean()), 4)
)

print(
    "Overall observed failure rate:",
    round(float(1 - observations["success"].mean()), 4)
)

print("\nObserved data directory:")
print(OBS)

print("\nHidden simulator state:")
print(HIDDEN)

print("\nSimulation protocol:")
print(OUT / "e2_0_simulation_protocol.json")

print("\nNo GNN was trained.")
print("No real-world trust labels were claimed.")
print("Phase I artifacts remain untouched.")
print("\nNEXT: E2.1 — Temporal trust evidence construction.")


In [ ]:

# ============================================================
# PHASE II — E2.1
# CAUSAL TEMPORAL TRUST-EVIDENCE CONSTRUCTION
# ============================================================
# Every snapshot t uses observations from time < t.
# No hidden simulator state is accessed.
# No model training.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import json
import hashlib

N = 16216
T = 12
GAMMA = 0.80
CONFIDENCE_K = 5.0

ROOT = Path(
    "/kaggle/working/phase2_adaptive_siot"
)

OBS = ROOT / "E2_temporal_trust" / "observed"
OUT = ROOT / "E2_temporal_trust" / "causal_features"
OUT.mkdir(parents=True, exist_ok=True)

topology = pd.read_csv(
    OBS / "siot_social_topology.csv.gz"
)

events = pd.read_csv(
    OBS / "all_interactions.csv.gz"
)

# Canonical undirected edge ordering.
topology = topology.sort_values(
    ["u", "v"]
).reset_index(drop=True)

M = len(topology)

assert M == 146117
assert len(events) == 263116
assert topology[["u", "v"]].duplicated().sum() == 0

u = topology["u"].to_numpy(np.int64) - 1
v = topology["v"].to_numpy(np.int64) - 1

assert np.all(u < v)
assert np.all((u >= 0) & (v < N))

edge_keys = u * N + v

assert np.all(np.diff(edge_keys) > 0)

# Convert observation endpoints to canonical edge IDs.
ev_u = (
    events["source"].to_numpy(np.int64) - 1
)
ev_v = (
    events["target"].to_numpy(np.int64) - 1
)

assert np.all(ev_u < ev_v)

event_keys = ev_u * N + ev_v
event_edge_id = np.searchsorted(
    edge_keys, event_keys
)

assert np.all(event_edge_id < M)
assert np.array_equal(
    edge_keys[event_edge_id], event_keys
)

event_time = events["time"].to_numpy(np.int16)
event_success = events["success"].to_numpy(np.int8)

assert set(np.unique(event_success)).issubset({0, 1})
assert event_time.min() == 0
assert event_time.max() == T - 1

# One observed outcome at most per edge and time.
event_unique_key = (
    event_time.astype(np.int64) * M
    + event_edge_id
)
assert len(np.unique(event_unique_key)) == len(events)

# Accumulators: updated only AFTER snapshot t is saved.
edge_s = np.zeros(M, dtype=np.float32)
edge_f = np.zeros(M, dtype=np.float32)

decay_s = np.zeros(M, dtype=np.float32)
decay_f = np.zeros(M, dtype=np.float32)

node_s = np.zeros(N, dtype=np.float32)
node_f = np.zeros(N, dtype=np.float32)

last_seen = np.full(M, -1, dtype=np.int16)

FEATURE_NAMES = [
    "edge_history_count",
    "edge_success_rate",
    "edge_decayed_count",
    "edge_decayed_success_rate",
    "edge_evidence_confidence",
    "edge_age_normalized",
    "edge_never_observed",
    "node_u_success_rate",
    "node_v_success_rate",
    "node_u_evidence_confidence",
    "node_v_evidence_confidence",
    "edge_recent_minus_longterm"
]

def success_rate(s, f):
    return (s + 1.0) / (s + f + 2.0)

def confidence(s, f):
    n = s + f
    return n / (n + CONFIDENCE_K)

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for part in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(part)
    return h.hexdigest()

audit = []
cumulative_observations = 0

print("=" * 72)
print("E2.1 — CAUSAL TRUST-EVIDENCE FEATURES")
print("=" * 72)
print("Edges:", M)
print("Devices:", N)
print("Time steps:", T)
print("Temporal decay factor:", GAMMA)
print("Feature count:", len(FEATURE_NAMES))

for t in range(T):

    # Advance exponential decay to current time.
    if t > 0:
        decay_s *= GAMMA
        decay_f *= GAMMA

    history_n = edge_s + edge_f
    history_rate = success_rate(edge_s, edge_f)

    decayed_n = decay_s + decay_f
    decayed_rate = success_rate(decay_s, decay_f)

    node_rate = success_rate(node_s, node_f)
    node_conf = confidence(node_s, node_f)

    never_seen = (last_seen == -1)

    age = np.where(
        never_seen,
        T,
        t - last_seen
    ).astype(np.float32)

    age_normalized = age / T

    X = np.column_stack([
        history_n,
        history_rate,
        decayed_n,
        decayed_rate,
        confidence(edge_s, edge_f),
        age_normalized,
        never_seen.astype(np.float32),
        node_rate[u],
        node_rate[v],
        node_conf[u],
        node_conf[v],
        decayed_rate - history_rate
    ]).astype(np.float32)

    assert X.shape == (M, len(FEATURE_NAMES))
    assert np.isfinite(X).all()

    # Strict causality invariant:
    # each previous observation contributes exactly
    # one unit to the historical edge count.
    assert int(history_n.sum()) == (
        cumulative_observations
    ), f"History mismatch at time {t}"

    # This snapshot is saved BEFORE time-t outcomes
    # are added to the history.
    snapshot_path = OUT / f"snapshot_t{t:02d}.npz"

    np.savez_compressed(
        snapshot_path,
        features=X,
        time=np.array(t, dtype=np.int16),
        feature_names=np.array(FEATURE_NAMES)
    )

    # Only now access observed outcomes at time t.
    mask_t = (event_time == t)

    indices = event_edge_id[mask_t]
    outcomes = event_success[mask_t]

    successes = outcomes.astype(np.float32)
    failures = 1.0 - successes

    # All timestamps were verified. No duplicate
    # edge-time observations are allowed.
    np.add.at(edge_s, indices, successes)
    np.add.at(edge_f, indices, failures)

    np.add.at(decay_s, indices, successes)
    np.add.at(decay_f, indices, failures)

    np.add.at(node_s, u[indices], successes)
    np.add.at(node_s, v[indices], successes)

    np.add.at(node_f, u[indices], failures)
    np.add.at(node_f, v[indices], failures)

    last_seen[indices] = t

    cumulative_observations += len(indices)

    record = {
        "time": int(t),
        "history_observations_before_t": int(
            history_n.sum()
        ),
        "current_t_observations_added_after_snapshot": int(
            len(indices)
        ),
        "edges_with_history_before_t": int(
            (history_n > 0).sum()
        ),
        "mean_historical_success_rate": float(
            history_rate.mean()
        ),
        "mean_evidence_confidence": float(
            confidence(edge_s - np.bincount(
                indices,
                weights=successes,
                minlength=M
            ).astype(np.float32),
            edge_f - np.bincount(
                indices,
                weights=failures,
                minlength=M
            ).astype(np.float32)
            ).mean()
        ),
        "snapshot_file": snapshot_path.name
    }

    audit.append(record)

    print(
        f"t={t:02d} | "
        f"past_events={record['history_observations_before_t']:6,d} | "
        f"observed_edges={record['edges_with_history_before_t']:6,d} | "
        f"new_events={len(indices):6,d}"
    )

# Final cumulative integrity verification.
assert cumulative_observations == len(events)

# Save the edge mapping separately.
# Same row index as features in every snapshot.
topology.to_csv(
    OUT / "edge_index_mapping.csv.gz",
    index=False,
    compression="gzip"
)

report = {
    "experiment": "E2.1",
    "observed_siot_topology": True,
    "trust_outcomes_are_synthetic": True,
    "strictly_past_observations_only": True,
    "feature_names": FEATURE_NAMES,
    "temporal_decay": GAMMA,
    "confidence_k": CONFIDENCE_K,
    "snapshots": audit,
    "total_observations": int(len(events)),
    "hidden_state_used": False,
    "model_trained": False
}

with open(
    OUT / "e2_1_causal_feature_audit.json",
    "w", encoding="utf-8"
) as f:
    json.dump(report, f, indent=2)

# Reopen first and last saved snapshot.
with np.load(OUT / "snapshot_t00.npz") as first:
    X0 = first["features"]

with np.load(OUT / "snapshot_t11.npz") as last:
    X11 = last["features"]

# t=0 must have no observational history.
assert np.all(X0[:, 0] == 0)
assert np.all(X0[:, 1] == 0.5)
assert np.all(X0[:, 6] == 1)

assert np.all(X11[:, 0] >= 0)

print("\n" + "=" * 72)
print("E2.1 COMPLETED")
print("=" * 72)
print("Snapshots saved:", T)
print("Feature dimensions:", X11.shape)
print("Initial snapshot history: ZERO — verified")
print("Final snapshot historical observations:",
      int(X11[:, 0].sum()))
print("Hidden simulator state accessed: NO")
print("No GNN trained.")
print("Output:", OUT)
print("\nNEXT: E2.2 — GNN sparsification model definition.")


In [ ]:

# ============================================================
# PHASE II — E2.2
# FREEZE TAGS ARCHITECTURE AND EXPERIMENT CONTRACT
# ============================================================

from pathlib import Path
import json

ROOT = Path(
    "/kaggle/working/phase2_adaptive_siot"
)
OUT = ROOT / "E2_temporal_trust" / "model"
OUT.mkdir(parents=True, exist_ok=True)

spec = {
    "method": "TAGS",
    "full_name": (
        "Trust-Aware Adaptive Graph Sparsification"
    ),
    "status": "pre-implementation specification",
    "dataset": "Santander SIoT",
    "nodes": 16216,
    "undirected_edges": 146117,
    "time_steps": 12,

    "encoder": {
        "type": "GraphSAGE",
        "layers": 2,
        "hidden_dim": 32,
        "symmetric_undirected_links": True,
    },

    "edge_scorer": {
        "type": "MLP",
        "inputs": [
            "endpoint_embeddings",
            "historical_trust_features",
            "social_relation_types"
        ],
        "future_observations_allowed": False
    },

    "adaptive_controller": {
        "type": "learnable_local_controller",
        "inputs": [
            "historical_node_evidence",
            "normalized_node_degree",
            "time-varying_context"
        ],
        "shared_undirected_edge_decision": True,
        "global_budget_constraint": True
    },

    "mask": {
        "training": "Hard-Concrete candidate",
        "inference": "deterministic global Top-K",
        "pilot_retention": 0.50,
        "sensitivity_retention": [0.30, 0.50, 0.70]
    },

    "objective": {
        "task": "next-step interaction success prediction",
        "regularizers": [
            "sparsity_budget",
            "trust_information_preservation"
        ],
        "avoid_sign_only_trust_filtering": True
    },

    "evaluation": {
        "train_times": list(range(0, 8)),
        "validation_times": [8, 9],
        "test_times": [10, 11],
        "pilot_seed": 42,
        "final_seeds": [42, 43, 44, 45, 46],
        "primary_metrics": [
            "failure_class_average_precision",
            "runtime",
            "retained_edge_fraction"
        ],
        "secondary_metrics": [
            "ROC_AUC",
            "connectivity",
            "node_coverage",
            "peak_memory",
            "adaptation_after_shift"
        ],
        "ablation": [
            "GNN-Fixed",
            "GNN-Adaptive",
            "Trust-GNN-Adaptive"
        ]
    },

    "scientific_safeguards": [
        "No future interaction outcomes in features",
        "No access to hidden simulator states",
        "Same edge budget in primary comparisons",
        "Same backbone and training protocol",
        "No test-driven hyperparameter selection",
        "Report complete sparsification overhead",
        "No claim of real-world SIoT trust labels",
        "No claim of attack robustness without testing"
    ]
}

path = OUT / "tags_architecture_spec_v1.json"

with open(path, "w", encoding="utf-8") as f:
    json.dump(
        spec, f,
        indent=2,
        ensure_ascii=False
    )

print("=" * 68)
print("E2.2 — TAGS ARCHITECTURE SPECIFICATION")
print("=" * 68)

print("Method:", spec["method"])
print("Encoder:", spec["encoder"]["type"])
print("Hidden dimension:", spec["encoder"]["hidden_dim"])
print("Edge scorer:", spec["edge_scorer"]["type"])
print("Controller:", spec["adaptive_controller"]["type"])
print("Training mask:", spec["mask"]["training"])
print("Pilot retention:", spec["mask"]["pilot_retention"])
print("Training times:", spec["evaluation"]["train_times"])
print("Validation times:", spec["evaluation"]["validation_times"])
print("Test times:", spec["evaluation"]["test_times"])

assert path.is_file()

print("\nE2.2 COMPLETED")
print("Protocol saved:", path)
print("No model trained.")
print("No previous artifacts modified.")
print("NEXT: E2.3 — PyTorch TAGS implementation and smoke test.")


In [ ]:

# ============================================================
# PHASE II — E2.3
# TAGS: PYTORCH IMPLEMENTATION AND GRADIENT SMOKE TEST
# ============================================================
# No model fitting or optimizer updates in this cell.
# Temporal features: only information before t=7.
# The simulator's hidden state is never accessed.
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import zipfile

ROOT = Path("/kaggle/working/phase2_adaptive_siot")
BASE = ROOT / "E2_temporal_trust"
OBS = BASE / "observed"
FEATURES = BASE / "causal_features"
MODEL_DIR = BASE / "model"

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
torch.manual_seed(42)
np.random.seed(42)

N = 16216
T_SMOKE = 7
KEEP_RATIO = 0.50
RELATIONS = ["OOR", "POR", "C_LOR", "SOR", "SOR2"]

print("=" * 72)
print("E2.3 — TAGS PYTORCH SMOKE TEST")
print("=" * 72)
print("Device:", DEVICE)

# ------------------------------------------------------------
# 1. Load real topology and causal historical features
# ------------------------------------------------------------

topology = pd.read_csv(
    FEATURES / "edge_index_mapping.csv.gz"
)

with np.load(
    FEATURES / f"snapshot_t{T_SMOKE:02d}.npz"
) as snapshot:
    edge_history = snapshot["features"].copy()
    feature_names = snapshot["feature_names"].tolist()

assert len(topology) == 146117
assert edge_history.shape == (146117, 12)
assert set(RELATIONS).issubset(topology.columns)

M = len(topology)

src = topology["u"].to_numpy(np.int64) - 1
dst = topology["v"].to_numpy(np.int64) - 1

assert np.all(src < dst)

edge_keys = src * N + dst
assert np.all(np.diff(edge_keys) > 0)

# Historical edge features:
# counts, smoothed success, recency, confidence,
# and node-level historical evidence.
#
# Counts are log-scaled; no labels from t=7 are used.

edge_history[:, 0] = np.log1p(edge_history[:, 0])
edge_history[:, 2] = np.log1p(edge_history[:, 2])

# Make endpoint-related fields permutation-invariant.
# Columns 7-8: endpoint historical success rates
# Columns 9-10: endpoint evidence confidence
for left, right in [(7, 8), (9, 10)]:
    a = edge_history[:, left].copy()
    b = edge_history[:, right].copy()
    edge_history[:, left] = (a + b) / 2
    edge_history[:, right] = np.abs(a - b)

relation_matrix = topology[RELATIONS].to_numpy(
    dtype=np.float32
)

edge_input = np.concatenate(
    [edge_history, relation_matrix],
    axis=1
).astype(np.float32)

# ------------------------------------------------------------
# 2. Build device features
# ------------------------------------------------------------

SANTANDER = (
    ROOT / "E1_trust_scoring" / "santander"
)

with zipfile.ZipFile(
    SANTANDER / "raw" / "objects_description.zip"
) as z:
    with z.open("objects_description.csv") as f:
        devices = pd.read_csv(f)

devices = devices.sort_values("id_device")
assert np.array_equal(
    devices["id_device"].to_numpy(),
    np.arange(1, N + 1)
)

device_types = devices["device_type"].to_numpy(
    dtype=np.int64
)

type_count = int(device_types.max()) + 1
type_onehot = np.eye(
    type_count, dtype=np.float32
)[device_types]

degree = np.bincount(
    np.concatenate([src, dst]),
    minlength=N
).astype(np.float32)

# Derive node-level historical evidence from
# the strictly causal edge-history snapshot.

historical_count = np.expm1(edge_history[:, 0])

# Recover the pre-log historical rate.
historical_success = edge_history[:, 1]

estimated_successes = (
    historical_success * (historical_count + 2.0)
    - 1.0
)

estimated_successes = np.clip(
    estimated_successes, 0, historical_count
)

node_count = np.bincount(
    np.concatenate([src, dst]),
    weights=np.concatenate([
        historical_count, historical_count
    ]),
    minlength=N
).astype(np.float32)

node_success_count = np.bincount(
    np.concatenate([src, dst]),
    weights=np.concatenate([
        estimated_successes, estimated_successes
    ]),
    minlength=N
).astype(np.float32)

node_success_rate = (
    node_success_count + 1.0
) / (node_count + 2.0)

node_confidence = node_count / (
    node_count + 5.0
)

node_dynamic = np.column_stack([
    np.log1p(degree),
    np.log1p(node_count),
    node_success_rate,
    node_confidence
]).astype(np.float32)

node_input = np.concatenate([
    type_onehot, node_dynamic
], axis=1).astype(np.float32)

# Controller sees historical evidence and degree,
# not outcomes from the prediction time step.
controller_input = node_dynamic

# ------------------------------------------------------------
# 3. Load labels ONLY for t=7 prediction targets
# ------------------------------------------------------------

train_events = pd.read_csv(
    OBS / "train.csv.gz"
)

targets = train_events.loc[
    train_events["time"] == T_SMOKE
].copy()

target_u = (
    targets["source"].to_numpy(np.int64) - 1
)
target_v = (
    targets["target"].to_numpy(np.int64) - 1
)

target_keys = target_u * N + target_v

target_ids = np.searchsorted(
    edge_keys, target_keys
)

assert np.all(target_ids < M)
assert np.array_equal(
    edge_keys[target_ids], target_keys
)

# Success = 1, failure = 0.
# This is used solely as a supervised target.
labels = targets["success"].to_numpy(
    dtype=np.float32
)

# ------------------------------------------------------------
# 4. Torch tensors
# ------------------------------------------------------------

def tensor(x):
    return torch.as_tensor(
        x, dtype=torch.float32, device=DEVICE
    )

X_node = tensor(node_input)
X_edge = tensor(edge_input)
X_controller = tensor(controller_input)

U = torch.as_tensor(
    src, dtype=torch.long, device=DEVICE
)
V = torch.as_tensor(
    dst, dtype=torch.long, device=DEVICE
)

target_ids_t = torch.as_tensor(
    target_ids, dtype=torch.long, device=DEVICE
)

Y = tensor(labels)

assert torch.isfinite(X_node).all()
assert torch.isfinite(X_edge).all()

# ------------------------------------------------------------
# 5. Weighted, undirected GraphSAGE
# ------------------------------------------------------------

class WeightedSAGELayer(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.self_linear = nn.Linear(
            input_dim, output_dim
        )
        self.neighbor_linear = nn.Linear(
            input_dim, output_dim, bias=False
        )

    def forward(self, x, u, v, weights):
        weights = weights.reshape(-1, 1)

        message = torch.zeros_like(x)
        message = message.index_add(
            0, u, x[v] * weights
        )
        message = message.index_add(
            0, v, x[u] * weights
        )

        degree = torch.zeros(
            (x.size(0), 1),
            device=x.device,
            dtype=x.dtype
        )

        degree = degree.index_add(
            0, u, weights
        )
        degree = degree.index_add(
            0, v, weights
        )

        mean_neighbor = message / degree.clamp(
            min=1e-6
        )

        return (
            self.self_linear(x)
            + self.neighbor_linear(mean_neighbor)
        )


class SAGEEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=32):
        super().__init__()
        self.layer1 = WeightedSAGELayer(
            input_dim, hidden_dim
        )
        self.layer2 = WeightedSAGELayer(
            hidden_dim, hidden_dim
        )

    def forward(self, x, u, v, weights):
        h = F.relu(
            self.layer1(x, u, v, weights)
        )
        return self.layer2(h, u, v, weights)


# ------------------------------------------------------------
# 6. Hard-Concrete differentiable masks
# ------------------------------------------------------------

class HardConcrete(nn.Module):
    def __init__(
        self,
        temperature=2/3,
        lower=-0.1,
        upper=1.1
    ):
        super().__init__()
        self.temperature = temperature
        self.lower = lower
        self.upper = upper

    def forward(self, logits, training=True):
        if training:
            uniform = torch.rand_like(
                logits
            ).clamp(1e-6, 1 - 1e-6)

            logistic_noise = (
                torch.log(uniform)
                - torch.log1p(-uniform)
            )

            z = torch.sigmoid(
                (logits + logistic_noise)
                / self.temperature
            )
        else:
            z = torch.sigmoid(logits)

        stretched = (
            z * (self.upper - self.lower)
            + self.lower
        )

        return stretched.clamp(0, 1)


# ------------------------------------------------------------
# 7. TAGS architecture
# ------------------------------------------------------------

class TAGS(nn.Module):
    def __init__(
        self,
        node_dim,
        edge_dim,
        controller_dim,
        hidden=32
    ):
        super().__init__()

        self.encoder = SAGEEncoder(
            node_dim, hidden
        )

        # Symmetric pair representation:
        # h_u + h_v and |h_u - h_v|.
        self.scorer = nn.Sequential(
            nn.Linear(
                hidden * 2 + edge_dim, 64
            ),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

        self.controller = nn.Sequential(
            nn.Linear(controller_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

        self.mask_layer = HardConcrete()

        # Prediction depends on sparse-graph
        # embeddings, not direct edge-history features.
        self.classifier = nn.Sequential(
            nn.Linear(hidden * 3, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(
        self,
        x_node,
        x_edge,
        x_controller,
        u,
        v,
        prediction_edge_ids
    ):
        full_weights = torch.ones(
            u.numel(),
            device=x_node.device
        )

        # First GNN: representations from the
        # observed social topology.
        h = self.encoder(
            x_node, u, v, full_weights
        )

        edge_representation = torch.cat([
            h[u] + h[v],
            torch.abs(h[u] - h[v]),
            x_edge
        ], dim=1)

        edge_logits = self.scorer(
            edge_representation
        ).squeeze(-1)

        # Local adaptive controller. Its outputs
        # affect one shared undirected edge decision.
        node_priority = self.controller(
            x_controller
        ).squeeze(-1)

        combined_logits = (
            edge_logits
            + 0.5 * (
                node_priority[u]
                + node_priority[v]
            )
        )

        mask = self.mask_layer(
            combined_logits,
            training=self.training
        )

        # Second GNN: masked social graph.
        z = self.encoder(
            x_node, u, v, mask
        )

        pu = u[prediction_edge_ids]
        pv = v[prediction_edge_ids]

        pair = torch.cat([
            z[pu] + z[pv],
            torch.abs(z[pu] - z[pv]),
            z[pu] * z[pv]
        ], dim=1)

        prediction_logits = self.classifier(
            pair
        ).squeeze(-1)

        return {
            "prediction_logits": prediction_logits,
            "edge_logits": combined_logits,
            "mask": mask,
            "node_priority": node_priority
        }


model = TAGS(
    node_dim=X_node.shape[1],
    edge_dim=X_edge.shape[1],
    controller_dim=X_controller.shape[1],
    hidden=32
).to(DEVICE)

# ------------------------------------------------------------
# 8. Smoke test: forward and gradients
# ------------------------------------------------------------

model.train()

output = model(
    X_node,
    X_edge,
    X_controller,
    U,
    V,
    target_ids_t
)

task_loss = F.binary_cross_entropy_with_logits(
    output["prediction_logits"],
    Y
)

# Pilot budget penalty, not the final objective.
budget_loss = (
    output["mask"].mean()
    - KEEP_RATIO
).square()

total_loss = task_loss + 2.0 * budget_loss

# Crucial test: do scorer and controller receive
# gradient from TASK LOSS, not just budget loss?
model.zero_grad(set_to_none=True)
task_loss.backward(retain_graph=True)

def gradient_norm(module):
    return sum(
        float(p.grad.detach().norm())
        for p in module.parameters()
        if p.grad is not None
    )

task_gradients = {
    "encoder": gradient_norm(model.encoder),
    "scorer": gradient_norm(model.scorer),
    "controller": gradient_norm(model.controller),
    "classifier": gradient_norm(model.classifier)
}

assert all(
    np.isfinite(g) and g > 0
    for g in task_gradients.values()
), f"Task gradient failure: {task_gradients}"

# Check complete objective independently.
model.zero_grad(set_to_none=True)
total_loss.backward()

total_gradients = {
    "encoder": gradient_norm(model.encoder),
    "scorer": gradient_norm(model.scorer),
    "controller": gradient_norm(model.controller),
    "classifier": gradient_norm(model.classifier)
}

assert torch.isfinite(total_loss)

# ------------------------------------------------------------
# 9. Deterministic exact-budget selection
# ------------------------------------------------------------

model.eval()

with torch.no_grad():
    deterministic = model(
        X_node,
        X_edge,
        X_controller,
        U,
        V,
        target_ids_t
    )

    scores = deterministic[
        "edge_logits"
    ]

    K = int(round(KEEP_RATIO * M))

    chosen = torch.topk(
        scores,
        k=K,
        largest=True,
        sorted=False
    ).indices

    selected = torch.zeros(
        M, dtype=torch.bool, device=DEVICE
    )
    selected[chosen] = True

    assert int(selected.sum()) == K

    # Exactly one decision per undirected edge.
    retained_fraction = (
        selected.float().mean().item()
    )

# ------------------------------------------------------------
# 10. Report
# ------------------------------------------------------------

results = {
    "experiment": "E2.3",
    "device": str(DEVICE),
    "time": T_SMOKE,
    "nodes": N,
    "undirected_edges": M,
    "prediction_examples": int(len(Y)),
    "task_loss": float(task_loss.detach().cpu()),
    "budget_loss": float(budget_loss.detach().cpu()),
    "total_loss": float(total_loss.detach().cpu()),
    "task_gradient_norms": task_gradients,
    "total_gradient_norms": total_gradients,
    "sampled_mask_mean": float(
        output["mask"].detach().mean().cpu()
    ),
    "deterministic_retained_edges": K,
    "deterministic_retained_fraction": (
        retained_fraction
    ),
    "no_optimizer_updates": True,
    "future_labels_as_features": False,
    "hidden_simulator_state_accessed": False
}

report_path = (
    MODEL_DIR / "e2_3_tags_smoke_report.json"
)

with open(
    report_path, "w", encoding="utf-8"
) as f:
    json.dump(results, f, indent=2)

print("\n" + "=" * 72)
print("E2.3 SMOKE TEST RESULTS")
print("=" * 72)

print("Task loss:", round(results["task_loss"], 6))
print("Budget loss:", round(results["budget_loss"], 6))
print("Sampled mask mean:",
      round(results["sampled_mask_mean"], 5))

print("\nTASK-LOSS GRADIENTS:")
for name, value in task_gradients.items():
    print(f"  {name:12s}: {value:.8f}")

print("\nTOTAL-LOSS GRADIENTS:")
for name, value in total_gradients.items():
    print(f"  {name:12s}: {value:.8f}")

print("\nEXACT-BUDGET INFERENCE:")
print("Original undirected edges:", M)
print("Retained edges:", K)
print(
    "Retained fraction:",
    round(retained_fraction, 6)
)

print("\nE2.3 COMPLETED")
print("All gradient assertions passed.")
print("No optimizer updates performed.")
print("Report:", report_path)
print("\nNEXT: E2.4 — Pilot training and validation.")


In [ ]:

# ============================================================
# PHASE II — E2.4
# TAGS PILOT TRAINING + HARD-BUDGET VALIDATION
# ============================================================
# Train: t=5,6,7 (subset of permitted training times)
# Validation: t=8,9
# Test: NEVER READ
# No simulator ground-truth files accessed.
# ============================================================

from pathlib import Path
import json
import time
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    log_loss
)

ROOT = Path("/kaggle/working/phase2_adaptive_siot")
BASE = ROOT / "E2_temporal_trust"
FEATURES = BASE / "causal_features"
OBS = BASE / "observed"
OUT = BASE / "model"

OUT.mkdir(parents=True, exist_ok=True)

TRAIN_TIMES = [5, 6, 7]
VAL_TIMES = [8, 9]
EPOCHS = 4
LR = 1e-3
WEIGHT_DECAY = 1e-4
BUDGET_WEIGHT = 0.002
KEEP_RATIO = 0.50
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert "TAGS" in globals(), (
    "Run E2.3 first to define the TAGS class."
)

# ------------------------------------------------------------
# 1. Reuse topology and static inputs from E2.3
# ------------------------------------------------------------

assert len(topology) == 146117
assert N == 16216

M = len(topology)

# Explicitly load only train and validation labels.
train_df = pd.read_csv(OBS / "train.csv.gz")
val_df = pd.read_csv(OBS / "validation.csv.gz")

assert set(train_df["time"].unique()).issubset(
    set(range(8))
)
assert set(val_df["time"].unique()).issubset(
    {8, 9}
)

# Use arrays from E2.3.
edge_keys_cpu = edge_keys.copy()

node_static = type_onehot.astype(np.float32)
degree_log = np.log1p(degree).astype(np.float32)

relation_static = relation_matrix.astype(np.float32)

# Topology tensors U, V were created in E2.3.
assert U.numel() == M
assert V.numel() == M

def prepare_snapshot(t, labels_df):
    with np.load(
        FEATURES / f"snapshot_t{t:02d}.npz"
    ) as data:
        history = data["features"].copy()

    assert history.shape == (M, 12)

    count = history[:, 0].copy()
    success_rate = history[:, 1].copy()

    # Undo Beta(1,1) smoothing to recover
    # historical success counts.
    successes = (
        success_rate * (count + 2.0) - 1.0
    )
    successes = np.clip(successes, 0, count)

    node_count = np.bincount(
        np.concatenate([src, dst]),
        weights=np.tile(count, 2),
        minlength=N
    ).astype(np.float32)

    node_success = np.bincount(
        np.concatenate([src, dst]),
        weights=np.tile(successes, 2),
        minlength=N
    ).astype(np.float32)

    node_rate = (
        node_success + 1.0
    ) / (node_count + 2.0)

    node_conf = node_count / (
        node_count + 5.0
    )

    dynamic_node = np.column_stack([
        degree_log,
        np.log1p(node_count),
        node_rate,
        node_conf
    ]).astype(np.float32)

    Xn = np.concatenate([
        node_static, dynamic_node
    ], axis=1)

    history[:, 0] = np.log1p(history[:, 0])
    history[:, 2] = np.log1p(history[:, 2])

    # Symmetric endpoint features.
    for a, b in [(7, 8), (9, 10)]:
        left = history[:, a].copy()
        right = history[:, b].copy()
        history[:, a] = (left + right) / 2
        history[:, b] = np.abs(left - right)

    Xe = np.concatenate([
        history, relation_static
    ], axis=1).astype(np.float32)

    step = labels_df[
        labels_df["time"] == t
    ]

    assert len(step) > 0

    su = step["source"].to_numpy(np.int64) - 1
    sv = step["target"].to_numpy(np.int64) - 1

    assert np.all(su < sv)

    keys = su * N + sv
    ids = np.searchsorted(edge_keys_cpu, keys)

    assert np.all(ids < M)
    assert np.array_equal(edge_keys_cpu[ids], keys)

    y = step["success"].to_numpy(np.float32)

    assert np.isfinite(Xn).all()
    assert np.isfinite(Xe).all()
    assert set(np.unique(y)).issubset({0.0, 1.0})

    def to_float(x):
        return torch.as_tensor(
            x, dtype=torch.float32, device=DEVICE
        )

    return {
        "time": t,
        "node": to_float(Xn),
        "edge": to_float(Xe),
        "controller": to_float(dynamic_node),
        "ids": torch.as_tensor(
            ids, dtype=torch.long, device=DEVICE
        ),
        "y": to_float(y),
        "y_numpy": y.astype(np.int8)
    }

snapshots = {}

for t in TRAIN_TIMES:
    snapshots[t] = prepare_snapshot(t, train_df)

for t in VAL_TIMES:
    snapshots[t] = prepare_snapshot(t, val_df)

print("=" * 70)
print("E2.4 — TAGS PILOT TRAINING")
print("=" * 70)
print("Device:", DEVICE)
print("Train times:", TRAIN_TIMES)
print("Validation times:", VAL_TIMES)
print("Epochs:", EPOCHS)
print("Global retention:", KEEP_RATIO)

# ------------------------------------------------------------
# 2. Create a fresh model, not the smoke-test instance
# ------------------------------------------------------------

pilot = TAGS(
    node_dim=snapshots[5]["node"].shape[1],
    edge_dim=snapshots[5]["edge"].shape[1],
    controller_dim=snapshots[5]["controller"].shape[1],
    hidden=32
).to(DEVICE)

optimizer = torch.optim.Adam(
    pilot.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

K = int(round(M * KEEP_RATIO))

# ------------------------------------------------------------
# 3. Exact hard-mask inference
# ------------------------------------------------------------

@torch.no_grad()
def predict_with_hard_mask(model, batch):
    model.eval()

    full_weights = torch.ones(
        M, device=DEVICE
    )

    h = model.encoder(
        batch["node"], U, V, full_weights
    )

    edge_rep = torch.cat([
        h[U] + h[V],
        torch.abs(h[U] - h[V]),
        batch["edge"]
    ], dim=1)

    score = model.scorer(
        edge_rep
    ).squeeze(-1)

    priority = model.controller(
        batch["controller"]
    ).squeeze(-1)

    score = score + 0.5 * (
        priority[U] + priority[V]
    )

    chosen = torch.topk(
        score, k=K, sorted=False
    ).indices

    mask = torch.zeros(
        M, device=DEVICE
    )
    mask[chosen] = 1.0

    assert int(mask.sum().item()) == K

    z = model.encoder(
        batch["node"], U, V, mask
    )

    pu = U[batch["ids"]]
    pv = V[batch["ids"]]

    pair = torch.cat([
        z[pu] + z[pv],
        torch.abs(z[pu] - z[pv]),
        z[pu] * z[pv]
    ], dim=1)

    logits = model.classifier(
        pair
    ).squeeze(-1)

    return (
        torch.sigmoid(logits)
        .detach().cpu().numpy(),
        chosen.detach().cpu().numpy()
    )

def evaluate_validation(model):
    outputs = []
    selected = {}

    for t in VAL_TIMES:
        batch = snapshots[t]

        probabilities, kept = predict_with_hard_mask(
            model, batch
        )

        selected[t] = kept

        outputs.append(pd.DataFrame({
            "time": t,
            "success": batch["y_numpy"],
            "pred_success": probabilities
        }))

    predictions = pd.concat(
        outputs, ignore_index=True
    )

    # Failure (negative) class is the primary outcome.
    y_failure = 1 - predictions[
        "success"
    ].to_numpy()

    p_failure = 1 - predictions[
        "pred_success"
    ].to_numpy()

    ap = average_precision_score(
        y_failure, p_failure
    )

    auc = roc_auc_score(
        y_failure, p_failure
    )

    bce = log_loss(
        y_failure,
        np.clip(p_failure, 1e-7, 1 - 1e-7),
        labels=[0, 1]
    )

    # Fraction of retained edges that changed
    # between consecutive validation snapshots.
    switch_fraction = (
        len(np.setxor1d(selected[8], selected[9]))
        / (2 * K)
    )

    return {
        "failure_AP": float(ap),
        "failure_ROC_AUC": float(auc),
        "BCE": float(bce),
        "selection_change_fraction": float(
            switch_fraction
        ),
        "predictions": predictions
    }

# ------------------------------------------------------------
# 4. Pilot training
# ------------------------------------------------------------

history_rows = []
best_ap = -float("inf")
best_epoch = None

checkpoint_path = (
    OUT / "tags_pilot_best_validation.pt"
)

start = time.perf_counter()

for epoch in range(1, EPOCHS + 1):

    epoch_task = []
    epoch_budget = []
    epoch_mask = []

    for t in TRAIN_TIMES:
        batch = snapshots[t]

        pilot.train()
        optimizer.zero_grad(set_to_none=True)

        result = pilot(
            batch["node"],
            batch["edge"],
            batch["controller"],
            U,
            V,
            batch["ids"]
        )

        task_loss = F.binary_cross_entropy_with_logits(
            result["prediction_logits"],
            batch["y"]
        )

        # Weak training-time budget regularization.
        # Exact 50% retention is enforced at validation.
        budget_loss = (
            result["mask"].mean()
            - KEEP_RATIO
        ).square()

        loss = (
            task_loss
            + BUDGET_WEIGHT * budget_loss
        )

        assert torch.isfinite(loss)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            pilot.parameters(), max_norm=5.0
        )

        optimizer.step()

        epoch_task.append(float(task_loss.item()))
        epoch_budget.append(float(budget_loss.item()))
        epoch_mask.append(
            float(result["mask"].mean().item())
        )

    validation = evaluate_validation(pilot)

    row = {
        "epoch": epoch,
        "train_task_loss": float(
            np.mean(epoch_task)
        ),
        "train_budget_loss": float(
            np.mean(epoch_budget)
        ),
        "train_sampled_mask_fraction": float(
            np.mean(epoch_mask)
        ),
        "val_failure_AP": validation["failure_AP"],
        "val_failure_ROC_AUC": (
            validation["failure_ROC_AUC"]
        ),
        "val_BCE": validation["BCE"],
        "val_selection_change_fraction": (
            validation["selection_change_fraction"]
        )
    }

    history_rows.append(row)

    if validation["failure_AP"] > best_ap:
        best_ap = validation["failure_AP"]
        best_epoch = epoch

        torch.save({
            "model_state_dict": pilot.state_dict(),
            "best_epoch": epoch,
            "validation_failure_AP": best_ap,
            "seed": SEED,
            "retention": KEEP_RATIO,
            "train_times": TRAIN_TIMES,
            "validation_times": VAL_TIMES
        }, checkpoint_path)

    print(
        f"Epoch {epoch:02d} | "
        f"train BCE={row['train_task_loss']:.5f} | "
        f"val failure AP={row['val_failure_AP']:.5f} | "
        f"val ROC={row['val_failure_ROC_AUC']:.5f} | "
        f"mask={row['train_sampled_mask_fraction']:.4f} | "
        f"selection change={row['val_selection_change_fraction']:.4f}",
        flush=True
    )

# ------------------------------------------------------------
# 5. Save pilot results
# ------------------------------------------------------------

elapsed = time.perf_counter() - start

history_df = pd.DataFrame(history_rows)

history_path = OUT / "e2_4_pilot_history.csv"
history_df.to_csv(history_path, index=False)

report = {
    "experiment": "E2.4",
    "pilot_only": True,
    "method": "TAGS",
    "seed": SEED,
    "train_times": TRAIN_TIMES,
    "validation_times": VAL_TIMES,
    "test_accessed": False,
    "epochs": EPOCHS,
    "best_epoch": best_epoch,
    "best_validation_failure_AP": best_ap,
    "retained_edges": K,
    "total_edges": M,
    "budget_weight": BUDGET_WEIGHT,
    "runtime_seconds": elapsed,
    "evaluation_mask": "exact hard Top-K",
    "model_checkpoint": str(checkpoint_path),
    "warnings": [
        "Pilot training uses only t=5,6,7.",
        "No baseline comparison has been completed.",
        "No independent test performance is available.",
        "Validation t=9 uses evidence observed at t=8.",
        "Controller adaptivity is not yet established.",
        "Training uses soft masks; inference uses hard Top-K."
    ]
}

report_path = OUT / "e2_4_pilot_report.json"

with open(report_path, "w") as f:
    json.dump(report, f, indent=2)

print("\n" + "=" * 70)
print("E2.4 COMPLETED — PILOT ONLY")
print("=" * 70)

print("Best epoch:", best_epoch)
print("Best validation failure AP:", round(best_ap, 6))
print("Runtime seconds:", round(elapsed, 2))
print("Checkpoint:", checkpoint_path)
print("History:", history_path)
print("Report:", report_path)

print("\nNo test data was read.")
print("No superiority claim is justified yet.")
print("NEXT: Diagnose task learning, mask utility,")
print("and adaptive-controller contribution.")


In [ ]:

# ============================================================
# PHASE II — E2.5
# TAGS DIAGNOSTIC: SIGNAL, SPARSIFICATION, BASELINES
# ============================================================
# Read-only evaluation; no neural-network training.
# Logistic regression is fitted on train times 5,6,7.
# Validation: t=8,9 only.
# Test files and hidden simulator states: NEVER ACCESSED.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import torch

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    log_loss
)

ROOT = Path("/kaggle/working/phase2_adaptive_siot")
OUT = ROOT / "E2_temporal_trust" / "model"
OUT.mkdir(parents=True, exist_ok=True)

REQUIRED = [
    "TAGS", "snapshots", "U", "V",
    "pilot", "M", "K", "DEVICE"
]
missing = [x for x in REQUIRED if x not in globals()]
assert not missing, f"Run E2.3 and E2.4 first: {missing}"

TRAIN_TIMES = [5, 6, 7]
VAL_TIMES = [8, 9]

checkpoint = torch.load(
    OUT / "tags_pilot_best_validation.pt",
    map_location=DEVICE,
    weights_only=True
)
pilot.load_state_dict(checkpoint["model_state_dict"])
pilot.eval()

assert checkpoint["best_epoch"] >= 1
assert int(M) == 146117
assert int(K) == 73058

print("=" * 72)
print("E2.5 — TAGS DIAGNOSTIC BASELINES")
print("=" * 72)
print("Best saved epoch:", checkpoint["best_epoch"])
print("Train times:", TRAIN_TIMES)
print("Validation times:", VAL_TIMES)
print("No test access.")

# ------------------------------------------------------------
# 1. Assemble training and validation features
# ------------------------------------------------------------
# Edge feature columns are from the causal snapshot.
# Columns 0:12 are historical trust evidence.
# Columns 12:17 are observed relationship indicators.

def get_features(batch):
    return batch["edge"][
        batch["ids"]
    ].detach().cpu().numpy()

def get_labels(batch):
    # Failure is the positive class here.
    return (
        1 - batch["y_numpy"].astype(np.int8)
    )

X_train = np.concatenate([
    get_features(snapshots[t])
    for t in TRAIN_TIMES
], axis=0)

y_train = np.concatenate([
    get_labels(snapshots[t])
    for t in TRAIN_TIMES
])

X_val = np.concatenate([
    get_features(snapshots[t])
    for t in VAL_TIMES
], axis=0)

y_val = np.concatenate([
    get_labels(snapshots[t])
    for t in VAL_TIMES
])

assert X_train.shape[1] == 17
assert X_val.shape[1] == 17
assert set(np.unique(y_train)) == {0, 1}
assert set(np.unique(y_val)) == {0, 1}

failure_prevalence = float(y_val.mean())

print("\nFAILURE PREVALENCE")
print("Training:", round(float(y_train.mean()), 6))
print("Validation:", round(failure_prevalence, 6))

# ------------------------------------------------------------
# 2. Lightweight prediction baselines
# ------------------------------------------------------------

# Historical smoothed failure rate, without fitting.
historical_failure = np.clip(
    1.0 - X_val[:, 1],
    1e-6, 1 - 1e-6
)

# Train-only constant prior.
constant_prior = np.full(
    len(y_val),
    float(y_train.mean()),
    dtype=np.float64
)

# A simple supervised model, with the same
# available historical edge evidence as TAGS.
linear = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=500,
        solver="lbfgs",
        random_state=42
    )
)

linear.fit(X_train, y_train)

linear_failure = linear.predict_proba(
    X_val
)[:, 1]

# ------------------------------------------------------------
# 3. Evaluate the frozen trained TAGS model using:
#    A: Full graph
#    B: Learned Top-K mask
#    C: Random Top-K masks, same budget
# ------------------------------------------------------------

@torch.no_grad()
def predict_with_mask(model, batch, mode, seed=42):
    model.eval()

    full_weights = torch.ones(
        M, device=DEVICE
    )

    h = model.encoder(
        batch["node"], U, V, full_weights
    )

    rep = torch.cat([
        h[U] + h[V],
        torch.abs(h[U] - h[V]),
        batch["edge"]
    ], dim=1)

    scores = model.scorer(rep).squeeze(-1)

    priority = model.controller(
        batch["controller"]
    ).squeeze(-1)

    scores = scores + 0.5 * (
        priority[U] + priority[V]
    )

    if mode == "full":
        mask = full_weights

    elif mode == "tags":
        kept = torch.topk(
            scores, k=K, sorted=False
        ).indices

        mask = torch.zeros(
            M, device=DEVICE
        )
        mask[kept] = 1.0

    elif mode == "random":
        generator = np.random.default_rng(seed)
        kept_np = generator.choice(
            M, size=K, replace=False
        )
        kept = torch.as_tensor(
            kept_np,
            dtype=torch.long,
            device=DEVICE
        )

        mask = torch.zeros(
            M, device=DEVICE
        )
        mask[kept] = 1.0

    else:
        raise ValueError(mode)

    z = model.encoder(
        batch["node"], U, V, mask
    )

    pu = U[batch["ids"]]
    pv = V[batch["ids"]]

    pair = torch.cat([
        z[pu] + z[pv],
        torch.abs(z[pu] - z[pv]),
        z[pu] * z[pv]
    ], dim=1)

    logits = model.classifier(
        pair
    ).squeeze(-1)

    # Return probability of FAILURE, not success.
    return (
        torch.sigmoid(-logits)
        .cpu().numpy()
    )

def across_validation(mode, seed=42):
    return np.concatenate([
        predict_with_mask(
            pilot, snapshots[t], mode,
            seed=seed + 1000 * t
        )
        for t in VAL_TIMES
    ])

p_tags = across_validation("tags")
p_full = across_validation("full")

random_predictions = [
    across_validation("random", seed=s)
    for s in [42, 43, 44]
]

# ------------------------------------------------------------
# 4. Comparable validation metrics
# ------------------------------------------------------------

def metrics(name, prediction):
    prediction = np.clip(
        prediction, 1e-7, 1 - 1e-7
    )

    return {
        "method": name,
        "failure_AP": average_precision_score(
            y_val, prediction
        ),
        "failure_ROC_AUC": roc_auc_score(
            y_val, prediction
        ),
        "BCE": log_loss(
            y_val, prediction,
            labels=[0, 1]
        )
    }

rows = [
    metrics("Train constant prior", constant_prior),
    metrics("Historical edge failure rate",
            historical_failure),
    metrics("Logistic regression", linear_failure),
    metrics("TAGS - full graph", p_full),
    metrics("TAGS - learned Top-K", p_tags)
]

for i, pred in enumerate(random_predictions):
    rows.append(
        metrics(f"TAGS - random Top-K seed {42+i}", pred)
    )

results = pd.DataFrame(rows)

# ------------------------------------------------------------
# 5. Save diagnostic report
# ------------------------------------------------------------

path = OUT / "e2_5_diagnostic_comparison.csv"
results.to_csv(path, index=False)

print("\nVALIDATION DIAGNOSTIC")
print(
    results.round(6).to_string(index=False)
)

print("\nREFERENCE:")
print("Random-ranking expected failure AP:",
      round(failure_prevalence, 6))
print("Random-ranking expected ROC-AUC: 0.5")

print("\nIMPORTANT:")
print("This is diagnostic, not final model comparison.")
print("The full graph and random masks reuse")
print("a checkpoint trained with soft TAGS masks.")
print("Therefore these are NOT fairly retrained baselines.")
print("No model architecture was changed.")
print("No test labels were accessed.")

print("\nE2.5 COMPLETED")
print("Saved:", path)


In [ ]:

# ============================================================
# PHASE II — E2.6
# TAGS-V2: DIRECT TRUST EVIDENCE IN PREDICTION HEAD
# ============================================================
# Architecture diagnostic, NOT final TAGS.
# Train: t=5,6,7
# Validation: t=8,9
# Test: NEVER READ
# Simulator hidden state: NEVER READ
# ============================================================

from pathlib import Path
import json
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    log_loss
)

REQUIRED = [
    "TAGS", "snapshots", "U", "V",
    "M", "K", "DEVICE"
]
missing = [x for x in REQUIRED if x not in globals()]
assert not missing, f"Missing E2.3/E2.4 variables: {missing}"

ROOT = Path("/kaggle/working/phase2_adaptive_siot")
OUT = ROOT / "E2_temporal_trust" / "model"
OUT.mkdir(parents=True, exist_ok=True)

TRAIN_TIMES = [5, 6, 7]
VAL_TIMES = [8, 9]
EPOCHS = 8
LR = 0.001
KEEP_RATIO = 0.50
BUDGET_WEIGHT = 0.02
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

assert M == 146117
assert K == 73058
assert all(t in snapshots for t in TRAIN_TIMES + VAL_TIMES)

# ------------------------------------------------------------
# 1. Corrected TAGS prediction head
# ------------------------------------------------------------

class TAGSv2(TAGS):
    def __init__(
        self,
        node_dim,
        edge_dim,
        controller_dim,
        hidden=32
    ):
        super().__init__(
            node_dim=node_dim,
            edge_dim=edge_dim,
            controller_dim=controller_dim,
            hidden=hidden
        )

        # GNN pair representation + direct historical
        # edge features from strictly earlier times.
        self.classifier = nn.Sequential(
            nn.Linear(hidden * 3 + edge_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def edge_scores(
        self, x_node, x_edge,
        x_controller, u, v
    ):
        full_weight = torch.ones(
            u.numel(), device=x_node.device
        )

        h = self.encoder(
            x_node, u, v, full_weight
        )

        pair = torch.cat([
            h[u] + h[v],
            torch.abs(h[u] - h[v]),
            x_edge
        ], dim=1)

        edge_score = self.scorer(pair).squeeze(-1)

        priority = self.controller(
            x_controller
        ).squeeze(-1)

        combined = edge_score + 0.5 * (
            priority[u] + priority[v]
        )

        return combined

    def predict_with_mask(
        self,
        x_node,
        x_edge,
        u,
        v,
        prediction_edge_ids,
        mask
    ):
        z = self.encoder(
            x_node, u, v, mask
        )

        pu = u[prediction_edge_ids]
        pv = v[prediction_edge_ids]

        pair_rep = torch.cat([
            z[pu] + z[pv],
            torch.abs(z[pu] - z[pv]),
            z[pu] * z[pv],
            x_edge[prediction_edge_ids]
        ], dim=1)

        return self.classifier(
            pair_rep
        ).squeeze(-1)

    def forward(
        self,
        x_node,
        x_edge,
        x_controller,
        u,
        v,
        prediction_edge_ids
    ):
        scores = self.edge_scores(
            x_node, x_edge,
            x_controller, u, v
        )

        mask = self.mask_layer(
            scores, training=self.training
        )

        logits = self.predict_with_mask(
            x_node, x_edge,
            u, v, prediction_edge_ids, mask
        )

        return {
            "prediction_logits": logits,
            "edge_logits": scores,
            "mask": mask
        }


# ------------------------------------------------------------
# 2. Independent model initialization
# ------------------------------------------------------------

sample = snapshots[5]

model_v2 = TAGSv2(
    node_dim=sample["node"].shape[1],
    edge_dim=sample["edge"].shape[1],
    controller_dim=sample["controller"].shape[1],
    hidden=32
).to(DEVICE)

optimizer = torch.optim.Adam(
    model_v2.parameters(),
    lr=LR,
    weight_decay=1e-4
)

# ------------------------------------------------------------
# 3. Hard Top-K validation
# ------------------------------------------------------------

@torch.no_grad()
def validation_v2(model):
    model.eval()

    truth = []
    probabilities = []
    selected = {}

    for t in VAL_TIMES:
        batch = snapshots[t]

        scores = model.edge_scores(
            batch["node"],
            batch["edge"],
            batch["controller"],
            U, V
        )

        chosen = torch.topk(
            scores, k=K, sorted=False
        ).indices

        hard_mask = torch.zeros(
            M, device=DEVICE
        )
        hard_mask[chosen] = 1.0

        assert int(hard_mask.sum().item()) == K

        logits = model.predict_with_mask(
            batch["node"],
            batch["edge"],
            U, V,
            batch["ids"],
            hard_mask
        )

        p_failure = torch.sigmoid(
            -logits
        ).cpu().numpy()

        y_failure = (
            1 - batch["y_numpy"]
        ).astype(np.int8)

        truth.append(y_failure)
        probabilities.append(p_failure)

        selected[t] = (
            chosen.cpu().numpy()
        )

    y = np.concatenate(truth)
    p = np.concatenate(probabilities)
    p = np.clip(p, 1e-7, 1 - 1e-7)

    change = len(
        np.setxor1d(selected[8], selected[9])
    ) / (2 * K)

    return {
        "failure_AP": float(
            average_precision_score(y, p)
        ),
        "failure_ROC_AUC": float(
            roc_auc_score(y, p)
        ),
        "BCE": float(
            log_loss(y, p, labels=[0, 1])
        ),
        "selection_change_fraction": float(change)
    }


# ------------------------------------------------------------
# 4. Train only on permitted snapshots
# ------------------------------------------------------------

records = []
best_ap = -np.inf
best_epoch = None

checkpoint_path = (
    OUT / "tags_v2_diagnostic_best.pt"
)

start = time.perf_counter()

for epoch in range(1, EPOCHS + 1):

    losses = []
    masks = []

    for t in TRAIN_TIMES:
        batch = snapshots[t]

        model_v2.train()
        optimizer.zero_grad(set_to_none=True)

        output = model_v2(
            batch["node"],
            batch["edge"],
            batch["controller"],
            U, V,
            batch["ids"]
        )

        task_loss = F.binary_cross_entropy_with_logits(
            output["prediction_logits"],
            batch["y"]
        )

        budget_loss = (
            output["mask"].mean()
            - KEEP_RATIO
        ).square()

        loss = task_loss + BUDGET_WEIGHT * budget_loss

        assert torch.isfinite(loss)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model_v2.parameters(),
            max_norm=5.0
        )

        optimizer.step()

        losses.append(float(task_loss.item()))
        masks.append(
            float(output["mask"].mean().item())
        )

    val = validation_v2(model_v2)

    row = {
        "epoch": epoch,
        "train_BCE": float(np.mean(losses)),
        "soft_mask_mean": float(np.mean(masks)),
        **{
            f"val_{key}": value
            for key, value in val.items()
        }
    }
    records.append(row)

    if val["failure_AP"] > best_ap:
        best_ap = val["failure_AP"]
        best_epoch = epoch

        torch.save({
            "model_state_dict": model_v2.state_dict(),
            "epoch": epoch,
            "validation_failure_AP": best_ap,
            "method": "TAGSv2-diagnostic",
            "seed": SEED,
            "train_times": TRAIN_TIMES,
            "validation_times": VAL_TIMES,
            "retained_edges": K
        }, checkpoint_path)

    print(
        f"Epoch {epoch:02d} | "
        f"train BCE={row['train_BCE']:.5f} | "
        f"val failure AP={val['failure_AP']:.5f} | "
        f"val ROC={val['failure_ROC_AUC']:.5f} | "
        f"val BCE={val['BCE']:.5f} | "
        f"mask={row['soft_mask_mean']:.4f} | "
        f"change={val['selection_change_fraction']:.4f}",
        flush=True
    )

# ------------------------------------------------------------
# 5. Save results
# ------------------------------------------------------------

elapsed = time.perf_counter() - start

history = pd.DataFrame(records)

history_path = OUT / "e2_6_tags_v2_history.csv"
history.to_csv(history_path, index=False)

report = {
    "experiment": "E2.6",
    "method": "TAGSv2-diagnostic",
    "change_from_v1": (
        "Direct causal edge evidence added "
        "to prediction head"
    ),
    "train_times": TRAIN_TIMES,
    "validation_times": VAL_TIMES,
    "test_accessed": False,
    "best_epoch": int(best_epoch),
    "best_validation_failure_AP": float(best_ap),
    "runtime_seconds": elapsed,
    "retained_edges": K,
    "total_edges": M,
    "pilot_only": True,
    "scientific_note": (
        "Better predictive performance does not "
        "prove GNN or sparsification utility."
    )
}

report_path = OUT / "e2_6_tags_v2_report.json"

with open(
    report_path, "w", encoding="utf-8"
) as f:
    json.dump(report, f, indent=2)

print("\n" + "=" * 72)
print("E2.6 COMPLETED — TAGS-V2 DIAGNOSTIC")
print("=" * 72)

print("Best epoch:", best_epoch)
print("Best validation failure AP:", round(best_ap, 6))
print("Logistic regression reference AP: 0.404299")
print("Original TAGS reference AP: 0.315786")
print("Elapsed seconds:", round(elapsed, 2))

print("\nCheckpoint:", checkpoint_path)
print("History:", history_path)
print("Report:", report_path)

print("\nNo test data accessed.")
print("No claim of GNN or sparsification benefit.")
print("NEXT: Isolate graph contribution and controller utility.")


In [ ]:

# ============================================================
# PHASE II — E2.7
# NEURAL TRUST BASELINE SANITY CHECK
# ============================================================
# Question:
# Can an ordinary MLP learn the signal already captured
# by Logistic Regression?
#
# No GNN. No sparsification. No synthetic hidden state.
# Train: t=5,6,7 | Validation: t=8,9
# Test: NEVER ACCESSED
# ============================================================

from pathlib import Path
import json
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    log_loss
)

ROOT = Path("/kaggle/working/phase2_adaptive_siot")
OUT = ROOT / "E2_temporal_trust" / "model"
OUT.mkdir(parents=True, exist_ok=True)

TRAIN_TIMES = [5, 6, 7]
VAL_TIMES = [8, 9]
SEEDS = [42, 43, 44, 45, 46]

MAX_EPOCHS = 80
PATIENCE = 16
BATCH_SIZE = 4096
LR = 1e-3

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert "snapshots" in globals(), (
    "Run E2.4 to prepare the temporal snapshots."
)

# ------------------------------------------------------------
# 1. Exactly the same features and labels as E2.5
# ------------------------------------------------------------

def extract_features(batch):
    return (
        batch["edge"][batch["ids"]]
        .detach().cpu().numpy()
        .astype(np.float32)
    )

def extract_failure_labels(batch):
    return (
        1 - batch["y_numpy"].astype(np.int8)
    )

X_train = np.concatenate([
    extract_features(snapshots[t])
    for t in TRAIN_TIMES
])

y_train = np.concatenate([
    extract_failure_labels(snapshots[t])
    for t in TRAIN_TIMES
])

X_val = np.concatenate([
    extract_features(snapshots[t])
    for t in VAL_TIMES
])

y_val = np.concatenate([
    extract_failure_labels(snapshots[t])
    for t in VAL_TIMES
])

assert X_train.shape[1] == 17
assert X_val.shape[1] == 17
assert len(X_train) == len(y_train)
assert len(X_val) == len(y_val)

# Fit scaling on training observations only.
scaler = StandardScaler()

X_train = scaler.fit_transform(
    X_train
).astype(np.float32)

X_val = scaler.transform(
    X_val
).astype(np.float32)

assert np.isfinite(X_train).all()
assert np.isfinite(X_val).all()

print("=" * 70)
print("E2.7 — NEURAL TRUST BASELINE SANITY CHECK")
print("=" * 70)
print("Device:", DEVICE)
print("Training examples:", len(y_train))
print("Validation examples:", len(y_val))
print("Features:", X_train.shape[1])
print("Training failure prevalence:", y_train.mean())
print("Validation failure prevalence:", y_val.mean())

# ------------------------------------------------------------
# 2. Common evaluation
# ------------------------------------------------------------

def evaluate(y, p):
    p = np.clip(p, 1e-7, 1 - 1e-7)

    return {
        "failure_AP": float(
            average_precision_score(y, p)
        ),
        "ROC_AUC": float(
            roc_auc_score(y, p)
        ),
        "BCE": float(
            log_loss(y, p, labels=[0, 1])
        )
    }

# ------------------------------------------------------------
# 3. Refit Logistic Regression reference
# ------------------------------------------------------------

lr_model = LogisticRegression(
    solver="lbfgs",
    max_iter=500,
    random_state=42
)

lr_model.fit(X_train, y_train)

lr_prob = lr_model.predict_proba(
    X_val
)[:, 1]

lr_metrics = evaluate(y_val, lr_prob)

print("\nLOGISTIC REGRESSION REFERENCE")
print(lr_metrics)

# ------------------------------------------------------------
# 4. Small neural trust predictor
# ------------------------------------------------------------

class TrustMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.network(x).squeeze(-1)

train_x = torch.from_numpy(
    X_train
).to(DEVICE)

train_y = torch.from_numpy(
    y_train.astype(np.float32)
).to(DEVICE)

val_x = torch.from_numpy(
    X_val
).to(DEVICE)

criterion = nn.BCEWithLogitsLoss()

results = []

# ------------------------------------------------------------
# 5. Five independent initializations
# ------------------------------------------------------------

for seed in SEEDS:

    torch.manual_seed(seed)
    np.random.seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    model = TrustMLP(
        input_dim=X_train.shape[1]
    ).to(DEVICE)

    # Initial intercept matching train failure prevalence.
    with torch.no_grad():
        prevalence = float(y_train.mean())
        intercept = np.log(
            prevalence / (1 - prevalence)
        )
        model.network[-1].bias.fill_(
            intercept
        )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=1e-4
    )

    best_ap = -np.inf
    best_epoch = 0
    best_metrics = None
    best_state = None
    patience_counter = 0

    for epoch in range(1, MAX_EPOCHS + 1):

        model.train()

        # Fixed seed-specific minibatch order.
        rng = np.random.default_rng(
            seed * 10000 + epoch
        )
        indices = rng.permutation(
            len(y_train)
        )

        for start in range(
            0, len(indices), BATCH_SIZE
        ):
            batch_idx = torch.as_tensor(
                indices[start:start + BATCH_SIZE],
                dtype=torch.long,
                device=DEVICE
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            logits = model(
                train_x[batch_idx]
            )

            loss = criterion(
                logits,
                train_y[batch_idx]
            )

            loss.backward()
            optimizer.step()

        model.eval()

        with torch.no_grad():
            val_prob = torch.sigmoid(
                model(val_x)
            ).cpu().numpy()

        metrics = evaluate(
            y_val, val_prob
        )

        # Validation AP selects the diagnostic checkpoint.
        if metrics["failure_AP"] > best_ap + 1e-5:
            best_ap = metrics["failure_AP"]
            best_epoch = epoch
            best_metrics = metrics
            best_state = copy.deepcopy(
                model.state_dict()
            )
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= PATIENCE:
            break

    assert best_state is not None

    record = {
        "method": "Trust-MLP",
        "seed": seed,
        "best_epoch": best_epoch,
        **best_metrics
    }

    results.append(record)

    print(
        f"Seed {seed} | "
        f"epoch={best_epoch:3d} | "
        f"AP={best_metrics['failure_AP']:.5f} | "
        f"AUC={best_metrics['ROC_AUC']:.5f} | "
        f"BCE={best_metrics['BCE']:.5f}",
        flush=True
    )

# ------------------------------------------------------------
# 6. Results and comparison
# ------------------------------------------------------------

result_df = pd.DataFrame(results)

ap_mean = result_df["failure_AP"].mean()
ap_std = result_df["failure_AP"].std(ddof=1)

auc_mean = result_df["ROC_AUC"].mean()
auc_std = result_df["ROC_AUC"].std(ddof=1)

bce_mean = result_df["BCE"].mean()
bce_std = result_df["BCE"].std(ddof=1)

report = {
    "experiment": "E2.7",
    "purpose": "Neural trust-prediction sanity check",
    "train_times": TRAIN_TIMES,
    "validation_times": VAL_TIMES,
    "test_accessed": False,
    "feature_count": 17,
    "standardization_train_only": True,
    "lr_reference": lr_metrics,
    "mlp_seeds": SEEDS,
    "mlp_mean": {
        "failure_AP": float(ap_mean),
        "ROC_AUC": float(auc_mean),
        "BCE": float(bce_mean)
    },
    "mlp_std": {
        "failure_AP": float(ap_std),
        "ROC_AUC": float(auc_std),
        "BCE": float(bce_std)
    },
    "mlp_minus_lr_AP": float(
        ap_mean - lr_metrics["failure_AP"]
    ),
    "architecture": "17-32-16-1 MLP",
    "GNN_used": False,
    "sparsification_used": False
}

result_path = OUT / "e2_7_trust_mlp_seeds.csv"
report_path = OUT / "e2_7_trust_mlp_report.json"

result_df.to_csv(
    result_path, index=False
)

with open(
    report_path, "w", encoding="utf-8"
) as f:
    json.dump(report, f, indent=2)

print("\n" + "=" * 70)
print("E2.7 COMPLETED — TRUST-MLP DIAGNOSTIC")
print("=" * 70)

print(
    f"LR failure AP: {lr_metrics['failure_AP']:.6f}"
)

print(
    f"MLP failure AP: {ap_mean:.6f} ± {ap_std:.6f}"
)

print(
    f"MLP ROC-AUC: {auc_mean:.6f} ± {auc_std:.6f}"
)

print(
    f"MLP BCE: {bce_mean:.6f} ± {bce_std:.6f}"
)

print(
    f"MLP minus LR AP: "
    f"{ap_mean - lr_metrics['failure_AP']:+.6f}"
)

print("\nAll five seeds:")
print(
    result_df.round(6).to_string(index=False)
)

print("\nSaved:", result_path)
print("Report:", report_path)
print("No GNN or sparsification used.")
print("No test data accessed.")
print("\nNEXT: Determine whether GNN adds utility.")


In [ ]:

# ============================================================
# PHASE II — E2.8
# FROZEN TRUST-MLP + RESIDUAL GRAPH SAGE
# ============================================================
# Purpose: isolate added predictive value of GNN.
# No sparsification, no controller, no Hard-Concrete.
# Train: t=5,6,7
# Validation: t=8,9
# Test and hidden simulator state: NEVER ACCESSED.
# ============================================================

from pathlib import Path
import copy
import json
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    log_loss
)

assert "snapshots" in globals()
assert "SAGEEncoder" in globals()
assert "U" in globals() and "V" in globals()
assert "TrustMLP" in globals()

SEED = 42
TRAIN_TIMES = [5, 6, 7]
VAL_TIMES = [8, 9]

BASE_EPOCHS = 80
BASE_PATIENCE = 16
GNN_EPOCHS = 40
GNN_PATIENCE = 12

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

ROOT = Path("/kaggle/working/phase2_adaptive_siot")
OUT = ROOT / "E2_temporal_trust" / "model"
OUT.mkdir(parents=True, exist_ok=True)

torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("=" * 70)
print("E2.8 — FROZEN TRUST-MLP + RESIDUAL GRAPH SAGE")
print("=" * 70)
print("Device:", DEVICE)
print("Train:", TRAIN_TIMES)
print("Validation:", VAL_TIMES)

# ------------------------------------------------------------
# 1. Prepare the EXACT trust-edge features
# ------------------------------------------------------------

def edge_array(batch):
    return (
        batch["edge"][batch["ids"]]
        .detach().cpu().numpy()
        .astype(np.float32)
    )

def failure_array(batch):
    return (
        1 - batch["y_numpy"]
    ).astype(np.float32)

raw_train = np.concatenate(
    [edge_array(snapshots[t]) for t in TRAIN_TIMES]
)
raw_val = np.concatenate(
    [edge_array(snapshots[t]) for t in VAL_TIMES]
)

y_train = np.concatenate(
    [failure_array(snapshots[t]) for t in TRAIN_TIMES]
)
y_val = np.concatenate(
    [failure_array(snapshots[t]) for t in VAL_TIMES]
)

scaler = StandardScaler()
x_train = scaler.fit_transform(raw_train).astype(np.float32)
x_val = scaler.transform(raw_val).astype(np.float32)

assert x_train.shape[1] == 17
assert len(y_train) == len(x_train)
assert len(y_val) == len(x_val)
assert np.isfinite(x_train).all()
assert np.isfinite(x_val).all()

tx = torch.from_numpy(x_train).to(DEVICE)
ty = torch.from_numpy(y_train).to(DEVICE)
vx = torch.from_numpy(x_val).to(DEVICE)

def metrics(y, p):
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return {
        "AP": float(average_precision_score(y, p)),
        "AUC": float(roc_auc_score(y, p)),
        "BCE": float(log_loss(y, p, labels=[0, 1]))
    }

# ------------------------------------------------------------
# 2. Retrain and freeze the successful Trust-MLP
# ------------------------------------------------------------

base = TrustMLP(17).to(DEVICE)

with torch.no_grad():
    prevalence = float(y_train.mean())
    base.network[-1].bias.fill_(
        np.log(prevalence / (1 - prevalence))
    )

base_optimizer = torch.optim.AdamW(
    base.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

best_base_ap = -np.inf
best_base_epoch = None
best_base_state = None
wait = 0

for epoch in range(1, BASE_EPOCHS + 1):
    base.train()

    rng = np.random.default_rng(
        SEED * 10000 + epoch
    )
    order = rng.permutation(len(y_train))

    for start in range(0, len(order), 4096):
        ids = torch.as_tensor(
            order[start:start + 4096],
            dtype=torch.long,
            device=DEVICE
        )

        base_optimizer.zero_grad(set_to_none=True)

        logits = base(tx[ids])
        loss = F.binary_cross_entropy_with_logits(
            logits, ty[ids]
        )
        loss.backward()
        base_optimizer.step()

    base.eval()
    with torch.no_grad():
        p = torch.sigmoid(base(vx)).cpu().numpy()

    ap = metrics(y_val, p)["AP"]

    if ap > best_base_ap + 1e-5:
        best_base_ap = ap
        best_base_epoch = epoch
        best_base_state = copy.deepcopy(
            base.state_dict()
        )
        wait = 0
    else:
        wait += 1

    if wait >= BASE_PATIENCE:
        break

base.load_state_dict(best_base_state)
base.eval()

for p in base.parameters():
    p.requires_grad_(False)

with torch.no_grad():
    base_val_p = torch.sigmoid(base(vx)).cpu().numpy()

base_metrics = metrics(y_val, base_val_p)

print("\nFROZEN TRUST-MLP")
print("Best epoch:", best_base_epoch)
print("Validation:", base_metrics)

# ------------------------------------------------------------
# 3. Prepare per-time inputs
# ------------------------------------------------------------
# Base logits are cached from the frozen model.
# Node features come from causal snapshots only.
# Device ownership and relation labels are NOT
# converted to hidden trust ratings.

batches = {}

for t in TRAIN_TIMES + VAL_TIMES:
    batch = snapshots[t]

    raw = edge_array(batch)
    scaled = scaler.transform(raw).astype(np.float32)

    with torch.no_grad():
        base_logits = base(
            torch.from_numpy(scaled).to(DEVICE)
        ).detach()

    batches[t] = {
        "node": batch["node"],
        "ids": batch["ids"],
        "base_logits": base_logits,
        "y_failure": torch.from_numpy(
            failure_array(batch)
        ).to(DEVICE)
    }

# ------------------------------------------------------------
# 4. Residual GraphSAGE
# ------------------------------------------------------------

class ResidualGraphSAGE(nn.Module):
    def __init__(self, node_dim, hidden=32):
        super().__init__()

        self.encoder = SAGEEncoder(
            node_dim, hidden
        )

        self.head = nn.Sequential(
            nn.Linear(hidden * 3, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

        # Start exactly from the frozen base prediction.
        nn.init.zeros_(self.head[-1].weight)
        nn.init.zeros_(self.head[-1].bias)

    def forward(self, x, edge_ids):
        weights = torch.ones(
            U.numel(), device=x.device
        )

        z = self.encoder(
            x, U, V, weights
        )

        a = U[edge_ids]
        b = V[edge_ids]

        pair = torch.cat([
            z[a] + z[b],
            torch.abs(z[a] - z[b]),
            z[a] * z[b]
        ], dim=1)

        return self.head(pair).squeeze(-1)

gnn = ResidualGraphSAGE(
    node_dim=batches[5]["node"].shape[1],
    hidden=32
).to(DEVICE)

optimizer = torch.optim.AdamW(
    gnn.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

# ------------------------------------------------------------
# 5. Validation of the frozen base + GNN correction
# ------------------------------------------------------------

@torch.no_grad()
def evaluate_residual():
    gnn.eval()

    y_all = []
    p_all = []
    delta_magnitudes = []

    for t in VAL_TIMES:
        b = batches[t]

        correction = gnn(
            b["node"], b["ids"]
        )

        logits = b["base_logits"] + correction

        probability = torch.sigmoid(
            logits
        ).cpu().numpy()

        y_all.append(
            b["y_failure"].cpu().numpy()
        )
        p_all.append(probability)

        delta_magnitudes.append(
            float(correction.abs().mean().item())
        )

    y = np.concatenate(y_all)
    p = np.concatenate(p_all)

    result = metrics(y, p)
    result["mean_abs_correction"] = float(
        np.mean(delta_magnitudes)
    )

    return result

# Initial residual is exactly zero.
initial = evaluate_residual()

assert abs(initial["AP"] - base_metrics["AP"]) < 1e-7
assert abs(initial["AUC"] - base_metrics["AUC"]) < 1e-7

print("\nZERO-INITIALIZATION CHECK: PASSED")

# ------------------------------------------------------------
# 6. Train the GNN residual only
# ------------------------------------------------------------

history = []
best_gnn_ap = base_metrics["AP"]
best_gnn_epoch = 0
best_gnn_state = copy.deepcopy(gnn.state_dict())
wait = 0

start_time = time.perf_counter()

for epoch in range(1, GNN_EPOCHS + 1):
    gnn.train()
    train_losses = []

    for t in TRAIN_TIMES:
        b = batches[t]

        optimizer.zero_grad(set_to_none=True)

        delta = gnn(
            b["node"], b["ids"]
        )

        logits = b["base_logits"] + delta

        task = F.binary_cross_entropy_with_logits(
            logits, b["y_failure"]
        )

        # Small correction penalty for stability.
        penalty = 1e-3 * delta.square().mean()

        loss = task + penalty
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            gnn.parameters(), 5.0
        )
        optimizer.step()

        train_losses.append(float(task.item()))

    val = evaluate_residual()

    record = {
        "epoch": epoch,
        "train_BCE": float(np.mean(train_losses)),
        "val_AP": val["AP"],
        "val_AUC": val["AUC"],
        "val_BCE": val["BCE"],
        "mean_abs_correction": val[
            "mean_abs_correction"
        ]
    }

    history.append(record)

    if val["AP"] > best_gnn_ap + 1e-5:
        best_gnn_ap = val["AP"]
        best_gnn_epoch = epoch
        best_gnn_state = copy.deepcopy(
            gnn.state_dict()
        )
        wait = 0
    else:
        wait += 1

    if epoch == 1 or epoch % 5 == 0:
        print(
            f"Epoch {epoch:02d} | "
            f"train BCE={record['train_BCE']:.5f} | "
            f"val AP={val['AP']:.5f} | "
            f"val AUC={val['AUC']:.5f} | "
            f"val BCE={val['BCE']:.5f} | "
            f"|delta|={val['mean_abs_correction']:.5f}",
            flush=True
        )

    if wait >= GNN_PATIENCE:
        break

gnn.load_state_dict(best_gnn_state)
final_metrics = evaluate_residual()

# ------------------------------------------------------------
# 7. Save the diagnostic
# ------------------------------------------------------------

elapsed = time.perf_counter() - start_time

report = {
    "experiment": "E2.8",
    "purpose": "Full-graph GNN incremental utility",
    "seed": SEED,
    "train_times": TRAIN_TIMES,
    "validation_times": VAL_TIMES,
    "test_accessed": False,
    "sparsification_used": False,
    "base_best_epoch": best_base_epoch,
    "base": base_metrics,
    "best_gnn_epoch": best_gnn_epoch,
    "base_plus_gnn": final_metrics,
    "delta_AP": (
        final_metrics["AP"] - base_metrics["AP"]
    ),
    "delta_AUC": (
        final_metrics["AUC"] - base_metrics["AUC"]
    ),
    "delta_BCE": (
        final_metrics["BCE"] - base_metrics["BCE"]
    ),
    "gnn_training_seconds": elapsed,
    "interpretation": (
        "Exploratory single-seed validation comparison; "
        "GNN benefit is not established by this alone."
    )
}

pd.DataFrame(history).to_csv(
    OUT / "e2_8_residual_gnn_history.csv",
    index=False
)

torch.save({
    "base_state_dict": base.state_dict(),
    "gnn_state_dict": gnn.state_dict(),
    "seed": SEED,
    "best_gnn_epoch": best_gnn_epoch
}, OUT / "e2_8_residual_gnn_checkpoint.pt")

with open(
    OUT / "e2_8_residual_gnn_report.json",
    "w", encoding="utf-8"
) as f:
    json.dump(report, f, indent=2)

print("\n" + "=" * 70)
print("E2.8 COMPLETED — GNN INCREMENTAL UTILITY")
print("=" * 70)

print("Frozen Trust-MLP:", base_metrics)
print("Trust-MLP + GraphSAGE:", final_metrics)
print("Best GNN epoch:", best_gnn_epoch)

print(
    f"Delta AP: {report['delta_AP']:+.6f}"
)
print(
    f"Delta AUC: {report['delta_AUC']:+.6f}"
)
print(
    f"Delta BCE: {report['delta_BCE']:+.6f}"
)

print("No sparsification used.")
print("No test data accessed.")
print("No claim of GNN superiority.")



In [ ]:

# ============================================================
# PHASE II — E2.9
# TRUST FEATURE-GROUP ABLATION
# ============================================================
# Goal: measure unique contribution of:
#   A. Direct edge-history evidence
#   B. Aggregated endpoint evidence
#   C. SIoT relationship categories
#
# Train: t=5,6,7
# Validation: t=8,9
# Test: NEVER ACCESSED
# No model changes or simulator changes.
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    log_loss
)

assert "snapshots" in globals(), (
    "Run E2.4 before this diagnostic."
)

ROOT = Path("/kaggle/working/phase2_adaptive_siot")
OUT = ROOT / "E2_temporal_trust" / "model"
OUT.mkdir(parents=True, exist_ok=True)

TRAIN_TIMES = [5, 6, 7]
VAL_TIMES = [8, 9]

def get_x(t):
    b = snapshots[t]
    return (
        b["edge"][b["ids"]]
        .detach().cpu().numpy()
        .astype(np.float32)
    )

def get_y(t):
    return (
        1 - snapshots[t]["y_numpy"]
    ).astype(np.int8)

X_train = np.concatenate(
    [get_x(t) for t in TRAIN_TIMES]
)

X_val = np.concatenate(
    [get_x(t) for t in VAL_TIMES]
)

y_train = np.concatenate(
    [get_y(t) for t in TRAIN_TIMES]
)

y_val = np.concatenate(
    [get_y(t) for t in VAL_TIMES]
)

assert X_train.shape[1] == 17
assert X_val.shape[1] == 17

# E2.4 feature layout:
# 0:6   direct edge-history / uncertainty / age
# 7:10  aggregated endpoint historical information
# 11    recent minus long-term edge success
# 12:16 five SIoT relation-type indicators

EDGE = [0, 1, 2, 3, 4, 5, 6, 11]
NODE = [7, 8, 9, 10]
REL = [12, 13, 14, 15, 16]

GROUPS = {
    "Edge history only": EDGE,
    "Endpoint history only": NODE,
    "Relation types only": REL,
    "Edge + endpoint": EDGE + NODE,
    "Edge + relations": EDGE + REL,
    "Endpoint + relations": NODE + REL,
    "All features": EDGE + NODE + REL,
}

def evaluate_features(columns):
    scaler = StandardScaler()

    xt = scaler.fit_transform(
        X_train[:, columns]
    )
    xv = scaler.transform(
        X_val[:, columns]
    )

    model = LogisticRegression(
        solver="lbfgs",
        max_iter=500,
        random_state=42
    )

    model.fit(xt, y_train)

    p = np.clip(
        model.predict_proba(xv)[:, 1],
        1e-7, 1 - 1e-7
    )

    return {
        "failure_AP": float(
            average_precision_score(y_val, p)
        ),
        "ROC_AUC": float(
            roc_auc_score(y_val, p)
        ),
        "BCE": float(
            log_loss(y_val, p, labels=[0, 1])
        )
    }

rows = []

print("=" * 72)
print("E2.9 — TRUST FEATURE-GROUP ABLATION")
print("=" * 72)
print("Train:", TRAIN_TIMES)
print("Validation:", VAL_TIMES)
print("Features:", X_train.shape[1])

for name, cols in GROUPS.items():
    result = evaluate_features(cols)

    rows.append({
        "feature_group": name,
        "feature_count": len(cols),
        **result
    })

    print(
        f"{name:25s} | "
        f"AP={result['failure_AP']:.6f} | "
        f"AUC={result['ROC_AUC']:.6f} | "
        f"BCE={result['BCE']:.6f}"
    )

results = pd.DataFrame(rows)

full_ap = float(
    results.loc[
        results["feature_group"] == "All features",
        "failure_AP"
    ].iloc[0]
)

without_node_ap = float(
    results.loc[
        results["feature_group"] == "Edge + relations",
        "failure_AP"
    ].iloc[0]
)

without_edge_ap = float(
    results.loc[
        results["feature_group"] == "Endpoint + relations",
        "failure_AP"
    ].iloc[0]
)

report = {
    "experiment": "E2.9",
    "train_times": TRAIN_TIMES,
    "validation_times": VAL_TIMES,
    "test_accessed": False,
    "model": "LogisticRegression",
    "results": rows,
    "delta_AP_adding_endpoint_history": (
        full_ap - without_node_ap
    ),
    "delta_AP_adding_edge_history": (
        full_ap - without_edge_ap
    ),
    "important_limitation": (
        "This feature ablation does not independently "
        "establish GNN utility or causal feature importance."
    )
}

csv_path = OUT / "e2_9_feature_ablation.csv"
json_path = OUT / "e2_9_feature_ablation.json"

results.to_csv(csv_path, index=False)

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print("\n" + "=" * 72)
print("E2.9 COMPLETED")
print("=" * 72)

print(
    "AP gain from adding endpoint history:",
    f"{report['delta_AP_adding_endpoint_history']:+.6f}"
)

print(
    "AP gain from adding edge history:",
    f"{report['delta_AP_adding_edge_history']:+.6f}"
)

print("\nSaved:", csv_path)
print("Report:", json_path)
print("No GNN was trained.")
print("No test data accessed.")
print("No previous artifacts modified.")


In [ ]:

# ============================================================
# PHASE II — E2.10
# DOES GRAPH MESSAGE PASSING ADD INFORMATION?
# ============================================================
# Controlled comparison:
# A. Node-wise MLP (no graph propagation)
# B. GraphSAGE (full social graph)
#
# Same causal inputs, prediction head, splits,
# loss function, optimizer and selection criterion.
# No sparsification. No simulator modifications.
# No test or hidden-state access.
# ============================================================

from pathlib import Path
import json
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    log_loss
)

assert "snapshots" in globals()
assert "SAGEEncoder" in globals()
assert "U" in globals() and "V" in globals()

ROOT = Path("/kaggle/working/phase2_adaptive_siot")
OUT = ROOT / "E2_temporal_trust" / "model"
OUT.mkdir(parents=True, exist_ok=True)

TRAIN_TIMES = [5, 6, 7]
VAL_TIMES = [8, 9]
SEEDS = [42, 43]

MAX_EPOCHS = 45
PATIENCE = 12
DEVICE = U.device

# Direct edge evidence + relationship types.
# Exclude endpoint-summary columns 7,8,9,10.
EDGE_COLS = [0, 1, 2, 3, 4, 5, 6, 11,
             12, 13, 14, 15, 16]

# ------------------------------------------------------------
# 1. Fit scalers ONLY on training snapshots
# ------------------------------------------------------------

edge_train = np.concatenate([
    snapshots[t]["edge"][snapshots[t]["ids"]]
    .cpu().numpy()[:, EDGE_COLS]
    for t in TRAIN_TIMES
])

node_train = np.concatenate([
    snapshots[t]["node"].cpu().numpy()
    for t in TRAIN_TIMES
])

edge_scaler = StandardScaler().fit(edge_train)
node_scaler = StandardScaler().fit(node_train)

def prepare(t):
    b = snapshots[t]

    xe = edge_scaler.transform(
        b["edge"].cpu().numpy()[:, EDGE_COLS]
    ).astype(np.float32)

    xn = node_scaler.transform(
        b["node"].cpu().numpy()
    ).astype(np.float32)

    return {
        "node": torch.from_numpy(xn).to(DEVICE),
        "edge": torch.from_numpy(xe).to(DEVICE),
        "ids": b["ids"],
        "y": torch.from_numpy(
            (1 - b["y_numpy"]).astype(np.float32)
        ).to(DEVICE)
    }

data = {
    t: prepare(t)
    for t in TRAIN_TIMES + VAL_TIMES
}

assert all(
    torch.isfinite(data[t]["node"]).all()
    and torch.isfinite(data[t]["edge"]).all()
    for t in data
)

print("=" * 70)
print("E2.10 — MESSAGE-PASSING CONTRIBUTION")
print("=" * 70)
print("Device:", DEVICE)
print("Train:", TRAIN_TIMES)
print("Validation:", VAL_TIMES)
print("Seeds:", SEEDS)
print("Direct edge inputs:", len(EDGE_COLS))
print("Node inputs:", data[5]["node"].shape[1])

# ------------------------------------------------------------
# 2. Two models with the same prediction head
# ------------------------------------------------------------

class ControlledEdgePredictor(nn.Module):
    def __init__(self, node_dim, edge_dim,
                 use_graph, hidden=32):
        super().__init__()

        self.use_graph = use_graph

        if use_graph:
            self.encoder = SAGEEncoder(
                node_dim, hidden
            )
        else:
            self.encoder = nn.Sequential(
                nn.Linear(node_dim, hidden),
                nn.ReLU(),
                nn.Linear(hidden, hidden)
            )

        self.head = nn.Sequential(
            nn.Linear(hidden * 3 + edge_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, batch):
        if self.use_graph:
            weights = torch.ones(
                U.numel(), device=DEVICE
            )
            z = self.encoder(
                batch["node"], U, V, weights
            )
        else:
            z = self.encoder(batch["node"])

        ids = batch["ids"]
        a = U[ids]
        b = V[ids]

        pair = torch.cat([
            z[a] + z[b],
            torch.abs(z[a] - z[b]),
            z[a] * z[b],
            batch["edge"][ids]
        ], dim=1)

        return self.head(pair).squeeze(-1)

def validate(model):
    model.eval()
    ys, ps = [], []

    with torch.no_grad():
        for t in VAL_TIMES:
            b = data[t]
            logits = model(b)
            probability = (
                torch.sigmoid(logits)
                .cpu().numpy()
            )

            ys.append(b["y"].cpu().numpy())
            ps.append(probability)

    y = np.concatenate(ys)
    p = np.clip(
        np.concatenate(ps), 1e-7, 1 - 1e-7
    )

    return {
        "AP": float(
            average_precision_score(y, p)
        ),
        "AUC": float(
            roc_auc_score(y, p)
        ),
        "BCE": float(
            log_loss(y, p, labels=[0, 1])
        )
    }

# ------------------------------------------------------------
# 3. Train controlled models
# ------------------------------------------------------------

results = []

for seed in SEEDS:
    for use_graph in [False, True]:

        name = (
            "GraphSAGE" if use_graph else "Node-MLP"
        )

        torch.manual_seed(seed)
        np.random.seed(seed)

        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        model = ControlledEdgePredictor(
            node_dim=data[5]["node"].shape[1],
            edge_dim=data[5]["edge"].shape[1],
            use_graph=use_graph,
            hidden=32
        ).to(DEVICE)

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=1e-3,
            weight_decay=1e-4
        )

        best_ap = -np.inf
        best_epoch = 0
        best_metrics = None
        best_state = None
        wait = 0

        for epoch in range(1, MAX_EPOCHS + 1):
            model.train()

            # Same chronological training order.
            for t in TRAIN_TIMES:
                optimizer.zero_grad(
                    set_to_none=True
                )

                logits = model(data[t])

                loss = F.binary_cross_entropy_with_logits(
                    logits, data[t]["y"]
                )

                loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(), 5.0
                )

                optimizer.step()

            m = validate(model)

            if m["AP"] > best_ap + 1e-5:
                best_ap = m["AP"]
                best_epoch = epoch
                best_metrics = m
                best_state = copy.deepcopy(
                    model.state_dict()
                )
                wait = 0
            else:
                wait += 1

            if wait >= PATIENCE:
                break

        assert best_state is not None

        results.append({
            "seed": seed,
            "model": name,
            "best_epoch": best_epoch,
            **best_metrics
        })

        print(
            f"{name:10s} | seed={seed} | "
            f"epoch={best_epoch:02d} | "
            f"AP={best_metrics['AP']:.6f} | "
            f"AUC={best_metrics['AUC']:.6f} | "
            f"BCE={best_metrics['BCE']:.6f}",
            flush=True
        )

        checkpoint = (
            OUT / f"e2_10_{name.lower()}_seed{seed}.pt"
        )

        torch.save({
            "model_state_dict": best_state,
            "model_type": name,
            "seed": seed,
            "best_epoch": best_epoch,
            "edge_columns": EDGE_COLS,
            "train_times": TRAIN_TIMES,
            "validation_times": VAL_TIMES
        }, checkpoint)

# ------------------------------------------------------------
# 4. Paired comparison
# ------------------------------------------------------------

df = pd.DataFrame(results)

wide = df.pivot(
    index="seed",
    columns="model",
    values=["AP", "AUC", "BCE"]
)

delta = pd.DataFrame({
    "seed": wide.index,
    "delta_AP": (
        wide[("AP", "GraphSAGE")]
        - wide[("AP", "Node-MLP")]
    ).values,
    "delta_AUC": (
        wide[("AUC", "GraphSAGE")]
        - wide[("AUC", "Node-MLP")]
    ).values,
    "delta_BCE": (
        wide[("BCE", "GraphSAGE")]
        - wide[("BCE", "Node-MLP")]
    ).values
})

df.to_csv(
    OUT / "e2_10_model_comparison.csv",
    index=False
)

delta.to_csv(
    OUT / "e2_10_paired_deltas.csv",
    index=False
)

report = {
    "experiment": "E2.10",
    "purpose": "Message passing vs node-wise encoding",
    "train_times": TRAIN_TIMES,
    "validation_times": VAL_TIMES,
    "seeds": SEEDS,
    "edge_feature_columns": EDGE_COLS,
    "same_prediction_head": True,
    "same_global_graph_topology": True,
    "sparsification_used": False,
    "test_accessed": False,
    "mean_delta_AP": float(
        delta["delta_AP"].mean()
    ),
    "mean_delta_AUC": float(
        delta["delta_AUC"].mean()
    ),
    "mean_delta_BCE": float(
        delta["delta_BCE"].mean()
    ),
    "warning": (
        "Exploratory two-seed comparison; "
        "different encoder parameterizations; "
        "not a statistical significance test."
    )
}

with open(
    OUT / "e2_10_report.json",
    "w", encoding="utf-8"
) as f:
    json.dump(report, f, indent=2)

print("\n" + "=" * 70)
print("E2.10 COMPLETED")
print("=" * 70)

print("\nMODEL RESULTS")
print(df.round(6).to_string(index=False))

print("\nPAIRED DELTAS: GraphSAGE minus Node-MLP")
print(delta.round(6).to_string(index=False))

print("\nMean Delta AP:",
      round(report["mean_delta_AP"], 6))
print("Mean Delta AUC:",
      round(report["mean_delta_AUC"], 6))
print("Mean Delta BCE:",
      round(report["mean_delta_BCE"], 6))

print("\nNo sparsification or test access.")
print("NEXT: Finalize revised TAGS experimental design.")


In [ ]:

# ============================================================
# PHASE III — E3.0
# TAGS-V3 RESEARCH PROTOCOL FREEZE
# ============================================================
# Records decisions BEFORE generating new scenarios.
# Does not train or simulate anything.
# Does not modify E1/E2 artifacts.
# ============================================================

from pathlib import Path
import json

ROOT = Path("/kaggle/working/phase2_adaptive_siot")
OUT = ROOT / "E3_tags_v3"
OUT.mkdir(parents=True, exist_ok=True)

protocol_path = OUT / "e3_0_research_protocol_v1.json"

# Never silently overwrite a frozen protocol.
if protocol_path.exists():
    raise FileExistsError(
        f"Protocol already exists: {protocol_path}"
    )

protocol = {
    "experiment": "E3.0",
    "research_title": (
        "Trust-Aware Adaptive Graph Sparsification "
        "Using Graph Neural Networks for "
        "Social Internet of Things"
    ),
    "algorithm": "TAGS-v3",
    "status": "pre-registered design; not validated",

    "dataset": {
        "name": "Santander SIoT",
        "nodes": 16216,
        "unique_undirected_edges": 146117,
        "relation_types": [
            "OOR", "POR", "C_LOR", "SOR", "SOR2"
        ],
        "real_trust_labels": False
    },

    "existing_negative_control": {
        "name": "Scenario A",
        "source": "E2.0",
        "recreate": False,
        "change_existing_artifacts": False
    },

    "future_scenarios": {
        "B": {
            "name": "Owner-correlated reliability",
            "mechanism": "Shared latent owner effects",
            "primary_question": (
                "Does relational structure provide "
                "incremental utility?"
            )
        },
        "C": {
            "name": "Clustered temporal shock",
            "mechanism": (
                "Group-localized time-dependent "
                "reliability perturbation"
            ),
            "primary_question": (
                "Does adaptive sparsification respond "
                "usefully to changing trust?"
            )
        }
    },

    "method": {
        "encoder": "GraphSAGE",
        "edge_scorer": "GNN + causal trust features",
        "controller": (
            "Learnable temporal node-priority controller"
        ),
        "global_budget": True,
        "undirected_symmetric_decisions": True,
        "initial_retention": 0.50,
        "retention_sensitivity": [
            0.25, 0.50, 0.75, 1.00
        ],
        "differentiable_selection": (
            "To be chosen by controlled "
            "training/inference comparison"
        ),
        "primary_task": (
            "Next-step interaction failure prediction"
        )
    },

    "evaluation": {
        "train_times": list(range(0, 8)),
        "validation_times": [8, 9],
        "test_times": [10, 11],
        "model_seeds": [42, 43, 44, 45, 46],
        "scenario_seeds": [101, 102, 103, 104, 105],
        "primary_metric": "Failure Average Precision",
        "secondary_metrics": [
            "ROC-AUC",
            "BCE",
            "Brier score",
            "retained edge fraction",
            "isolated device fraction",
            "end-to-end inference latency",
            "peak GPU memory",
            "mask churn",
            "post-shock performance"
        ]
    },

    "required_baselines": [
        "Trust-MLP without GNN",
        "Full-graph GraphSAGE",
        "Random edge selection",
        "Degree-based edge selection",
        "Trust-only edge selection",
        "GNN sparsification without trust",
        "Trust-aware GNN sparsification "
        "without adaptive controller",
        "TAGS-v3 complete"
    ],

    "safeguards": [
        "Never use future interaction outcomes",
        "Never expose hidden simulation states",
        "Do not modify the original Scenario A",
        "Keep evaluation budgets matched",
        "Report negative results",
        "Do not tune on test results",
        "Measure scorer and controller overhead",
        "Do not claim real-world SIoT trust labels",
        "Do not claim attack robustness without tests",
        "Do not infer GNN utility from model complexity",
        "Freeze scenario parameters before evaluation"
    ],

    "previous_findings": {
        "E2_7_Trust_MLP_AP_mean": 0.410647,
        "E2_8_residual_GNN_selected_delta_AP": 0.0,
        "E2_9_endpoint_only_AP": 0.401760,
        "E2_10_GNN_minus_Node_MLP_mean_delta_AP": -0.004123,
        "E2_10_seeds": 2,
        "interpretation": (
            "No stable incremental message-passing "
            "benefit demonstrated in Scenario A"
        )
    }
}

with open(
    protocol_path, "w", encoding="utf-8"
) as f:
    json.dump(
        protocol,
        f,
        indent=2,
        ensure_ascii=False
    )

print("=" * 72)
print("E3.0 — TAGS-V3 RESEARCH PROTOCOL")
print("=" * 72)
print("Algorithm:", protocol["algorithm"])
print("Scenarios: A, B, C")
print("Scenario A preserved:", True)
print("Encoder:", protocol["method"]["encoder"])
print("Primary task:", protocol["method"]["primary_task"])
print("Primary metric:",
      protocol["evaluation"]["primary_metric"])
print("Model seeds:",
      protocol["evaluation"]["model_seeds"])
print("Scenario seeds:",
      protocol["evaluation"]["scenario_seeds"])

print("\nE3.0 COMPLETED")
print("Frozen protocol:", protocol_path)
print("No simulation or training performed.")
print("No previous artifacts modified.")
print("NEXT: E3.1 — Scenario B generation and audit.")


In [ ]:

# ============================================================
# PHASE III — E3.1
# SCENARIO B: OWNER-CORRELATED TEMPORAL TRUST
# ============================================================
# Santander topology is observed.
# Trust outcomes and latent reliability are SYNTHETIC.
# No GNN training.
# No existing scenario is overwritten.
# Test outcomes are generated but NOT evaluated.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import zipfile
import json
import hashlib

ROOT = Path("/kaggle/working/phase2_adaptive_siot")
E3 = ROOT / "E3_tags_v3"
SANTANDER = (
    ROOT / "E1_trust_scoring" / "santander"
)

PROTOCOL = E3 / "e3_0_research_protocol_v1.json"
assert PROTOCOL.is_file(), "Complete E3.0 first."

with open(PROTOCOL, encoding="utf-8") as f:
    frozen = json.load(f)

assert frozen["algorithm"] == "TAGS-v3"

# ------------------------------------------------------------
# 1. Predefined scenario parameters
# ------------------------------------------------------------

SCENARIO = "B_owner_correlated"
SEED = 101

N = 16216
T = 12
INTERACTION_RATE = 0.15

# New structural factor, fixed BEFORE data generation.
OWNER_EFFECT_STD = 0.65

# Other mechanisms match Scenario A.
EDGE_EFFECT_STD = 0.45
TEMPORAL_RHO = 0.85
TEMPORAL_STD = 0.12
OBSERVATION_FLIP = 0.03

SHOCK_START = 6
SHOCK_FRACTION = 0.08
SHOCK_LOGIT = -1.6
SHOCK_DECAY = 0.90

rng = np.random.default_rng(SEED)

OUT = E3 / "scenarios" / SCENARIO / f"seed_{SEED}"
OBS = OUT / "observed"
HIDDEN = OUT / "simulation_ground_truth"

# Prevent partial or accidental re-generation.
if OUT.exists():
    raise FileExistsError(
        f"Scenario output already exists: {OUT}"
    )

OUT.mkdir(parents=True)
OBS.mkdir()
HIDDEN.mkdir()

print("=" * 72)
print("E3.1 — SCENARIO B: OWNER-CORRELATED TRUST")
print("=" * 72)
print("Seed:", SEED)
print("Owner-effect standard deviation:", OWNER_EFFECT_STD)
print("Devices:", N)
print("Time steps:", T)

# ------------------------------------------------------------
# 2. Read verified SIoT topology
# ------------------------------------------------------------

topology_path = (
    ROOT / "E2_temporal_trust" /
    "observed" / "siot_social_topology.csv.gz"
)

topology = pd.read_csv(topology_path)

assert len(topology) == 146117
assert not topology[["u", "v"]].duplicated().any()

topology = topology.sort_values(
    ["u", "v"]
).reset_index(drop=True)

M = len(topology)

u = topology["u"].to_numpy(np.int32) - 1
v = topology["v"].to_numpy(np.int32) - 1

assert np.all(u < v)
assert u.min() >= 0
assert v.max() < N

# ------------------------------------------------------------
# 3. Read observed device-to-owner mapping
# ------------------------------------------------------------

with zipfile.ZipFile(
    SANTANDER / "raw" / "objects_description.zip"
) as z:
    with z.open("objects_description.csv") as f:
        devices = pd.read_csv(f)

devices = devices.sort_values(
    "id_device"
).reset_index(drop=True)

assert len(devices) == N
assert np.array_equal(
    devices["id_device"].to_numpy(),
    np.arange(1, N + 1)
)
assert devices["id_user"].notna().all()

owner_codes, owner_ids = pd.factorize(
    devices["id_user"],
    sort=True
)

O = len(owner_ids)
owner_sizes = np.bincount(
    owner_codes, minlength=O
)

same_owner = (
    owner_codes[u] == owner_codes[v]
)

print("\nSTRUCTURAL OWNER AUDIT")
print("Unique owners:", O)
print("Mean devices per owner:", round(N / O, 4))
print("Same-owner undirected links:",
      int(same_owner.sum()))
print("Same-owner link fraction:",
      round(float(same_owner.mean()), 6))

# ------------------------------------------------------------
# 4. Generate hidden device and owner effects
# ------------------------------------------------------------

def sigmoid(x):
    return 1 / (
        1 + np.exp(-np.clip(x, -30, 30))
    )

def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

base_reliability = rng.beta(
    5.0, 2.0, size=N
)

# Hidden shared factor for devices with same owner.
owner_effect = rng.normal(
    0.0,
    OWNER_EFFECT_STD,
    size=O
)

# A device inherits its owner's hidden factor.
device_owner_effect = owner_effect[owner_codes]

# Independent hidden link effects.
edge_effect = rng.normal(
    0.0, EDGE_EFFECT_STD, size=M
)

base_logit = logit(base_reliability)

# ------------------------------------------------------------
# 5. Independent device shock, same mechanism as A
# ------------------------------------------------------------

shock_count = int(round(
    SHOCK_FRACTION * N
))

shocked_nodes = rng.choice(
    N,
    size=shock_count,
    replace=False
)

shock_indicator = np.zeros(
    N, dtype=np.float32
)
shock_indicator[shocked_nodes] = 1.0

temporal_state = np.zeros(
    N, dtype=np.float32
)

event_frames = []
audit_rows = []

# ------------------------------------------------------------
# 6. Generate chronological observed interactions
# ------------------------------------------------------------

for t in range(T):

    temporal_state = (
        TEMPORAL_RHO * temporal_state
        + rng.normal(
            0, TEMPORAL_STD, size=N
        )
    ).astype(np.float32)

    if t >= SHOCK_START:
        shock_strength = (
            SHOCK_LOGIT *
            (SHOCK_DECAY ** (t - SHOCK_START))
        )
    else:
        shock_strength = 0.0

    hidden_device_logit = (
        base_logit
        + device_owner_effect
        + temporal_state
        + shock_strength * shock_indicator
    )

    hidden_link_logit = (
        0.5 * (
            hidden_device_logit[u]
            + hidden_device_logit[v]
        )
        + edge_effect
    )

    success_probability = sigmoid(
        hidden_link_logit
    )

    # Same observation process as Scenario A.
    selected = np.flatnonzero(
        rng.random(M) < INTERACTION_RATE
    )

    success = (
        rng.random(len(selected))
        < success_probability[selected]
    ).astype(np.int8)

    flip = (
        rng.random(len(selected))
        < OBSERVATION_FLIP
    )

    success[flip] = 1 - success[flip]

    frame = pd.DataFrame({
        "time": np.full(
            len(selected), t, dtype=np.int16
        ),
        "source": (
            u[selected] + 1
        ).astype(np.int32),
        "target": (
            v[selected] + 1
        ).astype(np.int32),
        "success": success
    })

    event_frames.append(frame)

    audit_rows.append({
        "time": t,
        "observations": int(len(selected)),
        "observed_success_rate": float(
            success.mean()
        ),
        "hidden_mean_link_success_probability": float(
            success_probability.mean()
        ),
        "shock_strength": float(shock_strength)
    })

    # Test is sealed for later final evaluation.
    if t <= 9:
        print(
            f"t={t:02d} | "
            f"observations={len(selected):6,d} | "
            f"success={success.mean():.4f}"
        )
    else:
        print(
            f"t={t:02d} | "
            "TEST GENERATED — METRICS SEALED"
        )

events = pd.concat(
    event_frames, ignore_index=True
)

events = events.sort_values(
    ["time", "source", "target"]
).reset_index(drop=True)

assert events["time"].between(0, 11).all()
assert events["success"].isin([0, 1]).all()
assert not events[
    ["time", "source", "target"]
].duplicated().any()

train = events[events["time"] <= 7].copy()
validation = events[
    events["time"].between(8, 9)
].copy()
test = events[events["time"] >= 10].copy()

assert len(train) + len(validation) + len(test) == len(events)

# ------------------------------------------------------------
# 7. Save strictly separated observed / hidden data
# ------------------------------------------------------------

topology.to_csv(
    OBS / "siot_social_topology.csv.gz",
    index=False,
    compression="gzip"
)

events.to_csv(
    OBS / "all_interactions.csv.gz",
    index=False,
    compression="gzip"
)

train.to_csv(
    OBS / "train.csv.gz",
    index=False,
    compression="gzip"
)

validation.to_csv(
    OBS / "validation.csv.gz",
    index=False,
    compression="gzip"
)

test.to_csv(
    OBS / "test.csv.gz",
    index=False,
    compression="gzip"
)

# Hidden state is solely for later controlled
# simulation audits / oracle evaluation.
np.savez_compressed(
    HIDDEN / "hidden_simulator_state.npz",
    base_reliability=base_reliability,
    owner_effect=owner_effect,
    owner_codes=owner_codes,
    edge_effect=edge_effect,
    shocked_device_ids=shocked_nodes + 1
)

# Full latent audit must not be model input.
pd.DataFrame(audit_rows).to_csv(
    HIDDEN / "latent_temporal_summary.csv",
    index=False
)

# ------------------------------------------------------------
# 8. Reproducibility and integrity metadata
# ------------------------------------------------------------

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(chunk)
    return h.hexdigest()

metadata = {
    "experiment": "E3.1",
    "scenario": SCENARIO,
    "seed": SEED,
    "topology": "Observed Santander SIoT",
    "trust_observations": "Synthetic",
    "owner_mapping": "Observed device metadata",
    "nodes": N,
    "owners": O,
    "undirected_edges": M,
    "time_steps": T,
    "parameters": {
        "base_reliability": "Beta(5,2)",
        "owner_effect_std": OWNER_EFFECT_STD,
        "edge_effect_std": EDGE_EFFECT_STD,
        "temporal_rho": TEMPORAL_RHO,
        "temporal_std": TEMPORAL_STD,
        "interaction_rate": INTERACTION_RATE,
        "observation_flip": OBSERVATION_FLIP,
        "shock_start": SHOCK_START,
        "shock_fraction": SHOCK_FRACTION,
        "shock_logit": SHOCK_LOGIT,
        "shock_decay": SHOCK_DECAY
    },
    "splits": {
        "train": [0, 7],
        "validation": [8, 9],
        "test": [10, 11]
    },
    "owner_statistics": {
        "same_owner_edges": int(same_owner.sum()),
        "same_owner_fraction": float(same_owner.mean()),
        "mean_devices_per_owner": float(
            owner_sizes.mean()
        )
    },
    "integrity": {
        "all_events": int(len(events)),
        "train_events": int(len(train)),
        "validation_events": int(len(validation)),
        "test_events": int(len(test)),
        "topology_sha256": sha256(
            OBS / "siot_social_topology.csv.gz"
        ),
        "train_sha256": sha256(
            OBS / "train.csv.gz"
        ),
        "validation_sha256": sha256(
            OBS / "validation.csv.gz"
        )
    },
    "scientific_limitations": [
        "Latent owner effects are synthetic.",
        "Trust is not directly observed in Santander.",
        "Social topology is fixed.",
        "The relationship-type indicators are not trust labels.",
        "Shared owner effects may also be learnable using non-GNN baselines.",
        "No GNN advantage is assumed or established.",
        "Test outcomes must not guide model design."
    ]
}

metadata_path = (
    OUT / "e3_1_scenario_b_protocol.json"
)

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        metadata,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# 9. Final public audit
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("E3.1 COMPLETED — SCENARIO B")
print("=" * 72)

print("Devices:", N)
print("Owners:", O)
print("Undirected social links:", M)
print("Total simulated interactions:", len(events))
print("Train interactions:", len(train))
print("Validation interactions:", len(validation))
print("Test interactions: SEALED")
print("Owner effect std:", OWNER_EFFECT_STD)

print("\nObserved data:", OBS)
print("Hidden simulator state:", HIDDEN)
print("Protocol:", metadata_path)

print("\nScenario A preserved: YES")
print("No GNN was trained.")
print("No test outcomes were evaluated.")
print("No previous artifacts modified.")
print("\nNEXT: E3.2 — Causal feature construction for Scenario B.")


In [ ]:

# ============================================================
# PHASE III — E3.2
# SCENARIO B: CAUSAL TEMPORAL TRUST FEATURES
# ============================================================
# Same 12-feature definition as E2.1.
# Generate snapshots t=0,...,9 only.
# Read train and validation interactions only.
# NO test files or hidden simulator states accessed.
# ============================================================

from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd

ROOT = Path("/kaggle/working/phase2_adaptive_siot")

SCENARIO = (
    ROOT / "E3_tags_v3" / "scenarios" /
    "B_owner_correlated" / "seed_101"
)

OBS = SCENARIO / "observed"
OUT = SCENARIO / "causal_features"

N = 16216
M = 146117
TIMES = range(10)  # 0,...,9 only
GAMMA = 0.80
CONFIDENCE_K = 5.0

FEATURE_NAMES = [
    "edge_history_count",
    "edge_success_rate",
    "edge_decayed_count",
    "edge_decayed_success_rate",
    "edge_evidence_confidence",
    "edge_age_normalized",
    "edge_never_observed",
    "node_u_success_rate",
    "node_v_success_rate",
    "node_u_evidence_confidence",
    "node_v_evidence_confidence",
    "edge_recent_minus_longterm"
]

# Prevent silently replacing previous outputs.
if OUT.exists():
    raise FileExistsError(
        f"Output already exists: {OUT}"
    )

# ------------------------------------------------------------
# 1. Load only permitted observations
# ------------------------------------------------------------

topology = pd.read_csv(
    OBS / "siot_social_topology.csv.gz"
)

train = pd.read_csv(
    OBS / "train.csv.gz"
)

validation = pd.read_csv(
    OBS / "validation.csv.gz"
)

assert len(topology) == M
assert len(train) == 175503
assert len(validation) == 44216

assert train["time"].between(0, 7).all()
assert validation["time"].between(8, 9).all()

events = pd.concat(
    [train, validation],
    ignore_index=True
)

assert len(events) == 219719

topology = topology.sort_values(
    ["u", "v"]
).reset_index(drop=True)

assert topology[["u", "v"]].duplicated().sum() == 0

u = topology["u"].to_numpy(np.int64) - 1
v = topology["v"].to_numpy(np.int64) - 1

assert np.all(u < v)
assert u.min() >= 0
assert v.max() < N

edge_keys = u * N + v
assert np.all(np.diff(edge_keys) > 0)

# ------------------------------------------------------------
# 2. Map observed events to topology edges
# ------------------------------------------------------------

eu = events["source"].to_numpy(np.int64) - 1
ev = events["target"].to_numpy(np.int64) - 1

assert np.all(eu < ev)

event_keys = eu * N + ev

event_edge_id = np.searchsorted(
    edge_keys, event_keys
)

assert np.all(event_edge_id < M)
assert np.array_equal(
    edge_keys[event_edge_id],
    event_keys
)

event_time = events["time"].to_numpy(np.int16)
event_success = events["success"].to_numpy(np.int8)

assert set(np.unique(event_success)) == {0, 1}

# One observed outcome per edge and time.
unique_event_keys = (
    event_time.astype(np.int64) * M
    + event_edge_id
)

assert len(np.unique(unique_event_keys)) == len(events)

# ------------------------------------------------------------
# 3. Historical accumulators
# ------------------------------------------------------------

edge_s = np.zeros(M, dtype=np.float32)
edge_f = np.zeros(M, dtype=np.float32)

decay_s = np.zeros(M, dtype=np.float32)
decay_f = np.zeros(M, dtype=np.float32)

node_s = np.zeros(N, dtype=np.float32)
node_f = np.zeros(N, dtype=np.float32)

last_seen = np.full(M, -1, dtype=np.int16)

def smoothed_rate(s, f):
    return (s + 1.0) / (s + f + 2.0)

def evidence_confidence(s, f):
    count = s + f
    return count / (count + CONFIDENCE_K)

OUT.mkdir(parents=True)

audit = []
cumulative_events = 0

print("=" * 72)
print("E3.2 — SCENARIO B CAUSAL TRUST FEATURES")
print("=" * 72)
print("Scenario: B_owner_correlated")
print("Seed: 101")
print("Devices:", N)
print("Social links:", M)
print("Generated snapshots: t=0,...,9")
print("Features per link:", len(FEATURE_NAMES))
print("Temporal decay:", GAMMA)
print("Test data accessed: NO")

# ------------------------------------------------------------
# 4. Construct one snapshot before observing time t
# ------------------------------------------------------------

for t in TIMES:

    # Advance decay before preparing current snapshot.
    if t > 0:
        decay_s *= GAMMA
        decay_f *= GAMMA

    history_count = edge_s + edge_f
    history_rate = smoothed_rate(edge_s, edge_f)

    decayed_count = decay_s + decay_f
    decayed_rate = smoothed_rate(
        decay_s, decay_f
    )

    edge_conf = evidence_confidence(
        edge_s, edge_f
    )

    node_rate = smoothed_rate(
        node_s, node_f
    )

    node_conf = evidence_confidence(
        node_s, node_f
    )

    never_seen = last_seen == -1

    # Match E2.1: normalize age by original T=12.
    age = np.where(
        never_seen,
        12,
        t - last_seen
    ).astype(np.float32)

    age_normalized = age / 12.0

    X = np.column_stack([
        history_count,
        history_rate,
        decayed_count,
        decayed_rate,
        edge_conf,
        age_normalized,
        never_seen.astype(np.float32),
        node_rate[u],
        node_rate[v],
        node_conf[u],
        node_conf[v],
        decayed_rate - history_rate
    ]).astype(np.float32)

    assert X.shape == (M, 12)
    assert np.isfinite(X).all()

    # Strict causality invariant.
    assert int(history_count.sum()) == cumulative_events

    # Record PRE-UPDATE statistics.
    record = {
        "time": int(t),
        "history_before_t": int(history_count.sum()),
        "edges_with_history": int(
            (history_count > 0).sum()
        ),
        "mean_historical_success_rate": float(
            history_rate.mean()
        ),
        "mean_evidence_confidence": float(
            edge_conf.mean()
        ),
        "snapshot": f"snapshot_t{t:02d}.npz"
    }

    # Snapshot t is saved before any time-t outcomes
    # are used in the accumulators.
    np.savez_compressed(
        OUT / record["snapshot"],
        features=X,
        time=np.array(t, dtype=np.int16),
        feature_names=np.array(FEATURE_NAMES)
    )

    # Only NOW incorporate outcomes observed at t.
    current = event_time == t

    indices = event_edge_id[current]
    outcomes = event_success[current]

    successes = outcomes.astype(np.float32)
    failures = 1.0 - successes

    np.add.at(edge_s, indices, successes)
    np.add.at(edge_f, indices, failures)

    np.add.at(decay_s, indices, successes)
    np.add.at(decay_f, indices, failures)

    np.add.at(node_s, u[indices], successes)
    np.add.at(node_s, v[indices], successes)

    np.add.at(node_f, u[indices], failures)
    np.add.at(node_f, v[indices], failures)

    last_seen[indices] = t

    cumulative_events += len(indices)

    record["observations_at_t"] = int(
        len(indices)
    )

    audit.append(record)

    print(
        f"t={t:02d} | "
        f"past_events={record['history_before_t']:6,d} | "
        f"observed_edges={record['edges_with_history']:6,d} | "
        f"new_events={len(indices):6,d}",
        flush=True
    )

# ------------------------------------------------------------
# 5. Final integrity checks
# ------------------------------------------------------------

assert cumulative_events == len(events)

with np.load(OUT / "snapshot_t00.npz") as f:
    initial = f["features"]

with np.load(OUT / "snapshot_t09.npz") as f:
    last = f["features"]

assert np.all(initial[:, 0] == 0)
assert np.all(initial[:, 1] == 0.5)
assert np.all(initial[:, 6] == 1)

# t=9 must not contain observations from t=9.
expected_history_t9 = int(
    (events["time"] < 9).sum()
)

assert int(last[:, 0].sum()) == expected_history_t9

# The mapping is identical for all snapshots.
topology.to_csv(
    OUT / "edge_index_mapping.csv.gz",
    index=False,
    compression="gzip"
)

# ------------------------------------------------------------
# 6. Save audit and feature contract
# ------------------------------------------------------------

report = {
    "experiment": "E3.2",
    "scenario": "B_owner_correlated",
    "seed": 101,
    "feature_definition": "Identical to E2.1",
    "feature_count": 12,
    "feature_names": FEATURE_NAMES,
    "time_steps_generated": list(TIMES),
    "temporal_decay": GAMMA,
    "evidence_confidence_k": CONFIDENCE_K,
    "age_normalization_horizon": 12,
    "total_available_events_0_to_9": int(
        len(events)
    ),
    "future_observations_used": False,
    "hidden_simulator_state_used": False,
    "test_data_accessed": False,
    "snapshots": audit,
    "scientific_note": (
        "Validation t=9 features contain observed "
        "outcomes from t=8. This implements a "
        "prequential evaluation protocol and "
        "does not permit training on validation "
        "outcomes."
    )
}

report_path = (
    OUT / "e3_2_causal_feature_audit.json"
)

with open(
    report_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(report, f, indent=2)

print("\n" + "=" * 72)
print("E3.2 COMPLETED — SCENARIO B")
print("=" * 72)

print("Saved snapshots:", len(audit))
print("Snapshot shape:", last.shape)
print("t=0 historical observations: 0 — VERIFIED")
print(
    "t=9 historical observations:",
    int(last[:, 0].sum())
)

print(
    "Total train + validation events:",
    len(events)
)

print("Test accessed: NO")
print("Hidden state accessed: NO")
print("Scenario A modified: NO")

print("\nOutput:", OUT)
print("Audit:", report_path)

print(
    "\nNEXT: E3.3 — Establish predictive baselines "
    "before training TAGS-v3."
)


In [ ]:

# ============================================================
# PHASE III — E3.3
# SCENARIO B: PREDICTIVE BASELINES
# ============================================================
# Same preprocessing and MLP architecture as E2.7.
# Train: t=5,6,7 | Validation: t=8,9
# No Test or hidden simulator data accessed.
# No GNN or sparsification.
# ============================================================

from pathlib import Path
import json
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    log_loss
)

ROOT = Path("/kaggle/working/phase2_adaptive_siot")

SCENARIO = (
    ROOT / "E3_tags_v3" / "scenarios" /
    "B_owner_correlated" / "seed_101"
)

OBS = SCENARIO / "observed"
FEAT = SCENARIO / "causal_features"
OUT = SCENARIO / "baselines"

if OUT.exists():
    raise FileExistsError(
        f"Baseline output already exists: {OUT}"
    )

OUT.mkdir(parents=True)

TRAIN_TIMES = [5, 6, 7]
VAL_TIMES = [8, 9]
SEEDS = [42, 43, 44, 45, 46]

MAX_EPOCHS = 80
PATIENCE = 16
BATCH_SIZE = 4096
LEARNING_RATE = 1e-3

N = 16216
M = 146117

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 72)
print("E3.3 — SCENARIO B PREDICTIVE BASELINES")
print("=" * 72)
print("Device:", DEVICE)
print("Scenario seed: 101")
print("Train:", TRAIN_TIMES)
print("Validation:", VAL_TIMES)

# ------------------------------------------------------------
# 1. Load topology and ONLY permitted labels
# ------------------------------------------------------------

topology = pd.read_csv(
    FEAT / "edge_index_mapping.csv.gz"
)

train = pd.read_csv(OBS / "train.csv.gz")
validation = pd.read_csv(OBS / "validation.csv.gz")

assert len(topology) == M
assert train["time"].between(0, 7).all()
assert validation["time"].between(8, 9).all()

topology = topology.sort_values(
    ["u", "v"]
).reset_index(drop=True)

u = topology["u"].to_numpy(np.int64) - 1
v = topology["v"].to_numpy(np.int64) - 1

edge_keys = u * N + v

assert np.all(np.diff(edge_keys) > 0)

RELATIONS = ["OOR", "POR", "C_LOR", "SOR", "SOR2"]

relation_features = topology[
    RELATIONS
].to_numpy(np.float32)

# ------------------------------------------------------------
# 2. Reproduce E2.7 edge feature preprocessing
# ------------------------------------------------------------

def snapshot_features(t):
    path = FEAT / f"snapshot_t{t:02d}.npz"

    with np.load(path) as f:
        X = f["features"].copy()
        saved_time = int(f["time"])

    assert saved_time == t
    assert X.shape == (M, 12)

    # Log-transform count features only.
    X[:, 0] = np.log1p(X[:, 0])
    X[:, 2] = np.log1p(X[:, 2])

    # Undirected endpoint-invariant representation.
    for left, right in [(7, 8), (9, 10)]:
        a = X[:, left].copy()
        b = X[:, right].copy()

        X[:, left] = (a + b) / 2
        X[:, right] = np.abs(a - b)

    result = np.concatenate(
        [X, relation_features],
        axis=1
    ).astype(np.float32)

    assert result.shape == (M, 17)
    return result


def extract_examples(t, event_frame):
    step = event_frame[
        event_frame["time"] == t
    ].copy()

    su = step["source"].to_numpy(np.int64) - 1
    sv = step["target"].to_numpy(np.int64) - 1

    assert np.all(su < sv)

    keys = su * N + sv
    ids = np.searchsorted(edge_keys, keys)

    assert np.all(ids < M)
    assert np.array_equal(
        edge_keys[ids], keys
    )

    X = snapshot_features(t)[ids]

    # Positive target class = interaction FAILURE.
    y = (
        1 - step["success"].to_numpy(np.int8)
    )

    return X, y


train_parts = [
    extract_examples(t, train)
    for t in TRAIN_TIMES
]

val_parts = [
    extract_examples(t, validation)
    for t in VAL_TIMES
]

X_train = np.concatenate(
    [part[0] for part in train_parts]
)

y_train = np.concatenate(
    [part[1] for part in train_parts]
)

X_val = np.concatenate(
    [part[0] for part in val_parts]
)

y_val = np.concatenate(
    [part[1] for part in val_parts]
)

assert X_train.shape[1] == 17
assert X_val.shape[1] == 17
assert np.isfinite(X_train).all()
assert np.isfinite(X_val).all()

scaler = StandardScaler()

X_train = scaler.fit_transform(
    X_train
).astype(np.float32)

X_val = scaler.transform(
    X_val
).astype(np.float32)

# ------------------------------------------------------------
# 3. Shared metrics
# ------------------------------------------------------------

def evaluate(y, p):
    p = np.clip(p, 1e-7, 1 - 1e-7)

    return {
        "failure_AP": float(
            average_precision_score(y, p)
        ),
        "ROC_AUC": float(
            roc_auc_score(y, p)
        ),
        "BCE": float(
            log_loss(y, p, labels=[0, 1])
        )
    }


print("\nDATA AUDIT")
print("Train examples:", len(y_train))
print("Validation examples:", len(y_val))
print("Train failure prevalence:",
      round(float(y_train.mean()), 6))
print("Validation failure prevalence:",
      round(float(y_val.mean()), 6))

# ------------------------------------------------------------
# 4. Logistic Regression
# ------------------------------------------------------------

linear = LogisticRegression(
    solver="lbfgs",
    max_iter=500,
    random_state=42
)

linear.fit(X_train, y_train)

lr_p = linear.predict_proba(X_val)[:, 1]

lr_result = evaluate(y_val, lr_p)

print("\nLOGISTIC REGRESSION")
print(lr_result)

# ------------------------------------------------------------
# 5. Exactly the E2.7 MLP architecture
# ------------------------------------------------------------

class TrustMLP_B(nn.Module):
    def __init__(self, input_dim=17):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.network(x).squeeze(-1)


train_x = torch.from_numpy(
    X_train
).to(DEVICE)

train_y = torch.from_numpy(
    y_train.astype(np.float32)
).to(DEVICE)

val_x = torch.from_numpy(
    X_val
).to(DEVICE)

criterion = nn.BCEWithLogitsLoss()

results = []

# ------------------------------------------------------------
# 6. Five independent model seeds
# ------------------------------------------------------------

for seed in SEEDS:

    torch.manual_seed(seed)
    np.random.seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    model = TrustMLP_B(17).to(DEVICE)

    prevalence = float(y_train.mean())

    with torch.no_grad():
        model.network[-1].bias.fill_(
            np.log(prevalence / (1 - prevalence))
        )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=1e-4
    )

    best_ap = -np.inf
    best_epoch = None
    best_metrics = None
    best_state = None
    wait = 0

    for epoch in range(1, MAX_EPOCHS + 1):

        model.train()

        rng = np.random.default_rng(
            seed * 10000 + epoch
        )

        indices = rng.permutation(
            len(y_train)
        )

        for start in range(
            0, len(indices), BATCH_SIZE
        ):

            batch_ids = torch.as_tensor(
                indices[start:start + BATCH_SIZE],
                dtype=torch.long,
                device=DEVICE
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            logits = model(
                train_x[batch_ids]
            )

            loss = criterion(
                logits,
                train_y[batch_ids]
            )

            loss.backward()
            optimizer.step()

        model.eval()

        with torch.no_grad():
            probabilities = torch.sigmoid(
                model(val_x)
            ).cpu().numpy()

        score = evaluate(y_val, probabilities)

        if score["failure_AP"] > best_ap + 1e-5:
            best_ap = score["failure_AP"]
            best_epoch = epoch
            best_metrics = score
            best_state = copy.deepcopy(
                model.state_dict()
            )
            wait = 0
        else:
            wait += 1

        if wait >= PATIENCE:
            break

    assert best_state is not None

    result = {
        "seed": seed,
        "best_epoch": best_epoch,
        **best_metrics
    }

    results.append(result)

    print(
        f"Seed {seed} | "
        f"epoch={best_epoch:3d} | "
        f"AP={best_metrics['failure_AP']:.6f} | "
        f"AUC={best_metrics['ROC_AUC']:.6f} | "
        f"BCE={best_metrics['BCE']:.6f}",
        flush=True
    )

# ------------------------------------------------------------
# 7. Persist report and compare with Scenario A
# ------------------------------------------------------------

df = pd.DataFrame(results)

summary = {
    "experiment": "E3.3",
    "scenario": "B_owner_correlated",
    "scenario_seed": 101,
    "model_seeds": SEEDS,
    "train_times": TRAIN_TIMES,
    "validation_times": VAL_TIMES,
    "test_accessed": False,
    "hidden_state_accessed": False,
    "LR": lr_result,
    "MLP_mean": {
        k: float(df[k].mean())
        for k in ["failure_AP", "ROC_AUC", "BCE"]
    },
    "MLP_std": {
        k: float(df[k].std(ddof=1))
        for k in ["failure_AP", "ROC_AUC", "BCE"]
    },
    "scenario_A_reference": {
        "LR_failure_AP": 0.404299,
        "MLP_failure_AP_mean": 0.410647
    },
    "scientific_limitations": [
        "Only one scenario realization was used.",
        "MLP seeds do not represent simulation uncertainty.",
        "Failure prevalence differs across scenarios.",
        "No GNN or sparsification advantage established."
    ]
}

df.to_csv(
    OUT / "e3_3_trust_mlp_seeds.csv",
    index=False
)

report_path = OUT / "e3_3_baseline_report.json"

with open(
    report_path, "w", encoding="utf-8"
) as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 72)
print("E3.3 COMPLETED — SCENARIO B BASELINES")
print("=" * 72)

print("LR failure AP:",
      round(lr_result["failure_AP"], 6))

print(
    "MLP failure AP:",
    round(summary["MLP_mean"]["failure_AP"], 6),
    "+/-",
    round(summary["MLP_std"]["failure_AP"], 6)
)

print(
    "MLP ROC-AUC:",
    round(summary["MLP_mean"]["ROC_AUC"], 6)
)

print(
    "MLP BCE:",
    round(summary["MLP_mean"]["BCE"], 6)
)

print("\nScenario A MLP AP reference: 0.410647")
print("Scenario B results must be interpreted")
print("relative to their own failure prevalence.")

print("\nSaved:", report_path)
print("Test accessed: NO")
print("No GNN trained.")
print("No sparsification performed.")
print("\nNEXT: E3.4 — Evaluate structural information")
print("contribution before TAGS-v3 training.")


In [ ]:

# ============================================================
# PHASE III — E3.4
# OWNER-EVIDENCE INCREMENTAL UTILITY
# ============================================================
# Scenario B / seed 101
# Train: t=5,6,7 | Validation: t=8,9
# No test access; no hidden simulator access.
# No GNN training or sparsification.
# ============================================================

from pathlib import Path
import zipfile
import json
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    log_loss
)

ROOT = Path("/kaggle/working/phase2_adaptive_siot")

SCENARIO = (
    ROOT / "E3_tags_v3" / "scenarios" /
    "B_owner_correlated" / "seed_101"
)

OBS = SCENARIO / "observed"
FEAT = SCENARIO / "causal_features"
OUT = SCENARIO / "owner_evidence_audit"

if OUT.exists():
    raise FileExistsError(
        f"Output directory already exists: {OUT}"
    )

N = 16216
M = 146117
TRAIN_TIMES = [5, 6, 7]
VAL_TIMES = [8, 9]
RELATIONS = ["OOR", "POR", "C_LOR", "SOR", "SOR2"]

# ------------------------------------------------------------
# 1. Load permitted observations and public metadata
# ------------------------------------------------------------

topology = pd.read_csv(
    FEAT / "edge_index_mapping.csv.gz"
).sort_values(["u", "v"]).reset_index(drop=True)

train = pd.read_csv(OBS / "train.csv.gz")
val = pd.read_csv(OBS / "validation.csv.gz")

assert len(topology) == M
assert train["time"].between(0, 7).all()
assert val["time"].between(8, 9).all()

events = pd.concat([train, val], ignore_index=True)

device_archive = (
    ROOT / "E1_trust_scoring" /
    "santander" / "raw" /
    "objects_description.zip"
)

with zipfile.ZipFile(device_archive) as z:
    with z.open("objects_description.csv") as f:
        devices = pd.read_csv(f)

devices = devices.sort_values("id_device")

assert np.array_equal(
    devices["id_device"].to_numpy(),
    np.arange(1, N + 1)
)

owners, owner_ids = pd.factorize(
    devices["id_user"], sort=True
)

owner_count = len(owner_ids)

# Negative control: shuffle membership while
# preserving the number and sizes of owner groups.
rng = np.random.default_rng(20261116)
shuffled_owners = rng.permutation(owners)

u = topology["u"].to_numpy(np.int64) - 1
v = topology["v"].to_numpy(np.int64) - 1

edge_keys = u * N + v

assert np.all(np.diff(edge_keys) > 0)

relation_matrix = topology[
    RELATIONS
].to_numpy(np.float32)

print("=" * 72)
print("E3.4 — OWNER-EVIDENCE CONTRIBUTION")
print("=" * 72)
print("Scenario: B / seed 101")
print("Devices:", N)
print("Owners:", owner_count)
print("Edges:", M)
print("Train:", TRAIN_TIMES)
print("Validation:", VAL_TIMES)
print("Test accessed: NO")

# ------------------------------------------------------------
# 2. Historical device involvement accumulators
# ------------------------------------------------------------
# Each observed interaction adds one observation
# to EACH endpoint device.
#
# Owner totals are computed by summing endpoint
# involvement records. Same-owner interactions
# can therefore contribute two owner participations.
#
# Peer totals subtract the target device's
# own participations.

node_n = np.zeros(N, dtype=np.float64)
node_s = np.zeros(N, dtype=np.float64)

event_u = (
    events["source"].to_numpy(np.int64) - 1
)
event_v = (
    events["target"].to_numpy(np.int64) - 1
)
event_t = events["time"].to_numpy(np.int16)
event_success = events[
    "success"
].to_numpy(np.float64)

assert np.all(event_u < event_v)

def owner_peer_features(group_codes):
    group_n = np.bincount(
        group_codes,
        weights=node_n,
        minlength=owner_count
    )

    group_s = np.bincount(
        group_codes,
        weights=node_s,
        minlength=owner_count
    )

    # Leave the current device's own
    # interaction history out.
    peer_n = np.maximum(
        group_n[group_codes] - node_n, 0
    )

    peer_s = np.maximum(
        group_s[group_codes] - node_s, 0
    )

    peer_s = np.minimum(peer_s, peer_n)

    peer_rate = (
        (peer_s + 1) / (peer_n + 2)
    )

    peer_conf = peer_n / (peer_n + 5)

    node_peer = np.column_stack([
        np.log1p(peer_n),
        peer_rate,
        peer_conf
    ]).astype(np.float32)

    # Endpoint-order invariant features:
    # mean and absolute difference.
    a = node_peer[u]
    b = node_peer[v]

    return np.concatenate([
        (a + b) / 2,
        np.abs(a - b)
    ], axis=1).astype(np.float32)

def base_features(t):
    with np.load(
        FEAT / f"snapshot_t{t:02d}.npz"
    ) as f:
        X = f["features"].copy()
        assert int(f["time"]) == t

    assert X.shape == (M, 12)

    X[:, 0] = np.log1p(X[:, 0])
    X[:, 2] = np.log1p(X[:, 2])

    for a, b in [(7, 8), (9, 10)]:
        left = X[:, a].copy()
        right = X[:, b].copy()

        X[:, a] = (left + right) / 2
        X[:, b] = np.abs(left - right)

    return np.concatenate([
        X, relation_matrix
    ], axis=1).astype(np.float32)

# ------------------------------------------------------------
# 3. Construct strictly causal time snapshots
# ------------------------------------------------------------

feature_sets = {
    "Baseline": {"train": [], "val": []},
    "Owner evidence": {"train": [], "val": []},
    "Shuffled owner": {"train": [], "val": []}
}

labels = {"train": [], "val": []}
audit = []

for t in range(10):

    # Historical state currently contains
    # only outcomes with time < t.
    assert int(node_n.sum()) == (
        2 * int((event_t < t).sum())
    )

    if t in TRAIN_TIMES + VAL_TIMES:
        base_X = base_features(t)

        owner_X = owner_peer_features(owners)
        shuffled_X = owner_peer_features(
            shuffled_owners
        )

        frame = train if t <= 7 else val
        step = frame[frame["time"] == t]

        su = (
            step["source"].to_numpy(np.int64) - 1
        )
        sv = (
            step["target"].to_numpy(np.int64) - 1
        )

        keys = su * N + sv
        ids = np.searchsorted(edge_keys, keys)

        assert np.all(ids < M)
        assert np.array_equal(
            edge_keys[ids], keys
        )

        y = (
            1 - step["success"].to_numpy(np.int8)
        )

        split = "train" if t <= 7 else "val"

        feature_sets["Baseline"][split].append(
            base_X[ids]
        )

        feature_sets["Owner evidence"][split].append(
            np.concatenate([
                base_X[ids],
                owner_X[ids]
            ], axis=1)
        )

        feature_sets["Shuffled owner"][split].append(
            np.concatenate([
                base_X[ids],
                shuffled_X[ids]
            ], axis=1)
        )

        labels[split].append(y)

        audit.append({
            "time": t,
            "past_interactions": int(
                (event_t < t).sum()
            ),
            "prediction_examples": len(step)
        })

        print(
            f"t={t:02d} | "
            f"past={audit[-1]['past_interactions']:,} | "
            f"targets={len(step):,}"
        )

    # Update only AFTER computing snapshot t.
    current = event_t == t

    a = event_u[current]
    b = event_v[current]
    s = event_success[current]

    np.add.at(node_n, a, 1.0)
    np.add.at(node_n, b, 1.0)

    np.add.at(node_s, a, s)
    np.add.at(node_s, b, s)

# ------------------------------------------------------------
# 4. Train-only scaling and identical LR evaluation
# ------------------------------------------------------------

y_train = np.concatenate(labels["train"])
y_val = np.concatenate(labels["val"])

def score(y, p):
    p = np.clip(p, 1e-7, 1 - 1e-7)

    return {
        "failure_AP": float(
            average_precision_score(y, p)
        ),
        "ROC_AUC": float(
            roc_auc_score(y, p)
        ),
        "BCE": float(
            log_loss(y, p, labels=[0, 1])
        )
    }

rows = []

for name, parts in feature_sets.items():

    X_train = np.concatenate(parts["train"])
    X_val = np.concatenate(parts["val"])

    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)

    model = LogisticRegression(
        max_iter=500,
        solver="lbfgs",
        random_state=42
    )

    model.fit(X_train, y_train)

    p = model.predict_proba(X_val)[:, 1]
    m = score(y_val, p)

    rows.append({
        "method": name,
        "features": X_train.shape[1],
        **m
    })

results = pd.DataFrame(rows)

baseline_ap = float(
    results.loc[
        results["method"] == "Baseline",
        "failure_AP"
    ].iloc[0]
)

owner_ap = float(
    results.loc[
        results["method"] == "Owner evidence",
        "failure_AP"
    ].iloc[0]
)

shuffled_ap = float(
    results.loc[
        results["method"] == "Shuffled owner",
        "failure_AP"
    ].iloc[0]
)

# ------------------------------------------------------------
# 5. Save evaluation-only audit
# ------------------------------------------------------------

report = {
    "experiment": "E3.4",
    "scenario": "B_owner_correlated",
    "seed": 101,
    "train_times": TRAIN_TIMES,
    "validation_times": VAL_TIMES,
    "test_accessed": False,
    "hidden_state_accessed": False,
    "owner_labels_are_observed_metadata": True,
    "peer_history_excludes_own_device_events": True,
    "owner_history_uses_only_times_before_prediction": True,
    "same_owner_events_count_twice_as_participations": True,
    "owner_group_shuffle_seed": 20261116,
    "results": rows,
    "delta_AP_owner_minus_base": owner_ap - baseline_ap,
    "delta_AP_shuffled_minus_base": (
        shuffled_ap - baseline_ap
    ),
    "limitation": (
        "Single scenario seed and one shuffled grouping. "
        "Incremental prediction is not proof of GNN value."
    )
}

results.to_csv(
    OUT / "e3_4_owner_evidence_comparison.csv",
    index=False
)

with open(
    OUT / "e3_4_owner_evidence_report.json",
    "w", encoding="utf-8"
) as f:
    json.dump(report, f, indent=2)

print("\n" + "=" * 72)
print("E3.4 COMPLETED — OWNER-EVIDENCE AUDIT")
print("=" * 72)

print(results.round(6).to_string(index=False))

print(
    "\nOwner minus Baseline AP:",
    f"{owner_ap - baseline_ap:+.6f}"
)

print(
    "Shuffled minus Baseline AP:",
    f"{shuffled_ap - baseline_ap:+.6f}"
)

print("\nValidation failure prevalence:",
      round(float(y_val.mean()), 6))

print("No GNN training.")
print("No test or hidden-state access.")
print("Previous experiments remain unchanged.")
print("Output:", OUT)


In [ ]:

# ============================================================
# E3.4 — FIX OUTPUT DIRECTORY AND SAVE RESULTS
# ============================================================

from pathlib import Path
import json

# Create missing directory
OUT.mkdir(parents=True, exist_ok=True)

# Save results
results.to_csv(
    OUT / "e3_4_owner_evidence_comparison.csv",
    index=False
)

with open(
    OUT / "e3_4_owner_evidence_report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

# Display results
print("=" * 72)
print("E3.4 COMPLETED — OWNER-EVIDENCE AUDIT")
print("=" * 72)

print(results.round(6).to_string(index=False))

print(
    "\nOwner minus Baseline AP:",
    f"{owner_ap - baseline_ap:+.6f}"
)

print(
    "Shuffled minus Baseline AP:",
    f"{shuffled_ap - baseline_ap:+.6f}"
)

print("\nSaved successfully to:", OUT)
print("No additional training or simulation performed.")



In [ ]:

# ============================================================
# PHASE III — E3.5
# TAGS-v3: EXACT-BUDGET STRAIGHT-THROUGH SMOKE TEST
# ============================================================
# Scenario B, seed 101
# Causal snapshot: t=7
# Supervision: observed interaction outcomes at t=7
# No optimizer steps, test access, or hidden-state use
# ============================================================

from pathlib import Path
import json
import zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler

ROOT = Path("/kaggle/working/phase2_adaptive_siot")
SCENARIO = (
    ROOT / "E3_tags_v3" / "scenarios" /
    "B_owner_correlated" / "seed_101"
)
FEAT = SCENARIO / "causal_features"
OBS = SCENARIO / "observed"
OUT = SCENARIO / "tags_v3"

OUT.mkdir(parents=True, exist_ok=True)

SEED = 42
N = 16216
M = 146117
T = 7
KEEP_RATIO = 0.50
K = round(KEEP_RATIO * M)

torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

REL = ["OOR", "POR", "C_LOR", "SOR", "SOR2"]

# ------------------------------------------------------------
# 1. Topology and strictly past trust evidence
# ------------------------------------------------------------

topology = pd.read_csv(
    FEAT / "edge_index_mapping.csv.gz"
).sort_values(["u", "v"]).reset_index(drop=True)

assert len(topology) == M

src = topology["u"].to_numpy(np.int64) - 1
dst = topology["v"].to_numpy(np.int64) - 1

assert np.all(src < dst)

edge_keys = src * N + dst
assert np.all(np.diff(edge_keys) > 0)

relations = topology[REL].to_numpy(np.float32)

def historical_features(t):
    with np.load(
        FEAT / f"snapshot_t{t:02d}.npz"
    ) as f:
        X = f["features"].copy()
        assert int(f["time"]) == t

    assert X.shape == (M, 12)

    X[:, 0] = np.log1p(X[:, 0])
    X[:, 2] = np.log1p(X[:, 2])

    for a, b in [(7, 8), (9, 10)]:
        left = X[:, a].copy()
        right = X[:, b].copy()
        X[:, a] = (left + right) / 2
        X[:, b] = np.abs(left - right)

    return np.concatenate(
        [X, relations], axis=1
    ).astype(np.float32)

# Train-only reference times for feature scaling.
edge_scaler = StandardScaler()

edge_scaler.fit(np.concatenate([
    historical_features(t)
    for t in [5, 6, 7]
]))

Xe_np = edge_scaler.transform(
    historical_features(T)
).astype(np.float32)

# ------------------------------------------------------------
# 2. Causal node features
# ------------------------------------------------------------
# Derive historical device statistics only from
# observations before t=7.

train = pd.read_csv(OBS / "train.csv.gz")
past = train[train["time"] < T]
current = train[train["time"] == T]

assert past["time"].max() < T
assert current["time"].eq(T).all()

with zipfile.ZipFile(
    ROOT / "E1_trust_scoring" /
    "santander" / "raw" /
    "objects_description.zip"
) as z:
    with z.open("objects_description.csv") as f:
        devices = pd.read_csv(f)

devices = devices.sort_values("id_device")
assert np.array_equal(
    devices["id_device"].to_numpy(),
    np.arange(1, N + 1)
)

type_codes, type_names = pd.factorize(
    devices["device_type"], sort=True
)

type_onehot = np.eye(
    len(type_names), dtype=np.float32
)[type_codes]

degree = np.bincount(
    np.r_[src, dst],
    minlength=N
).astype(np.float32)

def get_node_history(t):
    subset = train[train["time"] < t]

    a = subset["source"].to_numpy(np.int64) - 1
    b = subset["target"].to_numpy(np.int64) - 1
    successes = subset["success"].to_numpy(np.float32)

    count = np.bincount(
        np.r_[a, b], minlength=N
    ).astype(np.float32)

    good = np.bincount(
        np.r_[a, b],
        weights=np.r_[successes, successes],
        minlength=N
    ).astype(np.float32)

    rate = (good + 1) / (count + 2)
    confidence = count / (count + 5)

    return np.column_stack([
        np.log1p(degree),
        np.log1p(count),
        rate,
        confidence
    ]).astype(np.float32)

node_scaler = StandardScaler()
node_scaler.fit(np.concatenate([
    get_node_history(t)
    for t in [5, 6, 7]
]))

node_dynamic = node_scaler.transform(
    get_node_history(T)
).astype(np.float32)

Xn_np = np.concatenate([
    type_onehot,
    node_dynamic
], axis=1)

# Controller uses time-varying causal node evidence.
Xc_np = node_dynamic.copy()

# ------------------------------------------------------------
# 3. Align targets with canonical undirected edges
# ------------------------------------------------------------

target_u = current["source"].to_numpy(np.int64) - 1
target_v = current["target"].to_numpy(np.int64) - 1

target_keys = target_u * N + target_v
target_ids = np.searchsorted(edge_keys, target_keys)

assert np.all(target_ids < M)
assert np.array_equal(
    edge_keys[target_ids], target_keys
)

# Positive class = FAILURE.
y_np = (
    1 - current["success"].to_numpy(np.float32)
)

def float_tensor(x):
    return torch.as_tensor(
        x, dtype=torch.float32, device=DEVICE
    )

Xn = float_tensor(Xn_np)
Xe = float_tensor(Xe_np)
Xc = float_tensor(Xc_np)
Y = float_tensor(y_np)

U = torch.as_tensor(src, dtype=torch.long, device=DEVICE)
V = torch.as_tensor(dst, dtype=torch.long, device=DEVICE)

TARGET = torch.as_tensor(
    target_ids, dtype=torch.long, device=DEVICE
)

# ------------------------------------------------------------
# 4. Sparse weighted GraphSAGE layer
# ------------------------------------------------------------

class TAGSWeightedSAGE(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.self_map = nn.Linear(in_dim, out_dim)
        self.neighbor_map = nn.Linear(
            in_dim, out_dim, bias=False
        )

    def forward(self, x, u, v, weight):
        w = weight[:, None]

        msg = torch.zeros_like(x)
        msg = msg.index_add(0, u, x[v] * w)
        msg = msg.index_add(0, v, x[u] * w)

        deg = x.new_zeros((x.shape[0], 1))
        deg = deg.index_add(0, u, w)
        deg = deg.index_add(0, v, w)

        agg = msg / deg.clamp_min(1e-6)

        return (
            self.self_map(x)
            + self.neighbor_map(agg)
        )

class TAGSEncoder(nn.Module):
    def __init__(self, input_dim, hidden=32):
        super().__init__()
        self.first = TAGSWeightedSAGE(
            input_dim, hidden
        )
        self.second = TAGSWeightedSAGE(
            hidden, hidden
        )

    def forward(self, x, u, v, weights):
        h = F.relu(
            self.first(x, u, v, weights)
        )
        return self.second(h, u, v, weights)

# ------------------------------------------------------------
# 5. Exact-budget Straight-Through Top-K
# ------------------------------------------------------------


def straight_through_topk(logits, k, tau=1.0):
    # Exact K selected relationships in forward pass.
    assert 0 < k <= logits.numel()
    assert tau > 0

    chosen = torch.topk(
        logits.detach(),
        k=k,
        sorted=False
    ).indices

    hard = torch.zeros_like(logits)
    hard[chosen] = 1.0

    # Detached threshold: surrogate gradient only.
    threshold = torch.topk(
        logits.detach(),
        k=k,
        sorted=True
    ).values[-1]

    soft = torch.sigmoid(
        (logits - threshold) / tau
    )

    # Forward is exactly hard.
    # Backward uses the soft surrogate.
    mask = hard + (soft - soft.detach())

    return mask, hard

# ------------------------------------------------------------
# 6. TAGS-v3
# ------------------------------------------------------------

class TAGSv3(nn.Module):
    def __init__(
        self, node_dim, edge_dim, controller_dim,
        hidden=32
    ):
        super().__init__()

        self.encoder = TAGSEncoder(
            node_dim, hidden
        )

        self.scorer = nn.Sequential(
            nn.Linear(hidden * 2 + edge_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

        self.controller = nn.Sequential(
            nn.Linear(controller_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

        # Direct trust pathway.
        self.trust_head = nn.Sequential(
            nn.Linear(edge_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

        # GNN correction pathway.
        self.graph_head = nn.Sequential(
            nn.Linear(hidden * 3, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(
        self, xn, xe, xc, u, v, target,
        keep_count, tau=1.0
    ):
        full = xn.new_ones(len(u))

        # Full-graph pass for edge selection.
        h = self.encoder(
            xn, u, v, full
        )

        edge_rep = torch.cat([
            h[u] + h[v],
            torch.abs(h[u] - h[v]),
            xe
        ], dim=1)

        edge_score = self.scorer(
            edge_rep
        ).squeeze(-1)

        priority = self.controller(
            xc
        ).squeeze(-1)

        score = edge_score + 0.5 * (
            priority[u] + priority[v]
        )

        # Shared symmetric decision for each
        # undirected social relationship.
        mask, hard = straight_through_topk(
            score, keep_count, tau
        )

        # Sparse-graph pass.
        z = self.encoder(
            xn, u, v, mask
        )

        a = u[target]
        b = v[target]

        pair = torch.cat([
            z[a] + z[b],
            torch.abs(z[a] - z[b]),
            z[a] * z[b]
        ], dim=1)

        trust_logit = self.trust_head(
            xe[target]
        ).squeeze(-1)

        graph_logit = self.graph_head(
            pair
        ).squeeze(-1)

        return {
            "logits": trust_logit + graph_logit,
            "scores": score,
            "mask": mask,
            "hard_mask": hard,
            "trust_logits": trust_logit,
            "graph_logits": graph_logit
        }

# ------------------------------------------------------------
# 7. Forward pass and task gradient verification
# ------------------------------------------------------------

model_v3 = TAGSv3(
    node_dim=Xn.shape[1],
    edge_dim=Xe.shape[1],
    controller_dim=Xc.shape[1]
).to(DEVICE)

model_v3.train()

result = model_v3(
    Xn, Xe, Xc, U, V, TARGET, K
)

task_loss = F.binary_cross_entropy_with_logits(
    result["logits"], Y
)

model_v3.zero_grad(set_to_none=True)
task_loss.backward()

def grad_norm(module):
    return sum(
        float(p.grad.norm().item())
        for p in module.parameters()
        if p.grad is not None
    )

gradients = {
    "encoder": grad_norm(model_v3.encoder),
    "scorer": grad_norm(model_v3.scorer),
    "controller": grad_norm(model_v3.controller),
    "trust_head": grad_norm(model_v3.trust_head),
    "graph_head": grad_norm(model_v3.graph_head)
}

assert torch.isfinite(task_loss)

assert all(
    np.isfinite(value) for value in gradients.values()
)

# Exact global retention is a forward-pass invariant.
assert int(result["hard_mask"].sum().item()) == K
assert result["mask"].shape == (M,)

# The differentiable mask has exactly the same
# forward values as the hard Top-K mask.
assert torch.equal(
    result["mask"].detach(),
    result["hard_mask"]
)

# ------------------------------------------------------------
# 8. Report, no optimizer step
# ------------------------------------------------------------

report = {
    "experiment": "E3.5",
    "method": "TAGS-v3",
    "scenario": "B_owner_correlated",
    "scenario_seed": 101,
    "model_seed": SEED,
    "snapshot": T,
    "target_count": int(len(Y)),
    "task_loss": float(task_loss.detach().cpu()),
    "total_edges": M,
    "retained_edges": K,
    "retention": float(K / M),
    "task_gradient_norms": gradients,
    "forward_mask_exact_budget": True,
    "optimizer_steps": 0,
    "test_accessed": False,
    "hidden_state_accessed": False,
    "important_limitation": (
        "Straight-through gradients are surrogate "
        "gradients, not exact derivatives of Top-K."
    )
}

path = OUT / "e3_5_smoke_report.json"

with open(path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print("\n" + "=" * 72)
print("E3.5 — TAGS-V3 SMOKE TEST RESULTS")
print("=" * 72)

print("Device:", DEVICE)
print("Target interactions:", len(Y))
print("Task BCE:", round(report["task_loss"], 6))
print("Original edges:", M)
print("Retained edges:", K)
print("Retained fraction:", round(K / M, 6))

print("\nTASK-LOSS GRADIENTS:")
for name, value in gradients.items():
    print(f"  {name:14s}: {value:.10f}")

print("\nE3.5 COMPLETED")
print("Exact-budget selection: PASSED")
print("Gradient values finite: PASSED")
print("No optimizer steps.")
print("No test data accessed.")
print("Report:", path)
print("NEXT: E3.6 — Controlled TAGS-v3 training.")


In [ ]:

# ============================================================
# PHASE III — E3.5b
# TAGS-v3 NUMERICAL STABILITY DIAGNOSTIC
# ============================================================
# Compare original vs stabilized aggregation.
# Identical model weights and identical input tensors.
# No optimizer updates, no test access.
# ============================================================

import copy
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

assert "model_v3" in globals()
assert "TAGSv3" in globals()
assert "TAGSWeightedSAGE" in globals()
assert "straight_through_topk" in globals()

class StableTAGSWeightedSAGE(TAGSWeightedSAGE):

    def forward(self, x, u, v, weight):
        w = weight.reshape(-1, 1)

        msg = torch.zeros_like(x)
        msg = msg.index_add(0, u, x[v] * w)
        msg = msg.index_add(0, v, x[u] * w)

        deg = x.new_zeros((x.size(0), 1))
        deg = deg.index_add(0, u, w)
        deg = deg.index_add(0, v, w)

        # Stabilized weighted aggregation:
        # avoids near-zero denominator gradients.
        agg = msg / (1.0 + deg)

        return (
            self.self_map(x)
            + self.neighbor_map(agg)
        )


class StableTAGSEncoder(nn.Module):

    def __init__(self, input_dim, hidden=32):
        super().__init__()

        self.first = StableTAGSWeightedSAGE(
            input_dim, hidden
        )
        self.second = StableTAGSWeightedSAGE(
            hidden, hidden
        )

    def forward(self, x, u, v, weights):
        h = F.relu(
            self.first(x, u, v, weights)
        )
        return self.second(h, u, v, weights)


class TAGSv3Stable(TAGSv3):

    def __init__(
        self, node_dim, edge_dim,
        controller_dim, hidden=32
    ):
        super().__init__(
            node_dim,
            edge_dim,
            controller_dim,
            hidden
        )

        self.encoder = StableTAGSEncoder(
            node_dim, hidden
        )


stable = TAGSv3Stable(
    node_dim=Xn.shape[1],
    edge_dim=Xe.shape[1],
    controller_dim=Xc.shape[1],
    hidden=32
).to(DEVICE)

# Exactly the same learned parameters.
stable.load_state_dict(
    copy.deepcopy(model_v3.state_dict())
)

def analyze_model(model):
    model.train()
    model.zero_grad(set_to_none=True)

    output = model(
        Xn, Xe, Xc,
        U, V, TARGET, K
    )

    loss = F.binary_cross_entropy_with_logits(
        output["logits"], Y
    )

    loss.backward()

    names = [
        "encoder", "scorer", "controller",
        "trust_head", "graph_head"
    ]

    grad_norms = {}
    grad_maxima = {}

    for name in names:
        module = getattr(model, name)

        tensors = [
            p.grad.detach()
            for p in module.parameters()
            if p.grad is not None
        ]

        grad_norms[name] = float(
            torch.linalg.vector_norm(
                torch.stack([
                    torch.linalg.vector_norm(g)
                    for g in tensors
                ])
            ).item()
        ) if tensors else 0.0

        grad_maxima[name] = float(
            max(
                g.abs().max().item()
                for g in tensors
            )
        ) if tensors else 0.0

    # Count nodes with no retained neighbors.
    with torch.no_grad():
        hard = output["hard_mask"].detach()

        degree = torch.zeros(
            Xn.size(0), device=DEVICE
        )

        degree.index_add_(0, U, hard)
        degree.index_add_(0, V, hard)

        isolated = int(
            (degree == 0).sum().item()
        )

        assert int(hard.sum()) == K

    return {
        "BCE": float(loss.detach().cpu()),
        "isolated_nodes": isolated,
        "gradient_norms": grad_norms,
        "gradient_max_abs": grad_maxima
    }


original_results = analyze_model(model_v3)
stable_results = analyze_model(stable)

print("=" * 72)
print("E3.5b — NUMERICAL STABILITY COMPARISON")
print("=" * 72)

print("\nOriginal BCE:", round(original_results["BCE"], 6))
print("Stable BCE:", round(stable_results["BCE"], 6))
print("Isolated nodes:", stable_results["isolated_nodes"])

print("\nGRADIENT NORM COMPARISON")

for name in original_results["gradient_norms"]:
    before = original_results["gradient_norms"][name]
    after = stable_results["gradient_norms"][name]

    print(
        f"{name:14s} | "
        f"original={before:12.5f} | "
        f"stable={after:12.5f}"
    )

print("\nMAXIMUM ABSOLUTE GRADIENT")

for name in original_results["gradient_max_abs"]:
    before = original_results["gradient_max_abs"][name]
    after = stable_results["gradient_max_abs"][name]

    print(
        f"{name:14s} | "
        f"original={before:12.5f} | "
        f"stable={after:12.5f}"
    )

report = {
    "experiment": "E3.5b",
    "original": original_results,
    "stabilized": stable_results,
    "weight_parameters_identical": True,
    "optimizer_updates": 0,
    "test_accessed": False,
    "scientific_note": (
        "The stabilized aggregation changes "
        "the message-passing normalization."
    )
}

path = OUT / "e3_5b_stability_report.json"

with open(path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print("\nE3.5b COMPLETED")
print("No optimizer updates.")
print("Report:", path)


In [ ]:

# ============================================================
# E3.5c — SPARSIFICATION STRUCTURAL AUDIT
# ============================================================
# Compare:
# 1. Original TAGS-v3 mask
# 2. Stabilized TAGS-v3 mask
# 3. Random Top-K (10 replicates)
# 4. Low-degree-priority Top-K
#
# All at EXACTLY the same edge budget.
# No optimization, no test access.
# ============================================================

import json
import numpy as np
import pandas as pd
import torch

from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

assert "model_v3" in globals()
assert "stable" in globals()
assert "Xn" in globals()
assert "Xe" in globals()
assert "U" in globals()
assert "V" in globals()
assert "TARGET" in globals()
assert "K" in globals()

N_AUDIT = int(Xn.shape[0])
M_AUDIT = int(U.numel())
assert M_AUDIT == 146117

u_np = U.detach().cpu().numpy()
v_np = V.detach().cpu().numpy()

# Historical evidence from t=7, strictly before
# the prediction-time interaction outcomes.
with np.load(
    FEAT / "snapshot_t07.npz"
) as f:
    historical = f["features"].copy()

assert historical.shape == (M_AUDIT, 12)

history_count = historical[:, 0]
history_success_rate = historical[:, 1]
history_confidence = historical[:, 4]

original_degree = np.bincount(
    np.concatenate([u_np, v_np]),
    minlength=N_AUDIT
)

assert np.all(original_degree > 0)

def model_mask(model):
    model.eval()
    with torch.no_grad():
        output = model(
            Xn, Xe, Xc, U, V, TARGET, K
        )
        return (
            output["hard_mask"]
            .detach()
            .cpu()
            .numpy()
            .astype(bool)
        )

original_mask = model_mask(model_v3)
stable_mask = model_mask(stable)

assert original_mask.sum() == K
assert stable_mask.sum() == K

def structural_metrics(mask, name):
    assert int(mask.sum()) == K

    selected_u = u_np[mask]
    selected_v = v_np[mask]

    degree = np.bincount(
        np.concatenate([selected_u, selected_v]),
        minlength=N_AUDIT
    )

    adjacency = coo_matrix(
        (
            np.ones(2 * K, dtype=np.uint8),
            (
                np.concatenate([selected_u, selected_v]),
                np.concatenate([selected_v, selected_u])
            )
        ),
        shape=(N_AUDIT, N_AUDIT)
    ).tocsr()

    n_components, labels = connected_components(
        adjacency,
        directed=False
    )

    component_sizes = np.bincount(labels)

    # Historical estimates are NOT true latent trust.
    previously_seen = history_count[mask] > 0
    sufficient_evidence = history_count[mask] >= 2

    selected_rates = history_success_rate[mask]
    selected_conf = history_confidence[mask]

    return {
        "method": name,
        "retained_edges": int(mask.sum()),
        "isolated_nodes": int((degree == 0).sum()),
        "isolated_fraction": float(
            (degree == 0).mean()
        ),
        "largest_component_fraction": float(
            component_sizes.max() / N_AUDIT
        ),
        "component_count": int(n_components),
        "mean_retained_degree": float(
            degree.mean()
        ),
        "seen_edge_fraction": float(
            previously_seen.mean()
        ),
        "edge_count_ge_2_fraction": float(
            sufficient_evidence.mean()
        ),
        "mean_historical_success_rate": float(
            selected_rates.mean()
        ),
        "mean_historical_confidence": float(
            selected_conf.mean()
        )
    }

rows = [
    structural_metrics(
        original_mask, "Original TAGS"
    ),
    structural_metrics(
        stable_mask, "Stable TAGS"
    )
]

# Random masks: identical budget each time.
for seed in range(101, 111):
    rng = np.random.default_rng(seed)
    ids = rng.choice(
        M_AUDIT, size=K, replace=False
    )

    mask = np.zeros(
        M_AUDIT, dtype=bool
    )
    mask[ids] = True

    rows.append(
        structural_metrics(mask, f"Random {seed}")
    )

# Static topology-only baseline:
# prioritize relationships incident to low-degree
# nodes. This is not an optimal coverage algorithm.
low_degree_score = (
    1.0 / original_degree[u_np]
    + 1.0 / original_degree[v_np]
)

ids = np.argpartition(
    low_degree_score, -K
)[-K:]

low_degree_mask = np.zeros(
    M_AUDIT, dtype=bool
)
low_degree_mask[ids] = True

rows.append(
    structural_metrics(
        low_degree_mask,
        "Low-degree priority"
    )
)

df = pd.DataFrame(rows)

# Aggregate random replicates for readable output.
random_rows = df[
    df["method"].str.startswith("Random ")
]

metric_cols = [
    "isolated_nodes",
    "isolated_fraction",
    "largest_component_fraction",
    "component_count",
    "seen_edge_fraction",
    "edge_count_ge_2_fraction",
    "mean_historical_success_rate",
    "mean_historical_confidence"
]

summary_rows = []

for name in [
    "Original TAGS",
    "Stable TAGS",
    "Low-degree priority"
]:
    summary_rows.append(
        df[df["method"] == name].iloc[0].to_dict()
    )

random_mean = {
    "method": "Random Top-K (mean of 10)",
    "retained_edges": K
}

for column in metric_cols:
    random_mean[column] = float(
        random_rows[column].mean()
    )

random_mean["mean_retained_degree"] = (
    2 * K / N_AUDIT
)

summary_rows.append(random_mean)

summary = pd.DataFrame(summary_rows)

overlap = int(
    np.logical_and(
        original_mask, stable_mask
    ).sum()
)

jaccard = overlap / (
    2 * K - overlap
)

print("=" * 78)
print("E3.5c — SPARSIFICATION STRUCTURAL AUDIT")
print("=" * 78)
print("Original nodes:", N_AUDIT)
print("Original edges:", M_AUDIT)
print("Retained edges:", K)
print(
    "Original vs Stable mask Jaccard:",
    round(jaccard, 6)
)

print("\nSTRUCTURAL COMPARISON")
print(summary[[
    "method",
    "isolated_nodes",
    "largest_component_fraction",
    "component_count"
]].round(5).to_string(index=False))

print("\nHISTORICAL TRUST-EVIDENCE COMPARISON")
print(summary[[
    "method",
    "seen_edge_fraction",
    "edge_count_ge_2_fraction",
    "mean_historical_success_rate",
    "mean_historical_confidence"
]].round(6).to_string(index=False))

# Save only new E3.5c artifacts.
output_csv = OUT / "e3_5c_structural_audit.csv"
output_json = OUT / "e3_5c_structural_report.json"

assert not output_csv.exists()
assert not output_json.exists()

df.to_csv(output_csv, index=False)

report = {
    "experiment": "E3.5c",
    "scenario": "B_owner_correlated",
    "scenario_seed": 101,
    "snapshot": 7,
    "retained_edge_budget": K,
    "random_replicates": 10,
    "original_vs_stable_mask_jaccard": float(
        jaccard
    ),
    "summary": summary.to_dict(
        orient="records"
    ),
    "no_model_training": True,
    "no_test_access": True,
    "no_hidden_state_access": True,
    "important_limitations": [
        "Models are untrained.",
        "Historical success estimates are not "
        "ground-truth trust.",
        "Low-degree priority does not guarantee "
        "optimal node coverage.",
        "Structural quality alone does not "
        "establish predictive utility."
    ]
}

with open(
    output_json, "w", encoding="utf-8"
) as f:
    json.dump(report, f, indent=2)

print("\nE3.5c COMPLETED")
print("No optimization performed.")
print("No test data accessed.")
print("Saved:", output_json)


In [ ]:

# ============================================================
# PHASE III — E3.5d
# EXACT-BUDGET EDGE SELECTION WITH NODE COVERAGE
# ============================================================
# Diagnostic only — NO TRAINING.
# Reuses the untrained stabilized TAGS-v3 scorer.
# No test or hidden simulator data accessed.
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch

from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

assert "stable" in globals()
assert "Xn" in globals()
assert "Xe" in globals()
assert "Xc" in globals()
assert "U" in globals()
assert "V" in globals()
assert "TARGET" in globals()
assert "K" in globals()

n_nodes = int(Xn.shape[0])
n_edges = int(U.numel())

u = U.detach().cpu().numpy()
v = V.detach().cpu().numpy()

# ------------------------------------------------------------
# 1. Obtain scores from unchanged stabilized model
# ------------------------------------------------------------

stable.eval()

with torch.no_grad():
    output = stable(
        Xn, Xe, Xc, U, V, TARGET, K
    )

    scores = (
        output["scores"]
        .detach()
        .cpu()
        .numpy()
        .astype(np.float64)
    )

    original_mask = (
        output["hard_mask"]
        .detach()
        .cpu()
        .numpy()
        .astype(bool)
    )

assert np.isfinite(scores).all()
assert int(original_mask.sum()) == K

original_degree = np.bincount(
    np.r_[u, v],
    minlength=n_nodes
)

assert np.all(original_degree > 0)

# ------------------------------------------------------------
# 2. Greedy score-aware edge cover
# ------------------------------------------------------------
# Phase 1: select an edge if it covers at least
# one currently uncovered device.
#
# Phase 2: use remaining budget for highest-scored
# relationships not already selected.
#
# This guarantees node coverage, not connectivity
# or optimal weighted edge cover.
# ------------------------------------------------------------

order = np.argsort(
    -scores, kind="stable"
)

covered = np.zeros(
    n_nodes, dtype=bool
)

selected = np.zeros(
    n_edges, dtype=bool
)

coverage_edges = 0

for edge_id in order:

    a = u[edge_id]
    b = v[edge_id]

    if not (covered[a] and covered[b]):
        selected[edge_id] = True
        covered[a] = True
        covered[b] = True
        coverage_edges += 1

    if covered.all():
        break

assert covered.all()

if coverage_edges > K:
    raise RuntimeError(
        "Coverage requires more edges than budget."
    )

remaining = K - coverage_edges

if remaining:
    eligible = order[~selected[order]]
    selected[eligible[:remaining]] = True

assert selected.sum() == K

# ------------------------------------------------------------
# 3. Structural and historical audits
# ------------------------------------------------------------

with np.load(
    FEAT / "snapshot_t07.npz"
) as f:
    historical = f["features"]

def metrics(mask):
    a = u[mask]
    b = v[mask]

    deg = np.bincount(
        np.r_[a, b],
        minlength=n_nodes
    )

    adjacency = coo_matrix(
        (
            np.ones(2 * len(a), dtype=np.uint8),
            (np.r_[a, b], np.r_[b, a])
        ),
        shape=(n_nodes, n_nodes)
    ).tocsr()

    count, labels = connected_components(
        adjacency, directed=False
    )

    sizes = np.bincount(labels)

    return {
        "retained_edges": int(mask.sum()),
        "isolated_nodes": int((deg == 0).sum()),
        "minimum_degree": int(deg.min()),
        "largest_component_fraction": float(
            sizes.max() / n_nodes
        ),
        "component_count": int(count),
        "seen_edge_fraction": float(
            (historical[mask, 0] > 0).mean()
        ),
        "mean_historical_success_rate": float(
            historical[mask, 1].mean()
        ),
        "mean_historical_confidence": float(
            historical[mask, 4].mean()
        ),
        "mean_selection_score": float(
            scores[mask].mean()
        )
    }

before = metrics(original_mask)
after = metrics(selected)

intersection = int(
    np.logical_and(original_mask, selected).sum()
)

jaccard = intersection / (
    int(original_mask.sum())
    + int(selected.sum())
    - intersection
)

# ------------------------------------------------------------
# 4. Report
# ------------------------------------------------------------

comparison = pd.DataFrame([
    {"method": "Unconstrained TAGS", **before},
    {"method": "Coverage-aware TAGS", **after}
])

print("=" * 74)
print("E3.5d — COVERAGE-CONSTRAINED SPARSIFICATION")
print("=" * 74)

print("Total nodes:", n_nodes)
print("Total edges:", n_edges)
print("Exact edge budget:", K)
print("Coverage-phase edges:", coverage_edges)
print("Remaining score-based edges:", remaining)
print("Mask Jaccard:", round(jaccard, 6))

print("\nSTRUCTURAL AND TRUST-EVIDENCE RESULTS")
print(
    comparison.round(6).to_string(index=False)
)

assert after["isolated_nodes"] == 0
assert after["minimum_degree"] >= 1
assert after["retained_edges"] == K

# ------------------------------------------------------------
# 5. Save without overwriting existing files
# ------------------------------------------------------------

output_dir = OUT / "e3_5d_coverage_audit"

if output_dir.exists():
    raise FileExistsError(
        f"Output already exists: {output_dir}"
    )

output_dir.mkdir(parents=True)

comparison.to_csv(
    output_dir / "comparison.csv",
    index=False
)

np.savez_compressed(
    output_dir / "selection_masks.npz",
    unconstrained=original_mask,
    coverage_aware=selected
)

report = {
    "experiment": "E3.5d",
    "scenario": "B_owner_correlated",
    "scenario_seed": 101,
    "snapshot": 7,
    "coverage_edges": int(coverage_edges),
    "remaining_score_based_edges": int(remaining),
    "mask_jaccard": float(jaccard),
    "unconstrained": before,
    "coverage_aware": after,
    "node_coverage_guaranteed": True,
    "global_budget_exact": True,
    "training_performed": False,
    "test_accessed": False,
    "hidden_state_accessed": False,
    "limitation": (
        "Greedy constrained selection is discrete. "
        "A compatible training-gradient surrogate "
        "has not yet been established."
    )
}

report_path = output_dir / "report.json"

with open(
    report_path, "w", encoding="utf-8"
) as f:
    json.dump(report, f, indent=2)

print("\nE3.5d COMPLETED")
print("Node coverage: GUARANTEED")
print("Exact edge budget: PASSED")
print("No training performed.")
print("No test data accessed.")
print("Report:", report_path)


In [ ]:

# ============================================================
# PHASE III — E3.6a
# COVERAGE-CONSTRAINED TAGS-v3 TRAINING PILOT
# ============================================================
# 8 optimizer steps on training snapshot t=7.
# Stable message passing + exact edge budget
# + full node coverage.
#
# No validation-driven model selection.
# No test access or hidden simulator state.
# ============================================================

from pathlib import Path
import copy
import json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

assert "stable" in globals()
assert "Xn" in globals()
assert "Xe" in globals()
assert "Xc" in globals()
assert "Y" in globals()
assert "U" in globals()
assert "V" in globals()
assert "TARGET" in globals()
assert "K" in globals()
assert "OUT" in globals()

STEPS = 8
LR = 1e-3
GRAD_CLIP = 1.0
TAU = 1.0

n_nodes = int(Xn.shape[0])
n_edges = int(U.numel())

u_cpu = U.detach().cpu().numpy()
v_cpu = V.detach().cpu().numpy()

torch.manual_seed(42)

# Copy the same untrained stable architecture.
pilot = copy.deepcopy(stable).to(Xn.device)

optimizer = torch.optim.AdamW(
    pilot.parameters(),
    lr=LR,
    weight_decay=1e-4
)

# ------------------------------------------------------------
# 1. Exact-budget, score-aware node-cover selection
# ------------------------------------------------------------

def coverage_topk(scores, k):
    order = np.argsort(
        -scores.detach().cpu().numpy(),
        kind="stable"
    )

    selected = np.zeros(
        n_edges, dtype=np.bool_
    )

    covered = np.zeros(
        n_nodes, dtype=np.bool_
    )

    coverage_count = 0
    uncovered = n_nodes

    for edge_id in order:
        a = u_cpu[edge_id]
        b = v_cpu[edge_id]

        if not (covered[a] and covered[b]):
            selected[edge_id] = True
            coverage_count += 1

            if not covered[a]:
                covered[a] = True
                uncovered -= 1

            if not covered[b]:
                covered[b] = True
                uncovered -= 1

        if uncovered == 0:
            break

    if uncovered != 0:
        raise RuntimeError(
            "Coverage infeasible: original graph "
            "may contain isolated nodes."
        )

    if coverage_count > k:
        raise RuntimeError(
            f"Coverage requires {coverage_count} "
            f"edges, exceeding budget {k}."
        )

    remaining = k - coverage_count
    available = order[~selected[order]]

    selected[available[:remaining]] = True

    assert int(selected.sum()) == k

    hard = torch.from_numpy(
        selected.astype(np.float32)
    ).to(scores.device)

    return hard, coverage_count


# ------------------------------------------------------------
# 2. Coverage-aware TAGS forward pass
# ------------------------------------------------------------

def pilot_forward(model):

    full_weights = Xn.new_ones(n_edges)

    # Full graph used to produce edge selection scores.
    h = model.encoder(
        Xn, U, V, full_weights
    )

    edge_repr = torch.cat([
        h[U] + h[V],
        torch.abs(h[U] - h[V]),
        Xe
    ], dim=1)

    edge_score = model.scorer(
        edge_repr
    ).squeeze(-1)

    priority = model.controller(
        Xc
    ).squeeze(-1)

    scores = (
        edge_score +
        0.5 * (priority[U] + priority[V])
    )

    # Discrete forward selection.
    hard, coverage_count = coverage_topk(
        scores, K
    )

    # Surrogate gradient; not the exact
    # derivative of constrained Top-K.
    threshold = torch.topk(
        scores.detach(), K
    ).values[-1]

    soft = torch.sigmoid(
        (scores - threshold) / TAU
    )

    mask = hard + (soft - soft.detach())

    assert torch.equal(
        mask.detach(), hard
    )

    # Second graph pass uses the selected edges.
    z = model.encoder(
        Xn, U, V, mask
    )

    a = U[TARGET]
    b = V[TARGET]

    pair = torch.cat([
        z[a] + z[b],
        torch.abs(z[a] - z[b]),
        z[a] * z[b]
    ], dim=1)

    trust_logits = model.trust_head(
        Xe[TARGET]
    ).squeeze(-1)

    graph_logits = model.graph_head(
        pair
    ).squeeze(-1)

    logits = trust_logits + graph_logits

    return {
        "logits": logits,
        "hard": hard,
        "scores": scores,
        "coverage_edges": coverage_count
    }


# ------------------------------------------------------------
# 3. Training diagnostics
# ------------------------------------------------------------

rows = []
initial_mask = None
previous_mask = None
initial_scores = None
initial_parameters = {
    name: p.detach().clone()
    for name, p in pilot.named_parameters()
}

for step in range(STEPS + 1):

    pilot.train()
    pilot.zero_grad(set_to_none=True)

    output = pilot_forward(pilot)

    loss = F.binary_cross_entropy_with_logits(
        output["logits"], Y
    )

    assert torch.isfinite(loss)

    loss.backward()

    # Total gradient norm before clipping.
    total_grad = float(
        torch.nn.utils.clip_grad_norm_(
            pilot.parameters(), GRAD_CLIP
        ).item()
    )

    hard_cpu = (
        output["hard"]
        .detach()
        .cpu()
        .numpy()
        .astype(bool)
    )

    score_cpu = (
        output["scores"]
        .detach()
        .cpu()
        .numpy()
    )

    if step == 0:
        initial_mask = hard_cpu.copy()
        initial_scores = score_cpu.copy()

    changed_initial = int(
        np.count_nonzero(
            hard_cpu != initial_mask
        )
    )

    changed_previous = (
        0 if previous_mask is None
        else int(np.count_nonzero(
            hard_cpu != previous_mask
        ))
    )

    # Confirm that every device has a retained edge.
    selected_degree = np.bincount(
        np.r_[
            u_cpu[hard_cpu],
            v_cpu[hard_cpu]
        ],
        minlength=n_nodes
    )

    isolated = int(
        (selected_degree == 0).sum()
    )

    assert isolated == 0
    assert int(hard_cpu.sum()) == K

    def module_grad(name):
        module = getattr(pilot, name)
        return float(sum(
            p.grad.detach().norm().item()
            for p in module.parameters()
            if p.grad is not None
        ))

    record = {
        "step": step,
        "training_BCE": float(loss.item()),
        "gradient_norm_before_clip": total_grad,
        "scorer_grad": module_grad("scorer"),
        "controller_grad": module_grad("controller"),
        "coverage_edges": int(
            output["coverage_edges"]
        ),
        "isolated_nodes": isolated,
        "mask_changes_from_start": changed_initial,
        "mask_changes_from_previous": changed_previous,
        "score_std": float(
            np.std(score_cpu)
        ),
        "score_change_mean_abs": float(
            np.mean(
                np.abs(score_cpu - initial_scores)
            )
        )
    }

    rows.append(record)

    print(
        f"step={step:02d} | "
        f"BCE={record['training_BCE']:.6f} | "
        f"grad={total_grad:.4f} | "
        f"coverage={record['coverage_edges']} | "
        f"isolates={isolated} | "
        f"mask_changed={changed_initial}",
        flush=True
    )

    previous_mask = hard_cpu.copy()

    # Step zero is the untrained starting point.
    # Do not update after the final diagnostic.
    if step < STEPS:
        optimizer.step()


# ------------------------------------------------------------
# 4. Verify actual parameter changes
# ------------------------------------------------------------

parameter_change = {}

for name, p in pilot.named_parameters():
    parameter_change[name] = float(
        (p.detach() - initial_parameters[name])
        .abs().max().item()
    )

scorer_changed = any(
    value > 0
    for name, value in parameter_change.items()
    if name.startswith("scorer.")
)

controller_changed = any(
    value > 0
    for name, value in parameter_change.items()
    if name.startswith("controller.")
)

assert scorer_changed
assert controller_changed

df = pd.DataFrame(rows)

# ------------------------------------------------------------
# 5. Save a separate pilot artifact
# ------------------------------------------------------------

PILOT_OUT = OUT / "e3_6a_training_pilot"

if PILOT_OUT.exists():
    raise FileExistsError(
        f"Pilot output already exists: {PILOT_OUT}"
    )

PILOT_OUT.mkdir(parents=True)

df.to_csv(
    PILOT_OUT / "training_diagnostics.csv",
    index=False
)

report = {
    "experiment": "E3.6a",
    "scenario": "B_owner_correlated",
    "scenario_seed": 101,
    "model_seed": 42,
    "training_time": 7,
    "optimizer_steps": STEPS,
    "learning_rate": LR,
    "gradient_clip": GRAD_CLIP,
    "surrogate_temperature": TAU,
    "retained_edges": K,
    "initial_BCE": float(
        df["training_BCE"].iloc[0]
    ),
    "final_BCE": float(
        df["training_BCE"].iloc[-1]
    ),
    "final_mask_changes_from_initial": int(
        df["mask_changes_from_start"].iloc[-1]
    ),
    "scorer_parameters_changed": scorer_changed,
    "controller_parameters_changed": controller_changed,
    "exact_budget_verified": True,
    "zero_isolates_verified": True,
    "validation_accessed": False,
    "test_accessed": False,
    "hidden_state_accessed": False,
    "limitations": [
        "Training diagnostic on a single time step.",
        "No generalization performance established.",
        "Coverage selection is discrete.",
        "Straight-through gradient is a surrogate.",
        "A direct trust head may dominate predictions.",
        "Temporal adaptivity has not been tested."
    ]
}

with open(
    PILOT_OUT / "report.json",
    "w", encoding="utf-8"
) as f:
    json.dump(report, f, indent=2)

torch.save(
    {
        "model_state_dict": pilot.state_dict(),
        "training_steps": STEPS,
        "scenario_seed": 101,
        "model_seed": 42
    },
    PILOT_OUT / "pilot_checkpoint.pt"
)

print("\n" + "=" * 72)
print("E3.6a COMPLETED — TRAINING PILOT")
print("=" * 72)

print(
    "Initial training BCE:",
    f"{report['initial_BCE']:.6f}"
)

print(
    "Final training BCE:",
    f"{report['final_BCE']:.6f}"
)

print(
    "Final mask changes:",
    report["final_mask_changes_from_initial"]
)

print(
    "Scorer weights changed:",
    scorer_changed
)

print(
    "Controller weights changed:",
    controller_changed
)

print(
    "Maximum pre-clipping gradient norm:",
    round(
        float(
            df["gradient_norm_before_clip"].max()
        ), 6
    )
)

print("Coverage invariant: PASSED")
print("Exact budget invariant: PASSED")
print("Test accessed: NO")
print("Report:", PILOT_OUT / "report.json")


In [ ]:

# ============================================================
# PHASE III — E3.6b
# DOES LEARNED SPARSIFICATION AFFECT PREDICTIONS?
# ============================================================
# Frozen trained pilot; training t=7 only.
# Compare learned, initial, random-coverage and full
# graph masks under identical trained model weights.
# No optimizer updates or validation/test access.
# ============================================================

import json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    log_loss
)

assert "pilot" in globals()
assert "pilot_forward" in globals()
assert "coverage_topk" in globals()
assert "Xn" in globals() and "Xe" in globals()
assert "Xc" in globals() and "Y" in globals()
assert "U" in globals() and "V" in globals()
assert "TARGET" in globals() and "K" in globals()
assert "OUT" in globals() and "FEAT" in globals()

pilot.eval()
n = int(Xn.shape[0])
m = int(U.numel())

u_np = U.detach().cpu().numpy()
v_np = V.detach().cpu().numpy()

# ------------------------------------------------------------
# 1. Recover trained model's learned mask
# ------------------------------------------------------------

with torch.no_grad():
    learned_output = pilot_forward(pilot)
    learned_mask = (
        learned_output["hard"]
        .detach()
        .cpu()
        .numpy()
        .astype(bool)
    )

# Initial untrained coverage mask saved in E3.5d.
initial_path = (
    OUT / "e3_5d_coverage_audit" /
    "selection_masks.npz"
)

with np.load(initial_path) as f:
    initial_mask = f["coverage_aware"].astype(bool)

assert initial_mask.shape == (m,)
assert learned_mask.shape == (m,)
assert int(initial_mask.sum()) == K
assert int(learned_mask.sum()) == K

# ------------------------------------------------------------
# 2. Generate random mask WITH exact node coverage
# ------------------------------------------------------------

def random_coverage_mask(seed):
    rng = np.random.default_rng(seed)
    order = rng.permutation(m)

    selected = np.zeros(m, dtype=bool)
    covered = np.zeros(n, dtype=bool)
    uncovered = n

    for eid in order:
        a = u_np[eid]
        b = v_np[eid]

        if not (covered[a] and covered[b]):
            selected[eid] = True

            if not covered[a]:
                covered[a] = True
                uncovered -= 1

            if not covered[b]:
                covered[b] = True
                uncovered -= 1

        if uncovered == 0:
            break

    assert uncovered == 0

    used = int(selected.sum())
    assert used <= K

    candidates = order[~selected[order]]
    selected[candidates[:K-used]] = True

    assert int(selected.sum()) == K

    return selected

random_masks = [
    random_coverage_mask(seed)
    for seed in [201, 202, 203]
]

# ------------------------------------------------------------
# 3. Check all masks and calculate predictions
# ------------------------------------------------------------

y = Y.detach().cpu().numpy()

def metrics(probabilities):
    p = np.clip(
        probabilities, 1e-7, 1 - 1e-7
    )

    return {
        "AP": float(
            average_precision_score(y, p)
        ),
        "AUC": float(
            roc_auc_score(y, p)
        ),
        "BCE": float(
            log_loss(y, p, labels=[0, 1])
        )
    }

with torch.no_grad():

    # Direct trust route (no graph representation).
    trust_logits = pilot.trust_head(
        Xe[TARGET]
    ).squeeze(-1)

    trust_prob = torch.sigmoid(
        trust_logits
    ).cpu().numpy()

    def evaluate_mask(name, mask, full=False):

        assert len(mask) == m

        if not full:
            assert int(mask.sum()) == K

            deg = np.bincount(
                np.r_[u_np[mask], v_np[mask]],
                minlength=n
            )

            assert deg.min() >= 1

        weights = torch.as_tensor(
            mask.astype(np.float32),
            device=Xn.device
        )

        z = pilot.encoder(
            Xn, U, V, weights
        )

        a = U[TARGET]
        b = V[TARGET]

        pair = torch.cat([
            z[a] + z[b],
            torch.abs(z[a] - z[b]),
            z[a] * z[b]
        ], dim=1)

        graph_logits = pilot.graph_head(
            pair
        ).squeeze(-1)

        combined = trust_logits + graph_logits

        probabilities = torch.sigmoid(
            combined
        ).cpu().numpy()

        return {
            "method": name,
            "retained_edges": int(mask.sum()),
            **metrics(probabilities),
            "mean_abs_graph_logit": float(
                graph_logits.abs().mean().item()
            ),
            "mean_abs_trust_logit": float(
                trust_logits.abs().mean().item()
            )
        }

    rows = [{
        "method": "Trust-only",
        "retained_edges": 0,
        **metrics(trust_prob),
        "mean_abs_graph_logit": 0.0,
        "mean_abs_trust_logit": float(
            trust_logits.abs().mean().item()
        )
    }]

    rows.append(
        evaluate_mask(
            "Learned TAGS", learned_mask
        )
    )

    rows.append(
        evaluate_mask(
            "Initial mask", initial_mask
        )
    )

    for i, mask in enumerate(random_masks):
        rows.append(
            evaluate_mask(
                f"Random coverage {i+1}", mask
            )
        )

    rows.append(
        evaluate_mask(
            "Full graph",
            np.ones(m, dtype=bool),
            full=True
        )
    )

# ------------------------------------------------------------
# 4. Measure task-gradient strength at final checkpoint
# ------------------------------------------------------------

pilot.train()
pilot.zero_grad(set_to_none=True)

output = pilot_forward(pilot)

loss = F.binary_cross_entropy_with_logits(
    output["logits"], Y
)
loss.backward()

def gradient_norm(module):
    grads = [
        p.grad.detach().norm()
        for p in module.parameters()
        if p.grad is not None
    ]

    if not grads:
        return 0.0

    return float(
        torch.linalg.vector_norm(
            torch.stack(grads)
        ).item()
    )

gradients = {
    name: gradient_norm(getattr(pilot, name))
    for name in [
        "encoder",
        "scorer",
        "controller",
        "trust_head",
        "graph_head"
    ]
}

# ------------------------------------------------------------
# 5. Report
# ------------------------------------------------------------

df = pd.DataFrame(rows)

random_rows = df[
    df["method"].str.startswith(
        "Random coverage "
    )
]

learned = df[
    df["method"] == "Learned TAGS"
].iloc[0]

random_ap = float(
    random_rows["AP"].mean()
)

random_bce = float(
    random_rows["BCE"].mean()
)

changes = int(
    np.count_nonzero(
        learned_mask != initial_mask
    )
)

print("=" * 76)
print("E3.6b — LEARNED SPARSIFICATION ATTRIBUTION")
print("=" * 76)

print("Training snapshot: t=7")
print("Positive class: interaction failure")
print("Failure prevalence:", round(y.mean(), 6))
print("Budget:", K)
print("Changed mask entries:", changes)

print("\nFROZEN-MODEL PERFORMANCE")
print(
    df[[
        "method",
        "retained_edges",
        "AP",
        "AUC",
        "BCE"
    ]].round(6).to_string(index=False)
)

print("\nCOMPARISONS")
print(
    "Learned minus random mean AP:",
    f"{learned['AP'] - random_ap:+.6f}"
)
print(
    "Learned minus random mean BCE:",
    f"{learned['BCE'] - random_bce:+.6f}"
)

print("\nTASK GRADIENTS AT TRAINED CHECKPOINT")
for name, value in gradients.items():
    print(f"{name:14s}: {value:.8f}")

# ------------------------------------------------------------
# 6. Save only new experiment files
# ------------------------------------------------------------

AUDIT = OUT / "e3_6b_attribution"

if AUDIT.exists():
    raise FileExistsError(
        f"Output already exists: {AUDIT}"
    )

AUDIT.mkdir(parents=True)

df.to_csv(
    AUDIT / "mask_comparison.csv",
    index=False
)

report = {
    "experiment": "E3.6b",
    "scenario": "B_owner_correlated",
    "scenario_seed": 101,
    "model_seed": 42,
    "training_snapshot": 7,
    "model_frozen_during_comparison": True,
    "retention_budget": K,
    "mask_entry_changes_vs_initial": changes,
    "results": rows,
    "final_task_gradient_norms": gradients,
    "learned_minus_random_mean_AP": float(
        learned["AP"] - random_ap
    ),
    "learned_minus_random_mean_BCE": float(
        learned["BCE"] - random_bce
    ),
    "validation_accessed": False,
    "test_accessed": False,
    "hidden_state_accessed": False,
    "limitations": [
        "Training-set diagnostic, not generalization.",
        "Random masks were not independently retrained.",
        "Initial mask comes from an untrained model.",
        "Trust-only uses the trained trust head.",
        "Mask sensitivity is not causal attribution "
        "of the training improvement.",
        "Three random masks are exploratory."
    ]
}

with open(
    AUDIT / "report.json",
    "w", encoding="utf-8"
) as f:
    json.dump(report, f, indent=2)

print("\nE3.6b COMPLETED")
print("No optimizer updates.")
print("No validation or test accessed.")
print("Report:", AUDIT / "report.json")


In [ ]:

# ============================================================
# PHASE III — E3.6c
# TEMPORAL TRAINING + VALIDATION OF TAGS-v3
# ============================================================
# Train times: 5, 6, 7
# Validation times: 8, 9
# Model seed: 42 (exploratory pilot)
# Stable GraphSAGE + coverage-constrained Top-K
# No test or hidden simulator access
# ============================================================

from pathlib import Path
import copy
import json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    log_loss
)

assert "stable" in globals()
assert "coverage_topk" in globals()
assert "historical_features" in globals()
assert "edge_scaler" in globals()
assert "node_scaler" in globals()
assert "type_onehot" in globals()
assert "degree" in globals()
assert "U" in globals() and "V" in globals()
assert "FEAT" in globals() and "OBS" in globals()
assert "OUT" in globals()

TRAIN_TIMES = [5, 6, 7]
VAL_TIMES = [8, 9]

SEED = 42
MAX_EPOCHS = 15
PATIENCE = 5
LR = 1e-3
TAU = 1.0
GRAD_CLIP = 1.0

torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = Xn.device
N = int(Xn.shape[0])
M = int(U.numel())

# ------------------------------------------------------------
# 1. Load only training and validation observations
# ------------------------------------------------------------

train_events = pd.read_csv(
    OBS / "train.csv.gz"
)

val_events = pd.read_csv(
    OBS / "validation.csv.gz"
)

assert train_events["time"].between(0, 7).all()
assert val_events["time"].between(8, 9).all()

available = pd.concat(
    [train_events, val_events],
    ignore_index=True
)

edge_u = U.detach().cpu().numpy()
edge_v = V.detach().cpu().numpy()
edge_keys = edge_u * N + edge_v

assert np.all(np.diff(edge_keys) > 0)

# ------------------------------------------------------------
# 2. Causal node histories for each snapshot
# ------------------------------------------------------------

def temporal_node_history(t):
    # Strictly previous events only.
    past = available[available["time"] < t]

    a = past["source"].to_numpy(np.int64) - 1
    b = past["target"].to_numpy(np.int64) - 1
    s = past["success"].to_numpy(np.float32)

    count = np.bincount(
        np.r_[a, b],
        minlength=N
    ).astype(np.float32)

    good = np.bincount(
        np.r_[a, b],
        weights=np.r_[s, s],
        minlength=N
    ).astype(np.float32)

    rate = (good + 1) / (count + 2)
    conf = count / (count + 5)

    return np.column_stack([
        np.log1p(degree),
        np.log1p(count),
        rate,
        conf
    ]).astype(np.float32)

def prepare_time(t):
    edge_x = edge_scaler.transform(
        historical_features(t)
    ).astype(np.float32)

    node_dynamic = node_scaler.transform(
        temporal_node_history(t)
    ).astype(np.float32)

    node_x = np.concatenate([
        type_onehot,
        node_dynamic
    ], axis=1).astype(np.float32)

    frame = (
        train_events if t <= 7
        else val_events
    )

    current = frame[frame["time"] == t]

    a = (
        current["source"].to_numpy(np.int64) - 1
    )
    b = (
        current["target"].to_numpy(np.int64) - 1
    )

    keys = a * N + b
    ids = np.searchsorted(edge_keys, keys)

    assert np.all(ids < M)
    assert np.array_equal(edge_keys[ids], keys)

    y = (
        1 - current["success"].to_numpy(np.float32)
    )

    def tensor(x):
        return torch.as_tensor(
            x, dtype=torch.float32, device=DEVICE
        )

    return {
        "time": t,
        "xn": tensor(node_x),
        "xe": tensor(edge_x),
        "xc": tensor(node_dynamic),
        "target": torch.as_tensor(
            ids, dtype=torch.long, device=DEVICE
        ),
        "y": tensor(y),
        "count": len(y)
    }

data = {
    t: prepare_time(t)
    for t in TRAIN_TIMES + VAL_TIMES
}

assert all(
    torch.isfinite(data[t]["xn"]).all()
    and torch.isfinite(data[t]["xe"]).all()
    for t in data
)

print("=" * 72)
print("E3.6c — TEMPORAL TAGS-v3 TRAINING")
print("=" * 72)
print("Train times:", TRAIN_TIMES)
print("Validation times:", VAL_TIMES)
print("Seed:", SEED)
print("Maximum epochs:", MAX_EPOCHS)
print("Train targets:", sum(
    data[t]["count"] for t in TRAIN_TIMES
))
print("Validation targets:", sum(
    data[t]["count"] for t in VAL_TIMES
))

# ------------------------------------------------------------
# 3. Coverage-constrained forward
# ------------------------------------------------------------

def temporal_forward(model, batch):
    xn = batch["xn"]
    xe = batch["xe"]
    xc = batch["xc"]
    target = batch["target"]

    full = xn.new_ones(M)

    h = model.encoder(
        xn, U, V, full
    )

    representation = torch.cat([
        h[U] + h[V],
        torch.abs(h[U] - h[V]),
        xe
    ], dim=1)

    edge_score = model.scorer(
        representation
    ).squeeze(-1)

    priority = model.controller(
        xc
    ).squeeze(-1)

    scores = edge_score + 0.5 * (
        priority[U] + priority[V]
    )

    hard, coverage_count = coverage_topk(
        scores, K
    )

    threshold = torch.topk(
        scores.detach(), K
    ).values[-1]

    soft = torch.sigmoid(
        (scores - threshold) / TAU
    )

    mask = hard + (soft - soft.detach())

    z = model.encoder(
        xn, U, V, mask
    )

    a = U[target]
    b = V[target]

    pair = torch.cat([
        z[a] + z[b],
        torch.abs(z[a] - z[b]),
        z[a] * z[b]
    ], dim=1)

    trust_logits = model.trust_head(
        xe[target]
    ).squeeze(-1)

    graph_logits = model.graph_head(
        pair
    ).squeeze(-1)

    return {
        "logits": trust_logits + graph_logits,
        "hard": hard,
        "scores": scores,
        "coverage_edges": coverage_count
    }

# ------------------------------------------------------------
# 4. Validation — no gradient or parameter updates
# ------------------------------------------------------------

def validate(model):
    model.eval()
    ys = []
    ps = []
    masks = []

    with torch.no_grad():
        for t in VAL_TIMES:
            batch = data[t]
            result = temporal_forward(
                model, batch
            )

            ys.append(
                batch["y"].cpu().numpy()
            )

            ps.append(
                torch.sigmoid(
                    result["logits"]
                ).cpu().numpy()
            )

            masks.append(
                result["hard"]
                .cpu()
                .numpy()
                .astype(bool)
            )

    y = np.concatenate(ys)
    p = np.clip(
        np.concatenate(ps),
        1e-7,
        1 - 1e-7
    )

    churn = float(
        np.count_nonzero(
            masks[0] != masks[1]
        ) / (2 * K)
    )

    return {
        "AP": float(
            average_precision_score(y, p)
        ),
        "AUC": float(
            roc_auc_score(y, p)
        ),
        "BCE": float(
            log_loss(y, p, labels=[0, 1])
        ),
        "mask_churn": churn
    }

# ------------------------------------------------------------
# 5. Train from UNTRAINED E3.5b initialization
# ------------------------------------------------------------
# Do not reuse the t=7-trained E3.6a pilot.

model = copy.deepcopy(stable).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4
)

best_ap = -np.inf
best_epoch = -1
best_metrics = None
best_state = None
patience_counter = 0
history = []

# Include an untrained reference.
initial_val = validate(model)

print(
    "\nEpoch 00 (untrained) | "
    f"Val AP={initial_val['AP']:.6f} | "
    f"AUC={initial_val['AUC']:.6f} | "
    f"BCE={initial_val['BCE']:.6f}"
)

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    losses = []
    grad_norms = []

    for t in TRAIN_TIMES:
        batch = data[t]

        optimizer.zero_grad(
            set_to_none=True
        )

        result = temporal_forward(
            model, batch
        )

        loss = F.binary_cross_entropy_with_logits(
            result["logits"],
            batch["y"]
        )

        assert torch.isfinite(loss)

        loss.backward()

        norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            GRAD_CLIP
        )

        grad_norms.append(float(norm.item()))

        optimizer.step()

        losses.append((
            float(loss.item()),
            batch["count"]
        ))

    weighted_train_bce = (
        sum(loss * count for loss, count in losses)
        / sum(count for _, count in losses)
    )

    val = validate(model)

    history.append({
        "epoch": epoch,
        "train_BCE": weighted_train_bce,
        "val_AP": val["AP"],
        "val_AUC": val["AUC"],
        "val_BCE": val["BCE"],
        "val_mask_churn": val["mask_churn"],
        "max_preclip_gradient": max(grad_norms)
    })

    print(
        f"epoch={epoch:02d} | "
        f"Train BCE={weighted_train_bce:.6f} | "
        f"Val AP={val['AP']:.6f} | "
        f"AUC={val['AUC']:.6f} | "
        f"BCE={val['BCE']:.6f} | "
        f"churn={val['mask_churn']:.4f}",
        flush=True
    )

    if val["AP"] > best_ap + 1e-5:
        best_ap = val["AP"]
        best_epoch = epoch
        best_metrics = dict(val)
        best_state = copy.deepcopy(
            model.state_dict()
        )
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= PATIENCE:
        break

assert best_state is not None

# ------------------------------------------------------------
# 6. Save pilot without modifying earlier artifacts
# ------------------------------------------------------------

SAVE = OUT / "e3_6c_temporal_training"

if SAVE.exists():
    raise FileExistsError(
        f"Output directory exists: {SAVE}"
    )

SAVE.mkdir(parents=True)

pd.DataFrame(history).to_csv(
    SAVE / "epoch_history.csv",
    index=False
)

torch.save({
    "model_state_dict": best_state,
    "seed": SEED,
    "best_epoch": best_epoch,
    "train_times": TRAIN_TIMES,
    "validation_times": VAL_TIMES,
    "retained_edges": K,
    "architecture": "TAGS-v3 Stable Coverage",
    "checkpoint_selection": "Validation Failure AP"
}, SAVE / "best_validation_checkpoint.pt")

report = {
    "experiment": "E3.6c",
    "scenario": "B_owner_correlated",
    "scenario_seed": 101,
    "model_seed": SEED,
    "train_times": TRAIN_TIMES,
    "validation_times": VAL_TIMES,
    "initial_validation": initial_val,
    "best_epoch": best_epoch,
    "best_validation": best_metrics,
    "epochs_run": len(history),
    "retained_edges": K,
    "node_coverage_required": True,
    "test_accessed": False,
    "hidden_state_accessed": False,
    "baseline_reference": {
        "Trust_MLP_val_AP_mean": 0.397888,
        "Trust_MLP_val_AUC_mean": 0.666459,
        "Trust_MLP_val_BCE_mean": 0.531655
    },
    "limitations": [
        "Exploratory single-model-seed pilot.",
        "Validation repeatedly used for checkpoint selection.",
        "No separately retrained sparsification baselines.",
        "No proof of causal controller benefit.",
        "Full-graph scoring cost remains."
    ]
}

with open(
    SAVE / "report.json",
    "w", encoding="utf-8"
) as f:
    json.dump(report, f, indent=2)

print("\n" + "=" * 72)
print("E3.6c COMPLETED — TEMPORAL VALIDATION")
print("=" * 72)

print("Best epoch:", best_epoch)
print("Best validation AP:",
      round(best_metrics["AP"], 6))
print("Best validation AUC:",
      round(best_metrics["AUC"], 6))
print("Best validation BCE:",
      round(best_metrics["BCE"], 6))
print("Validation mask churn:",
      round(best_metrics["mask_churn"], 6))

print("\nIndependent Trust-MLP reference:")
print("AP:  0.397888")
print("AUC: 0.666459")
print("BCE: 0.531655")

print("\nTest accessed: NO")
print("Checkpoint:", SAVE / "best_validation_checkpoint.pt")


In [ ]:

# E3.6c — Inspect existing output before saving

from pathlib import Path

SAVE = (
    Path("/kaggle/working/phase2_adaptive_siot")
    / "E3_tags_v3"
    / "scenarios"
    / "B_owner_correlated"
    / "seed_101"
    / "tags_v3"
    / "e3_6c_temporal_training"
)

print("Directory exists:", SAVE.exists())

if SAVE.exists():
    files = list(SAVE.iterdir())

    print("Number of items:", len(files))

    for item in files:
        print(
            "FILE" if item.is_file() else "DIR ",
            item.name
        )

print("\nTraining results in memory:")
print("Epochs completed:", len(history))
print("Best epoch:", best_epoch)
print("Best validation metrics:", best_metrics)


In [ ]:

# ============================================================
# E3.6c — VERIFY EXISTING ARTIFACTS (READ ONLY)
# ============================================================

from pathlib import Path
import json
import pandas as pd
import torch

SAVE = (
    Path("/kaggle/working/phase2_adaptive_siot")
    / "E3_tags_v3/scenarios/B_owner_correlated"
    / "seed_101/tags_v3/e3_6c_temporal_training"
)

with open(SAVE / "report.json", encoding="utf-8") as f:
    saved_report = json.load(f)

saved_history = pd.read_csv(
    SAVE / "epoch_history.csv"
)

# Only checkpoint metadata and weights from a
# locally generated, trusted experiment.
checkpoint = torch.load(
    SAVE / "best_validation_checkpoint.pt",
    map_location="cpu",
    weights_only=True
)

print("=" * 68)
print("E3.6c — SAVED ARTIFACT VERIFICATION")
print("=" * 68)

print("Saved best epoch:",
      saved_report.get("best_epoch"))

print("Saved validation metrics:",
      saved_report.get("best_validation"))

print("Checkpoint best epoch:",
      checkpoint.get("best_epoch"))

print("Saved history rows:", len(saved_history))
print("Current memory history rows:", len(history))

print("\nLAST FIVE SAVED EPOCHS")
print(saved_history.tail(5).to_string(index=False))

same_epoch = (
    saved_report.get("best_epoch") == best_epoch
    and checkpoint.get("best_epoch") == best_epoch
)

saved_metrics = saved_report.get("best_validation", {})

same_metrics = all(
    abs(float(saved_metrics.get(k, float("nan")))
        - float(best_metrics[k])) < 1e-8
    for k in ["AP", "AUC", "BCE", "mask_churn"]
)

same_history_length = (
    len(saved_history) == len(history)
)

print("\nINTEGRITY CHECKS")
print("Epoch agreement:", same_epoch)
print("Metric agreement:", same_metrics)
print("History length agreement:", same_history_length)

if same_epoch and same_metrics and same_history_length:
    print("\nE3.6c ARTIFACTS VERIFIED — NO RESAVE NEEDED")
else:
    print("\nEXISTING FILES DIFFER — DO NOT OVERWRITE")


In [ ]:

# ============================================================
# E3.6c — PRESERVE CURRENT IN-MEMORY RUN
# ============================================================
# No retraining.
# No overwriting existing artifacts.
# No validation/test re-evaluation.
# ============================================================

from pathlib import Path
import json
import torch
import pandas as pd
import numpy as np

assert "history" in globals()
assert "best_state" in globals()
assert "best_metrics" in globals()
assert "best_epoch" in globals()
assert best_state is not None

BASE = (
    Path("/kaggle/working/phase2_adaptive_siot")
    / "E3_tags_v3/scenarios/B_owner_correlated"
    / "seed_101/tags_v3"
)

ORIGINAL = BASE / "e3_6c_temporal_training"
NEW = BASE / "e3_6c_temporal_training_memory_run"

if NEW.exists():
    raise FileExistsError(
        f"Separate run already exists: {NEW}"
    )

with open(
    ORIGINAL / "report.json",
    encoding="utf-8"
) as f:
    original_report = json.load(f)

original_history = pd.read_csv(
    ORIGINAL / "epoch_history.csv"
)

current_history = pd.DataFrame(history)

assert len(current_history) == 15
assert best_epoch == 15

# Verify that the in-memory metrics agree
# with the corresponding in-memory history row.
current_best_row = current_history.loc[
    current_history["epoch"] == best_epoch
].iloc[0]

mapping = {
    "AP": "val_AP",
    "AUC": "val_AUC",
    "BCE": "val_BCE",
    "mask_churn": "val_mask_churn"
}

for metric, column in mapping.items():
    assert np.isclose(
        float(best_metrics[metric]),
        float(current_best_row[column]),
        atol=1e-9,
        rtol=0
    ), f"Memory mismatch: {metric}"

# Preserve current run in its own directory.
NEW.mkdir(parents=True)

current_history.to_csv(
    NEW / "epoch_history.csv",
    index=False
)

torch.save(
    {
        "model_state_dict": best_state,
        "seed": 42,
        "best_epoch": best_epoch,
        "train_times": [5, 6, 7],
        "validation_times": [8, 9],
        "retained_edges": 73058,
        "architecture": "TAGS-v3 Stable Coverage",
        "checkpoint_selection": "Validation Failure AP"
    },
    NEW / "best_validation_checkpoint.pt"
)

new_report = {
    "experiment": "E3.6c",
    "run_id": "memory_run",
    "scenario_seed": 101,
    "model_seed": 42,
    "best_epoch": int(best_epoch),
    "best_validation": {
        k: float(v)
        for k, v in best_metrics.items()
    },
    "epochs_run": len(current_history),
    "existing_run_preserved": True,
    "test_accessed": False,
    "note": (
        "Separate in-memory execution. "
        "Do not merge its metrics or checkpoint "
        "with the original saved execution."
    )
}

with open(
    NEW / "report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(new_report, f, indent=2)

print("=" * 72)
print("E3.6c — TWO RUNS PRESERVED")
print("=" * 72)

print("\nORIGINAL SAVED RUN")
print(original_report["best_validation"])

print("\nCURRENT MEMORY RUN")
print(new_report["best_validation"])

print("\nORIGINAL FINAL EPOCH")
print(
    original_history.tail(1).to_string(index=False)
)

print("\nCURRENT FINAL EPOCH")
print(
    current_history.tail(1).to_string(index=False)
)

print("\nNew run saved:", NEW)
print("Original run unchanged: YES")
print("No additional training performed.")
print("No test data accessed.")
print("\nE3.6c RUN PRESERVATION COMPLETED")


In [ ]:

# ============================================================
# PHASE III — E3.6d
# CONTINUE TAGS-v3 TEMPORAL TRAINING
# ============================================================
# Continue in-memory run from epoch 15.
# Preserve model AND AdamW optimizer state.
# Architecture, data and learning rate unchanged.
#
# Maximum total epochs: 60
# Early stopping patience: 12
# Selection metric: validation failure AP
#
# No test or hidden simulator access.
# No previous artifacts overwritten.
# ============================================================

from pathlib import Path
import copy
import json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

# ------------------------------------------------------------
# 0. Verify state and reserve output location
# ------------------------------------------------------------

required = [
    "model", "optimizer", "data",
    "temporal_forward", "validate",
    "history", "best_state",
    "best_metrics", "best_epoch",
    "TRAIN_TIMES", "VAL_TIMES",
    "OUT", "K"
]

missing = [
    key for key in required
    if key not in globals()
]

if missing:
    raise RuntimeError(
        "Training state is incomplete. "
        "Do not treat a checkpoint-only restart "
        f"as exact continuation. Missing: {missing}"
    )

assert TRAIN_TIMES == [5, 6, 7]
assert VAL_TIMES == [8, 9]
assert len(history) == 15
assert history[-1]["epoch"] == 15
assert best_epoch == 15

START_EPOCH = 15
MAX_TOTAL_EPOCHS = 60
PATIENCE = 12
MIN_AP_GAIN = 1e-5
GRAD_CLIP_E36D = 1.0

SAVE_D = OUT / "e3_6d_extended_training"

if SAVE_D.exists():
    raise FileExistsError(
        f"Output already exists: {SAVE_D}"
    )

# Confirm optimizer is attached to the actual model.
model_param_ids = {
    id(p) for p in model.parameters()
}

optimizer_param_ids = {
    id(p)
    for group in optimizer.param_groups
    for p in group["params"]
}

assert model_param_ids == optimizer_param_ids, (
    "Optimizer and model parameter objects differ."
)

# AdamW must have already accumulated state.
assert len(optimizer.state) > 0, (
    "Optimizer state is empty; exact continuation "
    "is not possible."
)

assert all(
    abs(group["lr"] - 1e-3) < 1e-12
    for group in optimizer.param_groups
)

# ------------------------------------------------------------
# 1. Verify the live model matches the memory run
# ------------------------------------------------------------

current_val = validate(model)

for metric in ["AP", "AUC", "BCE", "mask_churn"]:
    expected = float(best_metrics[metric])
    observed = float(current_val[metric])

    if not np.isclose(
        expected, observed,
        atol=2e-5, rtol=0
    ):
        raise RuntimeError(
            f"Current model differs from the "
            f"memory-run epoch 15: {metric} "
            f"{observed} != {expected}"
        )

print("=" * 72)
print("E3.6d — EXTENDED TEMPORAL TRAINING")
print("=" * 72)

print("Continuation verified: YES")
print("Current epoch:", START_EPOCH)
print("Current validation AP:",
      round(current_val["AP"], 6))
print("Maximum total epochs:", MAX_TOTAL_EPOCHS)
print("Early stopping patience:", PATIENCE)
print("Model and optimizer state preserved: YES")

# ------------------------------------------------------------
# 2. Restore best-checkpoint tracking
# ------------------------------------------------------------

extended_history = copy.deepcopy(history)

best_ap_d = float(best_metrics["AP"])
best_epoch_d = int(best_epoch)
best_metrics_d = dict(best_metrics)
best_state_d = copy.deepcopy(best_state)

best_optimizer_d = copy.deepcopy(
    optimizer.state_dict()
)

wait = 0
new_epochs = 0

# ------------------------------------------------------------
# 3. Continue training
# ------------------------------------------------------------

for epoch in range(
    START_EPOCH + 1,
    MAX_TOTAL_EPOCHS + 1
):

    model.train()

    losses = []
    grad_norms = []

    for t in TRAIN_TIMES:

        batch = data[t]

        optimizer.zero_grad(
            set_to_none=True
        )

        result = temporal_forward(
            model, batch
        )

        loss = (
            F.binary_cross_entropy_with_logits(
                result["logits"],
                batch["y"]
            )
        )

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite loss at epoch {epoch}"
            )

        loss.backward()

        grad = torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            GRAD_CLIP_E36D
        )

        if not torch.isfinite(grad):
            raise RuntimeError(
                f"Non-finite gradient at epoch {epoch}"
            )

        grad_norms.append(float(grad.item()))

        optimizer.step()

        losses.append((
            float(loss.item()),
            batch["count"]
        ))

    train_bce = (
        sum(loss * count for loss, count in losses)
        / sum(count for _, count in losses)
    )

    val = validate(model)

    record = {
        "epoch": epoch,
        "train_BCE": train_bce,
        "val_AP": val["AP"],
        "val_AUC": val["AUC"],
        "val_BCE": val["BCE"],
        "val_mask_churn": val["mask_churn"],
        "max_preclip_gradient": max(grad_norms)
    }

    extended_history.append(record)
    new_epochs += 1

    improved = (
        val["AP"] > best_ap_d + MIN_AP_GAIN
    )

    if improved:
        best_ap_d = float(val["AP"])
        best_epoch_d = epoch
        best_metrics_d = dict(val)

        best_state_d = copy.deepcopy(
            model.state_dict()
        )

        best_optimizer_d = copy.deepcopy(
            optimizer.state_dict()
        )

        wait = 0
    else:
        wait += 1

    print(
        f"epoch={epoch:02d} | "
        f"Train BCE={train_bce:.6f} | "
        f"Val AP={val['AP']:.6f} | "
        f"AUC={val['AUC']:.6f} | "
        f"BCE={val['BCE']:.6f} | "
        f"churn={val['mask_churn']:.4f} | "
        f"best={best_epoch_d:02d}",
        flush=True
    )

    if wait >= PATIENCE:
        print(
            f"\nEarly stopping at epoch {epoch}."
        )
        break

# ------------------------------------------------------------
# 4. Save as an entirely separate experiment
# ------------------------------------------------------------

SAVE_D.mkdir(parents=True)

history_df = pd.DataFrame(
    extended_history
)

history_df.to_csv(
    SAVE_D / "epoch_history.csv",
    index=False
)

torch.save(
    {
        "model_state_dict": best_state_d,
        "optimizer_state_dict": best_optimizer_d,
        "best_epoch": best_epoch_d,
        "model_seed": 42,
        "scenario_seed": 101,
        "train_times": TRAIN_TIMES,
        "validation_times": VAL_TIMES,
        "retained_edges": K,
        "architecture": (
            "TAGS-v3 Stable Coverage"
        ),
        "checkpoint_selection": (
            "Validation Failure AP"
        )
    },
    SAVE_D / "best_validation_checkpoint.pt"
)

report = {
    "experiment": "E3.6d",
    "scenario": "B_owner_correlated",
    "scenario_seed": 101,
    "model_seed": 42,
    "continuation_from": (
        "E3.6c in-memory run, epoch 15"
    ),
    "start_epoch": START_EPOCH,
    "max_total_epochs": MAX_TOTAL_EPOCHS,
    "early_stopping_patience": PATIENCE,
    "new_epochs_completed": new_epochs,
    "last_epoch": int(
        extended_history[-1]["epoch"]
    ),
    "best_epoch": best_epoch_d,
    "initial_validation": current_val,
    "best_validation": best_metrics_d,
    "architecture_changed": False,
    "scenario_changed": False,
    "optimizer_state_preserved": True,
    "validation_used_for_selection": True,
    "test_accessed": False,
    "hidden_state_accessed": False,
    "scientific_limitations": [
        "Exploratory continuation, not final comparison.",
        "Single model and scenario seed.",
        "Validation used repeatedly for selection.",
        "Baseline models were independently trained.",
        "Sparsification and controller contributions "
        "are not yet established."
    ]
}

report_path = SAVE_D / "report.json"

with open(
    report_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(report, f, indent=2)

# ------------------------------------------------------------
# 5. Final comparison with independent baseline
# ------------------------------------------------------------

baseline_ap = 0.397888
baseline_auc = 0.666459
baseline_bce = 0.531655

print("\n" + "=" * 72)
print("E3.6d COMPLETED — EXTENDED TEMPORAL TRAINING")
print("=" * 72)

print("Additional epochs:", new_epochs)
print("Final epoch:", report["last_epoch"])
print("Best epoch:", best_epoch_d)

print("\nBEST TAGS-v3 VALIDATION")
print(
    "AP:",
    round(best_metrics_d["AP"], 6)
)
print(
    "AUC:",
    round(best_metrics_d["AUC"], 6)
)
print(
    "BCE:",
    round(best_metrics_d["BCE"], 6)
)
print(
    "Mask churn:",
    round(best_metrics_d["mask_churn"], 6)
)

print("\nVERSUS TRUST-MLP BASELINE")
print(
    "Delta AP:",
    f"{best_metrics_d['AP'] - baseline_ap:+.6f}"
)
print(
    "Delta AUC:",
    f"{best_metrics_d['AUC'] - baseline_auc:+.6f}"
)
print(
    "Delta BCE:",
    f"{best_metrics_d['BCE'] - baseline_bce:+.6f}"
)

print("\nPrevious runs preserved: YES")
print("Test accessed: NO")
print("Saved:", report_path)


In [ ]:

# ============================================================
# E3.6e — FROZEN MODEL SPARSIFICATION ABLATION
# ============================================================
# Trained TAGS-v3 checkpoint at best epoch.
# Validation t=8,9 only.
# No training, test access, or hidden state.
# Random baselines: exact budget + full coverage.
# ============================================================

from pathlib import Path
import copy
import json
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    log_loss
)

assert all(x in globals() for x in [
    "stable", "data", "U", "V", "K",
    "coverage_topk", "OUT"
])

SOURCE = (
    OUT / "e3_6d_extended_training" /
    "best_validation_checkpoint.pt"
)
DEST = OUT / "e3_6e_frozen_ablation"

if DEST.exists():
    raise FileExistsError(f"Output exists: {DEST}")

checkpoint = torch.load(
    SOURCE, map_location="cpu", weights_only=True
)

frozen = copy.deepcopy(stable).to(U.device)
frozen.load_state_dict(
    checkpoint["model_state_dict"]
)
frozen.eval()

N = data[8]["xn"].shape[0]
M = U.numel()

u = U.cpu().numpy()
v = V.cpu().numpy()

assert K == 73058
assert checkpoint["best_epoch"] == 60

# ------------------------------------------------------------
# 1. Generate fixed random coverage masks
# ------------------------------------------------------------

def random_coverage(seed):
    rng = np.random.default_rng(seed)
    order = rng.permutation(M)

    selected = np.zeros(M, dtype=bool)
    covered = np.zeros(N, dtype=bool)
    uncovered = N

    for eid in order:
        a, b = u[eid], v[eid]

        if not (covered[a] and covered[b]):
            selected[eid] = True

            if not covered[a]:
                covered[a] = True
                uncovered -= 1

            if not covered[b]:
                covered[b] = True
                uncovered -= 1

        if uncovered == 0:
            break

    assert uncovered == 0
    assert selected.sum() <= K

    remaining = K - int(selected.sum())
    available = order[~selected[order]]
    selected[available[:remaining]] = True

    assert selected.sum() == K
    return selected

random_masks = {
    f"Random coverage {seed}": random_coverage(seed)
    for seed in [201, 202, 203]
}

def tensor_mask(mask):
    return torch.as_tensor(
        mask.astype(np.float32),
        device=U.device
    )

# ------------------------------------------------------------
# 2. Prepare graph selections per time
# ------------------------------------------------------------

@torch.no_grad()
def get_masks(batch):
    xn, xe, xc = (
        batch["xn"], batch["xe"], batch["xc"]
    )

    full = xn.new_ones(M)
    h = frozen.encoder(xn, U, V, full)

    edge_repr = torch.cat([
        h[U] + h[V],
        torch.abs(h[U] - h[V]),
        xe
    ], dim=1)

    edge_score = frozen.scorer(
        edge_repr
    ).squeeze(-1)

    priority = frozen.controller(
        xc
    ).squeeze(-1)

    combined = edge_score + 0.5 * (
        priority[U] + priority[V]
    )

    learned, _ = coverage_topk(combined, K)
    no_controller, _ = coverage_topk(
        edge_score, K
    )

    masks = {
        "Learned TAGS": learned,
        "No controller at inference": no_controller,
        "Full graph": full
    }

    for name, mask in random_masks.items():
        masks[name] = tensor_mask(mask)

    return masks

# ------------------------------------------------------------
# 3. Frozen prediction with alternative masks
# ------------------------------------------------------------

@torch.no_grad()
def probabilities(batch, mask):
    xn, xe = batch["xn"], batch["xe"]
    ids = batch["target"]

    z = frozen.encoder(xn, U, V, mask)

    a, b = U[ids], V[ids]

    pair = torch.cat([
        z[a] + z[b],
        torch.abs(z[a] - z[b]),
        z[a] * z[b]
    ], dim=1)

    logits = (
        frozen.trust_head(xe[ids]).squeeze(-1)
        + frozen.graph_head(pair).squeeze(-1)
    )

    return torch.sigmoid(logits).cpu().numpy()

@torch.no_grad()
def trust_probabilities(batch):
    logits = frozen.trust_head(
        batch["xe"][batch["target"]]
    ).squeeze(-1)

    return torch.sigmoid(logits).cpu().numpy()

def score(y, p):
    p = np.clip(p, 1e-7, 1 - 1e-7)

    return {
        "AP": float(average_precision_score(y, p)),
        "AUC": float(roc_auc_score(y, p)),
        "BCE": float(log_loss(y, p, labels=[0, 1]))
    }

# ------------------------------------------------------------
# 4. Evaluate both validation times
# ------------------------------------------------------------

predictions = {}
labels = []
selection_masks = {}

for t in [8, 9]:
    batch = data[t]
    labels.append(batch["y"].cpu().numpy())

    masks = get_masks(batch)

    for name, mask in masks.items():
        hard = mask.cpu().numpy().astype(bool)

        if name != "Full graph":
            assert hard.sum() == K
            degree = np.bincount(
                np.r_[u[hard], v[hard]],
                minlength=N
            )
            assert degree.min() >= 1

        selection_masks[(name, t)] = hard

        predictions.setdefault(name, []).append(
            probabilities(batch, mask)
        )

    predictions.setdefault(
        "Trust head only", []
    ).append(trust_probabilities(batch))

y = np.concatenate(labels)

rows = []
for name, pieces in predictions.items():
    p = np.concatenate(pieces)
    rows.append({"method": name, **score(y, p)})

df = pd.DataFrame(rows).sort_values(
    "AP", ascending=False
)

def mask_churn(name):
    a = selection_masks[(name, 8)]
    b = selection_masks[(name, 9)]
    return float(np.count_nonzero(a != b) / (2*K))

churn = {
    name: mask_churn(name)
    for name in predictions
    if (name, 8) in selection_masks
    and name != "Full graph"
}

learned = df[df.method == "Learned TAGS"].iloc[0]
random_rows = df[
    df.method.str.startswith("Random coverage")
]

print("=" * 72)
print("E3.6e — FROZEN MODEL VALIDATION ABLATION")
print("=" * 72)
print("Best epoch:", checkpoint["best_epoch"])
print("Validation times: 8, 9")
print("Validation failure prevalence:", round(y.mean(), 6))

print("\nPREDICTIVE PERFORMANCE")
print(df.round(6).to_string(index=False))

print("\nMASK CHURN")
for name, value in churn.items():
    print(f"{name:30s}: {value:.6f}")

print("\nLEARNED MINUS RANDOM MEAN")
for metric in ["AP", "AUC", "BCE"]:
    delta = (
        float(learned[metric])
        - float(random_rows[metric].mean())
    )
    print(f"{metric}: {delta:+.6f}")

# ------------------------------------------------------------
# 5. Save results without modifying previous stages
# ------------------------------------------------------------

DEST.mkdir(parents=True)

df.to_csv(
    DEST / "comparison.csv", index=False
)

report = {
    "experiment": "E3.6e",
    "source_checkpoint": str(SOURCE),
    "best_epoch": int(checkpoint["best_epoch"]),
    "validation_times": [8, 9],
    "results": rows,
    "mask_churn": churn,
    "random_mask_seeds": [201, 202, 203],
    "trained_parameters_frozen": True,
    "no_retraining_of_alternatives": True,
    "test_accessed": False,
    "hidden_state_accessed": False,
    "limitations": [
        "Frozen-mask intervention, not a fair "
        "independently retrained baseline.",
        "Controller removal occurs only at inference.",
        "Random coverage masks are fixed across time.",
        "Validation has been used for model selection.",
        "No statistical significance established."
    ]
}

with open(
    DEST / "report.json",
    "w", encoding="utf-8"
) as f:
    json.dump(report, f, indent=2)

print("\nE3.6e COMPLETED")
print("No model training or optimizer updates.")
print("No test or hidden-state access.")
print("Saved:", DEST / "report.json")


In [ ]:

# ============================================================
# PHASE III — E3.7a
# INDEPENDENTLY TRAINED NO-CONTROLLER ABLATION
# ============================================================
# Scenario B / simulator seed 101 / model seed 42
# Train times: 5, 6, 7
# Validation times: 8, 9
# Fixed training budget: 60 epochs
#
# Same Stable GraphSAGE, same trust/graph heads,
# same exact Top-K and node-coverage constraint.
#
# ONLY CHANGE:
# Edge selection does not use controller priority.
#
# No test or hidden-state access.
# No previous artifacts overwritten.
# ============================================================

from pathlib import Path
import copy
import json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    log_loss
)

# ------------------------------------------------------------
# 0. Preconditions and immutable configuration
# ------------------------------------------------------------

required = [
    "stable", "data", "U", "V", "K",
    "coverage_topk", "OUT"
]

missing = [x for x in required if x not in globals()]

if missing:
    raise RuntimeError(
        f"Missing notebook objects: {missing}"
    )

E37_TRAIN = [5, 6, 7]
E37_VAL = [8, 9]

E37_SEED = 42
E37_EPOCHS = 60
E37_LR = 1e-3
E37_TAU = 1.0
E37_CLIP = 1.0

E37_DEVICE = U.device
E37_M = int(U.numel())
E37_N = int(data[5]["xn"].shape[0])

E37_OUT = OUT / "e3_7a_no_controller_retrained"

# Check BEFORE any expensive computation.
if E37_OUT.exists():
    raise FileExistsError(
        f"Output already exists: {E37_OUT}"
    )

assert E37_M == 146117
assert K == 73058
assert E37_N == 16216

for t in E37_TRAIN + E37_VAL:
    assert t in data
    assert data[t]["xn"].shape[0] == E37_N
    assert data[t]["xe"].shape[0] == E37_M

torch.manual_seed(E37_SEED)
np.random.seed(E37_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(E37_SEED)

# Start from the original UNTRAINED stabilized model.
# The trained E3.6d checkpoint is NOT used.
e37_model = copy.deepcopy(stable).to(E37_DEVICE)

# Controller parameters are excluded from optimization.
e37_parameters = [
    p for name, p in e37_model.named_parameters()
    if not name.startswith("controller.")
]

e37_optimizer = torch.optim.AdamW(
    e37_parameters,
    lr=E37_LR,
    weight_decay=1e-4
)

# ------------------------------------------------------------
# 1. Forward: scorer only, NO controller
# ------------------------------------------------------------

def e37_forward(model, batch, training=False):
    xn = batch["xn"]
    xe = batch["xe"]
    target = batch["target"]

    full_weights = xn.new_ones(E37_M)

    # Full-graph encoder for learning edge scores.
    h = model.encoder(
        xn, U, V, full_weights
    )

    edge_rep = torch.cat([
        h[U] + h[V],
        torch.abs(h[U] - h[V]),
        xe
    ], dim=1)

    scores = model.scorer(
        edge_rep
    ).squeeze(-1)

    # No controller priorities added.
    hard, coverage_edges = coverage_topk(
        scores, K
    )

    if training:
        threshold = torch.topk(
            scores.detach(), K
        ).values[-1]

        soft = torch.sigmoid(
            (scores - threshold) / E37_TAU
        )

        mask = hard + (soft - soft.detach())
    else:
        mask = hard

    z = model.encoder(
        xn, U, V, mask
    )

    a = U[target]
    b = V[target]

    pair = torch.cat([
        z[a] + z[b],
        torch.abs(z[a] - z[b]),
        z[a] * z[b]
    ], dim=1)

    trust_logits = model.trust_head(
        xe[target]
    ).squeeze(-1)

    graph_logits = model.graph_head(
        pair
    ).squeeze(-1)

    logits = trust_logits + graph_logits

    return {
        "logits": logits,
        "hard": hard,
        "coverage_edges": coverage_edges
    }

# ------------------------------------------------------------
# 2. Validation with frozen parameters
# ------------------------------------------------------------

def e37_validate(model):
    model.eval()

    y_parts = []
    p_parts = []
    masks = []
    coverage_counts = []

    with torch.no_grad():
        for t in E37_VAL:
            batch = data[t]
            result = e37_forward(
                model, batch, training=False
            )

            probabilities = torch.sigmoid(
                result["logits"]
            ).cpu().numpy()

            labels = (
                batch["y"].cpu().numpy()
            )

            y_parts.append(labels)
            p_parts.append(probabilities)

            hard = (
                result["hard"]
                .cpu()
                .numpy()
                .astype(bool)
            )

            assert hard.sum() == K

            masks.append(hard)

            coverage_counts.append(
                int(result["coverage_edges"])
            )

    y = np.concatenate(y_parts)

    p = np.clip(
        np.concatenate(p_parts),
        1e-7,
        1 - 1e-7
    )

    # One replacement changes two binary entries.
    churn = float(
        np.count_nonzero(
            masks[0] != masks[1]
        ) / (2 * K)
    )

    return {
        "AP": float(
            average_precision_score(y, p)
        ),
        "AUC": float(
            roc_auc_score(y, p)
        ),
        "BCE": float(
            log_loss(y, p, labels=[0, 1])
        ),
        "mask_churn": churn,
        "coverage_edges_t8": coverage_counts[0],
        "coverage_edges_t9": coverage_counts[1]
    }

# ------------------------------------------------------------
# 3. Independent training, fixed 60 epochs
# ------------------------------------------------------------

e37_history = []

e37_best_ap = -np.inf
e37_best_epoch = -1
e37_best_metrics = None
e37_best_state = None

print("=" * 76)
print("E3.7a — INDEPENDENT NO-CONTROLLER TRAINING")
print("=" * 76)
print("Train:", E37_TRAIN)
print("Validation:", E37_VAL)
print("Seed:", E37_SEED)
print("Epoch budget:", E37_EPOCHS)
print("Controller in edge selection: NO")
print("Exact edge budget:", K)
print("Node coverage constraint: YES")

for epoch in range(1, E37_EPOCHS + 1):
    e37_model.train()

    losses = []
    grad_norms = []

    for t in E37_TRAIN:
        batch = data[t]

        e37_optimizer.zero_grad(
            set_to_none=True
        )

        output = e37_forward(
            e37_model,
            batch,
            training=True
        )

        loss = F.binary_cross_entropy_with_logits(
            output["logits"],
            batch["y"]
        )

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite loss at epoch {epoch}"
            )

        loss.backward()

        grad = torch.nn.utils.clip_grad_norm_(
            e37_parameters,
            E37_CLIP
        )

        if not torch.isfinite(grad):
            raise RuntimeError(
                f"Non-finite gradient at epoch {epoch}"
            )

        e37_optimizer.step()

        losses.append((
            float(loss.item()),
            int(batch["count"])
        ))

        grad_norms.append(float(grad.item()))

    train_bce = (
        sum(loss * count for loss, count in losses)
        / sum(count for _, count in losses)
    )

    val = e37_validate(e37_model)

    record = {
        "epoch": epoch,
        "train_BCE": train_bce,
        "val_AP": val["AP"],
        "val_AUC": val["AUC"],
        "val_BCE": val["BCE"],
        "val_mask_churn": val["mask_churn"],
        "max_preclip_gradient": max(grad_norms)
    }

    e37_history.append(record)

    if val["AP"] > e37_best_ap + 1e-5:
        e37_best_ap = val["AP"]
        e37_best_epoch = epoch
        e37_best_metrics = dict(val)
        e37_best_state = copy.deepcopy(
            e37_model.state_dict()
        )

    if epoch == 1 or epoch % 5 == 0:
        print(
            f"epoch={epoch:02d} | "
            f"Train BCE={train_bce:.6f} | "
            f"Val AP={val['AP']:.6f} | "
            f"AUC={val['AUC']:.6f} | "
            f"BCE={val['BCE']:.6f} | "
            f"churn={val['mask_churn']:.4f}",
            flush=True
        )

assert e37_best_state is not None
assert len(e37_history) == E37_EPOCHS

# ------------------------------------------------------------
# 4. Compare against existing TAGS-v3 experiment
# ------------------------------------------------------------

e37_reference_path = (
    OUT / "e3_6d_extended_training" /
    "report.json"
)

with open(
    e37_reference_path,
    encoding="utf-8"
) as f:
    e37_reference = json.load(f)

e37_full = e37_reference["best_validation"]

e37_delta = {
    metric: float(
        e37_full[metric] - e37_best_metrics[metric]
    )
    for metric in ["AP", "AUC", "BCE", "mask_churn"]
}

# ------------------------------------------------------------
# 5. Save as independent, non-overwriting artifact
# ------------------------------------------------------------

e37_report = {
    "experiment": "E3.7a",
    "scenario": "B_owner_correlated",
    "scenario_seed": 101,
    "model_seed": E37_SEED,
    "train_times": E37_TRAIN,
    "validation_times": E37_VAL,
    "epochs": E37_EPOCHS,
    "controller_in_training": False,
    "controller_in_inference": False,
    "encoder": "Stable GraphSAGE",
    "selection": "Coverage-constrained Top-K",
    "retained_edges": int(K),
    "best_epoch": int(e37_best_epoch),
    "best_validation": e37_best_metrics,
    "full_TAGS_reference": e37_full,
    "full_minus_no_controller": e37_delta,
    "test_accessed": False,
    "hidden_state_accessed": False,
    "limitations": [
        "Single model seed and scenario realization.",
        "Both checkpoints selected using validation AP.",
        "Full TAGS training length was chosen exploratorily.",
        "Controller training effect is not necessarily "
        "the same as temporal adaptivity.",
        "No conclusion about statistical significance."
    ]
}

E37_HISTORY = pd.DataFrame(e37_history)

E37_OUT.mkdir(parents=True)

E37_HISTORY.to_csv(
    E37_OUT / "epoch_history.csv",
    index=False
)

torch.save({
    "model_state_dict": e37_best_state,
    "model_seed": E37_SEED,
    "best_epoch": e37_best_epoch,
    "controller_enabled": False,
    "retained_edges": int(K)
}, E37_OUT / "best_validation_checkpoint.pt")

with open(
    E37_OUT / "report.json",
    "w", encoding="utf-8"
) as f:
    json.dump(
        e37_report,
        f,
        indent=2
    )

print("\n" + "=" * 76)
print("E3.7a COMPLETED — CONTROLLER ABLATION")
print("=" * 76)

print("No-controller best epoch:", e37_best_epoch)

print("\nFULL TAGS-v3")
for metric in ["AP", "AUC", "BCE", "mask_churn"]:
    print(
        f"  {metric}: "
        f"{e37_full[metric]:.6f}"
    )

print("\nRETRAINED NO-CONTROLLER")
for metric in ["AP", "AUC", "BCE", "mask_churn"]:
    print(
        f"  {metric}: "
        f"{e37_best_metrics[metric]:.6f}"
    )

print("\nFULL MINUS NO-CONTROLLER")
for metric in ["AP", "AUC", "BCE", "mask_churn"]:
    print(
        f"  Delta {metric}: "
        f"{e37_delta[metric]:+.6f}"
    )

print("\nNo previous artifacts overwritten.")
print("No test or hidden-state access.")
print("Report:", E37_OUT / "report.json")


In [ ]:

# ============================================================
# E3.7b — GPU / CPU EXECUTION AUDIT
# ============================================================
# Read-only timing diagnostic.
# No optimizer updates.
# No validation/test access.
# ============================================================

import time
import torch
import numpy as np

assert "e37_model" in globals()
assert "data" in globals()
assert "U" in globals()
assert "V" in globals()
assert "coverage_topk" in globals()
assert "K" in globals()

print("=" * 68)
print("E3.7b — GPU / CPU EXECUTION AUDIT")
print("=" * 68)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Model device:", next(e37_model.parameters()).device)
print("Node tensor device:", data[7]["xn"].device)
print("Edge tensor device:", data[7]["xe"].device)
print("Graph index device:", U.device)

assert next(e37_model.parameters()).is_cuda
assert data[7]["xn"].is_cuda
assert data[7]["xe"].is_cuda
assert U.is_cuda and V.is_cuda

def sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

e37_model.eval()
batch = data[7]

with torch.no_grad():
    sync()
    start = time.perf_counter()

    full = batch["xn"].new_ones(U.numel())

    h = e37_model.encoder(
        batch["xn"], U, V, full
    )

    sync()
    encoder_seconds = time.perf_counter() - start

    start = time.perf_counter()

    edge_rep = torch.cat([
        h[U] + h[V],
        torch.abs(h[U] - h[V]),
        batch["xe"]
    ], dim=1)

    scores = e37_model.scorer(
        edge_rep
    ).squeeze(-1)

    sync()
    scorer_seconds = time.perf_counter() - start

    start = time.perf_counter()

    hard, coverage_edges = coverage_topk(
        scores, K
    )

    sync()
    selection_seconds = time.perf_counter() - start

    start = time.perf_counter()

    z = e37_model.encoder(
        batch["xn"], U, V, hard
    )

    sync()
    sparse_encoder_seconds = (
        time.perf_counter() - start
    )

print("\nTIMING — ONE FORWARD SNAPSHOT")
print(f"Full GraphSAGE:  {encoder_seconds:.4f} s")
print(f"Edge scorer:     {scorer_seconds:.4f} s")
print(f"CPU selection:   {selection_seconds:.4f} s")
print(f"Second GraphSAGE:{sparse_encoder_seconds:.4f} s")

total = (
    encoder_seconds
    + scorer_seconds
    + selection_seconds
    + sparse_encoder_seconds
)

print(f"Measured total:  {total:.4f} s")
print(
    "Selection time share:",
    f"{100 * selection_seconds / max(total, 1e-9):.2f}%"
)

print("\nMASK AUDIT")
print("Scores device:", scores.device)
print("Returned mask device:", hard.device)
print("Retained edges:", int(hard.sum().item()))
print("Coverage-phase edges:", coverage_edges)

assert int(hard.sum().item()) == K

print("\nE3.7b COMPLETED")
print("No optimization or data modification.")


In [ ]:

# ============================================================
# PHASE III — E3.7c
# EXACT-EQUIVALENT GPU COVERAGE SELECTOR
# ============================================================
# Goal:
# Replace the sequential CPU greedy loop with
# vectorized GPU operations, WITHOUT changing
# the selected edges.
#
# No training, no test access, no file changes.
# ============================================================

import time
import numpy as np
import torch

assert torch.cuda.is_available()
assert "e37_model" in globals()
assert "data" in globals()
assert "coverage_topk" in globals()
assert "U" in globals() and "V" in globals()
assert "K" in globals()

assert U.is_cuda and V.is_cuda

N_GPU = int(data[7]["xn"].shape[0])
M_GPU = int(U.numel())

assert M_GPU == 146117
assert K == 73058

# ------------------------------------------------------------
# 1. GPU implementation of the SAME greedy selector
# ------------------------------------------------------------

@torch.no_grad()
def coverage_topk_gpu(scores, k):
    """
    Exact equivalent of the existing CPU greedy
    coverage selection, including stable tie order.

    Phase 1:
      Select the highest-priority incident edge
      for every node, then take their union.

    Phase 2:
      Fill the remaining budget with the
      highest-ranked unselected edges.
    """

    assert scores.is_cuda
    assert scores.ndim == 1
    assert scores.numel() == M_GPU
    assert bool(torch.isfinite(scores).all())
    assert 0 < k <= M_GPU

    device = scores.device

    # Same ranking semantics as:
    # np.argsort(-scores, kind="stable")
    order = torch.argsort(
        scores.detach(),
        descending=True,
        stable=True
    )

    rank = torch.empty_like(order)

    rank[order] = torch.arange(
        M_GPU,
        device=device,
        dtype=torch.long
    )

    # For each node, find its earliest incident
    # edge in the globally ranked ordering.
    first_rank = torch.full(
        (N_GPU,),
        M_GPU,
        dtype=torch.long,
        device=device
    )

    first_rank.scatter_reduce_(
        0, U, rank,
        reduce="amin",
        include_self=True
    )

    first_rank.scatter_reduce_(
        0, V, rank,
        reduce="amin",
        include_self=True
    )

    if bool((first_rank == M_GPU).any()):
        raise RuntimeError(
            "Original topology contains isolated nodes."
        )

    # One highest-ranked incident edge per node.
    # Multiple nodes can select the same edge.
    cover_ids = order[first_rank]

    selected = torch.zeros(
        M_GPU,
        dtype=torch.bool,
        device=device
    )

    selected[cover_ids] = True

    coverage_count = int(
        selected.sum().item()
    )

    if coverage_count > k:
        raise RuntimeError(
            "Node coverage exceeds edge budget."
        )

    # Complete the exact global budget, using
    # the original ranking order.
    remaining = k - coverage_count

    if remaining > 0:
        available = order[
            ~selected[order]
        ]

        selected[
            available[:remaining]
        ] = True

    hard = selected.to(
        dtype=scores.dtype
    )

    return hard, coverage_count

# ------------------------------------------------------------
# 2. Generate scores from the already trained model
# ------------------------------------------------------------

e37_model.eval()

with torch.no_grad():

    batch = data[7]

    full_weights = batch["xn"].new_ones(
        M_GPU
    )

    embeddings = e37_model.encoder(
        batch["xn"],
        U,
        V,
        full_weights
    )

    edge_rep = torch.cat([
        embeddings[U] + embeddings[V],
        torch.abs(
            embeddings[U] - embeddings[V]
        ),
        batch["xe"]
    ], dim=1)

    scores = e37_model.scorer(
        edge_rep
    ).squeeze(-1).detach()

assert scores.is_cuda

# ------------------------------------------------------------
# 3. Exact equivalence test
# ------------------------------------------------------------

cpu_mask, cpu_coverage = coverage_topk(
    scores, K
)

gpu_mask, gpu_coverage = coverage_topk_gpu(
    scores, K
)

equal = torch.equal(
    cpu_mask, gpu_mask
)

differences = int(
    torch.count_nonzero(
        cpu_mask != gpu_mask
    ).item()
)

assert cpu_coverage == gpu_coverage
assert equal, (
    f"CPU/GPU selector mismatch: "
    f"{differences} different mask entries."
)

# Validate exact edge count and node coverage.
assert int(gpu_mask.sum().item()) == K

degree = torch.zeros(
    N_GPU,
    dtype=torch.int32,
    device=U.device
)

chosen = gpu_mask.bool()

degree.index_add_(
    0,
    U[chosen],
    torch.ones_like(
        U[chosen],
        dtype=torch.int32
    )
)

degree.index_add_(
    0,
    V[chosen],
    torch.ones_like(
        V[chosen],
        dtype=torch.int32
    )
)

isolates = int(
    (degree == 0).sum().item()
)

assert isolates == 0

# ------------------------------------------------------------
# 4. Warm-up, then measure selector-only runtime
# ------------------------------------------------------------

for _ in range(3):
    coverage_topk_gpu(scores, K)

torch.cuda.synchronize()

def measure(fn, repeats):
    samples = []

    for _ in range(repeats):
        torch.cuda.synchronize()

        start = time.perf_counter()

        fn(scores, K)

        torch.cuda.synchronize()

        samples.append(
            time.perf_counter() - start
        )

    return np.array(samples)

cpu_times = measure(
    coverage_topk,
    repeats=5
)

gpu_times = measure(
    coverage_topk_gpu,
    repeats=15
)

cpu_median = float(
    np.median(cpu_times)
)

gpu_median = float(
    np.median(gpu_times)
)

speedup = cpu_median / gpu_median

# ------------------------------------------------------------
# 5. Final read-only report
# ------------------------------------------------------------

print("=" * 72)
print("E3.7c — EXACT-EQUIVALENT GPU SELECTOR")
print("=" * 72)

print("GPU:", torch.cuda.get_device_name(0))
print("Nodes:", N_GPU)
print("Original edges:", M_GPU)
print("Retained edges:", K)

print("\nCORRECTNESS")
print("CPU coverage edges:", cpu_coverage)
print("GPU coverage edges:", gpu_coverage)
print("CPU/GPU masks exactly equal:", equal)
print("Different mask entries:", differences)
print("GPU isolated nodes:", isolates)

print("\nPERFORMANCE — SELECTOR ONLY")
print(
    f"CPU median: {cpu_median * 1000:.3f} ms"
)
print(
    f"GPU median: {gpu_median * 1000:.3f} ms"
)
print(
    f"Speedup: {speedup:.2f}x"
)

print("\nE3.7c COMPLETED")
print("Exact mask equivalence: PASSED")
print("Exact edge budget: PASSED")
print("Node coverage: PASSED")
print("No training performed.")
print("No previous artifacts modified.")


In [ ]:

# ============================================================
# PHASE III — E3.7d
# END-TO-END GPU SELECTOR EQUIVALENCE
# ============================================================
# Same trained checkpoint and training snapshot t=7.
# Compare CPU vs GPU selector for:
#   mask, logits, BCE, gradients, step latency.
#
# No optimizer steps.
# No validation/test labels accessed.
# No hidden simulator data accessed.
# ============================================================

from pathlib import Path
import copy
import json
import time
import numpy as np
import torch
import torch.nn.functional as F

assert all(name in globals() for name in [
    "stable", "data", "U", "V", "K", "OUT",
    "coverage_topk", "coverage_topk_gpu"
])

assert U.is_cuda and V.is_cuda
assert data[7]["xn"].is_cuda

SOURCE = (
    OUT / "e3_6d_extended_training" /
    "best_validation_checkpoint.pt"
)

REPORT_PATH = OUT / "e3_7d_gpu_equivalence.json"

if REPORT_PATH.exists():
    raise FileExistsError(
        f"Report already exists: {REPORT_PATH}"
    )

checkpoint = torch.load(
    SOURCE, map_location="cpu", weights_only=True
)

assert checkpoint["best_epoch"] == 60

audit_model = copy.deepcopy(stable).to(U.device)
audit_model.load_state_dict(
    checkpoint["model_state_dict"]
)

audit_model.eval()

batch = data[7]
M = int(U.numel())

def synchronize():
    torch.cuda.synchronize()

# ------------------------------------------------------------
# 1. Identical forward, differing only in selector
# ------------------------------------------------------------

def audit_forward(model, batch, selector):
    xn = batch["xn"]
    xe = batch["xe"]
    xc = batch["xc"]
    ids = batch["target"]

    full = xn.new_ones(M)

    h = model.encoder(xn, U, V, full)

    edge_rep = torch.cat([
        h[U] + h[V],
        torch.abs(h[U] - h[V]),
        xe
    ], dim=1)

    edge_score = model.scorer(
        edge_rep
    ).squeeze(-1)

    priority = model.controller(
        xc
    ).squeeze(-1)

    scores = edge_score + 0.5 * (
        priority[U] + priority[V]
    )

    hard, coverage_edges = selector(scores, K)

    # Identical straight-through gradient surrogate.
    threshold = torch.topk(
        scores.detach(), K
    ).values[-1]

    soft = torch.sigmoid(
        scores - threshold
    )

    mask = hard + (soft - soft.detach())

    z = model.encoder(xn, U, V, mask)

    a = U[ids]
    b = V[ids]

    pair = torch.cat([
        z[a] + z[b],
        torch.abs(z[a] - z[b]),
        z[a] * z[b]
    ], dim=1)

    logits = (
        model.trust_head(xe[ids]).squeeze(-1)
        + model.graph_head(pair).squeeze(-1)
    )

    loss = F.binary_cross_entropy_with_logits(
        logits, batch["y"]
    )

    return {
        "hard": hard,
        "coverage_edges": coverage_edges,
        "logits": logits,
        "loss": loss
    }

# ------------------------------------------------------------
# 2. Compare forward results and task gradients
# ------------------------------------------------------------

def run_with_grad(selector):
    audit_model.zero_grad(set_to_none=True)

    result = audit_forward(
        audit_model, batch, selector
    )

    result["loss"].backward()

    gradients = {
        name: p.grad.detach().clone()
        for name, p in audit_model.named_parameters()
        if p.grad is not None
    }

    return result, gradients

cpu_result, cpu_grad = run_with_grad(
    coverage_topk
)

gpu_result, gpu_grad = run_with_grad(
    coverage_topk_gpu
)

mask_equal = torch.equal(
    cpu_result["hard"],
    gpu_result["hard"]
)

logit_diff = float(
    (cpu_result["logits"]
     - gpu_result["logits"])
    .detach().abs().max().item()
)

loss_diff = abs(
    float(cpu_result["loss"].item())
    - float(gpu_result["loss"].item())
)

gradient_differences = {
    name: float(
        (cpu_grad[name] - gpu_grad[name])
        .abs().max().item()
    )
    for name in cpu_grad
}

max_gradient_diff = max(
    gradient_differences.values()
)

coverage_equal = (
    cpu_result["coverage_edges"]
    == gpu_result["coverage_edges"]
)

assert mask_equal
assert coverage_equal
assert int(gpu_result["hard"].sum().item()) == K
assert logit_diff < 1e-5
assert loss_diff < 1e-6

# CUDA reductions may have small numerical differences.
gradient_close = all(
    torch.allclose(
        cpu_grad[name],
        gpu_grad[name],
        atol=1e-4,
        rtol=1e-3
    )
    for name in cpu_grad
)

assert gradient_close, (
    "Backward gradients differ beyond tolerance; "
    "investigate before replacing the selector."
)

# ------------------------------------------------------------
# 3. Measure full forward + backward time
# ------------------------------------------------------------

def timed_step(selector):
    audit_model.zero_grad(set_to_none=True)

    result = audit_forward(
        audit_model, batch, selector
    )

    result["loss"].backward()

    return result

# Warm up both paths.
for _ in range(2):
    timed_step(coverage_topk)
    timed_step(coverage_topk_gpu)

def benchmark(selector, repeats=5):
    samples = []

    for _ in range(repeats):
        synchronize()
        start = time.perf_counter()

        timed_step(selector)

        synchronize()
        samples.append(
            time.perf_counter() - start
        )

    return np.array(samples)

cpu_times = benchmark(coverage_topk)
gpu_times = benchmark(coverage_topk_gpu)

cpu_median_ms = float(
    np.median(cpu_times) * 1000
)

gpu_median_ms = float(
    np.median(gpu_times) * 1000
)

speedup = cpu_median_ms / gpu_median_ms

# ------------------------------------------------------------
# 4. Save a new diagnostic report
# ------------------------------------------------------------

report = {
    "experiment": "E3.7d",
    "checkpoint_epoch": 60,
    "training_snapshot": 7,
    "mask_exactly_equal": bool(mask_equal),
    "coverage_count_equal": bool(coverage_equal),
    "coverage_edges": int(
        gpu_result["coverage_edges"]
    ),
    "maximum_absolute_logit_difference": logit_diff,
    "absolute_BCE_difference": loss_diff,
    "gradient_allclose": bool(gradient_close),
    "maximum_absolute_gradient_difference": (
        max_gradient_diff
    ),
    "cpu_median_step_ms": cpu_median_ms,
    "gpu_median_step_ms": gpu_median_ms,
    "end_to_end_step_speedup": speedup,
    "optimizer_steps": 0,
    "validation_accessed": False,
    "test_accessed": False,
    "hidden_state_accessed": False,
    "note": (
        "Timing covers one training forward and "
        "backward pass, but excludes optimizer.step "
        "and data preparation."
    )
}

with open(
    REPORT_PATH, "w", encoding="utf-8"
) as f:
    json.dump(report, f, indent=2)

print("=" * 72)
print("E3.7d — FULL GPU PIPELINE EQUIVALENCE")
print("=" * 72)

print("GPU:", torch.cuda.get_device_name(0))
print("Checkpoint epoch:", 60)
print("Mask exactly equal:", mask_equal)
print("Coverage counts equal:", coverage_equal)
print("Coverage edges:", report["coverage_edges"])
print("Max logit difference:", f"{logit_diff:.10f}")
print("BCE difference:", f"{loss_diff:.10f}")
print("Gradients close:", gradient_close)
print("Max gradient difference:",
      f"{max_gradient_diff:.10f}")

print("\nFORWARD + BACKWARD TIMING")
print(f"CPU selector: {cpu_median_ms:.3f} ms")
print(f"GPU selector: {gpu_median_ms:.3f} ms")
print(f"Speedup: {speedup:.2f}x")

print("\nE3.7d COMPLETED")
print("No optimizer updates.")
print("No test or validation labels accessed.")
print("Saved:", REPORT_PATH)


In [ ]:

# ============================================================
# PHASE III — E3.8
# DOES GNN-BASED EDGE SCORING ADD VALUE?
# ============================================================
# Independent trust-only edge selector baseline.
#
# GNN scorer (E3.7a):
#   Full-graph GraphSAGE + edge history -> edge scores
#
# Trust-only scorer (E3.8):
#   Edge history -> edge scores
#
# Both use GraphSAGE AFTER sparsification for prediction.
# Both have exact 50% budget and full node coverage.
#
# Train: t=5,6,7 | Validation: t=8,9
# 60 epochs | Model seed 42
#
# No controller, no test or hidden state access.
# ============================================================

from pathlib import Path
import copy
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    log_loss
)

required = [
    "stable", "data", "U", "V", "K",
    "OUT", "coverage_topk_gpu"
]

missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError(f"Missing objects: {missing}")

E38_OUT = OUT / "e3_8_trust_only_selector"

if E38_OUT.exists():
    raise FileExistsError(
        f"Existing output: {E38_OUT}"
    )

SEED = 42
EPOCHS = 60
LR = 1e-3
CLIP = 1.0
TAU = 1.0

TRAIN_TIMES = [5, 6, 7]
VAL_TIMES = [8, 9]

DEVICE = U.device
M = int(U.numel())
N = int(data[5]["xn"].shape[0])

assert U.is_cuda
assert V.is_cuda
assert K == 73058
assert M == 146117
assert N == 16216

torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# 1. Independent trust-only selection model
# ------------------------------------------------------------

model38 = copy.deepcopy(stable).to(DEVICE)

EDGE_DIM = int(data[5]["xe"].shape[1])
HIDDEN = 64

# Replace only the selection scorer.
# The post-sparsification GNN and prediction
# heads remain architecturally unchanged.
model38.scorer = nn.Sequential(
    nn.Linear(EDGE_DIM, HIDDEN),
    nn.ReLU(),
    nn.Linear(HIDDEN, 1)
).to(DEVICE)

# Controller is unused and excluded from training.
parameters38 = [
    p for name, p in model38.named_parameters()
    if not name.startswith("controller.")
]

optimizer38 = torch.optim.AdamW(
    parameters38,
    lr=LR,
    weight_decay=1e-4
)

# ------------------------------------------------------------
# 2. Trust-only selector forward
# ------------------------------------------------------------

def forward38(model, batch, training=False):
    xn = batch["xn"]
    xe = batch["xe"]
    ids = batch["target"]

    # Selection uses only causal edge evidence.
    # No full-graph GNN pass for edge scoring.
    scores = model.scorer(xe).squeeze(-1)

    hard, coverage_edges = coverage_topk_gpu(
        scores, K
    )

    if training:
        threshold = torch.topk(
            scores.detach(), K
        ).values[-1]

        soft = torch.sigmoid(
            (scores - threshold) / TAU
        )

        mask = hard + (soft - soft.detach())
    else:
        mask = hard

    # The prediction model STILL uses GNN.
    z = model.encoder(xn, U, V, mask)

    a = U[ids]
    b = V[ids]

    pair = torch.cat([
        z[a] + z[b],
        torch.abs(z[a] - z[b]),
        z[a] * z[b]
    ], dim=1)

    logits = (
        model.trust_head(xe[ids]).squeeze(-1)
        + model.graph_head(pair).squeeze(-1)
    )

    return {
        "logits": logits,
        "hard": hard,
        "coverage_edges": coverage_edges
    }

# ------------------------------------------------------------
# 3. Evaluation
# ------------------------------------------------------------

def validate38(model):
    model.eval()

    ys, ps, masks = [], [], []

    with torch.no_grad():
        for t in VAL_TIMES:
            result = forward38(
                model, data[t], training=False
            )

            ys.append(
                data[t]["y"].cpu().numpy()
            )

            ps.append(
                torch.sigmoid(
                    result["logits"]
                ).cpu().numpy()
            )

            mask = (
                result["hard"]
                .cpu().numpy()
                .astype(bool)
            )

            assert mask.sum() == K
            masks.append(mask)

    y = np.concatenate(ys)
    p = np.clip(
        np.concatenate(ps),
        1e-7, 1 - 1e-7
    )

    churn = (
        np.count_nonzero(masks[0] != masks[1])
        / (2 * K)
    )

    return {
        "AP": float(
            average_precision_score(y, p)
        ),
        "AUC": float(
            roc_auc_score(y, p)
        ),
        "BCE": float(
            log_loss(y, p, labels=[0, 1])
        ),
        "mask_churn": float(churn)
    }

# ------------------------------------------------------------
# 4. Fixed-budget training
# ------------------------------------------------------------

history38 = []
best_ap38 = -np.inf
best_epoch38 = None
best_metrics38 = None
best_state38 = None

print("=" * 72)
print("E3.8 — TRUST-ONLY EDGE SELECTION BASELINE")
print("=" * 72)
print("Device:", DEVICE)
print("Selection scorer uses GNN: NO")
print("Prediction uses GNN: YES")
print("Controller: NO")
print("Epochs:", EPOCHS)
print("Exact edge budget:", K)

for epoch in range(1, EPOCHS + 1):
    model38.train()
    losses = []

    for t in TRAIN_TIMES:
        batch = data[t]

        optimizer38.zero_grad(set_to_none=True)

        result = forward38(
            model38, batch, training=True
        )

        loss = F.binary_cross_entropy_with_logits(
            result["logits"], batch["y"]
        )

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite loss: epoch {epoch}"
            )

        loss.backward()

        norm = torch.nn.utils.clip_grad_norm_(
            parameters38, CLIP
        )

        if not torch.isfinite(norm):
            raise RuntimeError(
                f"Non-finite gradient: epoch {epoch}"
            )

        optimizer38.step()

        losses.append((
            float(loss.item()),
            int(batch["count"])
        ))

    train_bce = (
        sum(loss * n for loss, n in losses)
        / sum(n for _, n in losses)
    )

    val = validate38(model38)

    history38.append({
        "epoch": epoch,
        "train_BCE": train_bce,
        "val_AP": val["AP"],
        "val_AUC": val["AUC"],
        "val_BCE": val["BCE"],
        "val_mask_churn": val["mask_churn"]
    })

    if val["AP"] > best_ap38 + 1e-5:
        best_ap38 = val["AP"]
        best_epoch38 = epoch
        best_metrics38 = dict(val)
        best_state38 = copy.deepcopy(
            model38.state_dict()
        )

    if epoch == 1 or epoch % 5 == 0:
        print(
            f"epoch={epoch:02d} | "
            f"Train BCE={train_bce:.6f} | "
            f"Val AP={val['AP']:.6f} | "
            f"AUC={val['AUC']:.6f} | "
            f"BCE={val['BCE']:.6f} | "
            f"churn={val['mask_churn']:.4f}",
            flush=True
        )

assert best_state38 is not None

# ------------------------------------------------------------
# 5. Compare with previously trained GNN scorer
# ------------------------------------------------------------

with open(
    OUT / "e3_7a_no_controller_retrained"
    / "report.json",
    encoding="utf-8"
) as f:
    gnn_report = json.load(f)

gnn_metrics = gnn_report["best_validation"]

comparison = pd.DataFrame([
    {
        "method": "GNN-based scorer",
        **{
            k: float(gnn_metrics[k])
            for k in ["AP", "AUC", "BCE", "mask_churn"]
        }
    },
    {
        "method": "Trust-only scorer",
        **best_metrics38
    }
])

delta_ap = (
    float(gnn_metrics["AP"])
    - float(best_metrics38["AP"])
)

# ------------------------------------------------------------
# 6. Save without overwriting previous runs
# ------------------------------------------------------------

E38_OUT.mkdir(parents=True)

pd.DataFrame(history38).to_csv(
    E38_OUT / "epoch_history.csv",
    index=False
)

comparison.to_csv(
    E38_OUT / "comparison.csv",
    index=False
)

torch.save({
    "model_state_dict": best_state38,
    "best_epoch": best_epoch38,
    "model_seed": SEED,
    "scenario_seed": 101,
    "scorer_type": "Trust-only Edge MLP",
    "prediction_encoder": "Stable GraphSAGE",
    "controller_enabled": False
}, E38_OUT / "best_validation_checkpoint.pt")

report = {
    "experiment": "E3.8",
    "scenario": "B_owner_correlated",
    "scenario_seed": 101,
    "model_seed": SEED,
    "epochs": EPOCHS,
    "best_epoch": best_epoch38,
    "best_validation": best_metrics38,
    "gnn_scorer_reference": gnn_metrics,
    "gnn_minus_trust_only_AP": delta_ap,
    "exact_budget": K,
    "controller_used": False,
    "selector_device": str(DEVICE),
    "test_accessed": False,
    "hidden_state_accessed": False,
    "limitations": [
        "Single model seed and scenario realization.",
        "Both models still use GNN for prediction.",
        "Scorer input sizes differ.",
        "Validation repeatedly used for model selection.",
        "60 epochs chosen during exploratory development.",
        "No statistical significance established."
    ]
}

with open(
    E38_OUT / "report.json",
    "w", encoding="utf-8"
) as f:
    json.dump(report, f, indent=2)

print("\n" + "=" * 72)
print("E3.8 COMPLETED — SCORER ABLATION")
print("=" * 72)

print(comparison.round(6).to_string(index=False))

print(
    "\nGNN scorer minus Trust-only scorer AP:",
    f"{delta_ap:+.6f}"
)

print("No controller in either method.")
print("Both methods retain exactly", K, "edges.")
print("No test or hidden-state access.")
print("Saved:", E38_OUT / "report.json")


In [ ]:

# ============================================================
# E3.9 — INDEPENDENT MODEL-SEED REPLICATION
# GNN scorer vs Trust-only scorer
# ============================================================
# Scenario B, simulation seed 101, model seed 43.
# Fixed 60 epochs; no early stopping.
# Same initial shared parameters and data splits.
# GPU-only coverage selector.
# No controller, no test, no hidden simulator state.
# ============================================================

from pathlib import Path
import copy
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    log_loss
)

required = [
    "stable", "data", "U", "V", "K",
    "OUT", "coverage_topk_gpu"
]
missing = [x for x in required if x not in globals()]
assert not missing, f"Missing: {missing}"

SEED39 = 43
EPOCHS39 = 60
TRAIN39 = [5, 6, 7]
VAL39 = [8, 9]
DEVICE39 = U.device
M39 = U.numel()
N39 = data[5]["xn"].shape[0]
TAU39 = 1.0

SAVE39 = OUT / f"e3_9_seed_{SEED39}"

if SAVE39.exists():
    raise FileExistsError(
        f"Existing results: {SAVE39}"
    )

assert U.is_cuda and V.is_cuda
assert K == 73058
assert M39 == 146117
assert N39 == 16216

torch.manual_seed(SEED39)
np.random.seed(SEED39)
torch.cuda.manual_seed_all(SEED39)

# ------------------------------------------------------------
# 1. Matched initialization
# ------------------------------------------------------------

base39 = copy.deepcopy(stable).to(DEVICE39)

gnn39 = copy.deepcopy(base39)
trust39 = copy.deepcopy(base39)

edge_dim39 = data[5]["xe"].shape[1]

trust39.scorer = nn.Sequential(
    nn.Linear(edge_dim39, 64),
    nn.ReLU(),
    nn.Linear(64, 1)
).to(DEVICE39)

# Common parts start from exactly matching weights.
for name, value in gnn39.state_dict().items():
    if name.startswith(("encoder.", "trust_head.",
                        "graph_head.")):
        assert torch.equal(
            value, trust39.state_dict()[name]
        )

# Controllers are excluded in both models.
def active_parameters(model):
    return [
        p for name, p in model.named_parameters()
        if not name.startswith("controller.")
    ]

# ------------------------------------------------------------
# 2. Shared forward function
# ------------------------------------------------------------

def forward39(model, batch, use_gnn, train_mode):
    xn = batch["xn"]
    xe = batch["xe"]
    ids = batch["target"]

    if use_gnn:
        full = xn.new_ones(M39)
        h = model.encoder(xn, U, V, full)

        representation = torch.cat([
            h[U] + h[V],
            torch.abs(h[U] - h[V]),
            xe
        ], dim=1)

        scores = model.scorer(
            representation
        ).squeeze(-1)
    else:
        scores = model.scorer(xe).squeeze(-1)

    hard, cover_count = coverage_topk_gpu(
        scores, K
    )

    if train_mode:
        threshold = torch.topk(
            scores.detach(), K
        ).values[-1]

        soft = torch.sigmoid(
            (scores - threshold) / TAU39
        )

        mask = hard + (soft - soft.detach())
    else:
        mask = hard

    z = model.encoder(xn, U, V, mask)

    a = U[ids]
    b = V[ids]

    pair = torch.cat([
        z[a] + z[b],
        torch.abs(z[a] - z[b]),
        z[a] * z[b]
    ], dim=1)

    logits = (
        model.trust_head(xe[ids]).squeeze(-1)
        + model.graph_head(pair).squeeze(-1)
    )

    return logits, hard, cover_count


@torch.no_grad()
def evaluate39(model, use_gnn):
    model.eval()

    ys, ps, masks = [], [], []

    for t in VAL39:
        logits, mask, _ = forward39(
            model, data[t], use_gnn, False
        )

        ys.append(
            data[t]["y"].cpu().numpy()
        )

        ps.append(
            torch.sigmoid(logits).cpu().numpy()
        )

        masks.append(
            mask.cpu().numpy().astype(bool)
        )

    y = np.concatenate(ys)
    p = np.clip(
        np.concatenate(ps), 1e-7, 1 - 1e-7
    )

    assert all(mask.sum() == K for mask in masks)

    return {
        "AP": float(average_precision_score(y, p)),
        "AUC": float(roc_auc_score(y, p)),
        "BCE": float(log_loss(y, p, labels=[0, 1])),
        "mask_churn": float(
            np.count_nonzero(masks[0] != masks[1])
            / (2 * K)
        )
    }

# ------------------------------------------------------------
# 3. Independent 60-epoch training
# ------------------------------------------------------------

results39 = []
histories39 = {}
checkpoints39 = {}

for model_name, model, use_gnn in [
    ("GNN_scorer", gnn39, True),
    ("Trust_only_scorer", trust39, False)
]:

    # Reset RNG before training each model.
    torch.manual_seed(SEED39)
    np.random.seed(SEED39)
    torch.cuda.manual_seed_all(SEED39)

    params = active_parameters(model)

    optimizer = torch.optim.AdamW(
        params,
        lr=1e-3,
        weight_decay=1e-4
    )

    history_rows = []
    best_ap = -np.inf
    best_epoch = None
    best_metrics = None
    best_state = None

    print("\n" + "=" * 65)
    print("MODEL:", model_name)
    print("MODEL SEED:", SEED39)
    print("=" * 65, flush=True)

    for epoch in range(1, EPOCHS39 + 1):
        model.train()

        losses = []

        for t in TRAIN39:
            batch = data[t]

            optimizer.zero_grad(
                set_to_none=True
            )

            logits, hard, _ = forward39(
                model, batch, use_gnn, True
            )

            loss = F.binary_cross_entropy_with_logits(
                logits, batch["y"]
            )

            assert torch.isfinite(loss)
            loss.backward()

            norm = torch.nn.utils.clip_grad_norm_(
                params, 1.0
            )

            assert torch.isfinite(norm)
            optimizer.step()

            losses.append((
                loss.item(), batch["count"]
            ))

        train_bce = sum(
            x * n for x, n in losses
        ) / sum(n for _, n in losses)

        metrics = evaluate39(
            model, use_gnn
        )

        history_rows.append({
            "epoch": epoch,
            "train_BCE": train_bce,
            **metrics
        })

        if metrics["AP"] > best_ap + 1e-5:
            best_ap = metrics["AP"]
            best_epoch = epoch
            best_metrics = dict(metrics)
            best_state = copy.deepcopy(
                model.state_dict()
            )

        if epoch == 1 or epoch % 10 == 0:
            print(
                f"epoch={epoch:02d} | "
                f"AP={metrics['AP']:.6f} | "
                f"AUC={metrics['AUC']:.6f} | "
                f"BCE={metrics['BCE']:.6f}",
                flush=True
            )

    final_metrics = history_rows[-1]

    results39.append({
        "model": model_name,
        "seed": SEED39,
        "best_epoch": best_epoch,
        "best_AP": best_metrics["AP"],
        "best_AUC": best_metrics["AUC"],
        "best_BCE": best_metrics["BCE"],
        "epoch60_AP": final_metrics["AP"],
        "epoch60_AUC": final_metrics["AUC"],
        "epoch60_BCE": final_metrics["BCE"],
        "epoch60_churn": final_metrics["mask_churn"]
    })

    histories39[model_name] = history_rows
    checkpoints39[model_name] = best_state

# ------------------------------------------------------------
# 4. Save without overwriting prior experiments
# ------------------------------------------------------------

SAVE39.mkdir(parents=True)

results_df39 = pd.DataFrame(results39)

results_df39.to_csv(
    SAVE39 / "comparison.csv",
    index=False
)

for name in histories39:
    pd.DataFrame(
        histories39[name]
    ).to_csv(
        SAVE39 / f"{name}_history.csv",
        index=False
    )

    torch.save(
        {
            "model_state_dict": checkpoints39[name],
            "model_seed": SEED39,
            "model_type": name
        },
        SAVE39 / f"{name}_checkpoint.pt"
    )

gnn_row = results_df39[
    results_df39["model"] == "GNN_scorer"
].iloc[0]

trust_row = results_df39[
    results_df39["model"] == "Trust_only_scorer"
].iloc[0]

deltas39 = {
    metric: float(
        gnn_row[metric] - trust_row[metric]
    )
    for metric in [
        "best_AP", "best_AUC", "best_BCE",
        "epoch60_AP", "epoch60_AUC",
        "epoch60_BCE"
    ]
}

report39 = {
    "experiment": "E3.9",
    "scenario_seed": 101,
    "model_seed": SEED39,
    "training_epochs": EPOCHS39,
    "train_times": TRAIN39,
    "validation_times": VAL39,
    "gpu_selector": True,
    "controller": False,
    "matched_shared_initialization": True,
    "results": results39,
    "gnn_minus_trust_only": deltas39,
    "test_accessed": False,
    "hidden_state_accessed": False,
    "limitations": [
        "One additional model seed only.",
        "Shared scenario realization.",
        "Different scorer architectures.",
        "Both predictors still use GraphSAGE.",
        "Exploratory validation comparison."
    ]
}

with open(
    SAVE39 / "report.json",
    "w", encoding="utf-8"
) as f:
    json.dump(report39, f, indent=2)

print("\n" + "=" * 72)
print("E3.9 COMPLETED — SEED 43 REPLICATION")
print("=" * 72)

print(results_df39.round(6).to_string(index=False))

print("\nGNN MINUS TRUST-ONLY")
for metric, value in deltas39.items():
    print(f"{metric}: {value:+.6f}")

print("\nNo previous files overwritten.")
print("No test or hidden-state access.")
print("Saved:", SAVE39 / "report.json")


In [ ]:

# ============================================================
# E3.10 — GENUINELY FRESH MODEL INITIALIZATION
# ============================================================
# Scenario B / simulation seed 101 / model seed 44.
# GNN scorer versus Trust-only scorer.
# Same 60 epochs, data, optimizer and edge budget.
#
# Critical correction:
# Construct a NEW model after setting the RNG seed.
# Do NOT deepcopy the previously initialized stable
# object as the source of random initialization.
#
# No test or hidden-state access.
# ============================================================

from pathlib import Path
import copy
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    log_loss
)

assert all(name in globals() for name in [
    "stable", "data", "U", "V", "K", "OUT",
    "coverage_topk_gpu"
])

SEED310 = 44
EPOCHS310 = 60
TRAIN310 = [5, 6, 7]
VAL310 = [8, 9]

SAVE310 = OUT / "e3_10_fresh_seed_44"
if SAVE310.exists():
    raise FileExistsError(f"Existing output: {SAVE310}")

DEVICE310 = U.device
M310 = int(U.numel())

assert DEVICE310.type == "cuda"
assert K == 73058

torch.manual_seed(SEED310)
np.random.seed(SEED310)
torch.cuda.manual_seed_all(SEED310)

# ------------------------------------------------------------
# 1. Fresh model constructor
# ------------------------------------------------------------

sample = data[5]

node_dim = sample["xn"].shape[1]
edge_dim = sample["xe"].shape[1]
controller_dim = sample["xc"].shape[1]

# Reinstantiate the stable architecture.
base = type(stable)(
    node_dim=node_dim,
    edge_dim=edge_dim,
    controller_dim=controller_dim,
    hidden=32
).to(DEVICE310)

gnn_model = copy.deepcopy(base)
trust_model = copy.deepcopy(base)

# A different selection scorer is the only
# architectural difference.
trust_model.scorer = nn.Sequential(
    nn.Linear(edge_dim, 64),
    nn.ReLU(),
    nn.Linear(64, 1)
).to(DEVICE310)

# ------------------------------------------------------------
# 2. Verify fresh initialization and shared weights
# ------------------------------------------------------------

shared_prefixes = (
    "encoder.",
    "trust_head.",
    "graph_head."
)

shared_equal = all(
    torch.equal(
        value,
        trust_model.state_dict()[name]
    )
    for name, value in gnn_model.state_dict().items()
    if name.startswith(shared_prefixes)
)

assert shared_equal

# The new initialization must differ from the
# old stable prototype in at least one shared tensor.
fresh_difference = any(
    not torch.equal(
        value,
        stable.state_dict()[name]
    )
    for name, value in base.state_dict().items()
    if name.startswith(shared_prefixes)
)

assert fresh_difference, (
    "Fresh initialization unexpectedly matches "
    "the old prototype."
)

def trainable_parameters(model):
    return [
        p for name, p in model.named_parameters()
        if not name.startswith("controller.")
    ]

# ------------------------------------------------------------
# 3. Reuse the identical GPU training/evaluation logic
# ------------------------------------------------------------

def forward310(model, batch, use_gnn, training):
    xn, xe = batch["xn"], batch["xe"]
    target = batch["target"]

    if use_gnn:
        full = xn.new_ones(M310)
        h = model.encoder(xn, U, V, full)

        rep = torch.cat([
            h[U] + h[V],
            torch.abs(h[U] - h[V]),
            xe
        ], dim=1)

        scores = model.scorer(rep).squeeze(-1)
    else:
        scores = model.scorer(xe).squeeze(-1)

    hard, _ = coverage_topk_gpu(scores, K)

    if training:
        threshold = torch.topk(
            scores.detach(), K
        ).values[-1]

        soft = torch.sigmoid(scores - threshold)
        mask = hard + (soft - soft.detach())
    else:
        mask = hard

    z = model.encoder(xn, U, V, mask)

    a, b = U[target], V[target]

    pair = torch.cat([
        z[a] + z[b],
        torch.abs(z[a] - z[b]),
        z[a] * z[b]
    ], dim=1)

    logits = (
        model.trust_head(xe[target]).squeeze(-1)
        + model.graph_head(pair).squeeze(-1)
    )

    return logits, hard

@torch.no_grad()
def validate310(model, use_gnn):
    model.eval()

    ys, ps, masks = [], [], []

    for t in VAL310:
        logits, mask = forward310(
            model, data[t], use_gnn, False
        )

        ys.append(data[t]["y"].cpu().numpy())
        ps.append(torch.sigmoid(logits).cpu().numpy())
        masks.append(mask.cpu().numpy().astype(bool))

    y = np.concatenate(ys)
    p = np.clip(np.concatenate(ps), 1e-7, 1 - 1e-7)

    assert all(int(m.sum()) == K for m in masks)

    return {
        "AP": float(average_precision_score(y, p)),
        "AUC": float(roc_auc_score(y, p)),
        "BCE": float(log_loss(y, p, labels=[0, 1])),
        "churn": float(
            np.count_nonzero(masks[0] != masks[1])
            / (2 * K)
        )
    }

# ------------------------------------------------------------
# 4. Train each model independently
# ------------------------------------------------------------

results310 = []
histories310 = {}
checkpoints310 = {}

print("=" * 72)
print("E3.10 — FRESH-INITIALIZATION COMPARISON")
print("=" * 72)
print("Model seed:", SEED310)
print("Device:", DEVICE310)
print("Fresh weights differ from prototype:", fresh_difference)
print("Shared starting weights equal:", shared_equal)
print("Epochs:", EPOCHS310)

for name, model, use_gnn in [
    ("GNN_scorer", gnn_model, True),
    ("Trust_only_scorer", trust_model, False)
]:
    params = trainable_parameters(model)

    optimizer = torch.optim.AdamW(
        params,
        lr=1e-3,
        weight_decay=1e-4
    )

    rows = []
    best_ap = -np.inf
    best_epoch = None
    best_metrics = None
    best_state = None

    print(f"\nMODEL: {name}", flush=True)

    for epoch in range(1, EPOCHS310 + 1):
        model.train()
        losses = []

        for t in TRAIN310:
            optimizer.zero_grad(set_to_none=True)

            logits, mask = forward310(
                model, data[t], use_gnn, True
            )

            loss = F.binary_cross_entropy_with_logits(
                logits, data[t]["y"]
            )

            assert torch.isfinite(loss)
            loss.backward()

            norm = torch.nn.utils.clip_grad_norm_(
                params, 1.0
            )
            assert torch.isfinite(norm)

            optimizer.step()

            losses.append((
                float(loss.item()),
                int(data[t]["count"])
            ))

        train_bce = (
            sum(value * count for value, count in losses)
            / sum(count for _, count in losses)
        )

        metrics = validate310(model, use_gnn)

        rows.append({
            "epoch": epoch,
            "train_BCE": train_bce,
            **metrics
        })

        if metrics["AP"] > best_ap + 1e-5:
            best_ap = metrics["AP"]
            best_epoch = epoch
            best_metrics = dict(metrics)
            best_state = copy.deepcopy(
                model.state_dict()
            )

        if epoch == 1 or epoch % 10 == 0:
            print(
                f"epoch={epoch:02d} | "
                f"AP={metrics['AP']:.6f} | "
                f"AUC={metrics['AUC']:.6f} | "
                f"BCE={metrics['BCE']:.6f}",
                flush=True
            )

    results310.append({
        "model": name,
        "seed": SEED310,
        "best_epoch": best_epoch,
        "best_AP": best_metrics["AP"],
        "epoch60_AP": rows[-1]["AP"],
        "epoch60_AUC": rows[-1]["AUC"],
        "epoch60_BCE": rows[-1]["BCE"],
        "epoch60_churn": rows[-1]["churn"]
    })

    histories310[name] = rows
    checkpoints310[name] = best_state

# ------------------------------------------------------------
# 5. Save independently
# ------------------------------------------------------------

SAVE310.mkdir(parents=True)

comparison310 = pd.DataFrame(results310)
comparison310.to_csv(
    SAVE310 / "comparison.csv", index=False
)

for name, rows in histories310.items():
    pd.DataFrame(rows).to_csv(
        SAVE310 / f"{name}_history.csv",
        index=False
    )

    torch.save(
        {
            "model_state_dict": checkpoints310[name],
            "model_seed": SEED310,
            "model_type": name,
            "fresh_initialization": True
        },
        SAVE310 / f"{name}_checkpoint.pt"
    )

gnn = comparison310[
    comparison310["model"] == "GNN_scorer"
].iloc[0]

trust = comparison310[
    comparison310["model"] == "Trust_only_scorer"
].iloc[0]

delta_ap = float(
    gnn["epoch60_AP"] - trust["epoch60_AP"]
)

report310 = {
    "experiment": "E3.10",
    "model_seed": SEED310,
    "scenario_seed": 101,
    "epochs": EPOCHS310,
    "fresh_initialization_verified": fresh_difference,
    "shared_initialization_verified": shared_equal,
    "gnn_minus_trust_epoch60_AP": delta_ap,
    "results": results310,
    "test_accessed": False,
    "hidden_state_accessed": False,
    "limitations": [
        "Single new initialization seed.",
        "Same simulation realization.",
        "Different scorer architectures.",
        "Validation reused during development.",
        "GPU selector uses straight-through surrogate."
    ]
}

with open(
    SAVE310 / "report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(report310, f, indent=2)

print("\n" + "=" * 72)
print("E3.10 COMPLETED — FRESH SEED 44")
print("=" * 72)

print(comparison310.round(6).to_string(index=False))

print(
    "\nGNN minus Trust-only AP at epoch 60:",
    f"{delta_ap:+.6f}"
)

print("Fresh initialization: VERIFIED")
print("No test or hidden state accessed.")
print("Saved:", SAVE310 / "report.json")


In [ ]:

# ============================================================
# PHASE III — E3.11
# FRESH MODEL SEED 45: SCORER REPLICATION
# ============================================================
# Scenario B, simulator seed 101.
# Fresh model initialization, seed 45.
# Fixed 60 epochs, GPU selector, exact coverage.
# No controller. No test or hidden-state access.
#
# Requires E3.10 definitions in the current kernel.
# ============================================================

from pathlib import Path
import copy
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

assert all(name in globals() for name in [
    "stable", "data", "U", "V", "K", "OUT",
    "forward310", "validate310",
    "trainable_parameters", "coverage_topk_gpu"
])

SEED311 = 45
EPOCHS311 = 60
DEVICE311 = U.device

SAVE311 = OUT / "e3_11_fresh_seed_45"

if SAVE311.exists():
    raise FileExistsError(
        f"Previous experiment exists: {SAVE311}"
    )

assert DEVICE311.type == "cuda"
assert K == 73058

torch.manual_seed(SEED311)
np.random.seed(SEED311)
torch.cuda.manual_seed_all(SEED311)

# ------------------------------------------------------------
# 1. Fresh initialization
# ------------------------------------------------------------

sample = data[5]

node_dim = sample["xn"].shape[1]
edge_dim = sample["xe"].shape[1]
controller_dim = sample["xc"].shape[1]

base311 = type(stable)(
    node_dim=node_dim,
    edge_dim=edge_dim,
    controller_dim=controller_dim,
    hidden=32
).to(DEVICE311)

gnn311 = copy.deepcopy(base311)
trust311 = copy.deepcopy(base311)

trust311.scorer = nn.Sequential(
    nn.Linear(edge_dim, 64),
    nn.ReLU(),
    nn.Linear(64, 1)
).to(DEVICE311)

shared_names = [
    name for name in gnn311.state_dict()
    if name.startswith((
        "encoder.",
        "trust_head.",
        "graph_head."
    ))
]

assert all(
    torch.equal(
        gnn311.state_dict()[name],
        trust311.state_dict()[name]
    )
    for name in shared_names
)

# Verify independence from seed 44 initialization.
assert any(
    not torch.equal(
        base311.state_dict()[name],
        base.state_dict()[name]
    )
    for name in shared_names
), "Seed 45 initialization matches seed 44."

print("=" * 72)
print("E3.11 — FRESH SEED 45 REPLICATION")
print("=" * 72)
print("GPU:", torch.cuda.get_device_name(0))
print("Model seed:", SEED311)
print("Fresh initialization: VERIFIED")
print("Shared weights: MATCHED")
print("Epochs:", EPOCHS311)
print("Retained edges:", K)

# ------------------------------------------------------------
# 2. Fixed training protocol
# ------------------------------------------------------------

results311 = []
histories311 = {}
checkpoints311 = {}

for name, model, use_gnn in [
    ("GNN_scorer", gnn311, True),
    ("Trust_only_scorer", trust311, False)
]:

    parameters = trainable_parameters(model)

    optimizer = torch.optim.AdamW(
        parameters,
        lr=1e-3,
        weight_decay=1e-4
    )

    rows = []
    best_ap = -np.inf
    best_epoch = None
    best_state = None

    print(f"\nMODEL: {name}", flush=True)

    for epoch in range(1, EPOCHS311 + 1):
        model.train()
        losses = []

        for t in [5, 6, 7]:
            optimizer.zero_grad(set_to_none=True)

            logits, mask = forward310(
                model, data[t], use_gnn, True
            )

            loss = F.binary_cross_entropy_with_logits(
                logits, data[t]["y"]
            )

            if not torch.isfinite(loss):
                raise RuntimeError(
                    f"Non-finite loss at epoch {epoch}"
                )

            loss.backward()

            gradient = torch.nn.utils.clip_grad_norm_(
                parameters, 1.0
            )

            if not torch.isfinite(gradient):
                raise RuntimeError(
                    f"Non-finite gradient at epoch {epoch}"
                )

            optimizer.step()

            losses.append((
                float(loss.item()),
                int(data[t]["count"])
            ))

        metrics = validate310(model, use_gnn)

        train_bce = sum(
            value * count
            for value, count in losses
        ) / sum(
            count for _, count in losses
        )

        rows.append({
            "epoch": epoch,
            "train_BCE": train_bce,
            **metrics
        })

        if metrics["AP"] > best_ap + 1e-5:
            best_ap = metrics["AP"]
            best_epoch = epoch
            best_state = copy.deepcopy(
                model.state_dict()
            )

        if epoch == 1 or epoch % 10 == 0:
            print(
                f"epoch={epoch:02d} | "
                f"AP={metrics['AP']:.6f} | "
                f"AUC={metrics['AUC']:.6f} | "
                f"BCE={metrics['BCE']:.6f}",
                flush=True
            )

    final = rows[-1]

    results311.append({
        "model": name,
        "seed": SEED311,
        "best_epoch": best_epoch,
        "best_AP": best_ap,
        "epoch60_AP": final["AP"],
        "epoch60_AUC": final["AUC"],
        "epoch60_BCE": final["BCE"],
        "epoch60_churn": final["churn"]
    })

    histories311[name] = rows
    checkpoints311[name] = best_state

# ------------------------------------------------------------
# 3. Save new results without overwriting
# ------------------------------------------------------------

SAVE311.mkdir(parents=True)

comparison311 = pd.DataFrame(results311)

comparison311.to_csv(
    SAVE311 / "comparison.csv",
    index=False
)

for name, rows in histories311.items():

    pd.DataFrame(rows).to_csv(
        SAVE311 / f"{name}_history.csv",
        index=False
    )

    torch.save(
        {
            "model_state_dict": checkpoints311[name],
            "model_seed": SEED311,
            "model_type": name,
            "fresh_initialization": True
        },
        SAVE311 / f"{name}_checkpoint.pt"
    )

gnn_row = comparison311[
    comparison311["model"] == "GNN_scorer"
].iloc[0]

trust_row = comparison311[
    comparison311["model"] == "Trust_only_scorer"
].iloc[0]

delta = {
    metric: float(
        gnn_row[metric] - trust_row[metric]
    )
    for metric in [
        "epoch60_AP",
        "epoch60_AUC",
        "epoch60_BCE"
    ]
}

report311 = {
    "experiment": "E3.11",
    "scenario_seed": 101,
    "model_seed": SEED311,
    "training_epochs": EPOCHS311,
    "fresh_initialization": True,
    "matched_shared_initialization": True,
    "controller": False,
    "results": results311,
    "gnn_minus_trust_only": delta,
    "test_accessed": False,
    "hidden_state_accessed": False,
    "limitations": [
        "Single scenario realization.",
        "Both methods use GNN for prediction.",
        "Scorer architectures differ.",
        "Validation is exploratory."
    ]
}

with open(
    SAVE311 / "report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(report311, f, indent=2)

print("\n" + "=" * 72)
print("E3.11 COMPLETED — FRESH SEED 45")
print("=" * 72)

print(comparison311.round(6).to_string(index=False))

print("\nGNN MINUS TRUST-ONLY AT EPOCH 60")
for metric, value in delta.items():
    print(f"{metric}: {value:+.6f}")

print("\nNo previous artifacts overwritten.")
print("No test or hidden-state access.")
print("Saved:", SAVE311 / "report.json")


In [ ]:

# ============================================================
# PHASE III — E3.12
# FRESH MODEL SEED 46 — FINAL INITIALIZATION REPLICATION
# ============================================================
# Scenario B: simulation seed 101
# Model seed: 46
# GNN scorer vs Trust-only scorer
# 60 fixed epochs, GPU selector, exact 50% budget
# No controller, no test, no hidden simulator state
# ============================================================

from pathlib import Path
import copy
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

required = [
    "stable", "data", "U", "V", "K", "OUT",
    "forward310", "validate310",
    "trainable_parameters", "coverage_topk_gpu"
]

missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError(f"Missing objects: {missing}")

SEED312 = 46
EPOCHS312 = 60
SAVE312 = OUT / "e3_12_fresh_seed_46"

if SAVE312.exists():
    raise FileExistsError(
        f"Output already exists: {SAVE312}"
    )

assert U.is_cuda and V.is_cuda
assert K == 73058

torch.manual_seed(SEED312)
np.random.seed(SEED312)
torch.cuda.manual_seed_all(SEED312)

# ------------------------------------------------------------
# 1. Construct FRESH model after setting seed
# ------------------------------------------------------------

sample = data[5]

node_dim = sample["xn"].shape[1]
edge_dim = sample["xe"].shape[1]
controller_dim = sample["xc"].shape[1]

base312 = type(stable)(
    node_dim=node_dim,
    edge_dim=edge_dim,
    controller_dim=controller_dim,
    hidden=32
).to(U.device)

gnn312 = copy.deepcopy(base312)
trust312 = copy.deepcopy(base312)

trust312.scorer = nn.Sequential(
    nn.Linear(edge_dim, 64),
    nn.ReLU(),
    nn.Linear(64, 1)
).to(U.device)

shared_names = [
    name for name in gnn312.state_dict()
    if name.startswith((
        "encoder.", "trust_head.", "graph_head."
    ))
]

assert all(
    torch.equal(
        gnn312.state_dict()[name],
        trust312.state_dict()[name]
    )
    for name in shared_names
)

# Confirm new initialization differs from seed 45.
assert "base311" in globals()

assert any(
    not torch.equal(
        base312.state_dict()[name],
        base311.state_dict()[name]
    )
    for name in shared_names
)

print("=" * 72)
print("E3.12 — FRESH SEED 46 REPLICATION")
print("=" * 72)
print("GPU:", torch.cuda.get_device_name(0))
print("Seed:", SEED312)
print("Fresh initialization: VERIFIED")
print("Shared model weights: MATCHED")
print("Epochs:", EPOCHS312)
print("Exact retained edges:", K)

# ------------------------------------------------------------
# 2. Train two models independently
# ------------------------------------------------------------

results312 = []
histories312 = {}
checkpoints312 = {}

for name, net, use_gnn in [
    ("GNN_scorer", gnn312, True),
    ("Trust_only_scorer", trust312, False)
]:

    params = trainable_parameters(net)

    optimizer = torch.optim.AdamW(
        params,
        lr=1e-3,
        weight_decay=1e-4
    )

    rows = []
    best_ap = -float("inf")
    best_epoch = None
    best_state = None

    print(f"\nMODEL: {name}", flush=True)

    for epoch in range(1, EPOCHS312 + 1):

        net.train()
        losses = []

        for t in [5, 6, 7]:

            optimizer.zero_grad(
                set_to_none=True
            )

            logits, mask = forward310(
                net, data[t], use_gnn, True
            )

            loss = F.binary_cross_entropy_with_logits(
                logits, data[t]["y"]
            )

            assert torch.isfinite(loss)

            loss.backward()

            grad = torch.nn.utils.clip_grad_norm_(
                params, max_norm=1.0
            )

            assert torch.isfinite(grad)
            optimizer.step()

            losses.append((
                float(loss.item()),
                int(data[t]["count"])
            ))

        metrics = validate310(net, use_gnn)

        train_bce = sum(
            value * count for value, count in losses
        ) / sum(
            count for _, count in losses
        )

        rows.append({
            "epoch": epoch,
            "train_BCE": train_bce,
            **metrics
        })

        if metrics["AP"] > best_ap + 1e-5:
            best_ap = metrics["AP"]
            best_epoch = epoch
            best_state = copy.deepcopy(
                net.state_dict()
            )

        if epoch == 1 or epoch % 10 == 0:
            print(
                f"epoch={epoch:02d} | "
                f"AP={metrics['AP']:.6f} | "
                f"AUC={metrics['AUC']:.6f} | "
                f"BCE={metrics['BCE']:.6f}",
                flush=True
            )

    final = rows[-1]

    results312.append({
        "model": name,
        "seed": SEED312,
        "best_epoch": best_epoch,
        "best_AP": best_ap,
        "epoch60_AP": final["AP"],
        "epoch60_AUC": final["AUC"],
        "epoch60_BCE": final["BCE"],
        "epoch60_churn": final["churn"]
    })

    histories312[name] = rows
    checkpoints312[name] = {
        "best": best_state,
        "epoch60": copy.deepcopy(net.state_dict())
    }

# ------------------------------------------------------------
# 3. Persist results without overwriting
# ------------------------------------------------------------

SAVE312.mkdir(parents=True)

comparison312 = pd.DataFrame(results312)

comparison312.to_csv(
    SAVE312 / "comparison.csv",
    index=False
)

for name, rows in histories312.items():

    pd.DataFrame(rows).to_csv(
        SAVE312 / f"{name}_history.csv",
        index=False
    )

    torch.save(
        {
            "best_model_state_dict":
                checkpoints312[name]["best"],
            "epoch60_model_state_dict":
                checkpoints312[name]["epoch60"],
            "seed": SEED312,
            "model": name,
            "training_epochs": EPOCHS312
        },
        SAVE312 / f"{name}_checkpoint.pt"
    )

gnn_row = comparison312[
    comparison312["model"] == "GNN_scorer"
].iloc[0]

trust_row = comparison312[
    comparison312["model"] == "Trust_only_scorer"
].iloc[0]

deltas = {
    metric: float(
        gnn_row[metric] - trust_row[metric]
    )
    for metric in [
        "epoch60_AP",
        "epoch60_AUC",
        "epoch60_BCE",
        "epoch60_churn"
    ]
}

report312 = {
    "experiment": "E3.12",
    "scenario": "B_owner_correlated",
    "simulation_seed": 101,
    "model_seed": 46,
    "epochs": 60,
    "fresh_initialization_verified": True,
    "shared_initialization_verified": True,
    "results": results312,
    "gnn_minus_trust_only": deltas,
    "controller_enabled": False,
    "gpu_selector_enabled": True,
    "test_accessed": False,
    "hidden_state_accessed": False,
    "limitations": [
        "One simulation realization only.",
        "Different scorer input dimensions.",
        "GNN is used in both prediction heads.",
        "Validation has guided earlier development."
    ]
}

with open(
    SAVE312 / "report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(report312, f, indent=2)

print("\n" + "=" * 72)
print("E3.12 COMPLETED — FRESH SEED 46")
print("=" * 72)

print(
    comparison312.round(6).to_string(index=False)
)

print("\nGNN MINUS TRUST-ONLY AT EPOCH 60")

for metric, value in deltas.items():
    print(f"{metric}: {value:+.6f}")

print("\nFresh initialization: VERIFIED")
print("Previous artifacts preserved: YES")
print("No test or hidden-state access.")
print("Saved:", SAVE312 / "report.json")


In [ ]:

# ============================================================
# PHASE III — E3.13
# FRESH MODEL-SEED AGGREGATION
# ============================================================
# Aggregate E3.10, E3.11, E3.12.
# Scenario B / simulation seed 101.
# Fresh model seeds: 44, 45, 46.
# Fixed checkpoint: epoch 60.
#
# No training, validation prediction, or test access.
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path("/kaggle/working/phase2_adaptive_siot")

BASE = (
    ROOT / "E3_tags_v3" / "scenarios"
    / "B_owner_correlated" / "seed_101"
    / "tags_v3"
)

sources = {
    44: BASE / "e3_10_fresh_seed_44" / "report.json",
    45: BASE / "e3_11_fresh_seed_45" / "report.json",
    46: BASE / "e3_12_fresh_seed_46" / "report.json"
}

OUT313 = BASE / "e3_13_fresh_seed_aggregation"

if OUT313.exists():
    raise FileExistsError(
        f"Output already exists: {OUT313}"
    )

records = []

for seed, path in sources.items():

    assert path.is_file(), f"Missing: {path}"

    with open(path, encoding="utf-8") as f:
        report = json.load(f)

    items = {
        row["model"]: row
        for row in report["results"]
    }

    gnn = items["GNN_scorer"]
    trust = items["Trust_only_scorer"]

    assert int(gnn["seed"]) == seed
    assert int(trust["seed"]) == seed

    for metric in [
        "AP", "AUC", "BCE", "churn"
    ]:
        field = f"epoch60_{metric}"

        g = float(gnn[field])
        r = float(trust[field])

        records.append({
            "seed": seed,
            "metric": metric,
            "GNN": g,
            "Trust_only": r,
            "GNN_minus_Trust": g - r
        })

paired = pd.DataFrame(records)

summary = (
    paired.groupby("metric", sort=False)
    .agg(
        GNN_mean=("GNN", "mean"),
        Trust_mean=("Trust_only", "mean"),
        delta_mean=("GNN_minus_Trust", "mean"),
        delta_sd=("GNN_minus_Trust", "std"),
        min_delta=("GNN_minus_Trust", "min"),
        max_delta=("GNN_minus_Trust", "max"),
        n=("seed", "nunique")
    )
    .reset_index()
)

assert (summary["n"] == 3).all()

# Explicitly descriptive.
# Three initializations from ONE simulated scenario
# are not independent scenario replications.

report313 = {
    "experiment": "E3.13",
    "scenario": "B_owner_correlated",
    "simulation_seed": 101,
    "model_seeds": [44, 45, 46],
    "evaluation_checkpoint": "Epoch 60",
    "metrics": summary.to_dict(orient="records"),
    "raw_paired_results": paired.to_dict(
        orient="records"
    ),
    "interpretation": (
        "Exploratory paired model-initialization "
        "comparison within one simulation realization."
    ),
    "no_significance_claim": True,
    "test_accessed": False,
    "limitations": [
        "Only three fresh model initializations.",
        "All runs share simulation seed 101.",
        "Validation data informed prior development.",
        "Both methods use GraphSAGE for prediction.",
        "Scorer architectures differ.",
        "Training budget was chosen exploratorily."
    ]
}

OUT313.mkdir(parents=True)

paired.to_csv(
    OUT313 / "paired_results.csv",
    index=False
)

summary.to_csv(
    OUT313 / "summary.csv",
    index=False
)

with open(
    OUT313 / "report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(report313, f, indent=2)

print("=" * 72)
print("E3.13 — FRESH MODEL-SEED AGGREGATION")
print("=" * 72)

print("Scenario seed: 101")
print("Fresh model seeds: 44, 45, 46")
print("Checkpoint: Epoch 60")

print("\nPAIRED AP RESULTS")
print(
    paired[paired.metric == "AP"][
        ["seed", "GNN", "Trust_only", "GNN_minus_Trust"]
    ].round(6).to_string(index=False)
)

print("\nAGGREGATE RESULTS")
print(
    summary.round(6).to_string(index=False)
)

print("\nE3.13 COMPLETED")
print("No training performed.")
print("No test or hidden-state access.")
print("Saved:", OUT313 / "report.json")


In [ ]:

# ============================================================
# PHASE III — E3.14
# SCENARIO B — INDEPENDENT SIMULATION SEED 102
# ============================================================
# Replicate E3.1 with the SAME simulator parameters
# and RNG call order; only the seed changes.
#
# No training, no test evaluation, no prior overwrite.
# ============================================================

from pathlib import Path
import json
import zipfile
import hashlib
import numpy as np
import pandas as pd

ROOT = Path("/kaggle/working/phase2_adaptive_siot")
E3 = ROOT / "E3_tags_v3"

with open(
    E3 / "e3_0_research_protocol_v1.json",
    encoding="utf-8"
) as f:
    protocol = json.load(f)

SEED = 102
assert SEED in protocol["evaluation"]["scenario_seeds"]

N = 16216
T = 12
INTERACTION_RATE = 0.15
OWNER_EFFECT_STD = 0.65
EDGE_EFFECT_STD = 0.45
TEMPORAL_RHO = 0.85
TEMPORAL_STD = 0.12
OBSERVATION_FLIP = 0.03
SHOCK_START = 6
SHOCK_FRACTION = 0.08
SHOCK_LOGIT = -1.6
SHOCK_DECAY = 0.90

SOURCE = (
    E3 / "scenarios" /
    "B_owner_correlated" / "seed_101"
)

OUT = (
    E3 / "scenarios" /
    "B_owner_correlated" / f"seed_{SEED}"
)

if OUT.exists():
    raise FileExistsError(
        f"Output already exists: {OUT}"
    )

# Verify generator parameters against E3.1.
with open(
    SOURCE / "e3_1_scenario_b_protocol.json",
    encoding="utf-8"
) as f:
    reference = json.load(f)

params = {
    "base_reliability": "Beta(5,2)",
    "owner_effect_std": OWNER_EFFECT_STD,
    "edge_effect_std": EDGE_EFFECT_STD,
    "temporal_rho": TEMPORAL_RHO,
    "temporal_std": TEMPORAL_STD,
    "interaction_rate": INTERACTION_RATE,
    "observation_flip": OBSERVATION_FLIP,
    "shock_start": SHOCK_START,
    "shock_fraction": SHOCK_FRACTION,
    "shock_logit": SHOCK_LOGIT,
    "shock_decay": SHOCK_DECAY
}

assert reference["parameters"] == params, (
    "Simulator parameters differ from E3.1."
)

rng = np.random.default_rng(SEED)

# ------------------------------------------------------------
# 1. Original observed SIoT topology
# ------------------------------------------------------------

topology = pd.read_csv(
    SOURCE / "observed" /
    "siot_social_topology.csv.gz"
).sort_values(["u", "v"]).reset_index(drop=True)

M = len(topology)
assert M == 146117

u = topology["u"].to_numpy(np.int32) - 1
v = topology["v"].to_numpy(np.int32) - 1

assert np.all(u < v)
assert u.min() >= 0
assert v.max() < N

# ------------------------------------------------------------
# 2. Original observed owner metadata
# ------------------------------------------------------------

archive = (
    ROOT / "E1_trust_scoring" /
    "santander" / "raw" /
    "objects_description.zip"
)

with zipfile.ZipFile(archive) as z:
    with z.open("objects_description.csv") as f:
        devices = pd.read_csv(f)

devices = devices.sort_values(
    "id_device"
).reset_index(drop=True)

assert np.array_equal(
    devices["id_device"].to_numpy(),
    np.arange(1, N + 1)
)

owner_codes, owner_ids = pd.factorize(
    devices["id_user"], sort=True
)

O = len(owner_ids)
same_owner = owner_codes[u] == owner_codes[v]

assert O == 4001
assert int(same_owner.sum()) == 65667

# ------------------------------------------------------------
# 3. Generate hidden trust state
# ------------------------------------------------------------

def sigmoid(x):
    return 1.0 / (
        1.0 + np.exp(-np.clip(x, -30, 30))
    )

def logit(p):
    p = np.clip(p, 1e-6, 1.0 - 1e-6)
    return np.log(p / (1.0 - p))

base_reliability = rng.beta(
    5.0, 2.0, size=N
)

owner_effect = rng.normal(
    0.0, OWNER_EFFECT_STD, size=O
)

device_owner_effect = owner_effect[
    owner_codes
]

edge_effect = rng.normal(
    0.0, EDGE_EFFECT_STD, size=M
)

base_logit = logit(base_reliability)

shock_count = int(round(
    SHOCK_FRACTION * N
))

shocked_nodes = rng.choice(
    N,
    size=shock_count,
    replace=False
)

shock_indicator = np.zeros(
    N, dtype=np.float32
)
shock_indicator[shocked_nodes] = 1.0

temporal_state = np.zeros(
    N, dtype=np.float32
)

event_frames = []
audit_rows = []

print("=" * 72)
print("E3.14 — SCENARIO B, SIMULATION SEED 102")
print("=" * 72)
print("Nodes:", N)
print("Owners:", O)
print("Edges:", M)
print("Owner effect std:", OWNER_EFFECT_STD)

# ------------------------------------------------------------
# 4. Same RNG sequence and interaction generator as E3.1
# ------------------------------------------------------------

for t in range(T):

    temporal_state = (
        TEMPORAL_RHO * temporal_state
        + rng.normal(
            0, TEMPORAL_STD, size=N
        )
    ).astype(np.float32)

    if t >= SHOCK_START:
        shock_strength = (
            SHOCK_LOGIT
            * SHOCK_DECAY ** (t - SHOCK_START)
        )
    else:
        shock_strength = 0.0

    hidden_device_logit = (
        base_logit
        + device_owner_effect
        + temporal_state
        + shock_strength * shock_indicator
    )

    hidden_link_logit = (
        0.5 * (
            hidden_device_logit[u]
            + hidden_device_logit[v]
        )
        + edge_effect
    )

    success_probability = sigmoid(
        hidden_link_logit
    )

    selected = np.flatnonzero(
        rng.random(M) < INTERACTION_RATE
    )

    success = (
        rng.random(len(selected))
        < success_probability[selected]
    ).astype(np.int8)

    flip = (
        rng.random(len(selected))
        < OBSERVATION_FLIP
    )

    success[flip] = 1 - success[flip]

    frame = pd.DataFrame({
        "time": np.full(
            len(selected), t, dtype=np.int16
        ),
        "source": (
            u[selected] + 1
        ).astype(np.int32),
        "target": (
            v[selected] + 1
        ).astype(np.int32),
        "success": success
    })

    event_frames.append(frame)

    audit_rows.append({
        "time": t,
        "observations": int(len(selected)),
        "observed_success_rate": float(
            success.mean()
        ),
        "hidden_mean_link_success_probability": float(
            success_probability.mean()
        ),
        "shock_strength": float(shock_strength)
    })

    if t <= 9:
        print(
            f"t={t:02d} | "
            f"observations={len(selected):6,d} | "
            f"success={success.mean():.4f}"
        )
    else:
        print(
            f"t={t:02d} | "
            "TEST GENERATED — METRICS SEALED"
        )

# ------------------------------------------------------------
# 5. Strictly separated data splits
# ------------------------------------------------------------

events = pd.concat(
    event_frames,
    ignore_index=True
).sort_values(
    ["time", "source", "target"]
).reset_index(drop=True)

assert not events[
    ["time", "source", "target"]
].duplicated().any()

train = events[events["time"] <= 7].copy()

validation = events[
    events["time"].between(8, 9)
].copy()

test = events[
    events["time"] >= 10
].copy()

assert len(events) == (
    len(train) + len(validation) + len(test)
)

# ------------------------------------------------------------
# 6. Write new scenario to its own directory
# ------------------------------------------------------------

OBS = OUT / "observed"
HIDDEN = OUT / "simulation_ground_truth"

OUT.mkdir(parents=True)
OBS.mkdir()
HIDDEN.mkdir()

topology.to_csv(
    OBS / "siot_social_topology.csv.gz",
    index=False,
    compression="gzip"
)

events.to_csv(
    OBS / "all_interactions.csv.gz",
    index=False,
    compression="gzip"
)

train.to_csv(
    OBS / "train.csv.gz",
    index=False,
    compression="gzip"
)

validation.to_csv(
    OBS / "validation.csv.gz",
    index=False,
    compression="gzip"
)

test.to_csv(
    OBS / "test.csv.gz",
    index=False,
    compression="gzip"
)

np.savez_compressed(
    HIDDEN / "hidden_simulator_state.npz",
    base_reliability=base_reliability,
    owner_effect=owner_effect,
    owner_codes=owner_codes,
    edge_effect=edge_effect,
    shocked_device_ids=shocked_nodes + 1
)

pd.DataFrame(audit_rows).to_csv(
    HIDDEN / "latent_temporal_summary.csv",
    index=False
)

# ------------------------------------------------------------
# 7. Reproducibility report
# ------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(chunk)

    return h.hexdigest()

metadata = {
    "experiment": "E3.14",
    "scenario": "B_owner_correlated",
    "seed": SEED,
    "reference_generator": "E3.1",
    "only_changed_parameter": "simulation seed",
    "parameters": params,
    "nodes": N,
    "owners": O,
    "undirected_edges": M,
    "train_events": int(len(train)),
    "validation_events": int(len(validation)),
    "total_events": int(len(events)),
    "topology_identity_verified": True,
    "integrity": {
        "train_sha256": sha256_file(
            OBS / "train.csv.gz"
        ),
        "validation_sha256": sha256_file(
            OBS / "validation.csv.gz"
        )
    },
    "test_generated_but_not_evaluated": True,
    "hidden_state_not_used_for_learning": True,
    "model_training_performed": False,
    "notes": [
        "Same topology and owner mapping as seed 101.",
        "Same simulator parameters and RNG call order.",
        "Trust observations are synthetic.",
        "Do not read test outcomes during development."
    ]
}

with open(
    OUT / "e3_14_scenario_b_seed102_report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(metadata, f, indent=2)

print("\n" + "=" * 72)
print("E3.14 COMPLETED — SCENARIO B SEED 102")
print("=" * 72)

print("Devices:", N)
print("Owners:", O)
print("Social edges:", M)
print("Total interactions:", len(events))
print("Train interactions:", len(train))
print("Validation interactions:", len(validation))
print("Test interactions: SEALED")
print("Parameter changes other than seed: NONE")

print("\nOutput:", OUT)
print("Previous scenario preserved: YES")
print("No GNN training performed.")
print("No test outcomes evaluated.")
print("NEXT: E3.15 — Causal features for seed 102.")


In [ ]:

# ============================================================
# PHASE III — E3.15
# CAUSAL TRUST FEATURES: SCENARIO B / SEED 102
# ============================================================
# Replicates E3.2 feature construction.
# 12 features per edge, snapshots t=0,...,9.
# History at t includes ONLY observations before t.
#
# No test, hidden simulator state, or model training.
# No previous artifacts overwritten.
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path("/kaggle/working/phase2_adaptive_siot")

SCENARIO = (
    ROOT / "E3_tags_v3" / "scenarios" /
    "B_owner_correlated" / "seed_102"
)

REFERENCE = (
    ROOT / "E3_tags_v3" / "scenarios" /
    "B_owner_correlated" / "seed_101"
)

OBS = SCENARIO / "observed"
OUT315 = SCENARIO / "causal_features"

if OUT315.exists():
    raise FileExistsError(
        f"Existing output: {OUT315}"
    )

N = 16216
M = 146117
GAMMA = 0.80
CONFIDENCE_K = 5.0
HORIZON = 12

FEATURE_NAMES = [
    "edge_history_count",
    "edge_success_rate",
    "edge_decayed_count",
    "edge_decayed_success_rate",
    "edge_evidence_confidence",
    "edge_age_normalized",
    "edge_never_observed",
    "node_u_success_rate",
    "node_v_success_rate",
    "node_u_evidence_confidence",
    "node_v_evidence_confidence",
    "edge_recent_minus_longterm"
]

# ------------------------------------------------------------
# 1. Verify the feature specification
# ------------------------------------------------------------

with open(
    REFERENCE / "causal_features" /
    "e3_2_causal_feature_audit.json",
    encoding="utf-8"
) as f:
    reference_report = json.load(f)

assert reference_report["feature_names"] == FEATURE_NAMES
assert reference_report["feature_count"] == 12
assert reference_report["temporal_decay"] == GAMMA
assert reference_report["evidence_confidence_k"] == CONFIDENCE_K
assert reference_report["age_normalization_horizon"] == HORIZON

# ------------------------------------------------------------
# 2. Read observed topology and permitted outcomes
# ------------------------------------------------------------

topology = pd.read_csv(
    OBS / "siot_social_topology.csv.gz"
).sort_values(["u", "v"]).reset_index(drop=True)

train = pd.read_csv(OBS / "train.csv.gz")
val = pd.read_csv(OBS / "validation.csv.gz")

assert len(topology) == M
assert train["time"].between(0, 7).all()
assert val["time"].between(8, 9).all()

# Verify identical topology and relation columns.
reference_topology = pd.read_csv(
    REFERENCE / "causal_features" /
    "edge_index_mapping.csv.gz"
).sort_values(["u", "v"]).reset_index(drop=True)

pd.testing.assert_frame_equal(
    topology.reset_index(drop=True),
    reference_topology.reset_index(drop=True),
    check_dtype=True
)

events = pd.concat(
    [train, val], ignore_index=True
)

assert events["time"].between(0, 9).all()
assert events["success"].isin([0, 1]).all()

u = topology["u"].to_numpy(np.int64) - 1
v = topology["v"].to_numpy(np.int64) - 1

assert np.all(u < v)
assert u.min() >= 0
assert v.max() < N

edge_keys = u * N + v
assert np.all(np.diff(edge_keys) > 0)

# ------------------------------------------------------------
# 3. Match interaction records to canonical edges
# ------------------------------------------------------------

eu = events["source"].to_numpy(np.int64) - 1
ev = events["target"].to_numpy(np.int64) - 1

assert np.all(eu < ev)

event_keys = eu * N + ev
event_ids = np.searchsorted(edge_keys, event_keys)

assert np.all(event_ids < M)
assert np.array_equal(
    edge_keys[event_ids], event_keys
)

event_time = events["time"].to_numpy(np.int16)
event_success = events["success"].to_numpy(np.int8)

unique_keys = (
    event_time.astype(np.int64) * M
    + event_ids
)
assert len(np.unique(unique_keys)) == len(events)

# ------------------------------------------------------------
# 4. Initialize causal historical accumulators
# ------------------------------------------------------------

edge_s = np.zeros(M, dtype=np.float32)
edge_f = np.zeros(M, dtype=np.float32)

decay_s = np.zeros(M, dtype=np.float32)
decay_f = np.zeros(M, dtype=np.float32)

node_s = np.zeros(N, dtype=np.float32)
node_f = np.zeros(N, dtype=np.float32)

last_seen = np.full(M, -1, dtype=np.int16)

def rate(s, f):
    return (s + 1.0) / (s + f + 2.0)

def confidence(s, f):
    count = s + f
    return count / (count + CONFIDENCE_K)

# Check output location before making any files.
OUT315.mkdir(parents=True)

audit = []
cumulative = 0

print("=" * 72)
print("E3.15 — CAUSAL FEATURES, SCENARIO B SEED 102")
print("=" * 72)
print("Nodes:", N)
print("Edges:", M)
print("Feature dimensions:", len(FEATURE_NAMES))
print("Temporal decay:", GAMMA)
print("Snapshots: t=0,...,9")
print("Reference specification: E3.2 VERIFIED")
print("Test accessed: NO")

# ------------------------------------------------------------
# 5. Generate features BEFORE updating with time-t data
# ------------------------------------------------------------

for t in range(10):

    # Same temporal decay convention as E3.2.
    if t > 0:
        decay_s *= GAMMA
        decay_f *= GAMMA

    history_count = edge_s + edge_f
    history_rate = rate(edge_s, edge_f)

    decayed_count = decay_s + decay_f
    decayed_rate = rate(decay_s, decay_f)

    edge_conf = confidence(edge_s, edge_f)

    node_rate = rate(node_s, node_f)
    node_conf = confidence(node_s, node_f)

    unseen = last_seen == -1

    age = np.where(
        unseen,
        HORIZON,
        t - last_seen
    ).astype(np.float32)

    age_normalized = age / HORIZON

    X = np.column_stack([
        history_count,
        history_rate,
        decayed_count,
        decayed_rate,
        edge_conf,
        age_normalized,
        unseen.astype(np.float32),
        node_rate[u],
        node_rate[v],
        node_conf[u],
        node_conf[v],
        decayed_rate - history_rate
    ]).astype(np.float32)

    assert X.shape == (M, 12)
    assert np.isfinite(X).all()

    # No current or future interaction may
    # contribute to historical features.
    assert int(history_count.sum()) == cumulative

    if t == 0:
        assert np.all(X[:, 0] == 0)
        assert np.all(X[:, 1] == 0.5)
        assert np.all(X[:, 6] == 1)

    filename = f"snapshot_t{t:02d}.npz"

    np.savez_compressed(
        OUT315 / filename,
        features=X,
        time=np.array(t, dtype=np.int16),
        feature_names=np.array(FEATURE_NAMES)
    )

    record = {
        "time": t,
        "history_before_t": int(cumulative),
        "edges_with_history": int(
            (history_count > 0).sum()
        ),
        "mean_historical_success_rate": float(
            history_rate.mean()
        ),
        "mean_evidence_confidence": float(
            edge_conf.mean()
        ),
        "snapshot": filename
    }

    # Apply the observed outcomes at t ONLY
    # after snapshot t is constructed and saved.
    current = event_time == t

    ids = event_ids[current]
    successes = event_success[current].astype(
        np.float32
    )
    failures = 1.0 - successes

    np.add.at(edge_s, ids, successes)
    np.add.at(edge_f, ids, failures)

    np.add.at(decay_s, ids, successes)
    np.add.at(decay_f, ids, failures)

    np.add.at(node_s, u[ids], successes)
    np.add.at(node_s, v[ids], successes)

    np.add.at(node_f, u[ids], failures)
    np.add.at(node_f, v[ids], failures)

    last_seen[ids] = t

    cumulative += len(ids)

    record["observations_at_t"] = int(len(ids))
    audit.append(record)

    print(
        f"t={t:02d} | "
        f"past={record['history_before_t']:7,d} | "
        f"observed_edges="
        f"{record['edges_with_history']:7,d} | "
        f"current={len(ids):6,d}",
        flush=True
    )

# ------------------------------------------------------------
# 6. Verify temporal integrity
# ------------------------------------------------------------

assert cumulative == len(events)

with np.load(
    OUT315 / "snapshot_t09.npz"
) as f:
    last_X = f["features"]
    last_time = int(f["time"])

assert last_time == 9
assert last_X.shape == (M, 12)

expected_past_t9 = int(
    (events["time"] < 9).sum()
)

assert int(last_X[:, 0].sum()) == expected_past_t9

# Compare definitions, not values, across seeds.
with np.load(
    REFERENCE / "causal_features" /
    "snapshot_t09.npz"
) as f:
    reference_names = f["feature_names"].tolist()

assert reference_names == FEATURE_NAMES

topology.to_csv(
    OUT315 / "edge_index_mapping.csv.gz",
    index=False,
    compression="gzip"
)

# ------------------------------------------------------------
# 7. Save audit metadata
# ------------------------------------------------------------

report315 = {
    "experiment": "E3.15",
    "scenario": "B_owner_correlated",
    "simulation_seed": 102,
    "reference_experiment": "E3.2",
    "feature_count": 12,
    "feature_names": FEATURE_NAMES,
    "temporal_decay": GAMMA,
    "evidence_confidence_k": CONFIDENCE_K,
    "age_normalization_horizon": HORIZON,
    "snapshots": audit,
    "total_train_events": int(len(train)),
    "total_validation_events": int(len(val)),
    "total_permitted_events": int(len(events)),
    "future_observations_used": False,
    "test_accessed": False,
    "hidden_state_accessed": False,
    "model_training_performed": False,
    "validation_protocol": (
        "Prequential: snapshot t=9 may use "
        "observed validation outcomes from t=8, "
        "without updating model parameters."
    )
}

report_path = (
    OUT315 / "e3_15_causal_feature_audit.json"
)

with open(
    report_path,
    "w", encoding="utf-8"
) as f:
    json.dump(report315, f, indent=2)

print("\n" + "=" * 72)
print("E3.15 COMPLETED — SEED 102 FEATURES")
print("=" * 72)

print("Snapshots saved:", len(audit))
print("Feature shape:", last_X.shape)
print("Historical observations at t=0: 0")
print(
    "Historical observations at t=9:",
    expected_past_t9
)
print(
    "Train + validation observations:",
    len(events)
)
print("Same feature definition as E3.2: VERIFIED")
print("Test accessed: NO")
print("Hidden state accessed: NO")
print("Previous artifacts modified: NO")

print("\nOutput:", OUT315)
print("Audit:", report_path)
print(
    "NEXT: E3.16 — Match model inputs and "
    "preprocessing for seed 102."
)


In [ ]:

# ============================================================
# E3.16a — INPUT AND PREPROCESSING SCHEMA AUDIT
# ============================================================
# Read-only audit for rebuilding model inputs for
# Scenario B, simulation seed 102.
#
# No training, no scaler fitting, no test access.
# No files modified.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import torch

ROOT316 = Path(
    "/kaggle/working/phase2_adaptive_siot/"
    "E3_tags_v3/scenarios/B_owner_correlated"
)

S101 = ROOT316 / "seed_101"
S102 = ROOT316 / "seed_102"

assert "data" in globals()
assert "U" in globals() and "V" in globals()

print("=" * 72)
print("E3.16a — INPUT AND PREPROCESSING SCHEMA AUDIT")
print("=" * 72)

# ------------------------------------------------------------
# 1. Existing model tensors (seed 101)
# ------------------------------------------------------------

print("\n[1] EXISTING MODEL TENSORS — SEED 101")

for t in [5, 6, 7, 8, 9]:
    batch = data[t]

    print(f"\nt={t}")
    for key, value in batch.items():
        if torch.is_tensor(value):
            print(
                f"  {key:12s} "
                f"shape={tuple(value.shape)!s:18s} "
                f"dtype={str(value.dtype):15s} "
                f"device={value.device}"
            )
        else:
            print(
                f"  {key:12s} "
                f"type={type(value).__name__} "
                f"value={value}"
            )

# ------------------------------------------------------------
# 2. Scaler specifications
# ------------------------------------------------------------

print("\n[2] SCALERS IN CURRENT KERNEL")

for name in [
    "edge_scaler",
    "node_scaler",
    "controller_scaler",
    "ctrl_scaler",
    "xc_scaler"
]:
    obj = globals().get(name, None)

    if obj is None:
        print(f"  {name}: NOT FOUND")
        continue

    print(f"\n  {name}: {type(obj).__name__}")

    for attr in [
        "n_features_in_",
        "with_mean",
        "with_std"
    ]:
        if hasattr(obj, attr):
            print(f"    {attr}: {getattr(obj, attr)}")

    for attr in ["mean_", "scale_"]:
        if hasattr(obj, attr):
            arr = np.asarray(getattr(obj, attr))
            print(
                f"    {attr}: "
                f"shape={arr.shape}; "
                f"first_values={arr[:12].tolist()}"
            )

# ------------------------------------------------------------
# 3. Topology and feature-file schema
# ------------------------------------------------------------

print("\n[3] TOPOLOGY SCHEMA")

top101 = pd.read_csv(
    S101 / "causal_features/edge_index_mapping.csv.gz"
)
top102 = pd.read_csv(
    S102 / "causal_features/edge_index_mapping.csv.gz"
)

print("Seed 101 columns:", top101.columns.tolist())
print("Seed 102 columns:", top102.columns.tolist())

print("Seed 101 shape:", top101.shape)
print("Seed 102 shape:", top102.shape)

pd.testing.assert_frame_equal(
    top101, top102, check_dtype=True
)

print("Topology identical: TRUE")

print("\n[4] CAUSAL SNAPSHOT SCHEMA")

for seed, folder in [
    (101, S101),
    (102, S102)
]:
    path = (
        folder / "causal_features/snapshot_t05.npz"
    )

    with np.load(path) as archive:
        print(f"\nSeed {seed}")
        print("  NPZ keys:", archive.files)

        for key in archive.files:
            arr = archive[key]

            print(
                f"  {key}: "
                f"shape={arr.shape}, dtype={arr.dtype}"
            )

            if key == "feature_names":
                print("  Names:", arr.tolist())

# ------------------------------------------------------------
# 4. Permitted interaction CSV schema
# ------------------------------------------------------------

print("\n[5] INTERACTION SCHEMA — TRAIN ONLY")

for seed, folder in [
    (101, S101),
    (102, S102)
]:
    train_path = folder / "observed/train.csv.gz"

    df = pd.read_csv(train_path)

    print(f"\nSeed {seed}:")
    print("  Columns:", df.columns.tolist())
    print("  Shape:", df.shape)
    print(
        "  Times:",
        sorted(df["time"].unique().tolist())
    )
    print("  Success mean:", float(df["success"].mean()))

    assert df["time"].between(0, 7).all()

# ------------------------------------------------------------
# 5. Graph indexing audit
# ------------------------------------------------------------

print("\n[6] GRAPH INDEXING")

print("U shape:", tuple(U.shape))
print("V shape:", tuple(V.shape))
print("U device:", U.device)
print("V device:", V.device)

expected_u = torch.as_tensor(
    top101["u"].to_numpy(np.int64) - 1,
    device=U.device
)

expected_v = torch.as_tensor(
    top101["v"].to_numpy(np.int64) - 1,
    device=V.device
)

print(
    "U matches canonical topology:",
    torch.equal(U, expected_u)
)
print(
    "V matches canonical topology:",
    torch.equal(V, expected_v)
)

assert torch.equal(U, expected_u)
assert torch.equal(V, expected_v)

# ------------------------------------------------------------
# 6. Available preprocessing-related variables
# ------------------------------------------------------------

print("\n[7] PREPROCESSING VARIABLES IN NOTEBOOK")

keywords = (
    "feature", "scaler", "type", "relation",
    "onehot", "encoder", "node", "edge",
    "snapshot"
)

matches = [
    name for name in globals()
    if not name.startswith("_")
    and any(word in name.lower() for word in keywords)
    and not callable(globals()[name])
]

print("Candidate variable names:")
print(sorted(matches)[:100])

print("\n" + "=" * 72)
print("E3.16a COMPLETED")
print("=" * 72)
print("No training performed.")
print("No scaler fitting performed.")
print("No validation or test outcomes read.")
print("No artifacts modified.")
print(
    "NEXT: Rebuild seed 102 tensors using "
    "the verified seed 101 preprocessing schema."
)


In [ ]:

# ============================================================
# E3.16b — EXACT INPUT RECONSTRUCTION AUDIT
# ============================================================
# Verify edge reconstruction against existing
# seed-101 tensors before preparing seed 102.
#
# Read-only. No training. No test. No hidden state.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import torch

ROOT316 = Path(
    "/kaggle/working/phase2_adaptive_siot/"
    "E3_tags_v3/scenarios/B_owner_correlated"
)

S101 = ROOT316 / "seed_101"

REL316 = ["OOR", "POR", "C_LOR", "SOR", "SOR2"]

assert all(x in globals() for x in [
    "data", "edge_scaler", "node_scaler",
    "U", "V", "type_onehot"
])

print("=" * 72)
print("E3.16b — EXACT INPUT RECONSTRUCTION AUDIT")
print("=" * 72)

# ------------------------------------------------------------
# 1. Reconstruct 17-dimensional edge features
# ------------------------------------------------------------

top = pd.read_csv(
    S101 / "causal_features/edge_index_mapping.csv.gz"
)

relations = top[REL316].to_numpy(
    dtype=np.float32
)

print("\n[1] EDGE FEATURE RECONSTRUCTION")

for t in [5, 7, 9]:

    path = (
        S101 / "causal_features" /
        f"snapshot_t{t:02d}.npz"
    )

    with np.load(path) as f:
        history = f["features"].astype(
            np.float32
        )

    raw_edge = np.concatenate(
        [history, relations],
        axis=1
    )

    assert raw_edge.shape == (146117, 17)

    reconstructed = edge_scaler.transform(
        raw_edge
    ).astype(np.float32)

    existing = (
        data[t]["xe"]
        .detach()
        .cpu()
        .numpy()
    )

    diff = np.abs(
        reconstructed - existing
    )

    print(
        f"t={t} | "
        f"max_abs_diff={diff.max():.9g} | "
        f"mean_abs_diff={diff.mean():.9g} | "
        f"allclose={np.allclose(reconstructed, existing, atol=1e-5, rtol=1e-5)}"
    )

# ------------------------------------------------------------
# 2. Inspect non-hidden node-related variables
# ------------------------------------------------------------

print("\n[2] NODE INPUT SOURCE STRUCTURES")

names = [
    "type_onehot",
    "type_codes",
    "type_names",
    "node_dynamic",
    "node_n",
    "node_rate",
    "node_conf",
    "relation_matrix",
    "feature_sets"
]

for name in names:

    if name not in globals():
        print(f"{name}: NOT FOUND")
        continue

    obj = globals()[name]

    print(f"\n{name}")
    print("  type:", type(obj).__name__)

    if isinstance(obj, dict):
        print("  keys:", list(obj.keys())[:20])

        for key, value in list(obj.items())[:5]:
            print(
                "  key:", key,
                "| value type:", type(value).__name__,
                "| shape:", getattr(value, "shape", None)
            )

    elif isinstance(obj, (np.ndarray, torch.Tensor)):
        print("  shape:", tuple(obj.shape))
        print("  dtype:", obj.dtype)

    elif isinstance(obj, (list, tuple)):
        print("  length:", len(obj))
        print(
            "  first item type:",
            type(obj[0]).__name__ if obj else "EMPTY"
        )

    else:
        print("  representation:", repr(obj)[:200])

# ------------------------------------------------------------
# 3. Identify static/dynamic node feature locations
# ------------------------------------------------------------

print("\n[3] EXISTING NODE TENSORS")

for t in [5, 7, 9]:

    xn = data[t]["xn"].detach().cpu().numpy()
    xc = data[t]["xc"].detach().cpu().numpy()

    print(f"\nt={t}")
    print("  xn shape:", xn.shape)
    print("  xc shape:", xc.shape)

    print(
        "  xn first 5 columns, first 3 nodes:\n",
        np.array2string(
            xn[:3, :5],
            precision=5
        )
    )

    print(
        "  xn last 4 columns, first 3 nodes:\n",
        np.array2string(
            xn[:3, -4:],
            precision=5
        )
    )

    print(
        "  xc first 3 nodes:\n",
        np.array2string(
            xc[:3],
            precision=5
        )
    )

    if "type_onehot" in globals():
        types = type_onehot

        if torch.is_tensor(types):
            types = types.detach().cpu().numpy()

        if (
            isinstance(types, np.ndarray)
            and types.ndim == 2
            and types.shape[0] == xn.shape[0]
        ):
            width = types.shape[1]

            print(
                "  type_onehot width:", width
            )

            if width <= xn.shape[1]:
                print(
                    "  static prefix matches:",
                    np.allclose(
                        xn[:, :width],
                        types,
                        atol=1e-6
                    )
                )

# ------------------------------------------------------------
# 4. Inspect scaler compatibility
# ------------------------------------------------------------

print("\n[4] NODE SCALER")

print(
    "node_scaler.n_features_in_:",
    node_scaler.n_features_in_
)

print(
    "node_scaler.mean_:",
    node_scaler.mean_.tolist()
)

print(
    "node_scaler.scale_:",
    node_scaler.scale_.tolist()
)

print("\n" + "=" * 72)
print("E3.16b COMPLETED")
print("=" * 72)
print("Read-only audit: YES")
print("Scaler refitting: NO")
print("Training performed: NO")
print("Validation labels accessed: NO")
print("Test accessed: NO")
print("Hidden simulator state accessed: NO")


In [ ]:

# ============================================================
# E3.16c — REVERSE-ENGINEER ORIGINAL FEATURE TRANSFORMS
# ============================================================
# Recover pre-standardization edge and node features
# from existing seed-101 tensors and fitted scalers.
#
# Diagnostic only: no fitting, training, test access
# or file modifications.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import torch

ROOT316C = Path(
    "/kaggle/working/phase2_adaptive_siot/"
    "E3_tags_v3/scenarios/B_owner_correlated/seed_101"
)

assert all(name in globals() for name in [
    "data", "edge_scaler", "node_scaler",
    "type_onehot", "U", "V"
])

REL316C = ["OOR", "POR", "C_LOR", "SOR", "SOR2"]

top = pd.read_csv(
    ROOT316C / "causal_features/edge_index_mapping.csv.gz"
)

relation_raw = top[REL316C].to_numpy(
    dtype=np.float32
)

history_names = None

print("=" * 76)
print("E3.16c — ORIGINAL FEATURE TRANSFORM DIAGNOSTIC")
print("=" * 76)

def compare_column(a, b):
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)

    diff = np.abs(a - b)

    correlation = (
        np.corrcoef(a, b)[0, 1]
        if np.std(a) > 1e-12
        and np.std(b) > 1e-12
        else np.nan
    )

    return (
        float(np.mean(diff)),
        float(np.max(diff)),
        float(correlation)
    )

# ------------------------------------------------------------
# 1. Recover actual PRE-SCALER edge inputs
# ------------------------------------------------------------

for t in [5, 7, 9]:

    with np.load(
        ROOT316C / "causal_features" /
        f"snapshot_t{t:02d}.npz"
    ) as archive:
        history = archive["features"].astype(
            np.float64
        )

        history_names = archive[
            "feature_names"
        ].tolist()

    standardized = (
        data[t]["xe"]
        .detach().cpu().numpy()
    )

    actual_raw = edge_scaler.inverse_transform(
        standardized
    )

    candidate_raw = np.concatenate(
        [history, relation_raw],
        axis=1
    )

    names = history_names + REL316C

    print(f"\n{'-' * 76}")
    print(f"EDGE SNAPSHOT t={t}")
    print(f"{'-' * 76}")

    rows = []

    for j, name in enumerate(names):

        observed = actual_raw[:, j]
        original = candidate_raw[:, j]

        tests = {
            "identity": original,
            "log1p": np.log1p(
                np.clip(original, 0, None)
            ),
            "sqrt": np.sqrt(
                np.clip(original, 0, None)
            )
        }

        candidates = []

        for operation, transformed in tests.items():
            mae, maximum, corr = compare_column(
                observed, transformed
            )

            candidates.append(
                (mae, maximum, corr, operation)
            )

        best = min(
            candidates, key=lambda x: x[0]
        )

        direct_mae, direct_max, direct_corr = (
            compare_column(observed, original)
        )

        rows.append({
            "col": j,
            "feature": name,
            "recovered_mean": np.mean(observed),
            "original_mean": np.mean(original),
            "identity_MAE": direct_mae,
            "identity_MAX": direct_max,
            "identity_corr": direct_corr,
            "best_transform": best[3],
            "best_MAE": best[0],
            "best_MAX": best[1]
        })

    display(
        pd.DataFrame(rows).round(6)
    )

# ------------------------------------------------------------
# 2. Recover original node-feature values
# ------------------------------------------------------------

print(f"\n{'=' * 76}")
print("NODE FEATURE RECOVERY")
print(f"{'=' * 76}")

for t in [5, 7, 9]:

    xc = (
        data[t]["xc"]
        .detach().cpu().numpy()
    )

    raw_node = node_scaler.inverse_transform(
        xc
    )

    print(f"\nNODE SNAPSHOT t={t}")

    print(
        pd.DataFrame(
            raw_node[:5],
            columns=[
                "raw_node_feature_0",
                "raw_node_feature_1",
                "raw_node_feature_2",
                "raw_node_feature_3"
            ]
        ).round(6).to_string(index=False)
    )

    print(
        "Means:",
        np.round(
            raw_node.mean(axis=0), 6
        ).tolist()
    )

    print(
        "Min:",
        np.round(
            raw_node.min(axis=0), 6
        ).tolist()
    )

    print(
        "Max:",
        np.round(
            raw_node.max(axis=0), 6
        ).tolist()
    )

# ------------------------------------------------------------
# 3. Check candidate static node degree transform
# ------------------------------------------------------------

n_nodes = int(type_onehot.shape[0])

u = U.detach().cpu().numpy()
v = V.detach().cpu().numpy()

degree = np.bincount(
    np.concatenate([u, v]),
    minlength=n_nodes
).astype(np.float64)

raw_node_5 = node_scaler.inverse_transform(
    data[5]["xc"].detach().cpu().numpy()
)

print(f"\n{'=' * 76}")
print("STATIC NODE DEGREE HYPOTHESIS")
print(f"{'=' * 76}")

for name, candidate in [
    ("degree", degree),
    ("log1p_degree", np.log1p(degree)),
    ("sqrt_degree", np.sqrt(degree))
]:
    mae, maximum, corr = compare_column(
        raw_node_5[:, 0], candidate
    )

    print(
        f"{name:17s} | "
        f"MAE={mae:.9f} | "
        f"MAX={maximum:.9f} | "
        f"corr={corr:.6f}"
    )

print("\n" + "=" * 76)
print("E3.16c COMPLETED")
print("=" * 76)
print("No scaler fitting.")
print("No model training.")
print("No test or hidden-state access.")
print("No artifacts modified.")


In [ ]:

# ============================================================
# E3.16d — COMPACT FEATURE MAPPING DIAGNOSTIC
# ============================================================
# Read only seed-101 training data and t=5 snapshot.
# No fitting, no training, no test or hidden state.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path(
    "/kaggle/working/phase2_adaptive_siot/"
    "E3_tags_v3/scenarios/B_owner_correlated/seed_101"
)

T = 5
REL = ["OOR", "POR", "C_LOR", "SOR", "SOR2"]

top = pd.read_csv(
    ROOT / "causal_features/edge_index_mapping.csv.gz"
)

with np.load(
    ROOT / "causal_features/snapshot_t05.npz"
) as f:
    history = f["features"].astype(np.float64)
    names = list(f["feature_names"])

relations = top[REL].to_numpy(np.float64)
raw_candidates = np.column_stack([history, relations])
names += REL

actual_edge = edge_scaler.inverse_transform(
    data[T]["xe"].detach().cpu().numpy()
)

print("=" * 73)
print("E3.16d — EDGE FEATURE MAPPING")
print("=" * 73)

matches = []

for j in range(actual_edge.shape[1]):
    actual = actual_edge[:, j]
    possibilities = []

    for k, name in enumerate(names):
        x = raw_candidates[:, k]

        transforms = {
            "identity": x,
            "log1p": np.log1p(np.maximum(x, 0)),
            "sqrt": np.sqrt(np.maximum(x, 0))
        }

        for transform, predicted in transforms.items():
            error = np.abs(actual - predicted)

            possibilities.append((
                float(error.mean()),
                float(error.max()),
                k, name, transform
            ))

    best = min(possibilities, key=lambda z: z[0])

    matches.append({
        "input_col": j,
        "source_col": best[2],
        "source": best[3],
        "transform": best[4],
        "MAE": best[0],
        "MAX": best[1]
    })

mapping = pd.DataFrame(matches)

print(mapping.to_string(
    index=False,
    float_format=lambda x: f"{x:.7f}"
))

print("\nColumns with matching error < 1e-5:",
      int((mapping["MAX"] < 1e-5).sum()),
      "/ 17")

# ------------------------------------------------------------
# NODE FEATURE HYPOTHESES — HISTORICAL DATA ONLY
# ------------------------------------------------------------

print("\n" + "=" * 73)
print("NODE HISTORICAL FEATURE CHECK — t=5")
print("=" * 73)

train = pd.read_csv(
    ROOT / "observed/train.csv.gz"
)

past = train[train["time"] < T]

N = 16216

src = past["source"].to_numpy(np.int64) - 1
dst = past["target"].to_numpy(np.int64) - 1
success = past["success"].to_numpy(np.float64)

counts = np.bincount(
    np.concatenate([src, dst]),
    minlength=N
)

success_counts = (
    np.bincount(src, weights=success, minlength=N)
    + np.bincount(dst, weights=success, minlength=N)
)

predicted_log_count = np.log1p(counts)
predicted_rate = (
    success_counts + 1
) / (counts + 2)

actual_node = node_scaler.inverse_transform(
    data[T]["xc"].detach().cpu().numpy()
)

for name, column, candidate in [
    ("log1p historical interactions", 1,
     predicted_log_count),
    ("smoothed historical success rate", 2,
     predicted_rate)
]:
    error = np.abs(
        actual_node[:, column] - candidate
    )

    print(
        f"{name}: "
        f"MAE={error.mean():.9f}, "
        f"MAX={error.max():.9f}"
    )

print("\nE3.16d COMPLETED")
print("Read-only: YES")
print("Test and hidden state accessed: NO")


In [ ]:

# ============================================================
# E3.16e — ENDPOINT FEATURE RECONSTRUCTION
# ============================================================
# Identify edge input columns 7..10 using actual
# historical node statistics from the existing
# seed-101 model input.
#
# No fitting, training, test access or file changes.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

S101_316E = Path(
    "/kaggle/working/phase2_adaptive_siot/"
    "E3_tags_v3/scenarios/B_owner_correlated/seed_101"
)

REL_316E = ["OOR", "POR", "C_LOR", "SOR", "SOR2"]

top_316e = pd.read_csv(
    S101_316E / "causal_features/edge_index_mapping.csv.gz"
)

uu_316e = top_316e["u"].to_numpy(np.int64) - 1
vv_316e = top_316e["v"].to_numpy(np.int64) - 1

relations_316e = top_316e[REL_316E].to_numpy(
    dtype=np.float32
)

print("=" * 75)
print("E3.16e — ENDPOINT FEATURE RECONSTRUCTION")
print("=" * 75)

for t in [5, 7, 9]:

    with np.load(
        S101_316E / "causal_features" /
        f"snapshot_t{t:02d}.npz"
    ) as f:
        hist = f["features"].astype(np.float32)

    # Recover the 17 inputs BEFORE standardization.
    observed_raw = edge_scaler.inverse_transform(
        data[t]["xe"].detach().cpu().numpy()
    )

    # Recover the 4 node inputs BEFORE standardization.
    node_raw = node_scaler.inverse_transform(
        data[t]["xc"].detach().cpu().numpy()
    )

    rate = node_raw[:, 2]
    confidence = node_raw[:, 3]

    candidates = {
        "rate_u": rate[uu_316e],
        "rate_v": rate[vv_316e],
        "confidence_u": confidence[uu_316e],
        "confidence_v": confidence[vv_316e],
        "snapshot_rate_u": hist[:, 7],
        "snapshot_rate_v": hist[:, 8],
        "snapshot_conf_u": hist[:, 9],
        "snapshot_conf_v": hist[:, 10]
    }

    print(f"\nSNAPSHOT t={t}")
    print("-" * 75)

    for col in range(7, 11):
        target = observed_raw[:, col]

        errors = sorted(
            [
                (
                    float(np.max(np.abs(target - values))),
                    float(np.mean(np.abs(target - values))),
                    name
                )
                for name, values in candidates.items()
            ],
            key=lambda row: row[0]
        )

        best = errors[:3]

        print(f"\nEdge input column {col}:")
        for maxerr, mae, name in best:
            print(
                f"  {name:19s} "
                f"MAX={maxerr:.9g} "
                f"MAE={mae:.9g}"
            )

    # Hypothesized feature order:
    # [rate_u, rate_v, confidence_u, confidence_v]
    reconstructed_raw = np.column_stack([
        np.log1p(hist[:, 0]),
        hist[:, 1],
        np.log1p(hist[:, 2]),
        hist[:, 3:7],
        rate[uu_316e],
        rate[vv_316e],
        confidence[uu_316e],
        confidence[vv_316e],
        hist[:, 11],
        relations_316e
    ]).astype(np.float32)

    assert reconstructed_raw.shape == (146117, 17)

    reconstructed_scaled = (
        edge_scaler.transform(reconstructed_raw)
        .astype(np.float32)
    )

    original_scaled = (
        data[t]["xe"].detach().cpu().numpy()
    )

    error = np.abs(
        reconstructed_scaled - original_scaled
    )

    print("\nFULL 17-COLUMN RECONSTRUCTION")
    print("Max absolute difference:", float(error.max()))
    print("Mean absolute difference:", float(error.mean()))
    print(
        "Allclose (atol=1e-5, rtol=1e-5):",
        bool(np.allclose(
            reconstructed_scaled,
            original_scaled,
            atol=1e-5,
            rtol=1e-5
        ))
    )

    # Inspect the suspected discrepancy between
    # saved E3.2 history and model endpoint inputs.
    for name, a, b in [
        ("node_u_rate", hist[:, 7], rate[uu_316e]),
        ("node_v_rate", hist[:, 8], rate[vv_316e]),
        ("node_u_conf", hist[:, 9], confidence[uu_316e]),
        ("node_v_conf", hist[:, 10], confidence[vv_316e])
    ]:
        print(
            f"{name:15s} "
            f"snapshot_vs_node_MAX="
            f"{np.max(np.abs(a - b)):.8f}"
        )

print("\n" + "=" * 75)
print("E3.16e COMPLETED")
print("=" * 75)
print("No model training or scaler fitting.")
print("No test or hidden-state access.")
print("No artifacts modified.")


In [ ]:

# ============================================================
# E3.16f — VERIFY SYMMETRIC ENDPOINT AGGREGATION
# ============================================================
# Hypothesis:
# col 7  = max(rate_u, rate_v)
# col 8  = abs(rate_u - rate_v)
# col 9  = max(conf_u, conf_v)
# col 10 = abs(conf_u - conf_v)
#
# Read-only; no training or scaler fitting.
# No test or hidden-state access.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

S101_F = Path(
    "/kaggle/working/phase2_adaptive_siot/"
    "E3_tags_v3/scenarios/B_owner_correlated/seed_101"
)

REL_F = ["OOR", "POR", "C_LOR", "SOR", "SOR2"]

top_f = pd.read_csv(
    S101_F / "causal_features/edge_index_mapping.csv.gz"
)

u_f = top_f["u"].to_numpy(np.int64) - 1
v_f = top_f["v"].to_numpy(np.int64) - 1

relation_f = top_f[REL_F].to_numpy(np.float32)

print("=" * 72)
print("E3.16f — SYMMETRIC ENDPOINT FEATURE AUDIT")
print("=" * 72)

all_passed = True

for t in [5, 7, 9]:

    with np.load(
        S101_F / "causal_features" /
        f"snapshot_t{t:02d}.npz"
    ) as f:
        h = f["features"].astype(np.float32)

    node_raw = node_scaler.inverse_transform(
        data[t]["xc"].detach().cpu().numpy()
    )

    rate_u = node_raw[u_f, 2]
    rate_v = node_raw[v_f, 2]

    conf_u = node_raw[u_f, 3]
    conf_v = node_raw[v_f, 3]

    endpoint = np.column_stack([
        np.maximum(rate_u, rate_v),
        np.abs(rate_u - rate_v),
        np.maximum(conf_u, conf_v),
        np.abs(conf_u - conf_v)
    ]).astype(np.float32)

    actual_raw = edge_scaler.inverse_transform(
        data[t]["xe"].detach().cpu().numpy()
    )

    print(f"\nSNAPSHOT t={t}")
    print("-" * 72)

    endpoint_pass = True

    for j in range(4):
        error = np.abs(
            endpoint[:, j] - actual_raw[:, j + 7]
        )

        max_error = float(error.max())
        mae = float(error.mean())

        passed = max_error < 1e-5
        endpoint_pass &= passed

        print(
            f"column={j + 7:2d} | "
            f"MAX={max_error:.9g} | "
            f"MAE={mae:.9g} | "
            f"PASS={passed}"
        )

    reconstructed_raw = np.column_stack([
        np.log1p(h[:, 0]),
        h[:, 1],
        np.log1p(h[:, 2]),
        h[:, 3:7],
        endpoint,
        h[:, 11],
        relation_f
    ]).astype(np.float32)

    assert reconstructed_raw.shape == (146117, 17)

    reconstructed_scaled = edge_scaler.transform(
        reconstructed_raw
    ).astype(np.float32)

    existing_scaled = (
        data[t]["xe"].detach().cpu().numpy()
    )

    diff = np.abs(
        reconstructed_scaled - existing_scaled
    )

    max_diff = float(diff.max())
    mean_diff = float(diff.mean())

    full_pass = bool(np.allclose(
        reconstructed_scaled,
        existing_scaled,
        atol=1e-5,
        rtol=1e-5
    ))

    print("\nFULL 17-FEATURE RECONSTRUCTION")
    print("Max absolute difference:", max_diff)
    print("Mean absolute difference:", mean_diff)
    print("Allclose:", full_pass)

    all_passed &= endpoint_pass and full_pass

print("\n" + "=" * 72)
print("E3.16f COMPLETED")
print("=" * 72)
print("All snapshots reconstructed:", all_passed)
print("No training performed.")
print("No scaler fitting.")
print("No test or hidden-state access.")
print("No existing artifacts modified.")

if not all_passed:
    print(
        "WARNING: Input reconstruction is not "
        "yet verified. Do not train seed 102."
    )
else:
    print(
        "SUCCESS: Edge preprocessing recovered "
        "for the tested seed-101 snapshots."
    )


In [ ]:

# ============================================================
# E3.16g — FULL MODEL-INPUT RECONSTRUCTION AUDIT
# ============================================================
# Verify all 17 edge features, 20 node features,
# and 4 controller features against seed-101 tensors.
#
# t=5,6,7: training history only.
# t=8,9: prequential history; t=9 uses observed
#        t=8 outcomes, never t=9 outcomes.
#
# No training, no fitting, no test access.
# No files modified.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

S101 = Path(
    "/kaggle/working/phase2_adaptive_siot/"
    "E3_tags_v3/scenarios/B_owner_correlated/seed_101"
)

REL = ["OOR", "POR", "C_LOR", "SOR", "SOR2"]
TIMES = [5, 6, 7, 8, 9]

top = pd.read_csv(
    S101 / "causal_features/edge_index_mapping.csv.gz"
)

u = top["u"].to_numpy(np.int64) - 1
v = top["v"].to_numpy(np.int64) - 1
relations = top[REL].to_numpy(np.float32)

N = 16216
M = len(top)

assert M == 146117
assert len(type_onehot) == N

train = pd.read_csv(
    S101 / "observed/train.csv.gz"
)

validation = pd.read_csv(
    S101 / "observed/validation.csv.gz"
)

# Only histories strictly earlier than the snapshot
# are used. No test data are loaded.
permitted = pd.concat(
    [train, validation],
    ignore_index=True
)

print("=" * 74)
print("E3.16g — FULL INPUT RECONSTRUCTION")
print("=" * 74)

def reconstruction_error(actual, expected):
    difference = np.abs(actual - expected)
    return (
        float(difference.max()),
        float(difference.mean()),
        bool(np.allclose(
            actual, expected,
            atol=1e-5,
            rtol=1e-5
        ))
    )

all_pass = True

for t in TIMES:

    # --------------------------------------------------------
    # 1. Reconstruct node history from interactions before t
    # --------------------------------------------------------

    past = permitted[
        permitted["time"] < t
    ]

    src = past["source"].to_numpy(np.int64) - 1
    dst = past["target"].to_numpy(np.int64) - 1
    outcomes = past["success"].to_numpy(np.float64)

    counts = np.bincount(
        np.concatenate([src, dst]),
        minlength=N
    )

    successes = (
        np.bincount(
            src, weights=outcomes, minlength=N
        )
        + np.bincount(
            dst, weights=outcomes, minlength=N
        )
    )

    degree = np.bincount(
        np.concatenate([u, v]),
        minlength=N
    )

    node_raw = np.column_stack([
        np.log1p(degree),
        np.log1p(counts),
        (successes + 1) / (counts + 2),
        counts / (counts + 5)
    ]).astype(np.float32)

    node_scaled = node_scaler.transform(
        node_raw
    ).astype(np.float32)

    xn_reconstructed = np.concatenate([
        type_onehot.astype(np.float32),
        node_scaled
    ], axis=1)

    # --------------------------------------------------------
    # 2. Reconstruct edge inputs
    # --------------------------------------------------------

    with np.load(
        S101 / "causal_features" /
        f"snapshot_t{t:02d}.npz"
    ) as archive:
        h = archive["features"].astype(np.float32)

    rate_u = node_raw[u, 2]
    rate_v = node_raw[v, 2]
    conf_u = node_raw[u, 3]
    conf_v = node_raw[v, 3]

    endpoint_features = np.column_stack([
        (rate_u + rate_v) / 2,
        np.abs(rate_u - rate_v),
        (conf_u + conf_v) / 2,
        np.abs(conf_u - conf_v)
    ]).astype(np.float32)

    raw_edge = np.column_stack([
        np.log1p(h[:, 0]),
        h[:, 1],
        np.log1p(h[:, 2]),
        h[:, 3:7],
        endpoint_features,
        h[:, 11],
        relations
    ]).astype(np.float32)

    assert raw_edge.shape == (M, 17)
    assert xn_reconstructed.shape == (N, 20)

    xe_reconstructed = edge_scaler.transform(
        raw_edge
    ).astype(np.float32)

    # --------------------------------------------------------
    # 3. Compare against original model tensors
    # --------------------------------------------------------

    original_xe = (
        data[t]["xe"].detach().cpu().numpy()
    )

    original_xn = (
        data[t]["xn"].detach().cpu().numpy()
    )

    original_xc = (
        data[t]["xc"].detach().cpu().numpy()
    )

    xe_max, xe_mae, xe_ok = reconstruction_error(
        original_xe, xe_reconstructed
    )

    xn_max, xn_mae, xn_ok = reconstruction_error(
        original_xn, xn_reconstructed
    )

    xc_max, xc_mae, xc_ok = reconstruction_error(
        original_xc, node_scaled
    )

    # Also check pre-scaler endpoint columns.
    raw_reference = edge_scaler.inverse_transform(
        original_xe
    )

    ep_max, ep_mae, ep_ok = reconstruction_error(
        raw_reference[:, 7:11],
        endpoint_features
    )

    passed = xe_ok and xn_ok and xc_ok and ep_ok
    all_pass &= passed

    print(
        f"t={t} | "
        f"xe_MAX={xe_max:.8g} | "
        f"xn_MAX={xn_max:.8g} | "
        f"xc_MAX={xc_max:.8g} | "
        f"endpoint_MAX={ep_max:.8g} | "
        f"PASS={passed}",
        flush=True
    )

print("\n" + "=" * 74)
print("E3.16g COMPLETED")
print("=" * 74)
print("All 5 snapshots reconstructed:", all_pass)
print("Scaler fitting: NO")
print("Model training: NO")
print("Test accessed: NO")
print("Hidden state accessed: NO")
print("Existing files modified: NO")

if all_pass:
    print(
        "SUCCESS — Exact preprocessing logic verified "
        "against seed-101 model inputs."
    )
else:
    print(
        "RECONSTRUCTION NOT VERIFIED — "
        "Do not train seed 102 yet."
    )


In [ ]:

# ============================================================
# E3.16h — VERIFIED INPUT PIPELINE FOR SCENARIO B SEED 102
# ============================================================
# 1. Refit seed-101 scalers and verify against originals.
# 2. Reconstruct seed-102 features with identical logic.
# 3. Fit NEW scalers only on training snapshots 5,6,7.
# 4. Transform snapshots 5,...,9.
# 5. Create CUDA tensors in data102.
#
# No model training.
# No test or hidden simulator-state access.
# Does not modify existing data or scaler objects.
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler
import joblib

ROOT_H = Path(
    "/kaggle/working/phase2_adaptive_siot/"
    "E3_tags_v3/scenarios/B_owner_correlated"
)

S101_H = ROOT_H / "seed_101"
S102_H = ROOT_H / "seed_102"

SAVE_H = S102_H / "e3_16h_model_inputs"

if SAVE_H.exists():
    raise FileExistsError(
        f"Existing output directory: {SAVE_H}"
    )

required = [
    "data", "edge_scaler", "node_scaler",
    "type_onehot", "U", "V", "K"
]

missing = [name for name in required if name not in globals()]
assert not missing, f"Missing objects: {missing}"

DEVICE_H = U.device
N_H = 16216
M_H = 146117
TIMES_H = [5, 6, 7, 8, 9]
TRAIN_H = [5, 6, 7]
REL_H = ["OOR", "POR", "C_LOR", "SOR", "SOR2"]

assert DEVICE_H.type == "cuda"
assert type_onehot.shape == (N_H, 16)

# ============================================================
# 1. Reusable exact feature-construction function
# ============================================================

def build_raw_snapshots_h(folder):

    topology = pd.read_csv(
        folder / "causal_features/"
        "edge_index_mapping.csv.gz"
    )

    assert len(topology) == M_H

    u = topology["u"].to_numpy(np.int64) - 1
    v = topology["v"].to_numpy(np.int64) - 1

    reference_u = U.detach().cpu().numpy()
    reference_v = V.detach().cpu().numpy()

    assert np.array_equal(u, reference_u)
    assert np.array_equal(v, reference_v)

    relations = topology[REL_H].to_numpy(
        dtype=np.float32
    )

    static_degree = np.bincount(
        np.concatenate([u, v]),
        minlength=N_H
    )

    degree_feature = np.log1p(
        static_degree
    ).astype(np.float32)

    # Only allowed observed outcomes.
    train = pd.read_csv(
        folder / "observed/train.csv.gz"
    )

    val = pd.read_csv(
        folder / "observed/validation.csv.gz"
    )

    assert train["time"].between(0, 7).all()
    assert val["time"].between(8, 9).all()

    assert train["success"].isin([0, 1]).all()
    assert val["success"].isin([0, 1]).all()

    events = pd.concat(
        [train, val], ignore_index=True
    )

    # Canonical edge mapping.
    keys = u * N_H + v

    assert np.all(np.diff(keys) > 0)

    source = events["source"].to_numpy(
        np.int64
    ) - 1

    target = events["target"].to_numpy(
        np.int64
    ) - 1

    assert np.all(source < target)

    event_keys = source * N_H + target

    event_edge_ids = np.searchsorted(
        keys, event_keys
    )

    assert np.all(event_edge_ids < M_H)
    assert np.array_equal(
        keys[event_edge_ids], event_keys
    )

    event_times = events["time"].to_numpy(
        np.int16
    )

    event_success = events["success"].to_numpy(
        np.float64
    )

    raw = {}

    for t in TIMES_H:

        # Strictly historical observations.
        # At t=9, t=8 validation observations may
        # contribute to causal history, but never
        # to model/scaler fitting.
        past = event_times < t

        past_u = source[past]
        past_v = target[past]
        past_success = event_success[past]

        node_count = np.bincount(
            np.concatenate([past_u, past_v]),
            minlength=N_H
        )

        node_success = (
            np.bincount(
                past_u,
                weights=past_success,
                minlength=N_H
            )
            + np.bincount(
                past_v,
                weights=past_success,
                minlength=N_H
            )
        )

        node_raw = np.column_stack([
            degree_feature,
            np.log1p(node_count),
            (node_success + 1.0) /
            (node_count + 2.0),
            node_count / (node_count + 5.0)
        ]).astype(np.float32)

        assert node_raw.shape == (N_H, 4)

        with np.load(
            folder / "causal_features" /
            f"snapshot_t{t:02d}.npz"
        ) as archive:

            history = archive["features"].astype(
                np.float32
            )

            assert int(archive["time"]) == t

        assert history.shape == (M_H, 12)

        # Independent temporal-integrity check.
        assert abs(
            float(history[:, 0].sum())
            - int(past.sum())
        ) < 1.0

        ru = node_raw[u, 2]
        rv = node_raw[v, 2]

        cu = node_raw[u, 3]
        cv = node_raw[v, 3]

        endpoints = np.column_stack([
            (ru + rv) / 2.0,
            np.abs(ru - rv),
            (cu + cv) / 2.0,
            np.abs(cu - cv)
        ]).astype(np.float32)

        edge_raw = np.column_stack([
            np.log1p(history[:, 0]),
            history[:, 1],
            np.log1p(history[:, 2]),
            history[:, 3:7],
            endpoints,
            history[:, 11],
            relations
        ]).astype(np.float32)

        assert edge_raw.shape == (M_H, 17)

        current = event_times == t

        current_ids = event_edge_ids[current]

        # Failure = 1 - success.
        failures = (
            1.0 - event_success[current]
        ).astype(np.float32)

        assert len(current_ids) == len(failures)

        raw[t] = {
            "node": node_raw,
            "edge": edge_raw,
            "target": current_ids.astype(np.int64),
            "y": failures,
            "count": int(len(current_ids))
        }

    return raw

# ============================================================
# 2. Verify fitted-scaler protocol on seed 101
# ============================================================

print("=" * 74)
print("E3.16h — VERIFIED SEED 102 INPUT PREPARATION")
print("=" * 74)

print("\n[1] VERIFY SCALER FITTING USING SEED 101")

raw101_h = build_raw_snapshots_h(S101_H)

edge_check_h = StandardScaler()
node_check_h = StandardScaler()

for t in TRAIN_H:
    edge_check_h.partial_fit(
        raw101_h[t]["edge"]
    )

    node_check_h.partial_fit(
        raw101_h[t]["node"]
    )

def scaler_error(a, b):
    return float(
        np.max(np.abs(np.asarray(a) - np.asarray(b)))
    )

checks_h = {
    "edge_mean": scaler_error(
        edge_check_h.mean_, edge_scaler.mean_
    ),
    "edge_scale": scaler_error(
        edge_check_h.scale_, edge_scaler.scale_
    ),
    "node_mean": scaler_error(
        node_check_h.mean_, node_scaler.mean_
    ),
    "node_scale": scaler_error(
        node_check_h.scale_, node_scaler.scale_
    )
}

for name, error in checks_h.items():
    print(
        f"{name:14s} max_diff={error:.10g}"
    )

scalers_match_h = all(
    value < 1e-5
    for value in checks_h.values()
)

assert scalers_match_h, (
    "Seed-101 scaler fitting does not match "
    "the original pipeline. Stop here."
)

# Cross-check reconstruction with fitted scalers.
for t in TIMES_H:

    xe_check = edge_check_h.transform(
        raw101_h[t]["edge"]
    ).astype(np.float32)

    xn_check = node_check_h.transform(
        raw101_h[t]["node"]
    ).astype(np.float32)

    xe_original = (
        data[t]["xe"].detach().cpu().numpy()
    )

    xc_original = (
        data[t]["xc"].detach().cpu().numpy()
    )

    assert np.allclose(
        xe_check, xe_original,
        rtol=1e-5, atol=1e-5
    )

    assert np.allclose(
        xn_check, xc_original,
        rtol=1e-5, atol=1e-5
    )

print("Seed-101 fitting reproduction: PASSED")

del raw101_h

# ============================================================
# 3. Generate seed-102 raw inputs
# ============================================================

print("\n[2] BUILD SEED 102 RAW SNAPSHOTS")

raw102_h = build_raw_snapshots_h(S102_H)

# ============================================================
# 4. Fit NEW scalers on seed-102 TRAINING only
# ============================================================

print("\n[3] FIT SEED 102 TRAIN-ONLY SCALERS")

edge_scaler102 = StandardScaler()
node_scaler102 = StandardScaler()

for t in TRAIN_H:

    edge_scaler102.partial_fit(
        raw102_h[t]["edge"]
    )

    node_scaler102.partial_fit(
        raw102_h[t]["node"]
    )

expected_edge_samples = M_H * len(TRAIN_H)
expected_node_samples = N_H * len(TRAIN_H)

assert int(edge_scaler102.n_samples_seen_) == (
    expected_edge_samples
)

assert int(node_scaler102.n_samples_seen_) == (
    expected_node_samples
)

print(
    "Edge scaler training samples:",
    edge_scaler102.n_samples_seen_
)

print(
    "Node scaler training samples:",
    node_scaler102.n_samples_seen_
)

# ============================================================
# 5. Create seed-102 CUDA model inputs
# ============================================================

print("\n[4] CONSTRUCT CUDA TENSORS")

data102 = {}

static_types_h = type_onehot.astype(
    np.float32, copy=False
)

for t in TIMES_H:

    r = raw102_h[t]

    edge_scaled = edge_scaler102.transform(
        r["edge"]
    ).astype(np.float32)

    node_scaled = node_scaler102.transform(
        r["node"]
    ).astype(np.float32)

    node_input = np.concatenate(
        [static_types_h, node_scaled],
        axis=1
    )

    assert edge_scaled.shape == (M_H, 17)
    assert node_input.shape == (N_H, 20)
    assert node_scaled.shape == (N_H, 4)

    assert np.isfinite(edge_scaled).all()
    assert np.isfinite(node_input).all()

    data102[t] = {
        "time": t,
        "xn": torch.as_tensor(
            node_input,
            device=DEVICE_H
        ),
        "xe": torch.as_tensor(
            edge_scaled,
            device=DEVICE_H
        ),
        "xc": torch.as_tensor(
            node_scaled,
            device=DEVICE_H
        ),
        "target": torch.as_tensor(
            r["target"],
            dtype=torch.long,
            device=DEVICE_H
        ),
        "y": torch.as_tensor(
            r["y"],
            dtype=torch.float32,
            device=DEVICE_H
        ),
        "count": r["count"]
    }

    batch = data102[t]

    assert batch["xn"].shape == (N_H, 20)
    assert batch["xe"].shape == (M_H, 17)
    assert batch["xc"].shape == (N_H, 4)

    assert torch.equal(
        batch["xn"][:, 16:],
        batch["xc"]
    )

    assert int(batch["target"].min()) >= 0
    assert int(batch["target"].max()) < M_H

    print(
        f"t={t} | "
        f"events={batch['count']:6,d} | "
        f"failure_rate="
        f"{batch['y'].mean().item():.4f} | "
        f"device={batch['xe'].device}",
        flush=True
    )

# ============================================================
# 6. Save scalers and reproducibility metadata
# ============================================================

report_h = {
    "experiment": "E3.16h",
    "scenario": "B_owner_correlated",
    "simulation_seed": 102,
    "reference_seed": 101,
    "reference_scaler_reproduction": {
        "passed": bool(scalers_match_h),
        "maximum_differences": checks_h
    },
    "train_snapshot_times": TRAIN_H,
    "validation_snapshot_times": [8, 9],
    "scaler_fit_times": TRAIN_H,
    "node_features": 20,
    "edge_features": 17,
    "controller_features": 4,
    "exact_edge_budget": K,
    "training_events_by_time": {
        str(t): int(data102[t]["count"])
        for t in TRAIN_H
    },
    "validation_events_by_time": {
        str(t): int(data102[t]["count"])
        for t in [8, 9]
    },
    "prequential_validation": True,
    "gpu_device": str(DEVICE_H),
    "test_accessed": False,
    "hidden_state_accessed": False,
    "model_training_performed": False,
    "old_seed101_tensors_modified": False
}

SAVE_H.mkdir(parents=True)

joblib.dump(
    edge_scaler102,
    SAVE_H / "edge_scaler102.joblib"
)

joblib.dump(
    node_scaler102,
    SAVE_H / "node_scaler102.joblib"
)

with open(
    SAVE_H / "report.json",
    "w", encoding="utf-8"
) as f:
    json.dump(report_h, f, indent=2)

print("\n" + "=" * 74)
print("E3.16h COMPLETED — SEED 102 INPUTS")
print("=" * 74)

print("Seed 101 scaler verification: PASSED")
print("Seed 102 tensors: READY")
print("Training snapshots:", TRAIN_H)
print("Validation snapshots:", [8, 9])
print("Scaler fitting used validation: NO")
print("Original data dictionary modified: NO")
print("Test accessed: NO")
print("Hidden state accessed: NO")
print("Model training performed: NO")
print("Saved:", SAVE_H / "report.json")


In [ ]:

# ============================================================
# E3.17 — INDEPENDENT SCORER TRAINING ON SCENARIO SEED 102
# ============================================================
# Scenario: B_owner_correlated / simulation seed 102
# Model seed: 44 (fresh initialization)
#
# Compare:
#   1. GNN-based edge scorer
#   2. Trust-only edge scorer
#
# Both models retain GraphSAGE for final prediction.
#
# Fixed 60 epochs, exact 50% edge budget,
# coverage-preserving GPU selector, no controller.
#
# IMPORTANT:
# Uses data102 explicitly throughout.
# Never uses global data for training/evaluation.
# No test or hidden-state access.
# ============================================================

from pathlib import Path
import copy
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    log_loss
)

# ------------------------------------------------------------
# 0. Preconditions
# ------------------------------------------------------------

required = [
    "stable", "data102", "U", "V", "K",
    "coverage_topk_gpu"
]

missing = [
    name for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing notebook objects: {missing}"
    )

ROOT317 = Path(
    "/kaggle/working/phase2_adaptive_siot/"
    "E3_tags_v3/scenarios/B_owner_correlated"
)

SCENARIO317 = ROOT317 / "seed_102"

SAVE317 = (
    SCENARIO317 / "tags_v3" /
    "e3_17_scorer_comparison_seed_44"
)

if SAVE317.exists():
    raise FileExistsError(
        f"Output already exists: {SAVE317}"
    )

SEED317 = 44
EPOCHS317 = 60
LR317 = 1e-3
WEIGHT_DECAY317 = 1e-4
CLIP317 = 1.0
TAU317 = 1.0

TRAIN317 = [5, 6, 7]
VALID317 = [8, 9]

DEVICE317 = U.device
N317 = 16216
M317 = 146117

assert DEVICE317.type == "cuda"
assert U.is_cuda and V.is_cuda
assert int(U.numel()) == M317
assert K == 73058
assert sorted(data102.keys()) == TRAIN317 + VALID317

for t in TRAIN317 + VALID317:
    batch = data102[t]

    assert batch["xn"].shape == (N317, 20)
    assert batch["xe"].shape == (M317, 17)
    assert batch["xc"].shape == (N317, 4)

    assert batch["xn"].device == DEVICE317
    assert batch["xe"].device == DEVICE317
    assert batch["target"].device == DEVICE317
    assert batch["y"].device == DEVICE317

    assert batch["count"] == len(batch["target"])
    assert batch["count"] == len(batch["y"])

# Verify the input preparation artifact exists.
assert (
    SCENARIO317 /
    "e3_16h_model_inputs/report.json"
).is_file()

# ------------------------------------------------------------
# 1. Fresh, matched model initialization
# ------------------------------------------------------------

random.seed(SEED317)
np.random.seed(SEED317)
torch.manual_seed(SEED317)
torch.cuda.manual_seed_all(SEED317)

sample317 = data102[5]

node_dim317 = sample317["xn"].shape[1]
edge_dim317 = sample317["xe"].shape[1]
controller_dim317 = sample317["xc"].shape[1]

base317 = type(stable)(
    node_dim=node_dim317,
    edge_dim=edge_dim317,
    controller_dim=controller_dim317,
    hidden=32
).to(DEVICE317)

gnn317 = copy.deepcopy(base317)
trust317 = copy.deepcopy(base317)

trust317.scorer = nn.Sequential(
    nn.Linear(edge_dim317, 64),
    nn.ReLU(),
    nn.Linear(64, 1)
).to(DEVICE317)

shared_prefixes317 = (
    "encoder.",
    "trust_head.",
    "graph_head."
)

shared_keys317 = [
    name
    for name in gnn317.state_dict()
    if name.startswith(shared_prefixes317)
]

assert shared_keys317

assert all(
    torch.equal(
        gnn317.state_dict()[name],
        trust317.state_dict()[name]
    )
    for name in shared_keys317
)

# Same initialization as the seed-44 architecture
# used in E3.10, subject to the same constructor
# and RNG behavior.

def active_params317(net):
    return [
        p
        for name, p in net.named_parameters()
        if not name.startswith("controller.")
    ]

# ------------------------------------------------------------
# 2. Scenario-scoped forward function
# ------------------------------------------------------------

def forward317(net, batch, use_gnn, training):

    xn = batch["xn"]
    xe = batch["xe"]
    target = batch["target"]

    if use_gnn:
        full_mask = xn.new_ones(M317)

        h = net.encoder(
            xn, U, V, full_mask
        )

        representation = torch.cat([
            h[U] + h[V],
            torch.abs(h[U] - h[V]),
            xe
        ], dim=1)

        scores = net.scorer(
            representation
        ).squeeze(-1)

    else:
        scores = net.scorer(
            xe
        ).squeeze(-1)

    hard, coverage_count = coverage_topk_gpu(
        scores, K
    )

    assert int(hard.sum().item()) == K

    if training:
        threshold = torch.topk(
            scores.detach(), K
        ).values[-1]

        soft = torch.sigmoid(
            (scores - threshold) / TAU317
        )

        mask = (
            hard + soft - soft.detach()
        )
    else:
        mask = hard

    z = net.encoder(
        xn, U, V, mask
    )

    a = U[target]
    b = V[target]

    pair_features = torch.cat([
        z[a] + z[b],
        torch.abs(z[a] - z[b]),
        z[a] * z[b]
    ], dim=1)

    logits = (
        net.trust_head(
            xe[target]
        ).squeeze(-1)
        +
        net.graph_head(
            pair_features
        ).squeeze(-1)
    )

    return logits, hard, coverage_count

# ------------------------------------------------------------
# 3. Validation: ONLY data102[t=8,9]
# ------------------------------------------------------------

@torch.no_grad()
def validate317(net, use_gnn):

    net.eval()

    all_y = []
    all_p = []
    masks = []
    cover_counts = []

    for t in VALID317:

        batch = data102[t]

        logits, hard, coverage_count = forward317(
            net,
            batch,
            use_gnn,
            training=False
        )

        all_y.append(
            batch["y"].detach().cpu().numpy()
        )

        all_p.append(
            torch.sigmoid(logits)
            .detach().cpu().numpy()
        )

        masks.append(
            hard.detach().cpu().numpy().astype(bool)
        )

        cover_counts.append(
            int(coverage_count)
        )

    y = np.concatenate(all_y)

    p = np.clip(
        np.concatenate(all_p),
        1e-7,
        1 - 1e-7
    )

    assert all(int(m.sum()) == K for m in masks)

    return {
        "AP": float(
            average_precision_score(y, p)
        ),
        "AUC": float(
            roc_auc_score(y, p)
        ),
        "BCE": float(
            log_loss(y, p, labels=[0, 1])
        ),
        "churn": float(
            np.count_nonzero(
                masks[0] != masks[1]
            ) / (2 * K)
        ),
        "coverage_t8": cover_counts[0],
        "coverage_t9": cover_counts[1]
    }

# ------------------------------------------------------------
# 4. Train both models with fixed hyperparameters
# ------------------------------------------------------------

results317 = []
history317 = {}
states317 = {}

print("=" * 76)
print("E3.17 — SCENARIO B SEED 102 / MODEL SEED 44")
print("=" * 76)
print("Device:", torch.cuda.get_device_name(0))
print("Simulation seed: 102")
print("Model seed:", SEED317)
print("Training epochs:", EPOCHS317)
print("Train:", TRAIN317)
print("Validation:", VALID317)
print("Retained edges:", K)
print("Controller: DISABLED")
print("GPU selector: ENABLED")
print("Data source: data102 ONLY")
print("Shared initialization: VERIFIED")

for model_name, net, use_gnn in [
    ("GNN_scorer", gnn317, True),
    ("Trust_only_scorer", trust317, False)
]:

    # Same random seed before training.
    random.seed(SEED317)
    np.random.seed(SEED317)
    torch.manual_seed(SEED317)
    torch.cuda.manual_seed_all(SEED317)

    params = active_params317(net)

    optimizer = torch.optim.AdamW(
        params,
        lr=LR317,
        weight_decay=WEIGHT_DECAY317
    )

    rows = []

    best_AP = -float("inf")
    best_epoch = None
    best_state = None
    best_metrics = None

    print("\n" + "-" * 76)
    print("MODEL:", model_name)
    print("-" * 76, flush=True)

    for epoch in range(1, EPOCHS317 + 1):

        net.train()
        losses = []

        for t in TRAIN317:

            batch = data102[t]

            optimizer.zero_grad(
                set_to_none=True
            )

            logits, hard, _ = forward317(
                net,
                batch,
                use_gnn,
                training=True
            )

            loss = F.binary_cross_entropy_with_logits(
                logits,
                batch["y"]
            )

            if not bool(torch.isfinite(loss)):
                raise RuntimeError(
                    f"Non-finite loss: "
                    f"{model_name}, epoch={epoch}, t={t}"
                )

            loss.backward()

            grad_norm = (
                torch.nn.utils.clip_grad_norm_(
                    params,
                    CLIP317
                )
            )

            if not bool(torch.isfinite(grad_norm)):
                raise RuntimeError(
                    f"Non-finite gradient: "
                    f"{model_name}, epoch={epoch}, t={t}"
                )

            optimizer.step()

            losses.append((
                float(loss.item()),
                int(batch["count"])
            ))

        train_BCE = (
            sum(loss * n for loss, n in losses)
            / sum(n for _, n in losses)
        )

        metrics = validate317(
            net,
            use_gnn
        )

        rows.append({
            "epoch": epoch,
            "train_BCE": train_BCE,
            **metrics
        })

        if metrics["AP"] > best_AP + 1e-5:
            best_AP = metrics["AP"]
            best_epoch = epoch
            best_metrics = dict(metrics)
            best_state = copy.deepcopy(
                net.state_dict()
            )

        if epoch == 1 or epoch % 10 == 0:
            print(
                f"epoch={epoch:02d} | "
                f"Train BCE={train_BCE:.6f} | "
                f"Val AP={metrics['AP']:.6f} | "
                f"AUC={metrics['AUC']:.6f} | "
                f"BCE={metrics['BCE']:.6f} | "
                f"churn={metrics['churn']:.4f}",
                flush=True
            )

    final = rows[-1]

    results317.append({
        "model": model_name,
        "simulation_seed": 102,
        "model_seed": SEED317,
        "best_epoch": best_epoch,
        "best_AP": best_metrics["AP"],
        "best_AUC": best_metrics["AUC"],
        "best_BCE": best_metrics["BCE"],
        "epoch60_AP": final["AP"],
        "epoch60_AUC": final["AUC"],
        "epoch60_BCE": final["BCE"],
        "epoch60_churn": final["churn"],
        "coverage_t8": final["coverage_t8"],
        "coverage_t9": final["coverage_t9"]
    })

    history317[model_name] = rows

    states317[model_name] = {
        "best": best_state,
        "epoch60": copy.deepcopy(
            net.state_dict()
        )
    }

# ------------------------------------------------------------
# 5. Paired comparison
# ------------------------------------------------------------

comparison317 = pd.DataFrame(results317)

gnn_row = comparison317.loc[
    comparison317["model"] == "GNN_scorer"
].iloc[0]

trust_row = comparison317.loc[
    comparison317["model"] == "Trust_only_scorer"
].iloc[0]

metrics_to_compare = [
    "epoch60_AP",
    "epoch60_AUC",
    "epoch60_BCE",
    "epoch60_churn"
]

deltas317 = {
    metric: float(
        gnn_row[metric] - trust_row[metric]
    )
    for metric in metrics_to_compare
}

# ------------------------------------------------------------
# 6. Save new artifacts
# ------------------------------------------------------------

SAVE317.mkdir(parents=True)

comparison317.to_csv(
    SAVE317 / "comparison.csv",
    index=False
)

for name, rows in history317.items():

    pd.DataFrame(rows).to_csv(
        SAVE317 / f"{name}_history.csv",
        index=False
    )

    torch.save(
        {
            "best_model_state_dict":
                states317[name]["best"],
            "epoch60_model_state_dict":
                states317[name]["epoch60"],
            "best_epoch": next(
                r["best_epoch"]
                for r in results317
                if r["model"] == name
            ),
            "model_seed": SEED317,
            "simulation_seed": 102,
            "method": name,
            "epochs": EPOCHS317
        },
        SAVE317 / f"{name}_checkpoint.pt"
    )

validation_failure_rate = float(
    (
        data102[8]["y"].sum()
        + data102[9]["y"].sum()
    ).item()
    /
    (
        data102[8]["count"]
        + data102[9]["count"]
    )
)

report317 = {
    "experiment": "E3.17",
    "scenario": "B_owner_correlated",
    "simulation_seed": 102,
    "model_seed": SEED317,
    "training_times": TRAIN317,
    "validation_times": VALID317,
    "epochs": EPOCHS317,
    "checkpoint_for_main_comparison": "epoch60",
    "validation_failure_rate": validation_failure_rate,
    "matched_shared_initialization": True,
    "fresh_initialization": True,
    "exact_edge_budget": K,
    "controller_enabled": False,
    "gpu_selector": True,
    "results": results317,
    "gnn_minus_trust_only": deltas317,
    "test_accessed": False,
    "hidden_state_accessed": False,
    "limitations": [
        "Single model initialization for scenario seed 102.",
        "Validation used for exploratory analysis.",
        "Both methods retain GNN for final prediction.",
        "Scorer architectures have different inputs.",
        "No claim of statistical significance."
    ]
}

with open(
    SAVE317 / "report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        report317,
        f,
        indent=2
    )

print("\n" + "=" * 76)
print("E3.17 COMPLETED — INDEPENDENT SCENARIO REPLICATION")
print("=" * 76)

print(
    comparison317.round(6).to_string(
        index=False
    )
)

print(
    "\nValidation failure prevalence:",
    f"{validation_failure_rate:.6f}"
)

print("\nGNN MINUS TRUST-ONLY AT EPOCH 60")

for metric, value in deltas317.items():
    print(
        f"{metric}: {value:+.6f}"
    )

print("\nNo previous artifacts overwritten.")
print("Original data dictionary unchanged.")
print("No test or hidden-state access.")
print("Saved:", SAVE317 / "report.json")


In [ ]:

# ============================================================
# E3.18 — SCENARIO 102 / FRESH MODEL SEED 45
# ============================================================
# Replicate E3.17 without changing the architecture,
# training procedure, evaluation or data.
#
# GNN scorer vs Trust-only scorer.
# 60 epochs, GPU selector, exact 50% edge budget.
#
# No controller, no test, no hidden-state access.
# ============================================================

from pathlib import Path
import copy
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

required = [
    "stable", "data102", "U", "V", "K",
    "forward317", "validate317",
    "coverage_topk_gpu", "base317"
]

missing = [x for x in required if x not in globals()]
assert not missing, f"Missing notebook objects: {missing}"

SEED318 = 45
EPOCHS318 = 60
TRAIN318 = [5, 6, 7]
VALID318 = [8, 9]

SAVE318 = Path(
    "/kaggle/working/phase2_adaptive_siot/"
    "E3_tags_v3/scenarios/B_owner_correlated/"
    "seed_102/tags_v3/e3_18_scorer_comparison_seed_45"
)

if SAVE318.exists():
    raise FileExistsError(
        f"Experiment already exists: {SAVE318}"
    )

assert U.is_cuda and V.is_cuda
assert K == 73058
assert sorted(data102.keys()) == TRAIN318 + VALID318

# ------------------------------------------------------------
# 1. Fresh model initialization
# ------------------------------------------------------------

random.seed(SEED318)
np.random.seed(SEED318)
torch.manual_seed(SEED318)
torch.cuda.manual_seed_all(SEED318)

sample = data102[5]

base318 = type(stable)(
    node_dim=sample["xn"].shape[1],
    edge_dim=sample["xe"].shape[1],
    controller_dim=sample["xc"].shape[1],
    hidden=32
).to(U.device)

gnn318 = copy.deepcopy(base318)
trust318 = copy.deepcopy(base318)

trust318.scorer = nn.Sequential(
    nn.Linear(sample["xe"].shape[1], 64),
    nn.ReLU(),
    nn.Linear(64, 1)
).to(U.device)

shared_keys = [
    name for name in gnn318.state_dict()
    if name.startswith((
        "encoder.",
        "trust_head.",
        "graph_head."
    ))
]

assert shared_keys

shared_match = all(
    torch.equal(
        gnn318.state_dict()[name],
        trust318.state_dict()[name]
    )
    for name in shared_keys
)

fresh_match = any(
    not torch.equal(
        base318.state_dict()[name],
        base317.state_dict()[name]
    )
    for name in shared_keys
)

assert shared_match
assert fresh_match

def trainable318(net):
    return [
        p for name, p in net.named_parameters()
        if not name.startswith("controller.")
    ]

# ------------------------------------------------------------
# 2. Train independently
# ------------------------------------------------------------

results318 = []
histories318 = {}
states318 = {}

print("=" * 76)
print("E3.18 — SCENARIO SEED 102 / MODEL SEED 45")
print("=" * 76)

print("GPU:", torch.cuda.get_device_name(0))
print("Scenario seed: 102")
print("Model seed:", SEED318)
print("Fresh initialization:", fresh_match)
print("Shared initialization:", shared_match)
print("Epochs:", EPOCHS318)
print("Train:", TRAIN318)
print("Validation:", VALID318)
print("Retained edges:", K)
print("Controller: NO")
print("Data source: data102 ONLY")

for model_name, net, use_gnn in [
    ("GNN_scorer", gnn318, True),
    ("Trust_only_scorer", trust318, False)
]:

    random.seed(SEED318)
    np.random.seed(SEED318)
    torch.manual_seed(SEED318)
    torch.cuda.manual_seed_all(SEED318)

    params = trainable318(net)

    optimizer = torch.optim.AdamW(
        params,
        lr=1e-3,
        weight_decay=1e-4
    )

    rows = []
    best_ap = -np.inf
    best_epoch = None
    best_metrics = None
    best_state = None

    print("\n" + "-" * 76)
    print("MODEL:", model_name)
    print("-" * 76, flush=True)

    for epoch in range(1, EPOCHS318 + 1):

        net.train()
        losses = []

        for t in TRAIN318:

            batch = data102[t]

            optimizer.zero_grad(
                set_to_none=True
            )

            logits, hard, _ = forward317(
                net, batch, use_gnn, True
            )

            loss = F.binary_cross_entropy_with_logits(
                logits, batch["y"]
            )

            if not bool(torch.isfinite(loss)):
                raise RuntimeError(
                    f"Non-finite loss: "
                    f"{model_name}, epoch={epoch}, t={t}"
                )

            loss.backward()

            grad_norm = torch.nn.utils.clip_grad_norm_(
                params, max_norm=1.0
            )

            if not bool(torch.isfinite(grad_norm)):
                raise RuntimeError(
                    f"Non-finite gradient: "
                    f"{model_name}, epoch={epoch}, t={t}"
                )

            optimizer.step()

            losses.append((
                float(loss.item()),
                int(batch["count"])
            ))

        train_bce = sum(
            loss * count
            for loss, count in losses
        ) / sum(
            count for _, count in losses
        )

        # Explicitly evaluates data102[8] and [9].
        metrics = validate317(net, use_gnn)

        rows.append({
            "epoch": epoch,
            "train_BCE": train_bce,
            **metrics
        })

        if metrics["AP"] > best_ap + 1e-5:
            best_ap = metrics["AP"]
            best_epoch = epoch
            best_metrics = dict(metrics)
            best_state = copy.deepcopy(
                net.state_dict()
            )

        if epoch == 1 or epoch % 10 == 0:
            print(
                f"epoch={epoch:02d} | "
                f"Train BCE={train_bce:.6f} | "
                f"Val AP={metrics['AP']:.6f} | "
                f"AUC={metrics['AUC']:.6f} | "
                f"BCE={metrics['BCE']:.6f} | "
                f"churn={metrics['churn']:.4f}",
                flush=True
            )

    final = rows[-1]

    results318.append({
        "model": model_name,
        "simulation_seed": 102,
        "model_seed": SEED318,
        "best_epoch": best_epoch,
        "best_AP": best_metrics["AP"],
        "best_AUC": best_metrics["AUC"],
        "best_BCE": best_metrics["BCE"],
        "epoch60_AP": final["AP"],
        "epoch60_AUC": final["AUC"],
        "epoch60_BCE": final["BCE"],
        "epoch60_churn": final["churn"],
        "coverage_t8": final["coverage_t8"],
        "coverage_t9": final["coverage_t9"]
    })

    histories318[model_name] = rows
    states318[model_name] = {
        "best": best_state,
        "epoch60": copy.deepcopy(net.state_dict())
    }

# ------------------------------------------------------------
# 3. Paired comparison
# ------------------------------------------------------------

comparison318 = pd.DataFrame(results318)

gnn_row = comparison318[
    comparison318["model"] == "GNN_scorer"
].iloc[0]

trust_row = comparison318[
    comparison318["model"] == "Trust_only_scorer"
].iloc[0]

deltas318 = {
    metric: float(
        gnn_row[metric] - trust_row[metric]
    )
    for metric in [
        "epoch60_AP",
        "epoch60_AUC",
        "epoch60_BCE",
        "epoch60_churn"
    ]
}

# ------------------------------------------------------------
# 4. Persist independently
# ------------------------------------------------------------

SAVE318.mkdir(parents=True)

comparison318.to_csv(
    SAVE318 / "comparison.csv",
    index=False
)

for name, rows in histories318.items():

    pd.DataFrame(rows).to_csv(
        SAVE318 / f"{name}_history.csv",
        index=False
    )

    torch.save(
        {
            "best_model_state_dict":
                states318[name]["best"],
            "epoch60_model_state_dict":
                states318[name]["epoch60"],
            "model_seed": SEED318,
            "simulation_seed": 102,
            "model_type": name
        },
        SAVE318 / f"{name}_checkpoint.pt"
    )

report318 = {
    "experiment": "E3.18",
    "scenario": "B_owner_correlated",
    "simulation_seed": 102,
    "model_seed": 45,
    "training_epochs": EPOCHS318,
    "train_times": TRAIN318,
    "validation_times": VALID318,
    "fixed_checkpoint": "epoch60",
    "fresh_initialization_verified": fresh_match,
    "shared_initialization_verified": shared_match,
    "results": results318,
    "gnn_minus_trust_only": deltas318,
    "exact_edge_budget": K,
    "controller_enabled": False,
    "gpu_selector_enabled": True,
    "test_accessed": False,
    "hidden_state_accessed": False,
    "limitations": [
        "Exploratory validation comparison.",
        "Same scenario realization as E3.17.",
        "Different scorer architectures.",
        "Both predictors use GraphSAGE.",
        "No statistical significance claim."
    ]
}

with open(
    SAVE318 / "report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(report318, f, indent=2)

print("\n" + "=" * 76)
print("E3.18 COMPLETED — SCENARIO 102 / MODEL SEED 45")
print("=" * 76)

print(
    comparison318.round(6).to_string(index=False)
)

print("\nGNN MINUS TRUST-ONLY AT EPOCH 60")
for metric, value in deltas318.items():
    print(f"{metric}: {value:+.6f}")

print("\nPrevious artifacts overwritten: NO")
print("Test accessed: NO")
print("Hidden simulator state accessed: NO")
print("Saved:", SAVE318 / "report.json")


In [ ]:

# ============================================================
# E3.19 — SCENARIO 102 / FRESH MODEL SEED 46
# ============================================================
# Same architecture and protocol as E3.17 and E3.18.
# Fixed 60 epochs, no controller, GPU selector.
# Explicitly uses data102 only.
# No test or hidden simulator-state access.
# ============================================================

from pathlib import Path
import copy
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

required = [
    "stable", "data102", "U", "V", "K",
    "forward317", "validate317", "base318"
]
missing = [x for x in required if x not in globals()]
assert not missing, f"Missing objects: {missing}"

SEED319 = 46
EPOCHS319 = 60
TRAIN319 = [5, 6, 7]
VAL319 = [8, 9]

SAVE319 = Path(
    "/kaggle/working/phase2_adaptive_siot/"
    "E3_tags_v3/scenarios/B_owner_correlated/"
    "seed_102/tags_v3/e3_19_scorer_comparison_seed_46"
)

if SAVE319.exists():
    raise FileExistsError(
        f"Output already exists: {SAVE319}"
    )

assert U.is_cuda and V.is_cuda
assert K == 73058
assert sorted(data102.keys()) == TRAIN319 + VAL319

# ------------------------------------------------------------
# 1. Fresh matched initialization
# ------------------------------------------------------------

random.seed(SEED319)
np.random.seed(SEED319)
torch.manual_seed(SEED319)
torch.cuda.manual_seed_all(SEED319)

sample = data102[5]

base319 = type(stable)(
    node_dim=sample["xn"].shape[1],
    edge_dim=sample["xe"].shape[1],
    controller_dim=sample["xc"].shape[1],
    hidden=32
).to(U.device)

gnn319 = copy.deepcopy(base319)
trust319 = copy.deepcopy(base319)

trust319.scorer = nn.Sequential(
    nn.Linear(sample["xe"].shape[1], 64),
    nn.ReLU(),
    nn.Linear(64, 1)
).to(U.device)

shared_keys = [
    name for name in gnn319.state_dict()
    if name.startswith((
        "encoder.", "trust_head.", "graph_head."
    ))
]

assert shared_keys

shared_equal = all(
    torch.equal(
        gnn319.state_dict()[name],
        trust319.state_dict()[name]
    )
    for name in shared_keys
)

fresh_different = any(
    not torch.equal(
        base319.state_dict()[name],
        base318.state_dict()[name]
    )
    for name in shared_keys
)

assert shared_equal and fresh_different

def active_params319(net):
    return [
        p for name, p in net.named_parameters()
        if not name.startswith("controller.")
    ]

# ------------------------------------------------------------
# 2. Independent fixed-budget training
# ------------------------------------------------------------

results319 = []
histories319 = {}
states319 = {}

print("=" * 76)
print("E3.19 — SCENARIO SEED 102 / MODEL SEED 46")
print("=" * 76)
print("GPU:", torch.cuda.get_device_name(0))
print("Scenario seed: 102")
print("Model seed:", SEED319)
print("Fresh initialization:", fresh_different)
print("Matched shared parameters:", shared_equal)
print("Training epochs:", EPOCHS319)
print("Exact edge budget:", K)
print("Data source: data102 ONLY")

for name, net, use_gnn in [
    ("GNN_scorer", gnn319, True),
    ("Trust_only_scorer", trust319, False)
]:
    random.seed(SEED319)
    np.random.seed(SEED319)
    torch.manual_seed(SEED319)
    torch.cuda.manual_seed_all(SEED319)

    params = active_params319(net)

    optimizer = torch.optim.AdamW(
        params, lr=1e-3, weight_decay=1e-4
    )

    rows = []
    best_ap = -np.inf
    best_epoch = None
    best_state = None
    best_metrics = None

    print(f"\nMODEL: {name}", flush=True)

    for epoch in range(1, EPOCHS319 + 1):
        net.train()
        losses = []

        for t in TRAIN319:
            batch = data102[t]

            optimizer.zero_grad(set_to_none=True)

            logits, hard, _ = forward317(
                net, batch, use_gnn, True
            )

            loss = F.binary_cross_entropy_with_logits(
                logits, batch["y"]
            )

            if not bool(torch.isfinite(loss)):
                raise RuntimeError(
                    f"Non-finite loss: {name}, "
                    f"epoch={epoch}, t={t}"
                )

            loss.backward()

            grad = torch.nn.utils.clip_grad_norm_(
                params, max_norm=1.0
            )
            assert bool(torch.isfinite(grad))

            optimizer.step()

            losses.append((
                float(loss.item()), int(batch["count"])
            ))

        train_bce = sum(
            loss * n for loss, n in losses
        ) / sum(n for _, n in losses)

        # Uses only data102[8] and data102[9].
        metrics = validate317(net, use_gnn)

        rows.append({
            "epoch": epoch,
            "train_BCE": train_bce,
            **metrics
        })

        if metrics["AP"] > best_ap + 1e-5:
            best_ap = metrics["AP"]
            best_epoch = epoch
            best_metrics = dict(metrics)
            best_state = copy.deepcopy(
                net.state_dict()
            )

        if epoch == 1 or epoch % 10 == 0:
            print(
                f"epoch={epoch:02d} | "
                f"Train BCE={train_bce:.6f} | "
                f"AP={metrics['AP']:.6f} | "
                f"AUC={metrics['AUC']:.6f} | "
                f"BCE={metrics['BCE']:.6f} | "
                f"churn={metrics['churn']:.4f}",
                flush=True
            )

    final = rows[-1]

    results319.append({
        "model": name,
        "simulation_seed": 102,
        "model_seed": SEED319,
        "best_epoch": best_epoch,
        "best_AP": best_metrics["AP"],
        "best_AUC": best_metrics["AUC"],
        "best_BCE": best_metrics["BCE"],
        "epoch60_AP": final["AP"],
        "epoch60_AUC": final["AUC"],
        "epoch60_BCE": final["BCE"],
        "epoch60_churn": final["churn"],
        "coverage_t8": final["coverage_t8"],
        "coverage_t9": final["coverage_t9"]
    })

    histories319[name] = rows
    states319[name] = {
        "best": best_state,
        "epoch60": copy.deepcopy(net.state_dict())
    }

# ------------------------------------------------------------
# 3. Save reproducible outputs
# ------------------------------------------------------------

comparison319 = pd.DataFrame(results319)

gnn_row = comparison319[
    comparison319["model"] == "GNN_scorer"
].iloc[0]

trust_row = comparison319[
    comparison319["model"] == "Trust_only_scorer"
].iloc[0]

deltas319 = {
    metric: float(
        gnn_row[metric] - trust_row[metric]
    )
    for metric in [
        "epoch60_AP",
        "epoch60_AUC",
        "epoch60_BCE",
        "epoch60_churn"
    ]
}

SAVE319.mkdir(parents=True)

comparison319.to_csv(
    SAVE319 / "comparison.csv",
    index=False
)

for name, rows in histories319.items():
    pd.DataFrame(rows).to_csv(
        SAVE319 / f"{name}_history.csv",
        index=False
    )

    torch.save(
        {
            "best_model_state_dict":
                states319[name]["best"],
            "epoch60_model_state_dict":
                states319[name]["epoch60"],
            "model_seed": SEED319,
            "simulation_seed": 102,
            "method": name,
            "epochs": EPOCHS319
        },
        SAVE319 / f"{name}_checkpoint.pt"
    )

report319 = {
    "experiment": "E3.19",
    "scenario": "B_owner_correlated",
    "simulation_seed": 102,
    "model_seed": 46,
    "training_epochs": EPOCHS319,
    "checkpoint_for_main_comparison": "epoch60",
    "train_times": TRAIN319,
    "validation_times": VAL319,
    "fresh_initialization_verified": True,
    "shared_initialization_verified": True,
    "results": results319,
    "gnn_minus_trust_only": deltas319,
    "controller_enabled": False,
    "exact_edge_budget": K,
    "gpu_selector_enabled": True,
    "test_accessed": False,
    "hidden_state_accessed": False,
    "limitations": [
        "Exploratory validation only.",
        "Shared simulation realization within seed 102.",
        "Scorer architectures differ.",
        "Both predictors contain GraphSAGE.",
        "No statistical significance claim."
    ]
}

with open(
    SAVE319 / "report.json",
    "w", encoding="utf-8"
) as f:
    json.dump(report319, f, indent=2)

print("\n" + "=" * 76)
print("E3.19 COMPLETED — SCENARIO 102 / MODEL SEED 46")
print("=" * 76)

print(
    comparison319.round(6).to_string(index=False)
)

print("\nGNN MINUS TRUST-ONLY AT EPOCH 60")
for metric, value in deltas319.items():
    print(f"{metric}: {value:+.6f}")

print("\nPrevious artifacts overwritten: NO")
print("No test or hidden-state access.")
print("Saved:", SAVE319 / "report.json")


In [ ]:

# ============================================================
# E3.20 — TWO-SCENARIO PAIRED ABLATION SUMMARY
# ============================================================
# Scenario seeds: 101, 102
# Fresh model seeds: 44, 45, 46
# Primary checkpoint: epoch 60
#
# No training or model evaluation.
# Only existing reports and validation labels.
# No test or hidden-state access.
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT320 = Path(
    "/kaggle/working/phase2_adaptive_siot/"
    "E3_tags_v3/scenarios/B_owner_correlated"
)

SOURCES320 = {
    (101, 44): "e3_10_fresh_seed_44",
    (101, 45): "e3_11_fresh_seed_45",
    (101, 46): "e3_12_fresh_seed_46",
    (102, 44): "e3_17_scorer_comparison_seed_44",
    (102, 45): "e3_18_scorer_comparison_seed_45",
    (102, 46): "e3_19_scorer_comparison_seed_46",
}

OUT320 = (
    ROOT320 / "e3_20_two_scenario_aggregation"
)

if OUT320.exists():
    raise FileExistsError(
        f"Output directory already exists: {OUT320}"
    )

rows320 = []
prevalence320 = {}

# ------------------------------------------------------------
# 1. Read validation prevalence per scenario
# ------------------------------------------------------------

for scenario_seed in [101, 102]:

    path = (
        ROOT320 / f"seed_{scenario_seed}"
        / "observed/validation.csv.gz"
    )

    val = pd.read_csv(path)

    assert set(val["time"].unique()) == {8, 9}
    assert val["success"].isin([0, 1]).all()

    prevalence320[scenario_seed] = float(
        (1 - val["success"]).mean()
    )

# ------------------------------------------------------------
# 2. Collect all six paired experiments
# ------------------------------------------------------------

for (scenario_seed, model_seed), folder in SOURCES320.items():

    path = (
        ROOT320 / f"seed_{scenario_seed}"
        / "tags_v3" / folder / "report.json"
    )

    assert path.is_file(), f"Missing report: {path}"

    with open(path, encoding="utf-8") as f:
        report = json.load(f)

    assert report["test_accessed"] is False
    assert report["hidden_state_accessed"] is False

    results = {
        item["model"]: item
        for item in report["results"]
    }

    assert set(results) == {
        "GNN_scorer",
        "Trust_only_scorer"
    }

    gnn = results["GNN_scorer"]
    trust = results["Trust_only_scorer"]

    for model in [gnn, trust]:
        assert int(model["model_seed"]) == model_seed if (
            "model_seed" in model
        ) else int(model["seed"]) == model_seed

    entry = {
        "scenario_seed": scenario_seed,
        "model_seed": model_seed,
        "failure_prevalence": prevalence320[
            scenario_seed
        ]
    }

    for metric in ["AP", "AUC", "BCE", "churn"]:

        key = f"epoch60_{metric}"

        a = float(gnn[key])
        b = float(trust[key])

        entry[f"GNN_{metric}"] = a
        entry[f"Trust_{metric}"] = b
        entry[f"delta_{metric}"] = a - b

    # Useful for cross-scenario descriptive context:
    # AP normalized against each scenario's baseline.
    p = prevalence320[scenario_seed]

    entry["GNN_AP_excess"] = entry["GNN_AP"] - p
    entry["Trust_AP_excess"] = entry["Trust_AP"] - p

    rows320.append(entry)

paired320 = pd.DataFrame(rows320).sort_values(
    ["scenario_seed", "model_seed"]
).reset_index(drop=True)

assert len(paired320) == 6
assert paired320["scenario_seed"].nunique() == 2

# ------------------------------------------------------------
# 3. Aggregate within each scenario
# ------------------------------------------------------------

metrics320 = ["AP", "AUC", "BCE", "churn"]

grouped320 = []

for seed, group in paired320.groupby(
    "scenario_seed", sort=True
):

    assert len(group) == 3

    record = {
        "scenario_seed": int(seed),
        "n_model_seeds": len(group),
        "failure_prevalence": prevalence320[int(seed)]
    }

    for metric in metrics320:

        delta = group[
            f"delta_{metric}"
        ].to_numpy(np.float64)

        record[f"GNN_mean_{metric}"] = float(
            group[f"GNN_{metric}"].mean()
        )

        record[f"Trust_mean_{metric}"] = float(
            group[f"Trust_{metric}"].mean()
        )

        record[f"delta_mean_{metric}"] = float(
            delta.mean()
        )

        record[f"delta_sd_{metric}"] = float(
            delta.std(ddof=1)
        )

    grouped320.append(record)

scenario_summary320 = pd.DataFrame(grouped320)

# ------------------------------------------------------------
# 4. Descriptive across-scenario result
# ------------------------------------------------------------

scenario_delta_ap = (
    scenario_summary320[
        "delta_mean_AP"
    ].to_numpy(np.float64)
)

overall320 = {
    "n_scenario_seeds": 2,
    "model_seeds_per_scenario": 3,
    "mean_scenario_delta_AP": float(
        scenario_delta_ap.mean()
    ),
    "sd_scenario_delta_AP": float(
        scenario_delta_ap.std(ddof=1)
    ),
    "mean_scenario_delta_AUC": float(
        scenario_summary320[
            "delta_mean_AUC"
        ].mean()
    ),
    "mean_scenario_delta_BCE": float(
        scenario_summary320[
            "delta_mean_BCE"
        ].mean()
    ),
    "mean_scenario_delta_churn": float(
        scenario_summary320[
            "delta_mean_churn"
        ].mean()
    ),
    "GNN_lower_churn_all_six": bool(
        (paired320["delta_churn"] < 0).all()
    )
}

# ------------------------------------------------------------
# 5. Persist analysis without overwriting
# ------------------------------------------------------------

OUT320.mkdir(parents=True)

paired320.to_csv(
    OUT320 / "paired_runs.csv",
    index=False
)

scenario_summary320.to_csv(
    OUT320 / "scenario_summary.csv",
    index=False
)

report320 = {
    "experiment": "E3.20",
    "comparison": "GNN scorer vs Trust-only scorer",
    "checkpoint": "epoch60",
    "scenario_seeds": [101, 102],
    "model_seeds": [44, 45, 46],
    "scenario_summaries": grouped320,
    "overall_descriptive_summary": overall320,
    "interpretation": (
        "Model initializations are nested within "
        "simulation scenarios. Two scenario seeds "
        "are insufficient for reliable inference "
        "about population-level superiority."
    ),
    "test_accessed": False,
    "hidden_state_accessed": False,
    "limitations": [
        "Only two independent simulator realizations.",
        "Validation used during architecture development.",
        "Both alternatives retain prediction GraphSAGE.",
        "Different edge-scorer input architectures.",
        "Churn is not a measure of adaptation quality.",
        "No significance or equivalence claim."
    ]
}

with open(
    OUT320 / "report.json",
    "w", encoding="utf-8"
) as f:
    json.dump(report320, f, indent=2)

# ------------------------------------------------------------
# 6. Console summary
# ------------------------------------------------------------

print("=" * 76)
print("E3.20 — TWO-SCENARIO PAIRED ABLATION")
print("=" * 76)

print("\nSIX PAIRED RUNS — EPOCH 60")

print(
    paired320[[
        "scenario_seed",
        "model_seed",
        "GNN_AP",
        "Trust_AP",
        "delta_AP",
        "delta_AUC",
        "delta_BCE",
        "delta_churn"
    ]].round(6).to_string(index=False)
)

print("\nSCENARIO-LEVEL SUMMARY")

print(
    scenario_summary320[[
        "scenario_seed",
        "failure_prevalence",
        "GNN_mean_AP",
        "Trust_mean_AP",
        "delta_mean_AP",
        "delta_sd_AP",
        "delta_mean_AUC",
        "delta_mean_BCE",
        "delta_mean_churn"
    ]].round(6).to_string(index=False)
)

print("\nDESCRIPTIVE CROSS-SCENARIO RESULT")

for key, value in overall320.items():
    print(f"{key}: {value}")

print("\nE3.20 COMPLETED")
print("No training or model inference performed.")
print("No test or hidden-state access.")
print("No previous artifacts overwritten.")
print("Saved:", OUT320 / "report.json")


In [ ]:

# ============================================================
# E3.21 — SCENARIO B, INDEPENDENT SIMULATION SEED 103
# ============================================================
# Same generator and RNG order as E3.14.
# Only simulation seed changes: 102 -> 103.
#
# No training or evaluation.
# Test outcomes generated and sealed.
# No existing artifacts overwritten.
# ============================================================

from pathlib import Path
import json
import zipfile
import hashlib
import numpy as np
import pandas as pd

ROOT321 = Path("/kaggle/working/phase2_adaptive_siot")
E3_321 = ROOT321 / "E3_tags_v3"

REFERENCE321 = (
    E3_321 / "scenarios/B_owner_correlated/seed_101"
)

SEED321 = 103
N321 = 16216
M321 = 146117
T321 = 12

INTERACTION_RATE321 = 0.15
OWNER_STD321 = 0.65
EDGE_STD321 = 0.45
TEMPORAL_RHO321 = 0.85
TEMPORAL_STD321 = 0.12
FLIP321 = 0.03
SHOCK_START321 = 6
SHOCK_FRACTION321 = 0.08
SHOCK_LOGIT321 = -1.6
SHOCK_DECAY321 = 0.90

OUT321 = (
    E3_321 /
    "scenarios/B_owner_correlated/seed_103"
)

assert not OUT321.exists(), (
    f"Output already exists: {OUT321}"
)

with open(
    REFERENCE321 / "e3_1_scenario_b_protocol.json",
    encoding="utf-8"
) as f:
    reference321 = json.load(f)

params321 = {
    "base_reliability": "Beta(5,2)",
    "owner_effect_std": OWNER_STD321,
    "edge_effect_std": EDGE_STD321,
    "temporal_rho": TEMPORAL_RHO321,
    "temporal_std": TEMPORAL_STD321,
    "interaction_rate": INTERACTION_RATE321,
    "observation_flip": FLIP321,
    "shock_start": SHOCK_START321,
    "shock_fraction": SHOCK_FRACTION321,
    "shock_logit": SHOCK_LOGIT321,
    "shock_decay": SHOCK_DECAY321
}

assert reference321["parameters"] == params321

# ------------------------------------------------------------
# 1. Load the fixed real-world graph
# ------------------------------------------------------------

topology321 = pd.read_csv(
    REFERENCE321 /
    "observed/siot_social_topology.csv.gz"
).sort_values(
    ["u", "v"]
).reset_index(drop=True)

assert len(topology321) == M321

u321 = topology321["u"].to_numpy(np.int32) - 1
v321 = topology321["v"].to_numpy(np.int32) - 1

assert np.all(u321 < v321)
assert u321.min() >= 0
assert v321.max() < N321

# Same device-owner metadata source as E3.14.
owner_archive321 = (
    ROOT321 /
    "E1_trust_scoring/santander/raw/"
    "objects_description.zip"
)

with zipfile.ZipFile(owner_archive321) as z:
    with z.open("objects_description.csv") as f:
        devices321 = pd.read_csv(f)

devices321 = devices321.sort_values(
    "id_device"
).reset_index(drop=True)

assert np.array_equal(
    devices321["id_device"].to_numpy(),
    np.arange(1, N321 + 1)
)

owner_codes321, owner_ids321 = pd.factorize(
    devices321["id_user"],
    sort=True
)

O321 = len(owner_ids321)

assert O321 == 4001
assert int(
    (owner_codes321[u321] ==
     owner_codes321[v321]).sum()
) == 65667

# ------------------------------------------------------------
# 2. Generate synthetic latent state
# ------------------------------------------------------------

rng321 = np.random.default_rng(SEED321)

def sigmoid321(x):
    return 1.0 / (
        1.0 + np.exp(-np.clip(x, -30, 30))
    )

def logit321(p):
    p = np.clip(p, 1e-6, 1.0 - 1e-6)
    return np.log(p / (1.0 - p))

base_reliability321 = rng321.beta(
    5.0, 2.0, size=N321
)

owner_effect321 = rng321.normal(
    0.0, OWNER_STD321, size=O321
)

device_owner_effect321 = (
    owner_effect321[owner_codes321]
)

edge_effect321 = rng321.normal(
    0.0, EDGE_STD321, size=M321
)

base_logit321 = logit321(
    base_reliability321
)

shock_count321 = int(round(
    SHOCK_FRACTION321 * N321
))

shocked_nodes321 = rng321.choice(
    N321,
    size=shock_count321,
    replace=False
)

shock_indicator321 = np.zeros(
    N321, dtype=np.float32
)

shock_indicator321[
    shocked_nodes321
] = 1.0

temporal_state321 = np.zeros(
    N321, dtype=np.float32
)

# ------------------------------------------------------------
# 3. Generate interactions with the same RNG call order
# ------------------------------------------------------------

frames321 = []
audit321 = []

print("=" * 72)
print("E3.21 — SCENARIO B / SIMULATION SEED 103")
print("=" * 72)
print("Nodes:", N321)
print("Owners:", O321)
print("Edges:", M321)
print("Parameters match E3.1:", True)

for t in range(T321):

    temporal_state321 = (
        TEMPORAL_RHO321 * temporal_state321
        + rng321.normal(
            0,
            TEMPORAL_STD321,
            size=N321
        )
    ).astype(np.float32)

    if t >= SHOCK_START321:
        shock_strength = (
            SHOCK_LOGIT321
            * SHOCK_DECAY321 **
            (t - SHOCK_START321)
        )
    else:
        shock_strength = 0.0

    hidden_device_logit = (
        base_logit321
        + device_owner_effect321
        + temporal_state321
        + shock_strength * shock_indicator321
    )

    hidden_link_logit = (
        0.5 * (
            hidden_device_logit[u321]
            + hidden_device_logit[v321]
        )
        + edge_effect321
    )

    success_probability = sigmoid321(
        hidden_link_logit
    )

    selected = np.flatnonzero(
        rng321.random(M321)
        < INTERACTION_RATE321
    )

    success = (
        rng321.random(len(selected))
        < success_probability[selected]
    ).astype(np.int8)

    flip = (
        rng321.random(len(selected))
        < FLIP321
    )

    success[flip] = 1 - success[flip]

    frame = pd.DataFrame({
        "time": np.full(
            len(selected), t,
            dtype=np.int16
        ),
        "source": (
            u321[selected] + 1
        ).astype(np.int32),
        "target": (
            v321[selected] + 1
        ).astype(np.int32),
        "success": success
    })

    frames321.append(frame)

    if t <= 9:
        audit321.append({
            "time": t,
            "observations": len(selected),
            "success_rate": float(
                success.mean()
            )
        })

        print(
            f"t={t:02d} | "
            f"observations={len(selected):6,d} | "
            f"success={success.mean():.4f}"
        )
    else:
        print(
            f"t={t:02d} | "
            "TEST GENERATED — METRICS SEALED"
        )

# ------------------------------------------------------------
# 4. Separate permitted time windows
# ------------------------------------------------------------

events321 = pd.concat(
    frames321, ignore_index=True
).sort_values(
    ["time", "source", "target"]
).reset_index(drop=True)

train321 = events321[
    events321["time"] <= 7
].copy()

validation321 = events321[
    events321["time"].between(8, 9)
].copy()

test321 = events321[
    events321["time"] >= 10
].copy()

assert len(events321) == (
    len(train321)
    + len(validation321)
    + len(test321)
)

assert not events321[
    ["time", "source", "target"]
].duplicated().any()

# ------------------------------------------------------------
# 5. Write to a NEW scenario directory
# ------------------------------------------------------------

obs321 = OUT321 / "observed"
hidden321 = (
    OUT321 / "simulation_ground_truth"
)

OUT321.mkdir(parents=True)
obs321.mkdir()
hidden321.mkdir()

topology321.to_csv(
    obs321 /
    "siot_social_topology.csv.gz",
    index=False,
    compression="gzip"
)

train321.to_csv(
    obs321 / "train.csv.gz",
    index=False,
    compression="gzip"
)

validation321.to_csv(
    obs321 / "validation.csv.gz",
    index=False,
    compression="gzip"
)

# Generate but do not inspect test outcomes.
test321.to_csv(
    obs321 / "test.csv.gz",
    index=False,
    compression="gzip"
)

np.savez_compressed(
    hidden321 /
    "hidden_simulator_state.npz",
    base_reliability=base_reliability321,
    owner_effect=owner_effect321,
    owner_codes=owner_codes321,
    edge_effect=edge_effect321,
    shocked_device_ids=shocked_nodes321 + 1
)

def sha256_321(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)
    return digest.hexdigest()

report321 = {
    "experiment": "E3.21",
    "scenario": "B_owner_correlated",
    "simulation_seed": 103,
    "reference_generator": "E3.14 / E3.1",
    "parameters": params321,
    "nodes": N321,
    "owners": O321,
    "undirected_edges": M321,
    "train_events": len(train321),
    "validation_events": len(validation321),
    "permitted_temporal_audit": audit321,
    "train_sha256": sha256_321(
        obs321 / "train.csv.gz"
    ),
    "validation_sha256": sha256_321(
        obs321 / "validation.csv.gz"
    ),
    "test_generated_but_not_evaluated": True,
    "hidden_state_not_used_for_learning": True,
    "model_training_performed": False,
    "storage_change": (
        "No all_interactions file in observed; "
        "separate train, validation, test files."
    )
}

with open(
    OUT321 / "e3_21_report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(report321, f, indent=2)

# Keep sealed data out of the notebook namespace.
del test321, events321, frames321
del success_probability, hidden_link_logit
del hidden_device_logit, success, flip
del base_reliability321, owner_effect321
del device_owner_effect321, edge_effect321
del shocked_nodes321, shock_indicator321

print("\n" + "=" * 72)
print("E3.21 COMPLETED — SEED 103")
print("=" * 72)
print("Training interactions:", len(train321))
print("Validation interactions:", len(validation321))
print("Test: SEALED")
print("Parameter changes other than seed: NONE")
print("Training performed: NO")
print("Previous artifacts overwritten: NO")
print("Output:", OUT321)
print("NEXT: E3.22 — Causal feature snapshots.")


In [ ]:

# ============================================================
# E3.22 — CAUSAL TRUST FEATURES, SCENARIO B / SEED 103
# ============================================================
# Reproduce the E3.15 temporal feature algorithm.
# 12 causal features, snapshots t=0,...,9.
#
# Strictly prequential:
# Snapshot t uses observations from times < t.
#
# No training, no test access, no hidden-state access.
# Existing artifacts are never overwritten.
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT322 = Path(
    "/kaggle/working/phase2_adaptive_siot/"
    "E3_tags_v3/scenarios/B_owner_correlated"
)

SOURCE322 = ROOT322 / "seed_103"
REFERENCE322 = ROOT322 / "seed_102"

OBS322 = SOURCE322 / "observed"
SAVE322 = SOURCE322 / "causal_features"

if SAVE322.exists():
    raise FileExistsError(
        f"Output already exists: {SAVE322}"
    )

N322 = 16216
M322 = 146117
DECAY322 = 0.80
CONF_K322 = 5.0
HORIZON322 = 12

FEATURE_NAMES322 = [
    "edge_history_count",
    "edge_success_rate",
    "edge_decayed_count",
    "edge_decayed_success_rate",
    "edge_evidence_confidence",
    "edge_age_normalized",
    "edge_never_observed",
    "node_u_success_rate",
    "node_v_success_rate",
    "node_u_evidence_confidence",
    "node_v_evidence_confidence",
    "edge_recent_minus_longterm"
]

# ------------------------------------------------------------
# 1. Verify the reference feature specification
# ------------------------------------------------------------

reference_report_path = (
    REFERENCE322 / "causal_features" /
    "e3_15_causal_feature_audit.json"
)

with open(
    reference_report_path,
    encoding="utf-8"
) as f:
    reference322 = json.load(f)

assert reference322["feature_count"] == 12
assert reference322["feature_names"] == FEATURE_NAMES322
assert reference322["temporal_decay"] == DECAY322
assert reference322["evidence_confidence_k"] == CONF_K322
assert reference322["age_normalization_horizon"] == HORIZON322

# ------------------------------------------------------------
# 2. Load the fixed observed graph
# ------------------------------------------------------------

topology322 = pd.read_csv(
    OBS322 / "siot_social_topology.csv.gz"
).sort_values(["u", "v"]).reset_index(drop=True)

reference_topology322 = pd.read_csv(
    REFERENCE322 /
    "causal_features/edge_index_mapping.csv.gz"
).sort_values(["u", "v"]).reset_index(drop=True)

pd.testing.assert_frame_equal(
    topology322,
    reference_topology322,
    check_dtype=True
)

assert len(topology322) == M322

u322 = topology322["u"].to_numpy(
    dtype=np.int64
) - 1

v322 = topology322["v"].to_numpy(
    dtype=np.int64
) - 1

assert np.all(u322 < v322)
assert u322.min() >= 0
assert v322.max() < N322

edge_keys322 = u322 * N322 + v322

assert np.all(np.diff(edge_keys322) > 0)

# ------------------------------------------------------------
# 3. Load ONLY train and validation observations
# ------------------------------------------------------------

train322 = pd.read_csv(
    OBS322 / "train.csv.gz"
)

validation322 = pd.read_csv(
    OBS322 / "validation.csv.gz"
)

assert train322["time"].between(0, 7).all()
assert validation322["time"].between(8, 9).all()

events322 = pd.concat(
    [train322, validation322],
    ignore_index=True
)

assert events322["time"].between(0, 9).all()
assert events322["success"].isin([0, 1]).all()

event_u322 = events322["source"].to_numpy(
    dtype=np.int64
) - 1

event_v322 = events322["target"].to_numpy(
    dtype=np.int64
) - 1

event_t322 = events322["time"].to_numpy(
    dtype=np.int16
)

event_success322 = events322["success"].to_numpy(
    dtype=np.float32
)

assert np.all(event_u322 < event_v322)

event_keys322 = (
    event_u322 * N322 + event_v322
)

event_ids322 = np.searchsorted(
    edge_keys322, event_keys322
)

assert np.all(event_ids322 < M322)
assert np.array_equal(
    edge_keys322[event_ids322],
    event_keys322
)

unique_event_keys322 = (
    event_t322.astype(np.int64) * M322
    + event_ids322
)

assert (
    np.unique(unique_event_keys322).size
    == len(events322)
)

# ------------------------------------------------------------
# 4. Initialize historical accumulators
# ------------------------------------------------------------

edge_success322 = np.zeros(
    M322, dtype=np.float32
)

edge_failure322 = np.zeros(
    M322, dtype=np.float32
)

decay_success322 = np.zeros(
    M322, dtype=np.float32
)

decay_failure322 = np.zeros(
    M322, dtype=np.float32
)

node_success322 = np.zeros(
    N322, dtype=np.float32
)

node_failure322 = np.zeros(
    N322, dtype=np.float32
)

last_seen322 = np.full(
    M322, -1, dtype=np.int16
)

def smoothed_rate322(s, f):
    return (s + 1.0) / (s + f + 2.0)

def evidence_conf322(s, f):
    count = s + f
    return count / (count + CONF_K322)

audit322 = []
cumulative322 = 0

# No writes until the input validations above pass.
SAVE322.mkdir(parents=True)

print("=" * 76)
print("E3.22 — CAUSAL FEATURES, SCENARIO B SEED 103")
print("=" * 76)
print("Nodes:", N322)
print("Edges:", M322)
print("Feature dimensions:", len(FEATURE_NAMES322))
print("Temporal decay:", DECAY322)
print("Reference specification: VERIFIED")
print("Snapshots: t=0,...,9")
print("Test accessed: NO")

# ------------------------------------------------------------
# 5. Construct each snapshot before updating history
# ------------------------------------------------------------

for t in range(10):

    if t > 0:
        decay_success322 *= DECAY322
        decay_failure322 *= DECAY322

    historical_count = (
        edge_success322 + edge_failure322
    )

    historical_rate = smoothed_rate322(
        edge_success322, edge_failure322
    )

    decayed_count = (
        decay_success322 + decay_failure322
    )

    decayed_rate = smoothed_rate322(
        decay_success322, decay_failure322
    )

    edge_confidence = evidence_conf322(
        edge_success322, edge_failure322
    )

    node_rate = smoothed_rate322(
        node_success322, node_failure322
    )

    node_confidence = evidence_conf322(
        node_success322, node_failure322
    )

    never_seen = last_seen322 == -1

    age = np.where(
        never_seen,
        HORIZON322,
        t - last_seen322
    ).astype(np.float32)

    normalized_age = age / HORIZON322

    features = np.column_stack([
        historical_count,
        historical_rate,
        decayed_count,
        decayed_rate,
        edge_confidence,
        normalized_age,
        never_seen.astype(np.float32),
        node_rate[u322],
        node_rate[v322],
        node_confidence[u322],
        node_confidence[v322],
        decayed_rate - historical_rate
    ]).astype(np.float32)

    assert features.shape == (M322, 12)
    assert np.isfinite(features).all()

    # Important temporal leakage checks.
    assert int(historical_count.sum()) == cumulative322
    assert cumulative322 == int(
        np.count_nonzero(event_t322 < t)
    )

    if t == 0:
        assert np.all(features[:, 0] == 0)
        assert np.all(features[:, 1] == 0.5)
        assert np.all(features[:, 6] == 1)

    filename = f"snapshot_t{t:02d}.npz"

    np.savez_compressed(
        SAVE322 / filename,
        features=features,
        time=np.array(t, dtype=np.int16),
        feature_names=np.array(FEATURE_NAMES322)
    )

    audit_record = {
        "time": int(t),
        "history_before_t": int(cumulative322),
        "edges_with_history": int(
            (historical_count > 0).sum()
        ),
        "observations_at_t": int(
            np.count_nonzero(event_t322 == t)
        ),
        "snapshot": filename
    }

    # Update history ONLY AFTER writing snapshot t.
    current = event_t322 == t

    ids = event_ids322[current]

    successes = event_success322[current]
    failures = 1.0 - successes

    np.add.at(
        edge_success322, ids, successes
    )

    np.add.at(
        edge_failure322, ids, failures
    )

    np.add.at(
        decay_success322, ids, successes
    )

    np.add.at(
        decay_failure322, ids, failures
    )

    np.add.at(
        node_success322, u322[ids], successes
    )

    np.add.at(
        node_success322, v322[ids], successes
    )

    np.add.at(
        node_failure322, u322[ids], failures
    )

    np.add.at(
        node_failure322, v322[ids], failures
    )

    last_seen322[ids] = t

    cumulative322 += len(ids)

    audit322.append(audit_record)

    print(
        f"t={t:02d} | "
        f"past={audit_record['history_before_t']:7,d} | "
        f"observed_edges="
        f"{audit_record['edges_with_history']:7,d} | "
        f"current={len(ids):6,d}",
        flush=True
    )

# ------------------------------------------------------------
# 6. Final integrity checks
# ------------------------------------------------------------

assert cumulative322 == len(events322)

with np.load(
    SAVE322 / "snapshot_t09.npz"
) as f:
    final_features322 = f["features"]
    final_time322 = int(f["time"])
    final_names322 = f["feature_names"].tolist()

assert final_time322 == 9
assert final_features322.shape == (M322, 12)
assert final_names322 == FEATURE_NAMES322

expected_t9_history322 = int(
    np.count_nonzero(event_t322 < 9)
)

assert int(
    final_features322[:, 0].sum()
) == expected_t9_history322

topology322.to_csv(
    SAVE322 / "edge_index_mapping.csv.gz",
    index=False,
    compression="gzip"
)

# ------------------------------------------------------------
# 7. Save audit report
# ------------------------------------------------------------

report322 = {
    "experiment": "E3.22",
    "scenario": "B_owner_correlated",
    "simulation_seed": 103,
    "reference_feature_pipeline": "E3.15",
    "feature_count": 12,
    "feature_names": FEATURE_NAMES322,
    "temporal_decay": DECAY322,
    "evidence_confidence_k": CONF_K322,
    "age_normalization_horizon": HORIZON322,
    "snapshots": audit322,
    "train_events": int(len(train322)),
    "validation_events": int(len(validation322)),
    "permitted_events": int(len(events322)),
    "historical_events_at_t9": expected_t9_history322,
    "prequential": True,
    "test_accessed": False,
    "hidden_state_accessed": False,
    "model_training_performed": False,
    "prior_artifacts_overwritten": False
}

REPORT322 = (
    SAVE322 / "e3_22_causal_feature_audit.json"
)

with open(
    REPORT322,
    "w",
    encoding="utf-8"
) as f:
    json.dump(report322, f, indent=2)

print("\n" + "=" * 76)
print("E3.22 COMPLETED — SEED 103 FEATURES")
print("=" * 76)
print("Snapshots saved:", len(audit322))
print("Feature shape:", final_features322.shape)
print("Historical observations at t=0: 0")
print(
    "Historical observations at t=9:",
    expected_t9_history322
)
print("Train + validation observations:", len(events322))
print("Feature specification unchanged: VERIFIED")
print("Test accessed: NO")
print("Hidden state accessed: NO")
print("Model training performed: NO")
print("Previous artifacts modified: NO")
print("Saved:", REPORT322)
print(
    "NEXT: E3.23 — Prepare seed-103 model inputs "
    "with train-only StandardScalers."
)


In [ ]:

# ============================================================
# E3.23 — VERIFIED MODEL INPUTS FOR SCENARIO B SEED 103
# ============================================================
# Reuse the exact E3.16h preprocessing function.
#
# Fit new StandardScalers on t=5,6,7 ONLY.
# Transform t=5,...,9.
# Create CUDA tensors in data103.
#
# No model training.
# No test or hidden-state access.
# No previous artifacts overwritten.
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
import joblib
from sklearn.preprocessing import StandardScaler

# ------------------------------------------------------------
# 0. Preconditions
# ------------------------------------------------------------

required_323 = [
    "build_raw_snapshots_h",
    "data102",
    "type_onehot",
    "edge_scaler102",
    "node_scaler102",
    "U", "V", "K"
]

missing_323 = [
    name for name in required_323
    if name not in globals()
]

if missing_323:
    raise RuntimeError(
        f"Missing notebook objects: {missing_323}"
    )

ROOT_323 = Path(
    "/kaggle/working/phase2_adaptive_siot/"
    "E3_tags_v3/scenarios/B_owner_correlated"
)

S102_323 = ROOT_323 / "seed_102"
S103_323 = ROOT_323 / "seed_103"

SAVE_323 = S103_323 / "e3_23_model_inputs"

if SAVE_323.exists():
    raise FileExistsError(
        f"Output already exists: {SAVE_323}"
    )

TRAIN_323 = [5, 6, 7]
VAL_323 = [8, 9]
TIMES_323 = TRAIN_323 + VAL_323

N_323 = 16216
M_323 = 146117

DEVICE_323 = U.device

assert DEVICE_323.type == "cuda"
assert U.is_cuda and V.is_cuda
assert K == 73058
assert type_onehot.shape == (N_323, 16)

assert (
    S103_323 /
    "causal_features/e3_22_causal_feature_audit.json"
).is_file()

print("=" * 76)
print("E3.23 — VERIFIED SEED 103 MODEL INPUTS")
print("=" * 76)
print("Device:", torch.cuda.get_device_name(0))
print("Scenario seed: 103")
print("Training snapshots:", TRAIN_323)
print("Validation snapshots:", VAL_323)

# ------------------------------------------------------------
# 1. Verify that the original preprocessing function
#    still reproduces the existing seed-102 tensors.
# ------------------------------------------------------------

print("\n[1] VERIFY EXISTING SEED 102 PIPELINE")

raw102_verify_323 = build_raw_snapshots_h(
    S102_323
)

max_edge_error_323 = 0.0
max_node_error_323 = 0.0

for t in TIMES_323:

    expected_edge = edge_scaler102.transform(
        raw102_verify_323[t]["edge"]
    ).astype(np.float32)

    expected_node = node_scaler102.transform(
        raw102_verify_323[t]["node"]
    ).astype(np.float32)

    existing_edge = (
        data102[t]["xe"]
        .detach().cpu().numpy()
    )

    existing_node = (
        data102[t]["xc"]
        .detach().cpu().numpy()
    )

    edge_error = float(
        np.max(np.abs(
            expected_edge - existing_edge
        ))
    )

    node_error = float(
        np.max(np.abs(
            expected_node - existing_node
        ))
    )

    max_edge_error_323 = max(
        max_edge_error_323, edge_error
    )

    max_node_error_323 = max(
        max_node_error_323, node_error
    )

    assert np.allclose(
        expected_edge,
        existing_edge,
        atol=1e-5,
        rtol=1e-5
    )

    assert np.allclose(
        expected_node,
        existing_node,
        atol=1e-5,
        rtol=1e-5
    )

print(
    "Maximum edge reconstruction error:",
    max_edge_error_323
)

print(
    "Maximum node reconstruction error:",
    max_node_error_323
)

print("Seed 102 reconstruction: PASSED")

del raw102_verify_323

# ------------------------------------------------------------
# 2. Build raw seed-103 features using the same function
# ------------------------------------------------------------

print("\n[2] BUILD SEED 103 RAW FEATURES")

raw103_323 = build_raw_snapshots_h(
    S103_323
)

assert set(raw103_323.keys()) == set(TIMES_323)

# Verify target observations match the scenario.
expected_counts_323 = {
    5: 21785,
    6: 22006,
    7: 21884,
    8: 22077,
    9: 22061
}

for t in TIMES_323:

    r = raw103_323[t]

    assert r["node"].shape == (N_323, 4)
    assert r["edge"].shape == (M_323, 17)

    assert r["count"] == expected_counts_323[t]
    assert len(r["target"]) == r["count"]
    assert len(r["y"]) == r["count"]

    assert np.isfinite(r["node"]).all()
    assert np.isfinite(r["edge"]).all()

print("Seed 103 raw feature schema: PASSED")

# ------------------------------------------------------------
# 3. Fit NEW scalers on training snapshots ONLY
# ------------------------------------------------------------

print("\n[3] FIT TRAIN-ONLY SCALERS")

edge_scaler103 = StandardScaler()
node_scaler103 = StandardScaler()

for t in TRAIN_323:

    edge_scaler103.partial_fit(
        raw103_323[t]["edge"]
    )

    node_scaler103.partial_fit(
        raw103_323[t]["node"]
    )

edge_samples_323 = int(
    edge_scaler103.n_samples_seen_
)

node_samples_323 = int(
    node_scaler103.n_samples_seen_
)

assert edge_samples_323 == M_323 * 3
assert node_samples_323 == N_323 * 3

assert edge_scaler103.n_features_in_ == 17
assert node_scaler103.n_features_in_ == 4

print("Edge scaler samples:", edge_samples_323)
print("Node scaler samples:", node_samples_323)
print("Validation used for scaler fitting: NO")

# ------------------------------------------------------------
# 4. Construct seed-103 CUDA tensors
# ------------------------------------------------------------

print("\n[4] CONSTRUCT CUDA TENSORS")

data103 = {}

static103 = type_onehot.astype(
    np.float32, copy=False
)

for t in TIMES_323:

    r = raw103_323[t]

    xe = edge_scaler103.transform(
        r["edge"]
    ).astype(np.float32)

    xc = node_scaler103.transform(
        r["node"]
    ).astype(np.float32)

    xn = np.concatenate(
        [static103, xc],
        axis=1
    ).astype(np.float32)

    assert xe.shape == (M_323, 17)
    assert xc.shape == (N_323, 4)
    assert xn.shape == (N_323, 20)

    data103[t] = {
        "time": t,
        "xn": torch.as_tensor(
            xn, dtype=torch.float32,
            device=DEVICE_323
        ),
        "xe": torch.as_tensor(
            xe, dtype=torch.float32,
            device=DEVICE_323
        ),
        "xc": torch.as_tensor(
            xc, dtype=torch.float32,
            device=DEVICE_323
        ),
        "target": torch.as_tensor(
            r["target"],
            dtype=torch.long,
            device=DEVICE_323
        ),
        "y": torch.as_tensor(
            r["y"],
            dtype=torch.float32,
            device=DEVICE_323
        ),
        "count": int(r["count"])
    }

    batch = data103[t]

    assert torch.equal(
        batch["xn"][:, 16:],
        batch["xc"]
    )

    assert bool(torch.isfinite(batch["xe"]).all())
    assert bool(torch.isfinite(batch["xn"]).all())

    assert int(batch["target"].min()) >= 0
    assert int(batch["target"].max()) < M_323

    assert batch["count"] == expected_counts_323[t]

    # No evaluation, just a dataset audit.
    print(
        f"t={t} | "
        f"observations={batch['count']:6,d} | "
        f"failure_rate={batch['y'].mean().item():.4f} | "
        f"device={batch['xe'].device}",
        flush=True
    )

# ------------------------------------------------------------
# 5. Independent temporal integrity checks
# ------------------------------------------------------------

print("\n[5] TEMPORAL INTEGRITY")

with np.load(
    S103_323 /
    "causal_features/snapshot_t09.npz"
) as archive:

    t9_history_323 = archive[
        "features"
    ][:, 0]

assert int(t9_history_323.sum()) == 197069

train_count_323 = sum(
    data103[t]["count"]
    for t in TRAIN_323
)

validation_count_323 = sum(
    data103[t]["count"]
    for t in VAL_323
)

assert train_count_323 == (
    21785 + 22006 + 21884
)

assert validation_count_323 == 44138

# Verify tensor data are independent of seed 102.
assert data103 is not data102

assert not torch.equal(
    data103[7]["xe"],
    data102[7]["xe"]
)

print("t=9 historical observations: 197069")
print("Training events t=5..7:", train_count_323)
print("Validation events t=8..9:", validation_count_323)
print("Seed 103 differs from Seed 102: VERIFIED")

# ------------------------------------------------------------
# 6. Save new scalers and audit report
# ------------------------------------------------------------

report323 = {
    "experiment": "E3.23",
    "scenario": "B_owner_correlated",
    "simulation_seed": 103,
    "reference_pipeline": "E3.16h",
    "reference_seed102_reconstructed": True,
    "seed102_max_edge_error":
        max_edge_error_323,
    "seed102_max_node_error":
        max_node_error_323,
    "train_times": TRAIN_323,
    "validation_times": VAL_323,
    "scaler_fit_times": TRAIN_323,
    "edge_scaler_samples": edge_samples_323,
    "node_scaler_samples": node_samples_323,
    "edge_feature_count": 17,
    "node_feature_count": 20,
    "controller_feature_count": 4,
    "exact_edge_budget": K,
    "temporal_history_at_t9": 197069,
    "training_events_by_time": {
        str(t): data103[t]["count"]
        for t in TRAIN_323
    },
    "validation_events_by_time": {
        str(t): data103[t]["count"]
        for t in VAL_323
    },
    "gpu_device": str(DEVICE_323),
    "prequential_validation": True,
    "test_accessed": False,
    "hidden_state_accessed": False,
    "model_training_performed": False,
    "previous_artifacts_modified": False
}

SAVE_323.mkdir(parents=True)

joblib.dump(
    edge_scaler103,
    SAVE_323 / "edge_scaler103.joblib"
)

joblib.dump(
    node_scaler103,
    SAVE_323 / "node_scaler103.joblib"
)

with open(
    SAVE_323 / "report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(report323, f, indent=2)

print("\n" + "=" * 76)
print("E3.23 COMPLETED — SEED 103 MODEL INPUTS")
print("=" * 76)

print("Seed 102 reconstruction: PASSED")
print("Seed 103 model tensors: READY")
print("Edge tensor shape:", tuple(data103[7]["xe"].shape))
print("Node tensor shape:", tuple(data103[7]["xn"].shape))
print("Controller tensor shape:", tuple(data103[7]["xc"].shape))
print("GPU device:", data103[7]["xe"].device)
print("Scaler fitting used validation: NO")
print("No test or hidden-state access.")
print("No model training.")
print("Previous artifacts preserved.")
print("Saved:", SAVE_323 / "report.json")


In [ ]:

# ============================================================
# E3.24 — SCENARIO B SEED 103 / MODEL SEED 44
# ============================================================
# GNN-based scorer vs Trust-only scorer
# Fresh, matched model initialization
#
# Train: t=5,6,7
# Validation: t=8,9
# Fixed: 60 epochs, AdamW lr=1e-3, wd=1e-4
# Exact 50% edge budget; GPU coverage selector
# No controller
#
# IMPORTANT:
# Uses data103 explicitly.
# No test or hidden simulator state access.
# Does not overwrite earlier experiments.
# ============================================================

from pathlib import Path
import copy
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    log_loss
)

required324 = [
    "stable", "data103", "U", "V", "K",
    "coverage_topk_gpu"
]

missing324 = [
    name for name in required324
    if name not in globals()
]
assert not missing324, f"Missing: {missing324}"

SCENARIO324 = Path(
    "/kaggle/working/phase2_adaptive_siot/"
    "E3_tags_v3/scenarios/B_owner_correlated/seed_103"
)

SAVE324 = (
    SCENARIO324 / "tags_v3" /
    "e3_24_scorer_comparison_seed_44"
)

if SAVE324.exists():
    raise FileExistsError(
        f"Output already exists: {SAVE324}"
    )

assert (
    SCENARIO324 / "e3_23_model_inputs/report.json"
).is_file()

SEED324 = 44
EPOCHS324 = 60
TRAIN324 = [5, 6, 7]
VAL324 = [8, 9]

DEVICE324 = U.device
N324 = 16216
M324 = 146117

assert DEVICE324.type == "cuda"
assert U.is_cuda and V.is_cuda
assert K == 73058
assert sorted(data103.keys()) == TRAIN324 + VAL324

for t in TRAIN324 + VAL324:
    b = data103[t]
    assert b["xn"].shape == (N324, 20)
    assert b["xe"].shape == (M324, 17)
    assert b["xc"].shape == (N324, 4)
    assert b["xn"].device == DEVICE324
    assert b["xe"].device == DEVICE324
    assert b["target"].device == DEVICE324
    assert b["y"].device == DEVICE324
    assert b["count"] == len(b["target"])

# ------------------------------------------------------------
# 1. Fresh model initialization
# ------------------------------------------------------------

random.seed(SEED324)
np.random.seed(SEED324)
torch.manual_seed(SEED324)
torch.cuda.manual_seed_all(SEED324)

sample324 = data103[5]

base324 = type(stable)(
    node_dim=sample324["xn"].shape[1],
    edge_dim=sample324["xe"].shape[1],
    controller_dim=sample324["xc"].shape[1],
    hidden=32
).to(DEVICE324)

gnn324 = copy.deepcopy(base324)
trust324 = copy.deepcopy(base324)

trust324.scorer = nn.Sequential(
    nn.Linear(17, 64),
    nn.ReLU(),
    nn.Linear(64, 1)
).to(DEVICE324)

shared_keys324 = [
    key for key in gnn324.state_dict()
    if key.startswith((
        "encoder.",
        "trust_head.",
        "graph_head."
    ))
]

assert shared_keys324

shared_equal324 = all(
    torch.equal(
        gnn324.state_dict()[key],
        trust324.state_dict()[key]
    )
    for key in shared_keys324
)

assert shared_equal324

# Check deterministic shared initialization against
# the model-seed-44 initialization from E3.17.
if "base317" in globals():
    matching_seed44 = all(
        torch.equal(
            base324.state_dict()[key],
            base317.state_dict()[key]
        )
        for key in shared_keys324
    )
    assert matching_seed44, (
        "Seed 44 shared initialization differs "
        "from the previous scenario run."
    )
else:
    matching_seed44 = None

def trainable324(net):
    return [
        p for name, p in net.named_parameters()
        if not name.startswith("controller.")
    ]

# ------------------------------------------------------------
# 2. Forward: use supplied batch, never global data
# ------------------------------------------------------------

def forward324(net, batch, use_gnn, training):

    xn = batch["xn"]
    xe = batch["xe"]
    ids = batch["target"]

    if use_gnn:
        full = xn.new_ones(M324)

        h = net.encoder(
            xn, U, V, full
        )

        scoring_input = torch.cat([
            h[U] + h[V],
            torch.abs(h[U] - h[V]),
            xe
        ], dim=1)

        scores = net.scorer(
            scoring_input
        ).squeeze(-1)
    else:
        scores = net.scorer(
            xe
        ).squeeze(-1)

    hard, coverage_count = coverage_topk_gpu(
        scores, K
    )

    if training:
        threshold = torch.topk(
            scores.detach(), K
        ).values[-1]

        soft = torch.sigmoid(
            scores - threshold
        )

        mask = hard + (soft - soft.detach())
    else:
        mask = hard

    z = net.encoder(
        xn, U, V, mask
    )

    a = U[ids]
    b = V[ids]

    pair_features = torch.cat([
        z[a] + z[b],
        torch.abs(z[a] - z[b]),
        z[a] * z[b]
    ], dim=1)

    logits = (
        net.trust_head(
            xe[ids]
        ).squeeze(-1)
        +
        net.graph_head(
            pair_features
        ).squeeze(-1)
    )

    return logits, hard, int(coverage_count)

# ------------------------------------------------------------
# 3. Scenario-isolated validation
# ------------------------------------------------------------

@torch.no_grad()
def validate324(net, use_gnn, dataset):

    net.eval()

    ys, ps, masks = [], [], []
    cover_counts = []

    for t in VAL324:
        batch = dataset[t]

        logits, hard, coverage = forward324(
            net, batch, use_gnn, False
        )

        ys.append(
            batch["y"].detach().cpu().numpy()
        )

        ps.append(
            torch.sigmoid(logits)
            .detach().cpu().numpy()
        )

        masks.append(
            hard.detach().cpu().numpy().astype(bool)
        )

        cover_counts.append(coverage)

    y = np.concatenate(ys)
    p = np.clip(
        np.concatenate(ps),
        1e-7,
        1 - 1e-7
    )

    assert all(
        int(mask.sum()) == K
        for mask in masks
    )

    return {
        "AP": float(
            average_precision_score(y, p)
        ),
        "AUC": float(
            roc_auc_score(y, p)
        ),
        "BCE": float(
            log_loss(y, p, labels=[0, 1])
        ),
        "churn": float(
            np.count_nonzero(
                masks[0] != masks[1]
            ) / (2 * K)
        ),
        "coverage_t8": cover_counts[0],
        "coverage_t9": cover_counts[1]
    }

# ------------------------------------------------------------
# 4. Fixed 60-epoch training
# ------------------------------------------------------------

results324 = []
histories324 = {}
states324 = {}

print("=" * 76)
print("E3.24 — SCENARIO B SEED 103 / MODEL SEED 44")
print("=" * 76)

print("GPU:", torch.cuda.get_device_name(0))
print("Scenario seed:", 103)
print("Model seed:", SEED324)
print("Fresh construction: YES")
print("Shared initialization:", shared_equal324)
print("Matches prior seed-44 initialization:",
      matching_seed44)
print("Train:", TRAIN324)
print("Validation:", VAL324)
print("Epoch budget:", EPOCHS324)
print("Retained edges:", K)
print("Controller: DISABLED")
print("Dataset: data103 ONLY")

for model_name, net, use_gnn in [
    ("GNN_scorer", gnn324, True),
    ("Trust_only_scorer", trust324, False)
]:

    random.seed(SEED324)
    np.random.seed(SEED324)
    torch.manual_seed(SEED324)
    torch.cuda.manual_seed_all(SEED324)

    params = trainable324(net)

    optimizer = torch.optim.AdamW(
        params,
        lr=1e-3,
        weight_decay=1e-4
    )

    rows = []
    best_ap = -np.inf
    best_epoch = None
    best_state = None
    best_metrics = None

    print("\n" + "-" * 76)
    print("MODEL:", model_name)
    print("-" * 76, flush=True)

    for epoch in range(1, EPOCHS324 + 1):

        net.train()
        losses = []

        for t in TRAIN324:

            batch = data103[t]

            optimizer.zero_grad(
                set_to_none=True
            )

            logits, hard, _ = forward324(
                net, batch, use_gnn, True
            )

            loss = F.binary_cross_entropy_with_logits(
                logits, batch["y"]
            )

            if not bool(torch.isfinite(loss)):
                raise RuntimeError(
                    f"Non-finite loss: {model_name}, "
                    f"epoch={epoch}, t={t}"
                )

            loss.backward()

            grad_norm = torch.nn.utils.clip_grad_norm_(
                params, max_norm=1.0
            )

            if not bool(torch.isfinite(grad_norm)):
                raise RuntimeError(
                    f"Non-finite gradient: {model_name}, "
                    f"epoch={epoch}, t={t}"
                )

            optimizer.step()

            losses.append((
                float(loss.item()),
                int(batch["count"])
            ))

        train_bce = sum(
            loss * n for loss, n in losses
        ) / sum(
            n for _, n in losses
        )

        metrics = validate324(
            net, use_gnn, data103
        )

        rows.append({
            "epoch": epoch,
            "train_BCE": train_bce,
            **metrics
        })

        if metrics["AP"] > best_ap + 1e-5:
            best_ap = metrics["AP"]
            best_epoch = epoch
            best_metrics = dict(metrics)
            best_state = copy.deepcopy(
                net.state_dict()
            )

        if epoch == 1 or epoch % 10 == 0:
            print(
                f"epoch={epoch:02d} | "
                f"Train BCE={train_bce:.6f} | "
                f"Val AP={metrics['AP']:.6f} | "
                f"AUC={metrics['AUC']:.6f} | "
                f"BCE={metrics['BCE']:.6f} | "
                f"churn={metrics['churn']:.4f}",
                flush=True
            )

    final = rows[-1]

    results324.append({
        "model": model_name,
        "simulation_seed": 103,
        "model_seed": SEED324,
        "best_epoch": best_epoch,
        "best_AP": best_metrics["AP"],
        "best_AUC": best_metrics["AUC"],
        "best_BCE": best_metrics["BCE"],
        "epoch60_AP": final["AP"],
        "epoch60_AUC": final["AUC"],
        "epoch60_BCE": final["BCE"],
        "epoch60_churn": final["churn"],
        "coverage_t8": final["coverage_t8"],
        "coverage_t9": final["coverage_t9"]
    })

    histories324[model_name] = rows

    states324[model_name] = {
        "best": best_state,
        "epoch60": copy.deepcopy(net.state_dict())
    }

# ------------------------------------------------------------
# 5. Paired comparison
# ------------------------------------------------------------

comparison324 = pd.DataFrame(results324)

gnn_row324 = comparison324[
    comparison324["model"] == "GNN_scorer"
].iloc[0]

trust_row324 = comparison324[
    comparison324["model"] == "Trust_only_scorer"
].iloc[0]

delta324 = {
    metric: float(
        gnn_row324[metric] - trust_row324[metric]
    )
    for metric in [
        "epoch60_AP",
        "epoch60_AUC",
        "epoch60_BCE",
        "epoch60_churn"
    ]
}

# ------------------------------------------------------------
# 6. Save without overwriting
# ------------------------------------------------------------

SAVE324.mkdir(parents=True)

comparison324.to_csv(
    SAVE324 / "comparison.csv",
    index=False
)

for name, rows in histories324.items():

    pd.DataFrame(rows).to_csv(
        SAVE324 / f"{name}_history.csv",
        index=False
    )

    torch.save(
        {
            "best_model_state_dict":
                states324[name]["best"],
            "epoch60_model_state_dict":
                states324[name]["epoch60"],
            "simulation_seed": 103,
            "model_seed": SEED324,
            "model": name,
            "epochs": EPOCHS324
        },
        SAVE324 / f"{name}_checkpoint.pt"
    )

failure_prevalence324 = float(
    (
        data103[8]["y"].sum()
        + data103[9]["y"].sum()
    ).item()
    / (
        data103[8]["count"]
        + data103[9]["count"]
    )
)

report324 = {
    "experiment": "E3.24",
    "scenario": "B_owner_correlated",
    "simulation_seed": 103,
    "model_seed": SEED324,
    "epochs": EPOCHS324,
    "train_times": TRAIN324,
    "validation_times": VAL324,
    "checkpoint_for_main_comparison": "epoch60",
    "validation_failure_prevalence": failure_prevalence324,
    "fresh_initialization": True,
    "matched_shared_initialization": True,
    "gpu_selector": True,
    "controller_enabled": False,
    "exact_edge_budget": K,
    "results": results324,
    "gnn_minus_trust_only": delta324,
    "test_accessed": False,
    "hidden_state_accessed": False,
    "limitations": [
        "Single model seed on simulation seed 103.",
        "Exploratory validation comparison.",
        "Both prediction models contain GraphSAGE.",
        "The scorer input dimensions differ.",
        "No statistical significance established."
    ]
}

with open(
    SAVE324 / "report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(report324, f, indent=2)

print("\n" + "=" * 76)
print("E3.24 COMPLETED — SCENARIO 103 / MODEL SEED 44")
print("=" * 76)

print(
    comparison324.round(6).to_string(index=False)
)

print(
    "\nValidation failure prevalence:",
    f"{failure_prevalence324:.6f}"
)

print("\nGNN MINUS TRUST-ONLY AT EPOCH 60")

for metric, value in delta324.items():
    print(f"{metric}: {value:+.6f}")

print("\nNo previous artifacts overwritten.")
print("No test or hidden-state access.")
print("Saved:", SAVE324 / "report.json")


In [ ]:

# ============================================================
# E3.25 — SCENARIO 103 / FRESH MODEL SEED 45
# ============================================================
# Same protocol as E3.24:
# 60 epochs, AdamW, GPU selector, 50% edge budget.
# GNN scorer vs Trust-only scorer.
# Controller disabled.
# No test / hidden-state access.
# ============================================================

from pathlib import Path
import copy, json, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

required = [
    "stable", "data103", "U", "V", "K",
    "forward324", "validate324", "base324"
]
assert all(k in globals() for k in required)

SEED325 = 45
EPOCHS325 = 60
TRAIN325 = [5, 6, 7]
VAL325 = [8, 9]

SAVE325 = Path(
    "/kaggle/working/phase2_adaptive_siot/"
    "E3_tags_v3/scenarios/B_owner_correlated/"
    "seed_103/tags_v3/e3_25_scorer_comparison_seed_45"
)

if SAVE325.exists():
    raise FileExistsError(SAVE325)

assert U.is_cuda and V.is_cuda
assert K == 73058
assert sorted(data103.keys()) == TRAIN325 + VAL325

# ------------------------------------------------------------
# 1. Fresh initialization, shared starting weights
# ------------------------------------------------------------

random.seed(SEED325)
np.random.seed(SEED325)
torch.manual_seed(SEED325)
torch.cuda.manual_seed_all(SEED325)

sample = data103[5]

base325 = type(stable)(
    node_dim=sample["xn"].shape[1],
    edge_dim=sample["xe"].shape[1],
    controller_dim=sample["xc"].shape[1],
    hidden=32
).to(U.device)

gnn325 = copy.deepcopy(base325)
trust325 = copy.deepcopy(base325)

trust325.scorer = nn.Sequential(
    nn.Linear(17, 64),
    nn.ReLU(),
    nn.Linear(64, 1)
).to(U.device)

shared_keys = [
    k for k in base325.state_dict()
    if k.startswith((
        "encoder.", "trust_head.", "graph_head."
    ))
]

assert shared_keys

shared_equal = all(
    torch.equal(
        gnn325.state_dict()[k],
        trust325.state_dict()[k]
    )
    for k in shared_keys
)

fresh_different = any(
    not torch.equal(
        base325.state_dict()[k],
        base324.state_dict()[k]
    )
    for k in shared_keys
)

assert shared_equal and fresh_different

# Optional consistency check with the previous
# scenario's independently constructed seed-45 model.
if "base318" in globals():
    assert all(
        torch.equal(
            base325.state_dict()[k],
            base318.state_dict()[k]
        )
        for k in shared_keys
    )

def active_params325(net):
    return [
        p for name, p in net.named_parameters()
        if not name.startswith("controller.")
    ]

print("=" * 76)
print("E3.25 — SCENARIO SEED 103 / MODEL SEED 45")
print("=" * 76)
print("GPU:", torch.cuda.get_device_name(0))
print("Fresh initialization:", fresh_different)
print("Matched shared parameters:", shared_equal)
print("Epochs:", EPOCHS325)
print("Exact edge budget:", K)
print("Data source: data103 ONLY")

# ------------------------------------------------------------
# 2. Independent training
# ------------------------------------------------------------

results325 = []
histories325 = {}
checkpoints325 = {}

for name, net, use_gnn in [
    ("GNN_scorer", gnn325, True),
    ("Trust_only_scorer", trust325, False)
]:

    random.seed(SEED325)
    np.random.seed(SEED325)
    torch.manual_seed(SEED325)
    torch.cuda.manual_seed_all(SEED325)

    params = active_params325(net)

    optimizer = torch.optim.AdamW(
        params,
        lr=1e-3,
        weight_decay=1e-4
    )

    rows = []
    best_ap = -float("inf")
    best_epoch = None
    best_state = None

    print(f"\nMODEL: {name}", flush=True)

    for epoch in range(1, EPOCHS325 + 1):
        net.train()
        losses = []

        for t in TRAIN325:
            batch = data103[t]

            optimizer.zero_grad(set_to_none=True)

            logits, hard, _ = forward324(
                net, batch, use_gnn, True
            )

            loss = F.binary_cross_entropy_with_logits(
                logits, batch["y"]
            )

            assert bool(torch.isfinite(loss))
            loss.backward()

            grad_norm = torch.nn.utils.clip_grad_norm_(
                params, max_norm=1.0
            )

            assert bool(torch.isfinite(grad_norm))
            optimizer.step()

            losses.append((
                float(loss.item()), int(batch["count"])
            ))

        train_bce = sum(
            loss * count for loss, count in losses
        ) / sum(count for _, count in losses)

        # Explicit seed-103 validation dataset.
        metrics = validate324(net, use_gnn, data103)

        rows.append({
            "epoch": epoch,
            "train_BCE": train_bce,
            **metrics
        })

        if metrics["AP"] > best_ap + 1e-5:
            best_ap = metrics["AP"]
            best_epoch = epoch
            best_state = copy.deepcopy(
                net.state_dict()
            )

        if epoch == 1 or epoch % 10 == 0:
            print(
                f"epoch={epoch:02d} | "
                f"AP={metrics['AP']:.6f} | "
                f"AUC={metrics['AUC']:.6f} | "
                f"BCE={metrics['BCE']:.6f} | "
                f"churn={metrics['churn']:.4f}",
                flush=True
            )

    final = rows[-1]

    results325.append({
        "model": name,
        "simulation_seed": 103,
        "model_seed": SEED325,
        "best_epoch": best_epoch,
        "best_AP": best_ap,
        "epoch60_AP": final["AP"],
        "epoch60_AUC": final["AUC"],
        "epoch60_BCE": final["BCE"],
        "epoch60_churn": final["churn"],
        "coverage_t8": final["coverage_t8"],
        "coverage_t9": final["coverage_t9"]
    })

    histories325[name] = rows

    checkpoints325[name] = {
        "best": best_state,
        "epoch60": copy.deepcopy(net.state_dict())
    }

# ------------------------------------------------------------
# 3. Save and compare
# ------------------------------------------------------------

comparison325 = pd.DataFrame(results325)

gnn_row = comparison325[
    comparison325.model == "GNN_scorer"
].iloc[0]

trust_row = comparison325[
    comparison325.model == "Trust_only_scorer"
].iloc[0]

deltas325 = {
    metric: float(
        gnn_row[metric] - trust_row[metric]
    )
    for metric in [
        "epoch60_AP",
        "epoch60_AUC",
        "epoch60_BCE",
        "epoch60_churn"
    ]
}

SAVE325.mkdir(parents=True)

comparison325.to_csv(
    SAVE325 / "comparison.csv", index=False
)

for name, rows in histories325.items():
    pd.DataFrame(rows).to_csv(
        SAVE325 / f"{name}_history.csv",
        index=False
    )

    torch.save({
        "best_model_state_dict": checkpoints325[name]["best"],
        "epoch60_model_state_dict": checkpoints325[name]["epoch60"],
        "simulation_seed": 103,
        "model_seed": SEED325,
        "method": name
    }, SAVE325 / f"{name}_checkpoint.pt")

report325 = {
    "experiment": "E3.25",
    "scenario": "B_owner_correlated",
    "simulation_seed": 103,
    "model_seed": SEED325,
    "epochs": EPOCHS325,
    "checkpoint_for_main_comparison": "epoch60",
    "fresh_initialization_verified": True,
    "matched_shared_initialization": True,
    "exact_edge_budget": K,
    "controller_enabled": False,
    "gpu_selector_enabled": True,
    "results": results325,
    "gnn_minus_trust_only": deltas325,
    "test_accessed": False,
    "hidden_state_accessed": False
}

with open(
    SAVE325 / "report.json", "w", encoding="utf-8"
) as f:
    json.dump(report325, f, indent=2)

print("\n" + "=" * 76)
print("E3.25 COMPLETED — SCENARIO 103 / MODEL SEED 45")
print("=" * 76)
print(comparison325.round(6).to_string(index=False))

print("\nGNN MINUS TRUST-ONLY AT EPOCH 60")
for metric, value in deltas325.items():
    print(f"{metric}: {value:+.6f}")

print("\nNo previous artifacts overwritten.")
print("No test or hidden-state access.")
print("Saved:", SAVE325 / "report.json")


In [ ]:

# ============================================================
# E3.26 — SCENARIO 103 / FRESH MODEL SEED 46
# ============================================================
# GNN scorer vs Trust-only scorer.
# Fixed 60 epochs, exact 50% edge budget.
# GPU selector; full node coverage; no controller.
# Uses data103 only. No test or hidden-state access.
# ============================================================

from pathlib import Path
import copy
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

required326 = [
    "stable", "data103", "U", "V", "K",
    "forward324", "validate324", "base325"
]
missing326 = [x for x in required326 if x not in globals()]
assert not missing326, f"Missing objects: {missing326}"

SEED326 = 46
EPOCHS326 = 60
TRAIN326 = [5, 6, 7]
VAL326 = [8, 9]

SAVE326 = Path(
    "/kaggle/working/phase2_adaptive_siot/"
    "E3_tags_v3/scenarios/B_owner_correlated/"
    "seed_103/tags_v3/e3_26_scorer_comparison_seed_46"
)

if SAVE326.exists():
    raise FileExistsError(f"Output exists: {SAVE326}")

assert U.is_cuda and V.is_cuda
assert K == 73058
assert sorted(data103.keys()) == TRAIN326 + VAL326

# ------------------------------------------------------------
# 1. Fresh and matched initialization
# ------------------------------------------------------------

random.seed(SEED326)
np.random.seed(SEED326)
torch.manual_seed(SEED326)
torch.cuda.manual_seed_all(SEED326)

sample326 = data103[5]

base326 = type(stable)(
    node_dim=sample326["xn"].shape[1],
    edge_dim=sample326["xe"].shape[1],
    controller_dim=sample326["xc"].shape[1],
    hidden=32
).to(U.device)

gnn326 = copy.deepcopy(base326)
trust326 = copy.deepcopy(base326)

trust326.scorer = nn.Sequential(
    nn.Linear(17, 64),
    nn.ReLU(),
    nn.Linear(64, 1)
).to(U.device)

shared_keys326 = [
    name for name in base326.state_dict()
    if name.startswith((
        "encoder.", "trust_head.", "graph_head."
    ))
]

assert shared_keys326

shared_equal326 = all(
    torch.equal(
        gnn326.state_dict()[name],
        trust326.state_dict()[name]
    )
    for name in shared_keys326
)

fresh_different326 = any(
    not torch.equal(
        base326.state_dict()[name],
        base325.state_dict()[name]
    )
    for name in shared_keys326
)

assert shared_equal326 and fresh_different326

# Check that model seed 46 reproduces the shared
# initialization used in scenario seed 102.
if "base319" in globals():
    assert all(
        torch.equal(
            base326.state_dict()[name],
            base319.state_dict()[name]
        )
        for name in shared_keys326
    )

def active_params326(net):
    return [
        p for name, p in net.named_parameters()
        if not name.startswith("controller.")
    ]

print("=" * 76)
print("E3.26 — SCENARIO 103 / MODEL SEED 46")
print("=" * 76)
print("GPU:", torch.cuda.get_device_name(0))
print("Scenario seed: 103")
print("Model seed:", SEED326)
print("Fresh initialization:", fresh_different326)
print("Matched shared weights:", shared_equal326)
print("Epochs:", EPOCHS326)
print("Retained edges:", K)
print("Data source: data103 ONLY")

# ------------------------------------------------------------
# 2. Train independently
# ------------------------------------------------------------

results326 = []
histories326 = {}
checkpoints326 = {}

for model_name, net, use_gnn in [
    ("GNN_scorer", gnn326, True),
    ("Trust_only_scorer", trust326, False)
]:

    random.seed(SEED326)
    np.random.seed(SEED326)
    torch.manual_seed(SEED326)
    torch.cuda.manual_seed_all(SEED326)

    params = active_params326(net)

    optimizer = torch.optim.AdamW(
        params, lr=1e-3, weight_decay=1e-4
    )

    rows = []
    best_ap = -np.inf
    best_epoch = None
    best_state = None

    print(f"\nMODEL: {model_name}", flush=True)

    for epoch in range(1, EPOCHS326 + 1):

        net.train()
        train_losses = []

        for t in TRAIN326:

            batch = data103[t]
            optimizer.zero_grad(set_to_none=True)

            logits, hard, _ = forward324(
                net, batch, use_gnn, True
            )

            loss = F.binary_cross_entropy_with_logits(
                logits, batch["y"]
            )

            assert bool(torch.isfinite(loss))
            loss.backward()

            grad_norm = torch.nn.utils.clip_grad_norm_(
                params, max_norm=1.0
            )
            assert bool(torch.isfinite(grad_norm))

            optimizer.step()

            train_losses.append((
                float(loss.item()),
                int(batch["count"])
            ))

        train_bce = (
            sum(value * count for value, count in train_losses)
            / sum(count for _, count in train_losses)
        )

        # Evaluates seed 103 only.
        metrics = validate324(
            net, use_gnn, data103
        )

        rows.append({
            "epoch": epoch,
            "train_BCE": train_bce,
            **metrics
        })

        if metrics["AP"] > best_ap + 1e-5:
            best_ap = metrics["AP"]
            best_epoch = epoch
            best_state = copy.deepcopy(net.state_dict())

        if epoch == 1 or epoch % 10 == 0:
            print(
                f"epoch={epoch:02d} | "
                f"AP={metrics['AP']:.6f} | "
                f"AUC={metrics['AUC']:.6f} | "
                f"BCE={metrics['BCE']:.6f} | "
                f"churn={metrics['churn']:.4f}",
                flush=True
            )

    final = rows[-1]

    results326.append({
        "model": model_name,
        "simulation_seed": 103,
        "model_seed": SEED326,
        "best_epoch": best_epoch,
        "best_AP": best_ap,
        "epoch60_AP": final["AP"],
        "epoch60_AUC": final["AUC"],
        "epoch60_BCE": final["BCE"],
        "epoch60_churn": final["churn"],
        "coverage_t8": final["coverage_t8"],
        "coverage_t9": final["coverage_t9"]
    })

    histories326[model_name] = rows
    checkpoints326[model_name] = {
        "best": best_state,
        "epoch60": copy.deepcopy(net.state_dict())
    }

# ------------------------------------------------------------
# 3. Compare fixed Epoch 60 results
# ------------------------------------------------------------

comparison326 = pd.DataFrame(results326)

gnn_row326 = comparison326[
    comparison326["model"] == "GNN_scorer"
].iloc[0]

trust_row326 = comparison326[
    comparison326["model"] == "Trust_only_scorer"
].iloc[0]

deltas326 = {
    metric: float(
        gnn_row326[metric] - trust_row326[metric]
    )
    for metric in [
        "epoch60_AP",
        "epoch60_AUC",
        "epoch60_BCE",
        "epoch60_churn"
    ]
}

# ------------------------------------------------------------
# 4. Save without overwriting
# ------------------------------------------------------------

SAVE326.mkdir(parents=True)

comparison326.to_csv(
    SAVE326 / "comparison.csv",
    index=False
)

for name, rows in histories326.items():

    pd.DataFrame(rows).to_csv(
        SAVE326 / f"{name}_history.csv",
        index=False
    )

    torch.save({
        "best_model_state_dict":
            checkpoints326[name]["best"],
        "epoch60_model_state_dict":
            checkpoints326[name]["epoch60"],
        "simulation_seed": 103,
        "model_seed": SEED326,
        "method": name,
        "epochs": EPOCHS326
    }, SAVE326 / f"{name}_checkpoint.pt")

report326 = {
    "experiment": "E3.26",
    "scenario": "B_owner_correlated",
    "simulation_seed": 103,
    "model_seed": SEED326,
    "training_epochs": EPOCHS326,
    "checkpoint_for_main_comparison": "epoch60",
    "train_times": TRAIN326,
    "validation_times": VAL326,
    "fresh_initialization_verified": True,
    "matched_shared_initialization": True,
    "controller_enabled": False,
    "exact_edge_budget": K,
    "gpu_selector_enabled": True,
    "results": results326,
    "gnn_minus_trust_only": deltas326,
    "test_accessed": False,
    "hidden_state_accessed": False
}

with open(
    SAVE326 / "report.json",
    "w", encoding="utf-8"
) as f:
    json.dump(report326, f, indent=2)

print("\n" + "=" * 76)
print("E3.26 COMPLETED — SCENARIO 103 / MODEL SEED 46")
print("=" * 76)

print(comparison326.round(6).to_string(index=False))

print("\nGNN MINUS TRUST-ONLY AT EPOCH 60")
for metric, value in deltas326.items():
    print(f"{metric}: {value:+.6f}")

print("\nPrevious artifacts overwritten: NO")
print("No test or hidden-state access.")
print("Saved:", SAVE326 / "report.json")


In [ ]:

# ============================================================
# E3.27 — THREE-SCENARIO NESTED ABLATION SUMMARY
# ============================================================
# Scenario seeds: 101, 102, 103
# Model seeds:    44, 45, 46
# Primary checkpoint: fixed epoch 60
#
# Read-only analysis of existing experimental reports.
# No model training/inference.
# No test or hidden simulator state access.
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

BASE327 = Path(
    "/kaggle/working/phase2_adaptive_siot/"
    "E3_tags_v3/scenarios/B_owner_correlated"
)

SOURCES327 = {
    (101, 44): "e3_10_fresh_seed_44",
    (101, 45): "e3_11_fresh_seed_45",
    (101, 46): "e3_12_fresh_seed_46",
    (102, 44): "e3_17_scorer_comparison_seed_44",
    (102, 45): "e3_18_scorer_comparison_seed_45",
    (102, 46): "e3_19_scorer_comparison_seed_46",
    (103, 44): "e3_24_scorer_comparison_seed_44",
    (103, 45): "e3_25_scorer_comparison_seed_45",
    (103, 46): "e3_26_scorer_comparison_seed_46"
}

OUT327 = BASE327 / "e3_27_three_scenario_aggregation"

if OUT327.exists():
    raise FileExistsError(
        f"Output already exists: {OUT327}"
    )

METRICS327 = ["AP", "AUC", "BCE", "churn"]

# ------------------------------------------------------------
# 1. Obtain validation failure prevalence
# ------------------------------------------------------------

prevalence327 = {}

for scenario_seed in [101, 102, 103]:

    path = (
        BASE327 / f"seed_{scenario_seed}"
        / "observed/validation.csv.gz"
    )

    val = pd.read_csv(path)

    assert set(val["time"].unique()) == {8, 9}
    assert val["success"].isin([0, 1]).all()

    prevalence327[scenario_seed] = float(
        (1.0 - val["success"]).mean()
    )

# ------------------------------------------------------------
# 2. Read and validate all nine paired reports
# ------------------------------------------------------------

records327 = []

for (scenario_seed, model_seed), folder in SOURCES327.items():

    path = (
        BASE327 / f"seed_{scenario_seed}"
        / "tags_v3" / folder / "report.json"
    )

    assert path.is_file(), f"Missing report: {path}"

    with open(path, encoding="utf-8") as f:
        report = json.load(f)

    assert report["test_accessed"] is False
    assert report["hidden_state_accessed"] is False

    assert int(report["simulation_seed"]) == scenario_seed
    assert int(report["model_seed"]) == model_seed

    results = {
        row["model"]: row
        for row in report["results"]
    }

    assert set(results.keys()) == {
        "GNN_scorer", "Trust_only_scorer"
    }

    gnn = results["GNN_scorer"]
    trust = results["Trust_only_scorer"]

    record = {
        "scenario_seed": scenario_seed,
        "model_seed": model_seed,
        "failure_prevalence": prevalence327[scenario_seed]
    }

    for metric in METRICS327:
        key = f"epoch60_{metric}"

        g = float(gnn[key])
        tr = float(trust[key])

        assert np.isfinite(g)
        assert np.isfinite(tr)

        record[f"GNN_{metric}"] = g
        record[f"Trust_{metric}"] = tr
        record[f"delta_{metric}"] = g - tr

    records327.append(record)

paired327 = pd.DataFrame(records327).sort_values(
    ["scenario_seed", "model_seed"]
).reset_index(drop=True)

assert len(paired327) == 9
assert paired327["scenario_seed"].nunique() == 3

assert all(
    set(group["model_seed"]) == {44, 45, 46}
    for _, group in paired327.groupby("scenario_seed")
)

# ------------------------------------------------------------
# 3. Aggregate paired differences WITHIN scenarios
# ------------------------------------------------------------

scenario_rows327 = []

for scenario_seed, group in paired327.groupby(
    "scenario_seed", sort=True
):

    assert len(group) == 3

    row = {
        "scenario_seed": int(scenario_seed),
        "n_model_seeds": 3,
        "failure_prevalence":
            prevalence327[int(scenario_seed)]
    }

    for metric in METRICS327:

        differences = group[
            f"delta_{metric}"
        ].to_numpy(np.float64)

        row[f"GNN_mean_{metric}"] = float(
            group[f"GNN_{metric}"].mean()
        )

        row[f"Trust_mean_{metric}"] = float(
            group[f"Trust_{metric}"].mean()
        )

        row[f"delta_mean_{metric}"] = float(
            differences.mean()
        )

        row[f"delta_sd_{metric}"] = float(
            differences.std(ddof=1)
        )

    scenario_rows327.append(row)

scenarios327 = pd.DataFrame(scenario_rows327)

# ------------------------------------------------------------
# 4. Equal-weight descriptive scenario aggregation
# ------------------------------------------------------------

overall327 = {
    "scenario_count": 3,
    "model_seeds_per_scenario": 3,
    "paired_runs": 9,
    "epoch": 60
}

for metric in METRICS327:

    scenario_differences = scenarios327[
        f"delta_mean_{metric}"
    ].to_numpy(np.float64)

    overall327[f"delta_mean_{metric}"] = float(
        scenario_differences.mean()
    )

    overall327[f"scenario_delta_sd_{metric}"] = float(
        scenario_differences.std(ddof=1)
    )

overall327["GNN_lower_churn_all_nine"] = bool(
    (paired327["delta_churn"] < 0).all()
)

overall327["GNN_higher_AP_runs"] = int(
    (paired327["delta_AP"] > 0).sum()
)

overall327["Trust_higher_AP_runs"] = int(
    (paired327["delta_AP"] < 0).sum()
)

# ------------------------------------------------------------
# 5. Store the descriptive analysis
# ------------------------------------------------------------

report327 = {
    "experiment": "E3.27",
    "scenario": "B_owner_correlated",
    "scenario_seeds": [101, 102, 103],
    "model_seeds": [44, 45, 46],
    "fixed_checkpoint": "epoch60",
    "comparison": "GNN scorer minus Trust-only scorer",
    "scenario_summary": scenario_rows327,
    "overall": overall327,
    "interpretation": (
        "Descriptive nested comparison. "
        "Three simulator realizations are insufficient "
        "to establish statistical superiority or "
        "equivalence. Model seeds are nested within "
        "scenario seeds."
    ),
    "test_accessed": False,
    "hidden_state_accessed": False,
    "limitations": [
        "Validation was used during architecture development.",
        "Both models retain GraphSAGE for prediction.",
        "Scorer architectures have different input sizes.",
        "Reduced mask churn does not establish adaptation quality.",
        "Only three independent simulator realizations.",
        "No equivalence or statistical significance claim."
    ]
}

OUT327.mkdir(parents=True)

paired327.to_csv(
    OUT327 / "paired_runs.csv",
    index=False
)

scenarios327.to_csv(
    OUT327 / "scenario_summary.csv",
    index=False
)

with open(
    OUT327 / "report.json",
    "w", encoding="utf-8"
) as f:
    json.dump(report327, f, indent=2)

# ------------------------------------------------------------
# 6. Report
# ------------------------------------------------------------

print("=" * 76)
print("E3.27 — THREE-SCENARIO NESTED ABLATION")
print("=" * 76)

print("\nNINE PAIRED RUNS")
print(
    paired327[[
        "scenario_seed",
        "model_seed",
        "delta_AP",
        "delta_AUC",
        "delta_BCE",
        "delta_churn"
    ]].round(6).to_string(index=False)
)

print("\nSCENARIO-LEVEL SUMMARY")
print(
    scenarios327[[
        "scenario_seed",
        "failure_prevalence",
        "GNN_mean_AP",
        "Trust_mean_AP",
        "delta_mean_AP",
        "delta_sd_AP",
        "delta_mean_churn"
    ]].round(6).to_string(index=False)
)

print("\nOVERALL — DESCRIPTIVE ONLY")
for key, value in overall327.items():
    print(f"{key}: {value}")

print("\nE3.27 COMPLETED")
print("No training or model inference.")
print("No test or hidden-state access.")
print("No previous artifacts overwritten.")
print("Saved:", OUT327 / "report.json")


In [ ]:

# ============================================================
# E3.27a — FIXED THREE-SCENARIO NESTED ABLATION
# ============================================================
# Fixes KeyError: 'simulation_seed'
# Supports older E3.10–E3.12 reports.
#
# No training, no inference, no test access.
# No existing artifact overwritten.
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

BASE327A = Path(
    "/kaggle/working/phase2_adaptive_siot/"
    "E3_tags_v3/scenarios/B_owner_correlated"
)

SOURCES327A = {
    (101, 44): "e3_10_fresh_seed_44",
    (101, 45): "e3_11_fresh_seed_45",
    (101, 46): "e3_12_fresh_seed_46",
    (102, 44): "e3_17_scorer_comparison_seed_44",
    (102, 45): "e3_18_scorer_comparison_seed_45",
    (102, 46): "e3_19_scorer_comparison_seed_46",
    (103, 44): "e3_24_scorer_comparison_seed_44",
    (103, 45): "e3_25_scorer_comparison_seed_45",
    (103, 46): "e3_26_scorer_comparison_seed_46"
}

OUT327A = BASE327A / "e3_27a_three_scenario_aggregation"

if OUT327A.exists():
    raise FileExistsError(
        f"Output already exists: {OUT327A}"
    )

METRICS327A = ["AP", "AUC", "BCE", "churn"]

print("=" * 76)
print("E3.27a — CORRECTED THREE-SCENARIO AGGREGATION")
print("=" * 76)

# ------------------------------------------------------------
# 1. Validation prevalence
# ------------------------------------------------------------

prevalence327A = {}

for scenario_seed in (101, 102, 103):

    path = (
        BASE327A / f"seed_{scenario_seed}"
        / "observed/validation.csv.gz"
    )

    val = pd.read_csv(path)

    assert set(val["time"].unique()) == {8, 9}
    assert val["success"].isin([0, 1]).all()

    prevalence327A[scenario_seed] = float(
        (1 - val["success"]).mean()
    )

# ------------------------------------------------------------
# 2. Compatible result extraction
# ------------------------------------------------------------

def extract_models327A(report):

    # New format: list under "results".
    results = report.get("results")

    if isinstance(results, list):
        mapped = {}

        for entry in results:
            name = entry.get(
                "model", entry.get("method")
            )
            if name is not None:
                mapped[str(name)] = entry

        if (
            "GNN_scorer" in mapped
            and "Trust_only_scorer" in mapped
        ):
            return (
                mapped["GNN_scorer"],
                mapped["Trust_only_scorer"]
            )

    # Alternative format: dict under "results".
    if isinstance(results, dict):
        if (
            "GNN_scorer" in results
            and "Trust_only_scorer" in results
        ):
            return (
                results["GNN_scorer"],
                results["Trust_only_scorer"]
            )

    raise KeyError(
        "Unrecognized results structure. "
        f"Top-level keys: {sorted(report.keys())}"
    )


def metric327A(row, metric):

    candidates = [
        f"epoch60_{metric}",
        f"final_{metric}",
        metric
    ]

    for key in candidates:
        if key in row:
            return float(row[key])

    raise KeyError(
        f"Missing Epoch-60 {metric}. "
        f"Available keys: {sorted(row.keys())}"
    )

# ------------------------------------------------------------
# 3. Read paired experimental records
# ------------------------------------------------------------

records327A = []

for (scenario_seed, model_seed), folder in SOURCES327A.items():

    path = (
        BASE327A / f"seed_{scenario_seed}"
        / "tags_v3" / folder / "report.json"
    )

    assert path.is_file(), f"Missing report: {path}"

    with open(path, encoding="utf-8") as f:
        report = json.load(f)

    # Verify explicit declarations if present.
    if "simulation_seed" in report:
        assert int(report["simulation_seed"]) == scenario_seed

    if "model_seed" in report:
        assert int(report["model_seed"]) == model_seed

    assert report.get("test_accessed", False) is False
    assert report.get("hidden_state_accessed", False) is False

    gnn, trust = extract_models327A(report)

    record = {
        "scenario_seed": scenario_seed,
        "model_seed": model_seed,
        "failure_prevalence":
            prevalence327A[scenario_seed]
    }

    for metric in METRICS327A:

        g = metric327A(gnn, metric)
        tr = metric327A(trust, metric)

        assert np.isfinite(g)
        assert np.isfinite(tr)

        record[f"GNN_{metric}"] = g
        record[f"Trust_{metric}"] = tr
        record[f"delta_{metric}"] = g - tr

    records327A.append(record)

    print(
        f"Loaded scenario={scenario_seed}, "
        f"model_seed={model_seed} | "
        f"delta_AP={record['delta_AP']:+.6f}"
    )

paired327A = pd.DataFrame(
    records327A
).sort_values(
    ["scenario_seed", "model_seed"]
).reset_index(drop=True)

assert len(paired327A) == 9

# ------------------------------------------------------------
# 4. Scenario-level nested aggregation
# ------------------------------------------------------------

scenario_rows327A = []

for scenario_seed, group in paired327A.groupby(
    "scenario_seed", sort=True
):

    assert len(group) == 3

    row = {
        "scenario_seed": int(scenario_seed),
        "n_model_seeds": 3,
        "failure_prevalence":
            prevalence327A[int(scenario_seed)]
    }

    for metric in METRICS327A:

        differences = group[
            f"delta_{metric}"
        ].to_numpy(np.float64)

        row[f"GNN_mean_{metric}"] = float(
            group[f"GNN_{metric}"].mean()
        )

        row[f"Trust_mean_{metric}"] = float(
            group[f"Trust_{metric}"].mean()
        )

        row[f"delta_mean_{metric}"] = float(
            differences.mean()
        )

        row[f"delta_sd_{metric}"] = float(
            differences.std(ddof=1)
        )

    scenario_rows327A.append(row)

scenarios327A = pd.DataFrame(
    scenario_rows327A
)

# ------------------------------------------------------------
# 5. Descriptive cross-scenario aggregation
# ------------------------------------------------------------

overall327A = {
    "scenario_count": 3,
    "model_seeds_per_scenario": 3,
    "paired_runs": 9,
    "checkpoint": "epoch60"
}

for metric in METRICS327A:

    values = scenarios327A[
        f"delta_mean_{metric}"
    ].to_numpy(np.float64)

    overall327A[f"delta_mean_{metric}"] = float(
        values.mean()
    )

    overall327A[
        f"scenario_delta_sd_{metric}"
    ] = float(values.std(ddof=1))

overall327A["GNN_lower_churn_all_nine"] = bool(
    (paired327A["delta_churn"] < 0).all()
)

overall327A["GNN_higher_AP_runs"] = int(
    (paired327A["delta_AP"] > 0).sum()
)

overall327A["Trust_higher_AP_runs"] = int(
    (paired327A["delta_AP"] < 0).sum()
)

# ------------------------------------------------------------
# 6. Save NEW artifacts only
# ------------------------------------------------------------

OUT327A.mkdir(parents=True)

paired327A.to_csv(
    OUT327A / "paired_runs.csv",
    index=False
)

scenarios327A.to_csv(
    OUT327A / "scenario_summary.csv",
    index=False
)

report327A = {
    "experiment": "E3.27a",
    "scenario": "B_owner_correlated",
    "scenario_seeds": [101, 102, 103],
    "model_seeds": [44, 45, 46],
    "checkpoint": "epoch60",
    "scenario_summary": scenario_rows327A,
    "overall": overall327A,
    "test_accessed": False,
    "hidden_state_accessed": False,
    "limitations": [
        "Only three simulator realizations.",
        "Model seeds nested within scenario seeds.",
        "Validation used for architecture development.",
        "Both prediction models retain GraphSAGE.",
        "Mask churn is not adaptation quality.",
        "No statistical significance or equivalence claim."
    ]
}

with open(
    OUT327A / "report.json",
    "w", encoding="utf-8"
) as f:
    json.dump(report327A, f, indent=2)

# ------------------------------------------------------------
# 7. Display
# ------------------------------------------------------------

print("\n" + "=" * 76)
print("NINE PAIRED RUNS — EPOCH 60")
print("=" * 76)

print(
    paired327A[[
        "scenario_seed",
        "model_seed",
        "delta_AP",
        "delta_AUC",
        "delta_BCE",
        "delta_churn"
    ]].round(6).to_string(index=False)
)

print("\nSCENARIO-LEVEL SUMMARY")

print(
    scenarios327A[[
        "scenario_seed",
        "failure_prevalence",
        "GNN_mean_AP",
        "Trust_mean_AP",
        "delta_mean_AP",
        "delta_sd_AP",
        "delta_mean_churn"
    ]].round(6).to_string(index=False)
)

print("\nOVERALL — DESCRIPTIVE ONLY")

for key, value in overall327A.items():
    print(f"{key}: {value}")

print("\nE3.27a COMPLETED")
print("No training or model inference.")
print("No test or hidden-state access.")
print("No previous artifacts overwritten.")
print("Saved:", OUT327A / "report.json")


In [ ]:

# ============================================================
# E3.28 — SCENARIO B / SIMULATION SEED 104
# ============================================================
# Same simulation procedure as E3.21.
# Change only the independent simulation seed.
#
# Train: t=0..7
# Validation: t=8..9
# Test: t=10..11, GENERATED AND SEALED
#
# No model training.
# No prior artifacts overwritten.
# ============================================================

from pathlib import Path
import json
import zipfile
import hashlib
import numpy as np
import pandas as pd

ROOT328 = Path(
    "/kaggle/working/phase2_adaptive_siot"
)
BASE328 = (
    ROOT328 /
    "E3_tags_v3/scenarios/B_owner_correlated"
)

REF328 = BASE328 / "seed_103"
OUT328 = BASE328 / "seed_104"

if OUT328.exists():
    raise FileExistsError(
        f"Existing output: {OUT328}"
    )

SIM_SEED328 = 104
N328 = 16216
M328 = 146117
T328 = 12

# Frozen Scenario B parameters
P328 = {
    "base_reliability": "Beta(5,2)",
    "owner_effect_std": 0.65,
    "edge_effect_std": 0.45,
    "temporal_rho": 0.85,
    "temporal_std": 0.12,
    "interaction_rate": 0.15,
    "observation_flip": 0.03,
    "shock_start": 6,
    "shock_fraction": 0.08,
    "shock_logit": -1.6,
    "shock_decay": 0.90
}

# ------------------------------------------------------------
# 1. Verify parameters against E3.21
# ------------------------------------------------------------

with open(
    REF328 / "e3_21_report.json",
    encoding="utf-8"
) as f:
    ref_report328 = json.load(f)

assert ref_report328["simulation_seed"] == 103
assert ref_report328["parameters"] == P328

topology328 = pd.read_csv(
    REF328 /
    "observed/siot_social_topology.csv.gz"
).sort_values(
    ["u", "v"]
).reset_index(drop=True)

assert len(topology328) == M328

u328 = (
    topology328["u"].to_numpy(np.int32) - 1
)
v328 = (
    topology328["v"].to_numpy(np.int32) - 1
)

assert np.all(u328 < v328)
assert u328.min() >= 0
assert v328.max() < N328

archive328 = (
    ROOT328 /
    "E1_trust_scoring/santander/raw/"
    "objects_description.zip"
)

with zipfile.ZipFile(archive328) as z:
    with z.open("objects_description.csv") as f:
        devices328 = pd.read_csv(f)

devices328 = devices328.sort_values(
    "id_device"
).reset_index(drop=True)

assert np.array_equal(
    devices328["id_device"].to_numpy(),
    np.arange(1, N328 + 1)
)

owner_codes328, owner_ids328 = pd.factorize(
    devices328["id_user"],
    sort=True
)

O328 = len(owner_ids328)
assert O328 == 4001

assert int(
    (owner_codes328[u328] ==
     owner_codes328[v328]).sum()
) == 65667

# ------------------------------------------------------------
# 2. Latent simulator initialization
# ------------------------------------------------------------

rng328 = np.random.default_rng(SIM_SEED328)

def sigmoid328(x):
    return 1.0 / (
        1.0 + np.exp(-np.clip(x, -30, 30))
    )

def logit328(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

base_reliability328 = rng328.beta(
    5.0, 2.0, size=N328
)

owner_effect328 = rng328.normal(
    0.0, P328["owner_effect_std"], size=O328
)

device_owner_effect328 = (
    owner_effect328[owner_codes328]
)

edge_effect328 = rng328.normal(
    0.0, P328["edge_effect_std"], size=M328
)

base_logit328 = logit328(
    base_reliability328
)

shock_count328 = int(round(
    P328["shock_fraction"] * N328
))

shocked_nodes328 = rng328.choice(
    N328,
    size=shock_count328,
    replace=False
)

shock_indicator328 = np.zeros(
    N328, dtype=np.float32
)

shock_indicator328[shocked_nodes328] = 1.0

temporal_state328 = np.zeros(
    N328, dtype=np.float32
)

# ------------------------------------------------------------
# 3. Generate events in frozen RNG order
# ------------------------------------------------------------

frames328 = []
audit328 = []

print("=" * 72)
print("E3.28 — SCENARIO B / SIMULATION SEED 104")
print("=" * 72)

print("Nodes:", N328)
print("Owners:", O328)
print("Undirected edges:", M328)
print("Parameters match Seed 103:", True)

for t in range(T328):

    temporal_state328 = (
        P328["temporal_rho"] * temporal_state328
        + rng328.normal(
            0.0,
            P328["temporal_std"],
            size=N328
        )
    ).astype(np.float32)

    if t >= P328["shock_start"]:
        shock_strength328 = (
            P328["shock_logit"]
            * P328["shock_decay"] **
            (t - P328["shock_start"])
        )
    else:
        shock_strength328 = 0.0

    device_logits328 = (
        base_logit328
        + device_owner_effect328
        + temporal_state328
        + shock_strength328 * shock_indicator328
    )

    edge_logits328 = (
        0.5 * (
            device_logits328[u328]
            + device_logits328[v328]
        )
        + edge_effect328
    )

    probs328 = sigmoid328(edge_logits328)

    selected328 = np.flatnonzero(
        rng328.random(M328)
        < P328["interaction_rate"]
    )

    success328 = (
        rng328.random(len(selected328))
        < probs328[selected328]
    ).astype(np.int8)

    flip328 = (
        rng328.random(len(selected328))
        < P328["observation_flip"]
    )

    success328[flip328] = (
        1 - success328[flip328]
    )

    frames328.append(
        pd.DataFrame({
            "time": np.full(
                len(selected328), t,
                dtype=np.int16
            ),
            "source": (
                u328[selected328] + 1
            ).astype(np.int32),
            "target": (
                v328[selected328] + 1
            ).astype(np.int32),
            "success": success328
        })
    )

    if t <= 9:
        audit328.append({
            "time": int(t),
            "observations": int(len(selected328)),
            "success_rate": float(
                success328.mean()
            )
        })

        print(
            f"t={t:02d} | "
            f"observations={len(selected328):,} | "
            f"success={success328.mean():.4f}"
        )
    else:
        print(
            f"t={t:02d} | "
            "TEST GENERATED — METRICS SEALED"
        )

# ------------------------------------------------------------
# 4. Separate temporal partitions
# ------------------------------------------------------------

events328 = pd.concat(
    frames328, ignore_index=True
).sort_values(
    ["time", "source", "target"]
).reset_index(drop=True)

train328 = events328[
    events328["time"] <= 7
].copy()

validation328 = events328[
    events328["time"].between(8, 9)
].copy()

# Do not calculate test metrics.
test328 = events328[
    events328["time"] >= 10
].copy()

assert len(events328) == (
    len(train328)
    + len(validation328)
    + len(test328)
)

assert not events328[
    ["time", "source", "target"]
].duplicated().any()

assert train328["time"].between(0, 7).all()
assert validation328["time"].between(8, 9).all()

# ------------------------------------------------------------
# 5. Persist separate data partitions
# ------------------------------------------------------------

OBS328 = OUT328 / "observed"
HIDDEN328 = OUT328 / "simulation_ground_truth"

OBS328.mkdir(parents=True)
HIDDEN328.mkdir()

topology328.to_csv(
    OBS328 / "siot_social_topology.csv.gz",
    compression="gzip",
    index=False
)

train328.to_csv(
    OBS328 / "train.csv.gz",
    compression="gzip",
    index=False
)

validation328.to_csv(
    OBS328 / "validation.csv.gz",
    compression="gzip",
    index=False
)

# Sealed partition: write only.
test328.to_csv(
    OBS328 / "test.csv.gz",
    compression="gzip",
    index=False
)

# Simulation-only ground truth, excluded from learning.
np.savez_compressed(
    HIDDEN328 / "hidden_simulator_state.npz",
    base_reliability=base_reliability328,
    owner_effect=owner_effect328,
    owner_codes=owner_codes328,
    edge_effect=edge_effect328,
    shocked_device_ids=shocked_nodes328 + 1
)

def sha256_328(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)
    return digest.hexdigest()

report328 = {
    "experiment": "E3.28",
    "scenario": "B_owner_correlated",
    "simulation_seed": SIM_SEED328,
    "reference_experiment": "E3.21",
    "parameters": P328,
    "nodes": N328,
    "owners": O328,
    "undirected_edges": M328,
    "train_events": int(len(train328)),
    "validation_events": int(len(validation328)),
    "permitted_temporal_audit": audit328,
    "train_sha256": sha256_328(
        OBS328 / "train.csv.gz"
    ),
    "validation_sha256": sha256_328(
        OBS328 / "validation.csv.gz"
    ),
    "test_generated_but_not_evaluated": True,
    "hidden_state_not_used_for_learning": True,
    "model_training_performed": False,
    "prior_artifacts_overwritten": False
}

REPORT328 = OUT328 / "e3_28_report.json"

with open(
    REPORT328, "w", encoding="utf-8"
) as f:
    json.dump(report328, f, indent=2)

# Remove sensitive simulation and sealed-test
# variables from notebook memory.
del test328, events328, frames328
del probs328, edge_logits328, device_logits328
del success328, flip328, selected328
del base_reliability328, owner_effect328
del device_owner_effect328, edge_effect328
del shocked_nodes328, shock_indicator328
del temporal_state328, base_logit328

print("\n" + "=" * 72)
print("E3.28 COMPLETED — SEED 104")
print("=" * 72)
print("Train interactions:", len(train328))
print("Validation interactions:", len(validation328))
print("Test: SEALED")
print("Parameters changed: SEED ONLY")
print("Training performed: NO")
print("Previous artifacts overwritten: NO")
print("Saved:", REPORT328)
print("NEXT: E3.29 — causal feature snapshots")


In [ ]:

# ============================================================
# E3.29 — CAUSAL TRUST FEATURES / SCENARIO B SEED 104
# ============================================================
# Reproduce E3.22 exactly with simulation seed 104.
#
# 12 causal features, snapshots t=0,...,9.
# Snapshot at t uses observations strictly before t.
#
# Read only:
#   observed/siot_social_topology.csv.gz
#   observed/train.csv.gz
#   observed/validation.csv.gz
#
# Do NOT access test or hidden simulator state.
# No model training, no overwriting previous artifacts.
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

BASE329 = Path(
    "/kaggle/working/phase2_adaptive_siot/"
    "E3_tags_v3/scenarios/B_owner_correlated"
)

SOURCE329 = BASE329 / "seed_104"
REFERENCE329 = BASE329 / "seed_103"

OBS329 = SOURCE329 / "observed"
SAVE329 = SOURCE329 / "causal_features"

if SAVE329.exists():
    raise FileExistsError(
        f"Output already exists: {SAVE329}"
    )

N329 = 16216
M329 = 146117
DECAY329 = 0.80
CONF_K329 = 5.0
HORIZON329 = 12

FEATURE_NAMES329 = [
    "edge_history_count",
    "edge_success_rate",
    "edge_decayed_count",
    "edge_decayed_success_rate",
    "edge_evidence_confidence",
    "edge_age_normalized",
    "edge_never_observed",
    "node_u_success_rate",
    "node_v_success_rate",
    "node_u_evidence_confidence",
    "node_v_evidence_confidence",
    "edge_recent_minus_longterm"
]

# ------------------------------------------------------------
# 1. Verify reference specification and scenario report
# ------------------------------------------------------------

with open(
    REFERENCE329 /
    "causal_features/e3_22_causal_feature_audit.json",
    encoding="utf-8"
) as f:
    ref329 = json.load(f)

assert ref329["feature_count"] == 12
assert ref329["feature_names"] == FEATURE_NAMES329
assert ref329["temporal_decay"] == DECAY329
assert ref329["evidence_confidence_k"] == CONF_K329
assert ref329["age_normalization_horizon"] == HORIZON329

with open(
    SOURCE329 / "e3_28_report.json",
    encoding="utf-8"
) as f:
    sim_report329 = json.load(f)

assert sim_report329["simulation_seed"] == 104
assert sim_report329["undirected_edges"] == M329
assert sim_report329["nodes"] == N329

# ------------------------------------------------------------
# 2. Fixed canonical topology
# ------------------------------------------------------------

topology329 = pd.read_csv(
    OBS329 / "siot_social_topology.csv.gz"
).sort_values(
    ["u", "v"]
).reset_index(drop=True)

reference_mapping329 = pd.read_csv(
    REFERENCE329 /
    "causal_features/edge_index_mapping.csv.gz"
).sort_values(
    ["u", "v"]
).reset_index(drop=True)

pd.testing.assert_frame_equal(
    topology329,
    reference_mapping329,
    check_dtype=True
)

assert len(topology329) == M329

u329 = topology329["u"].to_numpy(
    dtype=np.int64
) - 1

v329 = topology329["v"].to_numpy(
    dtype=np.int64
) - 1

assert np.all(u329 < v329)
assert u329.min() >= 0
assert v329.max() < N329

edge_keys329 = u329 * N329 + v329

assert np.all(np.diff(edge_keys329) > 0)

# ------------------------------------------------------------
# 3. Read permitted observations only
# ------------------------------------------------------------

train329 = pd.read_csv(
    OBS329 / "train.csv.gz"
)

validation329 = pd.read_csv(
    OBS329 / "validation.csv.gz"
)

assert train329["time"].between(0, 7).all()
assert validation329["time"].between(8, 9).all()

assert len(train329) == sim_report329["train_events"]
assert (
    len(validation329)
    == sim_report329["validation_events"]
)

events329 = pd.concat(
    [train329, validation329],
    ignore_index=True
)

assert events329["time"].between(0, 9).all()
assert events329["success"].isin([0, 1]).all()

event_u329 = (
    events329["source"]
    .to_numpy(np.int64) - 1
)

event_v329 = (
    events329["target"]
    .to_numpy(np.int64) - 1
)

event_t329 = events329[
    "time"
].to_numpy(np.int16)

event_success329 = events329[
    "success"
].to_numpy(np.float32)

assert np.all(event_u329 < event_v329)

event_keys329 = (
    event_u329 * N329 + event_v329
)

event_ids329 = np.searchsorted(
    edge_keys329,
    event_keys329
)

assert np.all(event_ids329 < M329)
assert np.array_equal(
    edge_keys329[event_ids329],
    event_keys329
)

# At most one interaction per edge per time step.
unique_keys329 = (
    event_t329.astype(np.int64) * M329
    + event_ids329
)

assert (
    np.unique(unique_keys329).size
    == len(events329)
)

# ------------------------------------------------------------
# 4. Initialize historical accumulators
# ------------------------------------------------------------

edge_success329 = np.zeros(
    M329, dtype=np.float32
)

edge_failure329 = np.zeros(
    M329, dtype=np.float32
)

decay_success329 = np.zeros(
    M329, dtype=np.float32
)

decay_failure329 = np.zeros(
    M329, dtype=np.float32
)

node_success329 = np.zeros(
    N329, dtype=np.float32
)

node_failure329 = np.zeros(
    N329, dtype=np.float32
)

last_seen329 = np.full(
    M329, -1, dtype=np.int16
)

def smoothed_rate329(s, f):
    return (s + 1.0) / (s + f + 2.0)

def evidence_conf329(s, f):
    count = s + f
    return count / (count + CONF_K329)

audit329 = []
cumulative329 = 0

print("=" * 76)
print("E3.29 — CAUSAL FEATURES / SEED 104")
print("=" * 76)

print("Nodes:", N329)
print("Edges:", M329)
print("Features:", len(FEATURE_NAMES329))
print("Temporal decay:", DECAY329)
print("Reference E3.22 specification: VERIFIED")
print("Train observations:", len(train329))
print("Validation observations:", len(validation329))
print("Test accessed: NO")

# ------------------------------------------------------------
# 5. Construct snapshots using strictly prior history
# ------------------------------------------------------------

# All checks above pass before creating the output folder.
SAVE329.mkdir(parents=True)

for t in range(10):

    # Same update order as the verified E3.22 code.
    if t > 0:
        decay_success329 *= DECAY329
        decay_failure329 *= DECAY329

    history_count329 = (
        edge_success329 + edge_failure329
    )

    history_rate329 = smoothed_rate329(
        edge_success329,
        edge_failure329
    )

    decayed_count329 = (
        decay_success329 + decay_failure329
    )

    decayed_rate329 = smoothed_rate329(
        decay_success329,
        decay_failure329
    )

    edge_conf329 = evidence_conf329(
        edge_success329,
        edge_failure329
    )

    node_rate329 = smoothed_rate329(
        node_success329,
        node_failure329
    )

    node_conf329 = evidence_conf329(
        node_success329,
        node_failure329
    )

    never_seen329 = (
        last_seen329 == -1
    )

    age329 = np.where(
        never_seen329,
        HORIZON329,
        t - last_seen329
    ).astype(np.float32)

    features329 = np.column_stack([
        history_count329,
        history_rate329,
        decayed_count329,
        decayed_rate329,
        edge_conf329,
        age329 / HORIZON329,
        never_seen329.astype(np.float32),
        node_rate329[u329],
        node_rate329[v329],
        node_conf329[u329],
        node_conf329[v329],
        decayed_rate329 - history_rate329
    ]).astype(np.float32)

    assert features329.shape == (M329, 12)
    assert np.isfinite(features329).all()

    # Strict prequential history checks.
    expected_history329 = int(
        np.count_nonzero(event_t329 < t)
    )

    assert cumulative329 == expected_history329

    assert int(
        history_count329.sum()
    ) == expected_history329

    if t == 0:
        assert np.all(features329[:, 0] == 0)
        assert np.all(features329[:, 1] == 0.5)
        assert np.all(features329[:, 6] == 1)

    filename329 = (
        f"snapshot_t{t:02d}.npz"
    )

    np.savez_compressed(
        SAVE329 / filename329,
        features=features329,
        time=np.array(
            t, dtype=np.int16
        ),
        feature_names=np.array(
            FEATURE_NAMES329
        )
    )

    current329 = (
        event_t329 == t
    )

    ids329 = event_ids329[
        current329
    ]

    successes329 = event_success329[
        current329
    ]

    failures329 = (
        1.0 - successes329
    )

    audit329.append({
        "time": int(t),
        "history_before_t": int(cumulative329),
        "edges_with_history": int(
            (history_count329 > 0).sum()
        ),
        "observations_at_t": int(
            len(ids329)
        ),
        "snapshot": filename329
    })

    # Update accumulators AFTER saving snapshot t.
    np.add.at(
        edge_success329,
        ids329,
        successes329
    )

    np.add.at(
        edge_failure329,
        ids329,
        failures329
    )

    np.add.at(
        decay_success329,
        ids329,
        successes329
    )

    np.add.at(
        decay_failure329,
        ids329,
        failures329
    )

    np.add.at(
        node_success329,
        u329[ids329],
        successes329
    )

    np.add.at(
        node_success329,
        v329[ids329],
        successes329
    )

    np.add.at(
        node_failure329,
        u329[ids329],
        failures329
    )

    np.add.at(
        node_failure329,
        v329[ids329],
        failures329
    )

    last_seen329[ids329] = t

    cumulative329 += len(ids329)

    print(
        f"t={t:02d} | "
        f"past={expected_history329:7,d} | "
        f"observed_edges="
        f"{audit329[-1]['edges_with_history']:7,d} | "
        f"current={len(ids329):6,d}",
        flush=True
    )

# ------------------------------------------------------------
# 6. Final integrity checks
# ------------------------------------------------------------

assert cumulative329 == len(events329)
assert len(events329) == 218911

with np.load(
    SAVE329 / "snapshot_t09.npz"
) as archive329:

    final_features329 = archive329[
        "features"
    ]

    assert int(
        archive329["time"]
    ) == 9

    assert (
        archive329["feature_names"].tolist()
        == FEATURE_NAMES329
    )

assert final_features329.shape == (M329, 12)

expected_t9_history329 = int(
    np.count_nonzero(event_t329 < 9)
)

assert int(
    final_features329[:, 0].sum()
) == expected_t9_history329

topology329.to_csv(
    SAVE329 / "edge_index_mapping.csv.gz",
    index=False,
    compression="gzip"
)

# ------------------------------------------------------------
# 7. Save audit report
# ------------------------------------------------------------

report329 = {
    "experiment": "E3.29",
    "scenario": "B_owner_correlated",
    "simulation_seed": 104,
    "reference_feature_pipeline": "E3.22",
    "feature_count": 12,
    "feature_names": FEATURE_NAMES329,
    "temporal_decay": DECAY329,
    "evidence_confidence_k": CONF_K329,
    "age_normalization_horizon": HORIZON329,
    "snapshots": audit329,
    "train_events": int(len(train329)),
    "validation_events": int(len(validation329)),
    "permitted_events": int(len(events329)),
    "historical_events_at_t9":
        expected_t9_history329,
    "prequential": True,
    "test_accessed": False,
    "hidden_state_accessed": False,
    "model_training_performed": False,
    "previous_artifacts_modified": False
}

REPORT329 = (
    SAVE329 / "e3_29_causal_feature_audit.json"
)

with open(
    REPORT329,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        report329, f, indent=2
    )

print("\n" + "=" * 76)
print("E3.29 COMPLETED — SEED 104 FEATURES")
print("=" * 76)

print("Snapshots saved:", len(audit329))
print("Feature shape:", final_features329.shape)
print("Historical observations at t=0: 0")
print(
    "Historical observations at t=9:",
    expected_t9_history329
)
print(
    "Train + validation observations:",
    len(events329)
)
print("Feature specification unchanged: VERIFIED")
print("Test accessed: NO")
print("Hidden state accessed: NO")
print("Model training performed: NO")
print("Previous artifacts modified: NO")
print("Saved:", REPORT329)

print(
    "NEXT: E3.30 — Prepare seed-104 model inputs "
    "with train-only StandardScalers."
)


In [ ]:

# ============================================================
# E3.30 — PREPARE SEED 104 MODEL INPUTS
# ============================================================
# Reuse verified preprocessing from E3.16h.
# Train-only scaling; CUDA tensors.
# No training, no test access, no overwriting.
# ============================================================

from pathlib import Path
import json
import numpy as np
import torch
import joblib
from sklearn.preprocessing import StandardScaler

required = [
    "build_raw_snapshots_h",
    "type_onehot",
    "data103",
    "edge_scaler103",
    "node_scaler103",
    "U", "V", "K"
]

missing = [x for x in required if x not in globals()]
assert not missing, f"Missing objects: {missing}"

BASE = Path(
    "/kaggle/working/phase2_adaptive_siot/"
    "E3_tags_v3/scenarios/B_owner_correlated"
)

SOURCE = BASE / "seed_104"
OUT = SOURCE / "e3_30_model_inputs"

if OUT.exists():
    raise FileExistsError(OUT)

TRAIN = [5, 6, 7]
VAL = [8, 9]
TIMES = TRAIN + VAL
N, M = 16216, 146117

assert U.is_cuda and V.is_cuda
assert K == 73058

assert (
    SOURCE /
    "causal_features/e3_29_causal_feature_audit.json"
).is_file()

print("=" * 70)
print("E3.30 — SEED 104 MODEL INPUT PREPARATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Verify preprocessing using existing Seed 103
# ------------------------------------------------------------

print("\n[1] VERIFY SEED 103 RECONSTRUCTION")

raw103_check = build_raw_snapshots_h(BASE / "seed_103")

edge_error = 0.0
node_error = 0.0

for t in TIMES:

    xe = edge_scaler103.transform(
        raw103_check[t]["edge"]
    ).astype(np.float32)

    xc = node_scaler103.transform(
        raw103_check[t]["node"]
    ).astype(np.float32)

    old_xe = data103[t]["xe"].cpu().numpy()
    old_xc = data103[t]["xc"].cpu().numpy()

    edge_error = max(
        edge_error,
        float(np.max(np.abs(xe - old_xe)))
    )

    node_error = max(
        node_error,
        float(np.max(np.abs(xc - old_xc)))
    )

    assert np.allclose(xe, old_xe, atol=1e-5)
    assert np.allclose(xc, old_xc, atol=1e-5)

print("Max edge error:", edge_error)
print("Max node error:", node_error)
print("Seed 103 reconstruction: PASSED")

del raw103_check

# ------------------------------------------------------------
# 2. Build Seed 104 raw features
# ------------------------------------------------------------

print("\n[2] BUILD SEED 104 RAW FEATURES")

raw104 = build_raw_snapshots_h(SOURCE)

assert set(raw104) == set(TIMES)

expected_counts = {
    t: int(raw104[t]["count"])
    for t in TIMES
}

assert sum(
    expected_counts[t] for t in VAL
) == 43832

for t in TIMES:
    assert raw104[t]["edge"].shape == (M, 17)
    assert raw104[t]["node"].shape == (N, 4)
    assert len(raw104[t]["target"]) == raw104[t]["count"]
    assert len(raw104[t]["y"]) == raw104[t]["count"]

print("Raw input dimensions: VERIFIED")

# ------------------------------------------------------------
# 3. Train-only scalers
# ------------------------------------------------------------

print("\n[3] FIT TRAIN-ONLY SCALERS")

edge_scaler104 = StandardScaler()
node_scaler104 = StandardScaler()

for t in TRAIN:
    edge_scaler104.partial_fit(raw104[t]["edge"])
    node_scaler104.partial_fit(raw104[t]["node"])

assert int(edge_scaler104.n_samples_seen_) == M * 3
assert int(node_scaler104.n_samples_seen_) == N * 3

print(
    "Edge scaler samples:",
    int(edge_scaler104.n_samples_seen_)
)
print(
    "Node scaler samples:",
    int(node_scaler104.n_samples_seen_)
)
print("Validation used for fitting: NO")

# ------------------------------------------------------------
# 4. Construct CUDA tensors
# ------------------------------------------------------------

print("\n[4] CONSTRUCT data104")

data104 = {}

for t in TIMES:

    r = raw104[t]

    xe = edge_scaler104.transform(
        r["edge"]
    ).astype(np.float32)

    xc = node_scaler104.transform(
        r["node"]
    ).astype(np.float32)

    xn = np.concatenate([
        type_onehot.astype(np.float32),
        xc
    ], axis=1)

    data104[t] = {
        "time": t,
        "xn": torch.as_tensor(
            xn, dtype=torch.float32,
            device=U.device
        ),
        "xe": torch.as_tensor(
            xe, dtype=torch.float32,
            device=U.device
        ),
        "xc": torch.as_tensor(
            xc, dtype=torch.float32,
            device=U.device
        ),
        "target": torch.as_tensor(
            r["target"], dtype=torch.long,
            device=U.device
        ),
        "y": torch.as_tensor(
            r["y"], dtype=torch.float32,
            device=U.device
        ),
        "count": int(r["count"])
    }

    assert data104[t]["xn"].shape == (N, 20)
    assert data104[t]["xe"].shape == (M, 17)
    assert data104[t]["xc"].shape == (N, 4)

    print(
        f"t={t} | "
        f"events={r['count']:,} | "
        f"failure={data104[t]['y'].mean().item():.4f}",
        flush=True
    )

# ------------------------------------------------------------
# 5. Save report and scalers
# ------------------------------------------------------------

OUT.mkdir(parents=True)

joblib.dump(
    edge_scaler104,
    OUT / "edge_scaler104.joblib"
)
joblib.dump(
    node_scaler104,
    OUT / "node_scaler104.joblib"
)

report = {
    "experiment": "E3.30",
    "simulation_seed": 104,
    "train_times": TRAIN,
    "validation_times": VAL,
    "edge_dim": 17,
    "node_dim": 20,
    "edge_budget": K,
    "reference_reconstruction_edge_error": edge_error,
    "reference_reconstruction_node_error": node_error,
    "test_accessed": False,
    "hidden_state_accessed": False,
    "training_performed": False
}

with open(OUT / "report.json", "w") as f:
    json.dump(report, f, indent=2)

print("\n" + "=" * 70)
print("E3.30 COMPLETED")
print("=" * 70)
print("data104 READY ON", U.device)
print("Validation events:", sum(data104[t]["count"] for t in VAL))
print("Train-only normalization: VERIFIED")
print("Test accessed: NO")
print("Saved:", OUT / "report.json")


In [ ]:

# ============================================================
# E3.31 — SCENARIO B / SEED 104 / MODEL SEEDS 44,45,46
# ============================================================
# GNN scorer vs Trust-only scorer
# Fixed protocol: 60 epochs, AdamW, 50% edge budget
# No controller; GPU coverage selector
# Validation only t=8,9; TEST remains sealed.
#
# Reuses verified E3.24 forward and validation.
# Saves each model-seed experiment separately.
# ============================================================

from pathlib import Path
import copy
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

required331 = [
    "stable", "data104", "U", "V", "K",
    "forward324", "validate324",
    "coverage_topk_gpu"
]

missing331 = [
    x for x in required331 if x not in globals()
]
if missing331:
    raise RuntimeError(f"Missing objects: {missing331}")

ROOT331 = Path(
    "/kaggle/working/phase2_adaptive_siot/"
    "E3_tags_v3/scenarios/B_owner_correlated/"
    "seed_104/tags_v3"
)

SEEDS331 = [44, 45, 46]
EPOCHS331 = 60
TRAIN331 = [5, 6, 7]
VALID331 = [8, 9]

assert U.is_cuda and V.is_cuda
assert K == 73058
assert sorted(data104.keys()) == TRAIN331 + VALID331

assert Path(
    "/kaggle/working/phase2_adaptive_siot/"
    "E3_tags_v3/scenarios/B_owner_correlated/"
    "seed_104/e3_30_model_inputs/report.json"
).is_file()

# Stop before training if any destination already exists.
destinations331 = {
    seed: ROOT331 / f"e3_31_model_seed_{seed}"
    for seed in SEEDS331
}

existing331 = [
    str(path)
    for path in destinations331.values()
    if path.exists()
]

if existing331:
    raise FileExistsError(
        "Refusing to overwrite prior experiment(s): "
        + str(existing331)
    )

def reset_rng331(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def active_parameters331(model):
    return [
        p for name, p in model.named_parameters()
        if not name.startswith("controller.")
    ]

def train_one331(net, use_gnn, seed, model_name):

    reset_rng331(seed)

    params = active_parameters331(net)

    optimizer = torch.optim.AdamW(
        params, lr=1e-3, weight_decay=1e-4
    )

    history = []
    best_ap = -float("inf")
    best_epoch = None
    best_state = None

    print(
        f"\nMODEL SEED {seed} | {model_name}",
        flush=True
    )

    for epoch in range(1, EPOCHS331 + 1):

        net.train()
        losses = []

        for t in TRAIN331:
            batch = data104[t]

            optimizer.zero_grad(set_to_none=True)

            logits, hard, coverage = forward324(
                net, batch, use_gnn, True
            )

            assert int(hard.sum().item()) == K

            loss = F.binary_cross_entropy_with_logits(
                logits, batch["y"]
            )

            if not bool(torch.isfinite(loss)):
                raise RuntimeError(
                    f"Nonfinite loss: {model_name}, "
                    f"seed={seed}, epoch={epoch}, t={t}"
                )

            loss.backward()

            norm = torch.nn.utils.clip_grad_norm_(
                params, max_norm=1.0
            )

            if not bool(torch.isfinite(norm)):
                raise RuntimeError("Nonfinite gradient")

            optimizer.step()

            losses.append((
                float(loss.item()),
                int(batch["count"])
            ))

        train_bce = (
            sum(loss * count for loss, count in losses)
            / sum(count for _, count in losses)
        )

        # Explicitly use Scenario Seed 104.
        val = validate324(net, use_gnn, data104)

        record = {
            "epoch": epoch,
            "train_BCE": train_bce,
            **val
        }
        history.append(record)

        if val["AP"] > best_ap + 1e-5:
            best_ap = val["AP"]
            best_epoch = epoch
            best_state = copy.deepcopy(
                net.state_dict()
            )

        if epoch == 1 or epoch % 10 == 0:
            print(
                f"epoch={epoch:02d} | "
                f"AP={val['AP']:.6f} | "
                f"AUC={val['AUC']:.6f} | "
                f"BCE={val['BCE']:.6f} | "
                f"churn={val['churn']:.4f}",
                flush=True
            )

    final = history[-1]

    result = {
        "model": model_name,
        "simulation_seed": 104,
        "model_seed": seed,
        "best_epoch": best_epoch,
        "best_AP": float(best_ap),
        "epoch60_AP": final["AP"],
        "epoch60_AUC": final["AUC"],
        "epoch60_BCE": final["BCE"],
        "epoch60_churn": final["churn"],
        "coverage_t8": final["coverage_t8"],
        "coverage_t9": final["coverage_t9"]
    }

    checkpoint = {
        "best_model_state_dict": best_state,
        "epoch60_model_state_dict":
            copy.deepcopy(net.state_dict()),
        "simulation_seed": 104,
        "model_seed": seed,
        "method": model_name,
        "epochs": EPOCHS331
    }

    return result, history, checkpoint

print("=" * 76)
print("E3.31 — THREE MATCHED SCORER COMPARISONS")
print("=" * 76)
print("Scenario seed: 104")
print("Model seeds:", SEEDS331)
print("Device:", torch.cuda.get_device_name(0))
print("Training:", TRAIN331)
print("Validation:", VALID331)
print("Epochs:", EPOCHS331)
print("Exact edge budget:", K)
print("Controller: DISABLED")
print("Test accessed: NO")

all_results331 = []
all_deltas331 = []

for seed in SEEDS331:

    # Fresh and matched shared initialization.
    reset_rng331(seed)

    b = data104[5]

    base = type(stable)(
        node_dim=b["xn"].shape[1],
        edge_dim=b["xe"].shape[1],
        controller_dim=b["xc"].shape[1],
        hidden=32
    ).to(U.device)

    gnn = copy.deepcopy(base)
    trust = copy.deepcopy(base)

    trust.scorer = nn.Sequential(
        nn.Linear(17, 64),
        nn.ReLU(),
        nn.Linear(64, 1)
    ).to(U.device)

    keys = [
        name for name in base.state_dict()
        if name.startswith((
            "encoder.",
            "trust_head.",
            "graph_head."
        ))
    ]

    assert keys
    assert all(
        torch.equal(
            gnn.state_dict()[name],
            trust.state_dict()[name]
        )
        for name in keys
    )

    experiment_results = []
    experiment_history = {}
    experiment_checkpoints = {}

    for model_name, model, use_gnn in [
        ("GNN_scorer", gnn, True),
        ("Trust_only_scorer", trust, False)
    ]:

        result, history, checkpoint = train_one331(
            model, use_gnn, seed, model_name
        )

        experiment_results.append(result)
        experiment_history[model_name] = history
        experiment_checkpoints[model_name] = checkpoint

        del result, history, checkpoint

    # Fixed-epoch paired comparison.
    gnn_result = experiment_results[0]
    trust_result = experiment_results[1]

    delta = {
        metric: (
            gnn_result[f"epoch60_{metric}"]
            - trust_result[f"epoch60_{metric}"]
        )
        for metric in ["AP", "AUC", "BCE", "churn"]
    }

    # Save this seed immediately, before starting the next.
    directory = destinations331[seed]
    directory.mkdir(parents=True, exist_ok=False)

    pd.DataFrame(experiment_results).to_csv(
        directory / "comparison.csv",
        index=False
    )

    for name, history in experiment_history.items():
        pd.DataFrame(history).to_csv(
            directory / f"{name}_history.csv",
            index=False
        )

        torch.save(
            experiment_checkpoints[name],
            directory / f"{name}_checkpoint.pt"
        )

    report = {
        "experiment": "E3.31",
        "scenario": "B_owner_correlated",
        "simulation_seed": 104,
        "model_seed": seed,
        "training_epochs": EPOCHS331,
        "checkpoint_for_main_comparison": "epoch60",
        "train_times": TRAIN331,
        "validation_times": VALID331,
        "matched_shared_initialization": True,
        "fresh_initialization": True,
        "controller_enabled": False,
        "exact_edge_budget": K,
        "results": experiment_results,
        "gnn_minus_trust_only": delta,
        "test_accessed": False,
        "hidden_state_accessed": False
    }

    with open(
        directory / "report.json",
        "w", encoding="utf-8"
    ) as f:
        json.dump(report, f, indent=2)

    all_results331.extend(experiment_results)
    all_deltas331.append({
        "scenario_seed": 104,
        "model_seed": seed,
        **{f"delta_{k}": v for k, v in delta.items()}
    })

    print("\n" + "=" * 60)
    print(f"SEED {seed} — COMPLETED AND SAVED")
    print("=" * 60)

    for metric, value in delta.items():
        print(f"delta_{metric}: {value:+.6f}")

    print("Saved:", directory / "report.json")

    # Release GPU memory from completed models.
    del base, gnn, trust
    del experiment_checkpoints
    torch.cuda.empty_cache()

print("\n" + "=" * 76)
print("E3.31 COMPLETED — SCENARIO SEED 104")
print("=" * 76)

comparison331 = pd.DataFrame(all_results331)
deltas331 = pd.DataFrame(all_deltas331)

print("\nSIX TRAINED MODELS")
print(
    comparison331[[
        "model_seed",
        "model",
        "epoch60_AP",
        "epoch60_AUC",
        "epoch60_BCE",
        "epoch60_churn"
    ]].round(6).to_string(index=False)
)

print("\nTHREE PAIRED DIFFERENCES")
print(
    deltas331.round(6).to_string(index=False)
)

print("\nMEAN DIFFERENCES")
for metric in ["AP", "AUC", "BCE", "churn"]:
    print(
        f"mean_delta_{metric}: "
        f"{deltas331[f'delta_{metric}'].mean():+.6f}"
    )

print("\nTest accessed: NO")
print("Hidden simulator state accessed: NO")
print("Previous artifacts overwritten: NO")
print("All three model seeds saved independently.")


In [ ]:

# ============================================================
# E3.32 — NO-GNN PREDICTION BASELINES
# ============================================================
# Scenario B: seed 104
# Model seeds: 44, 45, 46
# Models: Logistic Regression and Edge-MLP
#
# Input: same 17-dimensional causal edge features.
# Train: t=5,6,7 / Validation: t=8,9
# Fixed 60 epochs, AdamW, no GNN.
#
# No test access, no hidden-state access.
# No existing artifacts overwritten.
# ============================================================

from pathlib import Path
import copy
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    log_loss
)

assert "data104" in globals()
assert sorted(data104.keys()) == [5, 6, 7, 8, 9]

OUT332 = Path(
    "/kaggle/working/phase2_adaptive_siot/"
    "E3_tags_v3/scenarios/B_owner_correlated/"
    "seed_104/baselines/e3_32_no_gnn_predictors"
)

if OUT332.exists():
    raise FileExistsError(
        f"Output already exists: {OUT332}"
    )

SEEDS332 = [44, 45, 46]
TRAIN332 = [5, 6, 7]
VAL332 = [8, 9]
EPOCHS332 = 60

DEVICE332 = data104[5]["xe"].device
assert DEVICE332.type == "cuda"

def reset332(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def build332(model_type):
    if model_type == "LogisticRegression":
        return nn.Linear(17, 1)

    if model_type == "EdgeMLP":
        return nn.Sequential(
            nn.Linear(17, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    raise ValueError(model_type)

@torch.no_grad()
def evaluate332(model):
    model.eval()

    ys = []
    ps = []

    for t in VAL332:
        b = data104[t]

        x = b["xe"][b["target"]]

        logits = model(x).squeeze(-1)

        ys.append(
            b["y"].detach().cpu().numpy()
        )
        ps.append(
            torch.sigmoid(logits)
            .detach().cpu().numpy()
        )

    y = np.concatenate(ys)
    p = np.clip(
        np.concatenate(ps), 1e-7, 1 - 1e-7
    )

    return {
        "AP": float(
            average_precision_score(y, p)
        ),
        "AUC": float(
            roc_auc_score(y, p)
        ),
        "BCE": float(
            log_loss(y, p, labels=[0, 1])
        )
    }

results332 = []
histories332 = {}
states332 = {}

print("=" * 75)
print("E3.32 — NO-GNN PREDICTOR BASELINES")
print("=" * 75)

print("Scenario seed: 104")
print("Model seeds:", SEEDS332)
print("Models: LogisticRegression, EdgeMLP")
print("Epochs:", EPOCHS332)
print("Device:", DEVICE332)
print("Test accessed: NO")

for seed in SEEDS332:

    for model_type in [
        "LogisticRegression",
        "EdgeMLP"
    ]:

        reset332(seed)

        model = build332(
            model_type
        ).to(DEVICE332)

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=1e-3,
            weight_decay=1e-4
        )

        history = []
        best_ap = -np.inf
        best_epoch = None
        best_state = None

        print(
            f"\nseed={seed} | model={model_type}",
            flush=True
        )

        for epoch in range(1, EPOCHS332 + 1):

            model.train()
            train_losses = []

            for t in TRAIN332:
                b = data104[t]

                x = b["xe"][b["target"]]
                y = b["y"]

                optimizer.zero_grad(
                    set_to_none=True
                )

                logits = model(
                    x
                ).squeeze(-1)

                loss = (
                    F.binary_cross_entropy_with_logits(
                        logits, y
                    )
                )

                assert bool(torch.isfinite(loss))

                loss.backward()

                norm = torch.nn.utils.clip_grad_norm_(
                    model.parameters(), 1.0
                )

                assert bool(torch.isfinite(norm))
                optimizer.step()

                train_losses.append((
                    float(loss.item()),
                    int(b["count"])
                ))

            train_bce = (
                sum(v * n for v, n in train_losses)
                / sum(n for _, n in train_losses)
            )

            metrics = evaluate332(model)

            history.append({
                "epoch": epoch,
                "train_BCE": train_bce,
                **metrics
            })

            if metrics["AP"] > best_ap + 1e-5:
                best_ap = metrics["AP"]
                best_epoch = epoch
                best_state = copy.deepcopy(
                    model.state_dict()
                )

            if epoch in [1, 10, 20, 30, 40, 50, 60]:
                print(
                    f"epoch={epoch:02d} | "
                    f"AP={metrics['AP']:.6f} | "
                    f"AUC={metrics['AUC']:.6f} | "
                    f"BCE={metrics['BCE']:.6f}",
                    flush=True
                )

        final = history[-1]

        results332.append({
            "simulation_seed": 104,
            "model_seed": seed,
            "model": model_type,
            "best_epoch": best_epoch,
            "best_AP": best_ap,
            "epoch60_AP": final["AP"],
            "epoch60_AUC": final["AUC"],
            "epoch60_BCE": final["BCE"]
        })

        key = f"{model_type}_seed_{seed}"

        histories332[key] = history
        states332[key] = {
            "best": best_state,
            "epoch60": copy.deepcopy(
                model.state_dict()
            )
        }

        del model
        torch.cuda.empty_cache()

# ------------------------------------------------------------
# Compare against the existing E3.31 graph models
# ------------------------------------------------------------

ROOT332 = Path(
    "/kaggle/working/phase2_adaptive_siot/"
    "E3_tags_v3/scenarios/B_owner_correlated/"
    "seed_104/tags_v3"
)

graph_rows332 = []

for seed in SEEDS332:

    report_path = (
        ROOT332 /
        f"e3_31_model_seed_{seed}/report.json"
    )

    with open(
        report_path,
        encoding="utf-8"
    ) as f:
        report = json.load(f)

    assert report["test_accessed"] is False
    assert int(report["simulation_seed"]) == 104

    for row in report["results"]:
        graph_rows332.append({
            "simulation_seed": 104,
            "model_seed": seed,
            "model": row["model"],
            "epoch60_AP": float(
                row["epoch60_AP"]
            ),
            "epoch60_AUC": float(
                row["epoch60_AUC"]
            ),
            "epoch60_BCE": float(
                row["epoch60_BCE"]
            )
        })

summary332 = pd.concat([
    pd.DataFrame(graph_rows332),
    pd.DataFrame(results332)[[
        "simulation_seed",
        "model_seed",
        "model",
        "epoch60_AP",
        "epoch60_AUC",
        "epoch60_BCE"
    ]]
], ignore_index=True)

means332 = (
    summary332
    .groupby("model")[
        ["epoch60_AP", "epoch60_AUC", "epoch60_BCE"]
    ]
    .agg(["mean", "std"])
)

# ------------------------------------------------------------
# Save complete outputs
# ------------------------------------------------------------

OUT332.mkdir(parents=True)

pd.DataFrame(results332).to_csv(
    OUT332 / "baseline_results.csv",
    index=False
)

summary332.to_csv(
    OUT332 / "four_model_comparison.csv",
    index=False
)

means332.to_csv(
    OUT332 / "model_means.csv"
)

for key, history in histories332.items():

    pd.DataFrame(history).to_csv(
        OUT332 / f"{key}_history.csv",
        index=False
    )

    torch.save(
        states332[key],
        OUT332 / f"{key}_checkpoint.pt"
    )

report332 = {
    "experiment": "E3.32",
    "scenario": "B_owner_correlated",
    "simulation_seed": 104,
    "model_seeds": SEEDS332,
    "training_epochs": EPOCHS332,
    "training_times": TRAIN332,
    "validation_times": VAL332,
    "input_features": 17,
    "models": [
        "LogisticRegression",
        "EdgeMLP",
        "GNN_scorer",
        "Trust_only_scorer"
    ],
    "interpretation_note": (
        "No-GNN models are diagnostic prediction "
        "baselines. They do not perform edge "
        "sparsification and are not direct "
        "sparsification-policy baselines."
    ),
    "results": results332,
    "test_accessed": False,
    "hidden_state_accessed": False
}

with open(
    OUT332 / "report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(report332, f, indent=2)

print("\n" + "=" * 75)
print("E3.32 COMPLETED")
print("=" * 75)

print("\nALL MODELS — FIXED EPOCH 60")
print(
    summary332.sort_values(
        ["model_seed", "model"]
    ).round(6).to_string(index=False)
)

print("\nMODEL MEANS ACROSS THREE INITIALIZATIONS")
print(means332.round(6).to_string())

print("\nNo previous artifacts overwritten.")
print("No test or hidden-state access.")
print("Saved:", OUT332 / "report.json")


In [ ]:

# ============================================================
# E3.33 — RETRAINED FULL / RANDOM / SIMPLE TRUST BASELINES
# ============================================================
# Scenario seed 104, model seeds 44,45,46.
# Shared prediction architecture, fresh matched weights.
# Same 60 epochs / AdamW / t=5..7 / validation t=8..9.
# Random and Trust Top-K preserve all-node coverage.
# No test or hidden simulator-state access.
# ============================================================

from pathlib import Path
import copy, json, random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, roc_auc_score, log_loss

needed = ["stable", "data104", "U", "V", "K", "coverage_topk_gpu"]
assert all(x in globals() for x in needed)

OUT333 = Path(
    "/kaggle/working/phase2_adaptive_siot/"
    "E3_tags_v3/scenarios/B_owner_correlated/"
    "seed_104/baselines/e3_33_retrained_graph_baselines"
)
if OUT333.exists():
    raise FileExistsError(OUT333)

SEEDS333 = [44, 45, 46]
METHODS333 = ["Full_Graph", "Random_50", "Trust_TopK_50"]
TRAIN333 = [5, 6, 7]
VAL333 = [8, 9]
EPOCHS333 = 60
M333 = int(U.numel())
N333 = int(data104[5]["xn"].shape[0])
assert M333 == 146117 and K == 73058

def seed333(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

def select333(method, batch, random_mask):
    if method == "Full_Graph":
        return torch.ones(M333, device=U.device)

    if method == "Random_50":
        return random_mask

    # Raw feature index 3:
    # temporally decayed, Laplace-smoothed success rate.
    # StandardScaler is monotonic, preserving rankings.
    scores = batch["xe"][:, 3].detach()
    mask, _ = coverage_topk_gpu(scores, K)
    return mask.detach()

def predict333(model, batch, mask):
    x, xe, ids = batch["xn"], batch["xe"], batch["target"]
    z = model.encoder(x, U, V, mask)

    a, b = U[ids], V[ids]
    pairs = torch.cat([
        z[a] + z[b],
        (z[a] - z[b]).abs(),
        z[a] * z[b]
    ], dim=1)

    return (
        model.trust_head(xe[ids]).squeeze(-1)
        + model.graph_head(pairs).squeeze(-1)
    )

@torch.no_grad()
def validate333(model, method, masks):
    model.eval()
    ys, ps = [], []

    for t in VAL333:
        batch = data104[t]
        logits = predict333(model, batch, masks[t])
        ys.append(batch["y"].cpu().numpy())
        ps.append(torch.sigmoid(logits).cpu().numpy())

    y = np.concatenate(ys)
    p = np.clip(np.concatenate(ps), 1e-7, 1 - 1e-7)

    if method == "Full_Graph":
        churn = 0.0
    else:
        churn = float(
            (masks[8] != masks[9]).sum().item() / (2*K)
        )

    return {
        "AP": float(average_precision_score(y, p)),
        "AUC": float(roc_auc_score(y, p)),
        "BCE": float(log_loss(y, p, labels=[0, 1])),
        "churn": churn
    }

def get_masks333(method, seed):
    if method == "Random_50":
        # Fixed random selection across all time steps.
        # This produces a stationary random baseline,
        # not an independently re-sampled random mask at each t.
        gen = torch.Generator(device=U.device)
        gen.manual_seed(100000 + seed)
        scores = torch.rand(
            M333, device=U.device, generator=gen
        )
        random_mask, _ = coverage_topk_gpu(scores, K)
    else:
        random_mask = None

    masks = {
        t: select333(method, data104[t], random_mask)
        for t in TRAIN333 + VAL333
    }

    expected = M333 if method == "Full_Graph" else K

    for t, mask in masks.items():
        assert int(mask.sum().item()) == expected

        # Verify every node has at least one retained edge.
        active = torch.nonzero(mask > 0, as_tuple=True)[0]
        covered = torch.zeros(
            N333, device=U.device, dtype=torch.bool
        )
        covered[U[active]] = True
        covered[V[active]] = True
        assert bool(covered.all()), (method, seed, t)

    return masks

all_results333 = []
all_histories333 = {}
all_states333 = {}

print("=" * 72)
print("E3.33 — RETRAINED GRAPH BASELINES / SEED 104")
print("=" * 72)
print("Methods:", METHODS333)
print("Model seeds:", SEEDS333)
print("GPU:", torch.cuda.get_device_name(0))
print("Test accessed: NO")

for seed in SEEDS333:
    seed333(seed)

    sample = data104[5]
    initial = type(stable)(
        node_dim=20,
        edge_dim=17,
        controller_dim=4,
        hidden=32
    ).to(U.device)

    for method in METHODS333:
        # Matched initialization within the same model seed.
        model = copy.deepcopy(initial)

        # Edge scorer and controller are not used.
        params = [
            p for name, p in model.named_parameters()
            if not name.startswith(("scorer.", "controller."))
        ]

        optimizer = torch.optim.AdamW(
            params, lr=1e-3, weight_decay=1e-4
        )
        masks = get_masks333(method, seed)

        history = []
        best_ap = -float("inf")
        best_epoch = None
        best_state = None

        print(f"\nseed={seed} | method={method}", flush=True)

        for epoch in range(1, EPOCHS333 + 1):
            model.train()
            weighted_loss = 0.0
            total_count = 0

            for t in TRAIN333:
                batch = data104[t]
                optimizer.zero_grad(set_to_none=True)

                logits = predict333(
                    model, batch, masks[t]
                )
                loss = F.binary_cross_entropy_with_logits(
                    logits, batch["y"]
                )
                if not bool(torch.isfinite(loss)):
                    raise RuntimeError("Nonfinite training loss")

                loss.backward()
                norm = torch.nn.utils.clip_grad_norm_(params, 1.0)
                if not bool(torch.isfinite(norm)):
                    raise RuntimeError("Nonfinite gradient")
                optimizer.step()

                count = int(batch["count"])
                weighted_loss += float(loss.item()) * count
                total_count += count

            metrics = validate333(model, method, masks)

            history.append({
                "epoch": epoch,
                "train_BCE": weighted_loss / total_count,
                **metrics
            })

            if metrics["AP"] > best_ap + 1e-5:
                best_ap = metrics["AP"]
                best_epoch = epoch
                best_state = copy.deepcopy(model.state_dict())

            if epoch == 1 or epoch % 10 == 0:
                print(
                    f"epoch={epoch:02d} "
                    f"AP={metrics['AP']:.6f} "
                    f"AUC={metrics['AUC']:.6f} "
                    f"BCE={metrics['BCE']:.6f}",
                    flush=True
                )

        final = history[-1]
        key = f"{method}_seed_{seed}"

        all_results333.append({
            "scenario_seed": 104,
            "model_seed": seed,
            "model": method,
            "best_epoch": best_epoch,
            "best_AP": best_ap,
            "epoch60_AP": final["AP"],
            "epoch60_AUC": final["AUC"],
            "epoch60_BCE": final["BCE"],
            "epoch60_churn": final["churn"],
            "retained_edges": M333 if method == "Full_Graph" else K
        })
        all_histories333[key] = history
        all_states333[key] = {
            "best": best_state,
            "epoch60": copy.deepcopy(model.state_dict())
        }

        del model
        torch.cuda.empty_cache()

# ------------------------------------------------------------
# Save only after all runs succeed.
# ------------------------------------------------------------

OUT333.mkdir(parents=True, exist_ok=False)

results_df333 = pd.DataFrame(all_results333)
results_df333.to_csv(
    OUT333 / "results.csv", index=False
)

for key, history in all_histories333.items():
    pd.DataFrame(history).to_csv(
        OUT333 / f"{key}_history.csv", index=False
    )
    torch.save(
        all_states333[key],
        OUT333 / f"{key}_checkpoint.pt"
    )

summary333 = (
    results_df333.groupby("model")[
        ["epoch60_AP", "epoch60_AUC",
         "epoch60_BCE", "epoch60_churn"]
    ].agg(["mean", "std"])
)

summary333.to_csv(OUT333 / "summary.csv")

with open(OUT333 / "report.json", "w") as f:
    json.dump({
        "experiment": "E3.33",
        "simulation_seed": 104,
        "model_seeds": SEEDS333,
        "methods": METHODS333,
        "epoch": 60,
        "train_times": TRAIN333,
        "validation_times": VAL333,
        "random_baseline": "fixed mask per model seed",
        "trust_selection": "highest decayed historical success rate",
        "trainable_scoring_network": False,
        "all_node_coverage": True,
        "test_accessed": False,
        "hidden_state_accessed": False
    }, f, indent=2)

print("\n" + "=" * 72)
print("E3.33 COMPLETED")
print("=" * 72)
print("\nALL RESULTS")
print(results_df333.round(6).to_string(index=False))
print("\nMEAN RESULTS BY BASELINE")
print(summary333.round(6).to_string())
print("\nSaved:", OUT333 / "report.json")
print("Test accessed: NO")


In [ ]:

# ============================================================
# E3.34 — BUDGET SENSITIVITY, RETRAINED SCORERS
# ============================================================
# Scenario seed: 104 | Model seed: 44
# Budgets: 10%, 25%, 75%
# Compare GNN scorer vs Trust-only scorer.
#
# 50% results: existing E3.31
# Full graph: existing E3.33
#
# All new models trained independently for 60 epochs.
# Exact coverage-aware GPU selection.
# No test or hidden-state access.
# No existing outputs overwritten.
# ============================================================

from pathlib import Path
import copy
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    log_loss
)

required = [
    "stable", "data104", "U", "V",
    "coverage_topk_gpu"
]
missing = [x for x in required if x not in globals()]
assert not missing, f"Missing: {missing}"

ROOT334 = Path(
    "/kaggle/working/phase2_adaptive_siot/"
    "E3_tags_v3/scenarios/B_owner_correlated/"
    "seed_104"
)
OUT334 = ROOT334 / "budget_study/e3_34a_seed44"

if OUT334.exists():
    raise FileExistsError(OUT334)

MODEL_SEED = 44
BUDGETS = [0, 25, 0.75]
EPOCHS = 60
TRAIN = [5, 6, 7]
VALID = [8, 9]

M = U.numel()
N = data104[5]["xn"].shape[0]
assert M == 146117 and N == 16216
assert U.is_cuda and V.is_cuda

def reset334(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def check_mask334(mask, k):
    if int(mask.sum().item()) != k:
        raise RuntimeError(
            f"Budget violation: {int(mask.sum())} != {k}"
        )
    idx = torch.where(mask > 0)[0]
    coverage = torch.zeros(
        N, device=U.device, dtype=torch.bool
    )
    coverage[U[idx]] = True
    coverage[V[idx]] = True
    if not bool(coverage.all()):
        raise RuntimeError("Node coverage violated")

def forward334(model, batch, use_gnn, budget_k, training):
    xn, xe, ids = (
        batch["xn"], batch["xe"], batch["target"]
    )

    if use_gnn:
        full = xn.new_ones(M)
        h = model.encoder(xn, U, V, full)
        scorer_input = torch.cat([
            h[U] + h[V],
            (h[U] - h[V]).abs(),
            xe
        ], dim=1)
        scores = model.scorer(scorer_input).squeeze(-1)
    else:
        scores = model.scorer(xe).squeeze(-1)

    hard, _ = coverage_topk_gpu(scores, budget_k)
    check_mask334(hard, budget_k)

    if training:
        threshold = torch.topk(
            scores.detach(), budget_k
        ).values[-1]
        soft = torch.sigmoid(scores - threshold)
        mask = hard + soft - soft.detach()
    else:
        mask = hard

    z = model.encoder(xn, U, V, mask)
    a, b = U[ids], V[ids]

    pair = torch.cat([
        z[a] + z[b],
        (z[a] - z[b]).abs(),
        z[a] * z[b]
    ], dim=1)

    logits = (
        model.trust_head(xe[ids]).squeeze(-1)
        + model.graph_head(pair).squeeze(-1)
    )
    return logits, hard

@torch.no_grad()
def evaluate334(model, use_gnn, budget_k):
    model.eval()
    ys, ps, masks = [], [], []

    for t in VALID:
        batch = data104[t]
        logits, mask = forward334(
            model, batch, use_gnn, budget_k, False
        )
        ys.append(batch["y"].cpu().numpy())
        ps.append(torch.sigmoid(logits).cpu().numpy())
        masks.append(mask.cpu().numpy().astype(bool))

    y = np.concatenate(ys)
    p = np.clip(np.concatenate(ps), 1e-7, 1 - 1e-7)

    return {
        "AP": float(average_precision_score(y, p)),
        "AUC": float(roc_auc_score(y, p)),
        "BCE": float(log_loss(y, p, labels=[0, 1])),
        "churn": float(
            np.count_nonzero(masks[0] != masks[1])
            / (2 * budget_k)
        )
    }

def build_pair334():
    reset334(MODEL_SEED)

    base = type(stable)(
        node_dim=20,
        edge_dim=17,
        controller_dim=4,
        hidden=32
    ).to(U.device)

    gnn = copy.deepcopy(base)
    trust = copy.deepcopy(base)
    trust.scorer = nn.Sequential(
        nn.Linear(17, 64),
        nn.ReLU(),
        nn.Linear(64, 1)
    ).to(U.device)

    for key in base.state_dict():
        if key.startswith(
            ("encoder.", "trust_head.", "graph_head.")
        ):
            assert torch.equal(
                gnn.state_dict()[key],
                trust.state_dict()[key]
            )
    return gnn, trust

# ------------------------------------------------------------
# Feasibility check BEFORE training or writing
# ------------------------------------------------------------

print("=" * 75)
print("E3.34 — RETRAINED BUDGET SENSITIVITY")
print("=" * 75)

print("Scenario seed: 104")
print("Model seed: 44")
print("Budget fractions:", BUDGETS)

with torch.no_grad():
    probe_gnn, probe_trust = build_pair334()
    for frac in BUDGETS:
        k = int(round(M * frac))
        for name, net, use_gnn in [
            ("GNN", probe_gnn, True),
            ("Trust", probe_trust, False)
        ]:
            for t in TRAIN + VALID:
                try:
                    _, mask = forward334(
                        net, data104[t], use_gnn, k, False
                    )
                except Exception as exc:
                    raise RuntimeError(
                        f"Budget {frac:.0%} is infeasible "
                        f"or selector failed for {name} "
                        f"at t={t}: {exc}"
                    ) from exc

        print(f"Budget {frac:.0%}: k={k:,} PASSED")

    del probe_gnn, probe_trust
    torch.cuda.empty_cache()

# ------------------------------------------------------------
# Train and persist each budget independently
# ------------------------------------------------------------

OUT334.mkdir(parents=True, exist_ok=False)

records = []

for frac in BUDGETS:
    k = int(round(M * frac))
    gnn, trust = build_pair334()

    for name, model, use_gnn in [
        ("GNN_scorer", gnn, True),
        ("Trust_only_scorer", trust, False)
    ]:
        reset334(MODEL_SEED)

        params = [
            p for param_name, p in model.named_parameters()
            if not param_name.startswith("controller.")
        ]
        optimizer = torch.optim.AdamW(
            params, lr=1e-3, weight_decay=1e-4
        )

        history = []

        print(
            f"\nBudget={frac:.0%} | "
            f"edges={k:,} | model={name}",
            flush=True
        )

        for epoch in range(1, EPOCHS + 1):
            model.train()
            total_loss = 0.0
            total_n = 0

            for t in TRAIN:
                batch = data104[t]
                optimizer.zero_grad(set_to_none=True)

                logits, _ = forward334(
                    model, batch, use_gnn, k, True
                )

                loss = F.binary_cross_entropy_with_logits(
                    logits, batch["y"]
                )

                if not bool(torch.isfinite(loss)):
                    raise RuntimeError("Nonfinite loss")

                loss.backward()
                norm = torch.nn.utils.clip_grad_norm_(
                    params, 1.0
                )
                if not bool(torch.isfinite(norm)):
                    raise RuntimeError("Nonfinite gradient")
                optimizer.step()

                n = batch["count"]
                total_loss += float(loss.item()) * n
                total_n += n

            metrics = evaluate334(model, use_gnn, k)
            history.append({
                "epoch": epoch,
                "train_BCE": total_loss / total_n,
                **metrics
            })

            if epoch in [1, 20, 40, 60]:
                print(
                    f"epoch={epoch:02d} | "
                    f"AP={metrics['AP']:.6f} | "
                    f"AUC={metrics['AUC']:.6f} | "
                    f"BCE={metrics['BCE']:.6f}",
                    flush=True
                )

        final = history[-1]

        record = {
            "scenario_seed": 104,
            "model_seed": MODEL_SEED,
            "budget_fraction": frac,
            "retained_edges": k,
            "model": name,
            "epoch60_AP": final["AP"],
            "epoch60_AUC": final["AUC"],
            "epoch60_BCE": final["BCE"],
            "epoch60_churn": final["churn"]
        }
        records.append(record)

        tag = f"{name}_budget_{int(frac*100)}"

        pd.DataFrame(history).to_csv(
            OUT334 / f"{tag}_history.csv", index=False
        )
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "budget": frac,
                "retained_edges": k,
                "model_seed": MODEL_SEED,
                "simulation_seed": 104,
                "epoch": 60
            },
            OUT334 / f"{tag}_checkpoint.pt"
        )

    del gnn, trust
    torch.cuda.empty_cache()

result = pd.DataFrame(records)
result.to_csv(
    OUT334 / "results.csv", index=False
)

# Existing 50% results, no retraining.
for seed in [44]:
    path = (
        ROOT334 / "tags_v3"
        / f"e3_31_model_seed_{seed}/report.json"
    )
    with open(path, encoding="utf-8") as f:
        old = json.load(f)

    for row in old["results"]:
        records.append({
            "scenario_seed": 104,
            "model_seed": seed,
            "budget_fraction": 0.50,
            "retained_edges": 73058,
            "model": row["model"],
            "epoch60_AP": row["epoch60_AP"],
            "epoch60_AUC": row["epoch60_AUC"],
            "epoch60_BCE": row["epoch60_BCE"],
            "epoch60_churn": row["epoch60_churn"]
        })

combined = pd.DataFrame(records).sort_values(
    ["budget_fraction", "model"]
)
combined.to_csv(
    OUT334 / "budget_curve_with_existing_50.csv",
    index=False
)

pivot = combined.pivot(
    index="budget_fraction",
    columns="model",
    values="epoch60_AP"
)
pivot["delta_AP"] = (
    pivot["GNN_scorer"] - pivot["Trust_only_scorer"]
)

with open(
    OUT334 / "report.json", "w", encoding="utf-8"
) as f:
    json.dump({
        "experiment": "E3.34",
        "simulation_seed": 104,
        "model_seed": 44,
        "new_budgets": BUDGETS,
        "existing_budget": 0.50,
        "epochs": EPOCHS,
        "checkpoint": "epoch60",
        "design": "independent retraining per budget",
        "all_node_coverage_checked": True,
        "test_accessed": False,
        "hidden_state_accessed": False,
        "limitations": [
            "One scenario seed and one model seed.",
            "Exploratory budget sensitivity only.",
            "No statistical inference from this pilot."
        ]
    }, f, indent=2)

print("\n" + "=" * 75)
print("E3.34 COMPLETED — BUDGET SENSITIVITY")
print("=" * 75)
print("\nAP BY BUDGET")
print(pivot.round(6).to_string())

print("\nALL METRICS")
print(combined.round(6).to_string(index=False))
print("\nSaved:", OUT334 / "report.json")
print("Test accessed: NO")


In [ ]:

from pathlib import Path
import json

ROOT = Path(
    "/kaggle/working/phase2_adaptive_siot/"
    "E3_tags_v3"
)

BASE = (
    ROOT / "scenarios/B_owner_correlated"
)

checks = {
    "Research protocol":
        ROOT / "e3_0_research_protocol_v1.json",

    "Seed 101":
        BASE / "seed_101",

    "Seed 102":
        BASE / "seed_102",

    "Seed 103":
        BASE / "seed_103",

    "Seed 104":
        BASE / "seed_104",

    "9-run analysis":
        BASE /
        "e3_27a_three_scenario_aggregation/report.json",

    "Seed 104 input scalers":
        BASE /
        "seed_104/e3_30_model_inputs/report.json",

    "Seed 104 model seed 44":
        BASE /
        "seed_104/tags_v3/"
        "e3_31_model_seed_44/report.json",

    "Seed 104 model seed 45":
        BASE /
        "seed_104/tags_v3/"
        "e3_31_model_seed_45/report.json",

    "Seed 104 model seed 46":
        BASE /
        "seed_104/tags_v3/"
        "e3_31_model_seed_46/report.json",

    "No-GNN baselines":
        BASE /
        "seed_104/baselines/"
        "e3_32_no_gnn_predictors/report.json",

    "Graph baselines":
        BASE /
        "seed_104/baselines/"
        "e3_33_retrained_graph_baselines/report.json",

    "Budget experiment":
        BASE /
        "seed_104/budget_study"
}

print("=" * 70)
print("TAGS-v3 — SAFE RECOVERY AUDIT")
print("=" * 70)

print("Working directory exists:", ROOT.exists())

found = 0

for name, path in checks.items():
    exists = path.exists()

    print(
        f"{'FOUND  ' if exists else 'MISSING'}"
        f" | {name}"
    )

    if exists:
        found += 1

print("\nFound:", found, "/", len(checks))

print("\nSEARCH FOR SAVED CHECKPOINTS")

if BASE.exists():
    checkpoints = sorted(
        BASE.rglob("*checkpoint.pt")
    )

    print(
        "Checkpoint files:",
        len(checkpoints)
    )

    for path in checkpoints[-15:]:
        print(path.relative_to(BASE))

print("\nSEARCH FOR EXPERIMENT REPORTS")

if BASE.exists():
    reports = sorted(
        BASE.rglob("report.json")
    )

    print("Report files:", len(reports))

    for path in reports[-15:]:
        print(path.relative_to(BASE))

print("\nAUDIT FINISHED — READ ONLY")
print("No files changed.")
print("No models trained.")


In [ ]:

# ============================================================
# E3.RESTORE-1 — INSPECT SAVED ARTIFACTS
# ============================================================
# READ ONLY — NO TRAINING — NO FILE MODIFICATION
# ============================================================

from pathlib import Path
import json
import torch

BASE = Path(
    "/kaggle/working/phase2_adaptive_siot/"
    "E3_tags_v3/scenarios/B_owner_correlated/"
    "seed_104"
)

print("=" * 65)
print("STEP 1 — SEED 104 FILE INVENTORY")
print("=" * 65)

for folder in [
    BASE,
    BASE / "causal_features",
    BASE / "e3_30_model_inputs",
]:
    print("\nFOLDER:", folder)
    print("EXISTS:", folder.exists())

    if folder.exists():
        for p in sorted(folder.iterdir()):
            if p.is_file():
                print(
                    " FILE:",
                    p.name,
                    "| bytes:",
                    p.stat().st_size
                )
            else:
                print(" DIR: ", p.name)

print("\n" + "=" * 65)
print("STEP 2 — MODEL CHECKPOINT STRUCTURE")
print("=" * 65)

checkpoint = (
    BASE / "tags_v3/e3_31_model_seed_44/"
    "GNN_scorer_checkpoint.pt"
)

print("Checkpoint exists:", checkpoint.exists())

if checkpoint.exists():

    obj = torch.load(
        checkpoint,
        map_location="cpu",
        weights_only=True
    )

    print("Object type:", type(obj).__name__)

    if isinstance(obj, dict):

        print("\nTOP-LEVEL KEYS:")
        for key in obj.keys():
            print(" ", key)

        # Support direct state_dict and nested checkpoints.
        state = obj

        for candidate in [
            "model_state_dict",
            "state_dict",
            "epoch60",
            "model",
        ]:
            if (
                isinstance(state, dict)
                and candidate in state
                and isinstance(state[candidate], dict)
            ):
                state = state[candidate]
                print("\nSelected state:", candidate)
                break

        print("\nMODEL PARAMETER SHAPES:")
        if isinstance(state, dict):
            for key, value in state.items():
                if torch.is_tensor(value):
                    print(
                        f"{key:55s}",
                        tuple(value.shape)
                    )

print("\n" + "=" * 65)
print("STEP 3 — INPUT REPORT")
print("=" * 65)

report_path = (
    BASE / "e3_30_model_inputs/report.json"
)

if report_path.exists():
    with open(
        report_path,
        encoding="utf-8"
    ) as f:
        report = json.load(f)

    print(
        json.dumps(
            report,
            indent=2,
            ensure_ascii=False
        )[:12000]
    )

print("\n" + "=" * 65)
print("RESTORE INSPECTION COMPLETED")
print("No training performed.")
print("No files changed.")
print("=" * 65)


In [ ]:

# ============================================================
# E3.RESTORE-2 — SNAPSHOT SCHEMA + CHECKPOINT INSPECTION
# READ ONLY / NO TRAINING / NO FILE MODIFICATION
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import torch

BASE = Path(
    "/kaggle/working/phase2_adaptive_siot/"
    "E3_tags_v3/scenarios/B_owner_correlated/"
    "seed_104"
)

print("=" * 65)
print("STEP 1 — SNAPSHOT ARRAYS")
print("=" * 65)

for t in [5, 8]:
    path = (
        BASE / "causal_features"
        / f"snapshot_t{t:02d}.npz"
    )

    print(f"\nSNAPSHOT t={t}")

    with np.load(path, allow_pickle=False) as data:
        for key in data.files:
            value = data[key]
            print(
                f"{key:35s} "
                f"shape={str(value.shape):18s} "
                f"dtype={value.dtype}"
            )

print("\n" + "=" * 65)
print("STEP 2 — EDGE MAPPING")
print("=" * 65)

mapping = pd.read_csv(
    BASE / "causal_features/"
    "edge_index_mapping.csv.gz"
)

print("Columns:", mapping.columns.tolist())
print("Shape:", mapping.shape)
print(mapping.head(3).to_string(index=False))

print("\n" + "=" * 65)
print("STEP 3 — TRAINED MODEL PARAMETER SHAPES")
print("=" * 65)

checkpoint = (
    BASE / "tags_v3/e3_31_model_seed_44/"
    "GNN_scorer_checkpoint.pt"
)

obj = torch.load(
    checkpoint,
    map_location="cpu",
    weights_only=True
)

state = obj["epoch60_model_state_dict"]

for name, tensor in state.items():
    print(
        f"{name:55s} {tuple(tensor.shape)}"
    )

print("\n" + "=" * 65)
print("RESTORE-2 COMPLETED")
print("All operations read-only.")
print("=" * 65)


In [ ]:

# ============================================================
# E3.RESTORE-3 — RESTORE TRAINED GNN ARCHITECTURE
# + LOCATE REMAINING INPUT FILES
#
# READ ONLY — NO TRAINING — NO OVERWRITE
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import joblib

ROOT = Path(
    "/kaggle/working/phase2_adaptive_siot/"
    "E3_tags_v3"
)

BASE = (
    ROOT / "scenarios/B_owner_correlated/seed_104"
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ------------------------------------------------------------
# 1. Restore the exact stabilized GraphSAGE architecture
# ------------------------------------------------------------

class RestoredWeightedSAGE(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.self_map = nn.Linear(in_dim, out_dim)
        self.neighbor_map = nn.Linear(
            in_dim, out_dim, bias=False
        )

    def forward(self, x, u, v, weight):
        w = weight.reshape(-1, 1)

        msg = torch.zeros_like(x)
        msg = msg.index_add(0, u, x[v] * w)
        msg = msg.index_add(0, v, x[u] * w)

        deg = x.new_zeros((x.shape[0], 1))
        deg = deg.index_add(0, u, w)
        deg = deg.index_add(0, v, w)

        agg = msg / (1.0 + deg)

        return (
            self.self_map(x)
            + self.neighbor_map(agg)
        )


class RestoredEncoder(nn.Module):
    def __init__(self, node_dim=20, hidden=32):
        super().__init__()
        self.first = RestoredWeightedSAGE(
            node_dim, hidden
        )
        self.second = RestoredWeightedSAGE(
            hidden, hidden
        )

    def forward(self, x, u, v, mask):
        h = F.relu(
            self.first(x, u, v, mask)
        )
        return self.second(h, u, v, mask)


class RestoredTAGS(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = RestoredEncoder(20, 32)

        self.scorer = nn.Sequential(
            nn.Linear(81, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

        self.controller = nn.Sequential(
            nn.Linear(4, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

        self.trust_head = nn.Sequential(
            nn.Linear(17, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

        self.graph_head = nn.Sequential(
            nn.Linear(96, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )


# ------------------------------------------------------------
# 2. Strictly load the existing epoch-60 checkpoint
# ------------------------------------------------------------

checkpoint_path = (
    BASE / "tags_v3/e3_31_model_seed_44/"
    "GNN_scorer_checkpoint.pt"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

state = checkpoint["epoch60_model_state_dict"]

restored_model104 = RestoredTAGS()

# This fails if keys or tensor shapes do not match.
restored_model104.load_state_dict(
    state, strict=True
)

restored_model104 = (
    restored_model104.to(DEVICE).eval()
)

print("=" * 68)
print("STEP 1 — MODEL RESTORATION")
print("=" * 68)

print("Checkpoint:", checkpoint_path.name)
print("Simulation seed:", checkpoint["simulation_seed"])
print("Model seed:", checkpoint["model_seed"])
print("Architecture matching: PASSED")
print("Epoch-60 weights: LOADED")
print("Device:", DEVICE)

# ------------------------------------------------------------
# 3. Restore the exact graph index mapping
# ------------------------------------------------------------

mapping = pd.read_csv(
    BASE / "causal_features/"
    "edge_index_mapping.csv.gz"
)

assert len(mapping) == 146117
assert mapping[["u", "v"]].min().min() >= 1
assert mapping[["u", "v"]].max().max() <= 16216

U = torch.tensor(
    mapping["u"].to_numpy() - 1,
    dtype=torch.long,
    device=DEVICE
)

V = torch.tensor(
    mapping["v"].to_numpy() - 1,
    dtype=torch.long,
    device=DEVICE
)

M = len(mapping)
N = 16216
K = 73058

print("\n" + "=" * 68)
print("STEP 2 — GRAPH RESTORATION")
print("=" * 68)

print("Nodes:", N)
print("Edges:", M)
print("50% budget:", K)
print("Graph mapping: RESTORED")

# ------------------------------------------------------------
# 4. Restore scalers, inspect feature definitions
# ------------------------------------------------------------

scaler_dir = BASE / "e3_30_model_inputs"

edge_scaler104 = joblib.load(
    scaler_dir / "edge_scaler104.joblib"
)

node_scaler104 = joblib.load(
    scaler_dir / "node_scaler104.joblib"
)

print("\n" + "=" * 68)
print("STEP 3 — FEATURE METADATA")
print("=" * 68)

with np.load(
    BASE / "causal_features/snapshot_t05.npz",
    allow_pickle=False
) as snap:

    print("Historical feature names:")
    for i, name in enumerate(
        snap["feature_names"].tolist()
    ):
        print(f"  {i:02d}: {name}")

print(
    "\nEdge scaler input dimensions:",
    getattr(edge_scaler104, "n_features_in_", "unknown")
)

print(
    "Node scaler input dimensions:",
    getattr(node_scaler104, "n_features_in_", "unknown")
)

# ------------------------------------------------------------
# 5. Locate observed events and node feature sources
# ------------------------------------------------------------

print("\n" + "=" * 68)
print("STEP 4 — OBSERVED DATA FILES")
print("=" * 68)

observed_dir = BASE / "observed"

for p in sorted(observed_dir.rglob("*")):
    if not p.is_file():
        continue

    print(
        p.relative_to(BASE),
        "| bytes =", p.stat().st_size
    )

    if p.suffix == ".npz":
        try:
            with np.load(
                p, allow_pickle=False
            ) as z:
                print(
                    "  arrays:",
                    [(k, z[k].shape) for k in z.files]
                )
        except Exception as e:
            print("  inspection error:", str(e)[:120])

    elif p.suffix in [".csv", ".gz"]:
        try:
            header = pd.read_csv(p, nrows=2)
            print("  columns:", header.columns.tolist())
        except Exception:
            pass

print("\n" + "=" * 68)
print("STEP 5 — OTHER POSSIBLE INPUT DIRECTORIES")
print("=" * 68)

for folder in [
    ROOT,
    ROOT.parent,
    BASE / "e3_30_model_inputs",
]:
    print("\nDIRECTORY:", folder)

    if folder.exists():
        for p in sorted(folder.iterdir())[:45]:
            print(
                " DIR " if p.is_dir() else " FILE",
                p.name
            )

print("\n" + "=" * 68)
print("RESTORE-3 FINISHED")
print("Trained model successfully loaded.")
print("Graph restored.")
print("Scalers restored.")
print("No training or file modification.")
print("=" * 68)


In [ ]:
# E3.RESTORE-4 — READ ONLY
# No training, no test access, no file changes

import pandas as pd
import numpy as np
from pathlib import Path
import joblib

BASE = Path(
    "/kaggle/working/phase2_adaptive_siot/"
    "E3_tags_v3/scenarios/B_owner_correlated/seed_104"
)

print("=" * 60)
print("TRAIN AND VALIDATION DATA")
print("=" * 60)

for split in ["train", "validation"]:
    df = pd.read_csv(
        BASE / "observed" / f"{split}.csv.gz"
    )
    print(f"\n{split.upper()}")
    print("Rows:", len(df))
    print("Times:", sorted(df["time"].unique().tolist()))
    print("Columns:", df.columns.tolist())
    print(df.head(3).to_string(index=False))

print("\n" + "=" * 60)
print("SCALER DETAILS")
print("=" * 60)

edge_scaler = joblib.load(
    BASE / "e3_30_model_inputs/edge_scaler104.joblib"
)
node_scaler = joblib.load(
    BASE / "e3_30_model_inputs/node_scaler104.joblib"
)

for name, scaler in [
    ("EDGE", edge_scaler),
    ("NODE", node_scaler)
]:
    print(f"\n{name} SCALER")
    print("Type:", type(scaler).__name__)
    print("Input features:", scaler.n_features_in_)

    for attr in ["mean_", "scale_"]:
        if hasattr(scaler, attr):
            print(
                attr,
                np.asarray(
                    getattr(scaler, attr)
                ).round(6).tolist()
            )

print("\n" + "=" * 60)
print("RESTORE-4 COMPLETED")
print("No training performed.")
print("No files changed.")
print("No test accessed.")

In [ ]:
def validate324(net, use_gnn, dataset):

    net.eval()

    ys, ps, masks = [], [], []
    cover_counts = []

    for t in VAL324:
        batch = dataset[t]

        logits, hard, coverage = forward324(
            net, batch, use_gnn, False
        )

        ys.append(
            batch["y"].detach().cpu().numpy()
        )

        ps.append(
            torch.sigmoid(logits)
            .detach().cpu().numpy()
        )

        masks.append(
            hard.detach().cpu().numpy().astype(bool)
        )

        cover_counts.append(coverage)

    y = np.concatenate(ys)
    p = np.clip(
        np.concatenate(ps),
        1e-7,
        1 - 1e-7
    )

    assert all(
        int(mask.sum()) == K
        for mask in masks
    )

    return {
        "AP": float(
            average_precision_score(y, p)
        ),
        "AUC": float(
            roc_auc_score(y, p)
        ),
        "BCE": float(
            log_loss(y, p, labels=[0, 1])
        ),
        "churn": float(
            np.count_nonzero(
                masks[0] != masks[1]
            ) / (2 * K)
        ),
        "coverage_t8": cover_counts[0],
        "coverage_t9": cover_counts[1]
    }


# Reconstruct original E3.30 inputs using saved TRAIN-ONLY scalers.
raw104 = build_raw_snapshots_h(SOURCE)
assert set(raw104) == set(TIMES_H)
data104 = {}
for t in TIMES_H:
    r = raw104[t]
    xe = edge_scaler104.transform(r['edge']).astype(np.float32)
    xc = node_scaler104.transform(r['node']).astype(np.float32)
    xn = np.concatenate([type_onehot.astype(np.float32), xc], axis=1)
    data104[t] = {
        'time': t,
        'xn': torch.as_tensor(xn, dtype=torch.float32, device=DEVICE),
        'xe': torch.as_tensor(xe, dtype=torch.float32, device=DEVICE),
        'xc': torch.as_tensor(xc, dtype=torch.float32, device=DEVICE),
        'target': torch.as_tensor(r['target'], dtype=torch.long, device=DEVICE),
        'y': torch.as_tensor(r['y'], dtype=torch.float32, device=DEVICE),
        'count': int(r['count'])
    }
    assert data104[t]['xn'].shape == (N,20)
    assert data104[t]['xe'].shape == (M,17)
    assert data104[t]['xc'].shape == (N,4)
    print(f'RESTORED t={t}, observations={r["count"]:,}')
assert sum(data104[t]['count'] for t in VAL324) == 43832

stable = TAGSv3Stable(20,17,4,hidden=32).to(DEVICE)

# Strict, paired metric reproduction from the REAL saved epoch60 checkpoints.
# Scientific check: require differences <=2e-5 for AP, AUC, BCE and churn.
# No file writes.
for seed in [44,45,46]:
    report_file = SOURCE / 'tags_v3'/f'e3_31_model_seed_{seed}'/'report.json'
    report = json.loads(report_file.read_text(encoding='utf-8'))
    assert report['test_accessed'] is False
    for item in report['results']:
        method = item['model']
        ckpt = SOURCE / 'tags_v3'/f'e3_31_model_seed_{seed}'/f'{method}_checkpoint.pt'
        saved = torch.load(ckpt, map_location='cpu',weights_only=True)
        net = TAGSv3Stable(20,17,4,hidden=32).to(DEVICE)
        if method == 'Trust_only_scorer':
            net.scorer = nn.Sequential(nn.Linear(17,64), nn.ReLU(), nn.Linear(64,1)).to(DEVICE)
        net.load_state_dict(saved['epoch60_model_state_dict'], strict=True)
        actual = validate324(net, method=='GNN_scorer', data104)
        print(f'SEED {seed} {method}: AP={actual["AP"]:.6f}, AUC={actual["AUC"]:.6f}, BCE={actual["BCE"]:.6f}, churn={actual["churn"]:.6f}')
        for key in ['AP','AUC','BCE','churn']:
            expected = float(item['epoch60_'+key])
            error = abs(actual[key] - expected)
            if error > 2e-5:
                raise AssertionError(f'VALIDATION REPRODUCTION FAILED: {seed}, {method}, {key}, expected={expected}, actual={actual[key]}, error={error}. DO NOT TRAIN.')
        del net

print('\nE3.RECOVERY PASSED: all six epoch60 models reproduce original validation metrics.')
print('data104, stable, U, V, K, coverage_topk_gpu, forward324 and validate324 are ready.')
print('No training; no test; no files overwritten.')


In [ ]:
def validate324(net, use_gnn, dataset):

    net.eval()

    ys, ps, masks = [], [], []
    cover_counts = []

    for t in VAL324:
        batch = dataset[t]

        logits, hard, coverage = forward324(
            net, batch, use_gnn, False
        )

        ys.append(
            batch["y"].detach().cpu().numpy()
        )

        ps.append(
            torch.sigmoid(logits)
            .detach().cpu().numpy()
        )

        masks.append(
            hard.detach().cpu().numpy().astype(bool)
        )

        cover_counts.append(coverage)

    y = np.concatenate(ys)
    p = np.clip(
        np.concatenate(ps),
        1e-7,
        1 - 1e-7
    )

    assert all(
        int(mask.sum()) == K
        for mask in masks
    )

    return {
        "AP": float(
            average_precision_score(y, p)
        ),
        "AUC": float(
            roc_auc_score(y, p)
        ),
        "BCE": float(
            log_loss(y, p, labels=[0, 1])
        ),
        "churn": float(
            np.count_nonzero(
                masks[0] != masks[1]
            ) / (2 * K)
        ),
        "coverage_t8": cover_counts[0],
        "coverage_t9": cover_counts[1]
    }


# Reconstruct original E3.30 inputs using saved TRAIN-ONLY scalers.
raw104 = build_raw_snapshots_h(SOURCE)
assert set(raw104) == set(TIMES_H)
data104 = {}
for t in TIMES_H:
    r = raw104[t]
    xe = edge_scaler104.transform(r['edge']).astype(np.float32)
    xc = node_scaler104.transform(r['node']).astype(np.float32)
    xn = np.concatenate([type_onehot.astype(np.float32), xc], axis=1)
    data104[t] = {
        'time': t,
        'xn': torch.as_tensor(xn, dtype=torch.float32, device=DEVICE),
        'xe': torch.as_tensor(xe, dtype=torch.float32, device=DEVICE),
        'xc': torch.as_tensor(xc, dtype=torch.float32, device=DEVICE),
        'target': torch.as_tensor(r['target'], dtype=torch.long, device=DEVICE),
        'y': torch.as_tensor(r['y'], dtype=torch.float32, device=DEVICE),
        'count': int(r['count'])
    }
    assert data104[t]['xn'].shape == (N,20)
    assert data104[t]['xe'].shape == (M,17)
    assert data104[t]['xc'].shape == (N,4)
    print(f'RESTORED t={t}, observations={r["count"]:,}')
assert sum(data104[t]['count'] for t in VAL324) == 43832

stable = TAGSv3Stable(20,17,4,hidden=32).to(DEVICE)

# Strict, paired metric reproduction from the REAL saved epoch60 checkpoints.
# Scientific check: require differences <=2e-5 for AP, AUC, BCE and churn.
# No file writes.
for seed in [44,45,46]:
    report_file = SOURCE / 'tags_v3'/f'e3_31_model_seed_{seed}'/'report.json'
    report = json.loads(report_file.read_text(encoding='utf-8'))
    assert report['test_accessed'] is False
    for item in report['results']:
        method = item['model']
        ckpt = SOURCE / 'tags_v3'/f'e3_31_model_seed_{seed}'/f'{method}_checkpoint.pt'
        saved = torch.load(ckpt, map_location='cpu',weights_only=True)
        net = TAGSv3Stable(20,17,4,hidden=32).to(DEVICE)
        if method == 'Trust_only_scorer':
            net.scorer = nn.Sequential(nn.Linear(17,64), nn.ReLU(), nn.Linear(64,1)).to(DEVICE)
        net.load_state_dict(saved['epoch60_model_state_dict'], strict=True)
        actual = validate324(net, method=='GNN_scorer', data104)
        print(f'SEED {seed} {method}: AP={actual["AP"]:.6f}, AUC={actual["AUC"]:.6f}, BCE={actual["BCE"]:.6f}, churn={actual["churn"]:.6f}')
        for key in ['AP','AUC','BCE','churn']:
            expected = float(item['epoch60_'+key])
            error = abs(actual[key] - expected)
            if error > 2e-5:
                raise AssertionError(f'VALIDATION REPRODUCTION FAILED: {seed}, {method}, {key}, expected={expected}, actual={actual[key]}, error={error}. DO NOT TRAIN.')
        del net

print('\nE3.RECOVERY PASSED: all six epoch60 models reproduce original validation metrics.')
print('data104, stable, U, V, K, coverage_topk_gpu, forward324 and validate324 are ready.')
print('No training; no test; no files overwritten.')


In [ ]:
# TAGS-v3 / E3.RECOVER — faithful source extraction from uploaded notebook
# READ ONLY: no optimizer steps; never loads test.csv.gz or ground truth.
# Extracted from original notebook cells; do not run old training cells.
from pathlib import Path
import copy, json, zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import joblib
from sklearn.metrics import average_precision_score, roc_auc_score, log_loss

ROOT = Path('/kaggle/working/phase2_adaptive_siot')
SOURCE = ROOT / 'E3_tags_v3/scenarios/B_owner_correlated/seed_104'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE.type == 'cuda', 'Original validation uses CUDA; enable Kaggle GPU.'
N = N_H = N_GPU = 16216
M = M_H = M_GPU = 146117
K = 73058
REL_H = ['OOR', 'POR', 'C_LOR', 'SOR', 'SOR2']
TIMES_H = [5,6,7,8,9]
VAL324 = [8,9]
M324 = M

# Canonical edge order: do NOT sort independently, must equal saved mapping.
topology = pd.read_csv(SOURCE / 'causal_features/edge_index_mapping.csv.gz')
assert len(topology) == M
U = torch.as_tensor(topology['u'].to_numpy(np.int64)-1, dtype=torch.long, device=DEVICE)
V = torch.as_tensor(topology['v'].to_numpy(np.int64)-1, dtype=torch.long, device=DEVICE)
assert bool((U < V).all())
assert bool((U[1:] * N + V[1:] > U[:-1] * N + V[:-1]).all())

# The original 16-way device type one-hot from the source archive.
archive = ROOT / 'E1_trust_scoring/santander/raw/objects_description.zip'
assert archive.is_file(), f'Missing original device metadata: {archive}'
with zipfile.ZipFile(archive) as z:
    with z.open('objects_description.csv') as f:
        devices = pd.read_csv(f)
devices = devices.sort_values('id_device')
assert np.array_equal(devices['id_device'].to_numpy(), np.arange(1,N+1))
type_codes, type_names = pd.factorize(devices['device_type'], sort=True)
type_onehot = np.eye(len(type_names), dtype=np.float32)[type_codes]
assert type_onehot.shape == (N,16)

edge_scaler104 = joblib.load(SOURCE / 'e3_30_model_inputs/edge_scaler104.joblib')
node_scaler104 = joblib.load(SOURCE / 'e3_30_model_inputs/node_scaler104.joblib')
assert edge_scaler104.n_features_in_ == 17
assert node_scaler104.n_features_in_ == 4


# ========== original SAGE layer (notebook cell 101) ==========
class TAGSWeightedSAGE(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.self_map = nn.Linear(in_dim, out_dim)
        self.neighbor_map = nn.Linear(
            in_dim, out_dim, bias=False
        )

    def forward(self, x, u, v, weight):
        w = weight[:, None]

        msg = torch.zeros_like(x)
        msg = msg.index_add(0, u, x[v] * w)
        msg = msg.index_add(0, v, x[u] * w)

        deg = x.new_zeros((x.shape[0], 1))
        deg = deg.index_add(0, u, w)
        deg = deg.index_add(0, v, w)

        agg = msg / deg.clamp_min(1e-6)

        return (
            self.self_map(x)
            + self.neighbor_map(agg)
        )


# ========== original TAGS encoder (notebook cell 101) ==========
class TAGSEncoder(nn.Module):
    def __init__(self, input_dim, hidden=32):
        super().__init__()
        self.first = TAGSWeightedSAGE(
            input_dim, hidden
        )
        self.second = TAGSWeightedSAGE(
            hidden, hidden
        )

    def forward(self, x, u, v, weights):
        h = F.relu(
            self.first(x, u, v, weights)
        )
        return self.second(h, u, v, weights)


# ========== original TAGS v3 (notebook cell 101) ==========
class TAGSv3(nn.Module):
    def __init__(
        self, node_dim, edge_dim, controller_dim,
        hidden=32
    ):
        super().__init__()

        self.encoder = TAGSEncoder(
            node_dim, hidden
        )

        self.scorer = nn.Sequential(
            nn.Linear(hidden * 2 + edge_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

        self.controller = nn.Sequential(
            nn.Linear(controller_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

        # Direct trust pathway.
        self.trust_head = nn.Sequential(
            nn.Linear(edge_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

        # GNN correction pathway.
        self.graph_head = nn.Sequential(
            nn.Linear(hidden * 3, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(
        self, xn, xe, xc, u, v, target,
        keep_count, tau=1.0
    ):
        full = xn.new_ones(len(u))

        # Full-graph pass for edge selection.
        h = self.encoder(
            xn, u, v, full
        )

        edge_rep = torch.cat([
            h[u] + h[v],
            torch.abs(h[u] - h[v]),
            xe
        ], dim=1)

        edge_score = self.scorer(
            edge_rep
        ).squeeze(-1)

        priority = self.controller(
            xc
        ).squeeze(-1)

        score = edge_score + 0.5 * (
            priority[u] + priority[v]
        )

        # Shared symmetric decision for each
        # undirected social relationship.
        mask, hard = straight_through_topk(
            score, keep_count, tau
        )

        # Sparse-graph pass.
        z = self.encoder(
            xn, u, v, mask
        )

        a = u[target]
        b = v[target]

        pair = torch.cat([
            z[a] + z[b],
            torch.abs(z[a] - z[b]),
            z[a] * z[b]
        ], dim=1)

        trust_logit = self.trust_head(
            xe[target]
        ).squeeze(-1)

        graph_logit = self.graph_head(
            pair
        ).squeeze(-1)

        return {
            "logits": trust_logit + graph_logit,
            "scores": score,
            "mask": mask,
            "hard_mask": hard,
            "trust_logits": trust_logit,
            "graph_logits": graph_logit
        }


# ========== stabilized SAGE layer (notebook cell 102) ==========
class StableTAGSWeightedSAGE(TAGSWeightedSAGE):

    def forward(self, x, u, v, weight):
        w = weight.reshape(-1, 1)

        msg = torch.zeros_like(x)
        msg = msg.index_add(0, u, x[v] * w)
        msg = msg.index_add(0, v, x[u] * w)

        deg = x.new_zeros((x.size(0), 1))
        deg = deg.index_add(0, u, w)
        deg = deg.index_add(0, v, w)

        # Stabilized weighted aggregation:
        # avoids near-zero denominator gradients.
        agg = msg / (1.0 + deg)

        return (
            self.self_map(x)
            + self.neighbor_map(agg)
        )


# ========== stabilized encoder (notebook cell 102) ==========
class StableTAGSEncoder(nn.Module):

    def __init__(self, input_dim, hidden=32):
        super().__init__()

        self.first = StableTAGSWeightedSAGE(
            input_dim, hidden
        )
        self.second = StableTAGSWeightedSAGE(
            hidden, hidden
        )

    def forward(self, x, u, v, weights):
        h = F.relu(
            self.first(x, u, v, weights)
        )
        return self.second(h, u, v, weights)


# ========== stabilized TAGS (notebook cell 102) ==========
class TAGSv3Stable(TAGSv3):

    def __init__(
        self, node_dim, edge_dim,
        controller_dim, hidden=32
    ):
        super().__init__(
            node_dim,
            edge_dim,
            controller_dim,
            hidden
        )

        self.encoder = StableTAGSEncoder(
            node_dim, hidden
        )


# ========== original raw input builder (notebook cell 132) ==========
def build_raw_snapshots_h(folder):

    topology = pd.read_csv(
        folder / "causal_features/"
        "edge_index_mapping.csv.gz"
    )

    assert len(topology) == M_H

    u = topology["u"].to_numpy(np.int64) - 1
    v = topology["v"].to_numpy(np.int64) - 1

    reference_u = U.detach().cpu().numpy()
    reference_v = V.detach().cpu().numpy()

    assert np.array_equal(u, reference_u)
    assert np.array_equal(v, reference_v)

    relations = topology[REL_H].to_numpy(
        dtype=np.float32
    )

    static_degree = np.bincount(
        np.concatenate([u, v]),
        minlength=N_H
    )

    degree_feature = np.log1p(
        static_degree
    ).astype(np.float32)

    # Only allowed observed outcomes.
    train = pd.read_csv(
        folder / "observed/train.csv.gz"
    )

    val = pd.read_csv(
        folder / "observed/validation.csv.gz"
    )

    assert train["time"].between(0, 7).all()
    assert val["time"].between(8, 9).all()

    assert train["success"].isin([0, 1]).all()
    assert val["success"].isin([0, 1]).all()

    events = pd.concat(
        [train, val], ignore_index=True
    )

    # Canonical edge mapping.
    keys = u * N_H + v

    assert np.all(np.diff(keys) > 0)

    source = events["source"].to_numpy(
        np.int64
    ) - 1

    target = events["target"].to_numpy(
        np.int64
    ) - 1

    assert np.all(source < target)

    event_keys = source * N_H + target

    event_edge_ids = np.searchsorted(
        keys, event_keys
    )

    assert np.all(event_edge_ids < M_H)
    assert np.array_equal(
        keys[event_edge_ids], event_keys
    )

    event_times = events["time"].to_numpy(
        np.int16
    )

    event_success = events["success"].to_numpy(
        np.float64
    )

    raw = {}

    for t in TIMES_H:

        # Strictly historical observations.
        # At t=9, t=8 validation observations may
        # contribute to causal history, but never
        # to model/scaler fitting.
        past = event_times < t

        past_u = source[past]
        past_v = target[past]
        past_success = event_success[past]

        node_count = np.bincount(
            np.concatenate([past_u, past_v]),
            minlength=N_H
        )

        node_success = (
            np.bincount(
                past_u,
                weights=past_success,
                minlength=N_H
            )
            + np.bincount(
                past_v,
                weights=past_success,
                minlength=N_H
            )
        )

        node_raw = np.column_stack([
            degree_feature,
            np.log1p(node_count),
            (node_success + 1.0) /
            (node_count + 2.0),
            node_count / (node_count + 5.0)
        ]).astype(np.float32)

        assert node_raw.shape == (N_H, 4)

        with np.load(
            folder / "causal_features" /
            f"snapshot_t{t:02d}.npz"
        ) as archive:

            history = archive["features"].astype(
                np.float32
            )

            assert int(archive["time"]) == t

        assert history.shape == (M_H, 12)

        # Independent temporal-integrity check.
        assert abs(
            float(history[:, 0].sum())
            - int(past.sum())
        ) < 1.0

        ru = node_raw[u, 2]
        rv = node_raw[v, 2]

        cu = node_raw[u, 3]
        cv = node_raw[v, 3]

        endpoints = np.column_stack([
            (ru + rv) / 2.0,
            np.abs(ru - rv),
            (cu + cv) / 2.0,
            np.abs(cu - cv)
        ]).astype(np.float32)

        edge_raw = np.column_stack([
            np.log1p(history[:, 0]),
            history[:, 1],
            np.log1p(history[:, 2]),
            history[:, 3:7],
            endpoints,
            history[:, 11],
            relations
        ]).astype(np.float32)

        assert edge_raw.shape == (M_H, 17)

        current = event_times == t

        current_ids = event_edge_ids[current]

        # Failure = 1 - success.
        failures = (
            1.0 - event_success[current]
        ).astype(np.float32)

        assert len(current_ids) == len(failures)

        raw[t] = {
            "node": node_raw,
            "edge": edge_raw,
            "target": current_ids.astype(np.int64),
            "y": failures,
            "count": int(len(current_ids))
        }

    return raw


# ========== original GPU selector (notebook cell 115) ==========
@torch.no_grad()
def coverage_topk_gpu(scores, k):
    """
    Exact equivalent of the existing CPU greedy
    coverage selection, including stable tie order.

    Phase 1:
      Select the highest-priority incident edge
      for every node, then take their union.

    Phase 2:
      Fill the remaining budget with the
      highest-ranked unselected edges.
    """

    assert scores.is_cuda
    assert scores.ndim == 1
    assert scores.numel() == M_GPU
    assert bool(torch.isfinite(scores).all())
    assert 0 < k <= M_GPU

    device = scores.device

    # Same ranking semantics as:
    # np.argsort(-scores, kind="stable")
    order = torch.argsort(
        scores.detach(),
        descending=True,
        stable=True
    )

    rank = torch.empty_like(order)

    rank[order] = torch.arange(
        M_GPU,
        device=device,
        dtype=torch.long
    )

    # For each node, find its earliest incident
    # edge in the globally ranked ordering.
    first_rank = torch.full(
        (N_GPU,),
        M_GPU,
        dtype=torch.long,
        device=device
    )

    first_rank.scatter_reduce_(
        0, U, rank,
        reduce="amin",
        include_self=True
    )

    first_rank.scatter_reduce_(
        0, V, rank,
        reduce="amin",
        include_self=True
    )

    if bool((first_rank == M_GPU).any()):
        raise RuntimeError(
            "Original topology contains isolated nodes."
        )

    # One highest-ranked incident edge per node.
    # Multiple nodes can select the same edge.
    cover_ids = order[first_rank]

    selected = torch.zeros(
        M_GPU,
        dtype=torch.bool,
        device=device
    )

    selected[cover_ids] = True

    coverage_count = int(
        selected.sum().item()
    )

    if coverage_count > k:
        raise RuntimeError(
            "Node coverage exceeds edge budget."
        )

    # Complete the exact global budget, using
    # the original ranking order.
    remaining = k - coverage_count

    if remaining > 0:
        available = order[
            ~selected[order]
        ]

        selected[
            available[:remaining]
        ] = True

    hard = selected.to(
        dtype=scores.dtype
    )

    return hard, coverage_count


# ========== original scorer forward (notebook cell 140) ==========
def forward324(net, batch, use_gnn, training):

    xn = batch["xn"]
    xe = batch["xe"]
    ids = batch["target"]

    if use_gnn:
        full = xn.new_ones(M324)

        h = net.encoder(
            xn, U, V, full
        )

        scoring_input = torch.cat([
            h[U] + h[V],
            torch.abs(h[U] - h[V]),
            xe
        ], dim=1)

        scores = net.scorer(
            scoring_input
        ).squeeze(-1)
    else:
        scores = net.scorer(
            xe
        ).squeeze(-1)

    hard, coverage_count = coverage_topk_gpu(
        scores, K
    )

    if training:
        threshold = torch.topk(
            scores.detach(), K
        ).values[-1]

        soft = torch.sigmoid(
            scores - threshold
        )

        mask = hard + (soft - soft.detach())
    else:
        mask = hard

    z = net.encoder(
        xn, U, V, mask
    )

    a = U[ids]
    b = V[ids]

    pair_features = torch.cat([
        z[a] + z[b],
        torch.abs(z[a] - z[b]),
        z[a] * z[b]
    ], dim=1)

    logits = (
        net.trust_head(
            xe[ids]
        ).squeeze(-1)
        +
        net.graph_head(
            pair_features
        ).squeeze(-1)
    )

    return logits, hard, int(coverage_count)


# ========== original validation (notebook cell 140) ==========
@torch.no_grad()
def validate324(net, use_gnn, dataset):

    net.eval()

    ys, ps, masks = [], [], []
    cover_counts = []

    for t in VAL324:
        batch = dataset[t]

        logits, hard, coverage = forward324(
            net, batch, use_gnn, False
        )

        ys.append(
            batch["y"].detach().cpu().numpy()
        )

        ps.append(
            torch.sigmoid(logits)
            .detach().cpu().numpy()
        )

        masks.append(
            hard.detach().cpu().numpy().astype(bool)
        )

        cover_counts.append(coverage)

    y = np.concatenate(ys)
    p = np.clip(
        np.concatenate(ps),
        1e-7,
        1 - 1e-7
    )

    assert all(
        int(mask.sum()) == K
        for mask in masks
    )

    return {
        "AP": float(
            average_precision_score(y, p)
        ),
        "AUC": float(
            roc_auc_score(y, p)
        ),
        "BCE": float(
            log_loss(y, p, labels=[0, 1])
        ),
        "churn": float(
            np.count_nonzero(
                masks[0] != masks[1]
            ) / (2 * K)
        ),
        "coverage_t8": cover_counts[0],
        "coverage_t9": cover_counts[1]
    }


# Reconstruct original E3.30 inputs using saved TRAIN-ONLY scalers.
raw104 = build_raw_snapshots_h(SOURCE)
assert set(raw104) == set(TIMES_H)
data104 = {}
for t in TIMES_H:
    r = raw104[t]
    xe = edge_scaler104.transform(r['edge']).astype(np.float32)
    xc = node_scaler104.transform(r['node']).astype(np.float32)
    xn = np.concatenate([type_onehot.astype(np.float32), xc], axis=1)
    data104[t] = {
        'time': t,
        'xn': torch.as_tensor(xn, dtype=torch.float32, device=DEVICE),
        'xe': torch.as_tensor(xe, dtype=torch.float32, device=DEVICE),
        'xc': torch.as_tensor(xc, dtype=torch.float32, device=DEVICE),
        'target': torch.as_tensor(r['target'], dtype=torch.long, device=DEVICE),
        'y': torch.as_tensor(r['y'], dtype=torch.float32, device=DEVICE),
        'count': int(r['count'])
    }
    assert data104[t]['xn'].shape == (N,20)
    assert data104[t]['xe'].shape == (M,17)
    assert data104[t]['xc'].shape == (N,4)
    print(f'RESTORED t={t}, observations={r["count"]:,}')
assert sum(data104[t]['count'] for t in VAL324) == 43832

stable = TAGSv3Stable(20,17,4,hidden=32).to(DEVICE)

# Strict, paired metric reproduction from the REAL saved epoch60 checkpoints.
# Scientific check: require differences <=2e-5 for AP, AUC, BCE and churn.
# No file writes.
for seed in [44,45,46]:
    report_file = SOURCE / 'tags_v3'/f'e3_31_model_seed_{seed}'/'report.json'
    report = json.loads(report_file.read_text(encoding='utf-8'))
    assert report['test_accessed'] is False
    for item in report['results']:
        method = item['model']
        ckpt = SOURCE / 'tags_v3'/f'e3_31_model_seed_{seed}'/f'{method}_checkpoint.pt'
        saved = torch.load(ckpt, map_location='cpu',weights_only=True)
        net = TAGSv3Stable(20,17,4,hidden=32).to(DEVICE)
        if method == 'Trust_only_scorer':
            net.scorer = nn.Sequential(nn.Linear(17,64), nn.ReLU(), nn.Linear(64,1)).to(DEVICE)
        net.load_state_dict(saved['epoch60_model_state_dict'], strict=True)
        actual = validate324(net, method=='GNN_scorer', data104)
        print(f'SEED {seed} {method}: AP={actual["AP"]:.6f}, AUC={actual["AUC"]:.6f}, BCE={actual["BCE"]:.6f}, churn={actual["churn"]:.6f}')
        for key in ['AP','AUC','BCE','churn']:
            expected = float(item['epoch60_'+key])
            error = abs(actual[key] - expected)
            if error > 2e-5:
                raise AssertionError(f'VALIDATION REPRODUCTION FAILED: {seed}, {method}, {key}, expected={expected}, actual={actual[key]}, error={error}. DO NOT TRAIN.')
        del net

print('\nE3.RECOVERY PASSED: all six epoch60 models reproduce original validation metrics.')
print('data104, stable, U, V, K, coverage_topk_gpu, forward324 and validate324 are ready.')
print('No training; no test; no files overwritten.')


In [ ]:

# E3.34-PREFLIGHT — Coverage feasibility audit
# Run AFTER successful E3.RECOVERY
# No training, no file writes, no test access.

import torch
import torch.nn as nn

assert all(
    name in globals()
    for name in ["data104", "U", "V", "K",
                 "TAGSv3Stable", "coverage_topk_gpu"]
)

budgets = [0.10, 0.25, 0.50, 0.75]
M_audit = U.numel()
N_audit = data104[5]["xn"].shape[0]
results_audit = []

for model_name, use_gnn in [
    ("GNN_scorer", True),
    ("Trust_only_scorer", False)
]:
    ckpt_path = (
        SOURCE / "tags_v3/e3_31_model_seed_44"
        / f"{model_name}_checkpoint.pt"
    )

    net = TAGSv3Stable(20, 17, 4, hidden=32).to(U.device)

    if not use_gnn:
        net.scorer = nn.Sequential(
            nn.Linear(17, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        ).to(U.device)

    ckpt = torch.load(
        ckpt_path, map_location="cpu",
        weights_only=True
    )
    net.load_state_dict(
        ckpt["epoch60_model_state_dict"], strict=True
    )
    net.eval()

    for t in [5, 6, 7, 8, 9]:
        batch = data104[t]
        with torch.no_grad():
            if use_gnn:
                full = batch["xn"].new_ones(M_audit)
                h = net.encoder(batch["xn"], U, V, full)
                representation = torch.cat([
                    h[U] + h[V],
                    (h[U] - h[V]).abs(),
                    batch["xe"]
                ], dim=1)
                scores = net.scorer(
                    representation
                ).squeeze(-1)
            else:
                scores = net.scorer(
                    batch["xe"]
                ).squeeze(-1)

            # Exact mandatory coverage size under
            # the ORIGINAL GPU greedy selector.
            order = torch.argsort(
                scores, descending=True, stable=True
            )
            ranks = torch.empty_like(order)
            ranks[order] = torch.arange(
                M_audit, device=U.device
            )
            first = torch.full(
                (N_audit,), M_audit,
                dtype=torch.long, device=U.device
            )
            first.scatter_reduce_(
                0, U, ranks,
                reduce="amin", include_self=True
            )
            first.scatter_reduce_(
                0, V, ranks,
                reduce="amin", include_self=True
            )
            assert bool((first < M_audit).all())

            required_edges = int(
                torch.unique(order[first]).numel()
            )

        for fraction in budgets:
            budget_k = round(M_audit * fraction)
            results_audit.append({
                "model": model_name,
                "time": t,
                "budget": fraction,
                "allowed_edges": budget_k,
                "coverage_edges": required_edges,
                "feasible": required_edges <= budget_k
            })

    del net
    torch.cuda.empty_cache()

import pandas as pd

audit_df = pd.DataFrame(results_audit)

print("\nCOVERAGE AUDIT — ORIGINAL SELECTOR")
print(
    audit_df.to_string(index=False)
)

print("\nSUMMARY BY BUDGET")
print(
    audit_df.groupby("budget").agg(
        max_required=("coverage_edges", "max"),
        min_allowed=("allowed_edges", "min"),
        all_pass=("feasible", "all")
    ).to_string()
)

print("\nREAD-ONLY AUDIT COMPLETED")
print("No training, no test, no file changes.")


In [ ]:
logits=net.trust_head(xe[ids]).squeeze(-1)+net.graph_head(pairs).squeeze(-1)
    return logits,hard,int(coverage)

@torch.no_grad()
def validate_b(net,use_gnn,k):
    net.eval()
    ys,ps,masks,covs=[],[],[],[]
    for t in VAL_B:
        logits,hard,cover=forward_b(net,data104[t],use_gnn,k,False)
        ys.append(data104[t]['y'].detach().cpu().numpy())
        ps.append(torch.sigmoid(logits).detach().cpu().numpy())
        masks.append(hard.detach().cpu().numpy().astype(bool))
        covs.append(cover)
    y=np.concatenate(ys); p=np.clip(np.concatenate(ps),1e-7,1-1e-7)
    return dict(AP=float(average_precision_score(y,p)),
                AUC=float(roc_auc_score(y,p)),
                BCE=float(log_loss(y,p,labels=[0,1])),
                churn=float(np.count_nonzero(masks[0]!=masks[1])/(2*k)),
                coverage_t8=covs[0],coverage_t9=covs[1])

# Run once at the old 50% budget as an implementation parity check.
# No training, read existing checkpoint. Must agree with the recovered E3.31 numbers.
ckpt_path_b=SOURCE/'tags_v3/e3_31_model_seed_44/GNN_scorer_checkpoint.pt'
ckpt_b=torch.load(ckpt_path_b,map_location='cpu',weights_only=True)
parity_b=type(stable)(node_dim=20,edge_dim=17,controller_dim=4,hidden=32).to(U.device)
parity_b.load_state_dict(ckpt_b['epoch60_model_state_dict'],strict=True)
parity_metrics_b=validate_b(parity_b,True,K)
assert abs(parity_metrics_b['AP']-0.460809)<2e-6, parity_metrics_b
print('50% parity check PASSED:',{k:round(v,6) for k,v in parity_metrics_b.items()})
del parity_b

print('\nE3.34b — start: budgets 25%,75%; seed44; 60 epochs; TEST SEALED',flush=True)
rows_b=[]
for fraction_b in BUDGETS_B:
    k_b=round(M_B*fraction_b)
    reset_rng_b(MODEL_SEED_B)
    base_b=type(stable)(node_dim=20,edge_dim=17,controller_dim=4,hidden=32).to(U.device)
    gnn_b=copy.deepcopy(base_b)
    trust_b=copy.deepcopy(base_b)
    # Same trust-scorer initialization sequence as E3.31.
    trust_b.scorer=nn.Sequential(nn.Linear(17,64),nn.ReLU(),nn.Linear(64,1)).to(U.device)
    for name_b in base_b.state_dict():
        if name_b.startswith(('encoder.','trust_head.','graph_head.')):
            assert torch.equal(gnn_b.state_dict()[name_b],trust_b.state_dict()[name_b])
    for method_b, use_gnn_b, net_b in [
        ('GNN_scorer',True,gnn_b),('Trust_only_scorer',False,trust_b)
    ]:
        reset_rng_b(MODEL_SEED_B)  # same as E3.31 train_one331
        params_b=[p for n,p in net_b.named_parameters() if not n.startswith('controller.')]
        optimizer_b=torch.optim.AdamW(params_b,lr=1e-3,weight_decay=1e-4)
        history_b=[]
        print(f'\nBudget={fraction_b:.0%} k={k_b} | {method_b}',flush=True)
        best_ap_b=-float('inf'); best_epoch_b=None; best_state_b=None
        for epoch_b in range(1,EPOCHS_B+1):
            net_b.train()
            losses_b=[]
            for t_b in TRAIN_B:
                batch_b=data104[t_b]
                optimizer_b.zero_grad(set_to_none=True)
                logits_b,hard_b,_=forward_b(net_b,batch_b,use_gnn_b,k_b,True)
                loss_b=F.binary_cross_entropy_with_logits(logits_b,batch_b['y'])
                assert bool(torch.isfinite(loss_b)),('Nonfinite loss',method_b,epoch_b,t_b)
                loss_b.backward()
                norm_b=torch.nn.utils.clip_grad_norm_(params_b,max_norm=1.0)
                assert bool(torch.isfinite(norm_b)),('Nonfinite norm',method_b,epoch_b,t_b)
                optimizer_b.step()
                losses_b.append((float(loss_b.item()),int(batch_b['count'])))
            train_bce_b=sum(l*n for l,n in losses_b)/sum(n for l,n in losses_b)
            val_b=validate_b(net_b,use_gnn_b,k_b)
            history_b.append(dict(epoch=epoch_b,train_BCE=train_bce_b,**val_b))
            if val_b['AP']>best_ap_b+1e-5:
                best_ap_b=val_b['AP'];best_epoch_b=epoch_b
                best_state_b=copy.deepcopy(net_b.state_dict())
            if epoch_b==1 or epoch_b%10==0:
                print(f' epoch {epoch_b:02d}: AP={val_b["AP"]:.6f}, AUC={val_b["AUC"]:.6f}, BCE={val_b["BCE"]:.6f}, churn={val_b["churn"]:.6f}',flush=True)
        row_b=dict(simulation_seed=104,model_seed=MODEL_SEED_B,
                   budget_fraction=fraction_b,retained_edges=k_b,
                   model=method_b,best_epoch=best_epoch_b,best_AP=best_ap_b,
                   epoch60_AP=val_b['AP'],epoch60_AUC=val_b['AUC'],
                   epoch60_BCE=val_b['BCE'],epoch60_churn=val_b['churn'],
                   coverage_t8=val_b['coverage_t8'],coverage_t9=val_b['coverage_t9'])
        # Save immediately: earlier successful runs survive later errors.
        tag_b=f'{method_b}_{int(fraction_b*100)}pct_seed44'
        pd.DataFrame(history_b).to_csv(OUT_B/f'{tag_b}_history.csv',index=False)
        torch.save(dict(best_model_state_dict=best_state_b,
                        epoch60_model_state_dict=copy.deepcopy(net_b.state_dict()),
                        method=method_b,simulation_seed=104,model_seed=44,
                        budget_fraction=fraction_b,epochs=EPOCHS_B),
                   OUT_B/f'{tag_b}_checkpoint.pt')
        rows_b.append(row_b)
        pd.DataFrame(rows_b).to_csv(OUT_B/'completed_results.csv',index=False)
        print(f' SAVED {tag_b}',flush=True)
    del base_b,gnn_b,trust_b
    torch.cuda.empty_cache()

# Append exact 50% reference values for seed 44 (from the sealed E3.31 report).
ref_b=json.loads((SOURCE/'tags_v3/e3_31_model_seed_44/report.json').read_text())
combined_b=list(rows_b)
for r_b in ref_b['results']:
    combined_b.append(dict(simulation_seed=104,model_seed=44,budget_fraction=0.50,
                           retained_edges=K,model=r_b['model'],
                           epoch60_AP=float(r_b['epoch60_AP']),
                           epoch60_AUC=float(r_b['epoch60_AUC']),
                           epoch60_BCE=float(r_b['epoch60_BCE']),
                           epoch60_churn=float(r_b['epoch60_churn'])))
combined_df_b=pd.DataFrame(combined_b).sort_values(['budget_fraction','model'])
combined_df_b.to_csv(OUT_B/'budget_curve_with_50pct_reference.csv',index=False)
pivot_b=combined_df_b.pivot(index='budget_fraction',columns='model',values='epoch60_AP')
pivot_b['GNN_minus_Trust_AP']=pivot_b['GNN_scorer']-pivot_b['Trust_only_scorer']
(OUT_B/'report.json').write_text(json.dumps(dict(experiment='E3.34b',simulation_seed=104,
   model_seed=44,budgets=[0.25,0.50,0.75],new_budgets=BUDGETS_B,epochs=60,
   reference_50pct='E3.31',test_accessed=False,hidden_state_accessed=False,
   original_selector=True,matched_initializations=True,
   caveat='Exploratory one-scenario one-initialization sensitivity; no causal benefit claim.'),indent=2))
print('\nE3.34b COMPLETED\nAP BY BUDGET')
print(pivot_b.round(6).to_string())
print('\nALL METRICS')
print(combined_df_b.round(6).to_string(index=False))
print('\nSaved:',OUT_B)
print('TEST accessed: NO; existing experiments overwritten: NO')

In [ ]:
from pathlib import Path
import pandas as pd

p = Path(
    "/kaggle/working/phase2_adaptive_siot/"
    "E3_tags_v3/scenarios/B_owner_correlated/"
    "seed_104/budget_study/"
    "e3_34b_recovered_seed44_25_75"
)

print("Output folder exists:", p.exists())

if p.exists():
    files = sorted(p.iterdir())
    print("Saved files:", len(files))

    for f in files:
        print(f.name, "|", f.stat().st_size, "bytes")

    results = p / "completed_results.csv"

    if results.exists():
        print("\nCOMPLETED RUNS:")
        print(pd.read_csv(results).to_string(index=False))
    else:
        print("\nNo completed training runs saved yet.")

print("\nDIAGNOSTIC COMPLETED")

In [ ]:
import torch
import os

print("Python process:", os.getpid())
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Allocated GB:",
          round(torch.cuda.memory_allocated() / 1e9, 3))
    print("Reserved GB:",
          round(torch.cuda.memory_reserved() / 1e9, 3))
    free, total = torch.cuda.mem_get_info()
    print("GPU free GB:", round(free / 1e9, 3))
    print("GPU total GB:", round(total / 1e9, 3))

print("data104 available:", "data104" in globals())
print("stable available:", "stable" in globals())
print("DIAGNOSTIC PASSED")

In [ ]:
# E3.34b — STARTUP DIAGNOSTIC
# No training, no file changes, no test access.

import time
import copy
import torch
import torch.nn as nn
import traceback

print("STEP 1 — Starting", flush=True)

try:
    t0 = time.perf_counter()

    assert "stable" in globals()
    assert "data104" in globals()
    assert "U" in globals()

    print("STEP 2 — Variables OK", flush=True)

    torch.manual_seed(44)
    torch.cuda.manual_seed_all(44)

    print("STEP 3 — RNG OK", flush=True)

    base_test = type(stable)(
        node_dim=20,
        edge_dim=17,
        controller_dim=4,
        hidden=32
    ).to(U.device)

    print("STEP 4 — Base model created", flush=True)

    gnn_test = copy.deepcopy(base_test)

    print("STEP 5 — GNN copy created", flush=True)

    trust_test = copy.deepcopy(base_test)

    print("STEP 6 — Trust copy created", flush=True)

    trust_test.scorer = nn.Sequential(
        nn.Linear(17, 64),
        nn.ReLU(),
        nn.Linear(64, 1)
    ).to(U.device)

    print("STEP 7 — Trust scorer created", flush=True)

    params_test = [
        p for name, p in gnn_test.named_parameters()
        if not name.startswith("controller.")
    ]

    optimizer_test = torch.optim.AdamW(
        params_test,
        lr=1e-3,
        weight_decay=1e-4
    )

    print("STEP 8 — Optimizer created", flush=True)

    print(
        "Time:",
        round(time.perf_counter() - t0, 3),
        "seconds"
    )

    print("STARTUP DIAGNOSTIC PASSED")

except Exception:
    print("STARTUP DIAGNOSTIC FAILED")
    traceback.print_exc()

finally:
    for name in [
        "base_test", "gnn_test",
        "trust_test", "optimizer_test"
    ]:
        if name in globals():
            del globals()[name]

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# E3.34b — CONTINUE FROM THE ALREADY-EXECUTED SETUP
# Run in ONE new Kaggle code cell. No rerun of the preceding setup.
# Existing output folder is expected to exist and be empty.
from pathlib import Path
import torch, pandas as pd, numpy as np, copy, json
import torch.nn as nn
import torch.nn.functional as F
assert all(x in globals() for x in (
    'OUT_B','BUDGETS_B','M_B','U','V','data104','stable',
    'MODEL_SEED_B','METHODS_B','TRAIN_B','VAL_B','EPOCHS_B',
    'reset_rng_b','forward_b','validate_b','SOURCE'))
assert OUT_B.is_dir(), 'Missing prepared output directory. Do not rerun setup blindly.'
existing_b=list(OUT_B.iterdir())
assert not existing_b, f'Output is not empty; stop to avoid overwriting: {existing_b}'
print('E3.34b CONTINUATION: prerequisites OK, starting training', flush=True)

rows_b=[]
for fraction_b in BUDGETS_B:
    k_b=round(M_B*fraction_b)
    reset_rng_b(MODEL_SEED_B)
    base_b=type(stable)(node_dim=20,edge_dim=17,controller_dim=4,hidden=32).to(U.device)
    gnn_b=copy.deepcopy(base_b)
    trust_b=copy.deepcopy(base_b)
    # Same trust-scorer initialization sequence as E3.31.
    trust_b.scorer=nn.Sequential(nn.Linear(17,64),nn.ReLU(),nn.Linear(64,1)).to(U.device)
    for name_b in base_b.state_dict():
        if name_b.startswith(('encoder.','trust_head.','graph_head.')):
            assert torch.equal(gnn_b.state_dict()[name_b],trust_b.state_dict()[name_b])
    for method_b, use_gnn_b, net_b in [
        ('GNN_scorer',True,gnn_b),('Trust_only_scorer',False,trust_b)
    ]:
        reset_rng_b(MODEL_SEED_B)  # same as E3.31 train_one331
        params_b=[p for n,p in net_b.named_parameters() if not n.startswith('controller.')]
        optimizer_b=torch.optim.AdamW(params_b,lr=1e-3,weight_decay=1e-4)
        history_b=[]
        print(f'\nBudget={fraction_b:.0%} k={k_b} | {method_b}',flush=True)
        best_ap_b=-float('inf'); best_epoch_b=None; best_state_b=None
        for epoch_b in range(1,EPOCHS_B+1):
            net_b.train()
            losses_b=[]
            for t_b in TRAIN_B:
                batch_b=data104[t_b]
                optimizer_b.zero_grad(set_to_none=True)
                logits_b,hard_b,_=forward_b(net_b,batch_b,use_gnn_b,k_b,True)
                loss_b=F.binary_cross_entropy_with_logits(logits_b,batch_b['y'])
                assert bool(torch.isfinite(loss_b)),('Nonfinite loss',method_b,epoch_b,t_b)
                loss_b.backward()
                norm_b=torch.nn.utils.clip_grad_norm_(params_b,max_norm=1.0)
                assert bool(torch.isfinite(norm_b)),('Nonfinite norm',method_b,epoch_b,t_b)
                optimizer_b.step()
                losses_b.append((float(loss_b.item()),int(batch_b['count'])))
            train_bce_b=sum(l*n for l,n in losses_b)/sum(n for l,n in losses_b)
            val_b=validate_b(net_b,use_gnn_b,k_b)
            history_b.append(dict(epoch=epoch_b,train_BCE=train_bce_b,**val_b))
            if val_b['AP']>best_ap_b+1e-5:
                best_ap_b=val_b['AP'];best_epoch_b=epoch_b
                best_state_b=copy.deepcopy(net_b.state_dict())
            if epoch_b==1 or epoch_b%10==0:
                print(f' epoch {epoch_b:02d}: AP={val_b["AP"]:.6f}, AUC={val_b["AUC"]:.6f}, BCE={val_b["BCE"]:.6f}, churn={val_b["churn"]:.6f}',flush=True)
        row_b=dict(simulation_seed=104,model_seed=MODEL_SEED_B,
                   budget_fraction=fraction_b,retained_edges=k_b,
                   model=method_b,best_epoch=best_epoch_b,best_AP=best_ap_b,
                   epoch60_AP=val_b['AP'],epoch60_AUC=val_b['AUC'],
                   epoch60_BCE=val_b['BCE'],epoch60_churn=val_b['churn'],
                   coverage_t8=val_b['coverage_t8'],coverage_t9=val_b['coverage_t9'])
        # Save immediately: earlier successful runs survive later errors.
        tag_b=f'{method_b}_{int(fraction_b*100)}pct_seed44'
        pd.DataFrame(history_b).to_csv(OUT_B/f'{tag_b}_history.csv',index=False)
        torch.save(dict(best_model_state_dict=best_state_b,
                        epoch60_model_state_dict=copy.deepcopy(net_b.state_dict()),
                        method=method_b,simulation_seed=104,model_seed=44,
                        budget_fraction=fraction_b,epochs=EPOCHS_B),
                   OUT_B/f'{tag_b}_checkpoint.pt')
        rows_b.append(row_b)
        pd.DataFrame(rows_b).to_csv(OUT_B/'completed_results.csv',index=False)
        print(f' SAVED {tag_b}',flush=True)
    del base_b,gnn_b,trust_b
    torch.cuda.empty_cache()

# Append exact 50% reference values for seed 44 (from the sealed E3.31 report).
ref_b=json.loads((SOURCE/'tags_v3/e3_31_model_seed_44/report.json').read_text())
combined_b=list(rows_b)
for r_b in ref_b['results']:
    combined_b.append(dict(simulation_seed=104,model_seed=44,budget_fraction=0.50,
                           retained_edges=K,model=r_b['model'],
                           epoch60_AP=float(r_b['epoch60_AP']),
                           epoch60_AUC=float(r_b['epoch60_AUC']),
                           epoch60_BCE=float(r_b['epoch60_BCE']),
                           epoch60_churn=float(r_b['epoch60_churn'])))
combined_df_b=pd.DataFrame(combined_b).sort_values(['budget_fraction','model'])
combined_df_b.to_csv(OUT_B/'budget_curve_with_50pct_reference.csv',index=False)
pivot_b=combined_df_b.pivot(index='budget_fraction',columns='model',values='epoch60_AP')
pivot_b['GNN_minus_Trust_AP']=pivot_b['GNN_scorer']-pivot_b['Trust_only_scorer']
(OUT_B/'report.json').write_text(json.dumps(dict(experiment='E3.34b',simulation_seed=104,
   model_seed=44,budgets=[0.25,0.50,0.75],new_budgets=BUDGETS_B,epochs=60,
   reference_50pct='E3.31',test_accessed=False,hidden_state_accessed=False,
   original_selector=True,matched_initializations=True,
   caveat='Exploratory one-scenario one-initialization sensitivity; no causal benefit claim.'),indent=2))
print('\nE3.34b COMPLETED\nAP BY BUDGET')
print(pivot_b.round(6).to_string())
print('\nALL METRICS')
print(combined_df_b.round(6).to_string(index=False))
print('\nSaved:',OUT_B)
print('TEST accessed: NO; existing experiments overwritten: NO')

In [ ]:
net335=type(stable)(node_dim=20,edge_dim=17,controller_dim=4,hidden=32).to(U.device)
        if not use_gnn335:
            net335.scorer=nn.Sequential(nn.Linear(17,64),nn.ReLU(),nn.Linear(64,1)).to(U.device)
        net335.load_state_dict(obj335['epoch60_model_state_dict'],strict=True)
        m335=validate_b(net335,use_gnn335,K)
        ref335=next(x for x in report335['results'] if x['model']==method335)
        assert abs(m335['AP']-float(ref335['epoch60_AP']))<2e-6,(seed335,method335,m335)
        print(f'PARITY seed={seed335} {method335}: AP={m335["AP"]:.6f}',flush=True)
        del net335
    torch.cuda.empty_cache()

# Each completed model saved immediately. A resumed run skips complete
# checkpoint+history pairs; incomplete pairs stop rather than overwrite.
for seed335 in SEEDS335:
    for frac335 in BUDGETS335:
        k335=round(U.numel()*frac335)
        assert k335>=16216, 'Insufficient budget for guaranteed greedy coverage'
        reset_rng_b(seed335)
        base335=type(stable)(node_dim=20,edge_dim=17,controller_dim=4,hidden=32).to(U.device)
        gnn335=copy.deepcopy(base335)
        trust335=copy.deepcopy(base335)
        trust335.scorer=nn.Sequential(nn.Linear(17,64),nn.ReLU(),nn.Linear(64,1)).to(U.device)
        for pname335 in base335.state_dict():
            if pname335.startswith(('encoder.','trust_head.','graph_head.')):
                assert torch.equal(gnn335.state_dict()[pname335],trust335.state_dict()[pname335])
        del base335
        for method335,use_gnn335,net335 in (
            ('GNN_scorer',True,gnn335),('Trust_only_scorer',False,trust335)
        ):
            tag335=f'{method335}_{int(frac335*100)}pct_seed{seed335}'
            statepath335=OUT335/f'{tag335}_checkpoint.pt'
            histpath335=OUT335/f'{tag335}_history.csv'
            rowpath335=OUT335/f'{tag335}_result.json'
            trio335=[statepath335,histpath335,rowpath335]
            if all(p.exists() for p in trio335):
                print('SKIPPED completed:',tag335,flush=True)
                continue
            if any(p.exists() for p in trio335):
                raise RuntimeError(f'Partial output for {tag335}; inspect before continuing')
            reset_rng_b(seed335)
            params335=[p for n,p in net335.named_parameters() if not n.startswith('controller.')]
            opt335=torch.optim.AdamW(params335,lr=1e-3,weight_decay=1e-4)
            hist335=[]; best335=-float('inf'); bestepoch335=None; beststate335=None
            print(f'\nSTART seed={seed335} budget={frac335:.0%} model={method335}',flush=True)
            for ep335 in range(1,61):
                net335.train(); weighted335=0.0; total335=0
                for t335 in (5,6,7):
                    batch335=data104[t335]
                    opt335.zero_grad(set_to_none=True)
                    logits335,_,_=forward_b(net335,batch335,use_gnn335,k335,True)
                    loss335=F.binary_cross_entropy_with_logits(logits335,batch335['y'])
                    assert bool(torch.isfinite(loss335)),(seed335,frac335,method335,ep335,'loss')
                    loss335.backward()
                    norm335=torch.nn.utils.clip_grad_norm_(params335,1.0)
                    assert bool(torch.isfinite(norm335)),(seed335,frac335,method335,ep335,'grad')
                    opt335.step()
                    n335=int(batch335['count']);total335+=n335;weighted335+=float(loss335)*n335
                metrics335=validate_b(net335,use_gnn335,k335)
                hist335.append(dict(epoch=ep335,train_BCE=weighted335/total335,**metrics335))
                if metrics335['AP']>best335+1e-5:
                    best335=metrics335['AP'];bestepoch335=ep335
                    beststate335=copy.deepcopy(net335.state_dict())
                if ep335==1 or ep335%10==0:
                    print(f' epoch={ep335:02d} AP={metrics335["AP"]:.6f} '
                          f'AUC={metrics335["AUC"]:.6f} BCE={metrics335["BCE"]:.6f}',flush=True)
            result335=dict(simulation_seed=104,model_seed=seed335,budget_fraction=frac335,
                           retained_edges=k335,model=method335,best_epoch=bestepoch335,
                           best_AP=best335,epoch60_AP=metrics335['AP'],
                           epoch60_AUC=metrics335['AUC'],epoch60_BCE=metrics335['BCE'],
                           epoch60_churn=metrics335['churn'])
            # Save the model first, then history/metadata. A partial save is
            # deliberately detected on the next execution rather than overwritten.
            torch.save(dict(best_model_state_dict=beststate335,
                            epoch60_model_state_dict=copy.deepcopy(net335.state_dict()),
                            simulation_seed=104,model_seed=seed335,method=method335,
                            budget_fraction=frac335,epochs=60),statepath335)
            pd.DataFrame(hist335).to_csv(histpath335,index=False)
            rowpath335.write_text(json.dumps(result335,indent=2),encoding='utf-8')
            print('SAVED:',tag335,flush=True)
        del gnn335,trust335
        torch.cuda.empty_cache()

# Aggregate the three initializations without touching old files.
first335=SOURCE/'budget_study'/'e3_34b_recovered_seed44_25_75'/'completed_results.csv'
assert first335.is_file(), 'Missing seed44 budget results from E3.34b'
rows335=pd.read_csv(first335).to_dict('records')
for seed335 in SEEDS335:
    for frac335 in BUDGETS335:
        for method335,_ in METHODS335:
            path335=OUT335/f'{method335}_{int(frac335*100)}pct_seed{seed335}_result.json'
            rows335.append(json.loads(path335.read_text(encoding='utf-8')))
for seed335 in (44,45,46):
    report335=json.loads((SOURCE/'tags_v3'/f'e3_31_model_seed_{seed335}'/'report.json').read_text())
    for r335 in report335['results']:
        rows335.append(dict(simulation_seed=104,model_seed=seed335,budget_fraction=0.50,
                            retained_edges=K,model=r335['model'],
                            epoch60_AP=float(r335['epoch60_AP']),
                            epoch60_AUC=float(r335['epoch60_AUC']),
                            epoch60_BCE=float(r335['epoch60_BCE']),
                            epoch60_churn=float(r335['epoch60_churn'])))
all335=pd.DataFrame(rows335).drop_duplicates(subset=['simulation_seed','model_seed','budget_fraction','model'])
assert len(all335)==18,len(all335)
pairs335=all335.pivot(index=['model_seed','budget_fraction'],columns='model',values='epoch60_AP')
pairs335['GNN_minus_Trust_AP']=pairs335['GNN_scorer']-pairs335['Trust_only_scorer']
summary335=pairs335.groupby(level='budget_fraction')['GNN_minus_Trust_AP'].agg(['mean','std','min','max'])
all335.to_csv(OUT335/'all_three_seeds_metrics.csv',index=False)
pairs335.to_csv(OUT335/'paired_delta_ap.csv')
summary335.to_csv(OUT335/'budget_replication_summary.csv')
print('\nE3.35 COMPLETED — ALL 3 SEEDS, BUDGETS 25% / 50% / 75%')
print('\nPAIRED AP RESULTS')
print(pairs335.round(6).to_string())
print('\nMEAN DELTA AP BY BUDGET')
print(summary335.round(6).to_string())
print('TEST accessed: NO. Existing experiments overwritten: NO.')
print('Saved:',OUT335)

In [ ]:
# E3.35 — replicate budget sensitivity across model seeds 45 and 46
# Run in Kaggle AFTER E3.RECOVERY and E3.34b.
# Uses ORIGINAL restored forward_b, validate_b, reset_rng_b functions.
# Train/Validation only; no test access and no previous artifacts overwritten.
from pathlib import Path
import copy, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

required_335 = ['data104','stable','U','V','K','SOURCE','forward_b','validate_b','reset_rng_b']
missing_335 = [v for v in required_335 if v not in globals()]
assert not missing_335, f'Missing restored objects: {missing_335}'
assert sorted(data104) == [5,6,7,8,9]
assert U.is_cuda and V.is_cuda and U.numel() == 146117 and K == 73058
assert data104[5]['xn'].shape == (16216,20)
OUT335 = SOURCE/'budget_study'/'e3_35_replicate_seed45_46'
OUT335.mkdir(parents=True,exist_ok=True)
SEEDS335 = (45,46)
BUDGETS335 = (0.25,0.75)
METHODS335 = (('GNN_scorer',True),('Trust_only_scorer',False))

# Confirm the restored scorer/validation implementation against existing
# epoch-60 checkpoints before any new training.
print('E3.35 — REPLICATION PRECHECK',flush=True)
for seed335 in SEEDS335:
    report335 = json.loads((SOURCE/'tags_v3'/f'e3_31_model_seed_{seed335}'/'report.json').read_text())
    for method335,use_gnn335 in METHODS335:
        obj335=torch.load(SOURCE/'tags_v3'/f'e3_31_model_seed_{seed335}'/f'{method335}_checkpoint.pt',
                          map_location='cpu', weights_only=True)
        net335=type(stable)(node_dim=20,edge_dim=17,controller_dim=4,hidden=32).to(U.device)
        if not use_gnn335:
            net335.scorer=nn.Sequential(nn.Linear(17,64),nn.ReLU(),nn.Linear(64,1)).to(U.device)
        net335.load_state_dict(obj335['epoch60_model_state_dict'],strict=True)
        m335=validate_b(net335,use_gnn335,K)
        ref335=next(x for x in report335['results'] if x['model']==method335)
        assert abs(m335['AP']-float(ref335['epoch60_AP']))<2e-6,(seed335,method335,m335)
        print(f'PARITY seed={seed335} {method335}: AP={m335["AP"]:.6f}',flush=True)
        del net335
    torch.cuda.empty_cache()

# Each completed model saved immediately. A resumed run skips complete
# checkpoint+history pairs; incomplete pairs stop rather than overwrite.
for seed335 in SEEDS335:
    for frac335 in BUDGETS335:
        k335=round(U.numel()*frac335)
        assert k335>=16216, 'Insufficient budget for guaranteed greedy coverage'
        reset_rng_b(seed335)
        base335=type(stable)(node_dim=20,edge_dim=17,controller_dim=4,hidden=32).to(U.device)
        gnn335=copy.deepcopy(base335)
        trust335=copy.deepcopy(base335)
        trust335.scorer=nn.Sequential(nn.Linear(17,64),nn.ReLU(),nn.Linear(64,1)).to(U.device)
        for pname335 in base335.state_dict():
            if pname335.startswith(('encoder.','trust_head.','graph_head.')):
                assert torch.equal(gnn335.state_dict()[pname335],trust335.state_dict()[pname335])
        del base335
        for method335,use_gnn335,net335 in (
            ('GNN_scorer',True,gnn335),('Trust_only_scorer',False,trust335)
        ):
            tag335=f'{method335}_{int(frac335*100)}pct_seed{seed335}'
            statepath335=OUT335/f'{tag335}_checkpoint.pt'
            histpath335=OUT335/f'{tag335}_history.csv'
            rowpath335=OUT335/f'{tag335}_result.json'
            trio335=[statepath335,histpath335,rowpath335]
            if all(p.exists() for p in trio335):
                print('SKIPPED completed:',tag335,flush=True)
                continue
            if any(p.exists() for p in trio335):
                raise RuntimeError(f'Partial output for {tag335}; inspect before continuing')
            reset_rng_b(seed335)
            params335=[p for n,p in net335.named_parameters() if not n.startswith('controller.')]
            opt335=torch.optim.AdamW(params335,lr=1e-3,weight_decay=1e-4)
            hist335=[]; best335=-float('inf'); bestepoch335=None; beststate335=None
            print(f'\nSTART seed={seed335} budget={frac335:.0%} model={method335}',flush=True)
            for ep335 in range(1,61):
                net335.train(); weighted335=0.0; total335=0
                for t335 in (5,6,7):
                    batch335=data104[t335]
                    opt335.zero_grad(set_to_none=True)
                    logits335,_,_=forward_b(net335,batch335,use_gnn335,k335,True)
                    loss335=F.binary_cross_entropy_with_logits(logits335,batch335['y'])
                    assert bool(torch.isfinite(loss335)),(seed335,frac335,method335,ep335,'loss')
                    loss335.backward()
                    norm335=torch.nn.utils.clip_grad_norm_(params335,1.0)
                    assert bool(torch.isfinite(norm335)),(seed335,frac335,method335,ep335,'grad')
                    opt335.step()
                    n335=int(batch335['count']);total335+=n335;weighted335+=float(loss335)*n335
                metrics335=validate_b(net335,use_gnn335,k335)
                hist335.append(dict(epoch=ep335,train_BCE=weighted335/total335,**metrics335))
                if metrics335['AP']>best335+1e-5:
                    best335=metrics335['AP'];bestepoch335=ep335
                    beststate335=copy.deepcopy(net335.state_dict())
                if ep335==1 or ep335%10==0:
                    print(f' epoch={ep335:02d} AP={metrics335["AP"]:.6f} '
                          f'AUC={metrics335["AUC"]:.6f} BCE={metrics335["BCE"]:.6f}',flush=True)
            result335=dict(simulation_seed=104,model_seed=seed335,budget_fraction=frac335,
                           retained_edges=k335,model=method335,best_epoch=bestepoch335,
                           best_AP=best335,epoch60_AP=metrics335['AP'],
                           epoch60_AUC=metrics335['AUC'],epoch60_BCE=metrics335['BCE'],
                           epoch60_churn=metrics335['churn'])
            # Save the model first, then history/metadata. A partial save is
            # deliberately detected on the next execution rather than overwritten.
            torch.save(dict(best_model_state_dict=beststate335,
                            epoch60_model_state_dict=copy.deepcopy(net335.state_dict()),
                            simulation_seed=104,model_seed=seed335,method=method335,
                            budget_fraction=frac335,epochs=60),statepath335)
            pd.DataFrame(hist335).to_csv(histpath335,index=False)
            rowpath335.write_text(json.dumps(result335,indent=2),encoding='utf-8')
            print('SAVED:',tag335,flush=True)
        del gnn335,trust335
        torch.cuda.empty_cache()

# Aggregate the three initializations without touching old files.
first335=SOURCE/'budget_study'/'e3_34b_recovered_seed44_25_75'/'completed_results.csv'
assert first335.is_file(), 'Missing seed44 budget results from E3.34b'
rows335=pd.read_csv(first335).to_dict('records')
for seed335 in SEEDS335:
    for frac335 in BUDGETS335:
        for method335,_ in METHODS335:
            path335=OUT335/f'{method335}_{int(frac335*100)}pct_seed{seed335}_result.json'
            rows335.append(json.loads(path335.read_text(encoding='utf-8')))
for seed335 in (44,45,46):
    report335=json.loads((SOURCE/'tags_v3'/f'e3_31_model_seed_{seed335}'/'report.json').read_text())
    for r335 in report335['results']:
        rows335.append(dict(simulation_seed=104,model_seed=seed335,budget_fraction=0.50,
                            retained_edges=K,model=r335['model'],
                            epoch60_AP=float(r335['epoch60_AP']),
                            epoch60_AUC=float(r335['epoch60_AUC']),
                            epoch60_BCE=float(r335['epoch60_BCE']),
                            epoch60_churn=float(r335['epoch60_churn'])))
all335=pd.DataFrame(rows335).drop_duplicates(subset=['simulation_seed','model_seed','budget_fraction','model'])
assert len(all335)==18,len(all335)
pairs335=all335.pivot(index=['model_seed','budget_fraction'],columns='model',values='epoch60_AP')
pairs335['GNN_minus_Trust_AP']=pairs335['GNN_scorer']-pairs335['Trust_only_scorer']
summary335=pairs335.groupby(level='budget_fraction')['GNN_minus_Trust_AP'].agg(['mean','std','min','max'])
all335.to_csv(OUT335/'all_three_seeds_metrics.csv',index=False)
pairs335.to_csv(OUT335/'paired_delta_ap.csv')
summary335.to_csv(OUT335/'budget_replication_summary.csv')
print('\nE3.35 COMPLETED — ALL 3 SEEDS, BUDGETS 25% / 50% / 75%')
print('\nPAIRED AP RESULTS')
print(pairs335.round(6).to_string())
print('\nMEAN DELTA AP BY BUDGET')
print(summary335.round(6).to_string())
print('TEST accessed: NO. Existing experiments overwritten: NO.')
print('Saved:',OUT335)

In [ ]:
# E3.36 — ONE-TIME FINAL HELD-OUT TEST (NO TRAINING)
# Append AFTER executing the existing E3.RECOVER cell (159).
# Fixed protocol: scenario 104; seeds 44/45/46; budgets .25/.50/.75;
# methods GNN_scorer and Trust_only_scorer; EPOCH 60 weights only.
# Test times 10,11; chronological history updated before each time step.
# Never chooses a winning configuration from test results.
from pathlib import Path
from datetime import datetime, timezone
import json, zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import average_precision_score, roc_auc_score, log_loss

required_336 = ('SOURCE','ROOT','U','V','N','M','K','stable','data104',
                'edge_scaler104','node_scaler104','type_onehot','coverage_topk_gpu','validate324')
missing_336 = [name for name in required_336 if name not in globals()]
assert not missing_336, f'Run recovery and E3.34b setup first: {missing_336}'
assert SOURCE == Path('/kaggle/working/phase2_adaptive_siot/E3_tags_v3/scenarios/B_owner_correlated/seed_104')
assert U.is_cuda and V.is_cuda and (N,M,K)==(16216,146117,73058)
assert sorted(data104)==[5,6,7,8,9], 'Unexpected training/validation restoration'

METHODS_336 = [('GNN_scorer',True), ('Trust_only_scorer',False)]
SEEDS_336 = (44,45,46)
BUDGETS_336 = (.25,.50,.75)
TIMES_336 = (10,11)

def checkpoint_336(seed, frac, method):
    if frac == .50:
        return SOURCE/'tags_v3'/f'e3_31_model_seed_{seed}'/f'{method}_checkpoint.pt'
    if seed == 44:
        return SOURCE/'budget_study'/'e3_34b_recovered_seed44_25_75'/f'{method}_{int(frac*100)}pct_seed{seed}_checkpoint.pt'
    return SOURCE/'budget_study'/'e3_35_replicate_seed45_46'/f'{method}_{int(frac*100)}pct_seed{seed}_checkpoint.pt'

# Preflight only: verify existing inputs/checkpoints. t10/t11 causal features will be computed in memory.
cp_paths_336 = [checkpoint_336(seed,frac,method)
                for seed in SEEDS_336 for frac in BUDGETS_336 for method,_ in METHODS_336]
required_files_336 = [SOURCE/'observed/train.csv.gz', SOURCE/'observed/validation.csv.gz',
                      SOURCE/'observed/test.csv.gz',
                      SOURCE/'causal_features/edge_index_mapping.csv.gz',
                      SOURCE/'causal_features/snapshot_t09.npz', *cp_paths_336]
missing_files_336 = [str(p) for p in required_files_336 if not p.is_file()]
assert not missing_files_336, 'Missing files; TEST has not been read:\n'+'\n'.join(missing_files_336)
for cp_336 in cp_paths_336:
    obj_336 = torch.load(cp_336, map_location='cpu',weights_only=True)
    assert 'epoch60_model_state_dict' in obj_336, f'Wrong checkpoint format: {cp_336}'
    assert isinstance(obj_336['epoch60_model_state_dict'],dict)
    del obj_336

# Existing validation parity, before reading test labels.
ref_336 = json.loads((SOURCE/'tags_v3/e3_31_model_seed_44/report.json').read_text())
model_336 = type(stable)(20,17,4,hidden=32).to(U.device)
model_336.load_state_dict(torch.load(checkpoint_336(44,.50,'GNN_scorer'),
                                    map_location='cpu',weights_only=True)['epoch60_model_state_dict'], strict=True)
parity_336 = validate324(model_336,True,data104)
reference_336 = next(x for x in ref_336['results'] if x['model']=='GNN_scorer')
assert abs(parity_336['AP']-float(reference_336['epoch60_AP']))<2e-6, 'Validation parity failed; DO NOT access test'
del model_336

OUT_336 = SOURCE/'final_test'/'e3_36_frozen_epoch60_3seeds_3budgets'
assert not OUT_336.exists(), f'Final test already exists. Do not rerun or overwrite: {OUT_336}'
# Frozen choices declared and persisted BEFORE reading Test data.
OUT_336.mkdir(parents=True,exist_ok=False)
manifest_336 = dict(experiment='E3.36', scenario='B_owner_correlated', simulation_seed=104,
    model_seeds=list(SEEDS_336), budgets=list(BUDGETS_336),
    methods=[x[0] for x in METHODS_336], epochs=60, checkpoint='epoch60_model_state_dict',
    train_times=[5,6,7], validation_times=[8,9], test_times=list(TIMES_336),
    metrics=['AP','AUC','BCE','churn'],
    aggregation='pool predictions across t10 and t11; paired GNN-minus-Trust by seed and budget',
    historical_policy='Strict E3.29 accumulator reconstruction in memory; at t11, t10 outcomes are history; no current/future outcomes used in features',
    model_selection='NONE on test', optimizer_steps=0,
    checkpoint_paths=[str(x) for x in cp_paths_336],
    frozen_at_utc=datetime.now(timezone.utc).isoformat())
(OUT_336/'FROZEN_PROTOCOL_BEFORE_TEST.json').write_text(json.dumps(manifest_336,indent=2),encoding='utf-8')
print('Protocol frozen, validation parity passed. Reading held-out test exactly once.',flush=True)

# The same strictly historical feature recipe as the original E3.RECOVER cell.
train_336 = pd.read_csv(SOURCE/'observed/train.csv.gz')
val_336 = pd.read_csv(SOURCE/'observed/validation.csv.gz')
test_336 = pd.read_csv(SOURCE/'observed/test.csv.gz')
assert train_336.time.between(0,7).all()
assert val_336.time.between(8,9).all()
assert test_336.time.isin(TIMES_336).all() and set(test_336.time)==set(TIMES_336)
assert all(x.success.isin([0,1]).all() for x in (train_336,val_336,test_336))
events_336 = pd.concat([train_336,val_336,test_336],ignore_index=True)
top_336 = pd.read_csv(SOURCE/'causal_features/edge_index_mapping.csv.gz')
assert len(top_336)==M
u_336=U.detach().cpu().numpy(); v_336=V.detach().cpu().numpy()
assert np.array_equal(top_336.u.to_numpy(np.int64)-1,u_336)
assert np.array_equal(top_336.v.to_numpy(np.int64)-1,v_336)
keys_336=u_336*N+v_336
src_336=events_336.source.to_numpy(np.int64)-1
dst_336=events_336.target.to_numpy(np.int64)-1
assert np.all(src_336<dst_336)
event_keys_336=src_336*N+dst_336
edge_ids_336=np.searchsorted(keys_336,event_keys_336)
assert (edge_ids_336<M).all() and np.array_equal(keys_336[edge_ids_336],event_keys_336)
t_336=events_336.time.to_numpy(np.int16)
success_336=events_336.success.to_numpy(np.float64)
relations_336=top_336[['OOR','POR','C_LOR','SOR','SOR2']].to_numpy(np.float32)
static_degree_336=np.bincount(np.concatenate([u_336,v_336]),minlength=N)
degree_336=np.log1p(static_degree_336).astype(np.float32)

# Rebuild causal history in memory using the E3.29 accumulator recipe.
# No missing t10/t11 snapshots are needed or written to existing folders.
# Crucially: history at time t includes only observations with time < t.
def historical_features_336(current_time):
    edge_s = np.zeros(M, dtype=np.float32)
    edge_f = np.zeros(M, dtype=np.float32)
    decay_s = np.zeros(M, dtype=np.float32)
    decay_f = np.zeros(M, dtype=np.float32)
    node_s = np.zeros(N, dtype=np.float32)
    node_f = np.zeros(N, dtype=np.float32)
    last_seen = np.full(M, -1, dtype=np.int16)
    for tt in range(current_time + 1):
        if tt > 0:
            decay_s *= np.float32(0.80)
            decay_f *= np.float32(0.80)
        if tt == current_time:
            break  # do not update with current-time labels
        mask = t_336 == tt
        ids = edge_ids_336[mask]
        successes = success_336[mask].astype(np.float32)
        failures = np.float32(1.0) - successes
        np.add.at(edge_s, ids, successes)
        np.add.at(edge_f, ids, failures)
        np.add.at(decay_s, ids, successes)
        np.add.at(decay_f, ids, failures)
        np.add.at(node_s, u_336[ids], successes)
        np.add.at(node_s, v_336[ids], successes)
        np.add.at(node_f, u_336[ids], failures)
        np.add.at(node_f, v_336[ids], failures)
        last_seen[ids] = tt
    hist_count = edge_s + edge_f
    decay_count = decay_s + decay_f
    hist_rate = (edge_s + np.float32(1.0)) / (hist_count + np.float32(2.0))
    decay_rate = (decay_s + np.float32(1.0)) / (decay_count + np.float32(2.0))
    node_rate = (node_s + np.float32(1.0)) / (node_s + node_f + np.float32(2.0))
    node_conf = (node_s + node_f) / (node_s + node_f + np.float32(5.0))
    never_seen = last_seen == -1
    age = np.where(never_seen, 12, current_time - last_seen).astype(np.float32)
    out = np.column_stack((hist_count, hist_rate, decay_count, decay_rate,
        hist_count / (hist_count + np.float32(5.0)), age / np.float32(12.0),
        never_seen.astype(np.float32), node_rate[u_336], node_rate[v_336],
        node_conf[u_336], node_conf[v_336], decay_rate - hist_rate)).astype(np.float32)
    assert out.shape == (M, 12) and np.isfinite(out).all()
    assert int(out[:,0].sum()) == int(np.count_nonzero(t_336 < current_time))
    return out

# Strict parity check against the genuine previously-saved E3.29 t=9 snapshot.
with np.load(SOURCE/'causal_features/snapshot_t09.npz') as old_336:
    expected_336 = old_336['features']
    assert int(old_336['time']) == 9
trainval_336 = t_336 < 10
# At t=9, test outcomes must have zero effect. Compare all features.
reconstructed9_336 = historical_features_336(9)
assert np.allclose(reconstructed9_336, expected_336, rtol=1e-6, atol=1e-6), \
    'E3.29 historical-feature parity failed; STOP'
del reconstructed9_336, expected_336
print('E3.29 feature parity t=9: PASSED', flush=True)

data_test_336={}
for current_336 in TIMES_336:
    past_336=t_336<current_336
    pu_336=src_336[past_336]; pv_336=dst_336[past_336]; ps_336=success_336[past_336]
    node_count_336=np.bincount(np.concatenate([pu_336,pv_336]),minlength=N)
    node_success_336=(np.bincount(pu_336,weights=ps_336,minlength=N)
                      +np.bincount(pv_336,weights=ps_336,minlength=N))
    node_raw_336=np.column_stack((degree_336,np.log1p(node_count_336),
        (node_success_336+1)/(node_count_336+2),node_count_336/(node_count_336+5))).astype(np.float32)
    hist_336=historical_features_336(current_336)
    assert hist_336.shape==(M,12)
    assert abs(float(hist_336[:,0].sum())-int(past_336.sum()))<1.0, f'Historical snapshot mismatch at t={current_336}'
    ru_336=node_raw_336[u_336,2]; rv_336=node_raw_336[v_336,2]
    cu_336=node_raw_336[u_336,3]; cv_336=node_raw_336[v_336,3]
    ends_336=np.column_stack(((ru_336+rv_336)/2,abs(ru_336-rv_336),
                              (cu_336+cv_336)/2,abs(cu_336-cv_336))).astype(np.float32)
    edge_raw_336=np.column_stack((np.log1p(hist_336[:,0]),hist_336[:,1],
        np.log1p(hist_336[:,2]),hist_336[:,3:7],ends_336,hist_336[:,11],relations_336)).astype(np.float32)
    assert edge_raw_336.shape==(M,17)
    curr_336=t_336==current_336
    targets_336=edge_ids_336[curr_336]
    y_336=(1-success_336[curr_336]).astype(np.float32)
    assert len(targets_336)==len(y_336)>0
    xe_336=edge_scaler104.transform(edge_raw_336).astype(np.float32)
    xc_336=node_scaler104.transform(node_raw_336).astype(np.float32)
    xn_336=np.concatenate((type_onehot.astype(np.float32),xc_336),axis=1)
    data_test_336[current_336]=dict(time=current_336,
        xn=torch.as_tensor(xn_336,device=U.device),
        xe=torch.as_tensor(xe_336,device=U.device),
        xc=torch.as_tensor(xc_336,device=U.device),
        target=torch.as_tensor(targets_336,device=U.device),
        y=torch.as_tensor(y_336,device=U.device),count=int(len(y_336)))
    print(f't={current_336} historical features checked; n={len(y_336)}',flush=True)

@torch.no_grad()
def forward_test_336(net, batch, use_gnn, k):
    # Exact selector and inference equations from E3.34b (no training).
    xn,xe,ids=batch['xn'],batch['xe'],batch['target']
    if use_gnn:
        h=net.encoder(xn,U,V,xn.new_ones(M))
        inp=torch.cat((h[U]+h[V],(h[U]-h[V]).abs(),xe),dim=1)
        scores=net.scorer(inp).squeeze(-1)
    else:
        scores=net.scorer(xe).squeeze(-1)
    hard,coverage=coverage_topk_gpu(scores,k)
    z=net.encoder(xn,U,V,hard)
    a,b=U[ids],V[ids]
    pairs=torch.cat((z[a]+z[b],(z[a]-z[b]).abs(),z[a]*z[b]),dim=1)
    logits=net.trust_head(xe[ids]).squeeze(-1)+net.graph_head(pairs).squeeze(-1)
    return logits,hard,int(coverage)

records_336=[]
for seed_336 in SEEDS_336:
    for frac_336 in BUDGETS_336:
        k_336=round(M*frac_336)
        for method_336,use_gnn_336 in METHODS_336:
            net_336=type(stable)(20,17,4,hidden=32).to(U.device)
            if not use_gnn_336:
                net_336.scorer=nn.Sequential(nn.Linear(17,64),nn.ReLU(),nn.Linear(64,1)).to(U.device)
            ckpt_336=torch.load(checkpoint_336(seed_336,frac_336,method_336),map_location='cpu',weights_only=True)
            net_336.load_state_dict(ckpt_336['epoch60_model_state_dict'],strict=True)
            net_336.eval()
            ys_336=[]; probs_336=[]; masks_336=[]; covers_336=[]
            with torch.inference_mode():
                for tstep_336 in TIMES_336:
                    logit_336,mask_336,coverage_336=forward_test_336(net_336,data_test_336[tstep_336],use_gnn_336,k_336)
                    ys_336.append(data_test_336[tstep_336]['y'].cpu().numpy())
                    probs_336.append(torch.sigmoid(logit_336).cpu().numpy())
                    masks_336.append(mask_336.cpu().numpy().astype(bool))
                    covers_336.append(int(coverage_336))
            ys_336=np.concatenate(ys_336)
            probs_336=np.clip(np.concatenate(probs_336),1e-7,1-1e-7)
            assert all(int(x.sum())==k_336 for x in masks_336)
            assert len(np.unique(ys_336))==2, 'AUC undefined for a single-class test set'
            result_336=dict(simulation_seed=104,model_seed=seed_336,
                budget_fraction=frac_336,retained_edges=k_336,model=method_336,
                test_AP=float(average_precision_score(ys_336,probs_336)),
                test_AUC=float(roc_auc_score(ys_336,probs_336)),
                test_BCE=float(log_loss(ys_336,probs_336,labels=[0,1])),
                test_churn=float(np.count_nonzero(masks_336[0]!=masks_336[1])/(2*k_336)),
                coverage_t10=covers_336[0],coverage_t11=covers_336[1],
                test_n=int(len(ys_336)),test_failure_rate=float(ys_336.mean()))
            records_336.append(result_336)
            print(f'SEED {seed_336} | {frac_336:.0%} | {method_336}: AP={result_336["test_AP"]:.6f} AUC={result_336["test_AUC"]:.6f} BCE={result_336["test_BCE"]:.6f} churn={result_336["test_churn"]:.6f}',flush=True)
            del net_336,ckpt_336
        torch.cuda.empty_cache()

results_336=pd.DataFrame(records_336)
assert len(results_336)==18
results_336.to_csv(OUT_336/'test_metrics_all_18.csv',index=False)
paired_336=results_336.pivot(index=['model_seed','budget_fraction'],columns='model',values=['test_AP','test_AUC','test_BCE','test_churn'])
for metric_336 in ('test_AP','test_AUC','test_BCE','test_churn'):
    paired_336[(metric_336,'GNN_minus_Trust')]=paired_336[(metric_336,'GNN_scorer')]-paired_336[(metric_336,'Trust_only_scorer')]
paired_336.to_csv(OUT_336/'test_paired_differences.csv')
summary_336=results_336.groupby(['budget_fraction','model'])[['test_AP','test_AUC','test_BCE','test_churn']].agg(['mean','std'])
summary_336.to_csv(OUT_336/'test_summary_mean_std.csv')
(OUT_336/'SUCCESS.json').write_text(json.dumps(dict(finished_at_utc=datetime.now(timezone.utc).isoformat(),
    test_rows=18,train_steps=0,original_checkpoints_unchanged=True),indent=2),encoding='utf-8')
print('\nE3.36 COMPLETED — frozen held-out test, 18 evaluations, 0 training steps.')
print(summary_336.round(6).to_string())
print('Saved:',OUT_336)
